# 📊 SHAP Interpretation Report: RANDOM_FOREST
This notebook provides a post-hoc explanation of the model's predictions using **tree** SHAP.

---

### 🔍 What are SHAP Values?
**SHAP (SHapley Additive exPlanations)** decomposes a model's prediction into the contribution of each individual feature. 
* **Magnitude:** A larger absolute SHAP value means the feature had a bigger impact on the output.
* **Direction:** A positive SHAP value means the feature pushed the prediction *higher*, while a negative value pushed it *lower*.
* **Interpretation:** For any given sample, the sum of SHAP values plus the base value (average model output) equals the actual model prediction.

### 🧪 Methodology
**Tree SHAP** is an optimized algorithm for tree-based models (like Random Forest or XGBoost). It leverages the internal structure of the trees to calculate exact SHAP values significantly faster than model-agnostic methods.

---

### 📋 Metadata
**Model Architecture:** RANDOM_FOREST  
**Analysis Context:** tabular  
**Dataset Scope:** whole

---


In [1]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# Data provided by the explainer
all_shap_dict = {0: [[-0.12882951172061666, -0.008301979026064452, -0.004180297766117158, -0.03375413565499244, 0.02674874133209094, -0.03042615049763333], [-0.07866821009090592, -0.0082517759306007, -0.006936091899532251, -0.08073743826613226, 0.0013733736955561441, -0.007189857508384394], [-0.13515131652472642, -0.004980663599548114, -0.003483189544468591, 0.04485374976201017, 0.0011368858426556423, -0.05010213260258905], [-0.059848328465815136, -0.004342758067567689, -0.0002913324479366287, -0.09309425710790602, -0.0004810917152086385, -0.017352232195566062], [-0.1324017116529388, -0.007862958239774474, -0.00293135030799446, -0.024863724440870583, 0.00014266193859551946, -0.01195370161074246], [-0.24219511576617683, -0.000973206014538036, 0.0004304599322822375, 0.09281385936900075, -0.00434337354430868, -0.01382804966168503], [-0.003001079496425871, -0.013918348933339325, -0.01665822717090674, -0.1548558165470568, 0.0006090493483564554, 0.06138924862713926], [-0.09978541281089345, -0.00431215608786282, -0.000917183168545003, -0.06226453009293204, 0.012864407921468255, -0.01894967121578021], [-0.22936510902496285, 0.0021119848383978936, -0.0030222767190681145, 0.0686463467595503, -0.0022921096562696073, -0.013725678302911038], [0.6562056141236823, 0.0006468152703834199, 0.01235067453536752, 0.04940026489348019, 0.013162782721448461, 0.025510176016961966], [0.4889278126938745, -0.0026284573357367854, 0.0021735496970413035, 0.2678779024055831, 0.009398664385058052, -0.03571899565534587], [-0.1333130720879973, -0.002820232624116736, -0.002671207017943448, -0.01772717572080424, 0.0003258848021412802, -0.016704197351279208], [-0.10126291470676438, 0.0017709145271721592, 0.0040128427215277256, -0.06719520875943936, -0.002188485346527207, -0.008880481769302267], [-0.13964187264993577, -0.02073646813910325, -0.004743222659993646, 0.11596181055509856, -0.005829420696988315, -0.04798571054454962], [0.21370324033085, 0.02129235864353654, -0.006560317409871283, 0.3759122184373317, 0.002686154563263611, 0.028908604287147085], [-0.1525826271219774, 0.09181388190128409, 0.01437862731847467, 0.18020350919091876, 0.0876227825105389, 0.0874762468356819], [0.0339126985822719, -0.015063027035544322, 0.0621875141738575, -0.1603649842014339, -0.005577978022283331, -0.005126772516475646], [-0.12322658021361457, 0.03328428688690989, 0.00022149040973078752, -0.017733493969944475, -0.00026847540730792703, -0.01381718802323359], [-0.14763094157423434, -0.007718248182667612, -0.004810750902110619, -0.052718428419106775, -0.0029437450503266567, 0.03652322523955747], [0.2072637037844266, -0.009839810777608699, -0.012705424906526038, -0.2468807293993562, 0.0009259622860504194, -0.057768238802112765], [-0.13498057157286458, -0.008823553965321956, -0.0021297777393626495, -0.016596936356039833, -0.004249044668036809, -0.013630115698374062], [0.5632910755134818, 0.002758080783388963, 0.003281093031333149, 0.01224715817054912, -0.0017948386066578513, 0.17758124063171346], [0.5531761338013166, 0.011114409861530668, 0.014152157896498692, 0.13688003750817235, -0.0017172394487929721, 0.06961328826006206], [-0.12874061295592898, -0.0070262032855689174, -0.0020386783796055336, -0.0183129830408505, 0.0005727199464562147, -0.01571741688767675], [-0.1288322626889665, -0.005604983091262358, -0.003675319744096735, -0.06692454110940543, -0.00378008159152298, 0.03257385489192123], [-0.10194793243054233, 0.0011824198188054927, 0.000771231258693824, -0.05629699765512049, 0.00012527917231631058, -0.020052823693564367], [-0.11032679306029451, 0.0022387029828785775, -0.0023477855409477794, -0.06702993633037722, 0.00154614811538851, 0.0030096638333524735], [-0.04788919109437663, 0.0009210146034308865, -0.0009080326030592854, -0.10204578287225095, 8.972725570489478e-05, -0.01885084453314631], [0.03048865234750004, 0.0007161206931235494, -0.00530517948958625, -0.16727133265705924, 0.00023763891758076753, -0.026513995049654553], [-0.15663179308145872, 0.015113385466538308, -0.0007238207533098616, -0.03494702540956613, -0.003153664641272897, 0.009349585085735855], [-0.20059904778145649, -0.0018095816775514201, 0.00045340213172998585, 0.10083268372974546, 0.013644717149088983, -0.047143483075365596], [-0.0066799787581174885, -0.009097208372201323, -0.0029892113491419322, -0.10658221900960961, -0.0029120413510817828, -0.04548267449318158], [0.07690854697600462, -0.018094649877889155, -0.015253388251124017, -0.21261329761971345, -0.008019819824286534, 0.06601228179962264], [-0.09038085184958511, -0.005061878760727179, -0.0029412392882969024, -0.06434994769809994, -0.002776662662002392, -0.014899419741288486], [-0.1840346595059563, -0.009675023399517783, -0.002635200623301013, -0.019301757717462353, -0.002380768142178028, 0.03872852049952642], [-0.26029323232550505, 0.02302341907405265, -0.0016022736272995998, 0.08308854691558787, -0.005682953685862717, 0.03174696983950251], [0.36529065329247723, -0.010273407087872579, 0.00700810646693634, 0.33651978304535396, 0.0027383881584207392, 0.0713064761246791], [-0.10937899301094667, -0.013685793887710317, -0.001933289089001746, -0.03943221351255463, -0.0026060523064734766, 0.0020152306955759387], [-0.00040536171088579237, -0.01874963552130028, 0.0013320027506193838, -0.08416007971906822, 0.0022655181406555512, -0.04648656158707935], [-0.07960434396483687, -0.008144711666413978, -0.001442174128452117, -0.06740597537866304, 0.0006577302808894221, -0.02152934867193495], [-0.160933266671994, 0.002997891716718388, -0.0021681308823372093, -0.044431364249015004, -0.0004973661584719736, 0.029622236245099375], [-0.20994463147470782, -0.007917195632622668, -0.005899187495632016, 0.03957963135203815, -0.0017914467332840842, 0.005562829984207631], [-0.15783349347776943, 0.0007024465109263402, -0.0037803554602043508, -0.04490084560303151, -0.0007713734199670023, 0.03625695478337961], [0.057226761421896456, -0.01952415759471379, -0.003801585811957977, -0.1401663473213573, 0.0033423254155412205, -0.057486996109408633], [-0.11253136470025767, -0.007793232852608538, -0.0036876574379776234, -0.03464856949636023, -0.0022291470765606865, -0.019520028436234665], [-0.12313688727329013, -0.006415746035982165, -0.004057395992889035, -0.0502590790598921, 0.003342145170758764, 0.0026169631912949824], [-0.14260138306627104, -0.0049044099401268, 0.002110224866330622, -0.03629250834131067, -0.005152823837658538, 0.012621376509513615], [-0.05624894725568738, -0.005741800395449851, 0.0008624548124274415, -0.1278121015105906, 0.003952345870512628, 0.0357054994591799], [-0.15572804082383043, 0.01766105399532404, -0.01126397812222884, 0.02785577829751657, -0.0032292753376099597, 0.006559144530511519], [-0.21421228810107293, -0.02131527967291785, -0.012255070430476089, 0.13505261488989698, -0.00791836772498451, -0.03107113276996908], [0.22098046631246887, 0.012606623547417298, 0.1000710332385777, -0.10101459136688122, 0.012303953107897813, 0.15742671217265775], [0.18061284561697008, 0.06974248105316223, 0.013079108919511586, 0.08785658012923132, 0.013443569776273664, 0.09488734091177708], [-0.20071693356326045, -0.006971543592536048, -0.0023942971308112872, 0.06989718406891517, -0.0015589304896892022, -0.03130833643547547], [-0.2486741240599109, -0.014229031558388897, 0.023724624700108936, 0.06135136055335503, -0.010548043969282531, 0.028715214334118052], [-0.15540381944615364, 0.001008143952673169, -0.0027739067468618667, -0.051049302404364186, -0.003220059143581796, 0.033195610454954914], [-0.10430445413672186, 0.021890081845993076, -0.0017274130304474637, -0.06904571993113402, -0.0007758088213744302, -0.018946685926315047], [0.09224911308846381, -0.02540111551786971, -0.006042794554992393, -0.04015239614532642, 0.001257136835430123, -0.10349256275332393], [-0.06575001775704703, -0.004006259910031341, -0.0018510286368564925, -0.052356040165963416, -0.0025495661727239723, -0.04288518259547302], [-0.1373530291607763, -0.001723999585143344, 0.0061019319447455, -0.043258225685782754, -0.004372767339345105, 0.002696089826302077], [-0.11413156407770837, -0.005882913428268567, -0.00026892930931388765, -0.03136877938767923, 0.001946291064857279, -0.028204104861886924], [0.568544355771757, 0.00768791939999426, 0.007677927030670632, 0.09113236070052518, 0.021096360580763897, -0.013101858294000754], [0.007972800629412621, -0.027466934240930527, -0.00013490387299206055, -0.03530787091251416, -0.0011415436656376935, -0.04931894289532159], [-0.1462540856270211, -0.013464320871144384, -0.003995846323091865, -0.02933037429044829, 0.015890501311303547, 0.05856338986966657], [-0.10933575878234662, -0.006230152613617961, -0.002330964602090646, -0.03639586219559744, -0.002816347712437706, -0.019550914093909147], [-0.1464788511467111, -0.009155017362546288, 0.004956662182668845, 0.00913862438424304, 0.00045779743614230925, -0.02772207263665398], [-0.12008040700199146, 0.0027271687936474, -0.002013540505279719, -0.050196955704728506, 0.0004871357906860564, -0.011333401372333645], [0.5995929149853201, 0.007259184684066338, 0.0008786146032844593, 0.1136706618536414, 0.009546670970275369, 0.06164195290340988], [-0.10165749459105779, 0.011244105965642629, -0.0017257768802964016, -0.0693252660494024, 0.002892452758306153, -0.019838021203192367], [-0.11848867021558945, -0.0012041756295984502, 0.0024998170537231463, -0.02035795875509877, 0.0009189990996399804, -0.022528011553076394], [-0.1063491059068993, 0.030740916984477902, -0.0014931897126558174, -0.07397845823031633, 0.009552357833907574, -0.014007520968514611], [-0.11170925314688641, -0.01057955403069271, -0.0027407259086564225, -0.08287615271760221, 0.0007402148778096276, 0.03863047092602898], [-0.18984118431856523, -0.009501317812742213, -0.0030001381151315313, 0.04397389813723786, 0.0025783900738745257, -0.011476790821816496], [-0.09979355569507316, -0.00597458613387522, -0.0014031032861040362, -0.06042946611699955, 0.0025937207299405593, -0.01496183302730019], [-0.1589525718362047, 0.027618616468259482, 0.00766364150074533, -0.052615795796966826, 0.0006898237878947436, 0.025975968415954794], [-0.0912044874493675, -0.004929745280318761, -0.0028484230981227136, -0.06645121383605053, -0.0010517359657609194, -0.013924394370379413], [0.4071884691238871, 0.008358443468195479, 0.07670393847048852, -0.1853535403667131, 0.008385201781932729, 0.1396090748237947], [0.18124262997807003, -0.021479713357491978, -0.009040300190603668, -0.2437390931360634, -0.007349201202084289, -0.0648199631174684], [0.005361225961195533, -0.02117837450603421, -0.007203423742557965, -0.0850483702593747, 0.0013733686773936724, -0.050381092797288654], [0.5832597704493682, 0.006228749531059809, 0.0059036509329605125, 0.14205734693475172, 0.01045514073535342, -0.014217616736454245], [-0.16975013725064647, -0.008359763103727366, -0.004117098872585575, -0.03192721122015312, -0.0022012761145897572, 0.037483337599552656], [0.22360590999461755, -0.012873088016673667, -0.019004587273292215, -0.14264113386431423, -0.015914750054022493, -0.024332350786315123], [-0.12094463190578508, -0.005761809331312653, 0.003155955145437161, -0.03927946089068003, -0.001963301588760247, -0.013950084762232527], [0.2235283465646153, -0.031341535053080016, -0.01753351899960671, 0.21255521348859777, -0.00853582963261889, -0.14152785180650448], [-0.08395892116695725, -0.009501234446746736, -0.0035042579729096016, -0.04666995420437702, 0.0010612510881561734, -0.017658311868593998], [-0.11332351966183596, -0.003561900775953609, -0.0028829738613030467, -0.042511634482667884, -0.0008742911487310554, -0.012314503598919873], [0.48590878459198894, 0.010010311389310677, 0.01777081206020738, 0.2653617400237, 0.013138507703430665, -0.026900575936708037], [0.5229063675158182, -0.009678644845862987, 0.006632701903551937, 0.27332447748501537, 0.011191922106477991, -0.00895349083166786], [-0.08336806460351569, -0.005540864299141175, -0.0027481029521956063, -0.06498854286800261, -0.0022180862179199744, -0.021546339059224835], [-0.2173068389478598, -0.009629020536725746, -0.0014234962412449552, 0.03866076649692045, -0.002952431133545174, 0.028074353695788506], [-0.2395024365636488, -0.013812068981231508, 0.0034643075146966826, 0.20089770927516692, 0.00462842629672661, -0.01016927087504318], [-0.15171597593208674, -0.006354671958388057, -0.002537500959413383, 0.0012184615142641268, -0.006736891568448924, -0.014283421095927172], [-0.1253518028736543, -0.007957545782947644, 0.002615364506889841, -0.018809952593212158, -0.0022426633392543865, -0.02866339991782108], [0.21381610287138234, 0.020064487348268323, 0.02225590671115624, -0.03044219340622492, 0.006477685134558077, 0.17068334323119191], [-0.17590566279112252, -0.007966686070646874, 0.0015263651198198461, 0.029026425157241662, -0.002376615703724838, -0.013047159044900215], [-0.18927310655298285, -0.007287663103028533, 0.0002442631226698642, 0.06500068057993426, -0.006352578827612524, -0.031074928552313183], [0.09008778218715556, -0.006153770142668945, -0.0024365503644154262, -0.21299456906593997, 0.0003711975844944291, -0.02912169703623334], [-0.1806885353240903, -0.004542251146217031, -0.001764150037736585, -0.017776062059762284, -0.0024918648324139528, 0.03596992689228303], [-0.11689462598069232, -0.0024941660754410565, -0.00012200293542839475, -0.07248996100451663, -0.001926759022202317, 0.013517515018280782], [0.21661431200716622, -0.017451969450937087, -0.01064620204095081, -0.15465990960660156, -0.011164152955559252, -0.09928461763565749], [-0.11155136631852898, 0.026437260534311134, -0.0018148487623806618, -0.06624923594426828, -0.000421927179703186, -0.01997654899609656], [-0.14850138222583442, -0.008561442899245236, 0.007621864150649947, 0.002764925823882875, -0.0024817621637607617, -0.02875220268569242], [-0.11600444683270399, 0.019222952656089136, 0.0003511049213385791, -0.08028008560885265, -0.005618717003782376, 0.006627525201245774], [-0.07549489597418496, -0.0023556689065093446, -0.002549870563693236, -0.07711811204880696, -0.002074792975837005, -0.020816659530968736], [0.3418004593338409, 0.015006211433476229, -0.003279491718209524, 0.34014537147000456, -0.0009020060430189441, -0.013281788819988358], [-0.09540529189256675, -0.0071955782892523185, 0.01301833619886835, -0.055449010383123255, -0.0013612579078266071, -0.031052912011813906], [-0.09829884694410505, 0.00129173571080345, -0.0023587582798749065, -0.07398838066204398, -0.0023447713998100395, -0.004710978424968573], [-0.1909952596293549, -0.0030673127996167335, -0.003255502335962955, -0.001795414486305278, -0.001588840316486795, 0.025373937573247476], [-0.1083308082756262, -0.004260641854107221, -0.0027676703135126428, -0.0529104580490431, 0.0010259491151238425, -0.013166370622834557], [-0.16493172621691532, -0.0013671345366769192, -0.002544577730815164, -0.032082569233338236, -0.003362811257490836, 0.02854548564190352], [0.11584774574420316, -0.007468089555144606, -0.0037873756259921665, -0.20393964302865777, -0.0010401300222445532, -0.05379814853780578], [0.46584581746541714, 0.018910891214181495, -0.0011510682692607015, 0.23843294107021426, 0.006960060213586804, 0.057424691639192005], [-0.10430314071693987, -0.004602217209951916, -0.0018058498918128991, -0.07130617829878196, -0.0013775072555237873, 0.0029848933730105627], [-0.008286450384863895, 0.003934366510748968, 0.018834533402588675, 0.38109278819071934, 0.023671716895172543, -0.0018752085826220313], [-0.12837331183998102, -0.007132953295044574, -0.001945696836648107, -0.045541237806534754, 0.0024792062520178145, 0.0036039935261909435], [-0.19176020232258215, -0.004472580681987335, -0.0014610803440923865, -0.008943040685878229, -0.0051637046256123855, 0.03750171977126397], [-0.14552571053252483, 0.024752382005205336, -0.001665508421553358, -0.070485776711734, -0.0065169199451, 0.03626367646284984], [0.1452230557281344, -0.023990932784325746, -0.021689035628977327, -0.20329665689203205, -0.010092864534323467, 0.08742677220480322], [0.49498018696967283, 0.030772191652303656, 0.006953987348632414, -0.13063514185693897, 0.018328406747185957, 0.18786695644072937], [-0.20388348506524517, -0.008788331531215202, -0.004205942928057689, 0.037858771440635035, -0.0023154899582600017, 0.010091144708809292], [0.244641612927535, -0.011779235224869292, -0.01308546621032626, -0.27874246348420323, -0.007054504813562957, -0.05372327652790653], [0.21798318493195615, 0.02196621113877059, -0.012077703040438342, 0.014810785743832844, -0.015359490974163621, -0.13401870208567224], [-0.11137491812367337, 0.010253677668102187, -0.0020446093359128494, -0.0899967079515912, -0.004587271259922355, 0.01921482900299795], [-0.18757574829545623, -0.0098256520441859, -0.003865041876640787, -0.0067360506349649196, -0.0009314742313421157, 0.03519063374925635], [0.5269943406387819, 0.03709402477088545, -0.0029608874108697854, 0.22739729360122213, 0.010937711539136096, -0.02627599191108819], [-0.09616952620903912, -0.006166063751785459, 0.006038598950535851, -0.06486193436684438, 0.0012831958763425849, -0.018034270499209436], [-0.2582716645530724, 0.002602760709341007, 0.00253374779461386, 0.06066759105051329, -0.002768312831888755, 0.03196873497334884], [-0.12055042218274675, -0.008452032263831729, -0.0021185317186053084, -0.021335352637586142, 0.007344696405133903, -0.03196502426903055], [-0.2144622322980722, 0.022087387548536314, -0.0028170442343903885, 0.09829251197869952, -0.006745876132351566, -0.04105046114813591], [-0.10009037762934306, -0.004187677908951656, -0.0020565281244573396, -0.06140823571354945, 0.0035752817492800616, -0.016242462372978408], [-0.25655630821917996, 0.020350208541355573, -0.009467819401120226, 0.14761236465985583, -0.014797701879825221, -0.013015029415372222], [-0.13501918410533537, 0.033386546053187104, -0.0005565670146580005, -0.08726879174434025, -0.00447311164352222, 0.03447348940705074], [0.06704894937376571, -0.017695529051898123, -0.0049407947561951856, -0.1370008861443237, -0.0011572038104377257, -0.07133120227757771], [-0.06346585382814174, -0.0013792671137454792, -0.00247349393708585, -0.08620848840900441, -0.0008804713213391339, -0.026002425390683287], [-0.2805644205018833, -0.017516844684510056, 0.0009073476585945006, 0.12096218208864883, -0.007285010170986564, 0.032467697991089554], [0.5948627230127869, -0.0042956803563947606, 0.007074465247419086, -0.032388044931580706, 0.015707735990215597, -0.01648826245451355], [-0.0001415711571330138, -0.005831133218289268, -0.010304021578034814, -0.1210757197636299, -0.0029317971325695125, -0.03050670953129645], [-0.16272005727208255, -0.0077949505540866465, -0.0025722945961371537, 0.010521884907464849, 0.00013574764178668857, -0.017539153656357467], [0.09152085501588467, -0.017852210951390553, 0.008053162525671221, -0.11871737732693889, 0.0026290568729504904, -0.07794446016215026], [-0.09643233560367867, -0.006321652602342029, -0.00400757433916857, -0.04919352937664008, -0.0010948049431784472, -0.023360103134992143], [-0.09603471937988681, 0.015115705122932938, -0.002230041120086074, -0.08177485321363011, 0.0010368990978571924, -0.014022990507187163], [0.08510273996062306, -0.014106294401314529, -0.01559420267123596, -0.08816872890963907, -0.008763376525885548, -0.026816645389055364], [0.6308953952092378, 0.011412544927842157, 0.007349435719881525, 0.10373276561123797, 0.01386092592969565, -0.011902409389240018], [-0.1907079153382421, -0.012241099667546372, -0.003970424705297415, 0.050712427946598776, -0.0010413427317142623, -0.013161645503798418], [0.3204644759476598, 0.04636691550581389, 0.006617804264220124, 0.34739492553929896, 0.011538750112169973, -0.011514025215319986], [-0.09261028756229064, 0.016057798554355473, -0.0015892453001147614, -0.0811394031884278, -0.004985553790424416, -0.014000451570240699], [-0.15742980982330362, -0.0015746789800542618, 0.0012685639454983254, -0.02743160666320523, -0.002675847126705704, 0.012909569123960877], [-0.25706340057272425, -0.006135576665208466, -0.0026877360524499455, 0.08079537168892159, -0.004857193840374593, 0.03020520210850203], [-0.010516473711757082, -0.008297939245154479, 5.2536244356083575e-06, -0.09938771294855168, 0.0020968782158862546, -0.05056000593485874], [0.08495920681873351, -0.010135775435809166, -0.011410594132675722, -0.2335005472682535, -0.0018291865524630102, 0.051785141201653465], [0.23472573033803618, -0.020746966946721894, -0.01353384489342316, -0.28818630397111533, -0.0031549023443953167, -0.05312085503952434], [-0.1583242020128545, 0.024068297440097728, -0.00031608590365598513, 0.006276803560247983, -0.003917258326785466, -0.030745173804669193], [-0.13751725352312014, -0.003686389346376398, -0.0035214380019090935, 0.004741767399741026, -0.0033379608579010697, -0.027850630432338482], [-0.23332299911977764, -0.010652314370394638, -0.0011123230087775487, 0.04829380906329717, -0.004110297887035679, 0.025553649132211493], [-0.12220755642648277, -0.0007237247314863733, -0.004668234351814161, -0.05605536561611155, -0.002615830077947827, 0.010860711203843156], [0.26368514306792207, 0.13329642137631603, 0.015851224130957204, 0.07252228036540809, 0.022581880334395645, -0.010875899491449912], [-0.13434648006487984, -0.004713217964715196, -0.002266068156287325, -0.06118831301035868, -0.0034177273188049414, 0.026662832156072045], [-0.1597427289565766, 0.005902529988296724, 0.0012598227972773707, 0.002200117838853988, -0.0006575583571390666, -0.017524144095025968], [0.6323076533860043, 0.03222462286819021, 0.0006405879032835087, 0.033672184241510805, 0.000534154756989849, -0.006270804887584247], [0.21873054555349836, -0.01754540068026973, 0.015094596730575421, -0.22849177109697658, -0.012532542047799113, -0.07532019036379066], [-0.2340738818934335, 0.04247655290891267, -0.0006719922988140505, 0.10864184550420766, 0.005185639590841759, -0.031051497145047957], [0.2574541152562881, -0.02189086015216989, -0.0010383160097598122, -0.2543763909340595, -0.010643797078543044, -0.0662778463198508], [-0.12821653774215677, 0.001441030933442645, -0.002257104713014222, -0.03391501851380258, -0.0006097006350104288, -0.01629711377390273], [0.341634898311091, -0.020687267662757344, -0.010623529797978842, -0.21121886999378564, -0.019506169421114675, -0.0032471566735515723], [0.4355526314928668, 0.03559749380508874, 0.0092951290785506, 0.3039718127585158, 0.008706037689552587, -0.046247390538862905], [-0.20339468301414093, -0.010507958128868362, -0.002688085258188727, 0.03987459878606664, -0.004097734665187907, 0.003439576566033863], [-0.13814647823084247, -0.0008570067005747567, -0.0018692935504334737, -0.06934338111109013, -0.00026316281731572833, 0.03475201471794898], [-0.015394634902209381, -0.0028856543327802314, 0.04235736275104798, -0.10403976439344148, -0.003768902394862594, -0.020035549584897273], [-0.2653181417788144, -0.01779226919414146, 0.00211650768926857, 0.10944139001371203, -0.01371814928794765, 0.03611066255792311], [0.24743379459223588, -0.01824374663593157, -0.01034549708656662, -0.2549290023835299, -0.0018150141381303602, -0.09385577244331561], [-0.05415678784688198, -0.0056543624906013334, -0.003264840858925946, -0.08847695205857346, -0.00024177543660811631, -0.013948614641742293], [-0.1576407049221086, -0.007763447180671179, -0.0015470066732766877, 0.0050817801964443765, 0.004223321315559589, -0.01359727606928138], [-0.19502511136771358, -0.009984517359898493, -0.0029156669368950927, 0.0017651731484592972, -0.004668377750417237, 0.037918500266465656], [0.11482371666093659, -0.03434840709508795, -0.009316440131222282, 0.13987939025817292, -0.008747143553453837, -0.11051262407585218], [0.08288835394081338, -0.02558959020900588, -0.03492174691799749, 0.13525280754572608, -0.018288651445294537, -0.05795312210883766], [-0.27592156863791845, 2.405513747774049e-05, 0.00991308801731351, 0.13511440136133063, 0.005369874583416387, 0.009715149538380633], [-0.12589804070599594, -0.0005573878944150567, 0.0019187075549576904, -0.08130364478242313, 0.0004288309392566934, 0.030626534888620965], [-0.10140405760219505, -0.00275135217306735, -0.0022850818309792863, -0.06873204016782285, -0.002292434908328598, -0.0018339222064947833], [0.3369708595508407, -0.009621283398803146, 0.0027885726573396654, 0.26012562272851125, 0.00619012282240965, -0.005017708382533246], [0.03227705975673621, 0.00475193444137634, -0.008079825583074211, -0.18391863825504945, -0.005716188206119949, 0.018108991179464243], [-0.08447283909139099, -0.007101605530546546, -0.0027975763920238955, -0.07187339312090173, -0.002412925314700431, -0.01175166055043643], [-0.1707601907591465, -0.030421021664986822, 0.0031217719829462723, 0.16058974182703084, 0.0033300066898961592, -0.05854628210171288], [-0.10378819001987928, -0.006136868528089955, -0.0015845830777671815, -0.06937798928810761, -0.0008910610149020392, 0.003868691928745776], [-0.084398397563625, 0.005727044069327254, -0.0020704282385871848, -0.06765246151040753, 0.0004504122184019179, -0.026910613419553836], [-0.12922339663885712, -0.0023675965950642037, 0.003991986386446904, -0.029127133876318975, -0.0018450599408585443, -0.02017213266868091], [-0.2135176404783176, -0.002846514347029037, -0.004516415266086542, 0.08397537712003944, -0.006225838666766423, -0.03227896836183972], [-0.09681095735058187, -0.004465074186192334, 0.0037478037679063037, -0.067396370214425, -0.0007736856409424257, -0.014711716375764674], [-0.19705884253733252, -0.009600945560750472, 0.045869862378110214, 0.003958716054435798, -0.0009164679460512993, 0.03964918554809593], [-0.08925744874996466, -0.004668485953941695, -0.0030967595945798148, -0.059082724362062064, -0.0006774603145121386, -0.02362712102493952], [-0.1867742624435848, -0.009198737170238195, 0.0017903140277900449, 0.04119903752339005, -0.004872957139090956, 0.02879660520173397], [0.12397692668817344, 0.007679048197045178, -0.0030593859864996038, -0.2004159201659349, -0.011412899278570338, 0.04744968152617885], [-0.23369931085750112, -0.008501819560119228, -0.001318827841066092, 0.09837882630951422, -0.005844040168839628, -0.015610635538384448], [-0.09108794025075952, -0.005188508583452734, -0.003248829675668102, -0.05788750789955869, -2.472134818772937e-05, -0.022972492242373178], [-0.10696999547282805, 0.018414998229270067, 0.010674222263024056, -0.06697586331277679, 0.0032660515992643507, -0.016944413305953743], [-0.09140207638638673, -0.0014933535499046292, -0.0025471629669234904, -0.06469096298351547, 0.0026443810168701835, -0.022920825130139582], [0.1803255481736887, -0.01574293152605989, -0.006778405628180209, -0.15172458143855266, 0.008519185625904948, -0.06825881520680152], [-0.22410982686536848, -0.008118100613055356, -0.004049480474049277, 0.07799022424637768, -0.00294368023861688, -0.01873795958469962], [-0.05628470717665937, -0.006743624142349524, -0.0038537160849229576, -0.09015870030294935, -0.00016414155434209848, -0.01820511073877674], [-0.0906146953013157, -0.002113588917472861, -0.0029815671149405285, -0.06883027126614298, -0.00018469517708874785, -0.013685182223038973], [0.10600385485502314, -0.004420553268485404, -0.021370305241391295, 0.18873333793145156, -0.011483437976270078, -0.10959313439556563], [-0.09303434845853464, -0.004966965416263446, -0.0008614660002423718, -0.06459392799697712, -0.001773908491749981, -0.015179383636232405], [0.004434958128129109, -0.020204882097763145, -0.005982517532772118, -0.13730244218873627, -0.012242256460281542, 0.02101260700483223], [0.00041815220729208424, 0.00029980703400737604, -0.0020814398334920415, -0.11773999585505232, 0.0033996133871952673, -0.033872803606616916], [-0.1488552469684921, -0.006571460614603518, -0.0014932682979664717, -0.008044891166871548, -0.0008770987336005567, -0.012803328336112241], [0.3109394518898004, -0.02658757215704221, -0.01844279374060486, -0.20143236427798872, -0.012999031301845272, -0.037554357078987764], [0.31411423305630526, 0.04460105534933284, -0.010893137419754688, -0.2557643731384308, -0.003082082344325539, -0.07078648915392009], [0.3149535029937263, 0.005323645904994837, 0.007920764019571298, 0.07681266198171476, 0.007633373910925507, 0.16399457211258675], [-0.09054871890523371, 0.01737030564029452, 0.0007570502307692932, -0.09786866628332026, -0.005156940524679959, 0.004441731746932045], [-0.09068022029058273, -0.0044111714344482866, -0.0034234508924307606, -0.06305706284432279, -0.00257767549621708, -0.016260419041998438], [-0.024832475884755092, 0.026883276756361436, 0.02962192741358477, 0.23533991458282433, 0.023965747276261985, 0.14727113366524566], [-0.20875612014654135, -0.0010759843760382534, -0.0026124898196418465, 0.021476773730091592, -0.0036994598049423333, 0.030173947083737777], [0.0663738352869414, 0.03126469623595219, 0.0205811157384585, 0.36074291217119403, 0.01613714171480697, 0.10357958456692772], [-0.2527947659749733, 0.00674398646672319, 0.0030325913316055497, 0.07043544828245461, -0.004612067825684894, 0.03720147438654113], [0.0930139401108551, -0.0179034929912695, -0.005635869502910355, -0.19742123707705525, 0.011587822056169478, -0.04246782926245646], [-0.16908623639185402, -0.0090864321322416, -0.0027944296959524382, -0.022426434150748742, -0.0035927964535352633, 0.037360058983061877], [-0.09144038269065455, -0.006577118617212673, -0.002681129473467357, -0.06210165590823412, -0.004100128392479, -0.013509584917952155], [-0.09273277697445091, -0.001554701496383693, 0.0006466675391950957, -0.07022491643119241, -0.002026684982840937, -0.014517587654326989], [-0.09880092493203829, -0.007200719522182948, -0.0006822335811548908, -0.04493235732366896, -0.0013265601260613194, -0.027369165299207354], [-0.21569115861076085, -0.012243638214401322, 0.0025823315355475536, 0.07948621449216615, -0.002502446134970918, -0.02044130306758051], [-0.15690296939580056, 0.002384005887090876, -0.0007326120821518661, 0.002792118472224052, -0.0011075970733550277, -0.018112787077848788], [-0.09254515137003982, -0.005760441191688181, -0.002603569989589838, -0.06510486732833576, -0.0027533040842860016, -0.011642666036060699], [0.06548136180810527, -0.01921121390817073, -0.009510578166417651, -0.139455985579479, -0.0007986902138177244, -0.06824822727355333], [-0.1813799797874668, 0.01727387931339, -0.0004618216797851835, -0.03146690134182622, -0.002334876636167228, 0.027126366798522275], [0.08156892664609143, -0.014333151651400741, 0.002669566378456821, -0.17044057515477284, -0.0032872655012471117, -0.06067083405046097], [0.5659803099453093, 0.0014429831883868802, 0.014162505889900232, 0.13715822766841826, 0.0094234157081412, -0.06554768049539693], [0.017758536050158598, 0.005739473975541687, -0.005311063683123107, -0.12971470010914513, -0.0017115125870799483, -0.047855257455876196], [-0.09206333628242167, 0.001155043649640316, -0.0031823265889986702, -0.0662432392766529, -0.0011627506315711718, -0.018913390869995623], [-0.0928580172844276, -0.0025532579982399544, -0.0023209193349370454, -0.07010958274686828, 0.0003688084769770762, -0.012937031112504193], [0.4521562397843621, -0.0024311319284114606, 0.014526720617015725, 0.31456728917514076, -0.0005044145783351947, -0.010571925291995247], [-0.11429213036454783, -0.002608587505376512, -0.003832488310969904, -0.04641598785349698, -0.0005782943675304858, -0.012682511598078065], [0.5313438818799114, 0.0015467182387804343, 0.0017439552746019524, 0.18772169049433954, 0.0033327288108391763, 0.0813029860858398], [-0.26344776379419477, 0.004492699095820263, -0.004033972085927476, 0.11414387062564965, 0.0025866728477387147, 0.04600722346964334], [-0.2059441548803031, -0.00607560387857763, 0.0004151547656606739, 0.07402264199128163, 0.0026958518019665903, -0.0320238898000279], [0.0031198578612566122, -0.0067751750864711345, -0.0028841559273670675, -0.13349157319831914, -0.007659786937876601, -0.016677500044555833], [-0.0910224953549635, -0.006020980487855903, -0.002825613023961866, -0.06631352081045598, -0.0016947219748955308, -0.01253266834786717], [-0.1800216332669873, 0.009149689524024103, -0.002097425982891169, -0.010110979326441156, -0.00027316306023602275, 0.011275120118052541], [-0.24632723768046788, -0.016246556382485854, -0.002025048002074141, 0.11545908124987957, -0.005773092106284846, -0.012555970607979338], [-0.1499671097922924, -0.00547332571151469, -0.002417380608116376, -0.02876234005110738, 0.0003745025080766278, 0.01283962190892281], [-0.11531699047904234, -0.002980345170006338, -0.0011964581798369072, -0.08044234180866704, -0.0016043866790438714, 0.021338855649930374], [0.018858151961630428, -0.011037933720934241, -0.004712637325712361, -0.15296944869867726, -0.0028332155306116575, -0.02771491668569542], [-0.12083965590927126, -0.0036346120267159816, -0.0022507883856198512, -0.03673801895797034, -0.0005555832325963984, -0.016293302272139928], [-0.10381284503388997, -0.005334218715788065, -0.002729045531541063, -0.07702892939952476, -0.0029917816114021817, 0.011486820292147086], [-0.12806409084457612, -0.003312961219101842, 0.022296986070459615, -0.07000238623674455, -0.0013557256299538858, 0.027111511193250486], [0.499748443313464, 0.09405656164486455, -0.0039239300317906525, -0.002811415896857177, 0.023471119090798197, -0.06173649240619474], [-0.10206064423506024, -0.005682365121906001, -0.001716273065343836, -0.05727999138962577, 0.0005750526485574313, -0.01424577883662177], [-0.23762897155632792, -0.011907630122905454, -0.009957522919306058, 0.11773625212194407, 0.0026581024669206327, -0.013976896656991478], [-0.1735814666590221, -0.011372546632548417, -0.002920895425659782, -0.020499110651614617, -0.004372344193686019, 0.039995093721260866], [-0.21053310683993473, 0.014668129033288001, -0.0069668335685517105, 0.027539993272474465, -0.006450981166061893, 0.023116788226686464], [-0.14692573631313272, 0.020982538142599066, -0.0034502268985433376, -0.03231469646163044, -0.003270520655133668, -0.009910524480824912], [-0.1322365878317806, -0.00793785621236368, -0.00039896283231439464, -0.016612264989831788, -0.003911131940763761, -0.015538686389023892], [-0.1220714464478704, -0.004716654071785358, -0.003381722933732069, -0.07414208818651026, 0.02469337105843203, 0.027595445343371515], [-0.12813104246740695, -0.004986368747950889, -0.0033519106499166015, -0.061302824608734194, -0.0011710814872951515, 0.018533227961304133], [0.5555322798185808, 0.00772337643813707, -0.008318466060043614, 0.16722902114756322, 0.004493459964010873, 0.06334699535841633], [-0.10208957049872508, 0.003191753962556843, 0.0007926289832100533, -0.04146168563222114, -0.0011728984886215323, -0.03055118070715125], [-0.1072191533149623, -0.006575313863225826, -3.5719869917213535e-05, -0.04424651292968589, -0.0013883728540469017, -0.01800375069757356], [-0.19697668488470807, -0.008113162841885894, -0.0007814158465743364, 0.03189925692472822, 0.001549037313050176, 0.0032431280655481607], [0.5729718333012441, 0.013232512296266947, 0.005712439049806208, 0.11973806065117819, 0.002743800627023811, 0.06750998152545835], [0.48899430526715804, 0.025358459795869478, 0.009319122715084022, 0.21210396026429484, 0.008763865451327067, 0.06455028650626396], [-0.16722861304779557, -0.001202143660071418, -0.003181816994157892, -0.014539725985515776, -0.0024072974216046597, 0.019270781480329922], [0.5673279577702507, 0.014815686410639127, -0.007885855158635878, -0.1422593265816006, 0.013213788084813457, 0.054499502721285814], [-0.1224312169686368, -0.0002836728912943908, -0.0040499106136283, -0.038169121394043375, -0.0010307657410673224, -0.013889756835773875], [-0.1970096755848624, 0.022066729528990293, -0.0031355452933173257, 0.039543680403858125, 0.0030073398293223104, -0.020180147931610132], [0.46138352951133516, 0.0026222279560985055, 0.010461068035749902, 0.2850414674990188, 0.005250352616603535, 0.019074585006776962], [-0.12133752091849542, 0.014510086719680041, 0.004753291096915978, -0.10025232639782086, 0.030388191231520943, 0.03229018303010517], [-0.1722634280098947, 0.013383851461804213, -0.0015952308477636043, 0.004046369544530105, -0.0002445516270473531, -0.016237010521629378], [0.011436376586328513, -0.012138113162834082, -0.004647198761451151, -0.12226831282963327, -0.005698255068001835, -0.047094496764407585], [0.10010664108466881, -0.022307459348049383, -0.026632334613367427, -0.18327692980199817, -0.0032429270432271454, 0.06482259745450261], [-0.1893517275311798, -0.01123092284903464, 0.009713991403886523, 0.07144342643882864, -0.0009315028252320439, -0.03026278844679315], [0.48607924045318807, 0.03589963413253234, 0.008091640161953415, 0.2114260538113759, 0.002208969565435977, 0.05286975599315981], [-0.12407927443544518, -0.007301566958392756, -0.0041969643670358905, -0.05494101100728903, -0.002001179583394343, 0.012318329684891289], [-0.13084590323323228, 0.004127500451504983, -0.0031630868156849444, -0.04086314336969652, -0.0018788637685574392, 0.0005468300689998029], [0.06628108953584243, -0.02109147708586704, -0.0038634515236056753, -0.16509473681029144, 0.039661521123991156, -0.05234461190673622], [-0.2117461583272414, -0.018271116050485953, 0.0014246706269697201, 0.20177213096695154, -0.013470325974998398, -0.018350643818226854], [-0.12548724295150845, -0.0109693855987435, -0.00346410414928913, -0.0052675210954504816, 0.0018205367458453334, -0.03597085437942465], [-0.06164142536543036, -0.008807453492018596, -0.0027569820765937746, -0.08325394716124983, -0.0030621866473665113, -0.02088800525734097], [0.25008638327138716, 0.0019780965937360204, 0.020667234864483235, 0.36128321917372014, 0.01226546364359501, 0.004593382398621326], [0.2431978124562572, -0.003332543526365132, 0.020594562299866514, 0.35701441152841057, -0.010619654536397306, 0.007060158110324849], [0.5054269797473554, -0.003225572610844334, 0.007831932957522492, 0.22461855524238122, 0.0010672640879444935, 0.05637084057563695], [-0.24450403044488303, -0.01728672070948502, -0.0008306248291108245, 0.12811648284416197, -0.004197010333998375, -0.018904525098112832], [-0.2587002473508998, 0.010302855713216959, -0.003476856483657894, 0.06563144800200399, -0.0032712856135565824, 0.022532657161464076], [-0.16254423572322868, 0.023323860351846134, -0.0036144530987205544, -0.0575008090984541, -0.00033914797531697376, 0.02700104928013819], [-0.15672772463941076, 0.015652596357130812, -0.0004673567565944232, -0.05910422929285419, -0.0020072750088168676, 0.040470179816735966], [0.4069563345076816, 0.007501398291767057, 0.019573111027833122, -0.0478972179960673, 0.011024824769986315, 0.13049717821442705], [0.08776763295420716, -0.02156197142296684, 0.0034199107986921315, -0.15537599344217476, -0.0010456292334438911, -0.005375387562810914], [0.531297090152541, 0.016774625974504544, -0.0007667490687642226, 0.20069246697328566, 0.007037093961312046, 0.02017451962616532], [0.05442590008576007, -0.021300276829661496, -0.007261651627208038, -0.09667761361229918, -0.003434501920788964, -0.08741185609580197], [0.4732347693613267, -0.006797032234100091, 0.007482347054325328, 0.20724591667958567, 0.00923519705004854, -0.06666238838737838], [-0.2233236055728266, -0.0008959812620108844, -0.006009870863553732, 0.057676765196154625, -0.0068441458647882725, 0.007320171700358076], [-0.12270062045457149, 0.0006725984796227673, 0.0010241770049745826, -0.06676909814322876, -0.0006405160408529965, 0.008559014709612204], [-0.09366871440014228, 0.03094469280841212, -0.0023925745632317388, -0.08612414678467865, 0.011327090282375857, -0.0202463473427353], [0.10425154986783938, -0.014203192153677135, -0.005232162860964035, -0.21839626879637175, -0.00264686127457114, -0.038490757089947765], [0.22186165158111615, 0.0005340997062907094, -0.010385709240921482, -0.18292624909116892, -0.005964434304640354, -0.10821913456104044], [-0.11050626611720282, -0.005735424217434526, -0.005926809262490364, -0.03734080101004594, 0.0026338539784484297, -0.023534553371274625], [-0.1138172438157479, 0.019599777548029534, -0.0021734469908820057, -0.06013567525640801, -0.005122513475440224, -0.01709423134288453], [0.5566174573788912, -0.001231964042939353, 0.005649668053825464, 0.15299060680610382, 0.006963295466777829, 0.07730331728972026], [-0.12865608798730935, 0.026593214765918034, -7.639142490472642e-05, -0.04262701847312884, -0.0016606306411501986, -0.015899752906091177], [0.29041709210524475, 0.08132888100488664, -0.0028195638481516372, 0.10590387883965313, 0.019144945654438184, 0.10970431075738614], [0.02353334200374714, -0.009868437393254739, -0.010562845671470119, -0.07534472560308349, 0.005569680664907553, 0.026170351509460395], [-0.19055419146170569, -0.011080455657523062, -0.01207768321721951, 0.16251105327459617, -0.005691413277135064, -0.019191119184822174], [0.1065732220981364, 0.0028157188975610485, -0.005787635763735231, -0.2126934408643538, -0.003956885898655101, -0.0007339943419695012], [0.055589645587878544, 0.02762809466613157, 0.02442527477552271, 0.37080940741944335, 0.016982885388953917, 0.006501914384288621], [0.3786872402728818, 0.010302666339656909, 0.029507107720261976, -0.09586664139836759, 0.021184881068219013, 0.18240966663226574], [0.03339226963768801, -0.012353082440372, -0.001858893170321767, -0.16182128937468807, -0.0005992714187511674, -0.02700306656688895], [-0.11136455566365303, -0.006944046106572012, -0.0021906118882928914, -0.03462204181239473, -0.0034569954728763865, -0.021831749056210614], [-0.11774203891719946, 0.01686759990590912, 0.029879436202666247, 0.3379567960503315, 0.04788945606714765, 0.07333182428421724], [-0.21680101823766815, -0.010178534175357197, -0.003818527658741492, 0.07746217226896532, -0.002524622645461482, -0.016853242576098285], [0.3401430117553781, -0.004478319366982333, 0.0030726626459796572, 0.23854784125402853, 0.008391358058063526, 0.10195826358070219], [-0.21430013473324072, -0.009958338505253251, -0.0023800467552182445, 0.06915920196276674, -0.005267853761966796, -0.016494379009226873], [0.30151863553258296, -0.011740493778732092, -0.023854840321654396, -0.19971733513219966, -0.01946142487402661, -0.061170414441844706], [-0.15635018997870428, -0.008854031004771397, -0.0015123502595143711, 0.00033834063051465174, -0.0005336128626942364, -0.013056980054242438], [-0.1755230397279798, -0.009071129456948782, -0.005506595151520896, -0.020155600817757427, -0.003715060439954173, 0.03470245123518647], [0.5357245209376813, 0.01096326792142172, 0.006142314451473628, 0.23068652425901934, 0.003166096668910161, 0.02190727576149137], [0.08649163530113599, 0.012244880466610182, -0.017914053722878562, -0.133294668571996, -0.00868851631849143, 0.032990802210699346], [-0.1345902378730846, -0.006422367006476444, -0.0016045759403591543, -0.063311414253293, 0.027549937059354164, 0.039051991347193234], [-0.1089168794315011, -0.006327169120958194, -0.004056301964589637, -0.0707147725749366, -0.0013617036942761617, 0.010966826786262585], [0.03004563448118805, 0.006011447276678124, 0.005657307367135091, -0.08084661947842586, -0.00015171372848870598, -0.05886018290221352], [-0.14935585894461598, -0.009251220511814697, -0.002570892104524299, 0.00344987047730776, -0.002664111606525799, -0.01790994417257166], [-0.1262762411322139, 0.009476697191625982, -0.005600856397518728, -0.07230782620964293, 0.010613358385151121, 0.021518201495931893], [-0.174401905310303, 0.029054594863244455, -0.0017998267849804008, 0.007685992586605074, -0.0028329657620191255, -0.0231158895925475], [-0.12434055210377029, -0.01206327399788322, -0.0005141744017523706, -0.01890403543273256, 0.00035759901450621427, -0.01852889641170079], [-0.227320972701265, -0.008536635159648562, -0.0024604130956606563, 0.0723581085441265, -0.0015946339386988383, -0.011865023026844501], [0.10405674332905308, -0.03180502149248585, -0.02159205377092484, 0.20658619274665652, -0.008532028418409122, -0.14634407048912756], [-0.1260050099712369, -0.0019129052568170044, -0.0030033547320362816, -0.024898565179257655, -0.0043986521361926685, -0.015191512724458886], [-0.06868487119128813, -0.006692848142972127, 0.006020418470641372, -0.08353313579607359, 1.1949312421630109e-06, 0.002931622680831381], [-0.09234339532221261, -0.006021308657280052, -0.002235678206714424, -0.060504211873540395, 6.525348322709852e-05, -0.0193706594234798], [0.25572487393514665, 0.0077718810939567135, 0.020452092217181847, 0.37759485840669493, 0.00428861452888381, 0.09879247835293074], [-0.15572578101454054, 0.02114668352227587, -0.0010960324962919462, -0.011671166559344229, -0.004497174816881835, -0.01873319530188427], [0.2140077016470272, 0.006806957864915966, -0.0003404744695188094, 0.3973398173756903, 0.005521611217558061, -0.01066518018024131], [-0.22526251074247744, 0.013356506866143088, -0.005539763307687079, 0.02833607739113197, -0.0076432804516130505, 0.024676303577834145], [-0.06688436202553796, -0.02124998794001648, -0.016342881909485765, 0.09263196043476746, -0.005876945112968611, -0.04968778344675851], [0.5620039194235487, 0.005964654254862096, 0.007914588935292401, 0.15547260581601985, 0.0071508222984576625, 0.02698212330288235], [0.2528197488704337, -0.020987494691328608, -0.01995695224796776, -0.27338068247476033, -0.009492432475932882, 0.09359621638089995], [-0.1887931149964257, -0.01036023925911928, -0.004644854750646561, -0.004861748306951206, -0.00446213266918407, 0.03678789823204806], [-0.09254143234298025, -0.006497706312960887, -0.0025766541233194946, -0.06999622149526429, 0.01124725747504374, -0.018378576533852563], [-0.1340260537844136, 0.028898679244432002, 0.0010022398691422413, -0.0425521693648935, -0.006297893787557491, -0.016497302176709], [0.6060525905412224, -0.0002570482504135977, 0.01348849231321567, 0.1379189636640309, 0.015451395024616713, -0.0715762980545803], [0.469073997332124, 0.02818718814701532, 0.002997812457071911, -0.1481733634266061, 0.003401197859055343, 0.15782142159959145], [-0.24108349217665417, 0.0019746765681293525, -0.0011823611680235914, 0.04379111381573109, 0.0007524660474244801, 0.026567755643550343], [-0.09825867096495187, 0.007178362210928675, -0.0025726319289409764, -0.07189411632528918, -0.001476215208584381, -0.011720061116495123], [-0.2285265205276425, -0.0023425643757867907, -0.0033552868994911967, 0.07989508004267802, -0.009433657911806951, -0.015443587124487772], [0.013358945821564756, -0.007373531330209087, -0.004441911476010372, -0.14865089778271365, 0.0012073997480065014, -0.022724290694924362], [0.11840878499273902, -0.012497706077455449, -0.017543816889177128, -0.19667443484877092, 0.02469011507251169, -0.012400085106989284], [-0.10590715502680204, -0.0049507460616047515, -0.001500293497863759, -0.07856500542028332, -0.0033060059650033675, 0.016319205971557327], [0.010647567587073137, -0.011924311350391814, -0.00946246772143394, -0.1820606747972279, -0.00954197648294798, 0.05300375818976531], [-0.12878123276166611, 4.485015252405878e-05, 0.003390449731965147, -0.07670672662018586, -0.0017793195812359818, 0.030743407650028007], [0.03378789418060686, 0.04498523048076236, -0.004332258967894041, -0.14936887054382433, -0.0018789459472066323, -0.028698287297682493], [0.03973091297701458, 0.0017694337059603364, 0.00043157488815788664, 0.2431849792083303, 0.024913878739139216, 0.12652350619568245], [0.3985412058938403, -0.006238736463806073, 0.004397611882030665, -0.035061776544467835, 0.061418913023716784, 0.18490595070685367], [-0.09705835062090684, 0.0013498591340038069, 3.4253870282929762e-06, -0.062129751877428925, -0.0010770916301916637, -0.021498090392504617], [0.3911977922011568, 0.07242392538161334, -0.007118556550937298, 0.07565038427813063, 0.005198095236894227, 0.04917685151663216], [-0.18524942383230034, -0.009863485226353278, 0.0011952821348389139, -0.01325387815510838, -0.0027125710658828664, 0.032870901541630936], [-0.2530697415657948, -0.013247901707221416, -0.004499306688006568, 0.13447724682719372, 0.002415634748212448, 0.025962481084029133], [0.5851921621745485, 0.0021108181844177606, 0.0031286011995984796, 0.20582726204221294, 0.011842105445916478, -0.013594282380028803], [-0.19792794214608653, 0.008231669007821664, -0.0018461708380387762, 0.0009277616855230873, -0.0006261040348002904, 0.02841411965891372], [-0.24384948856257482, -0.011657300218443585, -0.006091452906921711, 0.08045259399360073, -0.006528499502194128, 0.0236371630695494], [0.5418477988733237, 0.003134969428064494, 0.005663237643958476, 0.006170412727788712, 0.01737436982215612, 0.16014307874857414], [-0.1709749293138385, 0.013019707580581146, -0.0035314621333119014, -0.043785438922674766, -0.0011558712195456542, 0.03156378155457754], [-0.27912767892168, -0.0031871368372732426, 0.0016161700397624632, 0.11897304830063833, -0.001391763988096335, 0.025207361406649222], [0.08673160437060791, -0.01711354396849827, -0.0012753664427420002, -0.19919369591057015, -0.007459587082037319, -0.030381462248811837], [-0.20153655976823937, -0.009969102198352581, -0.004442908441799075, 0.07069736120662702, -0.004029587866483415, -0.028866045037015724], [-0.17048370771982588, -0.0015985910657881968, -0.0032833633932602166, -0.029595294435023456, -0.00043555401560161765, 0.025597621740610286], [0.009709637497035795, -0.02619338807480778, -0.019543834011604047, -0.13400535111413983, -0.017391343440229478, 0.06029466052706831], [0.24760362285141127, -0.01851217068544715, -0.029078123184208375, -0.16673980479197148, -0.024655728059649804, 0.02829363244129369], [-0.13160971725501147, -0.0009688183431797992, -0.0020605004279577654, -0.028491624205431366, 0.0007033913809057663, -0.012884691933638651], [-0.12288788792115668, -0.0028732417561327835, -0.0022446845983851694, -0.03259713008569463, -0.0017012384885368758, -0.013105817150093702], [-0.13721032138915523, 0.0006655283665605509, -0.004393540538856144, -0.04528840357032077, -0.001462846546191455, 0.011237917011296924], [-0.07126531977094872, -0.006723227351040286, -0.000420109653687501, -0.07898241859027376, -0.0010903995817026716, -0.021928525052347318], [0.016879115248906458, -0.012760501867114422, 0.00022669396826214722, -0.14349722922690303, -0.005712204676496285, -0.02471254011332175], [0.40600644931471, 0.0037745219379539618, 0.03665720451156616, -0.11002988621644254, 0.01979522846859009, 0.20052104186817937], [0.527880420625539, 0.01606313091980207, 0.027372886829414227, -0.07824270811046716, 0.02742973427479527, -0.017877750253369688], [0.5015339577549013, 0.05002435601932464, 0.01009696904842506, 0.07494000222184118, 0.00741753990407312, 0.05722796870222632], [0.5007773123872049, -0.005305881860923405, 0.0033873566677745455, 0.27002070355540814, -0.004168290682051712, 0.019128799932584605], [-0.05854735042910612, -0.0066154397243558625, -0.0029245595658015902, -0.08730080848099592, -0.000624923696027193, -0.024396918103713507], [0.4817947720511327, 0.00872708263368778, 0.009142044069091753, -0.15324191340047422, 0.015740310091640616, 0.16080865693587335], [-0.10930552599883513, 0.0225705473069924, 0.0024778521069481318, -0.06341649100950282, 0.00036049146544638743, -0.02351354053771516], [-0.14994616610781153, -0.0057543973313330015, -0.0022272848347807575, -0.009679617706239938, 0.0009143055526067069, -0.013618800356754411], [-0.20248642161863428, 0.004234232522132047, 0.0016411795339899792, 0.05592901133652247, -0.00035165354221363706, -0.013979522834972112], [-0.10722461918721188, -0.006800927769959162, -0.009372143920344108, 0.0811556574212205, 0.0006110925232873308, -0.07930882097175465], [-0.2695104660511151, 0.0058165343412698856, -0.00136766095664599, 0.07549225754179478, -0.006009712738311757, 0.03314523833919847], [-0.22205962011013333, 0.03419184317494866, -0.006715605655668806, 0.08854566226788836, -0.011690081075625133, 0.003257087112875859], [-0.11592187873362914, -0.006802292610636125, -0.0019571707836388125, -0.040037750588254, -0.00023072745320516685, -0.013460179830636314], [-0.10876555437917741, -0.03725910931915882, -0.008802788408971893, 0.203925846902244, -0.015191385915371843, -0.03409677078432637], [-0.15026752541990307, -0.000745224786576789, 0.005192908191786906, -0.044796487239214464, 0.0004991038013974142, 0.03300484450012891], [0.15788512975033842, 0.03628474811475066, 0.043183315068683656, 0.014239881682742312, 0.005028247572281977, 0.16539887261639794], [-0.0947898231821625, -0.007090843287695617, 0.004660629204338573, 0.1426417576441043, -0.018611873135016856, -0.09421389486261524], [-0.20454168801679945, -0.005261241073163517, -0.0027100096917637936, 0.047269936991788386, 0.000955573759061177, -0.009895299241850489], [-0.10167559612194767, 0.023085904825473815, -0.00386804258839913, -0.06675172250859195, -0.0015743692180025027, -0.01962617438853282], [0.6146789911963557, 0.005538493023550164, 0.014336003521020992, 0.006403705488686369, -0.02403300741792305, -0.017313623041130866], [0.528955334366657, -0.0013160763323719505, 0.013516463916606006, 0.23979976578561596, 0.004034092733463893, -0.00808608840648063], [0.1253796058058172, -0.01397317542502798, 0.018192224370633253, -0.22584826598690935, -0.0071790860726409144, 0.08457425286368364], [-0.21543111495564027, -0.010244165921233872, -0.0030611482643966895, 0.06701047233092136, -0.00133760514674383, -0.01734643804290643], [-0.19323966977374463, 0.027505481320775096, -0.002628463256692135, 0.012814086393971308, 0.0011578503490426513, -0.008081785033352506], [-0.22207581002596857, -0.008128924252851416, -0.0005549930034758477, 0.07956243557133306, -0.005210462972276921, -0.01689440217950545], [0.5061580480223742, 0.008438047069415729, 0.036683096128749454, -0.09444206701851826, 0.008800481163657167, 0.19428771209463763], [0.26053298358268384, -0.00024340127650407628, 0.021422663862225706, 0.3011113540677253, 0.008919633099936234, 0.11609585091301663], [-0.1391283835053934, -0.002211412571451572, -0.003114028819036372, -0.011983943404955623, -2.7515522784380188e-05, -0.016346676960691784], [-0.07225101765289037, -0.03467210585015733, -0.0017970100201714465, 0.23303500447488104, -0.023373692399071687, -0.049358321409733945], [0.2158799403298258, 0.03222062072948274, 0.004673171649885396, 0.43520733560972136, 0.017717579102181212, 0.0158071034946514], [0.24683782723750008, 0.022401128692449417, 0.016749723721195462, 0.34204111690874756, 0.014960271089566917, 0.0847866916696519], [-0.18098065988197407, -0.008263972184656245, 8.702918031699991e-05, 0.030195024492182126, -0.0012876455898779917, -0.015159776015990768], [-0.2432606150153411, -0.005781777176802069, -0.00012878956791550063, 0.1617695749248003, 0.037950815564736895, -0.010758000215769517], [-0.11714245228390659, 0.001493576084354948, -0.011497131030664414, 0.04938044227603455, -0.0033560318155752616, -0.03033205402389401], [-0.140004336245983, 2.4708287247900522e-05, -0.004217899247543228, -0.06480928042534037, -0.0020720567037758597, 0.03448830877983938], [0.5524186666598132, 0.005840051681628481, -0.003617760695761339, 0.019977235557006562, 0.005819547075439465, 0.16640225972187367], [0.11346046709766103, -0.03012991382452939, -0.005499040803251868, 0.15691779737265982, -0.012412958519629282, -0.08187135132290957], [-0.21863997512226654, -0.004127353105566384, -0.0007021548825723631, 0.07200044966486006, -0.004952552926471399, -0.013886140900710348], [0.2703096638061417, -0.018636408500624868, -0.017685387787932563, -0.2894302683788661, -0.005806836946217415, -0.05282742885916805], [-0.13694893830234436, -0.0011600991227278107, 0.00022086880857159194, -0.06502564121315717, -0.0033037813199793408, 0.031918702260748937], [0.47653327653306654, -0.03940013718051748, -0.025448791410198877, -0.11376315381471776, -0.0015781305184184946, -0.11852813575928717], [-0.20186945617709248, -0.005585353196139638, -0.0017226939055576674, 0.04511499816620739, -0.0032812377133034686, -0.007338984446841603], [0.054607579095182004, -0.039898512227363604, -0.02552399438118356, 0.11371795467969248, -0.013557676255213194, -0.06126418929495225], [-0.18629933128727547, -0.0009862870484672545, -0.0008061380966390901, -0.018026775320038897, -0.003466107737845068, 0.043619083934710944], [-0.09288639699019816, 0.005459008670954614, -0.0022560876425481826, -0.07006193051034415, -0.003993909640515751, -0.01622950741676008], [-0.12503808407929676, 0.11113511719789852, 0.028359099145288565, 0.27158606892409193, 0.039168713219344814, 0.06461898458257063], [-0.12507480737994375, -0.0069712418544171754, -0.006239912809096858, -0.07218647385069266, -0.0016531049522224982, 0.031715540846373465], [0.5335070762356646, 0.006689105278573203, 0.00640660399222839, 0.21459495549759122, 0.007299864319723848, -0.01648183501565911], [-0.21912288039297134, -0.009394833735232606, -0.005548814115006641, 0.06266545766508985, -0.00578632022020404, 0.010610724131657869], [-0.12764030911791402, -0.010459705624950133, 0.002232383524878644, 0.15495266456634815, -0.008245176703210613, 0.0009572862119908885], [0.0023142288474796687, -0.010793987114960612, 0.008285822290560657, -0.10382705315724691, -0.006437635563259229, -0.05251089911209719], [0.13796168156104222, -0.01403967294626524, -0.018784473461075327, -0.1239278652520462, -0.014424581517962321, 0.05888824494964053], [-0.12874210739447337, -0.00655810190496676, -0.0022560291852242666, -0.02223387965816407, -0.0009076521281117324, -0.019271053258471598], [0.2606432030045489, 0.055929878056643324, 0.001463523913717888, -0.17021884054083755, -0.007307019922368979, -0.07769383253479163], [-0.15064938181031762, -0.005698094170693694, -0.00015892253788387932, -0.0044346857681574455, 0.0006411008265498559, -0.017115151926985095], [-0.2330884348350452, -0.0024435534450273013, -0.004074584055695413, 0.09982582463669992, -0.007610153125260623, -0.015300845207417873], [-0.11634966773714203, -0.007685204122498704, -0.002584968583371208, -0.07880780054305452, 0.0007461780060447374, 0.028021462980022793], [-0.13331802957075117, -0.004662709092286479, 0.002046735800928781, -0.05545511832933574, -0.004029957438915222, 0.019453523074805006], [0.37825546979306723, -0.028184969298171957, -0.021143233441898782, -0.24700938283970558, -0.0021813589878901508, -0.06996795379682956], [0.4900827443024418, -0.004060162529848084, 0.002360179346354455, -0.0004791127899940697, 0.0375098263137843, 0.06686483704557149], [-0.11707412925856121, -0.0065219061580260975, -3.0335922992232083e-05, -0.031921352075665065, -0.004776817406521564, -0.018418792511566938], [0.38359604359151145, 0.0409031933078361, 0.010627686114732268, 0.286276766029947, 0.004466433770443069, 0.05797012856813112], [0.03382629824985731, -0.001488450475829769, -0.001202364800812193, -0.11161232669441556, -0.0011373566858243827, -0.06206960911678524], [0.5688113665494058, 0.02222956854550659, 0.0017464628037029376, 0.12570958252427697, 0.014690551961669927, -0.005709815934347252], [0.4775621482285031, 0.06949877167946011, 0.017262817950569824, 0.1100241903044018, 0.007488378104975298, 0.08182512230351843], [-0.18096186220413968, -0.006050008926317103, -0.0032183656029170188, 0.0023075468925388273, 0.0004248827510764161, 0.015817965819916943], [-0.23240244662737505, -0.006061419981119092, 0.0032980455833130943, 0.06085737383001351, -0.001002099278516907, 0.010317213140350548], [-0.11386680656154523, -0.004894574742315764, -0.001804340435569762, -0.037126305353642426, -0.0022650658032428413, -0.02045290710368342], [0.24657083771379815, 0.018688674131835295, -0.006307068015518951, 0.3332181591294175, 0.011784921003152106, 0.053956972984811793], [-0.15006285338935538, 5.484939371578277e-06, -0.02282623501421182, 0.21111275077223549, -0.01666812330085697, -0.044224992261151494], [-0.23661228499735482, 0.003944957587140845, -0.0028364922535438265, 0.06375373956088978, -0.002486201061617515, 0.006623900212104829], [0.07869495509635431, -0.023773825503740978, -0.010690069926153055, -0.06033263456978139, -0.001808151543714597, -0.1261669402196297], [-0.21090018671281355, 0.014363348880339619, -0.0011503108907806798, 0.04510777728606402, -0.004778396649676754, -0.009224959185860538], [-0.13385346488168598, -0.005072666148075397, -0.0027823447751999545, -0.06487303852879528, -0.002292422242615937, 0.02846393657637225], [-0.1415556536550414, -0.0065566225934609885, -0.002912048884101745, -0.010190121755157948, 0.000622391917842129, -0.01981794503007974], [0.02603466210760311, -0.002500008519429964, -0.013308458976785068, -0.1942637649320204, -0.01175790371517571, 0.06061527201560631], [-0.2408999259151191, -0.0011011566548021097, -0.001031914105695389, 0.03956185390505623, -0.0013552490834877215, 0.028999725187380563], [0.2973010353321924, -0.01387694666198795, -0.008740144223027558, -0.25284109037378427, 0.006958954067269925, -0.08599752242637765], [-0.15002467179152004, -0.0073042014936867405, -0.0028923591185117436, -0.05672317177927252, -0.000434902481775271, 0.03808041777587791], [-0.09427557026127656, -0.0021045684899676093, -0.002877561624611408, -0.07151687498735769, 0.00014866319253729895, -0.009784087829324046], [-0.09749668836760308, -0.004539008227640383, 0.00010571457446724138, -0.0627989223220526, -0.0011951141630640802, -0.011985981494107043], [-0.1544793456038796, -0.008079719961722704, 0.016540129612401644, -0.001960246753818488, -0.006634641753695277, -0.013867604110713964], [-0.13856975718797643, -0.005237259604050908, -0.0035189291603730834, -0.06090237416875737, -0.0020108800706230266, 0.03665459701717844], [-0.2600642660336101, -0.010373887724658517, 0.0010199140673126706, 0.08295144999559967, -0.00443867257658943, 0.035257367033850226], [-0.14475383652023208, -0.0020407852191913455, 0.0043947053197152045, -0.06042238674646458, -0.0037400256974137332, 0.03202732886358698], [-0.1765678161042623, -0.008096261551497326, -0.0003955775800734118, 0.011628650505443011, -0.004694488695370875, 0.0007710489813163549], [0.4628002787067192, 0.008369348714329525, 0.03344006298835081, -0.1252528966924405, 0.012516219696933062, 0.19903841515753473], [-0.202451342781891, -0.001927422336431909, -0.0013100823720286077, 0.00920796905701811, 0.005833228094503964, 0.03474386151895425], [-0.1101421230500603, -0.008608856642517112, -0.007676571508756102, -0.08637125779802074, 0.0008248766307446995, 0.03156393236861016], [-0.150199205698864, -0.005081468588026134, -0.0034659446306745025, -0.04546306632289698, -0.0026667765748701476, 0.02874423959310981], [-0.15313429832401254, 0.015595151079636918, 0.0036283754421271904, -0.06579153969539871, -0.003291158718088242, 0.030912835295100846], [0.25751135236469375, -0.0017463622929651853, -7.0717421790826e-05, 0.2984999642218241, 0.020488327144285626, 0.125610068318936], [0.0012717918770962595, 0.018836596054023554, -0.005802226183765631, -0.0036036596689857717, -0.02284367876961665, -0.08679858521351365], [-0.05021231974269643, -0.008091441427318175, 0.0006388399718093122, -0.09579768867124243, -0.003586318293874878, -0.017086562032755923], [-0.1669117083451282, -0.002805589622455775, -0.0025539371636785005, -0.036052281163758235, -0.002545772219504117, 0.03407039962563623], [0.15090431073667548, -0.013814586082333751, 0.0019688510016144063, -0.20768036670785017, -0.004349301723687474, -0.07024446277997372], [-0.2626360980224937, -0.0018988568285350656, -0.00694165953360351, 0.0807403571128079, -0.0019968815935135515, 0.025299329341528176], [-0.2433769140624957, -0.014811924476327622, 0.0007395420420741918, 0.09352491116493492, -0.005302726556186083, 0.013483778554667009], [-0.09107192582011094, -0.004534461091551442, 0.0017515119997764247, -0.07318624078470184, 0.011801416086624501, -0.01812484584458253], [-0.22168543725777184, -0.006619375936608686, 0.0009660389908260289, 0.06812663411840063, -0.00441143342729078, -0.01511975982088824], [-0.09373216326194261, -0.001340731103063393, 0.001126539845993785, -0.06767885728410336, -0.0012125310756650504, -0.016322257121219045], [-0.1554211337737796, -0.00895176020114055, -0.0019760605862018414, -0.03513448582947004, -0.005038363135057927, 0.02888958130342819], [0.013075703091059388, -0.014421469548465366, -0.0012416386793890253, -0.08967701780646152, -0.0036341125979334754, -0.05327862132155505], [-0.24850925157159248, -0.00034934857297099, -0.008156276171962757, 0.12229370496608657, -0.006913597396694031, -0.013358564586199734], [0.022324281941725237, 0.014796375520353578, -0.00707021895980737, -0.19166681712259415, -0.005503626461945782, 0.03809889397115648], [-0.20582370981054338, 0.0016967623393093895, 0.002656666138020722, 0.07266518139298359, -0.0002961169796524282, -0.03155878308011805], [0.01759320228524314, -0.011585984717877522, 0.061671255909815724, -0.11201637927636178, -0.005637645228409418, -0.050184448972410235], [-0.04870970279062927, 0.0006464014405342384, -0.001656127338706523, -0.09879333391679687, -0.0009710496071539028, -0.025509521120581004], [0.25906757289983673, -0.03895824819388095, 0.01315559653052361, -0.0827757514085983, -0.024224282367562425, -0.10482369698412863], [0.07701296341967566, -0.013020379429094857, -0.007051889992398035, -0.10872746573755002, -0.003463001845221983, -0.08757689308207685], [-0.09345134949578508, 0.022203497423061918, -0.0032237166586725264, -0.0781999969316955, -0.001388511534765892, -0.024099922802142706], [-0.22140360835965725, -0.0011943050287897422, 0.03324310417685703, 0.06268736676026952, -0.0022428725516270656, -0.01602241226977985], [-0.12192947438308947, -0.005696072327325703, -0.0012992536652364347, -0.07641950162049255, -0.0017897712672609594, 0.026724073263405927], [-0.13138052914588869, 0.04078808760679822, -0.006407059912330005, -0.07638637298545585, -0.004799597319314694, 0.05284690032761997], [0.4979715343372846, -0.0008086392479641154, 0.010381402854691163, 0.29804145040239516, 0.008720824939044481, -0.029147945834472805], [-0.18349900852849452, 0.06989546703647695, 0.03098064062105359, 0.27335090592815087, 0.051901971176146366, 0.09501341482005676], [-0.10391861056194072, -0.007372758263769502, -0.001110216616621748, -0.05129772044853386, 0.0006271190832586314, -0.015573107310039472], [-0.1209731529411839, -0.005059809042368053, -0.0006366369629726406, -0.07431918236554019, -0.0028865795345665132, 0.0234653608466318], [0.33239942816331, -0.00496696092563494, 0.01633732775753619, 0.3419180092142351, 0.004548671148639809, 0.006501432289819388], [0.10090479041952476, -0.0356980569169881, -0.02131036361728188, 0.18690328492685512, -0.024411401783103148, -0.04824028158868278], [0.07095958107494299, -0.0036607511147335437, -0.009728909458165434, -0.19104455234583304, -0.013314585035794952, 0.07110306247843025], [-0.13167025511034872, 0.016051819692709603, -0.004252078124904474, -0.06639526871061044, 0.0008079624012741402, 0.011297819851880607], [-0.10638709593178966, 0.003465763846265061, -0.0014133176945457227, -0.058276955244622376, -0.0013309466670959965, -0.012692938504289552], [0.012389914787145729, 0.004217719073235302, -0.006178752288389514, -0.14016582171226863, -0.004528608069047822, -0.03288254702877043], [0.04340596917063893, -0.03666154391480151, -0.01520314120937206, -0.011135584984931729, -0.015800726082849204, 0.033080265116553605], [-0.12211375972925272, 0.0028134589890229235, 0.007968644942277982, -0.04724720073773586, 0.0020049635510140007, -0.014669440348659378], [-0.1103482338112688, -0.005963917125736548, -0.0017804969027701823, -0.045167305067279034, -0.0012199384887988615, -0.015488932133557976], [-0.2135893113662902, 0.014072321822400572, -0.0045118142665552495, 0.04371188229344442, -0.005524203874875894, -0.009341601880850953], [0.04069312486806942, -0.01148689347135346, -0.003032096272818305, -0.1703814045881755, -0.0018624523619249177, -0.025232435036542693], [-0.03092166632183068, -0.009215266133206123, -0.01193040585175118, -0.15480064601431914, -0.002458195997522603, 0.05024951365196305], [-0.08573973489523663, -0.0034706938706692034, -0.0004070511994107276, -0.06883562136001314, -0.00013716086279256747, -0.019319737811877857], [-0.16836163005470578, -0.010391418011969298, -0.004191969612889543, -0.016281187360372467, -0.005704190809113162, 0.025661421490075675], [-0.07928808597726741, 0.0007262277461468978, -0.0057039416211604925, -0.12488458125990429, 1.146461224430239e-05, 0.04372891649994216], [0.327416187966085, 0.04547132337251321, 0.006735898378431199, 0.2605982166445141, 0.015956019626273885, 0.026828104927931272], [-0.12586893160594148, -0.007581948250312044, 0.001379123853234611, -0.07901067812716504, -0.000671037628002436, 0.03134347175818716], [-0.12284640309165008, 0.007907096027509568, -0.0038367820967026307, -0.07592960935033682, -0.00346074517311737, 0.019423110350964286], [0.01126576184461047, -0.019765783523716975, -0.006240204169059944, -0.1290790116578676, -0.0032898111206198346, -0.02996761804001283], [-0.20505802935335596, -0.017020147316485035, 0.007887125839115786, 0.1357706931468086, -0.005654385561960209, -0.04833525675412242], [-0.23951627734188763, -0.006940009845479564, -0.007740214435456499, 0.062040736854279105, 0.0003084450098776396, 0.02727065309200013], [0.023500317305220127, -0.01231502434476665, -0.015870880615407715, -0.17831387857872738, -0.008566582133579258, 0.054795404109558535], [-0.09282979648287176, -0.00655323568165572, -0.0035780700144517567, -0.07166672829029092, 0.011319352661460268, -0.014601522192190703], [-0.1123931154266773, -0.002451237191990738, -0.001097522097766004, -0.04477100962070281, -0.0033858930089339804, -0.012977889320595636], [-0.24765566001047445, -0.01452770759087097, -0.006969810207813933, 0.1367021319085562, -0.004392129276110769, -0.01081682482328587], [-0.10718163082757688, 0.010529756720535154, 0.0010113212081785063, -0.05827992013450965, -0.0034526468829785668, -0.023036880083648115], [-0.09454181190247599, -0.004595334138153091, 0.0010138382572403336, -0.05502124297137354, -0.0017387261546825596, -0.02552672309055507], [-0.2776501554682836, -0.004163982730716821, -0.004462151745559576, 0.09814135801121088, -0.00793699877904159, 0.039054787855247075], [-0.17344950199666423, -0.006651124291877604, -0.004306547173139198, -0.006710307475941804, -0.0008108618663955489, 0.013581834867510686], [-0.20338026740994436, -0.007098080637264629, -0.005341216444158127, 0.07334722566300043, 0.00026332649388979173, -0.03143782977078608], [-0.10861548714536488, -0.0076007197203889435, -0.002567200754386949, -0.08174206154527917, 0.0007638489635036345, 0.03296273131302772], [-0.11317412346812929, -0.006795252009194186, -0.0008984204455730082, -0.03697629970282216, -0.00157189875552915, -0.017219495814830323], [-0.23093892049587972, -0.0035335314819253207, -0.002631578391492181, 0.09753065606730373, -0.0036922879905476507, -0.013477671040792527], [-0.008539351373127953, 0.03903259274648698, -0.008634945711440987, -0.1130539323025387, -0.0032369667604905425, -0.03714406326555567], [-0.22926425435377387, 0.01890448852928468, -0.00869212092997655, 0.07390342882231236, -0.005610633472527223, -0.017614112058522893], [-0.14777948795025916, 0.002137301864500129, -0.0022325140443896644, 0.00675746298900839, -0.0016404639551432048, -0.027295156046573715], [-0.10622726862227304, 0.034094959094976084, -0.004143776956993977, -0.06826499931422865, 0.023299749420793582, -0.0276210445746548], [-0.1635402976545404, -0.002069785099558602, -0.0023281946117057035, -0.02355724779372258, -0.004969197700460871, 0.019054722859988937], [-0.1993814670713709, -0.00851309773070353, -0.0038555788681631465, 0.07273816680189125, -0.006669284397587913, -0.029728738734066094], [-0.18408413095947135, -0.007166916874375401, -0.0025876143610143176, 0.05978476328960964, 0.002684180532054697, -0.023349805436327413], [-0.16531855114204144, -0.007106965781140343, -0.004164182020119694, -0.02288195588774835, -0.0038599079128772457, 0.027782673855038367], [-0.11512463074447078, -0.006325020157025967, -0.0033218110109563357, -0.08020841581496035, -0.0018891780948976208, 0.02645905582231203], [0.018568706609348316, -0.01042631836793344, -0.0018284362971115152, -0.15145013101885094, -0.0012452741307431353, -0.02533737032412136], [-0.08296143874998657, -0.005988111603951893, -0.003042176627104381, -0.06401836253649304, -0.0020382212412734454, -0.022361689241190717], [-0.14024526001830276, -0.009789894409378561, 0.0005283181377603338, -0.049856751885732936, -0.00025758082672573316, 0.03535219464340552], [0.0830151451412319, -0.012643030308681627, -0.005904470493758926, -0.17958937043842269, -0.009237695285788354, 0.032038707099705764], [0.027013300130870836, -0.012471651146903094, -0.0027961080365033826, -0.16005487195762919, -0.004073803271047242, -0.024669022581533338], [-0.22529037315618242, -0.005175106555072188, -0.0012822461735390858, 0.07845613174082273, -0.0006413544765462771, -0.011772335043187712], [-0.09169885123981325, -0.002252094309735007, -0.0012355495859240012, -0.06854399020868851, -0.0014588548020100058, -0.015220659853829154], [-0.1130272828055999, -0.005150706025265303, -0.003350364216195593, -0.08069375530488235, -0.0014958650371264628, 0.02747464005573744], [0.5857246649751958, -0.0052351317664515624, 0.009686888888796825, 0.05230112440269188, 0.012904875615873111, 0.09628081020712347], [0.5905233493608959, 0.002438426939893678, 0.008972654432182266, 0.1918375658194787, 0.0062918246812134065, -0.011604773614617262], [0.44503625359243854, 0.011614656053888452, 0.005832751272852096, 0.3485330131980615, -0.00015047289794690473, -0.02374842344151988], [-0.2352641498478072, -0.008355549725371504, -0.0018947471459784843, 0.03772578565836715, 0.00038986431601708475, 0.033635622141597286], [0.48974686784377924, 0.022558428846899903, 0.003368927433303066, 0.1959503798832174, 0.004455307974719916, 0.07966204880239222], [-0.09374820899922551, 0.0020245616505495732, -0.0026003122375856916, -0.06293281037359186, -0.0021191990525748303, -0.021034030987571627], [-0.2281849944483151, -0.01877097907519527, -0.0007266959444434815, 0.12631379213128033, -0.004709191706718569, -0.035998597623274654], [-0.08550772671892679, 0.0005696549872706462, -0.00176568080964585, -0.07843493892602803, 0.0015692495002394323, -0.012673891366242796], [0.06730309415816586, -0.014741023612194652, -0.006292638728580076, -0.17080949045818333, -0.0017096721311134274, -0.03993127763145596], [-0.139033432917456, 0.0038889831685931507, -0.01461292642558145, 0.028183453203867833, -0.004347437933533315, 0.04804390058664984], [-0.12312031901822292, -0.005555434309182998, 0.006476066638009944, -0.07240212107144149, -0.0015475026626055666, 0.025114310423443958], [0.2704832307763101, 0.02691133787330057, -0.01900761022648528, -0.2620658615441848, -0.016108218266095653, -0.06869656187402695], [-0.0901185219992607, -0.00016209428946735682, -0.002518341317564554, -0.06250788088980869, -0.0030544250432540786, -0.022048736460644498], [-0.1109711234048831, -0.005278675151353447, -0.003354623105244937, -0.04051728084213555, -0.0020649294634844037, -0.015282191562310149], [-0.26518556110175007, -0.006542209940197841, 0.009253386987001625, 0.09954543120145111, -0.007143340161749979, 0.026745626348578983], [-0.1268809703845157, -0.006270654075139675, -0.003093034185648377, -0.022470764870599964, -2.4717492411865095e-05, -0.02157181977599791], [-0.13405015915275204, -0.0038965286438451055, 0.011661422926924982, -0.04281626888440954, -0.0055409448813997265, 0.002149145302148123], [-0.1305104314106473, 0.034793423018854776, 0.00979287474261647, -0.08604302227933143, -0.006074891357898666, 0.042001094905454374], [-0.17665163869016992, -0.0065598531549031225, -0.002068067581020718, 0.032182912850082755, 0.0005434514526105827, -0.009404423924219021], [0.34393873485718035, -0.0004058870663972493, 0.010549323009211572, 0.29993479920294297, 0.0037967317041135893, 0.057108715875364584], [-0.10186692850337613, -0.004399428564562084, -0.003960381045689902, -0.07009312046706388, -0.0032169090391755495, 0.0031267676198676393], [-0.12797000733848274, -0.007782183030989372, 0.004302824619788554, -0.032618110839482614, -0.0014928990408455342, -0.013084918487634916], [0.09737479127228071, -0.029684902719069352, -0.011515130038635177, -0.08087812401064394, -0.00759948590239551, -0.023146364287810823], [0.21871339555562763, 0.011464015594065353, 0.02227028515685519, 0.3000063574785793, 0.004221902536601768, 0.003397268786493893], [0.5399323515327715, 0.008244773101693505, -0.0021351971618707288, -0.11729273633051415, 0.022327928802400556, 0.06931049910313701], [0.0175504845586599, -0.003564767154720196, -0.00376593851392388, -0.1536428569986266, 0.0023591673949024663, -0.02839370833391145], [0.626784395058611, 0.00515471324824824, -0.005046099129932012, -0.050360122095264206, 0.013538889251497991, 0.1260102871588988], [-0.21860641251361526, 0.0368689600175605, 0.0014470662984076407, 0.08540418328451398, 0.004007768091319165, -0.04351966041628187], [-0.242818332761335, -0.0006696143962508713, -0.005765896534006801, 0.07853256825602299, -0.00358295090634212, 0.0018704168181020253], [0.25544341673400683, -0.0005994320263306888, 0.0009932181213476074, -0.10544261666915984, -0.02803909255241952, -0.10166415050610131], [-0.2261016835257166, -0.016508890796944176, 0.004075585362060803, 0.0922229231814698, 0.008839180440913518, 0.04403193295726456], [-0.07974611137937988, -0.0066946717708270985, -0.0036132165041779965, -0.06778569340820743, -0.0010335508366728415, -0.021536756100734522], [0.528682204387298, -0.0010893652032278407, 0.009925483393433883, 0.2229392102723714, 0.0010554452612458398, 0.020077021888876816], [-0.07051992996352537, 0.00633638942663077, -9.70803931999411e-05, -0.07747796327381948, 0.001356095003630642, -0.029126558418764523], [-0.2140222929941789, -0.007381394876932628, -0.010073249843516601, 0.1572563993238547, -0.010721479345297304, -0.007475125121072207], [-0.06267456805373371, -0.00816101752911168, 0.0004100914907758351, -0.0954323595848933, 0.007412628103774694, -0.020298107760145494], [0.0761902855515531, -0.013613686652702357, 0.06268055551733169, -0.14029724341169095, -0.008250588803897015, -0.0697621793434511], [0.4633723068444175, 0.01710620534519894, 0.01156515382598933, 0.2531002506936679, 0.015495228748295375, -0.023777500435925504], [-0.11439212295294181, -0.012269642142904325, -0.008584097018763868, 0.05577646668792727, -0.009400310475811023, 0.029499388442176453], [0.005627856991292298, -0.011155090170349546, -0.00593928249311605, -0.10834326490671119, -0.005656846043498542, -0.054943373377617136], [-0.2102110812337191, 0.006600942716077034, 0.0026874845805375394, 0.006641727205536655, -0.0026036645502412192, 0.03030619928732973], [-0.2228343633309934, -0.002223179679096177, 0.004092465762994885, 0.023870313134482034, -0.0025160619475343825, 0.024657175266495395], [-0.17949558193098641, 0.003040949161264117, 0.0006283765228453425, 0.011311941869886943, -0.0010834249304901413, -0.011478927359186738], [-0.15013459226307657, -0.008648318201348844, -0.0023471980543059053, -0.05082621295612002, -0.0004352163549087549, 0.033092648940871564], [0.3002163425786921, 0.01624404981605915, 0.003897559207412879, 0.19548382983627127, 0.008241528105230598, 0.07728148673848197], [-0.23517028380909885, -0.009169649552652115, -0.0006885747343619811, 0.09245218869301697, -0.004071470060532432, -0.017042874317035422], [-0.09176041291042483, -0.005451382798363409, -0.0028848886683219277, -0.06604178358785989, -0.0016298958292040614, -0.012641636205826083], [-0.09887086140365446, -0.005658975868578913, 0.0008373442049075937, -0.059349153724850834, 0.00010642409064350739, -0.015808110631800164], [-0.12927589912417212, -0.0078340774296767, -0.0032277595796417224, -0.027134854120893646, 0.0006823565537478183, -0.01352172708367706], [-0.09723177255308507, -0.005274224915527613, -0.0007742533341927035, -0.057154466930228326, 0.0009817568824282907, -0.01952846772082294], [-0.1261914243519094, -0.0012104826093996151, -0.0033072466112365707, -0.07688578593669337, -0.0010134510466112711, 0.033198390555851144], [0.35645454006405375, -0.0017434207421191295, 0.010302496693972962, 0.3292373252571402, 0.011303868170231664, 0.0825076641615453], [0.024307673651801597, -0.014315718329020321, -0.007871173258542481, -0.15157708624896432, -0.0049068365423922036, -0.026046859272881938], [-0.12519317798152124, 0.0130750200924352, -0.0028242338603540547, -0.045723115134044784, 0.0025931689265620305, -0.016504328709743145], [0.053144468699700256, -0.034495774821902477, -0.013352993940111827, 0.14039541185459062, -0.010465908659734971, -0.13179863253939156], [-0.15241437288194726, 0.0010460664307075256, -0.004009627968447484, -0.0024625016161768914, -0.0025980388648491064, -0.016197015295365436], [-0.16375834128025282, -0.0013439570803276246, -0.003338188240348282, -0.04392191799844816, 0.00045523208729514786, 0.0396916169565258], [-0.12481704324558854, 0.011109388387003144, -0.004712054136698566, -0.08027497321999413, -0.0005726568248767325, 0.024690672373488584], [-0.16271047596162105, 0.02745413986869113, -0.002862276123757433, -0.00244055485969453, -0.006969375026995434, -0.022067965833130695], [-0.059440058587952614, -0.004648292459374291, -0.0017611626064533525, -0.07196547047538387, 0.0011708689047038856, -0.0339801704898256], [0.5754422727265674, 0.012666368412790601, 0.005320285381935268, 0.1025766018514978, -0.003423725292346964, -0.05613466022330332], [0.040071199356620464, -0.022151242255979458, 0.00012009364299651937, -0.06637424238342698, 0.0002124731206147631, 0.06562441693187332], [-0.13926498411605157, -0.004007227067090109, -0.0008176835823349989, -0.012322942459733102, -0.003019810588661769, -0.01613141381077802], [-0.14504266235156246, -0.0029542361992343676, -0.0036206804041433367, -0.05957628106891578, -0.000844653577085028, 0.03212851360094162], [-0.26879802451003076, -0.003171269710691162, -0.009593846941569301, 0.11468101651634587, -0.01299486528062765, 0.03382810103768385], [-0.2023304333813486, -0.006799363178196195, -0.0034039043373561787, 0.005314129914037687, -0.0011952368366323963, 0.03455070153930164], [-0.14888751616090434, 0.017143668309433202, -0.0034315019866832494, -0.07254173602300588, 0.025301902414464086, 0.037100421541934665], [-0.15024198143806594, 0.0002757629820035476, -0.0012932356688332642, 0.004288030527733023, -0.004076053001513926, -0.02686252340132277], [0.29927074634558465, -0.022666541937121688, -0.02692657182886429, -0.20254439517929204, 0.00960825973587934, -0.04393721142190236], [-0.10559833713025349, -0.004193273565232908, -0.004399986471513741, -0.041611195417656566, -0.0035379075231401808, -0.02106929989220328], [-0.2260453340700566, -0.014355438252319164, -0.013498523370256846, 0.13184297954282842, -0.006723853375887118, -0.02646316380764197], [-0.09798029207127393, 0.009271823100304397, -0.0031086245577753757, -0.05660936904169359, -0.0007505365374715694, -0.025399667558756667], [-0.17346225219388686, -0.006846234587569717, 0.04014830638476969, -0.029734241647400374, -0.005453935565147958, 0.042265738561616174], [-0.1460329421445233, -0.0006724124789281952, -0.001462665459547787, -0.06410564957162462, -0.0006991164158723336, 0.033618341626052296], [-0.09959513854045379, -0.0022680059699739727, -0.0013160100225559634, -0.05918813463606731, -0.001307691372418917, -0.01673501945853023], [-0.18299052693387607, 0.021954435508439094, -0.002305467998202241, 0.009525713567907355, -0.0061762218788878965, -0.013751265598713954], [0.62110624288326, 0.037760854353101066, 0.003078407942161585, 0.019172907402383733, 0.0058873429864448905, -0.02391016393676284], [0.23242822439640365, 0.016947566748562116, -0.010668336623504259, 0.30751023739328726, 0.02043521730114976, 0.090452048049056], [0.5005951926809293, -0.0074930352471678455, 0.004232093060496852, 0.2281344818713958, -0.004537723103777799, 0.034789943119075316], [0.5154798465152273, 0.008738496100041265, -0.0034576138422182066, 0.17266144015805734, -0.0003996745886510473, 0.07732941041944884], [-0.1277315502437562, -0.004476242210965411, 0.0051944672865366455, -0.05375817454918846, 0.0017724049566458081, 0.015196237617871441], [-0.19740190567931482, -0.008839946690013109, -0.0022383351116385154, 0.03715637735092661, -0.003306019356739673, 0.00452221043916065], [-0.18915357019876078, -0.010507495515792699, -0.002347127589731276, 0.05506686647688281, 0.0007977598569106005, -0.026266433029509186], [-0.11619127342409183, -0.005002323638464988, -4.508245428128534e-05, -0.06510189686102685, -0.0006678475201607913, 0.012669852469454805], [0.5355906673206743, 0.0006781938917527441, 0.009798167958016524, -0.04374392206450613, 0.0042234122738828836, 0.19253752823922718], [0.2542386715466407, -0.02272544301538625, -0.01431697538708708, -0.2939286562384794, -0.00207035654035363, -0.05769057369866843], [0.1831217570300014, 0.05394632031428561, 0.0006782055675426207, 0.24186897260440413, 0.007345867839221533, -0.0017122266965599374], [0.10455662275872322, 0.09575223356970282, 0.0043306626245646265, 0.19780433044462067, 0.02165176231491792, 0.017314097989532765], [0.024918329393248107, -0.009825114528260109, -0.03501382938346448, 0.18982633399765744, -0.018330608540288753, -0.05999106331984541], [-0.05782102754066167, -0.014217195643674897, -0.002984781277017054, -0.07853774956145738, 0.0010588075458357296, -0.022312815427786747], [-0.2100967545978842, -0.002049500838082594, -0.002776398033176258, 0.009866411156882793, -0.0003815558406999672, 0.029712580761655165], [-0.2282436722860875, -0.004887290813906449, 0.0007221686482871614, 0.07757221040032478, -0.0008210034866462747, -0.013526221985781737], [-0.10927679597993858, 0.0019791611885729, 0.008477337565979714, -0.05944898449109964, -0.0022660957330273646, -0.01470795588382029], [-0.09355263340898404, -0.0053072076405061615, 0.003952687914191903, -0.06144814044937316, 0.00034348148839922734, -0.023398187903727767], [-0.11177900846166695, 0.0017336416859700674, -0.001877039482976557, -0.0603219418102479, 0.0036684179244667894, 0.0046659301444546795], [-0.11644638174260591, -0.006015892846626322, 0.008696883881670722, -0.04485667958471094, -0.0021287279478356146, -0.015194916045605845], [-0.08677990678169367, -0.008973513369386987, -0.0008797609877376168, -0.07155118812516016, 0.006581858977627743, -0.018807489713649757], [-0.09935112199147772, -0.006465542342595587, -0.0026097184121007147, -0.05747760061385776, -0.0005454252621981285, -0.013960591377769897], [-0.2173893981809564, -0.011059605366981721, -0.0025124138617619, 0.07438265288984697, -0.000685124741673913, -0.01786833296069564], [-0.12725845883939219, -0.008443427699164653, -0.0012104580843120187, -0.029941101173728085, -0.0014987880710585944, -0.011616589661756202], [-0.12219153854951464, -0.010734119187757735, -0.0010093310465006714, -0.01085907956689598, -0.0002706196451494981, -0.019017114058336202], [-0.16020753664159643, 0.0019253476283957612, -0.005687652094273108, -0.04522670709474157, -0.0016860769944015979, 0.04205595852995001], [0.2141991876910745, -0.016489607566028686, -0.011187313895141174, -0.22203946613561867, 0.005065172012616096, -0.09366218456111462], [0.06533346325981748, 0.05246814435390066, -0.009306514220650684, -0.008050522344871611, -0.02078309344780485, -0.10077385855277184], [-0.13534443904979815, -0.008301507075906201, -0.00262562113199258, -0.017095563804245997, -0.0030084497950187707, -0.01359324267244952], [-0.20345158791976245, -0.009149974751764005, -0.0006354613406437749, 0.06325468984729948, -0.004343220865676318, -0.014417778302786864], [-0.22542152022511378, -0.0005098438476008624, -0.001323065453770323, 0.10586947127971283, 0.007126190631980939, -0.018282184766161117], [-0.1036471767887854, -0.004038907135553355, -0.0020172194655265507, -0.05700195489899017, 0.0005638107344697904, -0.014268552445614587], [-0.18863061487748373, 0.0077449037305690124, -0.0035395648349458944, 0.06716912489773665, -0.0008893194316093808, -0.03312167234140994], [-0.1256224439096525, -0.003236484463407014, -0.0031799142206612404, -0.06884028914406931, -0.002975648037006141, 0.02344477977479694], [-0.2521504454479134, 0.001576576793306529, -0.0014191908480489953, 0.12544401832351337, -0.008463828246422868, -0.013522130574435036], [-0.10261226744012948, -0.005084949495942222, -0.001294573395849306, -0.05774916163705077, 0.001051705210099294, -0.001387419907793528], [0.5054452490881867, -0.0005210080615116333, 0.00418605728407113, 0.25587105425729817, 0.004030561720927323, -0.016668126410185346], [-0.09222605372008531, 0.008710988427861166, -0.0031023120212534255, -0.07045446854559376, 0.00013392416786745786, -0.015805411642129636], [-0.11326095944710285, -0.008529473150696848, -0.002558067228071978, -0.040381425714494455, 0.007643951505253779, -0.023324025964887182], [-0.22183934545027123, 0.012481002186720528, -0.004567399300785326, 0.08490954710456051, -0.009876187300091285, -0.033184283906799686], [-0.10150637879628299, 0.00735746698482384, -0.0025526730753524874, -0.06358498057190375, -0.0012454728245734873, -0.018877961716711057], [-0.2685357005487258, -0.014573593639889623, -0.0026203176108605653, 0.14082827666991313, -0.010053755660713613, 0.04044786856805314], [0.44575266085170506, 0.03115890315909373, 0.02064243019798858, 0.0037275896448355703, 0.00911214155850779, -0.07206150929491534], [-0.06623131958634268, 0.011343390608187204, -0.016992484703223044, 0.16768454444859976, -0.018365980317388383, 0.0063542305025477445], [-0.1999571908960603, -0.00441327459416869, 0.0012011872641245886, 0.012656599865045848, -0.0026873213894073396, 0.0350578568933224], [-0.13295673423472096, -0.010360901721459167, -0.00038561471800139933, -0.008773946241866388, -0.005119105552851422, -0.02281369753110028], [-0.08208841126879024, -0.0048067176692611815, -0.00073451165701118, -0.06703363764482885, -0.0016224148428207951, -0.020790973583954313], [-0.11317427104876479, -0.004901329135297239, 0.000680002779979312, -0.043289587098566645, -0.0008967367117178907, -0.01882807878563254], [0.39592258184093443, 0.004335839330974989, 0.006248411301576237, 0.2898145940258716, 0.00975705418942607, 0.09016348009552758], [0.05721178758872486, -0.015678133958507885, -0.005141204482434218, -0.16421343735171628, -0.0019906438210441687, -0.03809836797502268], [0.39529652417693734, -0.0217118296651115, -0.01582671284408585, -0.0094134229333999, -0.004245889412759334, -0.1303896217025346], [-0.12063825533228872, 0.011677850069875689, -0.00016203412364545686, -0.07152528597926684, -0.002093257438381602, 0.0113468558195806], [-0.1059723783310574, -0.006987019494988818, -0.0009597844186192781, -0.04809553537331272, -0.0034959736062774712, -0.014458132305155707], [-0.20876985761570035, 0.004431370547938128, -0.003762166575260591, 0.07756674051461085, -0.0024028185561394575, -0.03180660164878227], [-0.04593339036143901, 0.02679596070948236, -0.0028555498726337038, -0.1130720004261249, -0.004692950117551165, -0.02511635564601953], [0.5559866451532525, -0.00851793041540332, 0.00225859685453392, 0.22319616426611333, 0.01278372014987601, -0.034279894421071065], [0.6185404529916261, 0.004452405762677491, 0.005372629081587474, 0.08958099580866143, 0.009479817993302976, -0.009183704235263844], [-0.17487606921807422, -0.003393601015345763, 0.0034822867510091423, -0.031085230360074885, -0.0031875286869307014, 0.04028109491036861], [-0.11170501193037591, -0.007933634255227317, -0.0021725791747951423, -0.039199704024550176, -0.0017855377638230573, -0.01383902304730646], [-0.13816784272259516, -0.003957106291119532, 0.0019292052329409695, -0.04490147636545297, 0.0019348536433825064, 0.014002366502844492], [-0.10247302060407161, -0.006430703946749601, -0.0009027765404622916, -0.04959759612248591, 0.0004623364840987772, -0.021468239270329162], [0.3824468372245456, 0.040919176127384715, -0.0002978948546418171, 0.2741924951578698, 0.007210542638403354, 0.08159131731126268], [-0.10100147066262351, -0.005164867894660488, -0.004989413931439078, -0.09563128248113091, -0.001151875695523306, 0.03432302831243627], [0.44152736194534975, 0.006890260843290412, -0.0036593496705491307, 0.3117324311481693, 0.0013724919438351942, -0.06326724382914353], [0.5199081123032512, 0.020635249976994504, 0.003486486968308809, 0.08642684570678053, -0.002350297152814162, 0.0910351894990643], [-0.1360699454052746, 0.016872920547926305, -0.002732746264509861, -0.050732762285192363, -0.0007715140842321531, 0.0037383332055689242], [0.5402777494862628, -0.0020425414046053012, 0.009093053139466753, 0.24799662611488266, 0.0167636500112071, -0.05542585564295592], [0.49898962342204545, 0.019324183848754974, 0.007549170104935222, 0.19605392299795785, -0.0060864594645773295, 0.062328186541863784], [-0.06609297690734908, -0.004688889451956669, -0.003658638594247642, -0.1278091076600213, -0.0031736066223216003, 0.04251321923589733], [-0.12912064435524054, -0.0055732663783693475, 0.0038123766084649634, -0.06729032083686477, -0.0035195398238741034, 0.030805204309693947], [0.4926232060399922, 0.007293905243594488, 0.07841950321539184, 0.01106276668558552, 0.018559631355359457, -0.029716234762147426], [0.3018781001928434, -0.02203832629848014, -0.022282175495327475, -0.30196491444901147, -0.0029638210820329067, 0.06514566094152892], [-0.273635416215001, 0.025052642089024076, 0.005587374365177987, 0.14686958045666437, -0.02335174024022073, 0.041948511925308025], [-0.21276642307365556, -0.010313159502564711, -0.005526864394119491, 0.06799860138503931, -0.0024568310501213064, -0.012688226075901695], [-0.13608572838628752, 0.034730522770806356, -0.0017072329421599533, -0.025823173757548685, 0.0005340734401352744, -0.03555846112494556], [-0.10907188423826857, -0.0012539999382058945, -0.002445458619722516, -0.054350163039080884, -0.0018896683472170042, -0.011398825817504756], [-0.10250358271732907, 0.002123673311953563, -0.0012867510780774794, -0.052236909576708734, -0.0005013929324440574, -0.022005037007394184], [0.09067345929951318, -0.017785141709875607, -0.016090281659656182, -0.23883816727268856, -0.004433971790020718, 0.045358220779786025], [0.07620914697951812, -0.012433882224415074, -0.009583880815334738, -0.20202697672461634, -0.003381546633016054, -0.009903178042453685], [0.4438562548472418, -0.0007202612542128685, 0.010349659313057038, 0.24815399620749798, -0.00015407364327620012, -0.050990813565547834], [0.3045526262614922, 0.024469277852721922, 0.002374039302179875, 0.3453410373071277, 0.0049209115898970765, -0.07743693993246852], [-0.0729669043498776, 0.011793537715717882, -0.005070646709105698, -0.0961484854180759, 0.03929698152983544, -0.029356149435161204], [-0.2156026888493492, 0.004924687915089595, -0.004328229094032217, 0.07568025660179697, -0.00232314150211552, -0.016708597489690273], [0.24052629102185225, -0.01541896199223712, -0.017326315122359742, 0.11084642212260493, -0.01686173528583696, -0.08054474836307114], [-0.17970234042628946, -0.008345128618206491, 0.0326056846550357, 0.03695063667551654, -0.006713733137768544, -0.02487773819590658], [-0.10362971505970767, -0.00627364367686125, -0.003121429607940791, -0.06869133076249062, -0.001702840558846228, 0.003008959665846543], [-0.1273507779521487, 0.014636587998514129, -0.002822292955431303, -0.0744839376059665, -0.0035625602112632254, 0.029566920120235353], [-0.2161176465412246, -0.0003031250327202427, 0.005131697104377387, 0.08848080454211477, -0.005601555009577922, -0.0233049369677318], [-0.06638996754074511, 0.018769438367030676, -0.004035758015771136, -0.11790625505785524, -0.0031668200692534904, 0.011777695649928066], [0.0053566897271547925, -0.015491311334218006, 0.0006103060068110321, -0.11054057015076797, 0.0012508898573159457, -0.03740552791581967], [0.47467120321768275, 0.02597691661980915, 0.08675537943103505, -0.046037359247730406, 0.016053379839149742, -0.029460472240899923], [0.0822360891996567, -0.017775060477603542, -0.00041183215003181867, -0.16418267968879846, -0.00900363361152003, -0.05311049010930943], [-0.21709173036913923, 0.012023149994789394, -0.0013274224562344365, 0.00493814504056136, -0.002266501965317152, 0.03408247569736845], [-0.1035614382322304, -0.0022175719995769718, -0.003914400826358568, -0.05653389141318251, 0.02734298279345079, -0.012722108893531044], [-0.1702923447889566, 0.002811360392816511, -0.0006435813513398479, 0.011325266770771274, -0.001815260483589439, -0.014295440539701536], [0.025727218951032138, -0.016732250332086248, 0.004625872750337674, -0.052709142530665984, 0.0321389009498471, -0.07886932994719478], [0.45738172071153377, -0.006687828836312266, 0.0039596735739234855, 0.2541599456472196, -0.0008372549515529994, 0.08211374385518676], [-0.0844528137900747, -0.002152245925165752, -0.0029815200843429962, -0.07606398689534966, -0.0014993792175420956, -0.013260054087524752], [0.07462795792371336, -0.027639589391335314, -0.009990760114224773, -0.1223547833603005, -0.0001389431963696992, -0.07391901724897079], [-0.14801914702890534, -0.008475108298836489, -0.003975629980432941, 0.008785514013905807, -0.0019171684868554287, -0.026808460218874428], [0.40701679006912145, 0.028877237330143173, -0.021833857298945103, -0.09197294535668385, -0.00619069699679588, -0.08342755949287187], [0.3962718206051163, -0.03082214780237994, -0.009616947588830917, -0.2098719891508158, -0.01160142236118023, -0.08449875093134634], [-0.26314694563971375, -0.015631924561149355, -0.004021426415441042, 0.08989164555119075, -0.005494629872132639, 0.03532661427057874], [-0.1529789714425177, -0.007256701894145933, -0.0023551803132286107, 0.0018071901866026335, -0.005975149090460799, -0.013651187446249753], [-0.09346493519448111, -0.005631116322164519, -0.001978965195656523, -0.06464366225855492, -0.001864999249764446, -0.012826321779378813], [-0.2051293990386796, -0.00220805102308532, -0.005123835326894104, 0.012713738201799638, -0.0027602129168589782, 0.02367356835343887], [-0.11601753362181602, 0.020420914522741845, 0.037531486211870226, 0.3006308239799897, 0.05906885325883489, 0.10301678320970546], [-0.14957221514815228, -0.005281228659975477, -0.0026258073366579463, -0.03180614359570441, 0.00034120606158261925, 0.01643101407573322], [0.3997830516671612, -0.025321403217032995, -0.020915564110852183, -0.05082257616582012, -0.039579654431698266, -0.1026193299322345], [0.344690037160595, -0.004238160864500353, 0.005458241394247221, 0.3194419077881166, 0.004017837382961125, 0.03916564995909051], [-0.09583622969487376, -0.0047178737605234776, -0.002402150374009239, -0.06031735562933607, -0.0014294645680371096, -0.015706925973220303], [0.49549920463356917, 0.02234450707488099, 0.005205513031662795, 0.22953358174785352, -0.0072505326776243105, 0.04829813023005997], [0.052959930491973714, 0.04228827731867451, 0.030214103674399077, 0.25249747125387695, 0.023330265431980537, 0.0017387091502036745], [-0.1267136666412797, -0.005322355187233072, -0.001513954477091896, -0.06710473088476954, 0.017028362171793876, 0.02519253549477122], [-0.08969881901214011, -0.0049712236092725485, -0.0013881386773800678, -0.0623656607023585, -0.0021938256961848692, -0.017292332302663728], [-0.1544374557012427, -0.0047427366878748005, -0.0029955359482830325, -0.04923617702355439, -6.783965871250032e-05, 0.03634752279744523], [-0.04684249453397272, -0.005776137245303771, -0.00200355931508277, -0.09698055115335572, -0.0007331315302438962, -0.028074126222041605], [-0.16821907279108175, 0.00426080885381409, -0.0005092275371496386, 0.0033325017119558774, -0.0001447246021718724, -0.013177904682986056], [-0.11431163906114974, -0.006345764106295475, -0.0014678409513484813, -0.056895303057346004, -0.0034403308233117765, 0.0020508779994518125], [0.12434754155189365, -0.03715581130064395, -0.010689930759999067, -0.028622601871990686, -0.01589683302258033, -0.07896379316810792], [0.23601812847347647, -0.02141468319974155, -0.015894407960266387, -0.27364636564837463, -0.013723898472291182, -0.0661892493832794], [0.5629855550914051, -0.01073335320241782, 0.008783938012625224, 0.16587085091109927, 0.0008939436761765107, 0.06585769296209011], [-0.20240343609584857, -0.006266340528439211, -0.002387939217370796, 0.060374496946614606, -0.006563672059102184, -0.012329775712520993], [-0.10605257470241435, 0.0007847936004445104, -0.0009040547561233707, -0.05899380812155594, 0.004238489549937361, -0.017816178903621117], [-0.1973675070005535, -0.00597344173044065, -0.00356733601606762, 0.008317577035125716, -0.0034394829346271693, 0.026092824293109093], [-0.13459955169401347, -0.0028081903029097935, -0.0004789789114822444, -0.020637851623635656, -0.001011729738727358, -0.015457031062564467], [-0.2075276080547767, 0.002854405014391316, -0.0034217134617943773, 0.11432403432004754, -0.005016419487447208, -0.024354249132558944], [-0.09549241403050757, -0.005502261903502879, -0.002543539251497656, -0.06080528464565448, -0.0002602154205001225, -0.015806284748337177], [-0.1678541017507642, -0.006233417019348079, 0.0028952812634494393, -0.029217758943860663, 0.0006949494722009868, 0.03110663427990931], [-0.09390252928741454, -0.007679113382435298, 0.0009841667166340794, -0.07583435987474067, 0.0014332429056843068, 0.001255259588938571], [0.07375502884973754, -0.0214385438374465, -0.0062765679954685585, -0.10937172582315567, 0.0019655800756211345, -0.08644573205360125], [-0.19227084338712636, -0.011889215555155309, -0.001982437846929501, 0.04150095476080946, 0.02168906866272584, 0.004916282889485934], [-0.09486461905028877, -0.00205388524855064, -0.0006351810686850808, -0.06379697083543437, -0.0018073117799982201, -0.01725203201704242], [-0.1426760302568767, -0.007494852107833588, -0.005692008568421011, -0.05246380052748506, -0.003969859268550129, 0.033027576370192456], [-0.22233631537198786, -0.008936866468801787, -0.0036380307542863612, 0.07729765389795143, -0.004013518282578148, -0.01824720873458295], [-0.1946582572306881, 0.004685006806049366, -0.003947533803797579, 0.09783933215549362, -0.011823101149288962, -0.015762589634911003], [0.5407691322436278, 0.038485833179063164, 0.006589180890217466, 0.1603506576139433, 0.017661951147394047, -0.032159612217104866], [0.019537594937166926, -0.005290322334185836, -0.002019233637816706, -0.10178841614103902, 0.0013647327600963145, -0.0690329830352016], [-0.08889972254264152, -0.0078061733952565155, 0.0014510855932572228, -0.051541848849549265, 0.0005184161144077517, -0.034033717704530904], [-0.21830526815763646, -0.0010272797746238992, 0.0011988372049161308, 0.0274970192220277, -0.004013362732812162, 0.032256720904794624], [-0.1489604085042237, -0.005571138596089298, -0.002679620180608752, -0.004097223886978419, 0.00033227243180383893, -0.017272349984724728], [-0.08202494864943134, -0.006933434346004223, -0.0032687749373030324, -0.07355435864258399, -0.0010373026216952156, -0.013591180802982104], [0.6096680430515263, 0.004325620340668311, 0.006782885779283056, 0.11213321478594448, 0.007393204502499454, 0.06392708756248262], [0.45646055214056885, 0.030577437271202282, 0.03496012799637191, -0.14819149471945758, 0.015015656532824237, 0.15144629220706096], [-0.10666817398379258, -0.0039675144147319815, -0.0024730583924916587, -0.07043238966165201, -0.0034009590012754626, 0.011532095453944412], [-0.08196364680411167, -0.004715191354176939, -0.004203604634523959, -0.11035978080535308, -0.0015689704122929967, 0.03228214639141172], [-0.1557494983390603, -0.002889701528519565, -0.0029601557332248046, -0.04200110375831374, -0.0018958486413113932, 0.034586308000429375], [-0.052234485887313234, 0.0006524092660605128, -0.0011603341323515595, -0.13212508463819167, 0.0007806746685063918, 0.023782132078600565], [0.03925574714532671, -0.009166136648836823, -0.0017638746796276616, -0.15850330146067312, -0.004532421646945698, -0.028700012709243928], [-0.0674773505444611, -0.0033321912011591605, -0.0037073244247346603, -0.07868719802442077, -0.0008877372247004638, -0.021377022109935678], [-0.09848198373994188, -0.0068547806321320156, -0.0015123529999532105, -0.049561727373366414, -0.0010604807996649763, -0.014605341121608047], [-0.11307154304633021, -0.00868301528734001, -0.002358545464429907, -0.03833125433915974, 0.0019325069041996735, -0.0173981487669394], [-0.15860150988041946, 0.011945488149523908, -1.7927327657410996e-06, -0.059739134661242296, -0.0042195649135119695, 0.03376206959397128], [0.4601392895323973, 0.00933555480957359, 0.004744355833483827, 0.27574115951469985, 0.019334809447909492, -0.06926983364023148], [0.5243477047895236, -0.008238393777250254, -0.0030929381248156512, 0.13953063460728143, 0.031913400683952, 0.08358792515464243], [-0.06843877620865939, -0.008207662719590348, -0.0036178119953868084, -0.07335412222568331, -0.005334585548456665, -0.017707041302223347], [-0.18308272745259874, 0.014472396402005178, -0.003236239597681528, -0.02936906786252304, -0.0015271396255182206, 0.024396270199808293], [-0.261472513712388, -0.01098581790853728, -0.0018181398357867329, 0.09455692029813861, -0.005702828765222606, 0.027262379923795366], [0.07952131716661308, -0.029596757851261354, -0.0028864508363231774, 0.17212124183847996, -0.024231198082722154, -0.1506198982665322], [0.547642646412667, 0.012104268409606034, 0.028970627959955614, -0.15457484643920935, -0.0005937339941603606, 0.09808757444767625], [0.02160040059236098, -0.006271521952777369, -0.0024229218926948358, -0.14982940447220416, 0.0004988657132521732, -0.036877574850681806], [-0.13933866247506332, -0.004332223418141224, -0.004869097341069882, -0.0618246745658833, -0.002962386690370299, 0.03736148893497289], [-0.14548448344658102, 0.02134485153522513, 0.010559683864822703, -0.05041220398440707, -0.0003492298483888664, 0.0032290009269490974], [-0.09914110689104616, -0.0062923869088440726, -0.002847604290147846, -0.049654183615730284, -0.0013504541940534281, -0.021124264100178217], [-0.15299041288964338, -0.0010570556334771092, -0.0014559124027232005, -0.020075296997701195, -0.0008226056254081665, 0.0009912835489531882], [-0.13808926599740123, -0.016908006899840324, -0.003569929879440506, 0.09404280764725227, -0.004536205798700655, 0.05791726759479709], [0.2590656186471828, 0.038808043372987606, 0.017947884594696262, 0.3188629592825194, 0.010422853066142185, 0.1025860670222469], [0.13018184016162399, -0.01576231668768353, -0.01246326452936005, -0.16472077674640617, 0.004302107507049126, -0.07228092303855682], [0.31010034461137465, -0.023091898008123262, -0.023633781964467148, -0.32839975092331525, -0.008238175947744446, 0.06581357969259087], [0.290996461198289, -0.013487742566022476, -0.01477103297958127, -0.2399002755646368, 0.0016402428463503926, -0.08795908150582686], [0.06946942838827014, -0.012509185801330714, -0.002929333018636795, -0.1772698078219118, -0.004162239926820548, -0.03372314753385578], [-0.09264208730783277, -0.0056511808306582, -0.0032587193933057584, -0.06554936917205059, -0.001835863446039198, -0.008139446516780181], [-0.2503689817080233, -0.008427816118209414, -0.002047082071668867, 0.0822679739975175, -0.007621591968725525, 0.013620831202442507], [0.03172874146204426, 0.03748449072564393, -0.0019271845988672893, -0.17789697548488148, -0.0012665702900746948, -0.02563964467100831], [-0.2168784841033221, 0.04256727690758906, -0.0012389218130088183, 0.03870327325878428, -0.005842531446246694, 0.016624625291442346], [0.027319151880707437, -0.008613990567360964, 0.00011661804910107668, -0.1352747065539679, 0.0021460103925106757, -0.05499594034384754], [-0.1816201217110885, -0.005369510329455831, 0.0032429933539109036, 0.044309436999017515, 0.003249984758388481, -0.03207992592791534], [-0.1273951431413442, -0.006405283329073969, -0.001170263962581956, -0.032161453841216756, -0.0003943427111765884, -0.01278547379891984], [-0.2480927666166056, -0.005070599778056603, -0.003052646401823504, 0.09365689784781733, -0.007692705952511183, 0.011508487567845877], [0.07760140367909088, -0.02056459109079273, -0.017713252425026083, -0.2374222507224195, -0.006406939360892915, 0.05736593804328894], [-0.017589904192964488, -0.01024348570171658, -0.00940505385049947, -0.14754998329093139, -0.004883794008462133, 0.05784022147551554], [-0.10830244417511233, -0.0015724642409841929, -0.0030129983034556792, -0.05027218773023307, -0.0013776024094364287, -0.015774263925091396], [0.3727517913070033, -0.03228064437710949, -0.010452212386182954, -0.09060570576321343, -0.018756315383820773, -0.13825738958715317], [0.20153559074617414, -0.02774838541114596, -0.02242313977980319, -0.18423985011688554, -0.009573679166946234, -0.06913047091191567], [0.24990146066458235, -0.026625530088691313, 0.021269565106488805, -0.16341114258750944, -0.017950693440468284, -0.0787523898131324], [-0.10722140490628249, -0.008298833512034632, -0.0017323036409555386, -0.04080264097812283, -0.0032012207491432464, -0.01915359621346097], [-0.119609726217025, 0.012101395148248732, 4.430580096695144e-05, -0.06775958737681316, -0.002247529060886494, 0.011899525543893555], [-0.21616294247559129, -0.007774204959988195, -0.004749549133709385, 0.07855100306043122, -0.00289030687826447, -0.014449124546423948], [-0.26735999638449504, -0.005168702561076046, -0.021485260165028032, 0.15625930545524608, -0.006752782436315798, 0.0026212456154773205], [0.09622863259713653, -0.03545021174197393, -0.004309213845645968, -0.03519028023568029, -0.003056628882941838, -0.11549259200854083], [-0.05184128547697877, 0.022447982553115222, -0.0029225461718267675, -0.12945195295901732, -0.006432079169338257, 0.0019565478907127213], [-0.05772798133724989, -0.0068279878319164955, -0.0029213998549810316, -0.08879044565689585, 0.0003464938378035961, -0.02448867915676047], [-0.21432370393227834, -0.00948378814410601, -0.004205038890371678, 0.03553966174573876, -0.004180824483915918, 0.016243693704932952], [0.24827510143288473, 0.0001450989166224696, 0.023775766182752407, 0.11521107677257299, 0.010446919564877541, 0.09856169275485854], [-0.11012339134133076, -0.004578725860495596, -0.002439194998311162, -0.036873806558143526, 0.00030763334465677597, -0.02660447537068927], [-0.26386296058113695, 0.002097286252927142, 0.002503149065037406, 0.0682242393764676, -0.0035754782235540637, 0.030632335538829707], [-0.09186165839187725, 0.0008969595937379538, -0.0028020813529563305, -0.06567990650240542, -0.001523894057229033, -0.019439419289269833], [0.22404195373543023, -0.003663391542294216, 0.0233009895560881, 0.28252881230963756, 0.00624563790957399, 0.12950728116019994], [0.2873542255618118, 0.10866665189390705, 0.008230903398552723, 0.05501995485148506, 0.016949801965653894, -0.020499898477272255], [0.2052347136387562, 0.006376556663901462, -0.004068800392958456, -0.2272329669922964, -0.007701839848690661, -0.08458909164014125], [0.37093077745076897, 0.028481783023704202, -0.0165131080308108, -0.2238095822208723, 0.006516999527179222, -0.0515585364166392], [-0.11224053431641395, -0.0008996601800318878, -0.005940683024139225, -0.04140385818457908, 0.00036454609966914283, -0.0019564770611710833], [0.0659014379284275, 0.012525972846441573, 0.02660845169365906, 0.2959372498337707, 0.017992873146144513, 0.046814490742030805], [-0.14930919126898734, -0.008178150152236606, -0.002569933408134725, -0.03379689230530477, -0.0055038673970736335, 0.02005914564284836], [0.023469981748254928, 0.0016507799551800297, -0.0051306949135126595, -0.18808109626732714, -0.00028717994390037576, 0.042458288786384586], [-0.15036804412300453, -0.011819513074491673, 0.03345114059435625, -0.03169283717886533, -0.005394792617812002, 0.032348570209341], [-0.23565190139203304, 0.030061711051586865, -0.0035204707359439294, 0.15284217269497807, -0.0015215261939608969, -0.024191413996056024], [-0.1710886047354731, -0.0009199702850935598, -0.0011560840648658125, 0.008696343067946989, 0.00043703940285403227, -0.01516957305857131], [-0.22445209890093393, -0.00803915665849816, 0.015787483201145387, 0.02941121395635578, 0.0014868385028775216, 0.0354611960895287], [-0.22026270589868086, -0.009364681151360358, -0.00475214834238972, 0.08272577895457961, -0.0030396278958272323, -0.00954816646846039], [0.5354111031333446, 0.05044138520097767, 0.00803415619808575, 0.170458491576061, 0.015059869824651555, -0.0007792916474087274], [-0.10831008714469724, 0.007228906430915793, 0.014348886585960202, -0.060304323288969265, -0.0006294053089370099, -0.02208851508939878], [0.058997799358193985, 0.0018787011466543868, -0.009081415861306922, -0.17526674130009642, -0.0015164109676157082, -0.04625526570916387], [-0.14496884822158426, 0.009600529182045287, -0.0037081798421803526, -0.06497533674652482, -0.0037849080318898536, 0.031234435967826185], [-0.12742852187848322, -0.0065517878814470975, 0.01165835417430418, -0.07092761858386494, -0.0007593936607386387, 0.029825158306420903], [-0.2048942404978831, -0.006581287244049991, 0.0018551680947349134, 0.051948977284695226, -0.0016015239659551426, -0.014207269110138894], [-0.21219596832211074, -0.0014301367149883591, -0.002515232504462435, 0.06811657949048443, -0.0018377957175774113, -0.02271411289801295], [-0.13223330247182152, -0.007095101217989719, -0.0015894035649271399, -0.011319162142246049, -0.005347727955049503, -0.0194919693146325], [-0.12780906250169263, 0.016380612349103228, -0.0027896622955592963, -0.08358315727515231, -0.0010093992201536928, 0.025067335610122397], [-0.042496864384569594, 0.10023401531327805, 0.03090724526810054, 0.30938294217955475, 0.024603759622684896, -0.011856574189526325], [0.5282030705839083, 0.005919121633954732, 0.030721357228615232, -0.03631968505487477, 0.011411938543573702, 0.18437840341402692], [-0.1318859471927113, -0.01071273842602823, -0.005658629704469654, 0.03314442232175391, 0.0025047028147876654, -0.008114688601210611], [0.16545105783675126, -0.014570417475807076, -0.022601702830889317, -0.05829653064331904, -0.008148765478975986, -0.0030690382331563437], [-0.17686531063780966, 9.309804816968654e-05, -0.0018104792016145942, 0.002936382175440544, -0.002280710418397203, 0.015088448605639838], [-0.18049091268487735, -0.009976909522632534, 0.036380217516721575, 0.02879370848242137, -0.0020555715205840515, -0.02426886560438209], [-0.09897204000393409, -0.005903870238235448, -0.005004820234837896, -0.05798490468727893, -0.00022287536076813348, -0.012321489474945732], [0.2439835373742017, -0.0008945508993922852, 0.0058998681094785685, 0.3470503493550054, 0.006773383460948435, -0.007266453664697858], [0.08416117206562815, -0.02027475499997576, -0.006439388117702788, -0.2148294866779052, -0.0017619312091776062, 5.23161100017294e-05], [-0.09898949514610847, -0.0037845256109439755, -8.028290729416706e-05, -0.057231882063411794, -0.0009415030543361185, -0.016441134747317024], [0.3687551605630616, -0.010762712301645777, 0.03941490843525489, 0.29586652384936213, 0.005066601823189714, -0.06678024427398654], [0.20720419303532997, 0.01173536298236037, 0.017071513522396355, 0.18974181369275772, 0.015764735163107713, 0.12587000065166543], [-0.2705450547105228, 0.006294472822373628, -0.014855903189546138, 0.14586667950391544, -0.0031873198253833126, 0.009850458732496339], [-0.011221504174680407, 0.030296091701515242, -0.009955612387505499, -0.07393533163999615, -0.007421776297813937, 0.02636979946514723], [0.5101082456115689, -0.004241197495471965, 0.004611279662253297, 0.2988098563372423, 0.007524178330355546, -0.010889029112615911], [0.4667305608764842, 0.014892799720956272, 0.03579880499961951, -0.22851939078060707, 0.00864690599686738, 0.0970125414089002], [0.25662901448297737, -0.023487903942921284, -0.013207701438164986, -0.2756510208575796, -0.014098003926110363, -0.06136224146105876], [-0.2167104618441545, -0.012400368023617186, -0.0005722664464408504, 0.07810409231101191, -0.0053100158636715045, -0.015015929628077416], [0.03985674384039923, 0.003700093244284038, 0.015277665014365269, -0.19743413126198406, -0.0057572340264431, 0.0436611489036641], [-0.13216396221272936, 0.0034182574062973465, -0.002398485547534209, -0.03747819826569622, 0.0005278297435225296, -0.01231544112385954], [-0.21606362621179173, -0.0035055661512086223, -0.001602830362348518, 0.07321680382416898, -0.007189158744138886, -0.01484895568801535], [-0.204898051774271, -0.0031527516274266693, -0.0019431695446721809, 0.026783757914013363, 1.6849048285649456e-05, 0.011930191380895815], [-0.15480874391438215, 0.017249870469001515, -0.00023218808625906004, -0.0197889800767213, -0.004270913786864195, -0.00928821127144084], [-0.13562320589678853, 0.030585874805525007, -0.003082054756347865, -0.04965385969894014, -1.1036702554201939e-05, -0.01835488441756018], [0.12289855790072705, -0.015927249619240663, -0.004928654406699486, -0.07015574479882644, 0.010904170999427676, -0.08194945669876455], [-0.22633470138727502, -0.010790790817798877, -0.00033536127594127753, 0.03062171289076567, -0.003980320822853327, 0.03715628680992738], [-0.11837985851534853, 0.022498435328537743, -0.0032939893381341255, -0.063281761956596, -0.0026213017740216176, -0.015331523744437175], [-0.20853607222643084, 0.011835995108254563, -0.0031880686021744996, 0.00968647480996592, -0.003951525554823326, 0.015927979073903434], [0.5144018834606646, -0.0007743809645094251, 0.008475348514183148, 0.23337774846706674, -0.0037638598822874636, 0.03487326040488137], [-0.09555355239488404, 0.0017371228735443786, -0.00299768608139708, -0.057622019553622075, -0.0014517888976460825, -0.02169994709445422], [-0.15581432306136764, 0.020301985533902125, -0.0018094514330724682, -0.04044454899544086, 0.019505638559346603, 0.004184032729966166], [-0.18326680822749336, 0.017984386410288784, -0.0053109836304592465, 0.022261967107128477, -0.008195876114985467, -0.014116018877813143], [0.48069770083314955, -0.005609114126861478, 0.007462971818666465, 0.16185306408755934, 0.0411022678520287, 0.10383310953545563], [-0.2621603303616063, -0.01021082319616454, -0.0014652094423685154, 0.0761998062929472, -0.005042749919503284, 0.03476930662669494], [-0.0887073120971141, -0.001826744681139998, -0.008019225378759019, -0.11273755241937193, -0.0036914581931796107, 0.04557046126773407], [0.3413431883447483, 0.1233806950514592, 0.019274129771357118, -0.10995746479122727, 0.012294947356147216, -0.011750303424792592], [-0.12566374467814959, -0.007472100871173069, -0.0021773664948762906, -0.044598937404015555, -0.0038724576482102634, 0.0033746070964252766], [-0.1395710423978306, -0.005428203220707568, -0.004041533153976388, -0.06072333186949388, 9.455116546053236e-05, 0.03037067058765906], [-0.08481784874266024, 0.0021128252359254953, 0.0006509973625846404, -0.07999316915906945, 0.005991043453531864, -0.012270514816978812], [0.40761619166456003, -0.01259162402453986, 0.0024772476354664725, -0.2664901785973908, -0.01436122446896652, 0.06716577826705872], [0.27359476997573545, 0.0005246716751357222, 0.018013428172694912, 0.3289095863367347, 0.0036245151240465144, 0.11813639867902027], [0.27716989303559075, -0.01449701713709294, 0.05098224582541571, -0.239710711164787, -0.0017660377903493173, -0.05051892832433482], [-0.21869256943561502, 0.01556525682564961, 0.003520319208644407, 0.10587852930166557, -0.005636749521745303, -0.022350341934154923], [0.0869998176362638, -0.032665418468473185, -0.03326798211499242, 0.14133395223333384, -0.020446023956567288, -0.056474207689427354], [-0.10262006032805542, -0.0030209257431734046, -0.0005745555698007507, -0.05089601371890743, -0.0034255035819288357, -0.015706274391467434], [-0.184649023253314, -0.0007147356191174822, -0.002335833739967951, 0.04004130396843842, -0.0017913048362736708, -0.028460406519765894], [0.06139644004521502, 0.06028260134911759, -0.008417047688942492, 0.023178546093130786, 0.004120494751324495, -0.10630436788317858], [-0.10656598583707007, 0.02307117271747842, -0.002662939689789923, -0.07201711578708644, 0.00021816761389315448, -0.017036632350758404], [-0.15558373750777682, -0.005096447794125506, -0.0038185923059427984, -0.021506667285146795, -0.0021165818006387495, 0.011858852090455745], [-0.11221524038709121, -0.00768822543756746, -0.003195690116325856, -0.059039388809732016, -0.00040801316311372943, 0.0021365579138305996], [-0.196010290441114, -0.006918948683483396, 0.0007378065908721097, 0.04464048384376711, -0.0031163164161601246, 0.003392979391833154], [-0.09875104373583687, -0.008728658036149596, -0.004077790983903361, -0.09773759978707608, -0.0032490695671761965, 0.041995273221253904], [-0.0913669419774007, 0.0007184603210942603, 0.00037093712992364677, -0.06542863079784914, 0.0010638969276084668, -0.01646216604782107], [-0.12077498048108805, -0.007914175947052711, -0.0036531457798147184, -0.03575710298381423, 9.983971035983655e-05, -0.012312395302903753], [-0.18186968296786088, -0.0049335651993176275, -0.004607026592584794, -0.013745893251743586, -0.0008569772706679894, 0.027111081790111395], [-0.011230394599632923, -0.016901227174191263, -0.01989583947004003, 0.23304613325295742, -0.017820968779872107, -0.044333893705412665], [0.6355414218401325, 0.005469653683196163, -0.003842369772202797, -0.014036098249335164, 0.015277278266769858, 0.10032693962826149], [-0.23018854610525544, -0.007811626625184043, -0.002016313103616933, 0.08322608874184703, -0.0009153160045866409, -0.010310347509264347], [-0.15008965894050044, 0.004484799224197914, 0.0034744882552645304, 0.015598185150275277, -0.0002507614975437552, -0.0331984807631213], [-0.21500839476444045, -0.007934026499699898, -0.0023446446788743603, 0.08733414608846399, -0.005934809194616602, -0.03035560428416577], [0.519438167734719, 0.04136237056433066, 0.0072325366237785274, -0.027343534821314394, 0.007173735727840239, 0.11154220036112153], [0.22699827025957553, 0.0009664518972992252, 0.012216534965740993, 0.2308640342954529, 0.0030519516867797644, 0.1442390400237876], [-0.16527911042689264, -0.008769120045563327, -0.003793446197915048, -0.026472318056114535, -0.0032400145798849554, 0.030255120417481646], [0.24500360798094137, -0.014923599128317849, 0.0087463768893505, -0.2763116081894425, -0.006798618299945139, -0.06922139734782518], [-0.0891218688889984, -0.005267805121339025, -0.0030466310078794442, -0.05894003883896204, -0.0027778962684535968, -0.02125575987436744], [-0.19431157990828873, -0.012100957674959768, -0.0031437885447654474, 0.04194736281186298, 1.6209072690022357e-05, -0.011364973029266782], [0.31092629361205193, -0.011343501706021941, 0.0022673274917635986, 0.2340026185435141, 0.014015554195649794, -0.004494678691580744], [0.09594247319213868, -0.004524259637235176, -0.003658558370130373, -0.19567332095170234, -0.002648441753123919, -0.04223678136883571], [0.27688542146655626, -0.03137810479862828, -0.022314155828508164, -0.030751647926001116, -0.018865679390026414, -0.1139858335233921], [-0.21208527619982112, -0.008175230177179934, -0.0022688761475685927, 0.024641252448221067, -0.0027420420740180986, 0.02821699754719086], [-0.13939237368754515, -0.006606284447738066, -0.0030569361822215583, -0.028867532633118864, -0.0009757609072504145, 0.0013857132546997435], [0.2391644771562157, 0.027277468834743746, 0.022121439483126944, 0.2975335090534935, 0.015109391448176293, 0.1218264257243072], [0.4948605576572037, 0.014525235172611036, 0.01573797468773403, 0.24936054879436115, 0.01471767375731845, -0.027233202190442077], [-0.12790055017050547, -0.0010367676239305065, -0.002871795616241796, -0.025830513030786605, -0.0001305867614260462, -0.020541747581423395], [-0.1196390629323994, -0.0036596654349109483, -0.004418747293516613, -0.05265878032238843, 0.009439179613900851, -0.0034729236306847034], [-0.23204341906731293, -0.007459132909354586, 0.0005553781787737184, 0.08120774241690308, -0.0003899997038699295, -0.011910439045009524], [-0.20896080103273573, 0.033193903067715996, 0.015027498449645349, 0.00787025297085097, -0.00316698462695752, 0.02945946450481445], [-0.07443451300295605, 0.01880934613620364, -0.002184391045318336, -0.0997909169089949, 0.002194978242652775, -0.012504503421587428], [-0.1943972899478672, -2.148989427917575e-05, -0.0002504480779222863, -0.0037782342657776188, -0.002012683193388968, 0.0317326850617748], [0.07258288174142782, -0.007448958303640388, -0.0005014463003462621, -0.19178036353062636, -0.0006926464677257695, -0.032033752853375806], [0.08439693295197716, -0.018495021811263817, -0.007751337445687305, -0.20507917037482526, -0.001198512443390352, -0.004988773229751453], [-0.14383136930841178, 0.014058940675440007, 0.006128786117825519, -0.06956631198239781, -0.0030196745866056416, 0.024152962417483395], [-0.12877246358273503, -0.008647388271621096, 2.4099999525453182e-05, -0.020416210935968238, 0.001754446028917952, -0.018519149904785412], [-0.1298508494132067, -0.005784876488057811, -0.0020842305967953044, -0.04236561200523578, -0.0034310107394438143, 0.0534858062958802], [0.49478695482531154, 0.10775595950101637, 0.011592168432256798, -0.057114334351655895, 0.022558617091621484, -0.027574530333716322], [-0.11134537173051104, -0.004624553684207467, -0.0008949717545480157, -0.043665786294825315, -0.0014038954976642454, -0.01847542103824382], [-0.1446207226826996, -0.0040014971969282, -0.005016875142267964, -0.053211070948102485, -0.003873982864480231, 0.03475859327892327], [-0.21833926180510124, -0.008115152828833827, -0.0006110834392331555, 0.07298327257175838, -0.006067473729927446, -0.016926967435329378], [0.10515719959573029, -0.013960852880000386, -0.008824905862847116, -0.2505378413922969, -0.0005775926686753967, 0.05108446006149738], [-0.1298000498771865, 0.005984981080308288, -0.003344670783052074, -0.048021402208808336, -0.003160642110952516, 0.001265117233024751], [0.5256364499938109, -0.004037298532974573, -0.00346201096397636, 0.2711351580623606, 0.002688193606659832, -0.010968531381568386], [-0.21477838771417548, -0.008001019217987312, -0.0030552012363033146, 0.040149046859500624, -0.00032145635840252583, 0.007263684334034012], [-0.2511972335774639, -0.015854775440180827, -0.013544282900709938, 0.09137925586541247, 0.012993324536203558, 0.03893672738975388], [-0.10906144330310066, -0.006945039352007037, 0.001461799687515637, -0.078068139452014, -0.003547512652737195, 0.015750335072343427], [-0.10470683624206775, -0.006115681013865929, -0.00493648526504147, -0.06612733447768719, 0.0006671557743942639, 0.005094895509982152], [0.2674929889217772, -0.003428548538710431, 0.016158323627433253, 0.22014031004946708, -0.01074982698779809, -0.00580499310391625], [-0.2383285215093305, -0.012868871266777226, 0.0019227568775911671, 0.10224554911779396, -0.0018545326176069108, -0.01815336472865439], [-0.02846627970231902, -0.001399012384779763, 0.007776585306107592, -0.12379022216933219, -0.002793730267395243, -0.02328495982990051], [-0.006072625355020645, -0.005947216088582616, 0.01343025581662547, -0.10439428685721049, -0.000345822017768352, -0.057246972164710516], [0.30765750454584284, -0.04112553084524034, -0.030248202204098296, 0.0507502571897047, -0.008806503281207828, -0.0817704619129384], [-0.0895369768928107, -0.00465129769250018, -0.0007692421349808862, -0.06650568808450724, -0.001593872493609175, -0.017352922701591608], [-0.20636033600910836, 0.010157778819545446, -0.0013274494493752359, 0.04153848991777661, -0.005032083148686018, -0.011389574733328389], [-0.09252771441780962, -0.006546517540506056, -0.00255782335872325, -0.06781581891131092, -0.00022972117276297648, -0.01073240459888709], [0.5150117280737342, -0.00932454483555201, 0.0038144112374865514, 0.13624752717141467, 0.009501125568590889, 0.10018102262559432], [-0.1317966559136239, -0.004711305622155218, -0.004870930962180424, -0.06024717582320633, -0.003800308755264302, 0.028627488187542075], [-0.11129690106565486, 0.0024107620118263813, -0.0009138024093991607, -0.04364333655188229, -0.000895100967094327, -0.0227382876844622], [-0.24363962046142418, 0.02320428760555662, -0.0020470682785378317, 0.04255525798508221, -0.008656478665301011, 0.026923621814622913], [-0.15269699703710607, -0.009199318478559133, -0.0012558776717004224, 0.010951899883815779, 0.0011341939704892035, -0.02517723400027211], [-0.024286871961329615, -0.016558975419796826, 0.0005662297757124005, -0.0961660290807719, -0.00242668933654822, -0.021120997310599462], [-0.19824552866310877, -0.010890263628618169, -0.0054390179200426445, 0.08200612004063451, 0.004594704675010523, -0.022289189107049988], [-0.09126292143574093, -0.0020675335025988452, -0.0029926731106277604, -0.07034330308386474, -0.0024947730035993296, -0.011248795863568237], [-0.1302176902882389, -0.010221934536527514, -0.003789937404974885, -0.012676256633366528, 0.009055664692281976, -0.03172651249584064], [-0.12524870460196552, -0.009589457674226485, -9.076359925883392e-05, -0.07510957554040813, -0.0017083045232409897, 0.03361458371687842], [-0.20925109189566823, -0.01046959465827246, -0.004615182581923745, 0.07509774586489736, 0.0004684180552130595, -0.00851529478424632], [-0.20060216762427222, -0.010890044160632018, 0.00033043186854200395, 0.06391719741930162, 0.0001772842845789231, -0.008598062113467161], [-0.23812222646154196, -0.00849964866310909, -0.006235944525002008, 0.06158356203287215, -0.0011961717427799747, 0.027310429359560313], [-0.27692374477498494, -0.006083596694150528, -0.0018947026731164192, 0.10402450863151148, -0.00672413042764588, 0.029667856414577035], [0.23937278769207018, 0.005078914805972785, 0.017458136682501797, 0.326918870282902, 0.0033659261345655705, 0.07077540103202248], [0.22603669261477194, -0.009541622334574808, -0.018935536611800894, -0.14644387947387136, -0.00740957380400522, -0.07032441372385335], [-0.11535985578523807, -0.007377882695738199, 8.880718585039365e-05, -0.02396625549794582, 0.006023258754428033, -0.029818071961356277], [-0.19743632892440735, 0.022557123709117506, 0.000983030198480565, 0.0382738648143299, -0.0034389224689671915, -0.023348767328553798], [-0.1496626378213137, -0.002860067419837373, -0.005258419664808386, -0.011904168717568185, 0.0014341774084414766, -0.0013453917214209285], [-0.21508850755721604, 0.0370464687912652, -0.004116239222303397, 0.06487935996725407, -0.003012400665328863, -0.01337474191973175], [0.24119897916595243, -0.005175214992451336, -0.009070720928736528, -0.27687988711452355, -0.006829083853258722, -0.06963209425500493], [-0.23876510454682326, -0.01689936763392842, -0.0056353515277746576, 0.12413162889712044, -0.0013423542805167394, -0.017554212812838782], [-0.13637437458542478, -0.003951407928110007, -0.0017962907637810092, -0.020514996886084776, 0.00035997549347345037, -0.017638040717560848], [-0.06014406477549415, 0.012020730225594954, -0.0029702822910249605, -0.12247565116426279, -0.0013595581650601323, 0.006018826170247317], [-0.13994959956261097, 8.958510428465248e-05, -0.005170740431567069, -0.0632599835599606, -0.000736690898796546, 0.029117429348650966], [0.23829220861008413, -0.016875302247116503, -0.01463376352216495, -0.2634316485119102, -0.013383811710538724, -0.07540149214216346], [0.19034540410807896, -0.021857369122974115, -0.015032046651748792, -0.23896972784306905, -0.0037073307034238942, -0.05305797740591018], [0.08576924639275259, -0.016282840635241026, -0.009575287455382898, -0.1305867361902284, -0.00272339700341815, 0.039568379970883166], [-0.2653302623778411, -0.006981315841304783, -0.0006388743750894472, 0.0997463107193571, -0.00266129199142601, 0.01962210053297119], [-0.2049226511522589, -0.003094626900880194, -0.0034969283866822464, 0.00501089807957066, -0.002170956479743093, 0.03237537595110471], [-0.1805075308029151, 0.022789473637698118, 0.03348998721430566, 0.0007986528115221705, -0.0023136045986604254, -0.010396144928616635], [-0.10836761679686369, 0.01396309714623771, -0.0024592471776385644, -0.06976694046782039, -0.0016604239744635275, -0.012118868729451325], [-0.1257270868175432, -0.0068488817645341905, -0.006073204473658478, -0.05841605308536601, -0.0013330324307757927, 0.019129284212903646], [-0.12228565695074457, -0.003986710697948087, -0.0028919997980770776, -0.060353214129462755, 8.519506060653317e-06, 0.013432395403505798], [-0.1253288155945273, -0.00203911712146625, 0.004651843819818449, -0.04899346331086661, 0.00024985070731027337, -0.0008947429447119375], [-0.0984322644620942, -0.0037779612269444956, 0.0012228790593364432, -0.05532653793448079, -0.002704864031041656, -0.021391251404774936], [0.007963718142540857, -0.02243704317259607, -0.003400127315472423, -0.12426474541985941, 0.0038542389063678413, -0.026224430496723218], [-0.10622618640907093, -0.00201505600978358, 0.003073134400719306, -0.04659749106777864, -0.001432887188954656, -0.027211513725131302], [-0.09796667314207302, -0.005998596251353089, -0.00029662093325314207, -0.05842719602446892, 0.0031537135079697442, -0.02087462715682134], [-0.0018653203289662784, -0.017694710140667497, -0.003903101752242365, -0.07365987886169817, 0.02752450609068443, -0.045999286790489986], [-0.1126441363069113, 0.000791546407468985, -0.0030244250031263306, -0.04257625227612866, -0.0009662466410505493, -0.01814244696456566], [-0.12457491435013043, -0.007574646263506272, 0.00033357455632616667, -0.0697373801605427, -0.0017564621131490679, 0.02289982833100295], [0.24649745502045492, -0.027729066121216513, -0.01025261022514525, -0.0858632301375264, 0.005431056308950837, -0.1142416207185321], [-0.1227434825665574, -0.007719465690817642, -0.004420471542803947, -0.030075521925740777, -0.0036702547568747733, -0.011780803517205264], [-0.11820832637995268, 0.009095599710924279, -0.0030850230091582233, -0.0860371596714259, -0.003107521189138285, 0.021765763872085193], [-0.10393448376868107, -0.0060120597666898645, -0.0010774390258286603, -0.05029575964403462, -0.003276208678775595, -0.015814049115989864], [-0.17822690347400708, 0.011123736194015844, -0.0034575350908151534, 0.03432920867913385, -0.005296279429529113, -0.008365560212131976], [-0.0419391930739967, -0.007708879201507981, -0.0035904174360183593, -0.10159612087631512, -0.0010868873368132474, -0.019488502075348572], [0.19919256196106208, 0.017625503834162174, -0.010930547411753111, -0.2653619161557001, -0.005542895065343283, -0.06208318335290564], [0.2726516404067173, -0.027493685521755563, -0.024378270258372418, -0.12581362665025567, 0.0019438724596803404, -0.03666913678522098], [-0.19330699813845093, -0.009247657507893652, -0.0023847244335194664, 0.04472793605907287, -0.0022204508577887585, -0.013179232940968736], [-0.2738447910040893, 0.001809405556539767, -0.004916533637342718, 0.12953790425455872, -0.00025078894899097543, -0.0018761486016269338], [-0.1858727418019795, -0.008333853218970495, -0.0014688747726366274, 0.04323039315445976, -0.006345260674987137, -0.014482866149089455], [-0.15432705310718406, -0.018724310844752172, -0.0005193380793649232, 0.10632831505835376, -0.00669631512446753, -0.04102253394724195], [-0.1456180692292711, -0.008951238653099885, -0.002064330141385723, -0.0006319232381913028, -0.0036417081475041166, -0.017002730590546952], [-0.09581952876643275, -0.0059655135898345115, -0.0036484006607335343, -0.05334324982447293, -0.0006605645053411957, -0.020972742653185127], [0.4734008733572673, -0.01027753260247551, 0.0025305168271874895, 0.1847747123158166, 0.001259254514924278, 0.103083898276354], [-0.11785812819978884, -0.0016680952030597841, -0.001309939338193201, -0.0335612016543314, -0.0033773465370253012, -0.016135289067601186], [-0.19769753795021933, -0.011032497576884808, -0.006726683507632216, 0.008099410434887716, -0.0018356894820287261, 0.03341436188715445], [-0.09868319573743481, -0.0021532362354854022, 0.0007438529010110648, -0.06396900683656803, 0.0006014992724451821, -0.015699913363967623], [0.6374609990883335, -0.008525509083780143, 0.002877650183119209, 0.12231959648808023, 0.014447393562236326, -0.01800798738085023], [-0.14568147951215796, -0.005435939216245624, -0.0008033846632001865, -0.030051026458498695, -0.000971606767147226, 0.002533436617249923], [-0.17641184120990494, -0.008143649973889744, -0.0017691198145795674, -0.0047461939058385935, -0.00366862314367315, 0.01919053915899746], [-0.12084182826712858, -0.009891781379474524, -0.0008492161561725506, -0.03114206163125611, 0.044882343194784825, -0.03902578909408628], [-0.11602162822090444, -0.005563137516273982, -0.0019620937054048958, -0.03569827628439606, -0.0005432927781966065, -0.020523532279137686], [-0.21529569659125444, -0.0011492406645576755, -0.005848429953944881, 0.02177239393820985, -0.0018988166373383152, 0.027565345464440345], [-0.14317155332895515, -0.0034118682189048307, -0.002043616895793353, -0.051004588796470324, 0.02649151164343404, 0.03585313146970566], [0.33327583130490634, -0.022527040918209444, -0.01618669165321865, -0.29467869187886375, -0.009452526343542373, 0.07484364329844814], [-0.06997080471940405, -0.006503335594629727, -0.0009697905866676859, -0.08201945365861912, -0.002445059136802734, -0.01806037983328845], [-0.18184645668200575, -0.002955250768803732, -0.005324726572290264, -0.02071334258353103, -0.0007701726808617019, 0.037737800325343136], [-0.1363834276077423, 0.024039450780051118, -0.001344350663653911, -0.0437183351207976, 0.00018877987130911704, -0.009087950592499257], [0.08711235425227849, -0.015110257651606049, 0.00707834448281297, -0.2037362879204733, -0.007361399825742698, -0.00940465809917427], [0.3823153640230951, -0.02860920232697676, -0.012429461083019958, -0.02013721441209198, -0.04087681012720014, -0.10653577131190287], [0.011818059101782488, -0.0078001138269796, -0.004790484393725449, -0.12818664278715075, -0.008724698941715852, 0.06317467449858308], [0.06646430543780169, 0.0018691430414516964, -0.008419931176007847, -0.17152614213012304, -0.00428212459025001, -0.04618191724954017], [0.5635274454424639, 0.015201820249305683, 0.0031590384735931437, 0.12344318921673852, 0.017019890363601162, -0.028246677863349785], [-0.2059139763618777, -0.011700018516178579, -0.0007502205597680952, 0.09288169125851421, -0.008871008706408985, -0.02008671921512116], [-0.21207813869689296, -0.007638515453012066, -0.002342703047343215, 0.09675391184196284, -0.002059676426523406, -0.015447857591758313], [0.5529879438791221, 0.0019016532674874926, 0.03786956447714538, 0.05363653404167404, 0.005028088687743315, 0.1037235749541832], [-0.0721662414573905, -0.009752870162500173, -0.003935238165953181, -0.05956422676888266, -0.0014472044511478278, -0.031044218994125593], [0.253942387272443, -0.01421634313806684, 0.0166281738018236, 0.015246682685123653, 0.022027351869649155, 0.18637047766775677], [0.6377361949311239, -0.004258447329611063, 0.009119329055647933, 0.06998987719018572, 0.01059284048499523, 0.06728899354644279], [-0.15699822872442457, 0.002266317445096069, -0.002435312234882943, -0.04975720800586486, -0.0017182555716182902, 0.039566020425028166], [-0.12986747973030519, -0.004562885162168344, -0.0024599083781101988, -0.010403599737898995, -0.0027410089017432127, -0.030375118089773934], [0.08001414190714493, -0.013494705381237485, -0.0008175960958196168, -0.19771050727395723, 0.00046710902330712983, -0.038093932375516316], [-0.1738188293398016, -0.009249250717930538, -0.0028668842202959842, -0.011147312693586688, -0.001959302331564588, 0.025139515811115506], [-0.1568178072632147, -0.007016845082276449, -0.006131243628101946, 0.021425721194165043, -0.006360377758121093, -0.018842780795784032], [0.08183393212433439, 0.0009193699542415724, -0.007967859824120318, -0.17658020490428625, -4.614347973449732e-05, -0.042886859438201495], [0.3754042191319508, -0.031196198306644857, -0.01660568128657954, -0.07768694771251777, -0.009581550020318685, -0.1344343179963664], [-0.12888830118764524, -0.005013704509489526, -0.002235842028269847, -0.06756053486537493, -0.00139861466896345, 0.024686997259742913], [-0.12999118034225765, 0.009929297330152862, 0.0013485031508108833, -0.07253418890137132, -0.004106436897514491, 0.014944005660179768], [0.15394193989260185, -0.006019031859464841, 0.06281950060464724, 0.03379731296930094, 0.02291639790729942, 0.15471324556498064], [-0.07620344719238718, -0.00688771097826688, -0.0031996985514358275, -0.06049912478618536, -0.000947462555731934, -0.02767255593599277], [-0.09312668400205339, -0.0014612828316949303, -0.0030831829229584357, -0.07112756259862982, 0.0005216691989856569, -0.012132956843648691], [-0.1348307384025182, 0.001921635572928161, -0.00013024089066392429, -0.06936115078672063, -0.001725576552210125, 0.024271626614740954], [0.5807683822145278, 0.013710613765260095, 0.033653356262192465, -0.09753710434241386, 0.006404210331793136, 0.04545165287974946], [0.3594880547423128, -0.015083817621173432, -0.0035033804034659733, 0.28295380921630964, -0.00539619115423381, 0.07377941899314291], [-0.11690860033993311, -0.005591642480591748, -0.0008147643188905729, -0.022914978140101717, -1.8616245978654545e-05, -0.03406335925881787], [-0.08105633621529462, -0.00542325234785967, -0.0023957969007951255, -0.06383197001996536, -0.003325048775850647, -0.024377595740234327], [-0.1024106906862231, -0.004176123410966373, 0.003475409333529472, -0.057626702379909245, 0.0010238767021974221, -0.011504593088039677], [-0.2211946861412023, 0.0129620453441162, -0.0019496934741669524, 0.07973635809996359, -0.0033540266816179673, -0.038776663813759195], [-0.19641175664257224, 0.017131803619808886, 0.034775912795904355, 0.07022112335166009, 5.712424109028121e-05, -0.04926754069922436], [-0.09955427298464807, 0.030339721208751076, -0.0028909933067786868, -0.08066822595364188, -0.0034376195966511047, -0.016594442700364344], [-0.2583627065138879, -0.004307999973159077, -0.0035809429763767214, 0.07853323595800778, -0.000804529249461329, 0.03561294275487678], [-0.16551901804672203, -0.009821010589226084, -0.0018155617547661502, -0.035420086126200755, -0.0019321905543163832, 0.035208978182342386], [0.5344348805250672, -0.00670441760173585, -0.002455472127473404, 0.25804023842949086, 0.007354874938457969, -0.010142429093780755], [-0.12524909797060635, 0.018998406180600568, -0.0019620977668075514, -0.04044810068739387, -5.40504750004972e-05, -0.016956964042696813], [-0.18815540246598353, -0.013862201108366182, -0.006852302546933487, 0.09247703859857413, -0.0003217486766986328, -0.030559669514877517], [0.5050918673583874, 0.016261149863080502, 0.02401729517660158, -0.19422921550957967, 0.015957546665980944, -0.01106221498304132], [-0.09234584403781156, -0.0009413010363375959, -0.0017561622347396094, -0.048425546522052304, -0.0024377433324425696, -0.01783673616994916], [0.4818170262607568, 0.03442953944429353, 0.011660223461401636, 0.26151058096260393, 0.003248722259467176, -0.00687371143614195], [-0.12060793337265346, 0.003187664006450931, -0.0002720391564923945, -0.07492745768780389, -0.00290116335953687, 0.015825215284321514], [-0.13143553283373954, -0.005440141898287114, -0.0007667962737482244, -0.026891299083052246, 8.466843698352668e-05, -0.015862859132469882], [0.29692097475269164, -0.022974078043105382, 0.058017381809788736, -0.1376482034644656, -0.022268884639705826, -0.11642346025647417], [-0.18934983766459762, -0.008102884726262178, -0.002955103182147047, 0.047763024767099566, -0.007185093171973179, -0.01558010602212018], [-0.267026066498967, -0.0007015635365562902, -0.005618601488084108, 0.07759259026660675, -0.002377917293743172, 0.04013822521740972], [-0.14812819211503017, 0.009361118272795368, -0.005587235568973851, 0.08295493670706905, -0.0040276392596383945, -0.04310322613145977], [-0.14606378043661095, -0.0035370976298204834, -0.005016424525176312, -0.024403580083396468, -0.0047435875846783, 0.00918780359301633], [-0.15785957422226893, 0.030464742514538023, -0.0008844495097660424, -0.018635558473937917, -0.0031712293991063103, -0.008011430909458366], [-0.16192403265909613, -0.0004565743384174741, -0.0048459567914362735, -0.04324016453055371, -0.00041616175874138094, 0.0372923345226892], [0.6139081662790113, -0.004938981639359496, 0.003083879824428681, 0.08251927852940875, 0.0035365748195511125, 0.08491756237343678], [0.4890311202727569, 0.013815146658089968, 0.002543679458822668, 0.26011213234761377, -0.0049029337754538885, 0.028157521704835174], [0.015101167245800325, -0.012676365806894626, -0.0012226576615116088, -0.13915900580543306, -0.007931130551511567, -0.029522007420449827], [-0.09828495248348039, 0.0200528601736464, -0.00589919889709025, -0.10827822023428132, -0.00039044634983109543, 0.026697650098728803], [-0.12334269820163231, -0.00867409275175825, -0.0015487436187840381, -0.028662823894972662, 0.005495397239016669, -0.017010372105202662], [-0.19618183252035779, -0.010850125308811895, -0.007212030478252248, 0.0701071273871763, -0.00511356853212536, -0.030896412652891923], [-0.19126754880869137, -0.00485571539452146, -0.003920616881019771, -0.0008526237423296678, -0.0020261863325034706, 0.028187294333668844], [0.5435331952457496, -0.002684037617519575, 0.012063610930861602, 0.19711058301846282, -0.0061044968137889625, -0.008069751122310077], [0.3802352562877143, 0.0358093899760755, 0.0059462329388451154, 0.2754660518828288, 0.005317908050842185, 0.07339477732566044], [0.5295513883826912, -0.007352295996298765, 0.011881540929059946, 0.23606420112624654, 0.016607868517527637, -0.020863916778088906], [0.031959981623213334, -0.010215709109216235, -0.0071561329863775594, -0.1874971070618931, -0.00911618127943971, 0.021416736115300473], [0.34044106465491847, -0.00039998790422658323, 0.021772759572697194, 0.3335628836380946, 0.0120524333653941, 0.023915883303156894], [-0.2135890885657087, -0.009302389558200636, -0.0051355806101544745, 0.0890354147776276, -0.0017470810065329187, -0.03283794170369732], [-0.12842878645275949, -0.003949554846882882, -0.0007673375142157905, -0.020861454604184117, -0.004394962213748048, -0.01867457103487586], [-0.08437893369740666, -0.003664909497635505, -0.004839697434004287, -0.0684550616786526, -0.000612285051834255, -0.01845911264046663], [-0.10966147402809487, -0.006864505129228012, -0.0037146276460065086, -0.08498458917952412, -0.0007593142130549535, 0.028907843529242534], [-0.02002695806101019, -0.019846996017304726, -0.010269706412338663, -0.026673756234898573, -0.014381205911179638, -0.0181637583156493], [-0.07431136410165051, 0.00569974295785018, 0.013077435217095675, -0.1018379971698957, -0.0016806476407841403, 0.013434497404051504], [-0.10748409340211898, 0.015275779121313367, -0.003047282193536159, -0.07767208692164602, 0.04109834434148073, -0.030574708564540642], [-0.10813491141903679, 0.017266964996733587, 0.003698225782903542, -0.06736115042552736, -0.001566053697771505, -0.0203130752373013], [-0.0494377738029081, -0.007060686382808443, -0.0016142725731894123, -0.08062756958658741, -0.0028726693137935302, -0.03636845691214177], [-0.1216336099886014, -0.007850270465305105, -0.0010748878737289309, -0.01666286439193969, -0.0010946630178501483, -0.02734370426257473], [-0.21133487579510118, -0.007327847635878699, 0.005656045202326245, 0.07882119453876381, -0.0047477318067144685, -0.03147678450339553], [0.5665969748564696, -0.001624060798046562, 0.008137304168245055, 0.1382281180948414, 0.00915499214210378, 0.08452921121892458], [-0.1016104103906003, -0.0017873962602797534, -0.0022225978571046605, -0.05522019893315669, -0.003675708224959099, -0.01589368833389932], [-0.1485899798207127, -0.0054342103168004184, -0.002521695186647315, 0.011428722560680466, -0.0020930449739618635, -0.02819979226255744], [-0.18541194169371966, 0.0065324870423028114, -0.002925239678082855, 0.0028787807605172044, -0.0036564410755968676, 0.01886391308613797], [0.3474150792484147, 0.018099915988425058, 0.01460259280355845, 0.32781105487630413, 0.013596410944361268, -0.007507398183409741], [0.04249304420588334, -0.01746870440065246, 0.0024583714575798932, -0.1058309985541576, 0.00019708172391432025, -0.0595683182420911], [0.5907037079518542, -0.0035020482766711723, 0.0032871278777290335, 0.2009879279910747, 0.017037383341260374, -0.05662647983763025], [-0.2402612167629785, -0.006956435598872328, 0.0007365767641039634, 0.07357735156454064, -0.005415828293990289, 0.014576218993863046], [-0.13016327143490497, -0.003355699710103196, 0.0004873781604851052, -0.012423920515350389, 0.00016817266452598064, -0.02803932583131876], [0.19608718393191624, -0.03366567433446485, -0.02414211812935138, -0.13352330760283185, -0.005852830494361429, -0.11172116653617235], [-0.25512940686880387, -0.0077167336046307, 0.0005990256831964087, 0.07671924116137459, 0.015491551320485034, 0.032376322308377066], [-0.0984964196351576, -0.005170213442657342, -0.004123322273460332, -0.05804626187165059, -0.0006372686667052259, -0.013936514110368874], [0.017880250986005538, 0.038365121477797706, -0.00501630280757071, -0.16327033808151684, -0.0006822108071642641, -0.036365092196123466], [-0.0785099958333879, -0.006711095908655154, 0.0010359232279112927, -0.07572268847317096, 0.002626291794065401, -0.01562843480676279], [-0.08771685977105467, -0.0021786419433599004, -0.005008592830316574, -0.06265028396488515, 0.004489245908569595, -0.026094867398953614], [-0.24699806551219938, 0.01742747515074262, -0.0006164662915642146, 0.047695779870260364, -0.007154138426880439, 0.033223510447735145], [-0.10943837641043301, -0.0029616674621931677, -0.0017953310366685244, -0.04724551365940145, -0.0020910362174735757, -0.015628075213830063], [0.11736039572290972, 0.02630932357948295, 0.021938520677530438, 0.0260497273873727, 0.045836970552647884, 0.14213059599059105], [-0.15584709862101304, -0.005782830780588489, -0.003393820153023325, -0.04341723454824137, 0.0005146430413099693, 0.032377452172666896], [-0.018945290609611337, 0.002442650881711925, -0.0014377292556542853, -0.15744668973262055, -0.000558511573109615, 0.0462121575908714], [-0.13568427182347015, -0.0010734650354210136, -0.004550792181708069, -0.06603625742719543, -0.0007910344727799504, 0.03272582094057548], [-0.16625381823493554, -0.008458211711137817, -0.0011080003579679028, -0.037943470445229374, -0.00108028602200158, 0.03554489788238291], [0.5849489323982859, 0.07221493733813132, 0.008437873057720887, -0.003102887267762932, 0.0038508401753772443, -0.011854933796993039], [-0.14792429777573546, -0.003343052609379526, -0.004063954958207692, 0.008410750846254617, -0.0006356373328354923, -0.029853808170095333], [0.13554341267713355, 0.11489189236931516, 0.02516910933006911, 0.050163504004614924, 0.024118425156103105, -0.01375467687056972], [-0.11656773886275923, -0.00601116368486603, 0.004582366759755048, -0.03429830253243764, -0.0013797922764634206, -0.01968705007549748], [-0.09048845970988362, -0.0032590619066227146, -1.2062967694994095e-05, -0.06176548905685466, -0.0029151173023214062, -0.021969809056622703], [-0.21435495681563385, -0.012437285025841018, -0.005017055441005924, 0.07349101181675893, -0.006572766182281395, -0.01479167562472378], [-0.16824846931771512, -0.007571261823353864, -0.0031477611662653667, -0.02856694386186089, -0.005044446965505426, 0.03705990877572613], [-0.18867921025988008, -0.0013815563462781068, -0.0016280970151474493, -0.0031367708949616075, -0.0026852744561463135, 0.03655924230574645], [0.024355000706977336, 0.04208103988757572, 0.021851388912262813, 0.31279297229887, 0.010243987718656715, 0.11844616603121147], [-0.057947746766007925, 0.04005366456186214, 0.01574568958358981, 0.20160324445526173, 0.014968058728506692, 0.10714208943678735], [-0.21497814174495314, 0.003116101432506026, -0.004171631165176434, 0.011508452064038013, -0.0011872831854936928, 0.027275136245624988], [-0.16612386347729777, 0.00013625516079927518, -0.0007086684580417698, -0.03875044514049145, 0.0007715677215558682, 0.04126515419347612], [-0.007902047935837774, 0.003102944878127366, -0.0031084082942330355, -0.10538028489337806, -0.006658137338470599, 0.08346252088538016], [0.0823909556470039, -0.023067864488868908, -0.005674113568184278, -0.04655056914983514, -0.002033482184366259, -0.12420643249460778], [-0.1541673431250939, -0.008617559732419818, -0.0018840228406680573, 0.007014897833800625, -0.0048884017176459805, -0.01786757041797241], [0.30394786864938483, 0.007280404444211522, 0.003950731554707502, 0.35256532675595964, 0.013459389353468446, 0.07762345872944543], [-0.25191363568212144, -0.01663954978579493, -0.02559786372672299, 0.1700031907518956, 0.004181511751849437, 0.03723293399248051], [-0.159448170275922, -0.008855524201829602, -0.002235831405340174, 0.0030541569988436225, 0.007015053865182433, -0.0014215690389049827], [-0.09339751429169488, -0.0018588772781874979, 0.0006478825257528547, -0.06798019265403304, -0.0014654261243742153, -0.01635587217746288], [-0.15072726881549817, -0.0010620413214849345, 0.013854990828002933, -0.05786551982506952, -0.0021984447847162715, 0.04106447439495614], [-0.27196447899403003, -0.013598269364476502, -0.00382033890817626, 0.1213503996103787, -0.005462154243734551, 0.022668175233371973], [-0.18307848446633246, -0.008011879305685617, -0.0035011377009820654, 0.06186410326670316, -0.005467183626918341, -0.03304875150011758], [-0.15651647032483623, -0.009579371503575303, -0.0013570384167378268, 0.00160048919183285, 0.006828324660287717, -0.0194811717022092], [-0.17023420496405817, -0.0011645775931931309, 0.0006413261662612913, -0.031643291963006116, -0.001236783547856159, 0.03447753190185161], [-0.13560177928732767, -0.006762754639266968, -0.00038856031559594947, -0.06814028009894126, -0.0010622274720390023, 0.03210115736872706], [-0.1083508796198051, -0.005161289726438032, -0.017325068989301563, 0.1779104070387319, -0.014464069066658949, -0.09379687741430646], [0.08982402055191299, -0.0036670198982255904, 0.014821935057195049, -0.19634153095528326, -0.0016770295802499766, -0.03983099788597231], [0.3540285048866206, 0.013197359401885125, -0.006461417306877043, 0.2752152228449942, 0.0059717638274034346, 0.07996503154743669], [-0.0706976304767076, 0.012142312985045358, -0.0006247502800988616, -0.13374306287632454, -0.004608140095429313, 0.037260159632404206], [0.6212833778183521, -0.004160190631054042, 0.006096081649837276, -0.010981345461178809, 0.014137210024915041, 0.09694105707531377], [-0.23037892609642166, 0.05149660989458899, -0.0031551380060129705, 0.08373712776946679, -6.899081673166799e-05, 0.014495031540824793], [-0.16396673744906184, -0.007578618672367748, -0.005540390514673979, -0.031206011873484558, 0.0006530925081736772, 0.032119691642439475], [-0.12624143407274357, 0.0017663693976186403, -0.00022184157730826791, -0.044662614848650035, 0.00011282773740473948, -0.011163306636321181], [0.5855784248859508, 0.006824604867656928, 0.011115367156181255, 0.04040869658815338, 0.016806342123870915, 0.0924218324434531], [0.08491136183675808, -0.0013352156483630703, -0.007880961079076383, -0.21248285279602144, -0.0013679756445196304, -0.024310217474639545], [-0.12247723391512352, -5.822056903568425e-06, -0.005252615168598371, -0.07353004407766299, -0.0011082101130234845, 0.021963925331312188], [-0.14758025583447246, -0.016608514973054846, -0.004236680483307785, 0.06270405085619166, -0.0018437205104141219, -0.04349487905494293], [-0.1081856362990112, -0.006172444806990259, -0.0022635882740469097, -0.04407990215074858, -0.0007257548554032697, -0.01854149714321151], [0.5310216093979767, -0.0007239523346967479, 0.008591940254616468, 0.2524949664583255, 0.007421050368256988, -0.027340614144481707], [-0.20423836792210975, -0.011043684460300757, 6.533533834589439e-05, 0.09277375858791483, 0.017323639769763918, -0.03794544321837617], [0.5015324754486368, 0.026892305761353513, -0.009666201861944805, 0.17432752450664676, -0.0017211418161399558, 0.0793174749362361], [-0.08878667237023335, -0.00447884255555035, 0.003229797726045109, -0.06636905034196833, -0.0014013006758023933, -0.02260393178249064], [-0.09258367303566671, -0.00032012839485912503, -0.003058418735290762, -0.05764117370671045, -0.001510053574436686, -0.025296552553036065], [0.2040341531663736, 0.05002681736642827, 0.021447733863410548, 0.19924453803824418, 0.029224784810264325, 0.007231285631943377], [-0.19271354387571465, -0.005112140895060567, -0.004539008836342539, 0.043297307831711, -0.0029798026741807903, -0.011993763931364738], [-0.21551745146402412, -0.01666601338563008, 0.0004233772074733772, 0.14657861527483107, 0.006456766573325903, -0.04859898468216593], [-0.12214077104970741, -0.006557681813130579, -0.002783959452964453, -0.061871730665345605, -0.004935885405791286, 0.018991139498051302], [0.28821949942826114, -4.519228772115016e-05, 0.01072458513086195, 0.3188998173066174, 0.017523030967934738, 0.11726454258268196], [0.32145944082600575, -0.00026401330576783565, 0.021479985360891015, 0.13933340660710158, 0.012930839388908785, 0.1678106946582141], [0.00932506149070373, -0.013238889107610964, -0.002946441156371608, -0.1545776910180856, -0.005404742477961507, -0.0019006310640073242], [0.09273321271535367, -0.014588918131829541, -0.010361424635804554, -0.18643979134411243, -0.0015821115580635903, -0.04186968499426164], [-0.14300271644249482, -0.006051042892306959, 0.0004981454455432536, 0.004645999240715465, -0.0026630984398919783, -0.028003953578230545], [-0.03480546380745218, 0.0019183429604320366, -0.003011217630532888, -0.12183427037572471, -0.0019575788014394354, -0.01562457425004502], [-0.08236405731587636, -0.0056313882908408395, 0.002815369313116858, -0.06883929468029039, -0.002341610259948046, -0.019049018766161316], [-0.1103510989243826, 0.015664598739008606, 7.671212777637743e-05, -0.06999860948182388, -0.0017503174392215427, -0.014051285021356724], [0.3266934781720604, -0.0026239608491824104, 0.009191329888561167, 0.09448829266867305, 0.010334955745778487, 0.096533501776706], [0.5343903443762901, 0.005756559989455093, 0.012308531684376815, -0.04995916646158392, 0.00607557518401387, 0.14526815522744624], [0.08944638823110196, -0.016799906681716444, -0.007859100639381389, -0.18773168669651125, -0.0025674335695483738, -0.0469570841733561], [-0.1041473911758243, -0.006637134730182088, 0.0019191152280298147, -0.048564265935375044, -0.0005046889481842905, -0.02247563443846403], [-0.10699217292376165, 0.01649458199247231, -0.000298088990699125, -0.07368847607171977, -0.001602848421559003, -0.014322995584732665], [-0.15554340733698174, 0.010197693394913769, -0.004548601266916519, -0.059208073710296676, -0.00023081665596231713, 0.03692320557524394], [-0.09758084774458038, 0.016210336202733013, -0.004041091067235154, -0.08042984486811919, -0.005303867611520658, -0.00832718491127751], [0.5784665210908442, -0.004997503477219203, 0.0031514210906113133, 0.20464539848374083, -0.007305664512564009, -0.01239100600874593], [-0.035892244330669704, 0.010110847064113115, 0.04087006084884652, 0.31226693502275443, 0.01207806000700107, 0.06872062710223832], [-0.10101513708195663, 0.007422691082679798, 0.00017928716952813695, -0.06565154992686216, -0.0017545838497924405, -0.018757374060262982], [-0.1479348738425413, -0.0012060524234987688, -0.0030808639775841444, -0.05018905427234963, -0.0014523721002070423, 0.02956432772729269], [-0.15322140405543924, -0.003172647128087369, -0.0018642096611570615, -0.04674470291802655, -0.002624790469602531, 0.03505108756564639], [-0.1353875450398131, -0.003882496409577479, -6.327526758287686e-05, -0.03375332278963493, -0.003579280252253932, 0.0013154435683865632], [-0.2592243451243852, -0.008278950501335678, -0.001973412694833098, 0.074477685111889, -0.006317255999614849, 0.023406279208279094], [-0.11461185270566797, -0.0026218021859048, 0.003487735852961349, -0.046419232857135194, -0.0009285852930615814, -0.01931626281119136], [-0.13208380872052783, 0.003689998778656752, 0.0005461032865491972, -0.03509686823024257, -0.0010304781581264354, -0.013379391400753369], [-0.25004677478733595, -0.007174325727470418, -0.016348203419720323, 0.1525551775427698, -0.00788163942588717, 0.005307194389073035], [0.002216440546264647, -0.0027580470621674895, -0.0060925916888692686, -0.12399376328081785, -0.008743571055913906, -0.00580387362096088], [0.48962706967151653, 0.014708832632519524, 0.0020422413330703133, 0.2897015363720065, 0.008713064585407994, -0.015976554118332216], [0.0266469372519543, -0.02842038331044507, -0.019597098928679803, 0.04457349856318734, -0.010035819274114165, -0.05423318005353657], [-0.09710669895847807, -0.004337202410069628, -0.003598591109416359, -0.048428379551521655, -0.0017200330835651235, -0.025219094886948828], [-0.13493192099085174, -0.005005285675504291, -0.0032658713479513466, -0.054998741940606465, -0.0012659166512818487, 0.020168847717307172], [0.5619585120877564, 0.011264487396432807, 0.008086253574344471, 0.02481034714076833, 0.005927854019495997, 0.1580901648288199], [-0.09845769663859752, -0.005010615625599711, -0.00046487199987457176, -0.06408610896848181, -8.897679518807546e-05, -0.010301729972258295], [0.24952559929672735, -0.021944785895215348, -0.01451001935368057, -0.23244822924426592, -0.012589893965082971, 0.0023728053519934547], [-0.10726923933355761, -0.006825709429200027, -0.002773596238688696, -0.043951735225580095, -0.0009735768535392611, -0.01567496644884609], [0.405888581527818, 0.09078312507703015, 0.0027132423562687166, 0.08438808959679442, 0.01679723771881492, -0.014290792149743861], [0.5097924645610835, -0.0018201463071508583, 0.0035249228469348072, 0.21668907739715249, 0.007222132616882719, 0.07918154888509547], [-0.14611672555312635, -0.005563513540411441, -0.002669909472069718, -0.04243209031625418, -0.0005518078657487297, 0.02177340572196923], [-0.11168488246635962, 0.0033827265041738203, -0.005220038881614303, -0.03627054228798205, 0.003946938815844898, 0.03336658492063741], [-0.11506449187873667, -0.0024374085616995322, -0.004488580318849537, -0.07808015729332744, -0.0035279022853576627, 0.023188540337971672], [-0.22360440376483773, 0.01611470602853183, -0.006573670294736548, 0.07442747959915878, -0.004668225754474728, -0.02594636200411777], [0.21330816332151256, -0.014452139105645004, -0.005357773065943256, -0.27476043071991896, -0.00489709914703858, 0.015832612050365846], [-0.14322482617536147, -0.01082114289273634, -0.00869892209358296, 0.08106547570810181, 0.00021189112352831837, -0.04417580900328284], [0.389058977384446, 0.08367941732137923, 0.015496215765823626, -0.028612737627431613, 0.02547836492736359, -0.011582686156530271], [-0.16186145869969, -0.0046691135222281955, -0.0037056245645627436, -0.02923569331284429, -0.002993763975593153, 0.027333431852696554], [-0.15687288892182547, -0.008401449743017823, -0.0011914613828103724, -0.04358366020651777, -0.0024648764448316615, 0.036965447810114495], [-0.08544066174312082, -0.0041743854313220285, -0.0033968685941352297, -0.07369905317361594, 0.001576905013674646, -0.01527593607148069], [-0.09117716236132148, -0.004476920063349948, -0.004796491267981662, -0.06523326734266044, -0.0011937448461764818, -0.011032414118510072], [-0.20806075138467311, -0.001116123351294693, 0.02813604227817691, 0.055081998121272534, -0.005449517078792887, -0.016293315251355567], [-0.12849877428440776, -0.006082948473435684, 0.0009120216635897109, -0.021007025035890046, -0.0005286268959110515, -0.0197097823614332], [-0.10961384121815096, -0.0064827429902099675, -0.0012247211136557467, -0.04003878226279578, 0.001372092367841234, -0.02442200478302862], [-0.20289740065707534, -0.0038234241385207116, -0.0022641640670617896, 0.011660867601249689, -0.0042045054001609, 0.03542418221712501], [-0.16183626198126988, 0.011988801301478488, 0.00017363865115091836, -0.026485444718793605, 0.0006228817520457223, 0.024401276770279947], [-0.09497571068538471, -0.0030564928806741543, -0.0004895425009444686, -0.05908524367822186, -0.00269412177275223, -0.018858888482022704], [-0.1032388614743289, 0.028212777089475016, 0.04018863032494154, 0.33064438137168606, 0.020945250721972515, 0.016565996569426266], [-0.20161163432606533, -0.004871462554135084, -0.0031924649838274027, 0.06419055118092952, -0.002483206585251822, -0.021425116064983565], [-0.16632240494543646, -0.01019671101470899, 0.0010316209640742642, -0.01898382495552152, -0.005143323616365603, 0.026422897536212138], [-0.14722756660988154, -0.004389204282247441, -0.004009489536983279, -0.04602514178119588, -0.004116412602864649, 0.026468925924284382], [0.5331566448605047, 0.016071214258345637, 6.078158685745513e-06, 0.21853090073125603, 0.009944742323785631, -0.024226723189721883], [0.090292115940071, -0.012919616876866553, -0.0018974869571838507, -0.196663722200089, -0.001043112766056683, -0.05065366733595384], [-0.12204142660711914, -0.005689912815611066, -0.002280962449574141, -0.08041766604475462, 0.0015643891653575976, 0.0411143089104325], [0.5512478546724263, -0.0016230062600319397, 0.009191911975577602, 0.2469613950733155, 0.010986235318893797, -0.0046743907801808345], [-0.11600932735564404, -0.003567362050423662, -0.003448637836343605, -0.02711065994541619, -0.001123400986361438, -0.029052572610124762], [-0.09006914305179817, -0.004197858638168011, -0.0029114436260863, -0.05870755814436477, 0.0023376285955831588, -0.02311162513516589], [0.3599149716637396, 0.014453236722900794, -0.019572045578184063, -0.05907494240465101, -0.025938353867592415, -0.09986097620432302], [0.25455904910855937, -9.654894517725336e-05, 0.050106409720193014, 0.25060198529120215, 0.0029233303365945955, 0.04154562904583425], [-0.2088786742145659, 0.004421675658556642, -0.004880208349919151, 0.04625907355790184, -0.0003017281378069521, -0.014530138514167483], [-0.21314356646987082, -0.010471415958316004, -0.006491577808021416, 0.07346568015474063, -0.005672198773220028, -0.01589454019293124], [-0.08488843719229885, -0.006687284582419568, -0.0036179360055342176, -0.06227460339067871, 0.000505025423260571, -0.023446764252329357], [-0.20698314166310144, -0.004007473443086411, 0.005067126790960291, 0.07396726240215981, -0.002473749520268541, -0.020063357899997213], [0.2367427376572526, 0.011138781991507868, 0.014698132063538177, -0.05606867005231229, 0.017647282241724373, 0.13691784720940017], [0.5650517322445892, -2.4696398876161402e-06, 0.014888960789568926, 0.04558972017926961, 0.0012386912425078843, 0.13246439082497638], [-0.2316966156282956, 0.03288356982744283, -0.004738251757920254, 0.07814613950506644, 0.0014618863447901358, -0.02625136032571597], [-0.21894243699301527, -0.007546707121709701, 0.0011351509693035308, 0.06408247164451741, -0.005231071816937485, 0.016692593317840728], [0.15867126685088517, -0.03233569746958438, -0.008105805769538467, -0.004322391129246026, -0.024651703946064592, -0.08100693837772126], [0.2753425474843408, 0.008952506321978335, -0.02132386190613547, -0.2677275120919907, -0.004375237834502641, -0.036839949910200946], [-0.11997018462825391, -0.007877381514178371, -0.00204353778134696, -0.02472633328645438, -0.00021422782206867932, -0.020980295752011432], [-0.06810022721826607, -0.002142748618676149, -0.0010941729090643314, -0.08567157449067307, -0.002751209602857958, -0.020650067160462544], [0.1887551604889839, -0.007637726694651364, 0.010352481399756905, 0.24035515357403736, 0.014513002941362308, 0.12154954733812944], [0.5488203783709352, 0.009357570512415554, 0.00862544354104003, 0.14665223973009023, 0.012968512210481137, -0.05289962055543922], [-0.11294557830074373, -0.004618197791887108, -0.0033388921025586157, -0.036654360722412724, -0.0006902518757911748, -0.01966271920660685], [0.2862884173636761, -0.010052745655982477, -0.014387202190604824, -0.2188259360511731, -0.020896560608677017, -0.1108633538096198], [0.06687655578997656, 0.02499804927595804, 0.021632261760650872, 0.3242609839984157, 0.010840682755238914, 0.12857273626102703], [0.013081419257924802, -0.013339131866245963, -0.006988562780472235, -0.13493863512652132, -0.008402343726157384, -0.020120364806147453], [-0.11754356187685683, -0.0066687363128441176, -0.0006727668497951786, -0.06660948013092717, -0.00010170590505583093, 0.012852917742146565], [0.28702441595114486, -0.028795747174454964, -0.015496437065108632, -0.27557972358084215, 0.0007178233148211925, -0.06803216294739285], [-0.10531505078610247, -0.0053049962044923, -0.0013692000222488927, -0.05046091119394598, -0.00033002299001100456, -0.017188642332610943], [-0.09138008825143525, -0.0035855898818745267, -0.003122434355999975, -0.06410300137490105, -0.0012024305898878395, -0.017016455545901123], [0.3512661016196626, 0.0036813727200929055, -0.00026850130110160874, 0.3133497290856951, 0.004086894202871059, 0.07107187727760524], [-0.12246832481854815, -0.0018859606980590816, -0.003642384187165206, -0.07839761360230953, -0.0016137631368247991, 0.03176471310957446], [-0.08994731425194179, -0.0016777775641681227, -0.0008794875414115192, -0.07155132634947071, 0.0035548249730298007, -0.01740891926603741], [-0.1569599583321924, -0.0005899922269204992, -0.001032375108400414, -0.02131063856020932, -0.0005487441047610913, 0.01372218452296032], [-0.11215136374992894, 0.0029419183906171497, -0.0003777655775333907, -0.020741612661325775, 0.00016553160557308165, -0.03969908895978291], [0.4274087931541535, 0.008113718202435932, 0.0015321019886257205, 0.359380350297359, 0.010083914256664377, -0.02555387789924167], [-0.23574256893879872, -0.007717250983278508, -0.001975535491470834, 0.09660500539003289, -0.0012112484517852206, -0.012982780609667432], [-0.05558725089044867, 0.009990240723974614, 0.020447301518118206, -0.1399812456358331, -0.0006650757428826159, 0.0353622205032622], [-0.12158843169751123, -0.007440905216459212, -0.0011073330844584295, -0.08047329320366227, -0.002164770988134118, 0.032364734190226355], [0.5439918597192143, -0.003236471364788723, 0.009263984937869812, 0.026157723313969965, 0.002771846078167804, 0.12636541628992654], [-0.10338568680369827, 0.0028475650487715227, 0.0005497804041955991, -0.05471387269134183, -0.0017785504194958502, -0.020179235538431296], [0.01952712294420603, 4.758668999825649e-05, -0.008833274356724239, -0.1914187998313572, -0.0042940495505647295, 0.02047808077110768], [-0.18523048220271573, 0.0021211091034403977, -0.0037407993594106305, -0.02331253708047307, -0.0005803523808471629, 0.04166639525333967], [-0.14840332778336046, -0.006829016225109171, 0.0021015741144301955, -0.009004571898434689, 0.005842998643530023, -0.02078432351772234], [-0.061249850423718555, 0.0003594147946122816, -0.0005552655935027692, -0.08464707609746144, 0.0021903576177028723, -0.027936151726203715], [-0.13047695939942058, -0.0011387544332334826, -0.0022796946412440232, -0.057742999331487584, -0.0008162049243689524, 0.019211279396422407], [-0.09763588641426825, 0.028637738368928575, -0.0009906709407802691, -0.1027069357624218, 0.0013982604916697818, 0.01472082759020604], [0.07921232498575029, -0.01582287680027904, -0.005756091044383266, -0.1691668521212886, -0.00925103458911995, 0.033190005759796554], [-0.21829673526780935, -0.009085710962836007, -0.0023005418443648685, 0.030148079349531714, -0.004121727332421843, 0.029079969391233113], [0.39548949539264755, -0.004763048010037212, -0.019249051621770864, -0.2552714331742287, -0.008660327868349488, -0.05899134900397528], [-0.09987028708134882, -0.007038135868478932, 0.00014141472626372188, -0.05421480368321886, -0.0035749069691782525, -0.01585328112403861], [0.6186576845139222, 0.0369868442338683, 0.004638754168162089, -0.04276924630643893, 0.0013162685241693973, 0.05380731391393389], [-0.12932290644254735, -0.007336329032189526, -0.003145461665885536, -0.024124217295126503, -0.0033519962471755334, -0.01312908931707525], [-0.1518641599027642, -0.00985623880298086, 0.0015574962644631274, -0.005184577507432584, -0.0004190385713567699, -0.012045442264242265], [0.42490496158414764, 0.041080210326388956, 0.0027403559522036283, 0.3088889828744252, 0.010032901085286991, -0.02685503087007555], [0.05340076405375534, -0.03490730418655787, -0.008407116210792674, 0.09753328032467991, -0.03129898599000173, -0.028600587486032624], [-0.23258303040708975, -0.0068166106876506255, 0.0004061543081175791, 0.04238881581423459, -0.0011839012677482927, 0.027950000811564582], [-0.12023480839367065, -0.006232013323227286, -9.604169996338989e-05, -0.03895830213293483, -0.001488085027724442, -0.012959572951890675], [0.3143592509911041, -0.017214955915059005, -0.025425543042494392, -0.21753638863843777, -0.00910679500173088, 0.017814034781219668], [0.37009438668740047, 0.02774574514301429, 0.00819022541222388, -0.08543841807541613, 0.005496820175863881, 0.16955329481190673], [0.2875542473407269, -0.017498007267761753, -0.0025035172169970135, -0.23532884572304535, 0.005157970328137623, -0.10304779984201326], [-0.16609230429797864, 0.0359003020720488, -0.012128258931370645, -0.019758175149432983, 0.0002552436832011789, 0.03647709611134922], [-0.09218188270987235, -0.008003567467399367, -0.0008172250246378057, -0.0672040937421609, 0.004322439365998866, -0.016525670421928672], [0.5520108132410009, 0.006558253640006845, 0.009587839143437828, 0.21797623530548793, 0.014441772904936302, -0.011547239164843548], [-0.11514204265177598, -0.007187555939391811, -0.0022239506063715633, -0.038794292704407186, -0.003190546194480997, -0.01343043543298348], [-0.21584943244618504, -0.00903763375661128, -0.003957939419127759, 0.06567407476801021, -0.0016755433767497046, -0.014573095147327145], [0.02858784942503975, -0.0003849218029483913, -0.0029562182646197107, -0.16774639089209809, -0.002409547205793522, -0.02166743792624715], [-0.11263948474479316, 0.02594004180007468, 0.0002533288700260322, -0.0687892020916984, -0.0007580155769637287, -0.010437501589978413], [-0.10332607897864657, -0.0022606062326719187, -0.0018244964035386208, -0.04206868060282113, -0.0008759130064350457, -0.024220891442553166], [0.216102887820351, -0.01613185809766418, 0.023989143776302402, -0.1655327138399412, -0.021700888835421147, -0.1025294279664842], [0.5748996805464255, -0.005915992596311875, -0.002101193369625466, 0.16219628746770973, 0.005213404913411703, 0.06129781303838783], [-0.20707276864796842, -0.011255764365073763, 5.745155224006691e-05, 0.07085444977489165, -0.0037825332268977734, -0.02367512080147762], [-0.2328475236713367, -0.009893492852513713, 0.027343482018129243, 0.13020995912157188, -0.0039543232483209385, -0.014416910891340092], [0.009740217384442883, -0.012367150680527673, -0.005068843953168137, -0.12963141977041498, 0.0024131472229692225, -0.03371023591758754], [0.0699941254312943, -0.012116318776301442, -0.004537825089055588, -0.1958976526360742, -0.0034743149151430426, -0.027592299729006124], [-0.14747296442205887, -0.0038447371108428986, 0.00012171074547103493, -0.048848802262488576, 0.0010669511686167078, 0.02841109584955684], [0.5065966780837703, 0.0016812196744771305, 0.017643865303987004, 0.11030969108926289, 0.02294448008627206, -0.020704345238533924], [0.5159558438121534, 0.037703190207047196, 0.0064676787182311605, 0.21733193695779593, 0.006007344385508089, 0.023275966703574645], [-0.1695076567793751, -0.007673367374359182, -0.0020619007972226696, 0.0010697160275235757, -0.0032824957371478604, 0.002712371327248373], [0.4789708216337735, 0.0030419847180658244, 0.038344071971713994, -0.05709739028035898, 0.02226114196716566, 0.13407532237058853], [0.5337897461990483, 0.0019200533326927437, 0.004174839665509043, 0.25337278052508877, 0.008777810955275375, -0.0028618973442800233], [0.09395662862125102, 0.02291505634007827, -0.011970897562662147, -0.2404850872084303, -0.012575365837217727, 0.061807205329520186], [-0.0008184176207334866, -0.01985505220493093, -0.005822459026869567, -0.1431606110386931, -0.013055577704743037, 0.07413941918327303], [-0.1083266022235318, -0.006006424609228466, -0.0027542761090727096, -0.06463708311262656, -0.0011681264665126684, 0.0024825125209721435], [0.02161810978337434, 0.049891573547773814, -0.003926339461521447, -0.06373870563127991, -0.00043191606164555164, -0.08439415074812967], [-0.244005923011145, -0.012313151758594054, 0.004979297274702805, 0.10410927865317608, -0.0012038755656501073, -0.012637241754105822], [-0.11719929107129906, -0.008127479426156856, 0.0002838350044404039, -0.03562329369713471, -0.0013670143529577682, -0.013278717241205397], [0.11834274159778826, -0.03304310802046327, -0.01856172584572374, -0.12385837944578042, -0.0051046648674901675, 0.048047898917373384], [-0.10281174708434297, -0.006941161978901003, -0.00041984856533837503, -0.07254204749861079, -0.0014016264713154509, 0.003706431598508532], [-0.11455725876374982, -0.0069268340436932085, 0.005277883087842334, -0.037390873824975586, -0.003627468942066131, -0.019256876084785983], [-0.2642133281415631, -0.010892105336015987, -0.006513356122425032, 0.07787174161585314, -0.006060597274921048, 0.032730978592404474], [0.5295056516441261, 0.06029769449827629, 0.011991196786962846, -0.013977523618752343, 0.012140513796804421, 0.06517016530528029], [-0.01489862303153452, -0.013014651099055675, -0.0005530821051108082, -0.14850608753700667, -0.0015809311381148306, 0.058161698907087535], [-0.22424508156280235, -0.004790924380477332, 0.00022316554462189592, 0.08572424599181397, -0.002235381106645906, -0.01420507210555853], [-0.12994192862835852, -0.0011028917686613884, -0.001746629375868642, -0.036584571254534494, 0.0007544580833520043, -0.004705103722595027], [-0.15850069771737949, -0.01433542020805714, -0.010615783966471039, 0.12058704515520356, -0.00646149888565679, -0.04064435866335359], [-0.03299958956523542, -0.008738242872620804, 0.0012931452889227224, -0.11142423305338046, -0.0035639404059042293, -0.016119996534639178], [-0.2277650114793052, 0.01572593904273354, 0.000442280933458998, 0.06641938740138556, -0.004991016981568353, -0.014574912250038529], [-0.23895424491113162, -0.010845541038901627, -0.0044132162639042676, 0.1036394436284367, -0.009172272314130552, -0.01553738656917447], [-0.20715187257278664, -0.006881397611452178, 0.005002713448582392, 0.07208521880865382, 0.0006359260556250124, -0.004578762731797198], [0.018030588667301697, 0.03269978474662909, -0.005315192133785497, -0.15484166816914008, 0.001385607957132713, -0.04646435916337631], [0.24030365218946573, -0.026069060445813693, -0.01723220342606209, -0.2073735344358689, -0.009161107853395217, -0.12188965079022927], [-0.11775537086497916, -0.006509269265468034, 0.003694433870006722, -0.03705790006130575, -0.00067121468658473, -0.017110678991668855], [-0.2102345725737213, 0.01508643358577166, -0.003773191886310807, 0.06728361467990626, -0.004212450371790812, -0.03679667553911787], [-0.09328720085759055, -0.004989416862798086, -0.002340698423634399, -0.072797523361161, 0.000278610150392227, -0.0035237706452070187], [-0.1116176130996909, 0.0015468617701416974, -0.0024428610257314584, -0.06812429328984428, 0.06687948282577039, 0.008518877364810147], [-0.09091975813880025, -0.001848188297464938, -0.003366187588497186, -0.06492781919105167, -0.0017458589421039856, -0.017602187842082048], [-0.10688228824227006, -0.002727642351880837, -0.004747563064611156, -0.05891690054203986, -0.00031221339004936367, -0.0028650590758144325], [0.3012106589716509, 0.024055664522827452, -0.010642127221281078, -0.2703754195708704, -0.0035478940764493007, 0.007207479567481581], [0.06228851496949048, 0.0034889772830435056, -0.010586417643640498, 0.22311095298357628, 0.0003538458089795665, -0.12203015911573478], [-0.14605085614111954, -0.0056405278967538485, -0.0038640081539908556, -0.054412933868568644, -0.0016807919021787987, 0.032380143603637446], [-0.24014163801805102, 0.0180165613421132, -0.0057889015560183615, 0.05839387917582734, -0.0037142834392478587, 0.03275771582870877], [-0.1615528619531548, -0.005972058880670242, -0.002194996889927546, -0.021861576101036607, -0.0035987674580672596, 0.01963137239396781], [-0.13079447940952305, -0.0040022943922047634, 0.03444838198313115, -0.025784710274835965, 2.1926815919055457e-05, -0.032007158055819725], [0.6210692428365164, 0.004483384918654647, 0.015541590029475466, -0.06172068121538272, 0.0077665032185642155, 0.10325749989470875], [-0.20908921958512067, -0.007485465894690525, -0.005697610648014604, 0.06853276360372544, -0.004539266633127968, -0.01364077022076177], [0.02540651545979213, 0.0074436970840922375, 0.013884019285188693, -0.06902479565434576, 0.004662364640376534, -0.05006156271986599], [0.13860989947662442, -0.014086424740557308, -0.0057353926398566075, -0.09305916712429206, -0.017790478055322148, -0.09015200834516755], [-0.14305154881557372, 0.014757083257691898, -0.004632503135602503, -0.05159115929427205, -0.0009661984135344444, 0.014360040687005152], [-0.10324605197814737, -0.005903233424318047, -0.0034461058623240356, -0.04486762400343856, -0.004210123242410362, -0.017070194822694576], [-0.18966967960898515, 0.025837467968117374, -0.003895003266451445, 0.042213552916225604, -0.0040404320579144806, -0.027147572617658794], [-0.1343336698075096, -0.009440148211148722, -0.0002859219226591866, -0.0054285033977834, 0.00020837159617311172, -0.028630128257071927], [-0.016763514974876683, -0.013299551702087988, 0.0012796792060180391, -0.1016208397725848, -0.0015412935794884566, -0.008237821847381793], [-0.13245099122270915, -0.007202739696262548, 0.003213359184512175, -0.061771182467147176, -0.0013048194809048765, 0.027532960984099165], [0.45497406867082335, -0.009021803970263639, 0.003832108934689686, 0.32791264914483587, -0.0028352426221293987, -0.06579360555478492], [0.5547241937227488, 0.004010920564023516, 0.024418656792681664, -0.035438435597650426, 0.010187034180283397, 0.15427493192521236], [0.21323309993131473, 0.04045066580048733, 0.03952085121794011, -0.017911461070245573, 0.018469969773157874, 0.17681857708904922], [-0.12125422039048672, -0.005770978180294465, -0.002821481176918316, -0.03238335381735267, -0.002279554738963838, -0.015900411695983968], [0.23345586325455614, 0.02026746261484243, 0.01885774518152922, 0.39242523479630226, 0.005146621306896732, -0.004784081000283058], [0.5800568996240637, -0.0005329768477826228, 0.025618215894012492, -0.02720726514859175, 0.019011998047599343, -0.025366792204224595], [-0.07062103586099747, -0.00616386034249839, -0.001680853242635645, -0.07454927358773596, 0.007363931504019668, -0.026758908470152182], [0.0027018189632262205, -0.01725452040293378, -0.0068892192500607085, -0.05659280757298482, -0.0052543882132735295, -0.0665256454287351], [-0.11144180258525604, -0.002340807539684309, 0.016420775361166987, -0.058858346568737716, 0.007939046391747074, -0.01591457934495039], [-0.13188262474643497, -0.0026768889151846486, 0.002285051082657707, -0.0179287606392336, -0.003485237430944041, -0.0200548726841934], [0.21783282582125904, 0.015064912789855934, 0.013947510548217298, 0.41548627120718024, 0.003916605259144537, 0.0011075319194092028], [-0.18034028181298353, 0.007549021202551094, -0.0028994679913918265, 0.0007998801543561712, -0.005279357942261365, 0.014987479117002428], [-0.13834751687905616, -0.00872204718810456, -0.003599185502612539, 0.0033572899658684996, -0.004227828864372817, -0.028037378198388106], [-0.08549238529041307, -0.004700669534724904, -0.004138250130255131, -0.05973849861573359, -0.0008496270801692082, -0.025490569348703838], [0.03550355502407895, 0.052479402241846294, 0.029811283638574125, 0.36822802772065133, 0.003974948994738556, 0.11425825857058293], [0.4336890180316187, 0.0123280032440546, 0.009101861766631003, 0.33702101846821336, 0.0033287624214642804, -0.026073108376427685], [0.5714935501363403, -0.004024809393055212, 0.008450411892722413, 0.20807199747688074, 0.013887695620834223, -0.06311027430515244], [0.47102864604876954, 0.004054677691803506, 0.0029940775768169544, 0.20765094908631102, 0.005009924992258631, 0.07007511395977965], [-0.1129119850055315, -0.0032241125342504285, 0.002906466930564282, -0.04431777537845232, -0.0022530838634196816, -0.018466653006052915], [0.5302284108635174, 0.010020529751130281, 0.002658370784557122, 0.12356894798334099, 0.00509344488749734, -0.07672375188909275], [-0.1330351392553863, -0.008355196707160386, -0.002038530926536873, -0.06501689432566438, -0.004221288354542382, 0.03336816068040232], [-0.09963631859372339, -0.005367560015483903, -0.0006828145811414944, -0.06816908080367304, -0.0033208757641810496, -0.003025016908462727], [0.023961195443280743, -0.0026008942414318397, -0.0013837156962831455, -0.18065960837525288, -0.006010977471097909, -0.002382666325881791], [-0.09200616479524534, -0.01110805347400279, 0.009406460575112597, -0.01800635806199085, -0.0011712097352311545, -0.04110800784197555], [0.28755731477201496, -0.01822296332644305, -0.021050423400321937, -0.2892316961658478, -0.007927425439031462, 0.01975090784534196], [-0.10522274917654591, -0.0031329506699995934, -0.0033085289117847196, -0.049654155208245336, -0.0030033222668268654, -0.01608829376659708], [-0.18179461872446895, 0.01967096881328562, -0.0076603594521255265, -0.02802783933578817, 0.008092142823087605, 0.027450731517035337], [0.5309226487269056, -0.007948460194108838, 0.0031185332278899545, 0.18905063582585238, -0.0037952266240021825, 0.07856329760889033], [0.24483610050370305, -0.025383988957178697, -0.015790160683851417, -0.20294023228790686, -0.0272559639704132, -0.1151733736519726], [0.45839608824296096, 0.009810786902941916, 0.009533112123701335, 0.30316910085558413, 0.007525868128751133, -0.04564455942854465], [-0.20918490495501815, 0.0007638806714668498, -0.0030386303121608775, 0.07500149201357706, -0.00488625151259655, -0.026065585905268404], [-0.1094703503520188, 0.017268605460253886, -0.0007913372886480072, -0.06918310612885073, -0.0009766739813931767, -0.01642380437600965], [-0.2379576780655024, 0.010993726984192997, -0.0019204735717738145, 0.08287674573941774, 0.0002758272746711138, -0.015309100741957554], [-0.12171023243884145, -0.002150597864820276, -0.0040068131070348425, -0.03931497773866256, -0.0009634541038083374, -0.012263924746832399], [0.34611430898397677, -0.0017895205563105954, 0.0062183529686163285, -0.14865966210294415, -0.025638488745871362, -0.09273832388080094], [-0.13035621783046175, -0.0054118021493122104, -0.0006497315445061886, -0.03736084077749855, 0.00020115984754764073, -0.005165900879101213], [-0.12592926043866695, -0.0034585664454899415, -0.0031561031006659395, -0.025220040043416322, -0.0013726768286296865, -0.01960668647646426], [-0.04635714761122618, -0.008157592805658337, 0.001930277967261479, -0.0981818719003204, 0.0032074739834072954, -0.01931472506763794], [-0.08910056904476803, -0.007170191247027524, -0.0039308237059384596, -0.0652229917534538, -0.00018785612571220415, -0.01479756812309993], [-0.07257551463542798, 0.009947704609575241, -0.0032942397190742646, -0.08953812017305417, -0.0022227765653578917, -0.021060386849994377], [-0.11759397601736658, -0.007476112848756169, -0.0019234450535773856, -0.02466371984081852, 0.002038530351791969, -0.026064385834970567], [-0.12318027538435472, -0.0017157521796449385, -0.002017355794762654, -0.034979070653315715, -0.0014558679051828271, -0.017061678082738983], [-0.20291548982555024, 0.0051703012773059815, -0.000832110754144585, 0.04583193381831208, 6.75836113257559e-05, -0.01117161206664325], [-0.17711114424844648, -0.005661072823275527, -0.0037755924043249814, -0.022843868544562062, -0.005594242358776706, 0.03685369815716354], [0.1678578432196314, 0.029311936016293293, 0.023211185577315795, 0.010836086757000388, 0.0258690479723979, 0.17706306351652437], [-0.24218636263503196, 0.0033219769667072103, -0.001571157985616229, 0.03949421020299699, -0.004830223034718469, 0.02952822315232827], [0.3206155834901342, 0.036245847791505824, 0.0033831734400132487, 0.3084347430335561, 0.03645573113598201, -0.018728610637225303], [0.3681353331470674, -0.003225925206451499, 0.001871945070841548, 0.32921080475799847, 0.011294393103584644, 0.09635925606511603], [-0.22156913051378915, -0.014676703089523683, -0.006491674742206403, 0.09821570701554881, -0.009052524660614614, 0.031247659323919007], [0.3596150209693672, -0.007111021216229765, 0.005970029213438464, 0.3324717974488397, 0.007625296040735238, 0.03202706543438791], [0.2464077137242261, 0.016544667372624052, 0.020071739139194637, 0.39644184722382325, 0.016950809632223736, -0.0006768991922184585], [-0.1624424993294172, -0.004371046197833678, -0.0019098568390454337, 0.0021460577605069853, 0.0010554982785736541, -0.006120037730754588], [-0.17969410707615882, 0.022763413868549236, -0.0023324266210225726, 0.006689940757548685, -0.0015032263895955558, -0.01385343580916252], [-0.24890299022411322, -0.01745134970321216, -0.0053103731853905165, 0.14263601990819913, -0.005988577359373789, -0.018351062769442372], [-0.13073584761298152, -0.0034550952440270313, -0.0006755364534000053, -0.07050756294958639, -0.0009075320580283074, 0.031982685429135384], [0.3473636325711234, 0.002465186421192613, -0.01476322097053893, -0.11370122934186801, -0.010163746714701638, -0.11199142172100786], [0.33873118332273117, -0.009582526106592479, 0.0028688844970212435, 0.3514650000906139, 0.001347833241877485, -0.012037742710668639], [-0.22350198458157802, -0.00961337992016376, -0.003974265066618913, 0.0821457056959469, -0.002745859990441942, 0.019844069577142017], [-0.25986361216457526, 0.04088834804529101, -0.01599714633431449, 0.15138257102669064, -0.012233893739423476, 0.015598256975854428], [-0.13506236455482154, -0.006163215281188384, -0.0012967285206106263, -0.014337321258888659, 0.0011696329549643872, -0.01918078765318055], [-0.1138823849418246, -0.002131777432208412, -0.0020449146665919337, -0.04691638253223232, -0.0011502258451313238, -0.014186275366324948], [-0.1560487813327911, -0.004227389413335248, 0.006936428569784206, -0.03628004007442628, -0.00021103195915781465, 0.021962480876592732], [0.5057307281963374, 0.0008175763097818784, 0.012333572496941593, 0.11733245134223358, 0.014128767045657116, -0.03391976205762037], [-0.2520326369210955, -0.00923238692994292, -0.0064940021317903985, 0.10669951548605043, -0.008334728767822942, 0.013150905931267998], [-0.2205169370576335, 0.00032612046739736495, -0.004860281468233768, 0.09736579711550437, 0.002238752806597938, -0.03663011853029884], [0.09353111960420203, -0.013293138633129918, 0.014766684756771922, -0.12892387792607488, -0.008498623855124619, -0.06001627872075871], [0.07277826063708033, -0.035724173818645164, -0.004002865400151044, -0.00428636641431143, -0.011578026033480585, 0.06322658372792095], [-0.15546305498542615, -0.0060663523859317905, 0.0018941031881011742, -0.049823373848790375, 0.0006646490474326414, 0.03412212422270966], [-0.10186945271308484, -0.007272627325710503, -0.0010831551785251733, -0.046774171148545735, -0.0023404043183953417, -0.021070189315738237], [0.6002659372727448, 0.004562299567559348, 0.006886759765002654, 0.19807826403249335, 0.011893447978555865, -0.01259670861635816], [0.2593154241039087, -0.0077579533446933465, 0.016240082481297838, 0.3886868245082104, 0.0024714795549478693, -0.009019667545719761], [-0.1407391971079106, -0.0034954133052777013, -0.003363171120808457, -0.05449826666758177, -0.0018737864748293873, 0.02636752698410014], [-0.09747178736625896, -0.0018443783969965153, -0.003236459752975646, -0.040142051188950235, -0.005257050865541861, -0.023493986714990648], [-0.07589475175510435, -0.005840840571836644, 0.004706993143468526, -0.06159833414367179, -0.004072966999857514, -0.005483909196807585], [-0.23330781944051887, -0.007449424912579713, 0.03326795461309398, 0.06471162064606982, -0.0005126037777629903, 0.007654082395508017], [-0.12193576255671901, 0.002041198413980307, -1.037817938368198e-05, -0.05158534183680208, 0.0016412329032018, -0.01056094874427721], [-0.04120564538373231, -0.0016048432655739909, -0.005100460681776434, -0.1359788667467312, -0.0030344300095807766, 0.0065142460873949225], [0.14289220007030626, -0.03441716810672717, -0.014561277120698437, 0.1926808697640648, 0.00298693302224053, -0.0950451290577574], [-0.15838966224309262, -0.003331679602778838, -0.0041090634315203, -0.041548421776288626, -0.0025934269000217405, 0.0301733650648132], [-0.11803386506115394, -0.007134668978983919, 0.005677891810560411, -0.02996648947955297, -0.0007939342209515096, -0.027560894854231528], [-0.21518145395013522, 0.015292491047761718, -0.0025994201883666346, 0.009905917542296269, -0.0016823167754209793, 0.021158612551607222], [-0.16199974145545804, -0.008302036892058742, -0.0037156464874421175, 0.009225963471418165, -0.006242417241365149, -0.006203243548302675], [-0.15011157614031195, -0.0045532796512488425, -0.0017490470964402762, -0.0027527569281671506, -0.0046289963717732535, -0.007012894536695298], [-0.10654612720227488, 0.017062583864565415, -0.002672776533910834, -0.07446510853185284, -0.0015276908292983266, -0.00715671410056184], [-0.14234253184851858, -0.0067520948927665084, -0.0031982414612700604, 0.035061009299567646, 0.0021387114304223477, -0.03148669379727585], [0.12146964397698218, -0.010759977899846266, -0.0037882279667551624, -0.24567498415544028, -0.0014168520458862459, 0.030565953646501658], [-0.2148812262494647, -0.007635683405443832, -0.0012039236178627167, 0.0690408395433707, 0.0017663155842599176, -0.013662988521526196], [-0.049129378638290234, -0.015340118366884037, -0.012720491745278945, 0.15530381518965333, -0.002187535341576458, -0.07577081490714782], [-0.09544590159793273, -0.0027873914162610137, -0.0006402466521609955, -0.06412439004000965, -0.0020237765515017567, -0.014947117271545485], [-0.15989790811057716, -0.0056312959038638324, 0.002852495735936755, 0.004140036387410631, -0.0011559891464272701, -0.017820513565653402], [-0.22011354876439815, -0.010621889039994722, -0.004470471322224509, 0.0768496430041915, 0.0037703182465267738, -0.01182405212410091], [0.1823961255416458, -0.02508734060762505, -0.021814752205922636, -0.18323690839222467, -0.0005399554798617781, -0.08794579630699247], [0.2418911702201452, 0.02338102818184747, 0.011678629299185932, 0.255845483737375, 0.008680213467904053, 0.10623475822217732], [-0.038077698351014995, -0.006197539967997926, -0.0012639040851786154, -0.11361907684167935, 0.002058558462517422, -0.01831033921664641], [-0.19435481880156927, -0.010544225778006807, -0.001837094552505417, 0.07708937373729675, 0.0034731635254607427, -0.04039116003543759], [-0.17211199910776437, -0.009630873155818847, -0.004970622096056567, -0.019529399317984, -0.0018579288416924644, 0.035498514827008706], [0.23580384219450662, 0.02544986292016426, 0.05419549346476128, -0.0711603225259458, 0.015937515057227073, 0.15995686285754096], [0.5389856092883284, -0.004029972338504248, 0.008737539677130231, 0.17639712686639658, 0.002996758879644116, 0.08415489841131989], [-0.23903173515661788, -0.013676690431775235, -0.00886184055823191, 0.1175934769770972, -0.00558697613400172, -0.014035628635864293], [-0.18428789639727877, -0.007091383577908649, -0.0004824751137794307, 0.030820797075002914, -0.002372370361062766, -0.013425243053544969], [0.5225969002432139, -0.0019032452454736597, 0.013652928949752615, 0.2704484144653609, -0.006774572933970788, -0.003652647701106317], [0.029591518250950646, -0.010665814034970687, 0.011225584551251212, -0.15598614759569346, 0.0030242244977007255, -0.03300412757400075], [-0.20222014155345214, -0.009655081792704126, -0.002043175090314072, 0.060627727279041047, 0.006958053121378651, -0.00999334835050421], [-0.11854541280425948, -0.00662712376454074, -0.0012372595820993406, -0.03270101333440964, -0.001839789706687129, -0.019018224337414937], [-0.09928846656809136, -0.004531253340712534, -0.0024542159394947394, -0.060795187957402616, 0.0007642385648243928, -0.01160511475912338], [-0.11959853464705586, -2.08030051784859e-05, -0.0038287104784908584, -0.0806134818972468, -0.0020201664993753695, 0.02588002986068184], [-0.223560639450062, 0.02192396411255271, -0.00409379817091825, 0.033678790777925745, -0.004746115827947369, 0.027487798558448368], [0.4909931927474186, -0.0033227802918767042, 0.0009662466607288007, 0.22682588138855775, -0.004871249071003731, 0.07974870856617353], [-0.20771562498359974, -0.010294995894751982, -0.005854033428487451, 0.11060503622576806, 0.010354429774495681, 0.02141185497324231], [0.43187737413646604, -0.007000579733644924, 0.00713780969539734, 0.33311122605022075, 0.007961970071102702, -0.019483911330656474], [-0.0019866361835115106, -0.01331563104403808, -0.006904539050600943, -0.1680296904069385, -0.005486836659289336, 0.02317704806103394], [-0.23266791073269544, -0.009013118597470693, -0.003620744566566318, 0.0917218973459973, -0.0001284466223921624, -0.01353501016020585], [-0.10716106813259403, -0.006463281452463552, -0.00852294567248221, 0.14115884144441818, 0.001332693761884132, -0.09285245423447651], [-0.10227011369704075, -6.196101160012094e-05, -0.004466734767923439, -0.09219112819302924, -0.00471699002467641, 0.038187953335295635], [-0.13343175405082142, 0.012367855096231846, -0.0028463623247894, -0.02510552842670853, -0.0014348639870446112, -0.02729267964020095], [-0.09763505217055402, -0.004501803494203613, 0.002110921374139344, -0.05434375727197172, -0.0019523538496925654, -0.024087954587717262], [0.49378890490294153, -0.001316429907576043, 0.00651074761330525, 0.2904010657928758, 0.005397451506815936, -0.036989358955983834], [-0.11457012561158013, -0.006535983175272284, -0.0007026403783949159, -0.07837769320047545, -0.0037062000624099437, 0.027232642428133762], [-0.18541776306411786, -0.010746660256704624, -2.6899837621428816e-05, 0.04827673694837013, 0.04287530368526427, -0.025620717475190024], [-0.09621645350833481, -0.001824669336234493, -0.0028106070099012683, -0.0618409334470889, -0.0015672022048619457, -0.01615013449357845], [-0.1457075847641419, 0.0055645276137710415, 0.004393124574031559, -0.06959600633458209, -0.0018326611497262528, 0.034268600060648476], [0.2277582601348435, -0.01936983061957027, -0.012348699515027498, -0.25827368808974444, -0.012214336639278688, -0.07381884812836571], [-0.08956599473160076, -0.006258066588354088, -0.0039605063939252205, -0.05480846605395416, -0.0007258719173211447, -0.024551878628569924], [0.1026571529262413, 0.01702747033938583, -0.004089449314148124, -0.21573174760579072, 0.002523682774905291, -0.020867850878835543], [-0.13225329286807228, 0.0032002382898663386, -0.0004425634843162489, -0.027249160862283343, -0.0020172533735573414, -0.016647967701636863], [-0.20356730716332624, -0.009996330739631656, 0.009244505838037539, 0.17246972099927574, -0.02651250586049543, 0.004309853434075802], [-0.11916555688717488, -0.00823480491482578, -0.0008519787751210548, -0.014633080743658248, -0.0038570642886274183, -0.03033418105725879], [-0.2272112334779672, -0.015114674347850163, 0.0005179655807697741, 0.12639386802217742, 0.0044709471259196595, -0.03625853956971632], [0.6012933093018188, 0.013292368524546848, -0.005566260658729923, -0.1008275202090378, 0.015643364700152436, 0.1308858433473521], [0.03579968900993904, 0.014172768751965296, -0.0011683535446522458, -0.17348721078243393, -0.0035909242215820566, -0.015580413657680906], [-0.09178816974953022, -0.005779268481629313, -0.0028694352971932145, -0.06182374194314161, -0.001607839751846035, -0.01654154477665936], [0.6201021989252329, 0.04167234562682976, 0.0033672965254740324, -0.0003820485677211162, 0.015429822235020057, -0.026921043316268066], [-0.24773142737609302, 0.04061571356661791, -0.0025965520166708568, 0.17691472279685244, 0.020146082600415997, 0.0031363016987183256], [0.05440032160037889, -0.008443803458466861, -0.008291554023957181, -0.16674401908328002, -0.0031303944706861323, -0.007200550563988681], [-0.16339200027352677, 0.0020939208472127865, 0.0028394417960690174, 0.0406756732886752, -0.010315901296669304, 0.034026960876334195], [-0.15518767720263096, 0.00042286920621022547, 0.0052242096322942235, -0.05270894585441139, -0.003151102285359126, 0.03099064650389693], [-0.11591589627748984, -0.006955206379117332, -0.0034466234071941475, -0.036084486084971695, -0.0016766504711198144, -0.015889960909518903], [-0.1278270392574699, 0.0028623643545573844, -0.002185897777703701, -0.07267314414334926, -0.0021532446020989192, 0.021566961426064724], [0.5432250723765835, 0.003706094084366038, 0.0002582208212343646, 0.1834540446787063, 0.002017781125846366, 0.06754503341186278], [-0.1697508697447018, -0.009673418285830775, -0.007235788412039107, -0.02191212027923589, -0.004445009642857798, 0.03374823200569059], [-0.2089111147514314, 0.022770837575880214, -0.0002149765120467399, 0.06916458329944915, -0.009166608206177963, -0.039694325415698645], [-0.13379564904872654, -0.004391462954852098, -2.4921492902878856e-05, -0.013267387415168625, -0.0055432636648425565, -0.023387315423507048], [-0.1253771277782639, -0.005514642286317396, -0.0035668427042644434, -0.07303170534376864, -0.00209312905068149, 0.030840113829963153], [-0.2078521276484539, -0.00994777684456961, 0.0025473084228685174, 0.09052971632556019, -0.0003746661238237493, -0.018799296236844076], [-0.11943892276042863, -0.0022210094553233176, -0.0035250109131636834, -0.07307223909363458, -0.0005057855616132707, 0.02210296778416388], [-0.08875624245119838, -0.006775402715282949, -0.00013601850389169734, -0.06800854921631727, -0.002674581741184288, -0.014059205372125318], [-0.10496793489206063, 0.023410943192382343, -0.00252203822296209, -0.06250554873797423, -0.005506781425738742, -0.027318639913646147], [-0.09623546529007856, -0.0050771644104487396, -0.0019070623784981184, -0.06251457354411487, 0.0006164407627479828, -0.010792175139607691], [-0.22993290529861637, -0.007993131685282909, -0.00422882387952196, 0.08923078241935002, -0.006266991987297597, -0.01152448512418634], [-0.12035516982251047, -0.006207719615437454, -0.0029109179900398507, -0.03397667238174953, -0.0010032245540586144, -0.01585825642051777], [-0.2598396930636149, -0.013351749147980456, -0.004623269271779514, 0.08152095425175096, -0.002065850314123052, 0.02919960754574647], [0.4317661960943452, 0.0014424452650189496, 0.03993248630433395, -0.07997264276532447, 0.0004886743554883564, 0.19172069178398649], [-0.15873430881687076, -0.007585017020625215, -0.0029480575316724383, 0.005671320354066581, 0.0014493750422373442, -0.015366486630310157], [0.333174886952686, 0.06912841674150447, 0.010700134086669042, 0.14814948294492208, 0.018020213960295642, 0.04919662721868334], [0.3174611390531783, 0.02259149339191109, 0.009265312122028425, 0.29772897423012074, 0.027224534039841545, 0.006158170259121612], [-0.012894909874976976, -0.003295362012611065, -0.008066155928351328, -0.1712250944218854, -0.00047102288382667225, 0.0555246879787943], [0.48625174011479183, 0.0022523064182059416, 0.039148242837143334, -0.0569938068034405, 0.015623165322265249, -0.02694164788896684], [0.279028425165089, 0.016089529165699907, -0.023717546450822094, -0.1313035257539799, -0.012598116554085848, 0.061515837602701996], [-0.12565619723575547, 0.00018889374308197603, -0.003543344850208969, -0.07265573638318272, -0.0029674649547124796, 0.0303349607918895], [0.10439852423663955, -0.013357238870882504, 0.0013500044407439677, -0.11478917730685466, -0.004720268657402442, 0.038214108538708984], [-0.11416046785476373, -0.005452231567205839, -0.0030651791655198646, -0.04393497020538256, -0.00013779332124069687, -0.013659357885887168], [-0.17241877458145022, -0.009083480014879425, 0.0014480215446347011, 0.03616512936957846, -0.0011347129501746082, -0.028886183367708592], [-0.13325958081188788, -0.007290215506670373, -0.0011175132521634322, -0.016859089564342577, -0.005203910863776646, -0.01596540428687334], [-0.23474408520488751, 0.014957035064206783, -0.003515513289186594, 0.08283081714651794, -0.012843552621872516, -0.016928034428110972], [0.3862484293226697, 0.0009403923860277788, 0.04015470456798083, 0.023849637330504343, 0.02956314085880184, -0.017676405476085944], [-0.10716115152174391, -0.004920034270978771, 0.001197073037934492, -0.045256850532135476, -0.0010812139042363478, -0.02318782280883989], [0.48304692440719793, -0.0013434981797194989, 0.0011355296967579939, 0.049959434005315516, 0.0215701401412193, -0.030108075525318127], [0.35261829680637574, 0.0352486571651637, 0.014192083916767623, 0.3261215518507336, 0.009815449089687338, -0.0025793083696478013], [-0.0966024881708982, -0.0006698154815899115, 0.0008517336648895778, -0.07053773653408887, 0.0014414825275238243, -0.014893176005836433], [0.23108669339158455, -0.015685613796736227, -0.019361166938345577, -0.20573188746141463, 0.003581270918389932, -0.0062437405579226736], [0.4289731156368021, -0.005652660403734889, -0.026002347127188123, -0.19142795708458993, -0.005589775825908262, -0.10663299424299862], [-0.160079440080434, -0.006967634566838164, -0.0008511693433626352, 0.01099710466825337, -0.001520368365541786, -0.019488492312077037], [0.5571267474420232, 0.005136786495487907, 0.007746682233992504, 0.14649049811013667, 0.007187866622820445, 0.051993856070327193], [-0.16783979270158098, -0.0032160946682227116, -0.0027692942585692553, -0.018133417289904967, -0.005108570167879828, 0.017157169086158258]], 1: [[-0.31416184572129374, 0.0005051400307708703, -0.014569731344532535, -0.07513733802123888, -0.009289355530027616, -0.08639305988986838], [0.15258579016726545, -0.03794501657558696, -0.02368503815381527, -0.15225046915213064, -0.04038705622723023, 0.04937369470340326], [0.2992940469897987, 0.014772640073617003, 0.0034543192052278546, 0.13545691294567103, 0.006254903711857323, -0.027860254240306585], [0.12587094053719816, 0.012587072436986882, -0.023396875420667148, -0.347294993681043, -0.006264804544424924, -0.101404672661382], [-0.5169888685559214, -0.010917069449249346, -0.0047216590055478656, -0.005380666040954293, 0.0020341164399331098, -0.022031750824160288], [-0.2986904593639313, 0.021598630666031356, 0.016426149263474802, 0.36611957968315495, 0.020237488583373195, -0.0026551858245838712], [0.3413377187207119, -0.005640274757654302, 0.0018393538048075402, 0.024093305709181674, -0.0042571482509193735, 0.0618705368373637], [-0.3205799356890624, -0.016401511624054087, 0.0008963247264232754, -0.128420902209024, -0.001139651572838489, -0.05597194267906369], [0.03468625347446552, 0.050072847147321325, 0.013947176235455518, 0.2483022671275304, 0.016766381538178724, -0.03014045207518108], [0.3737319830622036, -0.007180510558181077, 0.001822040707902754, 0.029470136208398973, -0.00028315403900903475, 0.03278617128535017], [0.3806413033604438, -0.009230584510920679, 0.006069101382733769, 0.06691613435562895, -0.0011213382916957645, -0.022347536332131424], [-0.24922991819216062, 0.004419510568534354, -0.008600004389577318, -0.06714870923852513, -0.021680913746589932, -0.1148805040852393], [-0.3541250158910438, 0.005111146932219845, -0.009888050126907556, -0.1604456345272149, -0.009392267884762766, -0.027592083264195787], [0.2665200166939038, -0.012940829152487081, 0.0008138318117491945, 0.17400675449387065, 0.0045183152252304985, -0.01958332716750528], [0.2618816107748601, 0.006467704505932321, -0.0008485719895677303, 0.12815051242067402, 0.001842737195259642, 0.02882197200512239], [0.131952998418941, 0.010605730537558483, 0.013713212173189188, 0.1645119922759148, -0.009432856817174028, 0.07213844722109501], [0.22247709857930076, -0.02769745476893475, -0.09303558820161016, -0.35552098261040205, -0.008806122003022554, 0.029792810909430974], [0.3155449979505117, 0.015240903479718685, -0.002413016781559319, 0.11352030336999223, 0.004506309135613418, -0.022579560047359063], [0.23662217018700135, -0.01030983647402655, 0.005690054510114816, 0.050416737285117946, -0.002399570693826185, 0.13024377851895183], [0.48808645705466625, -0.013197405091305875, 0.010098280777327709, -0.07772060330374862, 0.011749055694577638, -0.010835785131519915], [-0.48718455428403, -0.010089840165786734, -0.007938489932661203, -0.007471767019642736, -0.003563236632931071, -0.028932174858031394], [0.3612708683348043, -0.004639433085066803, 0.0014893915682103935, 0.014010823178143719, 0.0028063773857276335, 0.05346419484040059], [0.3311183401361483, -0.0031431796534967453, 0.004569429839184017, 0.05621595533831019, -0.006840625150080599, 0.04801007948993202], [-0.20219612465996983, -0.053056287331141545, -0.012557383176872905, -0.05549626480649409, -0.0018782125053242646, -0.1503855471450168], [-0.42867248290098636, -0.015726986043651775, -0.0009877313628026728, -0.15418239607461554, 0.001281077023405564, 0.05520066221579043], [-0.15542216177791862, -0.035786103266978724, 0.02347992007125444, -0.14857150793636353, -0.00401294052968591, -0.1430131589412608], [0.24645040670239424, -0.007699044615265624, 0.014079945693004327, -0.07175585452475045, 0.005859929571889557, 0.13208985526796493], [0.09731366553689329, -0.027501913554021193, 0.0029111527988370483, -0.4187730608616845, -0.010215305434468835, -0.11033231626333287], [0.4165434035090571, 0.017983140437826586, 0.005301122497368392, -0.1299069751311316, 0.002430636287898626, -0.027587002204195663], [-0.543391002309231, 0.008754012815881904, -0.00620780921078341, -0.009054233183102826, -0.009864519108265914, 0.019610217662164326], [-0.018460013947631224, 0.018558593458708098, 0.024202197515844365, 0.35868130801039544, 0.010768600409783629, -0.03932068544710088], [0.3959128410822092, -0.010183354604323677, 0.00367519470245778, 0.03325792016099766, 0.005388028076825541, -0.042572399869938896], [0.3994917001971966, -0.003454044163042174, 0.0040682071694923775, -0.041883031353644885, -0.0006887116238572293, 0.06422921310718835], [-0.3636387841833566, 0.0003241907761774448, -0.0015459150046960644, -0.15336912664001584, 7.989034591986515e-05, -0.03858692196069654], [-0.5627025816228225, -0.0152660058514795, -0.0008478301684532777, -0.004710321095834489, -0.0017614151764263422, 0.05655148724834329], [-0.5566301310329291, 0.00535556969695304, -0.0018534892604365378, 0.06507660245633451, -0.001881168638860509, -0.0002445260782103629], [0.2760950604290242, 0.00014005476169927166, 0.0019032315652358178, 0.11316508367137362, 0.004483535635664431, 0.03602899884928448], [0.3099638387023605, -0.009138042952337973, 0.005764846514955536, 0.06768390953013664, 0.0036099429680303673, 0.04947446698548825], [0.38934671677861427, -0.008176672360159083, 0.0021447637516117013, 0.05756243313843874, 0.0030691087800433104, -0.014217322072542326], [0.3325174850333206, -0.013489119287863413, 0.011132598415614633, 0.04355124027199252, 0.011011063921694236, -0.0321276839391752], [-0.5276997661958198, -0.008551061011762697, 0.001703107513083918, -0.0347905042888954, 0.0011012959211000035, 0.05383359472895661], [-0.5588518220672403, -0.003951054118495723, -0.0009590513830727213, 0.036403660743108385, -0.01095231045980743, 0.020157243952171788], [-0.06817360541743539, 0.01927864037507335, 0.010058154437852984, 0.04245976977707046, 0.0010718571313765826, 0.20689391385479206], [0.41470747941224867, -0.005531502273718168, 0.003960619877206667, 0.02850037748409897, -0.012606928618137994, -0.022592109373764324], [0.11797404276049783, -0.05608328776615504, -0.024194657535300465, 0.002958222749522753, -0.033569422454457216, -0.149494183468393], [-0.5038465041468277, -0.012624246676593872, -0.007609576590001201, -0.03324107902046723, -0.0064022832047363, 0.028272737257671342], [0.20029793323460016, 0.0057806827545642675, 0.009537869266890486, 0.07325258694894675, 0.0038503648229716607, 0.0838665249107736], [0.2446176748355908, 0.009506155823573885, 0.019809166414025633, -0.27413237230553045, 0.01692152985828572, 0.16234078188199194], [0.27457131014546043, 0.012248114416200315, 0.002004750677128766, 0.09899404884759103, 0.006040573817780846, 0.03400016384447258], [-0.03322820182037478, -0.011020507539345019, -0.010641011898345066, 0.40433674096154054, 0.024292008424063097, -0.025801091619603807], [0.35726581865649515, -0.011132684651951547, -0.03753487310894558, 0.010720692549709985, 0.0007348555214452129, 0.05876904817610172], [0.3162671228675879, 0.008683329571161276, 0.007655431165064293, 0.04637799468992451, 0.00301548627316686, 0.04646376384839463], [-0.5341594928159465, -0.01957798501695487, -0.010011494306237564, 0.045387153644283584, -0.00028963717275393706, -0.0395211084349578], [0.0028871632659014674, -0.006957137008270935, -0.01988112259667958, 0.2531562008446403, -0.009967065558452833, 0.08663856999946941], [0.21165595797240258, 0.018323714117038776, 0.0044266855099032885, 0.08033777642152488, 0.003986750553579676, 0.10641439320332954], [-0.3457697173070718, 0.002906488396781204, -0.004678197693717798, -0.17388145696687837, -0.0018316398578986958, -0.04223214323788112], [0.39941615224397975, -0.006084090306870074, 0.008779412447801019, 0.04966411017190075, -0.0060674593646079245, -0.020638188085287758], [0.34752882716479444, 0.01303186026418298, 0.009394123540258635, 0.08012694374877237, 0.0010666953333653814, -0.07016054961847429], [-0.14283888369579092, -0.02386326904959966, -0.04538553019573566, -0.0213467299956705, -0.002228063986182702, 0.08024723882774203], [-0.20341428407906223, -0.029998757191188802, -0.004894745656367517, -0.044243365354748876, -0.0033101177289926245, -0.17125177028268018], [0.4115882041099418, -0.0060619234841517555, 0.0038026269713927817, 0.032116681178470885, 0.002566884795599393, -0.01442630269810327], [0.3757147035179827, -0.00842245125310891, 0.0037915093889105854, 0.06819141287713408, 0.006444369256621096, -0.014649606680623699], [0.26396772614644476, -0.006749275772494401, 0.0076124242420911176, 0.07652349084804717, -0.009153669240099062, 0.07825311329982042], [-0.27623621437984097, -0.031045385185734783, 0.0029543264300979856, -0.08502157263924875, -0.0019444975143612607, -0.1418599900442458], [-0.19602239867329327, -0.036908708281978404, 0.014518311079174189, -0.001189172498027758, -0.01621182632497016, -0.17298239577709584], [0.275105345198704, 0.03212292241467773, 0.011590932646676318, 0.07502481660935026, 0.005888348804134037, -0.021912214158390588], [0.3246308586886197, -0.001662684111529663, -0.010298211235516087, 0.06452950304545706, 0.0075601000142053775, 0.04314662407495118], [0.03713886312507734, 0.007051531008962777, -0.019113712118350935, -0.1793061353279825, -0.03798724873844628, -0.16590488525084782], [0.06761427801603381, -0.0036244016891899796, -0.039758931753127315, 0.016775361118791507, -0.0470538505394468, -0.14286769324829854], [-0.3769681833721179, 0.010554348665838802, -0.003664010367776652, -0.142993375237246, 0.007212808178865037, -0.03879492120089909], [0.30067384240254336, -0.011771124791239469, 0.005159442199728729, 0.04794908531214223, -0.009581194709305662, 0.05643416011244481], [-0.18480798380533836, -0.030049400642884645, -0.010332239659198753, 0.10764043079371628, -0.035706031151735505, -0.12759435706413955], [-0.42268979054903616, -0.01252366979682687, -0.004508489100441557, -0.08078389878186107, -0.005419800024671433, -0.026918161270974603], [-0.49602562827386737, 0.007471906376607576, 0.0077024707274615705, -0.03980193777012965, -0.00015852617621744974, 0.02405123892566496], [-0.35487832757141413, -0.0058116488606748855, -0.008103812196305377, -0.15264664026657737, -0.0005740710712317356, -0.04047216670046342], [0.39514553522721235, 0.016033036739868358, 0.02121294461642203, -0.07574645918763034, 0.003754940522275634, 0.044530002081850734], [0.5000298654859843, 0.007022543690652228, 0.00459750807898861, -0.08673518522580106, 0.013439150849825264, -0.035340549546317886], [0.39772803449138266, -0.007971412966828092, 0.003602513044385564, 0.054145242068321824, -0.006665713934378985, -0.01743546922969108], [0.4067583618682603, -0.006359632680140505, 0.004614120772557788, 0.037984617300062175, -0.003894233992518407, -0.021968481346492468], [0.19765021953281167, -0.006270815431610002, 0.001712123871866619, 0.08841932791693845, 0.006310077225943093, 0.11710906688405187], [0.3877391211675378, -0.0048420298458142445, 0.0034673847517102028, 0.016527086427907567, 0.0067781328575907875, 0.01731538938682982], [-0.493119607024013, -0.00850376045534231, -0.005680362387825619, -0.014758167766273885, -0.006443678547758081, -0.029064423818789407], [0.3735900519491276, 0.010155188901466659, -0.00011279381513214501, 0.07464155760788553, 0.0020038182320459516, -0.034987106547698134], [0.34409501819324, 0.006396590248594625, 0.004924933512316577, 0.08634993155570853, 0.006572665942346298, -0.014912059488146277], [-0.4903691762135484, -0.005206716437464761, -0.004347367013821758, -0.0317834583579218, 0.0020263210804832023, -0.028722936391061486], [0.30936579714434354, 0.008299170699863667, 0.0151337017987563, 0.1149061985905356, 0.004799809606315187, -0.019658011173148207], [0.3128562488586658, -0.00981772194694469, 0.008334740094116217, 0.12863806771303474, 0.0010639151975611813, -0.013859535630719], [-0.19162308464697891, -0.017116008006227864, -0.0038591682652005816, -0.2229476946038903, 0.0018189905104883685, -0.10020251211237273], [0.045683107885671884, 0.0055750745505164555, 0.009292473678113833, 0.1589059831631539, 0.01293201860899861, 0.11874380010278028], [-0.2757077301329804, -0.004771692976488642, 0.01148701571797926, 0.40488583250123583, 0.02138724674277234, 0.061256471004622254], [-0.3794678905008544, 0.0010968842949725282, 0.0014718343946907885, -0.04267020241035621, -0.0034144718416152216, -0.08220216921087235], [-0.22463263677177328, 0.003076732801097837, 0.009357251350449458, -0.034067840127437195, -0.004360339997251189, -0.17367927836619662], [0.34821920116026783, 0.006159392963499098, 0.0039182491612878975, 0.018042051389344897, 0.0020353620513694675, 0.04781587187964132], [-0.12616059378512925, -0.054809463066312035, -0.028258835845378694, 0.10240811729537266, -0.009875132755975097, -0.13415980612829007], [-0.16799099930718792, -0.05445214431579869, -0.011502948348727766, 0.11957491189403238, -0.014774917339035852, -0.16388602379540274], [0.4769057442973997, 0.01618075704426689, -0.000688064123134148, -0.08412549256245602, 0.006076331608276954, -0.012675578785365087], [-0.5720384402781724, -0.0033959457483386823, 0.003403609954547688, -0.005255947929000473, -0.01346460216668357, 0.0466634690247869], [-0.20022161100126232, -0.0029647550266269644, 0.022439115529508814, -0.3391653669502492, 0.0005193577375958607, 0.052910561298335074], [0.41563636892067524, 0.011244960362382626, 0.0033471023427319477, 0.011308832410956215, 0.005240954400694681, -0.02022784806707331], [-0.4192979123492022, 0.011204853031018851, -0.003231042736764813, -0.11126207293097394, 0.0036821451945719527, -0.03283263687531757], [-0.5016986008404842, -0.0025439526922982096, 0.014668501203351166, -0.0022520160858370743, -0.0015770662653328462, -0.0547501986527337], [-0.19406300764077472, 0.008847908334654353, 0.005376845747593904, -0.30653049803682747, -0.025441394994546826, 0.03859728944704372], [0.3297468245520435, 0.01884426900030707, 0.01082022418778251, 0.042133665754670674, -0.0010922227584010341, -0.023950971414614865], [0.3502489299173073, 0.01213049452184047, -0.0032903401317108377, 0.07904918022046176, -0.0029952216363698368, -0.014781439117946518], [0.05560928052233037, -0.04434831857645174, 0.012359289791656534, -0.21961954474057185, -0.00842051071085058, -0.1664001962861138], [-0.18886428359960147, 0.0025897387464439274, -0.011534395619727056, -0.34808434906304087, -0.021884530561826484, 0.023303058192989952], [0.22997646459605506, 0.015687802989685376, -0.0014240505193336246, 0.10071412691312234, 0.00330317088216448, 0.07487188990021164], [-0.4868859387294771, 0.0038517598346789737, -0.005743248216190869, -0.047451562338678636, -0.00014584581878594808, -0.025028498064881655], [-0.4757070485968407, -0.0013862035178360671, 0.0018074707654390707, -0.06451304394254377, -0.004046941438311376, 0.06543501814251844], [0.4777881980778108, -0.003496589196781813, 0.005863623144649472, -0.051727485347662805, 0.009736178261191122, -0.020067258272542538], [0.2909075499446035, 0.004101106293523716, -0.011450406134263659, 0.11208672108565441, 0.002730025332938511, 0.012388336810877008], [-0.26720641939028633, 0.005926258968031485, -0.001901040560519768, -0.28665300287098894, -0.01045331609977727, 0.04613418662020804], [0.2171894715541042, -0.007245486104283402, 0.010384300604157291, 0.22127211405083463, 0.006243746857434526, -0.017854623152723768], [-0.4891793751192801, -0.007728702464591194, -0.005292925641617588, -0.048842106856621936, -0.007236767534140948, 0.03651940142577358], [-0.07332145264849227, 0.02342332575805845, 0.01683374564494004, 0.07698847254262467, 0.02401709300616681, 0.19910457384206548], [0.23689231828449128, 0.020317571163417692, 0.017472836592027544, -0.012491022697967676, 0.0028406989152460096, 0.12694819298088109], [0.392299889052568, 0.004232869803281378, 0.000504392597036506, -0.030771783019094206, 0.0056769984846785065, 0.04388001941459359], [0.41496637576695117, -0.004665748430289073, 0.0054973205625972665, -0.05283688783577709, 0.008851542384591638, 0.04911739755192481], [0.07704234522576364, -0.0023581871153288037, 0.005925903120150834, 0.1762998310858345, 0.01647924081373302, 0.11436863814229124], [0.4858839755624779, 0.0011391527829072, 0.00831410762309915, -0.11067563980665915, -0.01580744004608282, -0.010757489449077404], [0.39998418260285046, 0.010057188466537624, 0.0037078200343172896, 0.04341385801680899, 0.0018253635865344677, -0.0338351422668003], [-0.07983224508534387, 0.0018930014224485306, -0.014822267773653476, -0.41012809049294696, -0.02483194231738019, 0.08565154424687484], [0.2188684710266292, 0.00577483892809844, 0.008833913910004602, 0.10558756700290907, 0.009595117569515756, 0.08085342489617738], [0.3661024512596013, 0.006868227174312567, -0.001003201708342003, 0.07352813103635716, 0.006288799791008859, -0.018214470446022868], [-0.3751977963649534, -0.012362103351510301, -0.017261311286973655, -0.10821234580399805, 0.0007681344748107076, -0.04101886338166221], [-0.5918081737088988, 0.00393803324544665, -0.011086872978233143, 0.03844827720032063, 0.004710010090775544, 0.04940933392000835], [-0.34395612663852354, -0.005930555403106413, -0.009033714973317649, -0.06786708844659417, -0.009267643694104044, -0.09165772798721185], [-0.10262600452819602, 0.026411547370473454, 0.021217985443995196, 0.3873540143591432, -0.005913137049255459, -0.03593900877076515], [-0.3152075810787783, 0.002084650852033155, -0.007107781465352763, -0.1585593433058018, -0.013260982221445523, -0.04698721674890948], [-0.3491307695806675, 0.04117652202147051, 0.036307516497085926, 0.361042430692385, 0.010346284893778259, 0.07674753928546534], [-0.35491554675454484, 0.01030433046503287, -0.009550463841858321, -0.2297479448658586, -0.004200163832143286, 0.05756955073413568], [0.43462339408860406, -0.0047937111780751015, 0.005540663323963806, 0.018155131464289983, 0.006808075799946521, -0.0333202201653999], [0.09644637322200665, -0.029404219661286642, -0.013249342735939099, -0.41703940828390135, -0.014833880398598561, -0.11614428404704376], [-0.5971636255329612, -0.01494611498907789, -0.003705311444845743, 0.1410493227606028, 0.003785025616241132, 0.03577975120908331], [0.4341677291165743, -0.0062139569543717185, 0.0014928617380512137, 0.015211820485564687, 0.0012583036687161752, -0.021327667145447798], [0.39275332634312543, 0.010317460885427301, 0.006351257219841553, 0.029253175339190838, 0.006553798619797708, -0.014429970788338098], [-0.5208755221788902, -0.013748343876337369, 0.000628582844405298, 0.03548741479189949, -0.004701628102251859, -0.04352717014549561], [0.43612731447605146, -0.00973675710432274, -0.007138766124638921, 0.03396930057662568, -0.006474895530884684, -0.03629238676902465], [-0.42263663099339094, -0.005200415247610477, -0.006832386216348914, -0.08348459257243718, 0.00011624723688149726, -0.03828222220709256], [-0.028922234527203886, 0.023515240476566557, -0.012827902773690827, -0.2975086124054608, -0.005617580448551512, -0.11762557698832439], [0.3409859104565475, 0.008150391216739087, 0.00045868430005797104, 0.03895382623060607, 0.006257070275474531, 0.03399057926920703], [0.40957414277122284, -0.006430047621636134, -0.005349648428414012, 0.04136926753767563, 0.007157377676225162, -0.01645061574460032], [0.09080795655101324, 0.017410293337740038, 0.005426331503527012, 0.1641235725742525, 0.012285105041085863, 0.06407865785607628], [0.3000862199053264, 0.0011062806023551138, 0.0016781630597237024, 0.12559280901097641, 0.0009206971600035237, -0.014745836405052403], [0.07337206746415183, -0.003483467512226837, -0.0036306421443734354, -0.4317069033893582, -0.019114816701087643, -0.1038435393044071], [-0.3283000494127841, -0.0015704165462417706, -0.005478100801838172, -0.047955820398339226, -0.03318488627807646, 0.03448474962775575], [-0.5414011073867664, -0.017925067765824452, -0.011772000521545336, 0.0852729637893462, -0.014549751564601485, 0.004304963449385568], [0.4000816318019756, 0.006017051042547784, -0.000399695399714564, 0.03621690686914029, 0.005620370542095917, -0.022502098189380756], [0.31739726769506016, 0.02070482019035464, 0.02419681154654353, -0.24234674668829997, 0.01846673308766869, 0.11742778083533999], [0.48703199823386056, 0.013933369940685284, -0.00010800615420706587, -0.08318715812276405, 0.013427910224409277, -0.011091083309661721], [-0.5123667933724068, 0.022248246629622256, -0.0003401940693442319, 0.0029259337284371356, -0.0029047977383766413, -0.03778120470174399], [-0.5254528674640059, 0.005533448510674934, -0.004431405774576814, 0.008080342602765803, -0.003170460349380216, -0.03729572419214838], [-0.5840358041036887, -0.013388560042341157, 0.010410577764441068, 0.027937776360534967, 0.0001740367973672237, 0.03859426830564846], [-0.4939699149699461, -0.007947835781756443, -0.005512762959057618, -0.0669970940855668, -0.0005352657723635624, 0.029059540235355206], [0.39526426976299134, 0.00909974036177103, -0.0010136456953353228, 0.038344612437371424, 0.004656129464175963, -0.015781169224059793], [-0.26932234558640117, 0.001094589153365414, -0.016423759135500392, -0.27436002720905384, -0.0022667473137279776, 0.06341067104369848], [-0.46791826461101343, 0.0038920281016991184, -0.0139362159308553, -0.008214955844172374, -0.004696882462154427, -0.05427904258684023], [0.40340776371550063, 0.011317814480395792, 0.002506056819843682, 0.023855758910439485, 0.000289940920808539, -0.017064973497650276], [0.47493236599541444, -0.008441448775534426, -0.027971482604251467, -0.079967010546494, 0.011512188562244108, -0.013795677057153472], [-0.5536410855563849, 0.027704984647735187, -0.009761079474301765, 0.1403059844093658, 3.0479723649103582e-05, -0.046663468454253176], [0.4643180716076183, 0.007873010509319517, 0.026351841210102627, -0.1725438275132459, 0.007532042678936638, 0.0014762424596480794], [-0.5206179200157691, 0.008871696707052523, -0.0066905287778508075, -0.008222725356758281, -0.0006871126375906715, -0.027306743252421006], [0.402701131093514, -0.00987235398877887, 0.024000984543476375, -0.0697898593808441, 0.006433187148182974, 0.04302833915587789], [0.3060500994645957, 0.005706047523262976, 0.0014481722149866877, 0.13349403395863702, 0.005489955890712175, -0.02529618784007401], [-0.05128003367820824, -0.008262008046492908, 0.0026672912548770336, 0.19493952666212538, 0.011609980582746865, 0.11485249501687614], [0.15952028319522488, 0.028055144557993822, 0.011125394696345239, -0.06321149569670835, 0.026398475543862515, 0.15542315008423413], [0.31687907017056516, 0.012046482456131357, -0.02938652764043235, 0.04595925450825822, 0.005609459541810699, 0.022572260963665807], [-0.5871191113096832, -0.01610743563551196, 0.005135485987743026, 0.11769161458287329, -0.018848655618999937, 0.028690006755480432], [0.48278887816599286, 0.009372943663033586, 0.006157681763532862, -0.06718879154723481, 0.009474125707632985, -0.013674837752959402], [0.07104636461479967, -0.022828489180822652, -0.011577799479366631, -0.44523209548104126, -0.007384075359988763, -0.09296692098659483], [-0.40080362905951616, 0.00040246575073231444, -0.005933171559725459, -0.031337401044618386, 0.0019436140185108163, -0.0744519409984656], [0.03197454593067337, 0.010872938088429806, 0.0202206882999024, 0.10275253884488006, 0.0157949363913363, 0.21289577868798035], [0.35736505646214617, -0.00881541637463057, 0.009678855966086277, 0.08330258485952965, 0.00601406083013552, -0.020118061779208125], [0.3473131082915795, 0.00800842122770801, 0.007876948958261574, 0.08216114773577654, -0.00028734346796789676, -0.015822107306762867], [-0.32387552605975994, 0.03272299714781055, 0.015468300358676596, 0.34551980352660056, 0.031184955747765555, 0.08369839234414052], [-0.4158044239144896, 0.003421852230419948, -0.009897337538302672, -0.1552242334913292, 0.0019629847302237635, 0.06164377703109518], [-0.22164455970770874, -0.01878015282023396, -0.007300862412614875, -0.28130658673736314, -0.014320026164035106, 0.02953218784195636], [0.3757245227114154, -0.0021249270408107315, -0.001216919414269322, 0.072378536023259, -0.008346494603733962, -0.017023351997516464], [0.400726406448903, 0.027587774007857793, 0.006400232173686578, -0.08692499407456646, 0.008942573127895698, 0.02932300831622262], [0.03480258465015031, -0.03309403812720283, 0.008704485825462639, -0.3178400940879617, 0.008939699794642526, -0.06261780060496051], [0.12142661770155282, -0.0036890165794397248, 0.0021876920326002833, 0.2822969954956394, 0.0013341652058793119, -0.03913150436128308], [-0.28937697232873566, -0.027090072166845048, 8.452007697923975e-05, -0.25564612488865657, -0.0018618114570041213, 0.03920141314521643], [0.33027542698954815, 0.024452953063910057, 0.00557341374288312, 0.067505500021351, 0.010628378568100248, -0.044505672385793636], [-0.48522928428906376, 0.005261613158357756, -0.006245629542184249, -0.024239444364550437, 0.0010411108910027289, -0.03774169918689976], [-0.44769006822914686, 0.000864057908548635, -0.027084976521602, 0.21094585494411863, 0.00389258138543971, -0.03833172267031779], [-0.3560868013648717, 0.0034651822410522123, -0.0067312622314523235, -0.16626510560955884, -0.00014219646029303053, -0.03161537213043309], [0.2116218668020188, -0.011046316502115922, 0.018944114474805505, 0.09878370183674808, 0.009282124482513764, 0.08292784223936406], [-0.24886856908430208, -0.017482756848165918, -0.007699740274791921, -0.18336650301922675, -0.008593845231934957, -0.09231616129915352], [0.2685317106527556, -0.012962835010563544, 0.0025931614606300004, 0.11563377829163186, -0.008790880680566413, 0.0520083986194468], [0.4353608596790804, -0.0017822755480096735, 0.006803016017756624, -0.06912056389339173, 0.008956470418783747, 0.04554582665911287], [-0.5644359117198614, -0.016904977242174463, 0.005967075570356977, 0.14140856677427657, -0.013841495711733102, -0.03024143227404162], [-0.4234366345004769, 0.0005594984741713043, -0.003000884296885013, -0.08848798777150801, -0.00025306644784380515, -0.0488675921241239], [-0.33773428068647293, 0.0057279612996129345, 0.023480538220602937, -0.18993964438762825, 0.01273811737504619, -0.034715707694177346], [0.024490984141513118, 0.022957946541457136, -0.016974701659149976, -0.2615726369862299, -0.01860445719530347, -0.14138012185527385], [0.4162682943015404, 0.007232814402689954, 0.002931876982041169, 0.023879391499883985, -0.015148785850926497, -0.01596275800189852], [-0.5469596522928198, -0.015898376598598028, -0.009790391244786695, 0.07736910610045451, -4.013541179313995e-06, -0.03728667242307376], [0.043552578993706285, -0.013753558189163762, -0.018028152225115498, -0.3119550291583022, 0.0027323039189728776, -0.1387590163559703], [-0.26386128235926615, 0.013170981977803978, -0.008987787768352504, -0.22225127158026284, -0.002838451281388872, -0.06598076041710495], [0.3056326018950163, 0.010483838628930108, 0.002748596252413575, 0.0834396697341821, 0.0075384942364644415, 0.010806496222689494], [-0.19251224320720386, -0.0008449085227956958, 5.4714194052880025e-05, -0.2546163067813612, -0.02005279905285944, -0.0776262344076105], [0.35950977399308215, -0.00665906659907994, 0.0016582502634812284, 0.028812792021170437, 0.005562741343485654, 0.040600593723623284], [0.40424879507806694, -0.0061514481055784265, 0.0007932040880089296, 0.04248406160402801, -0.0028808105096367396, -0.04731975453584366], [-0.07438539349677913, 0.03991064444450291, 0.022691997058865758, 0.0664945004780298, 0.02484411585412641, -0.08313563863153846], [0.4167480220238195, 0.007936209457733592, -0.0008969039780886424, -0.045705597785307556, 0.005743259268565145, 0.013688344346611342], [0.4729215890847603, 0.01348448345611007, -0.0021603872775464604, -0.07340633327254709, 0.008695869389863746, -0.02010522138064365], [0.34642672014322573, -0.0020129617098367065, 6.970761688340536e-05, 0.03561740433027956, 0.006779170449035339, 0.0418955875857108], [0.2832735958437323, 0.031297518133879376, 0.020463210506516102, -0.1088689276198131, 0.0010431764769669128, 0.1085428552301494], [-0.3860659579178569, -0.011282491248894836, -0.003683523843166364, -0.11295322406780495, 0.001405029518980061, -0.03936483244125943], [0.2290647795043256, 0.005389798554480014, 0.0008400366857132397, 0.12767467204832395, 0.00361638086877237, 0.05856363058400015], [0.2266539574620483, 0.009722630266862688, -0.0031617175895505547, 0.11297535718560546, -0.005564972567440683, 0.07391585635358627], [0.22029416620914474, 0.009736576040987491, 0.0031353353619212074, 0.14081603035019566, 0.0049421480915722025, 0.05339170885845995], [-0.010867981285060497, -0.0032937087388303157, 0.005936382048649001, 0.17417932129258476, -0.012715829298424994, 0.13734675826102385], [0.48081066932787786, -0.001731031191817381, 0.003831499780049432, -0.09600966213881185, -0.019307330614101472, -0.020134733398493254], [0.28089294950446436, 0.004924588322018129, 0.006644328173175703, 0.06852467577736657, -0.007909609412811434, 0.06914473430245427], [-0.2668781480724617, -0.025470425108922518, -0.009405119812663431, -0.19866335664508195, 0.00012827942304392733, -0.05170384883153391], [-0.14593352034224574, 0.00689118289500566, 0.006024411000837129, -0.3164239762249426, 0.0029065535345828464, -0.07457631752990293], [0.031023798241201703, -0.012409750557288294, -0.040419666485124545, -0.043142117597634, -0.007538518392842756, -0.20727422139878898], [0.17807799136174737, 0.0032687196994025365, 0.0035618507100271413, 0.2304619880055388, 0.010487362371039735, -0.05263664420517164], [-0.35142993096031233, 0.00784095435515592, -0.016711736840096615, -0.03696755510856451, -0.01454864165818308, -0.09120943899435012], [-0.41482429564047824, -0.007866299555149878, -0.004671757513615947, -0.1063915443270401, 0.003271509682469623, -0.023337612646186896], [0.44320034499998046, 0.008262633580013277, 0.00391590006585681, -0.00019669862009369496, 0.002630118080768521, -0.06475134572557661], [-0.5457727873421238, 0.0058068444710743985, -0.0013757235317787288, -0.009864696208182238, -0.0012270807951939504, 0.03363281386624759], [0.46324244445283247, 0.016442811083655487, -0.01228557326085067, -0.2008539898038745, 0.03301212103469175, -0.045542846186197784], [0.34635628027017584, -0.0018304865244957763, 0.006945585808837068, 0.0676507781857172, 0.003934835390660395, 0.007289673535771381], [0.39951238932473293, 0.0153650592602182, 0.006784967306422735, 0.034481443546256166, 0.0018195113575956306, -0.025781687293545415], [-0.24945208795883422, 0.01368268577374989, -0.006732574390455908, -0.2430687329432307, -0.0005635111925698851, -0.058602445955325516], [-0.4174838418637634, 0.005331624088199223, -0.002362996196761717, -0.10666915623843405, -0.015194331545471647, -0.023969076021547544], [0.3574041201105262, -0.005712620016966263, 8.960776458776425e-05, 0.08890795790795752, 0.007123193638458169, -0.014548926071232616], [-0.3464893427594125, 0.007293043398383302, -0.008797270299490817, -0.08592797848501771, -0.014219374467645682, -0.081232107689848], [0.3128382074986167, -0.00392850750241525, 0.0010293631585582793, 0.08095301920359751, 0.0012078439386976369, 0.03991340703627775], [-0.5702786211110563, -0.0033724784799375046, -0.001360719245383801, 0.12445442818433933, -0.0020623651797327574, 0.02450357978559104], [-0.5442407227347097, -0.012977541770598715, -0.010061062735562357, 0.068939435407696, -0.008358732212697994, -0.036329709287463464], [0.3563899729647994, 0.007878122006546357, 0.005543233157488529, 0.02169276151459835, 0.005741889002230548, 0.01580426900736662], [-0.3591325926777071, -0.007482315996885075, -0.002682334793659343, -0.1517039766472652, -0.007403506645408737, -0.032915273239076004], [0.2788484338297914, 0.007831064895475687, -5.210821229962019e-06, 0.08488906316705343, 0.006247017381774301, 0.033131536309039984], [-0.556157081235676, -0.016110580929750946, 0.002852412719494028, 0.15782803080161503, -0.009491819011075969, -0.019540565519213654], [0.2112558592539635, -0.0029974664248511983, 0.0013376447884452713, 0.08813583733989261, 0.0023764644924345107, 0.08016689864535402], [0.015107053504434989, -0.007747109741862201, -0.012225926202405474, -0.4385168599874014, -0.024125583072855842, 0.07383525089691467], [0.42825450527354386, 0.0069167025868709505, 0.0045656246836353305, -0.046334359603042725, 0.007639323215192339, -0.028639573933977015], [-0.43972332790905627, -0.010814330084101738, -0.007337477139415527, -0.047160651377893034, -0.004560300501266877, -0.04540981042416491], [-0.05034199360307791, -0.03750986060602435, -0.009693857820006747, -0.34147662455956485, -0.026086000675049358, 0.018496670597056047], [-0.25956011220286385, -0.009233069377820176, 0.029306516600473457, -0.288944928827815, -0.0019812015083722724, 0.06950549372909842], [0.41901956220038244, 0.003960236958250858, 0.002905119312853966, 0.022151569034214242, 0.003768112674370606, -0.033666266846742496], [-0.44311103219067527, -0.010350222175952065, -0.0039080880303715565, -0.07574421106645125, -0.00021062885757011958, -0.02396803990120205], [-0.5437439037076817, -0.01034276455502551, -0.016277396727989307, 0.1332135251847218, -0.017475765307172212, -0.05147345679161764], [0.19640984432557484, -0.00880180744211482, 0.00743240286573212, 0.0957063002014621, 0.004395171390677009, 0.11648459426695328], [-0.5642382477894401, 0.002099792372116455, -0.0025745516825321775, 0.03182421343622197, -0.012427389732565264, 0.025728326253338794], [-0.5249710034848283, 0.010350862236520802, 0.0005689765705258491, -0.012432116675004552, -0.0017626517026200746, -0.02661573361126363], [-0.4557638759505026, -0.015364358283446192, -0.006879774298994438, -0.034577086315551224, 0.0017895360747569968, -0.03671491741674058], [-0.41924095555910595, -0.009975939929485877, -0.01106801921182599, -0.13287512822605368, 0.0017902754666624006, 0.04150810079314041], [-0.46623411850850455, -0.016522881040465755, -0.006504155002472263, -0.0996914731696641, -0.0024433064933151844, 0.0354092675477535], [0.3026152252079862, 0.006616271464168412, 0.0035066666090550432, 0.07027417721531748, 0.0010605294332182403, 0.04018793815106163], [0.32929261858694103, 0.018605610770036054, 0.0009759855078648284, 0.10214293291600993, 0.009843176462621336, -0.030320054883204967], [-0.4437892559522887, -0.011138666404785287, -0.005447074357490566, -0.03957100755648446, -0.007878421168517332, -0.050578907893768205], [-0.5748471763770853, -0.010557611571808326, 0.0007623165607479001, 0.03724467560259484, -0.0031730625639498473, 0.030625858349495747], [0.32293555912649957, 0.009830206298421315, 0.005278503807858153, 0.05300364978070474, 0.0003067405122623143, 0.04284617380758523], [0.2658109074493007, 0.009129535942370271, 0.0021237116352582847, 0.11394050579090229, 0.003635944869879057, 0.03787272764562337], [0.260480897914685, 0.0013733137860510495, -0.004078524167940102, 0.08454072417276769, -0.004261651206217998, 0.07625023950065526], [0.43295729123405247, 0.01218687673855856, 0.004962954013418413, -0.07550536657849649, 0.007882573399591506, 0.029433766430969843], [0.28419379483997603, 0.03013426413864789, 0.010677184271913758, 0.07574591275527845, 0.010426356275073016, -0.02498236076573771], [-0.03482158473644754, 0.0534916957170476, 0.031630693887805095, 0.2102522514935786, 0.022316506872781566, -0.050499087044289334], [0.3178977381151725, 0.00541521967804851, 0.0013176403187519655, 0.07437053894912458, -0.00021192501590142825, 0.026121990962322254], [0.18696791020443976, 0.02390841474868377, 0.037392891671917096, -0.2121498017085898, 0.005831676551706275, 0.1627408132937486], [-0.5141453986397199, 0.011950285778586903, -0.00686383037231689, 0.005094903931623, -0.0030749898147623125, -0.04390795501039931], [0.42851809169036303, -0.004588828009135625, -0.00899392475257668, -0.0927181330376408, 0.004536982159202523, -0.016890014380748594], [0.3832914751293229, 0.004930230174535117, 0.0021510265863092237, -0.01219479625277197, 0.002371783893437959, 0.04704694713583345], [0.25792530182252915, -0.010721273045089841, 0.018401760591996833, 0.16569818345561038, 0.009142464309641728, -0.03134078574205558], [0.3259184160473381, 0.006298223477722596, 0.0007616284953193778, 0.06155936041211203, 0.003176067754802192, 0.03574943222800407], [-0.5014869506394013, 0.0013800014791740685, -0.008173537740346883, -0.05327052133934855, -0.008708222636654335, 0.028439230876575648], [0.20409613857935485, -0.006569091165482808, 0.014504464725063526, 0.05079015663213261, 0.003734562196957276, 0.1119963140951987], [0.4160719187132455, 0.0035433649107829866, 0.005670707433139686, -0.006807555380254032, -0.04533060145724362, -0.026957913584752146], [-0.009210724986438266, -0.01801549535158453, 0.00462163573062036, 0.39294407310707913, 0.006658275581536731, -0.06028403392248324], [0.2968376358527989, -0.009247606849512766, 0.007184755653241001, 0.12381676510388863, -0.007313818402384725, -0.03785169961200093], [0.13387456774968468, -0.04799420683720971, -0.015342500256047657, -0.26028804450708987, -0.0027612674556613576, -0.10330096232579379], [0.2931820942502839, -0.0025189542257691645, 0.006853340984071318, 0.14111312456086214, 0.006669761309214415, -0.010928890688186498], [0.3028003898848736, -0.014177866571047327, -1.699394139469499e-05, 0.14185645850881232, -0.004271171878456155, -0.011677482669454564], [0.2963715652416335, -0.005501396487024459, 0.006645845085927478, 0.11320450119556737, 0.007657005383204658, 0.016135812914025265], [-0.5522659961048343, -0.024045843352089535, -0.0022060836014162, 0.16318314225333863, 0.006726948509263533, -0.03485502484712281], [-0.14080304349232386, 0.02976297364436281, 0.03867172632106877, 0.19808569154485078, 0.023267725873428974, 0.15854476377095011], [-0.047111924926290134, 0.030482652304573338, 0.004074787063804104, 0.035895640134610926, 0.02903375925203692, 0.19701044331412165], [0.2647882683019573, 0.007258634527344994, 0.015311434638100543, 0.04028442275705174, 0.01126075425981859, 0.07991490359482306], [0.3747245097021934, 0.002797897030467598, 0.0011200829606749776, 0.012999800322056011, -0.004543902172303296, 0.03816494549024393], [0.3851947519014163, -0.009920307556019196, -0.012103536184523688, 0.017257705622700962, 0.001497884261755867, 0.03217016862133433], [0.2974886227004067, 0.009211208021930886, -0.010730759332977511, 0.10097870835395523, 0.005027589710686635, 0.028590595458278748], [0.41605190646003476, -0.006366013327332178, 0.0021288553937435675, 0.034848581959597745, 0.004390466751256915, -0.061849987713494266], [0.3304346860082859, 0.00919682589189587, 0.005291171833942608, 0.07169893843668244, 0.003717578195612114, 0.0019771632699430264], [-0.5900952904069454, -0.0026531482340515335, -0.0021201842083797486, 0.040263246645620344, -0.0038597211097936407, 0.020043756031877177], [-0.4739020004306443, 0.008621020441773417, 0.0015824753606652326, -0.1084791429749077, -0.0020876167855784465, 0.016611931055355963], [-0.00696154748747049, 0.023699631087969675, -0.007140188549647425, -0.3677483959268043, -0.0023124267112432616, -0.08976778669851863], [0.5007057747075704, -0.000116348844964534, 0.00804777271628848, -0.08743723799741403, 0.0075183015274909, -0.007581552152264431], [0.4332213817155632, -0.006799200765456637, 0.0012647974809336484, 0.002659766572875524, 0.007258963934567542, -0.01909237560515205], [0.3153936861655364, 0.005937743486047546, 0.006617639863557418, 0.10182432108679322, -0.00995554865875739, -0.02426192619226238], [-0.45385423105840234, 0.014371059027186137, -0.004028360845958123, -0.062459763404774604, -0.006210995311558812, -0.03559604173982789], [0.32595278399638644, -0.008014717435557114, -0.018223481448380912, 0.07021073467225812, 0.002804874859160861, 0.04903963219595708], [-0.3726799858779509, 0.006227951285077356, -0.004315656710812039, -0.08835503943141104, -0.002769486847507785, -0.0853801633697773], [0.3349261356339528, 0.006707835753001305, 0.0038931953972174567, 0.040169467659674156, 0.005594296406605565, 0.030127554707704843], [0.34565477244166004, 0.0034480988026988225, 0.0019615121161105915, 0.040749431663448875, -0.002392115715432609, 0.04009163402484523], [0.15865578476007508, 0.00041548059120010203, -0.001843658979418926, 0.26198488903236783, 0.009974362001985612, -0.014454225827261429], [0.42796984791017517, 0.01777596788044072, 0.004599043036415743, -0.07195997214783961, -0.00815076777650142, 0.04361254776397469], [0.22719460118041823, 0.011694922513349425, 0.00020716854280619084, 0.20364613729896333, 0.005421247125915132, -0.022598111749171187], [0.3828291698722531, -0.0056632009891270825, 0.009656522844599185, -0.020374431758374077, 0.0038167065948875576, 0.059665233435760266], [0.4470980798601868, -0.012225231371264363, 0.008220682250540894, -0.10060155021131682, 0.009374493083398095, -0.008214882702455002], [-0.45792211043093944, -0.010710034740551114, -0.005676319946765873, -0.041524485723901945, -0.00016210966452361746, -0.03568208235046256], [0.12436824942252268, -0.008118765366297276, 0.00139611732923317, 0.2169611444275677, -0.001293874213900571, 0.06627982681357325], [-0.4904878179550976, -0.022537367052963372, -0.01091577841965363, 0.0422046597030149, -0.007134139262607264, -0.044702477048634975], [0.3199676106829895, 0.00663190230600302, 0.007080513493621698, 0.06888484012932014, -0.002856782948035516, 0.02397191633610022], [-0.3547652151824161, -0.04530526369361082, -0.009161611679289517, 0.08168536641120264, -0.012914944471273976, -0.08487618852746977], [0.41893139401505797, -0.010859723634991153, 0.004544441764885141, -0.045189424543783996, 0.007758937183785452, 0.009202708548379155], [0.2464904722579537, -0.007229276677786122, 0.01152160727463203, 0.14614361165508125, 0.0063224693853091325, -0.011635476369801296], [-0.48756637243336826, -0.013222961051856805, -0.004799259011830152, -0.062110449326637035, -0.002953310090904669, 0.06833783348575304], [0.27670448101083917, 0.005528092593783189, 0.00195606442122603, 0.11543674735265246, 0.005189003018893565, 0.024501576514887453], [0.3450114703779695, 0.010228098141136703, 0.001324336972712054, 0.025636191876415068, 0.0007674346959221303, 0.04203838601493984], [0.2022847875817908, 0.006314073718095903, 0.005892138977643464, 0.050296840085569866, -0.0229263418641611, 0.11662604118360184], [-0.44915357247264387, -0.011272650203656288, -0.009713398617593329, -0.10857200692200006, 0.0031783233368893084, 0.021088304879002664], [0.38184728038779125, 0.009118922268833672, -0.011196286262935907, 0.060268010953115424, -0.0036246179182874744, -0.01409337232160247], [-0.38054773200755415, -0.023699731426617597, 0.0021204007995400407, -0.031034625325057304, 0.0003503704158971307, -0.09059201578954315], [-0.45410687439986636, 0.0005914304637879345, -0.008391939216593732, -0.11642301922868167, 0.0008188627268853139, 0.03396931743224232], [-0.3486919194124385, 0.005509684501673497, 0.008012430470528608, 0.049566493106640616, -0.0125816884683925, -0.11500783569584715], [0.32571815336443394, -0.013345113775949331, 0.008861510694740699, 0.10983324959037429, 0.004144117954509694, -0.02251943659864772], [-0.003942251122723035, 0.038305238437066604, 0.0017953475213658182, 0.3197051429099499, 0.008237388903467011, -0.01695856141050616], [0.32724768211692934, -0.008431816195101891, 0.0005834414538991098, 0.13145261312617487, 0.008538138564522199, -0.05505529716166298], [0.20345281842241847, 0.020479762503709968, 0.009685746901707391, 0.10725812946140996, -0.0026808801167007056, -0.018352912792898402], [0.23209855695562429, 0.011957439020438967, 0.05297305904515522, -0.12759670396980535, 0.015200979713810982, 0.11814190733001458], [-0.24992541995267872, -0.011204160704756567, -0.009105564706659321, -0.20151799033578635, -0.005955459126615219, -0.08040469521679464], [0.264861134654874, 0.0055688315626325835, 0.00242993906284269, 0.12767317731332078, -0.0030229106329864768, 0.03450316137265147], [-0.26024342975953896, 0.027569804059025874, -0.004629526878312372, -0.03112962466881361, -0.02784535253846919, -0.10822705539907734], [0.2650981986267819, 0.005005507488350581, 0.010819596437778177, 0.16144393320732411, 0.0017757263946321523, -0.011219854385443916], [-0.5568017860676396, 0.0064737699854406725, -0.00905541937659065, 0.019950486528977702, -0.015910588877034827, 0.04989258542589113], [0.2870887417478265, -0.0056931254893567455, 0.007480725959463093, 0.08956311177760846, 0.007698151384990308, 0.009013023034768409], [0.3165209153590487, 0.009029394009007596, 0.001597119661893895, 0.06715935125828892, 0.00731995922617101, 0.029315165247492905], [0.45327517931974715, -0.0041808200838572744, 0.009326438252389553, -0.09200056133591235, -0.0006907858351590879, 0.058641025873266674], [0.21484653898936565, -0.009099928553464314, 0.005903068896149906, 0.11462902321280516, 0.0005515771577000182, 0.0948240036835059], [-0.40280428649893857, -0.015722740601592546, -0.004979686202267752, -0.08558123242711067, -0.0032621129302865554, -0.039803274673139294], [-0.485347778250217, 0.022081676699276354, -0.004248416377724949, -0.03189934975715054, -0.0019204133161740391, -0.03713452852182276], [0.3815561203324326, 0.010896544862859212, 0.008414425270299246, 0.0660610915535515, 0.005550632975796585, -0.053132148328273586], [0.4488159429658466, -0.004443354569977235, 0.0035352436012313806, -0.0790338881132991, 0.011394636880381054, 0.03091141923581619], [-0.5989285244736099, -0.0057376240128214325, 0.0018832411132193615, 0.04318417130836618, -0.001457251767946999, 0.04998896402326449], [-0.37032615773549504, 0.010427048031065457, -0.004664394956448223, -0.15452767077699966, -0.00869679038708318, -0.02809552623853285], [-0.5736452353810753, 0.0013572050727223089, -0.008702105878901046, 0.08247404531230683, 0.0027483039694683395, -0.02334265510865286], [0.42546139149551426, 0.008826956429431125, 0.004815125756761376, -0.04206390720534475, 0.009955410293188274, -0.019624500579074177], [0.4100007849920837, 0.024298917633848664, 0.012455600678798969, -0.06981452659084227, -0.017856323738983052, 0.021178880358424516], [-0.025602011948608724, -0.019954778084892145, -0.007223926135111051, -0.3452911534676963, -0.0003733748589746419, 0.06948238735242601], [0.3784897242484023, 0.005248189013375416, 0.0022329078410499126, -0.006381401527629098, 0.005942931591407835, 0.0336369441054721], [-0.4121083939745596, 0.0026503415200872793, -0.004858396397496315, -0.1708966814034598, -0.0055898850311754174, 0.04698301528660295], [0.41067159449525414, 0.012553701464070886, 0.0035903288772857825, 0.015215039171990017, -0.003495899190092363, -0.044604764818509945], [0.25970324060363054, -0.00727373588244032, 0.003718020469546419, 0.1101722558046287, 0.002855060922012734, 0.05343936860893892], [0.3608363327881319, -0.004970519270025997, 0.001695333838431267, 0.016155311229979385, -0.022080145140452877, 0.06355955956980677], [-0.37682841727998595, 0.008665385177469022, -0.01312784166951266, -0.13438595536869619, -0.004619618018858201, -0.041440219507082966], [0.35097912742292836, 0.006472319482706555, 0.004277790834311022, 0.030973277017135394, 0.006281874236034802, 0.018749572755516062], [-0.0809273296666372, -0.012001640629433971, 0.0032927549622610206, 0.07694569177859253, 0.014442934786205543, 0.18275496972139318], [-0.5790438939727555, -0.004586893538956695, -0.00623527643434702, 0.13710797200519573, 0.00441561897199905, 0.004281852478239478], [0.3830027106549757, -0.009026655228239679, 0.0015402653436988364, 0.07245015994232618, 0.005247042850676795, -0.03441988719980476], [-0.552032311741944, 0.00963612403063709, -0.013797858845108521, 0.013521899055189647, -0.00435887472906614, 0.04563959365886021], [0.05094175138608861, 0.017572016976375327, 0.003694622510919933, 0.20567301887035683, 0.013728376560900066, 0.10263691453837612], [0.3712147310915715, 0.006469614075192958, 0.0022209834395757405, 0.014446240955056588, 0.0019482540923384621, 0.035401009679597785], [0.16407303433054107, 0.013515021852473186, -0.013315085994335675, 0.06864932935997174, -0.007505669029967329, 0.12004166851279385], [-0.2655041769015769, 0.0131553728247395, 0.00542948424668699, 0.38142074034266893, 0.017089123103782952, 0.08678891335946946], [0.47574239222801823, 0.004059047137619633, 0.00471422323694191, -0.1764157820712343, 0.01571591148052193, -0.00716094373922933], [-0.1487956760226809, -0.003974151583424626, -0.0192801751413154, 0.20444403235884698, -0.03128795794932516, -0.10190301767206418], [-0.5703417543453178, -0.010884605004781639, -0.00017938365176079044, -0.011636801101516875, -0.0006719755524170234, 0.042894519655787146], [0.34772588445023966, -0.009417497069542386, -0.002306660094557685, 0.03138318334845753, -0.0011120459671826225, 0.05109134585889987], [0.385976393643779, 0.0029942696728130786, 0.0023851382690198452, 0.005157429431012459, 0.006231489007249206, 0.025424575248205083], [0.2930983257042922, 0.01892145564116791, -0.00015329713418959, 0.10926597653269501, 0.009648609300867684, -0.016836132937915294], [0.21584785097549003, 0.002901922071931929, 0.011933974619532219, 0.08008933141488185, 0.020969725841738222, -0.01961412166239099], [-0.5435083232818274, 0.0038111157882044708, -0.007051205801141206, -0.006850301487980277, -0.009001721279147147, 0.028363769395220645], [0.07168680023562464, -0.02317949805931488, -0.04911792039298591, -0.22877938182974555, -0.007903424711600372, -0.1534293530197553], [0.17142593679355286, -0.030147282615206862, -0.021945236375228442, -0.45016333448137674, 0.0004786428245392489, -0.07462150392405534], [0.3849681928318709, 0.004976581301356362, 0.012622299100351815, -0.04266553766589221, 0.004276070069939382, 0.06045477531475528], [0.4088582216321111, 0.01276281721887391, 0.011668772960466861, 0.0073714379333127645, -0.008636537513065977, -0.016600664612653537], [0.3445528248409413, 0.005707565493596557, 0.0007492068104095536, 0.03507914792580263, 0.0023894167517952546, 0.02452326674888079], [0.28503154443448064, -0.006125946344563724, 0.005706466254298556, 0.11523928427642711, 0.005109823592115745, 0.02830216112057592], [0.29931959136214603, -0.007460961229421319, 0.010807134528264642, -0.0308229369615937, 0.014654680999906673, -0.041460005091798514], [0.4479897691156585, -0.011458179924744828, 0.005431237057976838, -0.0775956069854924, 0.008112159245531051, 0.04761728815773697], [0.051284295514215476, 0.010338496199819665, 0.011559127189017892, -0.20274253519286803, -0.017037209546388647, -0.18860222466884746], [-0.5220142442904548, -0.014287541539136049, -0.0052991851321613865, 0.005793541276371061, 9.100539221652002e-05, -0.027270242373504117], [-0.5580488493473154, 0.004805737324500374, -0.008979450378374216, 0.04627233242700365, -0.007641380776680684, -0.03689505591580497], [0.29936019894305044, 0.013155144629641694, 0.002743270449518069, 0.1401475791311979, 0.0062556503155054176, -0.03356849556226825], [-0.6038215443002783, 0.007760793486498919, -0.007601251843827584, 0.04065056262254756, -0.006145842355609554, 0.03950102508072313], [0.19050636682504588, 0.013015568343938824, -0.0005333080832503572, 0.15628560984050252, 0.0038398192578030232, 0.05806594381596033], [-0.21162147831457637, -0.03921176715805261, -0.0014895203358538155, -0.09104703918853584, -0.0028240602754704045, -0.14023465121102716], [0.21171416263854337, -0.008473691319859353, -0.008353892511785935, 0.23070102510088902, -0.006252137008971335, -0.03272491134326037], [0.24098278693867017, 0.007417502441889264, -0.017085592321588704, 0.05792810555517539, -0.0009572707036586951, 0.0959758172958625], [0.32668319722717626, 0.0036039348902195995, 0.0036479390157969343, 0.03805096325156817, 0.005220256653917865, 0.04875683737662058], [0.23694774665621418, 0.012730292069902597, 0.002480889832543487, 0.19012910718731799, 0.003286314114304365, -0.025186956376573404], [-0.5692811372980998, 0.004367291009370552, -0.005954641569052639, 0.043499320507675056, -0.00020762443111738198, -0.01864042839901874], [-0.3674413031577805, 0.007085688853990113, -0.0015990282814000592, -0.15684991028945683, 0.00881190758384666, -0.03735116423301132], [0.4166064013635663, 0.006223874218065499, 0.008258896092000173, 0.013156609208358034, 0.00020129062118560083, -0.019898023884131102], [0.3642521529313363, -0.0019497301715689353, 0.0003101825727374119, 0.08028225691718847, -0.008955239872376968, -0.01095699079837223], [0.41468145655065736, -0.004108974558300992, 0.01599020541905514, -0.08553200379675578, 0.008553470771652484, 0.06058394085178577], [-0.5449424247321817, 0.004519037940675482, -0.009572207130599366, 0.05677167772680959, 0.00435911916688807, -0.036594490584568415], [-0.5389591041793267, 0.006174630890605833, -0.0001459027582879015, 0.012702782222250315, -0.001365333547487528, -0.0278659615166467], [-0.44226887916413704, -0.024657824195835315, -0.00364505194510584, 0.1447781863648183, -0.025869309934569373, -0.04484958144263311], [0.40238808058535513, 0.0064340407633898915, 0.007375716475969789, -0.044654533533046416, -0.008192502386680876, 0.0567875314283453], [0.3096827400254681, -0.002174945909956855, 0.005264580335841862, 0.08087801408596128, -0.005881732044353864, 0.04049467684037201], [-0.32613442182359575, -0.022866851289257785, -0.014869447557624208, -0.03852569629262105, -8.465983915394183e-05, -0.0945889231977478], [0.24410152367218388, -0.01274743316983879, 0.001352718852467548, 0.20262831141941187, -0.004181299232415886, -0.022223821541809283], [0.2593319171466155, 0.0096048557115213, -0.0024416627513516656, 0.16934805873816536, 0.0014944903946854102, -0.013021694327355633], [0.2506153430071265, -0.004412680392593417, 0.003048393531852599, 0.13506039436064904, 0.004341105926321345, 0.045027443566645294], [-0.5436801291195652, -0.011443368312941203, -0.0032575578565891207, 0.03900806989623256, -5.004778848349534e-05, -0.033188633485324734], [-0.1249618618486304, 0.0035662565861120854, 0.004296902131229883, 0.3727409996128426, 0.0030997571856222323, 0.07500937490425118], [0.2943676287230893, 0.013333670369433538, 0.009500624154005738, 0.12889013400916335, 0.001817672988689134, -0.01750053152188433], [-0.23330617427708364, 0.0018030747522725609, -0.013214572582952144, -0.26596661572692515, -0.027002803496206016, 0.08428375799756183], [0.3619919973601076, -0.002119191546868872, 0.0027921031078785266, 0.018240383537222047, 0.002569150608103949, 0.0497231007932029], [0.3258178355123667, -0.00527331119147327, -0.0020612155801727435, 0.07964161752714442, 0.008150016077193357, 0.00030371637327230417], [-0.539871829557853, -0.017082043942603722, -0.0025915409442991023, 0.052204655204964326, -0.009719051678184626, -0.034260189082027374], [0.4818472876889084, 0.007050651110576225, 0.010325239779693855, -0.07764582358155545, 0.009444499791583067, -0.006645776357835622], [-0.36422244613693966, -0.00787994781678057, -0.007323217961193241, -0.17262925518320396, -0.011291371399945953, 0.05955401627583929], [0.4452709333533797, -0.007336996183729447, 0.003666860907126531, 0.011221432542203678, 0.0064485197625635884, -0.05987585416665085], [-0.5639850077109703, -0.0005436837628059819, -0.0075647173802254695, 0.04611066157898135, 0.0017676675806219365, -0.01960387208757441], [0.3608317263560689, -0.008477267590647888, 0.009067359090723306, 0.08996595801579792, -0.006352785893610109, -0.015734283640646732], [-0.0420763419494824, 0.02603543339824351, -0.00010852962540733224, 0.09251356340340747, 0.02296017063185185, 0.21583494784490748], [-0.032309053369972786, -0.036267220555198924, 0.003588123285501047, -0.25843077813196647, -0.00898781892424364, -0.08672872849459483], [0.07750011973153338, 0.00881566538173723, 0.0012333720689711141, 0.2194592978516378, 0.006757944953638019, 0.0883897904886732], [-0.22564850415526633, -0.03343696631436705, -0.01133467868273693, -0.29574747589480765, 0.007447246521022321, 0.09705514043091723], [0.31899152874555364, 0.0075988279606302445, -5.3489503293751634e-05, 0.12259433025671235, 0.0056206598652649275, -0.023988523991534786], [0.12474011359248817, 0.01388098130922772, 0.005169600941140846, 0.1759099376920221, 0.01241226863471755, 0.07562647083444922], [0.24238261235018396, 0.003874488133423153, 0.0007490820661405177, 0.14554980361334197, 0.002317468364412316, 0.03944251038477925], [0.399618274757493, -0.009344571516013268, -0.01600784640855175, 0.02244926772011831, -0.0032392500577590358, -0.029028017352433804], [0.36989837112877166, 0.002905844000514369, 0.0029922176153172257, 0.014801575848753568, 0.002897575961696501, 0.02398203449256345], [-0.20001503635438053, -0.04539154393546586, -0.014520051327745443, -0.0641738533677702, -0.003684608172003775, -0.17057765576038442], [0.43004591644575435, 0.008543960414980574, 0.003970907445236963, 0.006550888621605182, -0.006001052146211476, -0.042555620781369645], [-0.5160016960723414, -0.011792785363346114, -0.004049294074059194, 0.003123992895983212, -0.0008761656148732552, -0.026724051771365095], [-0.03254695171658328, 0.016897040559194488, 0.02371046548210642, 0.37241182136033735, 0.022271712711719856, -0.018385516968204], [-0.16640480214837802, -0.011305123843364889, -0.008499689947738864, -0.3144401207976211, 0.0048938856167822726, 0.06886442254889376], [-0.5116436231708816, -0.0018908137969944829, -0.004866677898572778, -0.06920304686967531, -0.0017169114305095716, 0.040024882690442604], [0.4800364595760646, -0.006374372410929828, 0.004115433038894646, -0.05280659901725639, 0.008528224936281852, -0.017485812789724012], [0.37276132341990764, -0.004913808629848401, 0.0003691341054889481, 0.01811955821310811, -0.021717349969068205, 0.04691034920961531], [-0.33644988119730784, -0.022338162836889365, 0.0009853828884132231, -0.0615248586235546, 0.004315889023888809, -0.10600281369899602], [0.3274133907337551, 0.006390592804594291, -0.0002530293474671896, 0.06069277373487993, 0.0010834637498998879, 0.033448436739635976], [0.40695972116416235, 0.010864533255826283, -0.007847466921618746, 0.04001615511968695, 0.0026872304930218595, -0.0348335064444146], [0.39680688481008625, 0.01058295488880826, 0.00026787461337617584, 0.033591194642983775, 0.006049375538323754, -0.013728347386663288], [0.3415458864556571, 0.007900060216107894, 0.0007338083618347122, 0.03300339682632141, 0.0042097350479094785, 0.04190357484080131], [-0.5579801102269661, -0.013813118608598119, -0.0064367760889865455, 0.005910067597729754, -0.005327283555690071, 0.020702220882505794], [-0.09482063478632458, 0.020071184062310482, 0.02471548456878347, 0.19891814169476252, 0.018200337229504072, 0.06175133138680972], [-0.18170339568613877, -0.0040693268345313605, -0.0009487929158804491, -0.09086988635601514, -0.010038347889760471, -0.18553747253989497], [0.2793748441727545, -0.006376917586841521, 0.006696146343886843, 0.13027828334063696, -0.0018236306258963054, 0.014281274355459578], [0.10558908666253793, 0.014543059940861545, -0.014124630260707665, 0.2857433550002085, 0.0067058342494110065, -0.03560502639431552], [-0.5871862137987519, -0.00019318725037142877, -0.006226687944325247, 0.04277387642466744, 0.002915943603665693, 0.031125091020248973], [0.4020030636955561, 0.004889629976612478, 0.003215487903089674, 0.04111922443343225, 0.009608638814626876, -0.029908964859258102], [0.23944160322762556, 0.01987195440814158, 0.01329536562664879, 0.14970304121862296, 0.004358618877188335, -0.015399347550010862], [-0.45682656476030636, -0.014670751128250475, -0.00954017460557796, -0.11102841924249057, -0.009299848538177726, 0.04204575827480155], [-0.48101911488108523, -0.0023828202108030147, -0.0018360647190366743, -0.03144305147214482, -0.0006640487424397131, -0.039391566641159254], [0.38127155905543486, 0.00916093250964186, 8.916353802088274e-06, -0.038557053825233625, 0.006329022046111197, 0.05947852862214916], [-0.26590794767843984, 0.03128857888735016, 0.001586931757253964, 0.15187265213145926, 0.028322317530217384, 0.15132203086422474], [0.4753530709858403, 0.006803685417858833, 0.000730308063770979, -0.055209474476273855, 0.003702594503670861, -0.017839073383758564], [0.2609484691815082, -0.0002500433581622316, 0.006961534219786811, 0.0493964157939036, 0.008345064718338062, 0.1066118927779595], [-0.2953124271763454, 0.009659029722556113, 0.002879674184806878, -0.22283070110640768, -0.0015078391071928377, -0.039707736517417554], [-0.25547559446594015, 0.005795367985987342, 0.0158983309800166, -0.24565681111256613, -0.00031635032270745534, -0.04481494306479075], [-0.4026508548649401, -0.00016738238863067136, 0.008804870037806339, -0.022826470807760683, -0.0030259106446999568, -0.06712091799844376], [-0.47702067217596095, -0.00027864421264686755, -0.0069308792011212065, -0.1142343329523761, 0.00016515152467528986, 0.05522937701742841], [0.0025039614170040175, 0.017349732672663345, -0.004609536623338456, 0.25477213556026396, 0.005590376395984219, 0.0993439028414173], [-0.2625681237999639, 0.002911590621123185, -0.028702205371152555, -0.2777044691896008, -0.0015815701720038617, 0.07049144457826524], [0.03553752324492312, -0.01222035762073839, 0.0069477437762504295, 0.1436157569501687, -0.00640242020363164, 0.12388658291285626], [0.40637537455341205, -0.0012328593320265595, -0.02765747575813613, -0.0547410781612301, -0.0014808504119136778, 0.06615498434798878], [-0.5641069922779384, 5.3603942047838715e-06, -0.010806944953861674, 0.012131277271268791, 0.006021602177999621, 0.026191649769275342], [0.1811714725297982, -0.019278580706614083, -0.0011496804816820306, -0.21970769371820373, 0.024951287559229793, 0.18433902815080586], [-0.49962697096558667, -0.0023247149819308546, -0.011103488334757309, -0.04784984158757159, 0.0011173942473701574, 0.03341525022788553], [-0.5015852737898536, 0.010077446682044258, -0.0018629572595423001, -0.08844834693117991, -0.0021210620454967474, 0.04570352667735769], [0.31633175938853847, -0.007702990739754895, 0.00560676681084354, 0.07817762026757553, 0.002789648805428286, 0.03817416516433813], [0.34993056020803404, 0.010403649593190884, 0.0007368175926672601, 0.08665948628505425, -0.001450918645650186, -0.04086658433330551], [0.14881234080881686, -0.053791029227465315, -0.009866473702115002, -0.27824543301665666, -0.028309951407008177, -0.0903242153603348], [-0.5745833149636501, -0.0020940651130729837, -0.009624665711456678, -0.008132720146249363, -0.0007193624694725716, 0.04193087981632656], [0.48396967978569644, -0.0066569292954416345, -0.006543726508882061, -0.05589750259055515, 0.00930225943545751, -0.027131035728238478], [-0.603398286443951, -0.0029823613179290843, 0.0023209635492396925, 0.05143332945519561, 0.0006751601670845364, 0.04079493728041307], [0.08385601797730703, 0.004821078509661953, -0.002590252900547737, 0.23638576268952516, 0.014681741476278303, 0.048912015884139605], [-0.2070393688495846, -0.025269109468898248, -0.0038570121820915106, -0.22469469764181071, -0.00333908315623831, -0.08284691917756674], [-0.5356609505860958, 0.0067982284104140915, -0.0008977337919929319, 0.06612849371269944, 0.0024968043288692166, -0.03062863702439627], [-0.021721816986349295, 0.01330473906009037, 0.0024345535739844775, -0.2512099422667751, -0.017536450884976774, -0.09166052694041783], [-0.5579147428418941, -0.011539937136212168, -0.007948174780675988, -0.013701267563841928, 0.0016287008799665212, 0.04969017126018598], [0.4009948443826737, -0.005791109036445856, 0.004018813381892292, 0.03897521054332678, 0.000464212890408704, -0.016818225531130663], [-0.5802667539497656, -0.003159336310397301, -0.011720898485095756, 0.13594125535894735, -0.013142436038453451, -0.02686364344073574], [0.37248006201265804, 0.027335512978732508, 0.0048231127117867156, -0.059795565714561065, -0.014276381167014091, 0.051547782987920565], [-0.542878972435013, 0.007224282157414232, -0.004891059855273255, 0.0556384188817224, -0.0016648847902169255, -0.04500968872054188], [0.24405293016182997, -0.03384484354395847, -0.12959509943995168, -0.22576427693609558, 0.0010986869408826827, -0.08719398448429358], [0.2536470938517896, 0.06584338954648625, 0.0371131386909082, -0.1634016088387709, 0.02267755759350351, -0.03897139624074312], [0.41738455561616417, 0.0034514851161710517, -0.014182652818911633, 0.017798645052421132, 0.0004068330524573287, -0.020428866018304126], [0.4177649294804218, 0.010619376646886402, 0.001960173867905904, 0.01460395441571566, 0.002485862767615731, -0.02436436007163071], [0.05942636911890108, 0.023471411483083283, 0.00020310110048030031, -0.40815514326707686, -0.00589372812079994, -0.0972103941529698], [0.04908182511549627, 0.02669806619358184, -0.026038662966930564, 0.2858189345781913, 0.01594301311709366, -0.020205068268008754], [-0.41204594188594534, -0.00782582948250374, -0.010380346499822387, -0.1692534850697804, -0.0053979734858050354, 0.048702624042903264], [0.2809845279244487, 0.01189828907853653, 0.004897968922125496, 0.041803982476861595, 0.007117830206078792, 0.0768732347252817], [0.32982797180618795, -0.002716031592999523, 0.0036471339249611267, 0.11567718890974428, 0.0027471459077664535, -0.014670075622326906], [-0.0069354770190608326, 0.015692269359405115, 0.0008054916158734332, 0.28534725928660004, 0.018690913504165754, 0.08853192420539915], [-0.1640405031191244, -0.03462164759197082, -0.003280024629239599, -0.09145142899312453, -0.003939413533777778, -0.14588128227706318], [-0.28200300971596864, -0.01801025860560254, -0.012652173116734746, -0.25954068714414885, -0.016252631570709383, 0.07242447443887962], [0.28235584730453195, 0.007930993373803989, -0.037165972042056794, 0.14014609497898214, 0.004414297100331767, -0.008603391041407652], [0.3440223529587625, 0.007726401474829806, 0.0023136281361269477, 0.08945459657127967, 0.0030746559903848465, -0.016091459692788905], [0.3462038169394698, 0.009377617900680419, 0.00506270943121622, 0.021940624287847068, 0.0026196597695547982, 0.04316307167123086], [-0.5158073460598203, 0.005765495830559435, -0.008929779739091648, -0.041587224499403, -0.0006051719300710505, 0.02969673019120272], [-0.44975409221759727, -0.009347243235364911, 0.0014622033287171794, -0.07124144232055234, 0.0009354345991179532, -0.030666526820986303], [0.40033401686270326, 0.013334726383628847, 0.0029831493079832444, 0.03788444548898297, -0.006928365755016455, -0.01801888137919489], [0.3330388323354488, -0.0045590255760695995, 0.007842746462301162, 0.047442169599464575, 0.004548612044685132, 0.04212896021613601], [-0.4170644147855438, 0.006041593969310607, -0.010046383091452112, -0.07460506722649383, -0.010723856373396448, -0.044157983603538085], [-0.5030180322256279, -0.010982548402424021, -0.0012899454134385091, -0.019327127652439807, -0.0009612259790011382, -0.02990778699373702], [-0.5603212548808727, 0.004477550791268083, -0.005757787784016821, 0.043711569652373945, -0.0023850695291413706, -0.02786719235481803], [0.439116097026225, -0.003811149235862142, 0.016295867023580716, -0.12418833721876306, 0.006941995342711861, -0.0031953062712269547], [0.33087300463074687, 0.005125675443760313, 0.006983493825491963, 0.028798718147160226, 0.006618430141658026, 0.05611401114451442], [-0.16874488799697202, 0.005838921999765942, -0.00789280677989968, -0.3070717223299549, 0.0024597663060670957, -0.06899260453233974], [-0.04087092365669104, -0.012240896315914374, 0.011255018333450434, 0.09044984255356527, 0.018019921031157626, 0.20983876011293123], [0.22267590362844478, 0.015098505404606869, 0.02527040587271699, -0.2842292222953212, 0.010135390701583123, 0.17357804266199578], [0.30437065631523597, 0.0031815614956702313, 0.006822507145179791, 0.060509749850370276, -0.003584485635869877, 0.03396334416274493], [0.17936079573455116, -0.009613910512334246, 0.03813176508164355, -0.19406513946608087, 0.008401646244790166, 0.16034182704441333], [-0.4285480597040233, 0.0029869013535858894, 0.00035918196087694407, -0.1605405788859989, -0.014262588673549162, 0.04785181061577355], [0.40339677599770635, -0.006361430410050201, 0.00551788784787765, 0.026635522791620465, -0.011637490504206243, -0.01719269429438044], [0.008048610581037188, -0.008057588562051668, 0.0023225843609035314, 0.3836246098967119, 0.023504948800805437, -0.05717983174407392], [-0.5955870714371698, -0.00332763748635488, -0.0022156413088651322, 0.04209819311766807, 0.0021428359718831252, 0.04140273151302064], [0.3115693433960662, 0.015423538965112232, 0.02361670197618657, -0.2562987068427123, -0.006096387667783689, 0.11534249430011599], [-0.37148836580394995, -0.01650107545623901, -0.010106703103393282, -0.1326588054070097, -0.0016308902421740543, -0.028934159987235454], [-0.5114011731274702, 0.0050362336263186465, -0.006103498463971465, -0.016796383195713974, 0.006879356394960213, -0.026601201900792883], [-0.14381169510502306, 0.0018726765391486977, 0.006559026372078235, 0.4042568811402878, 0.02076680789478614, -0.032527188904770216], [-0.4093376362407912, 0.010285537792107285, -0.01176889324622036, -0.09324204972670311, -0.005933164729857001, -0.042573793848536545], [-0.41260465295250476, 0.0010609907676181367, 0.009263043951217943, -0.10768392022931779, 0.009095892812692569, -0.038701354349706854], [-0.06168423730265042, 0.01113387357537673, 0.013693422141164204, 0.3331109813088154, 0.01019356193903865, 0.07770963309180709], [-0.5604602930113376, -0.014734385520614247, -0.0034965021174970697, 0.009189241430298067, -0.005217510045330867, 0.021453020693047994], [-0.4022705877159267, -0.01977553695718354, -0.001558076418860314, 0.1330699713614549, -0.03243556796593033, -0.07205095987931372], [0.2756419254299637, -0.0014057525855254126, 0.02046038376598639, -0.06409373345397873, 0.006615218078853223, 0.15312862543136674], [-0.12059407771388038, -0.01856148039578653, -0.018031676730698963, -0.03907277464157901, -0.004260063625915777, -0.1364040033712151], [-0.020655207184833387, 0.02683286391420749, 0.012815783070285176, 0.36991394121875887, 0.0021059927011837514, -0.03481248817240239], [0.3576191322967907, 0.00958319342645649, 0.006041136198829891, 0.03483888246076207, 0.0035298425618665807, -0.030205996468516613], [0.04430250587584062, 0.01898632853203455, 0.009795461662307103, 0.27963987147901576, 0.014691855905568737, -0.03233522980397236], [0.19293507029817153, -0.015072587015114585, 0.010447102384702242, 0.14142013876074946, -0.003172664254721918, -0.07144390999063652], [-0.36994601090374546, 0.012676339951032378, -0.004343831355730034, -0.12430866287961773, 0.005654562238815486, -0.05498692086028062], [-0.571654818744528, 0.002495548193394581, -0.005864809994835365, -0.007366632373503684, -0.0024972724237360606, 0.033643903422301544], [0.00830500692572435, -0.02538723338771718, 0.013761654299409258, 0.2834322353444081, 0.02271265082794729, -0.05640026639072459], [-0.5299481052982821, -0.0074273505927278365, -0.004807394287761119, 0.0427453243674792, 0.0007997586274143758, -0.05006859645248884], [-0.4263721842986988, -0.02006592513260365, -0.019843247377018473, -0.06238230736034511, -0.006006201037114482, 0.07960733090392282], [-0.12057194191828155, -0.022736754836153266, -0.010011441749732822, -0.3889741296944693, -0.017645606979050075, 0.0970424942253065], [0.1895260460266887, -0.024177002736299844, -0.026900954931161805, -0.4490376866140882, -0.010650004947555143, -0.09951492060710691], [-0.154225760634467, -0.016729307703761097, -0.005906495570881881, -0.26642671297304016, 0.011826118480796488, -0.08440184253235186], [0.14205102658315175, -0.00957036102307133, -0.011487367375339237, 0.03494242835673534, 0.010377469635993678, 0.16033108953681643], [0.36258314539516623, 0.005784042567071146, 0.0029690344276450246, 0.012757152951883947, 0.005181795195585584, 0.03698074754174258], [0.43045181697227014, -0.0067737975242682, 0.017292060855529336, -0.12509671554818572, 0.015343174960927021, -0.016402961284901384], [0.0013355206415153963, 0.03650250837343719, 0.0016462130859643436, 0.32083796772946543, 0.016275626100101335, -0.013926309912641635], [-0.4258678071352199, 0.0037334713383826387, -0.003725579039576585, -0.09499285307783707, -0.006635005005798783, -0.0337766715243968], [0.09519287180957353, -0.03735429761612263, -0.014398531741279827, -0.30793855885912647, -0.03093384070975747, 0.06764888309073873], [0.37104777858883636, -0.004331115013652934, 0.0009666430275027533, 0.02667118135773254, -0.0056192224979621395, 0.04002806787087662], [0.35968041004965884, -0.008755175495920912, -0.009188411620040382, 0.09832537710813703, 0.0052697559022424605, -0.0306102892774132], [0.31409133532394096, -0.003970613687142302, 0.004935742929223118, 0.1329300071729797, 0.00637232041591549, -0.01984545882158352], [-0.37789626367490525, 0.012512005301991029, 0.032288133803264514, 0.1190649397557267, 0.024372964516669877, 0.13612195045597966], [0.27265039967995774, 0.006428731351784307, 0.002045091570249237, 0.10877891449520226, 0.004805566993418737, 0.03813796257605496], [-0.39130338732087483, -0.005505921158985655, -3.0489546921230837e-05, -0.10526371295310598, 0.0024865708784875566, -0.04195305989860072], [-0.23902019781995326, -0.009890276172104848, 0.015018174241171092, 0.413800457515074, 0.022358529993556694, -0.017045021091077563], [0.0531531394066561, 0.0038501472211699118, -0.011399879442470194, -0.41819863630045107, -0.01780120967355548, -0.12020133898912842], [0.4326940700016956, -0.008717221321781297, 0.0024440506102878135, 0.007418374864258257, 0.003512105205103918, -0.02219482813301682], [0.2720447359654076, 0.016461157140535795, 0.003586851987865027, 0.08531407020308875, -0.0013205636040660802, 0.046864581640502306], [-0.17548335674625617, -0.023314537185232024, -0.011789935594609804, -0.30393950336849873, -0.0025822012850575775, 0.058664534179654594], [0.46946933559252485, 0.014838802367249205, 0.002823080537594234, -0.07725562055267984, -0.006675150535750081, -0.02171092359941758], [-0.435326697969349, 0.008808844732927673, -0.005211227622648936, -0.08115466924928343, 0.0034702158791593265, -0.04253146577080636], [-0.2981526203533502, -0.02762920111350267, -0.012041975691296997, -0.07508509786033178, -0.009146131315739254, -0.12006800396881069], [-0.022715849107021967, -0.008946463420549363, 0.00481040625571959, 0.3131396238139503, -0.011072373599895682, 0.09446465605779725], [-0.4558434479594862, -0.011851238205295716, -0.0027819658150950665, -0.04267573318249144, -0.0021633213667091003, -0.046420960137589645], [-0.5539808294104775, -0.0038099292779022933, 0.020480482422698645, -0.022564505782218885, -0.0014219384370046655, 0.031615138563995775], [-0.11392515063310459, 0.010712739322081978, -0.03017668173510033, -0.3749132328265518, -0.015832191397665882, 0.08527880298462735], [-0.49267900154327404, -0.0003485591401775683, -0.0037824971876367886, 0.025342996155173374, 0.003619800851804965, -0.040527246473417815], [0.3412632186185986, -0.01132033930592516, 0.0010832283552881718, 0.0692355828938071, 0.002000571753295286, 0.022435281544583616], [-0.42957995571894697, 0.00018781816059182068, -0.008506321535607672, -0.15010273972736576, 0.0005707172881687093, 0.029443814866492633], [0.14186305293565998, 0.015394194489769853, -0.02355215638423972, 0.07168542470542785, 0.01284036483384733, -0.029821623726207294], [0.35503445926240373, 0.0027471837223082742, 0.001196809753555448, 0.030726538146770784, 0.0036004202882723903, 0.03615442776830355], [0.3203094132538211, 0.006898159204382787, -0.014315465800333785, 0.10146032662984127, 0.002551519733054332, -0.012364717668233811], [0.42330823036276066, 0.0015758080586047174, 0.0043396135231201696, -0.04362985606666724, 0.00024355049226881646, 0.03034265362991251], [0.41939515558509083, 0.015081462423215411, 0.007830252328411318, -0.0791725974198343, 0.012271508485001522, -0.025855818031921506], [0.3741564378163606, -0.0056209341849234, 0.002235865916196448, 0.006818361416280629, 0.0020947341909332315, 0.05149553484515048], [-0.3337339804628538, 0.03663664195652904, 0.011919260707820033, 0.28508290170459055, 0.01806324983960249, -0.012262099719716065], [-0.005021708278088468, 0.02216526781775637, 0.015383010196143052, 0.2665774141983135, 0.001616537385199994, 0.05917877692629147], [0.4214012584548163, 0.011747451681257133, 0.003556970460876281, 0.015505881061645466, 0.0022811735931443393, -0.02416458710359453], [0.2388347851055497, -0.00031849464618195056, -0.002309500748536553, 0.1369533842382584, 0.00542463086207678, 0.04919607238181784], [-0.0025764524914784993, -0.015056650770433175, -0.0053847281890395835, -0.4024382349641255, -0.0071494062581265935, -0.08843278129505072], [0.34237010801956386, -0.004357226247194679, 0.0036110429619525103, 0.06571341883402718, -0.005384260247499673, 0.03033802779026034], [0.11882330092684334, -0.05893308137672882, 0.005046968976060164, -0.2872708113849345, -0.005719756858763858, -0.11901662028247659], [0.1755833358272583, 0.016865906891957376, 0.0005023171164904446, 0.17906032681142, -0.02791396390624231, 0.050236839163878805], [0.21735326637644645, 0.004624281995670918, 0.030520762659418048, -0.16989684950779854, 0.017438153368984927, -0.031216757749864897], [0.4173755690821047, -0.0007145688123989758, 0.02104267425368519, -0.06865111753447978, 0.014745219344167783, -0.06116539538070008], [0.3770650694715065, 0.008708150153237355, 0.002350884268572831, 0.06842138459081751, -0.004707675294236751, -0.017324479856565843], [0.28570692880589593, -0.011707562661090815, 0.0038342543773174018, 0.10036056968427659, -0.005771129608590727, 0.048872324017577136], [0.4146671517140139, 0.006090623295174458, 0.006880336875337704, 0.020246734068869905, 0.0018948072719462035, -0.06730203417772501], [-0.20540424049075875, 0.030198093313001186, 0.033411540494777126, 0.08030686255623383, 0.020329913183707804, 0.1657316801493877], [-0.5679553378663774, -0.0045376953815162435, -0.005335776056418579, 0.029644225325096776, -0.005867994431416282, 0.03848257841063062], [-0.5344472216004764, 0.010163552289209704, -0.006699988962555252, 0.006045355167360728, -0.00011418174245838858, -0.02780434572093383], [0.20531233072260213, -0.016260947924533788, 0.0105423090823365, 0.05729257972767617, 0.010102782598689861, 0.14446475531703812], [0.3139191218242642, -0.005774905969472477, 0.004949339484123767, 0.06985781864155823, 0.007597489111731297, 0.03797676532309407], [-0.5100354358656899, -0.013555803588158055, -0.007032810848766102, 0.19235395892788215, 9.62100602182013e-05, -0.03468778535215634], [-0.37575516608431875, -0.006724157921714019, -0.004288389855192097, -0.15085717933328244, -9.595900152521482e-05, -0.02526581447063582], [-0.44367745481897436, -0.0060218114319767835, 0.0026543339416891373, -0.08184916921300542, 0.0009951566613648855, -0.02719089640893813], [-0.20249400829053946, -0.054825284166109885, -0.007211353787471317, -0.06202729922833123, -0.003194788674209173, -0.14136138839746157], [-0.41046927218202495, -0.013648362613534502, 0.004012396073622568, -0.08550422656805438, 0.005644869427166289, -0.040635166041938384], [-0.12002459051429548, 0.0003692450742751583, -0.016231432828383716, -0.3753200557120614, -0.02428675379566423, 0.10066763539517769], [0.29553885808080727, -0.0012436853150960378, 0.00320112745758698, 0.08800086663907263, 0.005874286211226477, 0.04314188025973554], [0.23201626370278952, -0.026400996410788805, 0.0019444154479002693, -0.38648582253513075, -0.023682981291221577, -0.04558191065957798], [-0.3768601141518556, 0.00783298850282314, -0.004810679240088496, -0.07672891914725063, -0.014260430582585026, -0.06846109934930047], [0.36246130981582275, -0.007832078490889727, 0.0008616414117852607, 0.09216833238241273, 0.007602201177749973, -0.030683970399447816], [-0.5172480845591906, -0.0073231500398205215, 0.00038691567260689026, 0.006045661626027821, -0.002378484582320558, -0.04021952478397315], [-0.11257495997209621, 0.0010030046379825902, 0.018802133790145958, 0.016204422050608222, 0.0049643652099177205, 0.21578103428344184], [0.11811718972410824, 0.05481262535622891, 0.011802228159457544, -0.19594035183273123, 0.031204430572872255, 0.14824340182958903], [0.1740421650580433, 0.026915977385980384, 0.005620481729302935, 0.13026298849208087, 0.020312590174410266, -0.07034216820778197], [0.1303483892355446, -0.04810254582254666, -0.007360334020065004, -0.27109109107394047, -0.0203977281825538, -0.13971272188247139], [0.40884305306422325, 0.009475638803930547, 0.004114350448773427, 0.032663007873626956, -0.0032920880981060544, -0.034567358318867224], [0.3298014157890717, -0.004295021394350809, 0.006765535251114471, 0.04346010739118364, 0.006449139919682548, 0.0498321563766302], [-0.518788344520004, -0.01161004925888457, -0.0034471035095861476, 0.0018531900649792663, 0.00021731441918965852, -0.025878340529030687], [-0.016428280645369026, 0.023460053940931237, 0.010415931241675877, -0.0722242978850495, 0.0019161835849884315, 0.1870562827786962], [-0.5392885432049257, -0.004684605512323055, -0.0021775583192023148, 0.1606379708426561, -0.015087671306274085, 0.010014534484193663], [-0.5689027270148629, -0.0032998484166241707, -0.007227753469340041, 0.01227404615541503, -0.008405493998843942, 0.053460030712506754], [0.2376994923556652, 0.01595680837498864, -0.00044429338429629353, 0.04493071951193615, -0.015151979657852052, 0.11666544327574889], [-0.4399939282233572, 0.003194782159123298, -0.007104636109135441, -0.02437704514824352, -2.7029061344644898e-05, -0.06479785790275865], [0.42106925802825157, 0.0063785698168999285, 0.004463884333568196, -0.04576927074838256, -0.006744552314605357, 0.02536544421760088], [-0.5015906851844085, -0.0008032795812771479, -0.004371782936741677, -0.01754852592172419, 0.0010355619535684548, -0.03345795499608571], [-0.5165485899321154, -0.01688024184154721, -0.006592266012716691, 0.20451058361697072, -0.016805352938452926, -0.028295799558807296], [-0.28203623994947635, 0.007235270442755142, -0.007671957068930857, -0.1499852302081547, -0.009696989929213968, -0.0754402861874131], [0.012593404813372967, 0.010564574685832679, 0.02342894713050949, 0.05422679602521827, 0.012345594506961866, 0.20918933363175443], [0.2411158560505946, 0.01983047675574239, 0.014355539233263005, 0.03508280172056724, 0.003206416416371172, 0.11277640982346146], [-0.4096358145271023, -0.007846537557632626, -0.00930317614508883, -0.09452653597883608, 0.0006018946882249673, -0.04077649714623338], [-0.4860913150735294, 0.009170227738647946, -0.007725727795439425, -0.005949530741586257, -0.010730159683219428, -0.038410161111544786], [0.41825158003867374, 0.014099986858988533, 0.0019399031301210285, 0.030295361757788523, -0.003693292455666912, -0.03221353932990869], [0.2927295611715688, 0.007774428814052092, -0.0045044081748033486, 0.0883190834564858, 0.003002424841353816, 0.024161940194373673], [0.3411384770144489, 0.005714422974300731, -9.544530770453551e-06, 0.0568627837908972, 0.0018484094973724534, 0.017571025416429523], [0.3084161154061685, 0.007450363949359434, 0.007437827832017529, 0.060185085781244735, -0.00671774793677387, 0.03819040625003327], [-0.4800615211295651, -0.0016012609647050717, 0.011765541124953935, -0.072950677789104, -0.007788818236683084, 0.024483403661768124], [0.2383092599607887, -0.004080526282567174, 0.002732077226724974, 0.12241650926564669, 0.005936571427717779, 0.06391362411098332], [-0.23663355139379572, -0.06282733086043046, -0.02693905351776236, 0.10774587246786961, -0.014593854271040149, -0.12007453830229693], [-0.011749657768196977, -0.031857578831844424, -0.04221507621350509, -0.28794456684748343, -0.020431012227433886, 0.04180844744401944], [0.35733306762359335, 0.006179482770930563, 0.0028225158236382104, 0.010030063109848237, 0.002895785201749692, 0.04991167021600086], [0.5057140894666514, -0.004359863357100048, 0.010059011956557005, -0.10079591184032775, 0.009682456729215196, -0.031717997240713565], [0.32890120371051995, 0.012595378347384733, 0.010716708282466262, 0.09484881007979412, -0.002093302138566112, -0.011541718317538483], [0.32496615667871515, 0.01097065770258151, 0.0026637367492895483, 0.09606053991801947, 0.006513330944673532, -0.012470612469470494], [0.26332507675231637, -0.004584469408750028, 0.0038866685171128714, 0.142915485493922, -0.006599425797820699, 0.005236664443219561], [0.35640398072633606, -0.013987050810205405, 0.004022963958791681, 0.05718419428483396, -0.009087275132352393, -0.02704728921788211], [-0.5781678685342985, 0.0023149918265716516, -0.009003397068522791, 0.029486142169455592, -0.014550632853064589, 0.04261862160271296], [-0.582600398239693, 0.0029254729572569926, -0.011678070499699085, 0.07332994344117068, 0.003822970287491489, -0.019028824829122874], [-0.18937942704934008, 0.016815147622048457, -0.024630489923201507, -0.20465572880845104, -0.01740892952561262, -0.08722723898210961], [0.01937805006337802, -0.039617034460232195, -0.048142583077351614, -0.3006554911785681, -0.01194412243354215, -0.11066619986606442], [0.21463012181453048, -0.0013648071212529742, 0.026885723542300233, -0.07856009590398501, 0.0197940564316153, 0.15497555679234754], [-0.462097781414694, -0.010861628666277778, 0.026464350284905448, -0.04651449756500482, -0.0009800012457690714, -0.04251633882905848], [0.0518989131856915, -0.05027428367129622, -0.013522984353853754, -0.27223142202303047, -0.005989553255956103, -0.10540900321488883], [-0.4451228247372625, -0.011249206250969965, -0.002492757114057657, -0.07224000534477779, -0.0017475837815264527, -0.029420003723785722], [-0.024365912678419997, 0.04421807133199023, 0.026347922462685986, 0.26915508056555676, 0.021431004657722345, -0.019223121463192782], [-0.5096732274159244, -0.0123554272338271, 0.0013418094380017078, -0.011792048642997448, 0.0029519035684934538, -0.02560253352327101], [0.292471629075931, -0.012004048802284398, -0.02040056304233666, 0.13656583342598583, 0.005817625603659286, -0.019277364550862798], [-0.5533976330087305, -0.011157780585849902, -0.011388239287229106, -0.024498581965244712, -0.000986992355874229, 0.05352589386958983], [0.45460492240154937, 0.009668130094794744, 0.004574469885124916, -0.02819390533848771, 0.0057112829826947925, -0.01601823335901143], [0.3782299542677168, 0.008703369419349882, 0.008436455681813665, 0.05551034835032215, -0.0008095125798026704, -0.020200138948926735], [-0.5204006621200841, -0.012513609718604561, -0.005160987584887656, 0.007320466211336339, 0.0006093641220869541, -0.021110468345746602], [-0.5486741392883349, -0.0168358581073054, -0.013753225733850221, 0.03783612859569545, 0.0032856741183319433, 0.014541735547550342], [-0.06601610238131167, 0.02102476733568882, 0.01597152720904537, 0.3720585959533063, 0.011712739886636563, -0.027858723264246384], [-0.4589344547479094, 0.0005248928607211508, -0.006681176205236207, -0.07134922694424065, 0.0015516626986154598, -0.029487253217506792], [0.20194515198104762, 0.002805431549582266, 0.01158420572260507, 0.17223885324465324, 0.006343665652168872, -0.10513749130024048], [-0.49189597991196277, 0.004986778648012319, -0.008017654597391021, -0.09762030585444674, 0.0012873011831082857, 0.03614819386600902], [-0.5563523754846065, 0.001843194083654867, -0.01038483802421485, 0.15762593866693872, -0.020377468165324993, -0.0440236574256567], [-0.48685933062554804, -0.007937490558904877, -0.0046876777154648, -0.06674800292596249, 0.004566576428040595, 0.021804258731170766], [0.38025972851348755, -0.005852637099469247, 0.0023955054026060527, 0.06598810970999187, -0.011676744656694037, -0.019541104727069592], [0.036275084194041296, -0.03953010199913926, -0.027313334079637767, -0.3032530198094257, -0.028608520280065778, -0.09091788580355115], [-0.2567720124827555, -0.0320292458418057, 0.004149920949156319, -0.07651683710765798, -0.0002245933801056851, -0.14296889880349897], [-0.5560183361803585, 0.011348450133080116, -0.018114868148373053, 0.14439542532213137, -0.01026778128602502, -0.04104781047537686], [-0.2546175328380763, 7.139617512594896e-05, -0.010723977041887956, -0.1788628870035992, -0.0035146518221424897, -0.07502353794561066], [-0.5763163509416727, -0.00385506170556287, -0.018186101856830075, 0.15476544379033597, -0.010825610453974772, 0.01740720497722291], [0.4214627273568447, 0.01004385502050236, 0.0012213478474831787, 0.01654171270907213, 0.000518907326964132, -0.0307752169275372], [0.2685729461033486, 0.012878685246665522, 0.0012236797115670042, 0.1077827315304881, 0.004019649111991185, 0.039838273208221586], [-0.5686475668188203, -0.009976592340865696, 0.013085544387310258, 0.012800793240594499, -0.010791474685507294, 0.038783014166000794], [0.29085993131655613, -0.0018290097264873255, 0.0017794618090980145, 0.14000665396080111, -0.0044007370751731134, -0.029281548363061776], [-0.04753622421583383, -0.029560182175979374, -0.017527569071686146, -0.30433290018372716, -0.020707711429162876, -0.12226652403472146], [-0.49111927434883074, -0.008606572209126476, -0.006063724681043125, -0.018787845591427908, -0.005794326704954077, -0.02761492313128472], [0.30539424147481964, 0.0060828935216729624, -0.00019312268296677704, 0.08358847005440533, 0.002324121994975493, 0.03595309260679077], [0.41784243254556985, -0.008187215730060092, 0.0028029682886548924, 0.025685715947410027, 0.00017552981236812913, -0.019241994966509256], [0.4109810533108882, 0.009290905962145669, 0.0028871342767723233, 0.03752965415499606, -0.00020847777491081685, -0.032667908580551526], [-0.41528332141014135, -0.0041356682685507115, -0.016866945482857, -0.13233538347181703, -0.011483389137965339, 0.02211804110466137], [-0.49007732510082347, -0.011775472072648941, 0.0006426359138280447, -0.02577269827739483, -0.0017274790975848726, -0.031419185174901665], [-0.5372888454669075, 0.00044652788824909277, -0.011118014864427433, 0.07290902979613266, 8.885344031104564e-05, -0.057069089254898536], [0.2923360085043343, 0.07450959902292709, 0.029011801158884583, -0.14277977226764657, -0.0021426261055351205, -0.024010601943556094], [0.37750127998167327, -0.008477177019506746, 0.004282106272956943, 0.06958847959673502, 0.006921682655118757, -0.021265665149292203], [0.40166993648115423, 0.007195988349983375, 0.002558821369384296, 0.03132418691948164, -0.0026799608625562893, -0.018339944241440784], [-0.5547787431175669, -0.0021776914455415484, -0.005955918922915773, -0.013164654017237615, -0.0020734893370427864, 0.05479279842760456], [-0.3895576067507933, -0.018911966886611852, -0.01144250881579218, -0.06365930926157937, -0.004672934594811913, -0.07064710226184205], [-0.4620164277043105, -0.007830364481990676, 0.003948457230663154, -0.038155189987634036, -0.005587446977150195, 0.03304716239660931], [-0.2294699650704614, -0.027136651373329015, -0.010145008979127974, -0.08181056885575928, 0.004863961822344774, -0.15582414849604784], [0.312780541994287, 0.005990218527978524, 0.003886953760195039, 0.0635923508168995, 0.0022364934074365008, 0.03963788593764625], [0.3151033229886367, -0.008191746323910514, 0.0026250099904104435, 0.037806964578558366, 0.005438114454595582, 0.07589833431170984], [0.3220465393685363, 0.0035464022992752722, -0.017768182070800487, 0.13662595539601086, -0.005608392453597372, -0.05559089396799678], [0.3498405242279185, 0.005597034023412624, 0.005511480069101989, 0.030564418546671066, 0.0006224702278031092, 0.018514311000329825], [-0.5030254022452713, 0.014731288968947886, -0.008387007128043628, -0.051645074652306035, -0.012428882853277952, 0.035077935052803814], [0.36733738021599954, -0.002055083375554812, 0.004025248972574514, 0.08548230114453827, 0.007526294037189337, -0.030889061030687965], [0.32358418612958817, 0.00829312232834469, 0.00407869254133749, 0.05729120522523174, 0.00139285814348774, 0.03529236229292295], [0.2559239306291213, 0.017379014634085907, 0.007357667033055238, -0.26625556671066364, 0.011560062741104877, 0.17318512976853612], [-0.020404877733451978, -0.04381850831969329, 0.010239085987742605, -0.2849733498768601, -0.00567665098388142, 0.055441285053128724], [0.41077400118083524, 0.005751569935821952, -0.02746608626209398, 0.027418641078804148, 0.006504499196826348, -0.01742762513019714], [0.44346714031743173, 0.019295636566416204, 0.006032964335217191, -0.11855713614966228, 0.011597447098651806, 0.04406120973670792], [-0.024245897020424973, 0.016048369655166757, 0.0007647521145973217, 0.3041179230179149, -0.006122332494008121, 0.08559337520294438], [0.03270093048831475, 0.03449496759665595, 0.0070277293398927744, 0.24959830890749385, 0.013808581783335059, -0.04997169401857067], [-0.5056281303756494, 0.016153202114763874, -0.00395916869206016, -0.014593259032819134, -0.0025468800012457145, -0.04532909734632642], [0.16358844066395092, 0.03521615358661722, 0.013520661121892761, 0.04274287926424592, 0.02034204487968084, -0.03006315209936069], [-0.3602150387110841, 0.010636326920658473, -0.004306929862195112, -0.11482924260493717, 0.0014674065387639261, -0.07049297682666071], [0.40658408761626413, 0.018426250596150395, 0.014907927525866654, -0.19869818110027443, 0.01912107246222936, 0.08429122385214409], [0.2520838402574458, -0.015119471836120451, -0.02263911454767684, -0.3530236762277888, -0.01089752058208483, 0.004728323888606013], [0.29882304324844494, -0.006964661532902955, 0.0030572655441160348, 0.1208153339994432, -0.003992804117273036, 0.004441822858171682], [0.3093293237638491, -0.007284957613253711, 0.01078345577585744, 0.1440788727170179, -0.0004126535270512954, -0.023980707783086645], [0.20302007370742928, -0.0026113112440787163, -0.008156107760720501, -0.09645179056171353, -0.0859482635051477, -0.1385356958738644], [-0.5368662872153015, 0.012812748131530648, 0.0017860182813368576, 0.05422657800792184, -0.0020993393071256287, -0.030128130596779778], [0.3462441839906716, 0.012091367254551136, 0.0032202313155149658, 0.06036997258653156, -0.005442277312371893, 0.008686906780485825], [-0.5343540597798272, -0.005060627714745292, 0.03450980978167107, 0.04117615191058327, 0.0017046828828953216, -0.03426117870382109], [-0.3954770902555869, -0.017681280998520073, -0.008347380216447453, -0.16677122611911485, -0.0004982395020898644, 0.029121883758424505], [-0.20097826942406685, 0.0132782146365035, -0.011636496077700769, -0.3292573668659297, -0.01655922839913275, 0.087410527082708], [0.1866906863877517, 0.013624796838947474, 0.0024770605072865163, 0.2106821361411038, -0.006621870386190567, -0.02206636598113924], [0.11350996750757304, -0.0189403856585866, -0.019464159637570654, -0.4090963491297203, -0.02415098931331614, 0.0401433448030499], [0.40832315756328363, -0.012594861195180818, -0.00804712837507535, 0.042081377040608124, -0.0024838092974483026, -0.01945984684730225], [0.40247187998148937, 0.015559698310648025, -0.04109212261808974, 0.022945424979774857, 0.004275408198183974, -0.016271955518676582], [0.42925488041972193, 0.0052180172015974515, 0.0017831298694647858, 0.0021476427609685724, 0.00020632798081409633, -0.020179998232570537], [-0.5834238064871019, 0.0034715430654770987, -0.005562232567763278, 0.012416667564106965, -0.0046452679300423355, 0.049718129532527175], [-0.35567852657834276, 0.0020423647556843676, -0.0087673567568249, -0.12938938339305728, -0.0013976715672037205, -0.053907204238035696], [-0.5348397057521941, 0.006948149726636129, -0.005586413622549324, 0.011835710189275348, -0.001563657565896286, -0.02544741630860878], [0.38128207939347303, -0.005082199770532919, 0.013733365536636798, 0.055755557928990375, -0.008622794652897557, -0.019405239204902433], [0.2854760728585516, 0.0009184195786971642, 0.001636966701677746, 0.10798242017394899, -0.012427798904959797, 0.03942725292541932], [-0.022116128791294085, 0.006926233630663181, -0.0009901425665760335, -0.3249131606838702, -0.014774680941182448, -0.06924973969535897], [0.4412773147135105, -0.01019974213298645, 0.003108525062694819, 0.015303031731899239, 0.0052819153956103915, -0.0359016508313371], [-0.1754518865269694, -0.036733196543848275, -0.014203636903206274, 0.02427696820977585, -0.0006274529536510979, -0.15423252484184893], [0.4235628791068434, 0.010105803713612789, 0.0017654103913521775, 0.014372357310155966, 0.0076260179510667945, -0.03937746847303318], [0.4908013338220845, -0.004051892681722281, 0.0131255744284704, -0.09014468426717642, 0.01220572176873641, -0.05765188640372976], [0.020299536615335116, 0.0019591845674146405, 0.009457802394530231, 0.261771959373614, 0.01693187606001027, 0.10523583146528777], [-0.5261385919038827, -0.00618494530126881, 0.00024113083174622188, 0.0063738236854126345, 4.137597546802152e-05, -0.028212317097001064], [-0.24258534551948616, -0.01137916784718105, -0.011230551517916068, -0.22911529763251826, 0.0032583534520433935, -0.033184657601608665], [-0.15587416168388218, 0.02265438509685783, 0.03513373689747667, 0.1579500972963015, -0.01284039169852278, 0.1702337150441494], [0.15190173311296448, 0.005984761272998401, 0.003068130319323338, 0.19481234812444342, -0.003712754934499762, 0.05918025540924387], [0.08366018711364966, -0.0039222177115577905, 0.010490720063473607, 0.07864310782145174, -0.008177957133032713, 0.16735520746506394], [0.42095118163648343, 0.011351910677670983, 0.0007789502718565971, 0.019043810176279465, -0.014062391477633109, -0.04838346128466012], [0.29579386170358457, 0.006911519259796425, 0.005070976636153503, 0.10991866738383999, -0.0013051119289944583, 0.01092645058198493], [-0.4567779067519019, -0.0009813095547617585, -0.0024637877893582422, -0.08195960878857513, 0.00010288840548052078, -0.023073608854217494], [0.2839438266580927, 0.006368378867503025, 0.004228431224710412, 0.11439163414680366, -0.006731511632987568, 0.0323125740692117], [0.2890798879024019, 0.012865982763846965, 0.001363498705428961, 0.13747326673439178, 0.004661097176028586, -0.013737831262899106], [-0.4778510566343526, -0.008900898850850235, -0.003012930818237455, -0.09296513596048486, 0.003258378392629756, 0.035984977204626484], [-0.14423274507060244, -0.027279975857522842, -0.010566112588080857, -0.19078818457547872, -0.0051285683623449865, -0.13124108021263578], [0.03470233607311011, -0.0015369015157256373, 0.005811510338815695, 0.02154110714470419, 0.010135325363548628, 0.2328163051352292], [0.111371417660718, -0.006922664468206561, -0.01251224939604995, -0.46333824539514634, -0.01246004634020673, -0.1158629739658682], [-0.49719463766152455, 0.006508946642591417, -0.00518839374148129, -0.01793102470524668, 0.00019888499448853916, -0.034879393977469766], [-0.48523805271841647, -0.012500549728093245, -0.00045051450097105696, -0.09482293877334962, 0.0008273286406916797, 0.0366980604134696], [0.4075371519094738, -0.008709834546246704, 0.012758148043428532, 0.03699899540341652, -0.002476255231109892, -0.03678931669007607], [0.4781134113541241, 0.018842277656945624, 0.007161580536188056, -0.07808987460268989, 0.007942125858818103, -0.011551775705349201], [0.3289827738967754, -0.005568427142986686, 0.0031523092800165883, 0.059748842692116604, 0.0026324629994207817, 0.04306537160798865], [-0.26557619628507356, -0.05458664712038627, -0.03925908849715694, 0.11240988102491344, -0.009951479673673408, -0.0007492495546251116], [-0.24737070055002228, 0.012114516894139777, 0.00022208774078672045, -0.17184514642627552, -0.013747396560549085, -0.09410209125680917], [-0.577210632141738, -0.0054218111266187505, -0.002122131427099077, 0.010125895338277116, -0.014930603731745292, 0.04309047356511364], [0.2203368717404601, 0.028194579215012998, 0.012741057886497442, 0.10421641933424282, 0.016559619098682858, -0.01121832878269615], [0.21169361297013564, -0.012173052050866303, 0.00984163503536998, 0.22355907862205585, 0.0028317972755576787, -0.02015640518558637], [-0.2243518379335993, -0.03193762333457661, -0.004516729333063344, -0.21794040734188297, -0.000729974380733645, -0.07934342767614412], [-0.0464884196675462, 0.01159810318196739, 0.022921700130747576, 0.03678414484760823, 0.001800853217092801, 0.215170761147272], [0.2755196976855707, -0.007253998991554812, 0.02234681012714002, -0.03413354184401406, 0.01163796180861235, 0.11936862676980126], [0.42887377417846373, -0.010160767878218036, 0.006238717916565005, 0.014009674335540012, 0.006546184624649811, -0.027629642229062692], [0.2079667409995578, 0.004125002023383885, 0.003155948226811932, 0.1324926615183453, -0.009175617095878962, 0.06894859766111411], [0.020995302011917988, -0.011490394647786388, 2.372775735777078e-05, -0.20272314246789783, -0.017927605871110346, -0.11755899789359243], [0.23630517087792818, -0.011639212042076885, 0.0010283830076473722, 0.047496345242322935, 0.003358923845505884, 0.1367688071477698], [-0.5221719653711723, 0.002061922562575093, -0.002292939134898696, 0.061133608037161925, -0.0044555962715058985, -0.038084064909883156], [0.17145837670001152, 0.023024178881100116, 0.0008247511680648128, 0.18880828359956459, 0.006253779288416294, 0.02261366066587432], [0.3697617501626037, 0.011903325959487697, 0.007285471050859452, 0.059603185781656264, 0.004290442587813683, -0.019417095578361902], [0.41095467684283393, -0.015429015685391283, 0.006307445616529702, 0.03708362225058536, 0.0013055891380575324, -0.03790245802275823], [0.3300917026150023, -0.0002780807555944268, -0.009402969759334297, 0.08163019876845301, 0.006734680628285337, -0.05039711879839922], [0.1936186942623651, 0.00624510933790299, 0.007327558058033152, 0.1236066731159829, 0.0015589188083026458, 0.07097414531851307], [-0.39484710556050523, 0.0019406217869195718, -0.010032860612403336, -0.040622943158446306, -0.004369789221525377, -0.07238074225987848], [-0.0366790745499555, -0.019136073918927354, -0.012503238920875124, -0.34990850027046566, -0.010313294569246673, -0.10189092888164177], [0.32507067723886646, 0.005424541145230981, -0.0006420905198126014, 0.056816800469332594, 0.0022208424357917016, 0.042851583542944285], [0.3912645770046714, 0.012821915469310125, 0.010935552603155138, -0.06455265678363692, -0.002704462315829494, 0.05374840735566076], [-0.3403069281370005, -0.025845693582699558, -0.0004844590098394901, -0.20118075488860587, 0.0003473918697274467, 0.024031396129370143], [0.2501079945076519, 0.005961440519622414, 0.018003857906244185, -0.24839501161031796, 0.009195696566310623, 0.17770483163429912], [0.10196263345318311, -0.005161658913991165, 0.004017087356907376, 0.06373165637451314, -0.0018309220997167397, 0.20587787049577067], [0.3584261445180385, 0.02577778610376028, 0.006316500537225553, -0.08805457315681213, 0.004647063675602893, 0.08865041165551824], [0.2527616266056815, -0.002552411639582359, -0.006934982663630106, -0.4060202020899488, -0.008476908271635542, -0.07669236003611977], [0.3415027014432844, -0.0008577481008720707, 0.007671133387927497, 0.07107170133655366, 0.010861388150814985, -0.02783244998098295], [-0.16043606989268672, -0.03679806956981752, -0.008712924808474508, -0.09820528065385757, -0.012781359575263582, -0.14039820026180494], [0.05985448561642905, -0.04026936878358422, -0.03871485551288109, -0.02182766288481705, -0.024100184673054904, -0.16164048014015764], [-0.5092258095489597, 0.01242207620297064, 0.006572167542257289, -0.05688429548162387, -0.0036463246073502116, 0.061025519226034546], [0.3056174308886772, -0.0014072136211051708, 0.0040712934076644605, 0.14041261067413405, -0.025033399925886057, -0.04461365793142231], [0.32409703306143484, 0.0023373961947239477, 0.0019751887559601457, 0.05557815418695097, -0.02059957089998137, 0.05622433838344823], [0.31725732511746524, -0.013170027131538583, -0.000519310809349214, 0.043933830997551236, 0.008646818265724512, -0.039059906281124705], [-0.38355526968407516, 0.0003861847373242106, -0.026803857141771546, -0.05384904879624646, -0.006231148801682911, 0.030250996829307798], [-0.2970050458178311, -0.00010034689531847526, 0.023976889149621546, 0.3198609942188271, -0.0052981347043081885, 0.06722778690615104], [0.36432497644158957, -0.005915117946948733, 0.004079778177765633, 0.09438242072357003, 0.002067959615779871, -0.052676683678425026], [0.42475641878145093, -0.009769769678101175, 0.0090680870220831, -0.10904708601963652, 0.013972180077331248, 0.05572100315020456], [0.4480574226297254, -0.020819476642114604, 0.009938783193583824, -0.08324651111270832, 0.01498671451672337, -0.03140359925187644], [-0.25297489811477575, -0.03359562801041331, -0.013523197243002176, -0.22717258025153692, -0.0065920829582936874, 0.10383005324468889], [0.21311816179558324, 0.021759892350402616, -0.04973803717100439, 0.04585848256727707, -0.002882680405012032, 0.10022787133894383], [-0.35917755574237187, -0.007661489092861594, -0.0027971217241768696, -0.1159954760124163, 5.187738667971275e-05, -0.0675259491005665], [-0.5625236572350957, 0.00267789020511572, -0.0008831279735464324, -0.00775847615709533, -0.009524782137600798, 0.030442153298219213], [0.2612762706520763, -0.005085575780655101, 0.005314971190386895, 0.12273594726691406, -0.002207165718695381, 0.03597888572330778], [0.3141216897131169, 0.011523651117145676, 0.00015471211476017558, 0.07786230022282019, 0.006688087859008332, 0.021662892306482023], [0.42810698223575, 0.0008163534717376273, 0.0012377412559308478, 0.014916773979803554, -0.0053541990895611844, -0.022944445504456942], [0.4503995637918436, 0.015318095342365586, 0.005829666413053681, -0.1574901644378589, -0.0009930677285059912, 0.04420816852386334], [0.46859262767515386, 0.013414493280740604, 0.008692896411594658, -0.09402569108670884, -0.015377045979335265, -0.06364866558283344], [0.3952743415352237, 0.009213005940536936, 0.004553992961913234, 0.013447366926860792, 0.005090974708126165, -0.02284412651710776], [-0.44319534226395624, -0.0008481086422803721, -0.0021538804198176554, -0.09490488397192044, -0.00020374350004762865, -0.016347374535310863], [-0.45182578808641005, -0.01736886720024862, -0.02330015582119127, 0.11100862001481256, -0.006429654446927283, -0.008503674092452116], [0.4119199208348027, 0.03302783603655679, 0.014390387421612999, -0.08478804509656954, 0.01148167419047612, -0.025125582910689205], [-0.5682896304636775, 0.01747340088951597, -0.004180338719083575, 0.03284947283970279, -0.0028856051407852327, 0.01567103392765515], [0.25151781639358406, 0.002796570053664125, -0.016253614306685944, -0.39992259947943454, -0.01960392987842115, -0.10131456024302263], [-0.5117439889071352, -0.0012976996382033702, 0.001955779812902968, 0.045454252570279054, -0.00902590079473158, -0.04209101447168518], [-0.5141468965114162, -0.0037831961382216116, -0.004238789684543757, -0.003810640821804231, -0.0012524099198941467, -0.028254733590788048], [-0.023204282764404692, 0.013216577185523609, 0.014668166472447416, 0.3036559170864868, 0.016047856624007182, 0.021105289205463066], [0.4267491850931605, 0.004313018265440959, 0.0046051465799223125, -0.08494947162799082, 0.010682319495848016, 0.060380812294629334], [0.33990988039361275, -0.00034782891473777875, 0.001282965485437293, 0.004404078526603929, -0.008852465672300603, 0.07262067176868536], [-0.19346927557097615, 0.005679702084097617, -0.016390782461613487, -0.0819491565375132, -0.015492745970806467, -0.13369269103813664], [0.43541159301612636, -0.007870560412791692, 0.005015660824367884, 0.026265001876193732, -0.009270730513912726, -0.03368745385647691], [0.4314298523369617, -0.0008014446053448954, 0.0024219961072529553, 0.010803634749404097, 0.003862504435818206, -0.015703209690762086], [0.43848000473240223, -0.006496063286686229, -0.010499241723861269, 0.015470522659738003, 0.0018914138663431047, -0.014166636247940912], [-0.3805977427638817, -0.0016408263946565155, 0.0011036742332088093, -0.07437968532226669, -0.0011334954877676858, -0.08098442426463688], [-0.4122876961896089, 0.014139298784670636, -0.001259157303368692, -0.15861758959566913, 0.0036461104198083566, 0.024967764042895253], [-0.4642427279542091, -0.02556232807051629, -0.009657461282276644, 0.08937029356251124, -0.00581785229134011, -0.04968098623523209], [-0.3804547722627641, 0.0205745009098798, 0.04513039414683928, 0.35242751235688385, 0.013516259649836921, 0.08834826058778958], [0.4157050101584841, -0.00865829949304439, 0.002274777143660175, 0.03857825137339263, 0.003971850944453237, -0.024714823183199473], [0.3049519148050553, 0.03961002372755856, 0.005496196031725224, -0.13407398550216357, -0.007305817069316626, 0.06975166800714132], [0.12557407244401067, -0.0313646826145724, -0.01586034945387004, -0.24805634524639486, -0.026864123293761696, -0.13702238135921918], [-0.022254368382343022, -0.009911423925462343, 0.028342600473158185, 0.19398936016316207, 0.01665005131808602, 0.13039740717380272], [0.3417611197440207, -0.002442276194071804, 0.0011114736533616454, 0.04890524148768816, 0.0015674794891533152, 0.039443628486513053], [-0.5012842608272379, 0.0012921929694632637, -0.0023166602930397326, -0.019336600907605146, 0.0006829210949742171, -0.04244092536988915], [-0.6022960722356059, 0.006712815548321175, -0.01332946974525744, 0.039190737855026206, -0.004827902466890768, 0.03639363373446059], [-0.22593232598418195, 0.00387582014258558, -0.0024831010613651865, -0.23474758543029745, -0.0181450724457463, -0.06797106855432809], [0.2947430469106554, -0.002578589253826217, 0.002368194859413166, 0.0801587811889044, -0.00027198097778194353, 0.05113847233865332], [0.4017187547144375, 0.008786875854414474, 0.00048255550221265706, 0.034010927027433836, 0.0021750109467111233, -0.015270853604962161], [0.4837486910923579, -0.017324280693804924, 0.003893678723551035, -0.08008213340848011, 0.01428079282413025, -0.026670081871090438], [0.41609172909436676, 0.02192350225530236, -0.0024267771048354656, -0.11862325287791102, -0.0041529391499304175, 0.012076071116339463], [0.30979325072107783, 0.012348379050313855, -0.0006290820662301815, 0.06419060642275985, 0.004294575914028839, 0.018890603291383395], [0.227353368217323, -0.0030088476251726066, -0.002666401706344593, 0.1591132930501804, 0.0033218314515591893, 0.039702721524736036], [-0.5385749340642743, -0.011958332672198115, -0.004650954676757658, -0.026542570175585728, -0.0005115660383123151, 0.03985209734364609], [0.3775135449969984, 0.017408751153803446, 0.010813232157060041, -0.05723020729199146, 0.009174641621675255, 0.058458370695786965], [0.26957269552126784, -0.013233832293124552, 0.005623590797399832, 0.05856867593731703, 0.0078002008303083605, 0.07220581206397436], [-0.012754443314439031, 0.032670980752479095, 0.006200553928571325, 0.35683628637155734, -0.0008607232621039649, -0.05628765447606466], [-0.29368425924397684, 0.013944372957812567, -0.015476397655998585, 0.00765518442392388, 0.003938204375137256, -0.10504309848591238], [-0.38617043211028906, 0.014495268082344222, 0.04826245444957331, 0.09076309071158341, 0.02305975595602919, 0.11544675546264808], [0.0586399164583785, -0.017086329907413603, 0.009169923680941976, 0.3180677836957114, 0.02160882953723703, -0.03093440917914176], [0.37265950363321493, 0.005909813215093183, 0.002532637105947726, 0.05648145806686234, 0.0038046331786196347, -0.015439320214035816], [-0.2527500503242164, -0.01155162416436232, 0.031198546664320877, -0.17875206666582077, -0.003960785011000411, -0.08668259192749236], [0.23211631381918627, -0.01263958454183366, -0.018041948132284456, -0.4358601937156385, -0.016225985576848283, -0.10874598280496083], [-0.47990174855943996, 0.012974589039634596, -0.005757755402720795, -0.09941069353203577, -0.015038235019621095, 0.0485519387122765], [-0.00529721852222174, -0.05491908070485624, -0.05147265385171803, -0.3294335739615683, -0.02322695034120982, 0.034168366270463206], [0.012813040036367827, 0.043218651933123645, 0.014628296775111337, 0.23634358503659622, -0.004295564385879042, -0.04176156916189894], [-0.5698417436898977, 0.008616325213073848, 0.0021983502020881977, 0.05592008061360214, 0.00041498593449053187, -0.026792572444637903], [-0.45527949446465654, -0.015595506759139016, 0.006583039987468264, -0.023978569901597493, -0.0031677282770931165, -0.056024597727840506], [-0.2430612283248144, 0.014137989807400267, -0.017251578017999085, -0.30135719661876975, -0.00458854088500954, 0.08075293499157432], [0.20657799542412772, 0.00990913781827812, 0.00034849712103990994, 0.2124476022163466, 0.008087951989506145, -0.01277451790263244], [0.3582766715086107, 0.006540256071134373, -0.016245497247345902, 0.020096696832410114, 0.0007207467396916992, 0.05508973720660771], [0.2817810322056808, -0.010564905029940308, 0.004819942191694621, 0.10481814568548416, -0.004484689769797658, 0.028762855669259374], [0.3762901244859998, 0.007808266662824373, 0.0014803214454070875, 0.021220299101141264, 0.003286031300354607, 0.014280994130986166], [-0.5610723857959163, -0.008759388849153385, -0.011903042318572938, 0.007333799097811382, -0.0011363066982658866, 0.021675657897424046], [-0.17639311353917908, -0.049598939518386835, -0.025172136588622645, 0.095589004604974, -0.02705390269667748, -0.1102958212420167], [-0.4582898841569114, -0.004907318081950174, -0.005401607873253856, -0.06743133762690487, 0.00034031123729807874, -0.02646349683161091], [0.3082599834334043, -0.006413447867513353, 0.007054973228644419, 0.14110793717268152, -0.01113072619160545, -0.017019232596124347], [0.44704400678932815, -0.004967553551688924, 0.006718536066369084, -0.13559267049467366, -0.000704686909104827, 0.042530582385483635], [-0.3719865874573987, -0.009239842421910456, 0.00830706738133069, -0.13902527090194322, -0.004645759484316405, -0.04000738489354003], [0.31153364031375036, 0.011926696515863124, 0.01610257273236219, 0.12045385220965223, 0.0026284806256185542, -0.04174500430200882], [0.3029630960321065, 0.004868976263192295, 0.0037740149194132296, 0.07237772553030003, -0.0034455818226279814, 0.049153673839520594], [-0.5195874355400559, 0.01606694624788834, 0.00512146339348019, 0.2214523110631536, 0.0012212878269718596, 0.04576758239702839], [0.3182906976986827, 0.011812063491670969, 0.006956305192744693, 0.037914320162907314, -0.001638171923570169, 0.04624954728232523], [0.3145844192571697, -0.00888631504854366, 0.000679686825103594, 0.13098989044323986, 0.0086026282301126, -0.013837928754700649], [0.4430460946434806, 0.010079295123110479, 0.006430556686435371, -0.1793831694964126, 0.011707825040171729, 0.09121606466988054], [0.4928351295925644, 0.0026665054141250663, 0.0042097528239973, -0.09859835463979809, 0.011760385188295051, -0.018530078043052654], [-0.5531597582884537, -0.018205359575678044, -0.00851576282154055, 0.04922959065208062, -0.0023730784397252274, -0.02496229819335393], [0.2948171939812782, 0.003962528286546495, 0.00011072139447774405, -0.2937147960058841, 0.00895748925785314, 0.10921352975239727], [-0.5289746177163693, 0.012934769133156183, 0.004111908479721905, -0.01303634891378582, 0.0002707766674253391, -0.025168154316817747], [-0.587066737792926, 0.0037277050447072596, 0.003239625600424723, 0.07763679185882248, -0.0003951564348268333, -0.0036753842722564156], [-0.5647669001250042, -0.008235219158254646, 0.003212799463353862, 0.03921553863349969, -0.003296947158176628, 0.02321739501124396], [-0.5170214627735182, 0.010603119475036223, -0.007022979256093347, -0.007521891772761581, -0.007529836582661273, -0.030743615756671942], [-0.384608642167547, 0.02424088867803435, -0.009068749323691588, -0.07862344860245597, -0.0011454103884238759, -0.06119580702708609], [0.4046013564083138, 0.009272398923552885, -0.00532735004492113, 0.04417751185814411, -0.015106797397564304, -0.03447680228721013], [0.20699093887670264, -0.015778910474271005, 0.00120901421958514, 0.13335951564029075, -0.0012422573059570822, 0.08589169904365047], [-0.41686268859084197, 0.01372585865985399, -0.0039931046752554825, -0.10341114899286556, -0.0005077179781597926, -0.026372388898922017], [-0.5606800801175171, 0.015360854532667277, -0.010508416318586095, 0.005287236238596224, -0.0016054191772228905, 0.04352561992402599], [0.33148869705322825, -0.002132167769310304, 0.0036751542832670105, 0.0780535555095586, 0.003622066189178607, 0.015823571927060267], [0.3223371890342535, -0.020177917833459274, 0.0034999124117658224, 0.06993421737806942, 0.009677622908837403, -0.03483416964261392], [0.2520069157227848, 0.015252633143881499, -0.00021810700879034355, 0.06075190501126215, -0.016191707000778006, 0.08644740775068824], [0.27666920216490115, 0.015728988555713294, -0.007014029433838051, 0.14984581611237946, -0.007058131413242084, -0.02001857554566245], [0.3289850957456907, -0.004828008450179932, 0.0006318640636458105, 0.05878923283071561, -0.01958643280319539, 0.05670412162919294], [-0.5857787809847811, -0.006700896997191477, -0.012126669054400178, 0.04795049375407538, 0.0011570992129050336, 0.036900976291607844], [0.28417694077751404, 0.031083916249325678, 0.0013200085753998318, -0.12040466638756861, 0.013924394690587164, 0.13443456482490165], [0.4290996031519968, 0.0043331323522354, -0.0005389518792570601, 0.000130009323166607, 0.0044335410465148875, -0.018944000661327197], [-0.5502872317688411, -0.013551980999265772, -0.006764432587899985, -0.014220943686149118, 0.00011929247430748571, 0.033152369617700524], [-0.47776357781614076, -0.011572851203180981, -0.009330668936539445, -0.0755279871932956, 0.007767549843566935, 0.044839678162730184], [-0.09693180818412406, -0.01881119736990856, -0.00781827711778953, -0.3194999299728739, -0.00035758562085712125, -0.07431786840111382], [0.4241274759056046, 0.007629802985731103, 0.01244836904883568, -0.19216352871400913, -0.0007794969116806467, 0.07955626657440633], [0.31475013680288155, -0.0061798765088045, 0.004036141734155776, 0.09904561085495212, -0.0038974135467773822, 0.01975873399692596], [0.42701748329654526, 0.011693717004052454, 0.026980380006448475, -0.11346729822959718, -0.012124213668908073, -0.03114030650378023], [-0.09351898824383047, 0.003774474357202563, 0.026801095125069743, 0.3758455821541628, 0.027182669256408918, -0.017067531061710144], [0.3322517928162197, 0.010112756683724472, 0.014131763444494886, 0.0966930184500371, -0.005528906494006155, -0.015550937720983888], [0.2021223563219618, 0.030391779095506548, -0.008520477483653576, 0.021482481625406922, 0.011370491517566096, -0.04731014959530576], [-0.5328033930482229, -0.006511053593717635, -0.008773870078561355, 0.04449755551181638, -0.008092889002319106, -0.033941294843944106], [0.37033289876033804, 0.003194573849305077, 0.009315965158671696, 0.059927715924900885, 0.008424402436279933, -0.0707655561294982], [-0.2683311365685129, 0.02027925971341687, -0.004916667667460503, -0.22200632889378724, -0.0010716279866270757, -0.06463064145417213], [-0.5058489335381072, -0.00983696523604995, 0.003118960958824603, -0.037590037812539556, -0.00020159084697882998, 0.030943328379610398], [-0.4850216574826023, -0.0023906358040576194, -0.0022956058048367537, -0.0879952484217449, -0.0023553269647684882, 0.03298847447800833], [0.25806161714171766, 0.002947454825736948, 0.0029648992179547667, 0.10855584695174846, 0.002246591382407562, 0.052486061000999534], [0.2867772631255754, -0.00803301362664228, -0.001162225108809092, -0.0768545082808982, 0.008796501087650765, 0.1483226494697906], [0.3252028258753017, 0.02042127558009074, -0.0005788940625255368, 0.07104527947954503, 0.008114240045153402, -0.03819139358423366], [-0.26892991048592374, 0.004222180886667323, -0.012685594174855472, -0.082281803953945, 0.005053931334785584, -0.10952104386646896], [-0.5835808320105554, -0.0026158028973995414, -0.007850519892263121, 0.0036484577377080907, -0.002029597752818617, 0.039024961481992596], [0.23787749188243473, 0.013987066429518358, 0.00019905604846113465, 0.15301714566982527, 0.0070366423284368524, 0.01232396127768862], [0.3530064257577673, 0.004127612720283435, 0.006986082700120777, 0.01378313256516367, 0.0021419488601409223, 0.051343130729855975], [-0.5459745840049712, -0.017653384622793793, -0.011206574201087791, 0.10524032743896711, -0.001871488794133401, -0.0405209624826491], [0.15628688823674486, -0.003574794124746614, -0.024101753439719553, 0.1554840270728185, 0.01044772164100826, -0.04967161319562842], [-0.1059657156303027, -0.01695433784408048, 0.03245455078519677, 0.4184355267807942, 0.027515386182647483, -0.020222076940920538], [0.34933324915490177, 0.00781253187349066, -3.00784361992762e-05, 0.011883726397600358, 0.0049751101667649565, 0.05253137892253605], [0.30125987441557744, 0.0026400797401545037, 0.006277755749654885, 0.06766594796138431, 0.0036919834046744795, 0.045356015851988785], [-0.5287162130007587, -0.0026154278653409218, -0.00923112388549533, -0.029124266556339602, -0.0023438468533977974, 0.029477951211184603], [0.48171294095299866, 0.012745202187317391, -0.01809096557396518, -0.19475322451444727, 0.007887208469399514, 0.011657013081868176], [-0.2693730640312834, -0.009521456524649249, -0.004448714861790749, -0.2099804706830354, 0.0014191480716144456, -0.053832108637522386], [0.2470207789434999, -0.021298513233802305, 0.005701493185918428, 0.171884767306471, 0.007621814072540789, -0.0212454701447575], [0.3684684104923202, -0.007178409906571679, 0.012849216945751816, 0.05926020352264212, 0.006831144815779021, -0.010637901490278557], [0.47896524637338184, -0.018921747495514417, 0.008690460691864292, -0.08904367566880955, 0.01433650583995244, -0.018180123074211314], [0.3905604121773089, 0.009809045164302204, 0.006384568848746811, 0.03527772152947175, 0.0010362681238494783, -0.027539745403429367], [0.20052135630572263, -0.0007981765427797662, 0.0072667410235914736, 0.13938867378494701, 0.003403071042200593, 0.08014833438631899], [0.07100310096905575, -0.008530073790553556, 0.0017236977494329761, 0.054123620706749385, 0.011701636557516208, 0.1514329095826914], [0.2931630740464647, 0.008570211569878115, 0.0015112984096400507, 0.07802640735934582, 0.003218801816421994, 0.047097403767946695], [0.37315791836141915, 0.008606975333152292, 0.004167832171566011, 0.07088981268700728, -0.005754546201545225, -0.016554659018268456], [-0.3008920055587708, 0.013472701973847824, -0.012339580498635027, -0.07462283813364307, -0.004851889746635166, -0.10863148183125793], [-0.5205832372141769, 0.0073989429257747775, -0.006245160895757, -0.02619174396989781, 0.0016848722891892327, 0.02541394591248189], [-0.37566815316331487, -0.04123959769073901, -0.021851919333503552, 0.15354166450371884, -0.003406384870456401, -0.0627042258293221], [-0.5748284431181774, 0.00841552374532793, 0.011125477886571457, 0.022193535775203388, -0.004619797166229819, 0.0344655217687901], [0.10563484744428916, 0.019715676873693074, -0.015478844693143297, -0.30313640158649674, -0.023513394606637313, -0.131107359622181], [-0.5563886230450443, -0.010555731411995471, -0.012147856212798918, 0.007582258926969109, -0.005469741698621807, 0.020618026774819585], [0.4275586774799801, 0.026091138558083454, 0.008199732967342856, -0.08208797011485357, 0.010291898133470718, -0.011712762738310109], [0.44222203834191687, -0.0034246770575609097, 0.008410334992764603, -0.07042191666027033, 0.0037379253509462152, 0.03898962836553615], [0.1184656706316638, 0.015612327742027666, 0.041132034708669475, -0.08889125194660419, 0.013003007162771344, 0.14875702122528084], [-0.5060796928171893, -0.006217459437089129, 0.003348618259074674, -0.01091165183981464, 0.007326349510123203, -0.028940925579868444], [0.2911552706910227, 0.008219990119362305, 0.0016131325407792551, 0.0696039536481205, -0.0008005020283680267, 0.06052009947352792], [0.42867319983501084, 0.0018934812893824312, 0.0005523614740687389, 0.003204306886107083, 0.003063272784360268, -0.025539955602266733], [-0.4891652855808231, -0.009546292177661913, -0.0016657066401095252, -0.017440969045816476, -0.007254330413781193, -0.039164082808476205], [0.255099322214628, -0.0097786794733377, 0.0027079726872400634, 0.026537093939894445, 0.00613252951214871, 0.12903684586519032], [-0.015850513881797906, 0.030820426441336576, 0.012940905695322526, 0.3095989689612479, -0.008836759556338608, -0.026229619771363608], [0.44404948883046624, 0.02094734565496043, 0.004522039599487574, -0.1931490182784436, 0.003852098080123834, 0.08244614135150045], [0.1916076712237727, -0.004919147041612356, 0.005251922135111829, 0.07582400350967587, 0.0032479906332361563, 0.105286607158864], [0.3028965951831568, -0.008834939142099284, 0.009141485736709866, 0.12712800641483044, 0.007386718956002508, -0.01153786714860051], [-0.5710884275131572, -0.01354463333695542, -0.006209831258883269, 0.03777116214128687, -0.008244038898923606, 0.014162435533293693], [-0.5739851884237114, -0.012281888303044136, -0.009658919685360345, 0.09993644267025022, 0.0016559022637520405, 0.025287461001919774], [-0.014219296624472888, -0.03145495856303548, -0.03397799573168547, -0.4136621456787493, -0.010059928307117163, 0.05738368998442565], [-0.44314459517682453, 0.00021573375163086062, -0.002823861115697444, -0.1297821414164456, 0.0014812637519369663, 0.030650266872063212], [0.3507212723112141, 0.002627229386319597, -0.011998992919267652, 0.07681496123899878, -0.005870275748164883, -0.014724257162183368], [-0.5596728145526166, -0.020611251433879513, -0.01870654349422003, 0.14456093116615604, -0.002304033648101955, -0.035145811846863884], [0.22882124839150902, 0.008202136766819027, -0.07238347765988586, -0.24216816240841735, -0.011583403559113052, -0.07813492883249872], [0.3748904406790998, 0.010321857288033148, -0.03595780459760265, 0.04618760384418444, 0.00623853086438805, -0.04662364395112023], [0.39027942796208087, 0.0038106864784596354, 0.01468792750728466, 0.03498462999997017, 0.008342992907247049, -0.03234233152171048], [-0.25352053752535825, -0.018476756319195826, -0.006603526019204571, -0.22439204711543795, 0.0010240898799097685, -0.05518455623404601], [0.22721689324425634, 0.02270432791426588, 0.00893146332614983, 0.16259008603916988, 0.0025343812353464563, -0.013469985215040718], [-0.3598809772823303, -0.007568372925350705, -0.005746927907284353, -0.16176653371947217, -0.0018644493588541038, -0.02865940547337537], [0.33131948953855905, -0.006287448284155145, 0.005759299049140433, 0.04977064085655736, 0.004465095665361198, 0.046379113650726264], [-0.2789362770888, -0.01729383495588006, -0.020962241954796568, -0.25538573152507044, 0.0003546364044326303, 0.04578440150106795], [-0.5063903074245164, 0.006682346418663467, -0.0001492234407883201, -0.01892874441034673, -0.006059159084573274, -0.03564157872510623], [-0.5845878672336857, 0.008873007888516977, -0.010753585726894966, 0.040837615020417876, -0.014863006776611669, 0.03744764635206346], [-0.031181067669513614, -0.01772166175679491, 0.01940901966705263, 0.15962782247906038, 0.019166186487655657, -0.0861157544859331], [0.379989402618543, -0.0028555758522436723, 0.0028907844704542576, 0.052787891149372074, 0.0024224585938477914, -0.016221627646641378], [0.061420805878060876, -0.020538835480602487, 0.006508116627785471, 0.26292464894778944, 0.01739633449562377, -0.054144165706752526], [-0.3801469613713655, 0.004884518924963727, -0.006746650634057833, -0.1551439495060606, 0.0035669110875344023, -0.02690053516768237], [-0.4899243869848015, -0.013977059851937527, -0.0012058513924403638, 0.004757302006237079, -0.01430134612133279, -0.03833532432239304], [0.2284798173449833, -0.008045219360850123, 0.02942391159573862, -0.1007245532050269, 0.02037439187562015, 0.13164387397175947], [-0.5811658333251296, -0.003630409051000019, -0.011643484917713488, 0.047315013184920464, -0.005743390255256429, -5.5808985663205455e-08], [0.24717257968920014, -0.013590847905873412, 0.004929483145030968, 0.17558309017203186, 0.009243243215085998, -0.015500295191241015], [-0.5101210264747421, -0.01826484330447469, -0.009516783830249967, 0.030418771082702308, -0.018105675794059115, 0.018813514364775973], [-0.10901991597266261, 0.012264952479848788, 0.012556386786446064, 0.34375970059281796, 0.019546577946077486, 0.09045826307975315], [0.2991310071349246, -0.0097426597390844, 0.0024385441319035774, 0.10014531539537228, 0.006120443490356117, 0.016741798208082276], [0.4263766555813207, -0.010272568816693313, 0.004149747903585501, 0.016756705243577752, 0.0005585007308295326, -0.02190058465416792], [-0.4833940662477892, -0.00400395505360814, -0.008772727851690035, -0.01222730621353959, 0.001639138559705061, -0.05253727366927088], [-0.5544600302258983, 0.011419119050979695, -0.004001047721975878, 0.03853587405679033, -0.0009168785242597, -0.03198036996897484], [-0.19877873761547754, -0.004744750812621891, -0.020999893715022044, 0.011981277211943343, -0.031010905748896703, 0.036149677346740704], [0.23572902797607995, 0.01623122098698366, 0.010644852142380477, 0.16124469410783532, 0.010196006329044363, -0.012402598079120296], [0.48674943572727064, 0.008446678409509798, 0.009211526190598547, -0.1932652302523639, 0.008368780631246856, 0.0020338886588143956], [-0.53895691314662, -0.019918407150647858, -0.014042736306889297, 0.15108643102257396, 0.002458440277698287, -0.05317300517230745], [-0.11390226842365238, 0.06182279239800006, 0.025826955819229162, 0.027190682103256685, 0.026720812499755272, -0.0808792008298335], [0.17152242702725534, -0.0060000111981738675, -0.01946838037492272, -0.33450794248514293, -0.00315517760118504, 0.044360513203597476], [-0.4893582003172566, 0.007889099163305291, -0.007308740640049875, -0.10421285686196383, 0.001890088217911657, 0.0476972771047171], [0.4593493335864629, 0.012934972093883248, 0.008687485696381636, -0.19920865912232302, 0.036649384167202675, -0.048764262453357285], [0.4935120553122543, -0.004028319195197543, 0.003989682634198522, -0.08213325286044464, 0.012642072428687296, -0.016717270999238535], [0.35568837581884905, 0.0019100613715316548, 0.0038766350137276976, 0.023807008396319253, -0.005790377949544692, 0.04272996401578337], [-0.07241956762951135, 0.0022769303208599097, 0.007382140366658841, 0.3432540292108607, 0.014011046898711644, 0.07237184940384873], [0.09523266827008875, 0.011391651204715466, 0.00824032447968488, 0.10624642864421521, 0.01721629280813421, 0.1595107630084622], [-0.4499457672173523, 0.007283100663968372, 0.0004091321977821935, -0.025878855969700593, -0.0016409323636173284, -0.0499781687755944], [-0.45439278807672945, 0.013485050743791709, -0.0008565354808664058, -0.09009560142968347, -0.0006115931664438048, -0.02134853259006991], [-0.43882664981764413, -0.0021197714241069383, -0.0019386795239501678, -0.13170997488071906, -0.0008838884087359127, 0.04545658310277418], [-0.5095687819575064, 0.0060279304336282764, -0.0071073273380874175, -0.05330915643657126, -0.00551765019471571, 0.025750223588486023], [-0.5005026672002081, 0.0026217281539300647, 0.010737397113700366, -0.05118810637583808, -0.0019913134267848073, 0.024669628401863774], [0.2691706503155627, 0.015532923196823619, 0.027364494951588336, 0.05218567968704055, 0.005029211607529781, -0.047391726547313356], [0.4185564793425244, -0.008800845362122448, 0.0005212697719341782, 0.030503741135496566, 0.005668044312139007, -0.020015192696477516], [0.13910677506851216, -0.008947475350583337, -0.05977807037856977, 0.0009417771936873485, -0.022947480197820034, -0.17637806601776557], [-0.39436163041129674, 0.0005951901109990482, -0.0030371197656585878, -0.10853938180473344, -0.0024744451610865495, -0.05324070820631932], [0.3610830723665547, -0.007801687555541079, 0.009678400666157317, 0.06587548248021118, -0.006726530604203301, -0.014154317328760036], [-0.462926548654134, 0.0029704220222737054, -0.006105067390066416, -0.04351017919040166, 0.00040182002658523716, -0.049678224592036725], [-0.35128352237848104, -0.0234382505162735, 0.002923912774000216, -0.21609882476132422, 0.007051577626373147, 0.06132272630332492], [0.43437531623398346, -0.003687166415559419, 0.008879249808687949, 0.01933082568285131, 0.00440669101564821, -0.054585233785931], [-0.507745838954899, -0.004967052202524331, -0.00461301151152535, -0.012253496766685768, -0.0010509740309222874, -0.024668793200111682], [-0.2343474358862815, 0.013077266326767277, -0.018647685091449283, -0.30655366037167103, -0.001577630100134311, 0.049038668932293986], [-0.4659976267019028, -0.011524316844700969, 0.0023469806237127874, -0.05143863716109293, 0.0023008357753102936, -0.030923902357993516], [0.21812477765190527, -0.0033037658670683956, 0.006224059209824985, 0.13370990304740235, 0.008980717936087162, 0.0347899364371497], [0.1469286639467545, -0.017038704581047143, -0.02107764546634537, -0.2683207034150682, -0.018413392341679533, -0.10107118733028957], [0.4733786688068971, 0.011764867696735947, 0.006523959825535075, -0.08152257091639663, 0.00701257399864239, -0.015608451792367518], [0.3817161814410032, 0.004654906466822376, 0.0036995927400302366, 0.019240999775022326, 0.0001423582697281254, 0.014475961307393637], [-0.5361162564294398, -0.011305938833446274, -0.002820003179435252, 0.04432929380560577, 0.0004845805319360746, -0.03726072351427258], [-0.6063738707658144, -0.006031314769824281, -0.007266037071452866, 0.138161358677811, -0.0012586440304574696, 0.028032885570427657], [0.2615855242238373, 0.005055858549238702, -0.004356958238855485, 0.15800623845895567, 0.0029952018674997767, -0.019932235568750307], [0.25671141790163843, -0.006859425833964456, -0.00916774064052167, 0.18156746188584597, 0.002768724190295835, -0.018855006267864287], [0.19941355637803812, -0.02104891344897644, 0.006950553195035547, 0.14468135388186676, -0.002910148286692094, -0.04108100489387452], [-0.4439666331490648, -0.010778765971845938, -0.0019907904194269273, -0.06819044376682383, -0.005356113577471528, -0.03520391978203302], [0.33277937118405876, -0.0071027475850303605, -0.001067531360610357, 0.05844801790167497, -0.00503299794541288, 0.04517343166496595], [-0.19825506073360308, 0.016352903426241037, -0.01577835427917767, -0.06459506623031962, -0.011864887591171677, -0.1417774181898516], [-0.5650524327596418, -0.016134274151230175, -0.00757847959213947, 0.009881867011972204, -0.0015873749424954557, 0.058209904959845286], [-0.31776873427584085, 0.006168250762013382, 0.0037459777373754985, -0.19138546036448964, 0.0008601045505251793, -0.04286870983815343], [0.40690545041763587, -0.009771070139942526, 0.001749628369698148, 0.05465816544647185, 0.005578621285531661, -0.03360746204606396], [-0.4855867681962681, -0.012459718956824744, 0.006672042700580314, -0.05042355964204758, 0.0027884239545503753, 0.03599910394952972], [-0.29873324584910094, -0.036020708452920006, -0.013919834901218219, 0.0024436177589548733, -0.009899781178979888, 0.09598264870773711], [-0.48603107076408575, -0.0008970228221326217, -0.004504946393647502, -0.008255444956746077, -0.012106038790659546, -0.0412516667489212], [-0.5102118735127386, -0.00022541953659156017, -0.004758566084050121, -0.011520958609924997, 0.003926775942697613, -0.030759124866060535], [-0.574605181209094, -0.008832436137355218, 0.0004819359715483014, 0.03529728220635204, -0.00821433636357873, 0.04121940219878643], [0.2057097997684211, 0.012093051343734836, 0.0059755441461702054, 0.056005671961578254, -0.0021874023767514427, 0.11822619229970449], [0.44485639071036787, 0.0008294012766517968, 0.006605310010953189, -0.12379854205834663, 0.008107131694422355, 0.07100590360404462], [0.0881923054164405, -0.023975014729074692, -0.008102362162883682, -0.43748403316439016, -0.0038759653929642773, -0.10615337929557676], [0.2213345226971316, 0.009957836869231563, 0.0059309392161194675, 0.09899846786512236, 0.0034747433555291176, 0.08188626777464468], [-0.51350532885903, 0.010523675321357817, -0.005203072966732951, -0.021463537932965984, -0.0002975328690278249, -0.028040869360271736], [0.4226567290159119, 0.010503253153630251, 0.02427442299116732, -0.18805990390188687, -0.009215692730219844, 0.06314619147139727], [0.4272195876842377, -0.0035888129751712144, 0.0014095836908189039, 0.02627974333982121, 0.0028171244217435683, -0.03670722616145243], [0.3316206027439188, -0.016222502940282595, 0.0024103092879078393, 0.02927107187596836, 0.005449824035865818, 0.052312922599524976], [0.4163204818316446, 0.01375832793308513, 0.007217511218315389, 0.012619981540905146, -0.0015614188843786846, -0.01787329633798772], [0.4114711949632467, -0.002598631756946016, 0.002110650131148072, 0.033149076628505816, 0.004373271597406225, -0.014992228230030442], [0.19798500665808536, -0.009321680173566984, 0.03115300775881454, 0.22776952767345807, 0.004471776031442053, -0.04586481743541237], [0.16275748583118635, -0.0030546407375266595, 0.012146674855341775, 0.24918018547032644, 0.00951746469825711, -0.006570633321047276], [0.3367036472535959, 0.011033273140352212, -0.03712038837194193, 0.044090407213193655, -0.003116121407402411, 0.04959315042616852], [0.3476758884125724, -0.009298344784287404, 0.001254068261805757, 0.07512352947136428, -0.001023447679816464, -0.02689990796735437], [0.3453943892845294, -0.006933453662919248, 0.008309840595346549, 0.031760727706921475, 0.0007566326768735099, 0.051409407258894646], [0.3596693371176377, -0.004284655970447583, -0.005570455145499747, 0.042690297148009565, -0.005661006119481739, 0.04766981630311249], [0.19215219467457237, 0.01933642178954883, 0.00953976876362479, 0.06621932575750512, 0.011325811454834698, 0.12251572897234377], [-0.17761558852565448, -0.04121989782339606, -0.015206142365671603, -0.0037338416581753435, -0.013771332974841694, -0.13582381270287675], [0.48838574163037685, -0.0025956241681183204, 0.009663736278800331, -0.13244615811296692, 0.012872199398967204, -0.0028695378842036917], [-0.5736207962797074, -0.011805626320615173, -0.005575351924918175, -0.0030598207939408336, -0.0004123153892802159, 0.0422923287875516], [0.17882345944594372, 0.009821453093168777, -0.015481262455568684, 0.17396665318875404, -0.0075393405497569945, -0.07890501034158727], [0.4143454845539977, 0.01382924517774799, 0.007107747843421551, 0.007121597119145252, 0.006933878111651027, -0.017967476615489298], [0.4161999580422589, 0.003637739756515211, 0.017048452201195954, 0.02363709767217306, -0.005606482241566798, -0.055704027335340106], [-0.19671611585895887, -0.020461517177159442, -0.018464154877260802, -0.3262121669240065, 0.002359567899193627, 0.0827160536048587], [-0.2844029517938889, 0.011222167596804644, -0.015548034756894908, -0.24835846530959604, -0.009804600950479284, 0.04882188521405415], [0.3250275801226168, -0.0035808789301193826, 0.010656393786327219, 0.04425321299957382, 0.0016575943785278003, 0.050614815591790145], [-0.14481816803828804, -0.023152033582368853, -0.009632457881687833, -0.20193821094211942, -0.0021642080404607276, -0.13457325484840713], [-0.3784345326554007, 0.0006150674261249629, -0.004114989985837296, -0.15117884343813776, -0.00777244290707912, -0.015747750503162475], [-0.47983847550829895, 0.010334885988060174, -0.002740660267722049, -0.10751115219438605, -0.0032215725948606935, 0.019698641243871648], [0.42201979289780894, 0.008555213842374773, 0.009320719164594404, -0.06138884967608365, 0.008921053776664894, 0.03125206999463905], [0.32151236224508667, 0.005103906947573972, 0.005613545818530478, 0.05811127196590043, -0.012900575209404958, 0.04606091680373966], [0.29631694501721206, 0.014213717914602476, 0.000997708574162855, 0.11771834232698379, 0.008559374131292001, -0.042160393281577176], [-0.03943125455689743, -0.00527249087856387, -0.010609181531620635, -0.27447582901337136, -0.005413612782266728, -0.12043453981889389], [-0.16673887983610478, -0.011009656709060976, -0.008681177525973234, -0.16022025286791083, -0.0010373521911976815, -0.11506215417422569], [-0.029056442435707073, 0.018744500867775737, 0.02191319743679218, 0.3732158377499134, 0.017177744893679233, -0.03870769565531183], [0.23399931253109937, 0.017852536797157138, 0.02386128215938507, 0.16076647120265078, -0.0037102167703962447, -0.049729210717740314], [0.08241763581856537, 0.09167342602651889, 0.02711733975015693, -0.2094195780839837, 0.013536960942730743, -0.01649607305427624], [-0.5209636452570536, -0.015225725906581684, 0.000732389866639762, 0.08074765021114279, -0.03299499442258407, 0.050562896937005324], [0.003473472378855637, 0.010741086003482135, 0.020717208341850875, 0.05782011724773506, 0.017776858791458013, 0.2066033249057904], [0.37095128542291683, -0.01047416102217419, 0.005610240405913101, 0.06915264193635329, 0.008043009113224383, -0.014419865673085739], [0.028292829652076114, 0.011632730398577608, -0.01715912247013723, -0.02599599127822609, -0.04510253278516433, -0.16622600875522128], [0.22484598503150843, 0.004892846977541515, -0.0028809174045077205, 0.20235784142786992, 0.0039025789496459277, -0.01880237006977801], [0.4835636730152492, -0.0031848933949561037, 0.013736664646155335, -0.08298822427683653, 0.009135991950182464, -0.0062617833683680955], [-0.1604986746496101, 0.0066429112624369875, -0.013022506193340774, -0.08145680405461747, -0.015848152903502114, -0.18166238673697874], [0.3343686450294435, 0.005306565813646056, 0.002221368464691911, 0.10593759044746799, -0.008945900745290272, -0.011791602343294572], [-0.4062736754594716, -0.0002735244348726332, -0.00903674597765242, -0.1656898693303212, -0.0071158663178715765, 0.035545871996378424], [-0.2818607888894898, -0.025504728720692885, 0.0037935768504418757, -0.06265605103402105, -0.00546918590681961, -0.11527805595465353], [0.4222224890891021, 0.010719121139082084, -0.03338013074279511, 0.010786731345282824, 0.00013227532488967625, -0.04000881948889833], [-0.5547880291248228, -0.014934147225873117, 0.007406231120025455, 0.02784435765727135, -0.003574676669999753, 0.007321892658693718], [-0.5930016404674241, -0.003999411255044541, -0.007455899616934998, 0.06690657035519851, -0.010281776708281135, 0.039819156249482894], [0.27951356357899554, 0.01874345272371832, 0.00038843768424653145, 0.13711977520658467, 0.008868857375609636, -0.016716063337834593], [0.2674248527758671, -0.00797318280820088, -0.003098504371014699, 0.09866119083250023, 0.00816330203352898, 0.04243130328595327], [-0.5005715472380059, 0.012741344092711493, -0.007310593766596844, -0.02375302412682034, 0.0004870794455758287, -0.03457992507353539], [-0.5708458959176127, -0.0010590851602190335, -0.005030451812941585, -0.0036075085227303826, -0.0067960986369333875, 0.052713484494877776], [0.3462436677030373, 0.0007503310547834374, 0.00025389977213941835, 0.028772268806286038, 0.006957470331492789, 0.04915910185867101], [0.2970843185098819, -0.005108429629268784, 0.001577946920494731, 0.11281189521578573, 0.00758097760303644, 0.014733291380070773], [0.3677338132667087, -0.0219771460150353, 0.01933745823052006, -0.096885549285871, 0.001825765275278811, -0.038149976392237], [0.302016773294033, 0.01232521257091551, -0.004607937700857987, 0.030326727939378212, 0.005156309695839545, 0.07290041420069168], [-0.46373200706654305, 0.0015426979793664525, 0.0030348249809823455, -0.02917024857016001, -0.011070578467838595, -0.045119133300254216], [-0.5627007936513021, -0.0013867014960275022, -0.008457226084974376, 0.06964747756013709, -0.004044745338328122, -0.03803691787209921], [-0.582635311937086, -0.003000384983663298, -0.007566123894679895, 0.009465876064687614, -0.00946775752165493, 0.040217035605728105], [0.34309995653213843, -0.005833126725636721, 0.01841515108341344, 0.08163636540644546, 0.005627186387089578, -0.011527437445356276], [0.32615484744831014, 0.005629361453678251, 0.00296967918356248, 0.07366516323687866, 0.006046556601935781, 0.015047725408966293], [0.36784733230446315, 0.008835249965362092, 0.0039227090708644176, 0.067183675675344, -0.0032266089562752835, -0.016177606138028242], [0.41287615332334715, -0.020346874094166172, 0.010160444889545569, -0.08970202640130182, 0.014109987702084522, 0.018487076485252437], [0.2833204424827078, -0.0033079302998391172, 0.01058432138752035, 0.10813477149361794, 0.0035873949201271006, 0.02719433334919916], [-0.03737721556577783, 9.41648802705455e-05, -0.0033766228060526497, 0.38199360477383104, 0.008301611364920827, -0.0518960188376701], [-0.37730723741854644, 0.0008393489188500558, -0.011030159879229805, -0.059250299605336836, -0.004079430127577627, -0.09063111077704851], [-0.03799915310259988, -0.01055463161422408, -0.01050601523467688, -0.3348816916989152, -0.020645625717781457, -0.08117444107336136], [0.2119346945854236, -0.00694112509730854, 0.018303741982990177, -0.21711373312281831, 0.015489295633284337, 0.1792382768120798], [0.32236223831943694, 0.002231157951686276, 0.003519599075509131, 0.06319858440771121, -0.008380873241489013, 0.018564456394412933], [0.2646048982515224, 0.026790014060100748, -0.0260985609597834, -0.07858275920366899, 0.0071435347569964785, 0.08951930166626126], [-0.35335992981090164, 0.0025017793334758194, -0.0035065661446992235, -0.14800343459793297, 0.005974153606082735, -0.048693859528883665], [-0.3383627527884024, 0.005694753208133669, -0.010679271548275346, -0.1712538865310632, -0.0012479068842646108, -0.0471376021227948], [0.3687768640687427, -0.01172938682423652, 0.007883245136235398, 0.052478162823031374, -0.008432165096338092, -0.049748129808847456], [0.2827785275779926, -0.013174002924421533, 0.013823381117895129, 0.10506137121956151, -0.00631252965812635, -0.03769411181328641], [-0.03195987016603161, 0.02246778191251711, -0.01541736628535164, 0.3764463695674235, 0.0006480342505821763, -0.038083270081145475], [0.3263855864852193, 0.005259246173277145, 0.004432127417448969, 0.04806773108630168, 0.002498580416943616, 0.03922422842080764], [-0.12208673660605884, -0.0016835204362514239, -0.0070003124673476734, -0.08431281160946077, -0.017142107874867297, -0.1498881617996635], [-0.447787446196001, -0.012510656558931, -0.005909861639801708, -0.004470952467927639, -0.011740393917711893, -0.05508658665552589], [-0.17556500283887863, 0.03145851158777793, 0.015530609462104075, 0.054521850210251126, 0.011496617826209374, 0.14041439137418443], [0.36339654241492714, 0.011470362441305379, 0.0037514681761772056, 0.07613718986529242, -0.010731395080867411, -0.02159416781683661], [0.410860128781132, -0.010493985668931853, -0.005033142877241871, 0.03944843119057065, -0.008129603207705926, -0.0472575425035393], [0.3619547677993722, 0.005647507729091565, 0.0039851541024379915, 0.09059451239757112, 0.006793527801454993, -0.05080462733908712], [-0.5066603634831551, 0.00023553887880486572, -0.015008518274699234, 0.04597138944795395, -0.0027744008334427113, 0.012136051234230854], [-0.5001126787926783, -0.008951938318532133, -0.004598699008675738, -0.003519764990010615, 1.6466690060352394e-05, -0.03450594968272876], [0.42786742962114566, 0.0025043956832313035, 0.0033087628820632846, 0.020981817747218218, 0.002070880277831906, -0.034623361969069996], [-0.5614774373728223, -0.012219893210099572, 0.004844276666153397, 0.04696695794405186, -0.007275873438389166, 0.037008636077769545], [-0.4475926165272167, -0.00527624326710172, -0.008442828621437479, -0.07047745536449064, -0.0017503886314049638, -0.03194713425501517], [0.39773192923400813, 0.026179342553429637, 0.011082828032925165, -0.14560753305721655, 0.014002561528183731, -0.021476985434189813], [0.04172948150108708, -0.027754061985726816, -0.01993658526922446, -0.30976359326260294, 0.003741311975557856, -0.14014310057813859], [-0.19537660884583202, 0.015354553547520387, -0.01138041933940347, -0.21492233918254167, 0.010048239784330268, -0.10835294977359682], [-0.03423700445310298, 0.019612904377680217, 0.0002870668724488921, 0.18874016006174188, -0.015585424596948262, 0.13224325011913196], [-0.2535930702141664, 0.012067706491904738, -0.008105356707079273, -0.10270750364916506, -0.0037185609208435512, -0.11993168541912057], [0.32080565746311634, 0.011652825418405491, 0.0021004145523945313, 0.049059755954283406, -0.013607258360705622, 0.050272084825984005], [-0.4324566919921109, -0.02394641672794692, -0.0098153491394421, -0.07050954035770253, -0.004455421234147896, 0.08433167341960096], [0.3269617423443953, 0.0120828851190923, 0.00012122777244667272, 0.02456920604235716, 0.0029467539514014926, 0.05549818477030633], [-0.2604656012029504, 0.00861248011403137, -0.008511340528282672, -0.2768712085762091, -0.010740247476077432, 0.07115591766949082], [0.2501864402657525, -0.0021644611229627497, 0.00872208996395911, 0.06048403125729667, 0.003808947857085513, 0.09039295177886955], [0.4188283978917813, 0.0070498291023473655, 0.0023953910120866105, 0.010149263499327238, 0.0014568890052533983, -0.020093289029317823], [-0.5024081711548107, 0.004093405619518438, -0.0062803153819740625, 0.009060057167819602, 0.004394677378447, -0.054167748867098464], [0.364100821617314, 0.004802482711923188, 0.007218741739547239, 0.048223009633812305, 0.0004264682718238037, -0.014549857307755609], [-0.5014245307213087, -0.012162007534701232, 0.0198208460323264, -0.01799871294095483, 0.0024130363184011933, -0.032926964487098774], [-0.41288387694820305, 0.000983231180914816, -0.0002857684362237953, -0.11106496665142547, 0.0005542226944209549, -0.03745617517281758], [0.05273550059732535, 0.028770223039448983, 0.009385600268279448, 0.29298977529923015, 0.004080135195444001, -0.034434188901366], [-0.21856689383611339, 0.0125687318689226, 0.030636998650089682, 0.006841358485960725, 0.016147062780367902, 0.19258996965367847], [-0.49280142503527774, 0.004199702620975975, -0.00878366648298572, -0.0426527895569975, -0.013492951064729605, 0.04960696285234413], [0.23369082255421242, 0.005802372894178533, -0.01057308169117739, 0.1419224011556097, -0.0016805010552756222, 0.04590434977881699], [0.2528020867709632, 0.014919604057839141, 0.005839451191139537, 0.09779493090680257, 0.006860605371443218, 0.04854412978262177], [-0.16484290935871335, 0.028964387231737054, 0.031958363936806226, 0.1642474627068569, 0.020220579192459554, 0.17902596549720165], [0.2380403514193045, 0.020693579655271587, 0.0033903803695795954, 0.05643103744410988, -2.6732052616516627e-05, 0.09713054983101824], [0.3183236822407026, 0.006593727911002595, 0.008072144759604362, 0.030428514175325747, 0.005260812192656262, 0.05751124732611968], [0.4018657898507663, 0.008629650229258868, 0.003208023404238977, 0.03653665902191132, 0.005019325350171955, -0.021832367892288304], [-0.26037685026753776, -0.03483620786693572, -0.012148338565521263, -0.010677829531244009, -0.0074959184244667715, -0.1593384267728648], [0.2555198038037798, 0.007673732726105334, 0.00427264878742602, 0.12145277178133461, 0.005329401526027926, 0.04006760628760803], [-0.34926076435608244, 0.004698615709928254, 0.01976188308543201, 0.349016892240543, 0.015971118262847493, 0.08232558839065916], [-0.5391059341198919, -0.01464122178732655, -0.0049584581559639525, 0.005627233053905595, 0.000919557591644007, 0.017231367277277268], [-0.2287920637488293, 0.0015687332171082458, 0.01391238813668877, -0.25741320294668363, -0.010391169461289308, -0.05917690741921685], [-0.41006897053558866, 0.0081001439144159, -0.011753462221774704, -0.1483915045157571, -0.008562279485376892, 0.05252273951074577], [-0.07265545278806682, 0.004268060132548919, 0.016945060548190997, 0.34695681447539456, 0.007322971116233369, 0.082103511427979], [-0.4877721789277493, -0.0028140217892078353, -0.004042945754545396, 0.04031979405409894, 0.0034529979615017243, -0.06483864554410167], [-0.4623750628283091, -0.010579387843105314, -0.008824183881007922, -0.009715093059028018, -0.013857422673146083, -0.045802183048739384], [-0.055643470604531134, -0.0011341579179313403, 0.02129369715112094, 0.04739406397088887, 0.004865237683865008, 0.20980939162134898], [-0.4864080081393299, -0.005798671726556931, 0.0016151566010098105, -0.11075967965850478, 0.0007493146279281802, 0.04719855496212002], [0.2068762656777977, 0.010204873311958367, 0.00017660635021872703, 0.21931551723791118, 0.003180241728749331, -0.07588302811615853], [0.43314038330047255, 0.015048783840189473, 0.008176527207362788, -0.07083584909735972, 0.00597648472958657, -0.02193942521834732], [0.3203078393404935, 0.005532702018350272, 0.002144505843586219, 0.06070499145031721, -0.006308882803401208, 0.04241099953912228], [0.2910616224982999, 0.03392491292762708, 0.01869900543274093, -0.14205962182264004, -0.004044975163734143, 0.11174191327056417], [0.371087880724142, -0.0034252047988292903, 0.008327424540469261, 0.023594267455843258, -0.002107629756311588, 0.031286595168019404], [0.05616311593399165, 0.003476000455475389, -0.006197783671800916, 0.20049254199135977, 0.008872063814366957, 0.0669752519527971], [-0.47715568047262746, -0.01742461138997634, 0.0005890115286921806, -0.04780596165100959, -0.013312276417830737, 0.05758713745036925], [-0.502828506509559, 0.00842660907849496, -0.0053150395757899545, -0.02283758708878004, -0.00030413460498119113, -0.03562800796605396], [0.36305946006295514, 0.009058893098323102, 0.00010550503983623617, 0.023206928469065094, -0.005277625030878316, 0.03686017169403058], [0.4708424496319837, 0.01520412152419342, 0.004421819050349719, -0.07747206467728787, 0.005088278075155438, -0.005495512695305136], [-0.43017258245513595, -0.006942406404035497, 9.856097075710001e-05, -0.15198291026837585, -0.004452667258979286, 0.0510941266278874], [0.2794194134528385, 0.0030072351280867188, 0.013368777145027268, 0.1407763252013528, -0.008265919294549931, -0.03166749829942234], [-0.436791715185966, -0.014411473168502048, -0.004107584381939028, -0.05450437430299005, -0.002359227522929119, -0.05331229210434189], [0.38341017690820944, -0.009929053243562547, -0.005657880061573974, 0.07645990319694022, 0.004008361560057389, -0.033421032169596875], [-0.5556716387177566, -0.010091721562706484, -0.012029692546076562, 0.07866471219478273, 0.00011612279431630777, -0.03247444882922944], [0.297095821599175, 0.005591502075681887, 0.0006234216589397345, 0.07669503755486477, -0.0017304901832512474, 0.02873804062792224], [-0.13600119758907295, 0.0020867644452487906, -0.020977579398128176, -0.2912917858301712, -0.010092678456332856, -0.09226574539376538], [-0.18340006652247545, 0.012686796860893185, -0.004267792417690218, -0.18689867489643205, -0.0171234509759331, -0.14924105447260338], [0.3486524025896808, 0.010657768458214428, -0.0007988672433273165, 0.07766588192100642, 0.0077273335078924255, -0.01333458212654984], [-0.09085939614720578, 0.03664355256160961, 0.03935717735427906, 0.19325869565707002, 0.026232027957109508, -0.02448672345054756], [-0.08541664973103949, 0.017652757104131608, 0.007388783509260467, 0.41903531372634345, 0.021787197315493542, -0.05231898922577805], [-0.26148964266118346, -0.03069578681737811, -0.011345414462859683, -0.2770379231782662, 0.006363966316231803, 0.06677170556536155], [0.294201596148993, -0.007712784058318539, 0.007038402769140837, 0.09241151186513759, 0.0054842184612681364, 0.04309038814711215], [0.3525498828264837, 0.0012673489706302157, -0.00039797352374300473, 0.03828220856964388, 0.006106447588673388, 0.027429702287704055], [0.20384495976191658, -0.04169325444422349, -0.008464752720833493, -0.35810340320393913, -0.017326649974738376, 0.037829846613564526], [0.4277452551750489, 0.009860703306420441, 0.002939221523777289, -0.0013114075394880013, 0.0053316480411760925, -0.015968753840270928], [-0.5090262205292369, -0.010526454110104154, -0.0038919993766685554, 0.010352908490082963, -0.00880418547517555, -0.03894327976813049], [0.142500535597817, 0.007195380353520348, -0.011718301574015736, -0.455590812142199, -0.02199117456818727, -0.08274248969379647], [-0.10595653245244249, -0.015099982456321789, -0.013846317331396388, -0.3170452239092122, -0.006922908070246663, -0.0676990357803819], [-0.33072098955778406, 0.0041515539564594564, 0.005517363654976773, -0.18624233277344152, -0.0030140009683649765, -0.04384492764517828], [0.3663521791066485, 0.009155031133897686, 0.001394312057149063, 0.03349520476271712, -0.00015811491138588668, 0.018024721184306345], [0.37684706093077636, 0.0067499291789847446, 0.004004942803096461, 0.0048529968631157204, 0.0057960380346601765, 0.028067450268462385], [0.4625199564122409, -0.0012946779222488798, 0.008029579046456604, -0.026689137466947913, 0.007430105396619282, -0.025482492132788217], [-0.22070043823745247, -0.03102944166640986, -0.014116028083260682, -0.13455232022208624, -0.00470976655752398, -0.12238859253485378], [-0.1840857536951925, 0.02063042216236156, 0.004730869087146473, -0.2588891023998589, 0.0005769937411977284, -0.08056319080041625], [-0.07666083996824474, 0.03081482696861323, 0.004488755152056754, 0.007256329166542627, 0.030098002518314466, 0.2112394737817642], [0.06795353868350257, 0.0034727908697178757, -0.011058572051210675, -0.44664410101921753, -0.024185320953922857, -0.08444166886220315], [0.34449965791214804, -0.0031846153132408794, 0.0008918297122920412, 0.10361471989024729, -0.00022312133228561635, -0.011085137535829691], [0.20696433924165838, -0.008341433547573178, 0.0012194487074569777, 0.1776736825906043, -0.009479498390051092, 0.049143461397905104], [-0.34721817998202414, 0.006451489731218351, -0.000672903022612067, -0.1812101220860683, -0.004666315037566946, -0.038170636269613666], [-0.5626261816651409, 0.004369510837038189, -0.006689049727602865, -0.01535723654175973, -0.0008751480772410032, 0.04903490044678171], [0.17947299022576027, -0.01249935848379772, -0.0075184693769932654, 0.051736138252623486, 0.007850603202659625, 0.1476098475921771], [0.021028476321713566, 0.014589624034042493, 0.031239097556128866, 0.023548780902022615, 0.012433752379113958, 0.16024027836477098], [-0.2297869449965121, 0.027766739769736985, 0.0029136531953542397, 0.22536744350837745, 0.007985223287855616, 0.1222910280923309], [-0.5087078652205013, 0.008070221137042375, 0.020366283807531988, -0.02214115761161636, -0.005421518134746545, -0.03227763064437922], [0.21387150451420064, 0.04407053332241446, 8.46885966941604e-05, 0.0785976433551968, 0.012804162946167147, -0.03259449233063284], [-0.14792187745001342, 0.0008195441645769221, -0.00032377859022750645, 0.37542857360949733, 4.675315135728173e-06, 0.08800619628436268], [0.3631230148923655, -0.007251243019602219, 0.0018811091042383874, 0.028988039478266853, 0.004845706494447875, 0.038731791129378156], [0.30578223904138224, 0.012018969813494743, 0.0067148638483021635, 0.12952186453868317, -0.015327474375315557, -0.01694712953321397], [0.3616290110438489, -0.01111371933137667, 0.005321218762803754, 0.08837474214106084, -0.0048448619580766334, -0.019853057324928782], [-0.49287921262051815, 0.002565103103344752, -0.004885504927997474, -0.031020016514859416, -0.0011513882241586712, -0.03811564748247912], [-0.42419975145551136, -0.020988265017499848, -0.0128930084340235, -0.12488218353593979, -0.00628712634678896, 0.05018033478976357], [0.36111092200013545, -0.005056810683376384, 0.0029147486753155942, 0.016983637682588773, 0.0007758551003893659, 0.052368313891611284], [-0.4028927446362159, 0.0021384089290786227, 0.0035312273688158603, -0.1336640955996794, 0.0009072429068256753, -0.020351943730730623], [0.4244115021027415, 0.01079430178794552, 0.004076577547372478, -0.07193417562281679, 0.0067308313593275595, 0.0385176294920955], [-0.2864901066236036, -0.026018340342488396, -0.009232877122432906, -0.0828414366266168, -0.01053191660156737, -0.10867501965298884], [0.39240619756085965, 0.009922397324943566, 0.0015271869514881869, 0.038142689727681124, 0.009037665441643602, -0.01760905704255795], [0.28846599380703003, -0.00558644165124431, 0.0013004115467425395, 0.10779845364513042, 0.0022465228704146097, 0.03778839311526079], [-0.5544805962068877, -0.006361955164371138, -0.0020936802408976015, -0.013651834380956865, -0.010675696368328805, 0.034693762361437994], [0.29510077143717234, 0.008717063244067954, 0.0026682695439931947, 0.07094142302021267, -0.015914606850669213, 0.04829803198617446], [-0.3383446932309177, 0.008683392968656233, -0.013903430978590392, -0.2154230642590234, 0.0010217512394381848, 0.03402699664139036], [0.17052943216133445, 0.017173107759210954, 0.013183515699165886, 0.21553181054308918, 0.006560784597293555, -0.05456055552199709], [0.4316346213049885, 0.01390328999723418, 0.008974505140663912, -0.19492963559596913, 0.005908715678266389, 0.038353005639318004], [0.27898763907989704, 0.010385367134644328, 0.007919543952939883, 0.15670013627689072, 0.011918373204792965, -0.04874894089073086], [0.40547765555600446, 0.012039517945461061, 0.0019295414251130187, 0.01906355116145216, 0.004522230013822618, -0.01777673852609952], [-0.573533102609576, -0.00019524049527524544, -0.007562656987784671, -0.008975879256859139, -0.002824998418551462, 0.04334450637345454], [-0.5145812178545671, -0.012073636163282662, -0.009463331996460624, -0.04031618044887956, 0.0007457841114138425, 0.06230279287808842], [-0.016062583075327437, -0.012436206616773623, 0.002407556677994936, -0.31121888243589224, -0.02279816175388408, -0.10018501339440843], [-0.32321840225245185, -0.018271238170758996, -0.0026095899359676734, -0.16614517324631942, -0.004220872591127868, -0.041521390470041664], [-0.4639822870814734, -0.0005786312057791351, 0.022241392460876946, 0.04194117660253967, -0.0067195839985608324, 0.0011668221112822324], [-0.4502838721795511, -0.012399930312820237, 3.0916791387080136e-06, -0.03515153642607833, 3.955644786723858e-06, -0.054741708405477596], [-0.43912967047659063, -0.008048165627863339, 0.005249265901604665, -0.04335216297983164, -0.0068658856566199515, -0.05948588116070192], [-0.16690348615443015, 0.010047615814462332, 0.03775082048845289, 0.14421128015747425, 0.013812235622683354, 0.19228573391522846], [0.19369548226907457, -0.01007124858799213, 0.007466574401181228, 0.10120168305408025, 0.007382294230275752, 0.09140997653814346], [-0.4350770142143103, 0.003123719397082816, -0.012023344792654713, -0.0768804255586464, 0.0020084609955563788, -0.04413806249369595], [0.0419144459646709, 0.017549695473763935, 0.00495724330686379, 0.35151737524691834, 0.014971413678340371, -0.0287251611392281], [0.18703472325689943, 0.014141746420076474, 0.007165317548050344, 0.1994727748860422, 0.012492926991533333, -0.06022870367749917], [0.09682124028464664, -0.006973643929953144, 0.02172542759153931, 0.07880682170293436, 0.00367406245532539, 0.1764293500408705], [0.002599940173638513, -0.013719879697678565, 0.005031423702802597, 0.013191175486850447, 0.011926163967233169, 0.21187292777958175], [0.3475590280637134, 0.011421368413429337, 0.00935575918936872, 0.07184716288215261, 0.0007229476928108754, -0.01802085294408346], [0.4692466374044846, -0.003546130157890405, 0.009650587100241267, -0.055657964725233766, 0.006433518711446515, -0.009613314999717288], [0.2366596075575757, 0.010744561165883102, 0.017919244224781136, -0.11804290595066456, 0.02355253019887279, 0.16119418502577604], [0.32119884460369197, -0.009943222993042433, 0.007916581711097517, 0.12574280952309577, 0.0007997582115432057, -0.013201437723053098], [0.09734879608279123, -0.06307677380546922, -0.015584976590531477, 0.0080405185367043, -0.014124844804141286, -0.1297244177210513], [-0.19632182691126157, 0.003716083269605831, -0.0007871011695170941, -0.186513848294659, 0.0027185345969790117, -0.1503818414911464], [0.41950181088834226, 0.008560884094997993, 0.0016483272211436151, 0.026413389835253938, 0.00022644690939280403, -0.03017085894913458], [0.30704710896292337, -0.004315211083387957, 0.012898592912213161, 0.07053749417555179, 0.0038833034007082065, 0.02923982274310207], [-0.5671234932929408, 0.008181623567158902, 0.0010976493256890864, 0.048978267565066416, 0.0017341260751460093, -0.027688173240123865], [-0.18814496495650554, 0.027299176574856015, 0.017346436543993312, 0.24614931909278917, 0.011864970400371537, -0.03395622178349111], [-0.2736650730039344, -0.020227020658336658, -0.015455046088374919, -0.1942439227108366, 0.002601873591285523, -0.05891414446313577], [-0.5482443881265713, -0.013035363013387079, 0.017641983123724708, 0.05642776134748161, -0.006529386810855024, -0.03028590750871633], [0.357765737449974, 0.0018393496641628847, 0.006237595469816889, 0.009641742755026838, 0.006028742330035774, 0.046305250410078914], [0.3528000586994397, 0.0028817639683397143, 0.0017351773266288354, 0.02447441557221309, 0.002673399752834049, 0.04389831309584184], [-0.5524217759463195, 0.014148369900650876, -0.005542711316155257, 0.08955324982652438, -0.006515056635529297, -0.03161350440060428], [0.25553483432699764, -0.01244028263825685, 0.0031971621590311003, 0.13041339199950852, -0.0036434997595265867, 0.03713029867415229], [0.4057988402953959, 0.0030684095551178118, -0.005913624828311103, 0.03627756519774127, -0.002863183157841877, -0.016646881520979193], [0.4457060161527846, 0.016578903976269883, 0.002477186417573329, -0.19586729983831058, 0.0011900088070431883, 0.03128566067511349], [-0.19551858704465025, -0.03283329086119694, -0.014257585392220646, -0.05413548028304126, -0.009793203802095932, -0.14225569821564046], [0.26263741526564893, 0.03682958966175092, 0.021860694433290907, -0.11581562250827086, 0.02055470870438717, -0.031982373792101046], [0.294818932708516, -0.007188259463922657, -0.013164685225132852, 0.09000734335474216, 0.006600396551296746, 0.05683246255069063], [0.33757602778573215, -0.007660485570063686, 0.008234576770514148, 0.07380037594495233, -0.02202462483417303, 0.00615889180779908], [-0.5125573969579239, -0.0008431955366868375, -0.004917162576442429, -0.009254024701024034, -0.006384137668247154, -0.029343249226343793], [0.4720452845049102, -0.006393259833061145, 0.003950381658173872, -0.05675390415853971, 0.009668328003801597, -0.04136460795306647], [0.23143159725952292, 0.008187261826036853, 0.0037746797437974095, 0.13555835366186955, -0.0013016504768698395, 0.05133278828867397], [0.4050858733721884, -0.002760078253502098, 0.00026757954579362015, -0.012206531959624246, -0.007716657792914948, -0.013841375388133124], [-0.33704938063812967, -0.031003261485021926, -0.001085444911477512, -0.19142994628937626, 0.0007771252695290335, 0.026601860435427806], [0.48639753788078866, -0.008827148484094643, 0.004028372847457603, -0.07819134866886823, 0.010420171867858876, -0.014463411773677368], [-0.308735120399634, -0.024192980302412428, -0.01231938218967055, -0.10762808716728171, -0.003380418397357568, -0.07723067821031093], [-0.19130947359928785, 0.002270423313700415, -0.007391712856020444, -0.216757095069341, -0.0079606535549456, -0.11478945360206916], [0.3165705717722258, -0.008835867229520957, 0.006766290548176857, 0.07532069846782263, -0.0006459764919522036, 0.0432863342152968], [-0.20331418275579, 0.00043643794785251194, -0.013347564404253961, -0.3690927937998045, -0.010617641967743207, 0.08816336402735762], [0.08346215033184638, 0.045172793326741276, 0.03711471560291137, -0.19071482188549915, 0.04206930218431841, -0.0414589519701299], [-0.5682882487140307, 0.0024215464681271873, 0.003928905908880841, -0.0029491871832825053, -0.0014974803102246854, 0.024202881909621513], [0.31637235628938437, -0.012161126488720636, 0.0063382719449415455, 0.1210507546047307, -0.0017417219106776486, -0.04454300330412738], [0.32041572285180475, 0.006950904395817605, 0.001405704080091272, 0.1289365003897382, 0.003648457298299595, -0.029957990770137024], [-0.04277851435450539, -0.017298520432360937, 0.020545991752625947, 0.3900508028609677, 0.011933271375637556, -0.024917872472207194], [0.2584692289740427, 0.034449649960587615, -0.043335416390059237, -0.23020483566901367, -0.00031622324581837545, 0.1387961677988332], [0.03663410319537858, -0.033229603315973986, 0.0025268795967927763, -0.40009049233725363, -0.015882103231461157, 0.11030653355283351], [0.37049790871964533, -0.003609647350445867, 0.0008929245321418969, 0.022988680034792257, -0.004374066450744284, 0.042117533847940564], [-0.34741173415071863, -0.0069473283307406805, 0.0030322326378214655, -0.12040657485107788, -0.010015181660925855, -0.0613214136443584], [0.3990997621690207, 0.026236606434468613, 0.009922260247569608, -0.09464839700477243, 0.011990730881112589, 0.05175941245297675], [0.005531991543827298, 0.021901242471936474, 0.001195838485691212, 0.06555995350895032, -0.0059790613760845645, 0.2016813448894885], [-0.5088175900388867, -0.0006937483375849782, 0.0004235789268169271, 0.0011709435129408621, -0.016209207875642385, -0.03017016666383665], [0.13035297905172294, -0.05226893932287655, -0.010535738322242324, -0.2708350903163843, -0.027028150745907753, -0.11903247737172974], [-0.47064918134684675, 0.007686645463732505, -0.008417788491582584, -0.09112562338402086, 0.005435200629947385, 0.046042413795434355], [0.2631256840138938, 0.023196612736572513, 0.011457367484315273, -0.055259608024970465, -0.002411552572674842, 0.08947030588667258], [0.35244980172872464, 0.004033316736386631, 0.0042537123121402275, 0.025805056631113536, -0.01679803129756352, 0.022686143889198304], [-0.5727392484752057, -0.0056294123236087885, 0.002578918345042892, 0.0362237577046958, 0.004426554251045286, 0.036874986053578564], [0.48858333306113466, 0.014310492302949847, 0.0075488653105056465, -0.07923871777874122, 0.005612499913859428, -0.035025361698600564], [-0.44683813565357805, -0.012805225834839125, 0.0009212295542153447, -0.06285367656240641, -0.0009749405114159764, -0.04043591765864289], [0.37161344189287354, 0.009494119681847414, 0.0013097270120513587, 0.008122764608172147, 0.0009765640960686332, 0.035989300788081574], [-0.5088682255915011, 0.0020711488752912007, -0.00352157332226101, -0.017052954619250712, -0.000831497923955686, -0.026866897418324907], [0.2462774912656463, -0.013924604344403626, 0.008550373541440809, 0.14696058043641055, 0.010273306254308638, -0.014295565024839191], [0.30517295826766416, 0.007295778096403465, 0.0036326738314870145, 0.12612478429981056, 0.003123367078623385, -0.01633622824065502], [0.34359597583092777, -0.006603442065513442, -0.0015608646660817082, 0.09460672523053466, 0.0009809146344243566, -0.016675562333565735], [-0.4617304722485446, -0.011982101774325666, -0.023471223768902105, 0.059456636751084924, 0.0015455914613828844, 0.02314529973802923], [-0.511586989238606, -0.010974867568034005, -0.001278056190121503, -0.013352095403314466, 0.0012602448810793273, -0.02705490314767159], [0.41884685069553323, 0.016470870288344423, 0.0028375900051583213, -0.0748482278258247, -0.010042978711297421, 0.028844466976655485], [0.369648636944398, -0.005672246530700944, 0.004139822481998313, 0.002012643832536839, 0.0022952350055073433, 0.05347765967868855], [0.4542921642445511, -0.005090029978023524, 0.013423859157160859, -0.06553358941260509, -0.019015107980207713, -0.030970347978929145], [0.27729037220807556, 0.0086386234000121, -0.0030556599272035913, 0.06494517385453558, 0.005601684617745817, 0.06680483902403948], [-0.37364778652618846, -0.01036231019434111, -0.0015029962942996974, -0.14474353221130665, 0.006289731927784948, -0.021823344796888845], [0.36641067850068393, 0.009364131788603403, 0.002885019205532243, 0.06618784326229436, 0.0005713099977498502, -0.01603423083313248], [0.22362739012716012, 0.003736226829135735, 0.02056566065581712, 0.056774977553386057, -0.001594324380553076, -0.033987560655076056], [-0.5400258541666296, -0.01908830908273078, -0.0019772801144922673, 0.04851833757717977, 0.0014812759671848537, -0.03731150351384822], [0.4072793080655892, 0.019305573582381275, 0.009380474481447095, -0.05215987695876046, 0.008548222785585834, -0.019751433048680288], [-0.18815267820817383, 0.017679202155643284, 0.001864281108147456, -0.21246439862635927, -0.016102111398175696, -0.08832286645965316], [-0.47919722518811814, -0.006636717871909195, -0.0009161791548377157, -0.019698783805653867, 0.0003987685275052764, -0.04776528375240833], [0.4100101948245006, -0.013048032444408846, 0.008717433976981956, 0.0035599347551941536, 0.005687160408014834, -0.01631812009171355], [0.3242869411917583, 0.002626268912624387, 0.00031825212592704244, 0.057748339358543704, 0.005358031959579372, 0.03928734189016172], [-0.16677561273089925, -0.07158218163547883, -0.024340782716650532, 0.14653400303832437, -0.012216098703280187, -0.12347882941651733], [-0.02763159967142369, 0.012794613398913678, 0.013783187668270892, 0.36219220027543275, 0.01816133889131569, -0.02052251834028671], [0.3993966361091326, 0.0019939313498316853, 0.003249950294248738, 0.03218999324541053, -0.006602506968520506, -0.020936892918994254], [0.26320584298032923, -0.012028443698993441, -0.014395528841187437, -0.4036537823903664, -0.025331797129791882, -0.07861430679300659], [0.21730444226731208, 0.007935672409450421, 0.0042237319269003955, 0.052578015892228026, -0.002029169338283463, 0.12650064017572532], [0.4132739072764417, -0.007478667518581405, -0.0026027658918828804, 0.03634614916830712, 0.003951355579178309, -0.01514731423382069], [0.3214193960263628, 0.009239418816757666, 0.00415578910092818, 0.06331670624281828, 0.006116263640510006, 0.029953259505954462], [-0.509647615317705, -0.004134854460731156, -0.0029082826277070464, -0.02529722920626368, -0.0020811820061669827, 0.03266536619654994], [0.3603259154758518, -0.002856766012399522, 0.015299896404696882, 0.007622417917065581, 0.005135313030592144, 0.040153223184192344], [0.30957045243398096, -0.01159909843476554, 0.0024687436955039034, 0.13408898511043882, 0.008654763733437628, -0.013670513205262136], [0.4045367887668094, 0.013119157812106826, 0.005051743145025247, -0.0694659673072378, -0.002716169122348222, 0.05622587527707161], [0.3344836676724883, -0.009786807016350284, 0.00611485425785877, 0.03450317416917216, -0.002933415685391334, 0.0554827371285361], [-0.27588147084606457, -0.026607495656628996, -0.011817280673938387, -0.2676003426673589, -0.0004381667553062731, 0.04519142326596519], [0.3788638132624918, 0.014288204495371128, 0.010516492562951292, 0.03646733209950176, 0.00032581291225112634, -0.03496617914209418], [-0.5857006455484991, -0.019308840410878537, -0.014019140726886193, 0.12886621621707534, -0.010567860645793303, -0.030714728885020744], [0.1705171300712613, -0.02598163750326149, -0.001232157452834879, 0.06101369404073255, 0.01831418916654421, -0.04034100908723195], [0.3749503430389337, -0.009834214016426486, 0.00022174027982465423, 0.020649600275575643, -0.0025883538626215235, 0.03979842814436055], [-0.21480481059892945, -0.0016785495010893554, -0.013535103367746059, -0.3378904485320195, 0.002774598548312674, 0.04136193249908979], [-0.4667240395683361, -0.009729144197277537, -0.0030659172011731424, -0.015353883874660737, -3.060569356723759e-05, -0.04933307613165514], [-0.30091348881223146, 0.009833229363633716, 0.034771927316825914, 0.2284046949895741, 0.022328952224524974, 0.11534397063195764], [0.3662620983663228, 0.007543696467198254, 0.0019023027530732325, 0.016252814385316748, 4.105548111795808e-05, 0.03767505635649193], [0.33718994687760917, -0.004570458135172615, 0.0016111359176390534, 0.038144216627434856, -0.0077309418331226315, 0.05674443387894473], [0.0360156759937658, 0.03638835635496391, 0.006223076481496051, 0.3060415297345487, 0.011668513334855699, -0.05457421729038126], [-0.4116964361159341, 0.006240673669455105, 0.005531317308340614, -0.0538742683630864, -0.0036462280223921533, 0.02759319549186963], [0.21851515262240492, -0.007858508670759929, -0.0004934999886251646, 0.19958907350817515, 0.0060308035618587925, -0.019214132144165244], [0.1792703843751967, -0.030838043555303955, -0.030065204390376292, -0.2757785151147439, -0.030220825463141397, -0.07068977997861428], [-0.5678726432059297, 0.008129235926890093, -0.008910417907650004, 0.048454042457400995, -0.002516403476532046, -0.02872502591539662], [-0.2542063453066915, -0.008310733324875547, 0.0419614976960525, 0.3904442453633953, 0.023772984338034785, -0.020517363051629524], [-0.02095445550994591, -0.004164777446308385, 0.027261203232018118, 0.21496631813595846, 0.027395169827947766, 0.035117017950806316], [0.2215904371283617, -0.022140972509343732, -0.007753498167302195, -0.43986348549966203, -0.014812174812847887, -0.11601292518682463], [0.46745956781578146, 0.003510773927923994, 0.005922504370744284, -0.03827697492347531, 0.006968722603349121, -0.04107126046099264], [-0.2629234156030034, -0.007381747993431753, -0.005762183557898577, -0.0836340500654412, -0.00509633450638683, -0.1302841730357424], [-0.4457364022914028, 0.007207003614850952, -0.01728906822784633, 0.09847849198884648, -0.0023317828029463966, -0.07556216169542271], [-0.0014909706113447206, -0.05268631451985052, 0.0029599114817810013, -0.3404659892737801, -0.03322079342437826, -0.00012219285877656768], [-0.3931424353915143, -0.011936834649741553, -0.006273600396264925, -0.1356891776744132, 0.0008343555147506329, 0.020613883073370056], [-0.3320155899309242, -0.006223449893806075, -0.0056055589568790596, -0.1606429486123234, -0.017469107692449207, -0.04353001158028445], [-0.4578995188897673, 0.002755686442954908, -0.0065343199556437415, -0.0949489022245955, -0.011711247213976947, 0.021315920888645777], [0.4383145477653535, -0.015301978672801845, 0.013547084800811113, -0.10667049480160437, 0.005270481761835936, 0.019698930574975484], [0.262026480259247, 0.0010185155938629274, 0.006101879035801722, 0.16652168697703393, 0.005181314850507806, -0.035044876716453684], [-0.30827154339559965, -0.03095534098046454, 0.0019465840170632658, -0.16314305213545655, -0.01733980041994705, 0.08122886720011704], [0.19589192329017904, 0.009390996857817026, 0.005950678339798562, 0.13555487204208216, 0.004763522759078113, 0.07132780179301323], [-0.5170855325375009, -0.006113225388181381, -0.011107771170944926, -0.05283350619299446, -0.0004681016706250873, 0.03911076556565448], [-0.28216519792106143, 0.015855385932062243, 0.021322917547118493, -0.06728967801362615, -0.021399940330504495, -0.10731195763245935], [0.381548135657731, -0.010678292334312904, 0.00483694837700765, -0.021701405570261095, -0.01615550588489802, 0.053764643564256476], [-0.5531714282811279, 0.005121965666729027, -0.004020869974878184, 0.055866070633123696, -0.006257225911635241, -0.02424927677968332], [0.3580043080864208, 0.01426446827088297, 0.01243598813050621, 0.0493033923532559, 0.006126954448748622, -0.014868745611469466], [0.41855310233558496, 0.011591471600537674, -0.001533839705063783, 0.02765087308232336, -0.002497796511394683, -0.03575047746865669], [-0.49713987429507367, -0.001633258005808627, -0.0022735580294287842, -0.04510025230189625, -0.008339390560191852, 0.029999666525728528], [-0.4071929586363984, -0.017261069138441727, -0.01031394190469952, -0.0631191509609567, -0.0009339565357212855, -0.0666655894904499], [-0.5235790448893538, 0.004511474560759922, -0.006164891586841474, 0.042201942541915304, -0.012168249234988491, -0.059371231391496596], [-0.20259352405913963, -0.04631377270809271, -0.004781858055223443, -0.007904848853453321, -0.004678818534625653, -0.18718545617774346], [0.3349707658256073, 0.004075209954436021, 0.0023160784426453935, 0.05039352961391473, 0.005488952330703243, 0.03601879716602597], [0.047908842779475526, 0.013776577659703929, 0.030539369319226514, -0.09427261235957511, 0.007721650982598625, 0.18252998114237964], [0.3236652553304934, -0.006714019468186971, -0.0035824742352805416, 0.13064569835525183, 0.00802581142924353, -0.028122176173427264], [0.3735750630946442, -0.0031412365938961843, -0.00748665154394498, 0.011467410262746178, 0.002514660117111591, 0.05258408799667072], [0.33294596341061755, 0.00876905363816639, 0.007278164749408504, 0.025664939122725456, 0.003064974873602846, 0.04978282228457424], [-0.512888499724715, 0.004621449113325373, 0.004416237623423883, -0.014275863594287931, -0.006292198832766062, -0.024678902362759464], [0.3041024347102065, 0.010200815919126758, 0.003439451036161918, 0.1450218338397671, -0.012193401115003189, -0.030224467723591913], [0.434701744065321, -0.007670153946110525, 0.004667235735825898, 0.011200165030989154, 0.0038796110776753924, -0.02478449939960185], [0.15688740970654913, -0.014194432353146935, -0.017136993773294768, -0.02005372607007917, -0.09688761512805741, -0.15673226142959035], [0.3919391406865818, 0.0012932790071764587, -0.003688689601394781, 0.04822516484363588, 0.005165356984373815, -0.024780981480123206], [-0.4472905292337693, 0.002402086589315706, 0.03139790924279963, -0.06582530332490673, -0.0080549679345052, -0.038240862005601096], [-0.5165013870138573, -0.0012065996970423577, -0.006178920833048474, 0.0002677566802813611, -0.0009593025137123474, -0.026574879955957302], [0.27364287334743276, -0.00843554227981333, 0.007102846293405072, 0.17857941184849155, -0.01492327305607566, -0.0174252050423297], [0.021406678708887007, 0.0072916313984006794, 0.01206355535246938, 0.09811735662457687, 0.015271932676700599, 0.16966018793998047], [-0.2631287208215903, -0.03886892888933315, -0.016069038252475902, -0.02318429604675042, -0.002896424051765581, -0.17320830622379943], [0.013500963234690076, -0.00882709446330482, -0.028793972691698616, -0.21835250665819575, -0.012456618426915674, -0.18945426305806712], [0.22276584216210293, 0.007437620138362018, 0.0013092060128371587, 0.14616103587897766, -0.001020648779697926, 0.051848373158847494], [0.3134201228272752, 0.009422672234549156, 0.0037812514382491295, 0.12688426147041293, -0.0035732757388798183, -0.01642169889827336], [0.3775171262302448, 0.0071904410193303694, 0.007907770868910096, 0.07275632240749172, 0.008911379425205883, -0.051882260730405305], [0.34366245711123194, 0.007522778352678568, 0.0009173566532529458, 0.057709469236894836, -0.00542493525702271, 0.013908258518347852], [0.27569602837874063, 0.023876974731126897, 0.024733563285549154, 0.06438966774267789, 0.014220727049988332, -0.026567394643516887], [0.4096518502537284, -0.0032828387888321317, 0.0074203993359821425, 0.04869189155182804, -0.0016575058662801517, -0.0329130272556588], [-0.014569775316285373, 0.018978628975448268, 0.011172890222249899, -0.1544244286629052, 0.021662085137807215, 0.18323559964368577], [-0.020357136305078308, -0.04021100965081382, -0.0006641943581798402, -0.3306585597318015, -0.021084783929993803, 0.003235049055233156], [0.3943089247244071, 0.016173711223981965, 0.017240725828515567, -0.11368517245178962, 0.014107703488557517, 0.04681188496410563], [0.3240652429912168, -0.00643723755883535, -0.008942923140742156, 0.11984605989296948, 0.007616560295334225, -0.03657026658250789], [0.44826079338962044, 0.005859648778520068, 0.004964763635016084, -0.1006850327919627, 0.01326850241399713, 0.0220657076916926], [-0.4846115818585092, 0.005112070228209509, -0.005446204685668202, -0.04248223750076329, -0.0009586819017080232, -0.02868336428156306], [-0.5226519769270965, 0.010622641926438714, -0.014442406793738058, -0.03888915168577494, 0.006044307534523377, 0.050472776421832434], [0.3306903371047185, 0.0048039630656415925, -6.691108026537503e-05, 0.07128909343510206, -0.005865532101270965, 0.033662382909406804], [0.4664925357122764, 0.01203082649236908, 0.0075923924734088545, -0.021997380542088826, 0.006749402156855633, -0.039479442959488985], [0.35129493951655255, -0.002899878291762129, 0.00466509420249312, 0.09836992020894036, 0.008961493583659908, -0.025878235886553184], [-0.32643798677026975, -0.030637865433064813, -0.027645256521880216, 0.10260433638692262, -0.0031375981698293772, -0.09759081264206178], [-0.26921411592993993, 0.019901707633704448, 0.0007665292408661603, -0.22311105086044583, -0.0016853497634360343, -0.05718605365408242], [-0.5555299550215337, 0.0014494221987246272, -0.002844724460019775, 0.08931694178448173, 0.0008313709570236617, -0.04004900783963127], [-0.5160998651548336, 0.0008249068345184491, -0.003133702196816416, -0.008868576377365057, -0.006743783796104095, -0.027611479309401325], [0.4085352585230492, 0.011693715740112043, 0.013816077860354976, 0.003618566959435574, 0.0022590476880494474, -0.022532349310685117], [-0.3519859521204596, -0.016145974503038167, -0.02047019750654198, -0.047581127896700236, -0.016449963230434777, 0.025063215257175215], [-0.516247912674208, 0.0008362997244360993, -0.0013823291976285906, -0.008616824082328474, -0.003714978550091641, -0.032194255220182065], [0.3649148747475138, -0.003721202062512796, 0.009848598439114169, 0.04781078550391092, 0.0009863773122200756, -0.031146613427427357], [0.006930509846216544, -0.03775892934202863, -0.024519128686968185, -0.28799152968460934, -0.009731412076379254, -0.11541617672289872], [0.12078668920442681, 0.004052204820571123, -0.013048336121412851, -0.21299640293512856, -0.006931750608907361, -0.14926573769288118], [0.28714913688935423, -0.012088927176438509, 0.006128366518975385, 0.11666297900943899, 0.012260638994502108, -0.05881909899773779], [-0.49981129295040777, 0.00826711798078255, -0.0020009743160849408, -0.02774594917316128, -0.005332787311108986, -0.028723892007801958], [-0.5422133149288207, 0.003654038995448616, -0.004510645646243084, 0.044347055322562084, -0.0018412420146218455, -0.033583272680709805], [-0.5672182396550876, -0.007165901974944183, -0.009135556009227785, -0.0068112657333478255, 0.0038452026318934952, 0.05438751215313696], [0.3359520453289373, 0.0022772193175131456, 0.004319514332551063, 0.03379125553646026, 0.0013295593925455411, 0.053718739425325596], [0.06321268412519002, 0.021451299755753328, 0.009171577542914162, 0.17481643129310115, 0.006883636717351293, 0.12006346389327174], [0.30727117897140477, 0.008227407166433793, -0.002608524169228243, 0.13656513414946359, -0.037689047272903585, -0.03525876789279132], [0.278829093750262, -0.006070595458140324, 0.005245212719047986, 0.11211691285318671, 0.0026500195507915365, 0.04149268991818562], [0.2042351060771185, 0.006285540712325352, -0.011480924414919795, 0.17010665416314827, 0.0059245399579264295, 0.04699544714076627], [0.3402875308992503, -0.006461817686664528, 0.004224843958186856, 0.06121714156902732, 0.004541552765067554, 0.03070408182846395], [0.2863371625983826, -0.006955223314275169, 0.003351539043711262, 0.1528287291630907, 0.009122269471522771, -0.014504476962432305], [0.04922794301803997, 0.017582067753360416, 6.741623167608097e-05, 0.12951425148176177, 0.021526532754571354, 0.11317845542725727], [-0.32915199896137176, 0.015348304163123721, -0.014672420428176161, -0.025968527358460166, 0.0019611958341598196, -0.09419766436038744], [-0.05252470074919648, 0.00273620525868325, 0.006997375110259929, 0.4089598504616172, 0.020437731478201496, -0.05512884251194648], [-0.2290715841552113, -0.0046792998061113865, 7.317553065658168e-05, -0.31467057725575176, 0.010115305650347156, 0.08037131336940499], [0.42109969586793017, 0.012709522321599345, 0.0018236886611769556, 0.013249194164413091, 0.007046442049418014, -0.024100394916392535], [0.308782186831593, 0.003130806017958588, 0.0055897331226560515, 0.11353313087306517, -0.001278275649877145, -0.009077581195397665], [0.19903163721615127, -0.0077667186061174025, 0.0023567164278708486, 0.15319228236454652, 0.006462262188102422, 0.06758239183801935], [-0.41026480846791674, 0.04924359958845384, 0.027342312974922144, 0.2942136787996458, 0.006309735784687746, 0.07847833846305886], [-0.5139824974203043, -0.01035296287369234, -0.003543888823587104, 0.007832140626444595, 0.00046338440412529243, -0.0392361759129883], [-0.2622142436082432, 0.016151677382422458, -0.0024933196020278875, -0.09565014245346574, -0.011436894903015032, -0.11688919802779216], [-0.48563246452026565, -0.010393525415689829, -0.0008155012977391218, -0.04612948845021331, -0.010418057212484722, 0.04131903689639144], [0.3970497561902093, -0.007377576994102486, 0.019448515688537626, 0.03484354765100518, 0.0036770313768157846, -0.03293854663973951], [-0.2615542971327149, 0.011589628008638634, 0.02628240182757727, 0.3444309867200883, -0.0018924656906956798, 0.05466303198139068], [-0.5687896216297493, -0.0005399835571284882, -0.015290113203938851, 0.1522409521594006, -0.0006642908359901742, -0.040447577853229694], [0.41499169956070536, 0.007205055988305615, -0.013301727200116806, 0.020169820667577724, -0.0007665514224032246, -0.013618297594072128], [0.34640885362875434, -0.006231723597916418, 0.0051941504041958155, 0.045499642334246686, 0.009328553640656372, 0.02774281867202788], [-0.4845790462825835, -0.014375838554281303, -0.010744892447198515, -0.07046798367151183, -0.003338572174433456, 0.04752363471730748], [0.07910333775559916, -0.04296777833843819, 0.0003213547994945786, -0.04479306222887337, -0.017606780048889922, -0.12481459275141268], [0.35459196972033086, -0.004402707228082693, -0.0007437320668823271, 0.0918133353007098, 0.007521507302996316, -0.03623673666543735], [0.30507123230639765, -0.01040636396678224, 0.0020174482616381875, 0.1449999295209524, 0.005489957460372464, -0.01765887024924471], [-0.5115167448029024, -0.00010125793688123636, -0.005595550571208627, -0.05981237261139001, -0.010160743072792858, 0.05104524042374443], [0.32527814376327496, 0.013831020135894247, -0.0035342691735829723, 0.0879807972008048, -0.0033706894435084813, -0.034903812006693194], [0.30030477572754044, -0.003498391329346385, 0.010959492860142871, 0.050823438506661656, 0.0072224847518474605, 0.036089950895582146], [-0.4924685512912607, -0.016114390649884337, 0.023054152667771072, 0.0411478926158099, -0.012652454613440684, 0.032149859207507965], [-0.26562282374600876, 0.017090658825728824, -0.005580329024984481, -0.08970984043211962, -0.007154127202310178, -0.10620428878105675], [0.1933304249779348, -0.0043738483012315515, -0.020597121068122167, -0.3903683096224525, -0.02145688313664152, 0.04966954667432286], [0.3104651321464952, 0.0016441525701497875, 0.0053602612060618445, 0.09132483903414534, 0.002316739851821093, -0.0025030945056419038], [-0.5657945957841365, -0.00814423752313391, -0.008885916252980137, -0.011433670994942911, 0.0033876414122800103, 0.048480863888670414], [-0.4837027263284742, -0.012281107325237725, -0.0031931004801935992, -0.010679698318362818, -0.002503072735056049, -0.0351461922485753], [-0.06593786918979787, 0.024031683398067025, 0.010087849145908106, 0.16519550922073525, 0.011273175184120592, 0.17107429509810884], [-0.3600513256174713, -0.010017432152052436, -0.02413004635867269, 0.0055580850559896556, 0.006641259428491631, 0.023745012871045038], [0.259274965266981, 0.010329278968725032, 0.003341791286164265, 0.10941490107615467, 0.006877327235193191, 0.02592990844173362], [-0.21364369883551382, 0.008598069745660417, -0.012296633697820868, -0.2822201830851606, 0.0017718555958049543, -0.05586274305630414], [0.2937544540583764, 0.005832086309255034, -0.0004906240285480723, 0.15150786634007155, 0.012183866811840676, -0.05499401312735979], [0.4329385386747094, 0.01036237021868628, 0.005729722548920239, -0.20987165358190996, 0.005237323595094938, 0.06725988902068884], [-0.05567261048758628, 0.0547567887948646, 0.001778907928984235, 0.23752988827413865, 0.004620649299236147, -0.04525699839103064], [0.2833673114795729, 0.01133140819460563, 0.0034442454444115833, 0.1429857598159109, 0.005213701248308316, -0.02183598508005218], [-0.2462773282732745, -0.020079544184386122, 0.0165148556665666, -0.22620070701253775, 0.0002335542380379776, -0.05217749710107302], [-0.4069101274497923, 0.0034835499696079016, -0.004885406653139298, -0.020301798265486424, -0.012018806999964191, -0.08277074393456127], [-0.056887620696663393, 0.021891914599952277, 0.02277176454895971, 0.2981053370490861, -0.03250903984812086, -0.01587092708178587], [0.4334782360716472, 0.0024628829177678335, 0.0005052873489643806, -0.005454511428335789, 0.003532296800811232, -0.024689429806094215], [0.309427347224462, 0.007743454618981888, 0.002639049172407493, 0.0689215827915004, 0.005711443053539842, 0.021719115190771576], [0.10571101566156652, -0.013297924789387175, -0.00856677320669362, -0.4351709462545826, -0.009837283182146255, -0.11904697711764468], [-0.5270083945741133, -0.007967679936757633, -0.001675408093393908, 0.07869486807665949, 0.0061722208051178526, -0.05211893961085135], [-0.5567819380407443, -0.0062868978219531675, -0.010410998339028602, -0.015511371732590675, 0.002338755438774888, 0.051127434231199566], [0.3440227477611899, 0.011590491368867329, -0.021590420941686932, 0.026352741310588363, 0.009026417109402192, 0.05282663450274832], [0.3109165251754884, 0.004270074700845078, 0.006429208769248378, 0.06746930969649242, 0.005800641533325171, 0.03826393709429657], [-0.5675357261202493, -0.021346469417207095, -0.021280375362949317, 0.12844419703304363, -0.01532940356474854, -0.03845674637741336], [-0.09500553882970925, 0.04606166367368672, 0.003860329423462571, 0.18079945527599353, 0.03063466779343221, -0.03700881483312128], [0.31558003124273193, -0.00028415182205754233, -0.009931768936179596, 0.131230995079155, 0.00542791616452331, -0.01020705681589373], [0.3015812668920599, -0.02170629431127401, -0.10774353468020918, -0.2795402317984224, -0.010130149113927319, -0.07214812048028994], [-0.5266887243672086, -0.017205505477929664, -0.007466393993298368, 0.037924346678735954, -0.008179752017903068, -0.021662304155735057], [-0.4896384231423078, -0.012357099858438027, 0.004844547365931623, -0.016430958255935775, -0.0034359732759510187, -0.044421140452348115], [-0.4568635590546866, 0.0013193109442549966, -0.005860599222055434, -0.07484466042764258, 0.000979599106048855, -0.02627231356814186], [-0.20351926446001647, 0.0008335836806129322, -0.010386323893760074, -0.3624139363900855, -0.025634316872308587, 0.07296692460222363], [0.20899129702229255, 0.01848503361048123, 0.0060096368117900706, 0.13361953419741698, 0.003802376876868034, 0.05370108322978566], [0.3417975601234268, -0.0045566972533680255, 6.05478668418865e-05, 0.05947294686106988, -0.005344125990974038, 0.0426984863417194], [0.14486315359149587, -0.01092104108005807, 0.012134257690684176, 0.19678608649410798, -0.0024242957501677863, 0.057634696196795704], [0.35137135100290184, -0.007671886935416379, -0.004422432853691795, 0.09764120112785185, 0.007322283892882465, -0.019697420996434664], [0.36325583484224555, -0.012236264602819248, 0.00019796253879683908, -0.0471573874315345, 0.00818845516774962, 0.06824092329508516], [-0.5125639203919078, -0.002431504700401222, -0.007972377803980215, 0.1592699099741775, -0.0010854199281816741, -0.03303668714970962], [0.23246007863943685, 0.01395450667271546, -0.0005939605060606662, 0.19034433249502836, -0.014829809797898587, -0.031364420686179104], [0.3081192439758774, 0.014278154631363312, 0.007010714645885011, 0.044046491196051814, -0.00374996292099091, 0.06341980291625775], [0.2365994312639036, 0.03639885062848112, 0.012389625777595796, 0.06940972574630136, 0.012691485168391966, -0.07773141091498433], [-0.3960854359784266, -0.008412467186427165, -0.00998642759091453, -0.09465716592697354, -0.0075336809078247755, -0.03381148907610108], [0.3102211739911142, 0.008537320262060985, 0.005081441052776526, 0.13220267059055174, -0.011561380809981552, -0.017301225086521342], [-0.20396968014430072, -0.027983329616590846, -0.018982262427390772, -0.3121443465690059, 0.007389740848134799, 0.06090559219486995], [-0.5285909723566481, -0.0010163327090965592, -0.00632017404474863, 0.043396305199837414, -0.010169278408613038, -0.030119547680734972], [0.1923214188111191, 0.034703444193339386, 0.018673452848103025, -0.07051282087882493, 0.016776276940778642, -0.04098884262158636], [-0.4957471198936124, 0.009754794177747937, 0.009262347332732308, -0.0923778943074696, -0.004209503074698866, 0.047902137670060414], [0.4675939474646752, 0.020202894600228758, 0.008743518537504673, -0.21000646101010695, 0.03586580641923323, -0.0387355790274122], [0.3312239284312429, -0.010901201405016922, 0.008876120580893966, 0.06699420950921058, 0.01371545518842761, -0.05976422659047428], [0.4511474466236156, 0.022335611902512036, -0.0010122922714507557, -0.09883590479687022, -0.03596591231538882, -0.022167520570992857], [0.2394679712740439, -0.014785630186891044, 0.0024532550452938963, 0.07726426580532451, 0.00862662819036115, -0.04088556216720375], [0.2032285571599133, 0.012647006082439587, 0.0022876206431378097, 0.18046126053782116, -0.002441300164278837, 0.03145518907430132], [0.10510319827806638, -0.032372434597304996, -0.0015511238016824888, 0.06209656516710097, -0.017784640055266247, -0.1694742634036116], [-0.3421458411067216, 0.012362966573339983, 0.015865657299888546, 0.3379166406695703, 0.03379876622713793, -0.010396907290618576], [0.4324358960815725, 0.008041654370643836, 0.005410531299660734, -0.07032491701251907, -0.0032978275262429036, 0.05129345066567142], [0.3964921027953506, 0.03275925947827096, 0.015803420978270324, -0.09519838109284297, 3.2725196774111466e-05, -0.0010841273558233062], [-0.19805243024288446, 0.0007380469626323106, -0.009316500652824876, -0.23268112914740477, 0.004631956282075796, -0.10005660986825941], [0.4139140025704948, 0.015806660677085475, 0.004829843608946493, 0.029843861409290923, -0.0017411975259880928, -0.03313983740649896], [-0.18179201127769679, 0.028830317768441742, 0.01051804171546418, 0.37774407143101485, 0.012387031659533832, 0.07424850108419234], [0.23969636324665614, -0.035944316876134544, -0.01909976195898782, -0.3556045431610278, -0.026749140119693007, -0.009618601130811935], [0.27188036641010443, 0.00918319868125353, 0.0003888437566856333, 0.09392426939617264, 0.0010889780574852594, 0.05596677035921356], [-0.4603988622804309, -0.011983736709574415, -0.003415710575698978, -0.08440862034850423, -0.0003003926400334148, 0.05360398922090518], [-0.5020210344996828, -0.010733461065677205, -0.001077762719899196, -0.008796488169171814, -0.004650905220468512, -0.029040348325102328], [-0.41026118202301437, -0.0062618192894147735, -0.011865535389893378, -0.16505500496314354, 0.0003994117157304596, 0.048783653759256654], [0.3187313551220884, 0.0040979482590970725, 0.0023455106716168067, 0.06211274624310338, 0.005288050276970961, 0.037771056093787905], [-0.37570972754144927, -0.020285211132840434, -0.006023082782747327, -0.05542816907257966, -0.016342887531383965, 0.0979489235079495], [-0.42967069421585613, 0.01635865121789983, -0.007590965854908356, 0.11115248205855964, -0.01924855256058295, -0.06453322223241557], [-0.2591627393304498, 0.0033394541636783182, -0.019348151432519625, -0.02975036892767434, -0.006397841463263965, -0.11900110854354434], [-0.4260463452880449, -0.0094661720060749, -0.0007623750735524692, -0.15897965112179352, 0.0032973491757584805, 0.04897052764703878], [0.17728670884194236, 0.006041146616208965, 0.006089348079628268, 0.23364447281225753, 0.010122304037347023, -0.02898880672089483], [-0.43041366496043726, -0.0015973571280685072, -0.008563600950057058, -0.1600208690408965, -0.0016341190268635715, 0.04315961110632267], [-0.3574282133393323, -0.003794337825647996, -0.005822315299048103, -0.15663170719905267, -0.0003343959731490404, -0.040642363697104535], [-0.3832176363252366, 0.034264712831849876, -0.00034289695951152, -0.11197596044626766, -0.00319494034921025, -0.04762708827543339], [-0.25983637342527616, -0.014109822069901117, -0.008664483945819215, -0.21134356928340686, 0.0011579136023383456, -0.06054747440174578], [-0.05668015632578793, -0.0207511417195523, -0.004575935822705473, 0.37923544915324786, -0.0027979173203084022, -0.023083631298228043], [-0.5087392863625788, -0.009894925166286156, -0.003586590147997392, -0.012572618382529394, -0.0017210787808471341, -0.02599139859566031], [-0.5749148235185433, -0.01824566685505449, -0.005102275755883699, 0.07807925608886832, -0.011901967686888419, 0.03420595391797221], [0.3830565494533995, 0.004723395371869344, -0.010290810428069688, -0.011393012978556202, 0.005335048499443931, 0.04520669260545268], [-0.28094555895233947, -0.039291264883058834, -0.002720074830479204, -0.022715721909715818, -0.017877332195581023, -0.119779787488567], [0.3334550020885723, 0.00712439276218091, -0.0010003902556560123, 0.05153353080583778, 0.0027426059973453373, 0.020866525268384197], [0.3351720507774748, 0.01461355233601817, 0.00562970099514972, 0.08619282745278405, -0.013489931113623867, -0.01611677187637605], [0.3313077047348871, 0.008456317763528147, 0.0026160723906767094, 0.02989884774496049, 0.0008197841392111833, 0.0587132176711803], [0.43127666692244043, 0.007057798957293574, -0.007820017018620997, 0.007713064782322104, 0.004717537184886398, -0.016765050828325734], [0.3509712978148186, 0.0057163358717990335, 0.005777975096274041, 0.012694184770611188, 0.004290624000379627, 0.023701333858545386], [-0.4428678380289042, -0.0009165585222438202, -0.0051699177891616046, -0.1359770846563868, 0.0035765622976851445, 0.04795150336567579], [0.34552136484413154, 0.008632979590293783, -0.008225337982907479, 0.04287612850800019, -0.0028151136197034432, 0.028451883422088983], [-0.48148600859447366, -0.007207780985540885, -0.0011873102503996127, -0.024319870052129614, -0.005177163449225177, -0.03194186666823304], [0.05022960401052477, 0.003317538824126925, 0.013010008363500565, 0.20472660888581176, 0.024070266604656658, -0.05698051815313043], [-0.37710037811419, -0.01735613954737311, -0.003175247481199, -0.04571911749212832, 0.0018426802090591627, -0.07995465471702715], [-0.17530368442574976, 0.02441846493969733, 0.004328755148557985, 0.34932345857623154, -0.012989575594942415, -0.018156942453319943], [0.41722627433024134, -0.0007774930396696746, 0.0008681321495768591, 0.020539329454797084, 0.001852464994964714, -0.01613877078299496], [-0.4407518072216636, -0.015179328464173289, 0.011502446522170832, -0.05392864813247515, -0.007725891068414005, -0.04058933573800926], [0.41605200792649893, 0.0023808792889093685, 0.0010525033491633185, 0.012199140807483097, 0.005831167591380903, -0.017167984078741858], [0.35854719432384524, 0.01094630276960904, 0.0007212702621250788, 0.07078327814551416, 0.005923214101151774, -0.014003236370929196], [-0.2544581549174935, 0.0019057143107657228, 0.014791711880220179, -0.23111963014110268, -0.008298119191865997, -0.05647882352782588], [0.40912847388733264, 0.007713672275870339, 0.004079395821444476, -0.04229626728275895, -0.0069784819697653756, 0.050545112029781174], [0.47095230128348387, 0.009288454517258312, -0.0004231401397905884, -0.0675604828005618, 0.0010417654005615018, -0.05295223159428815], [-0.25162585901886675, 0.0014904835859259205, -0.015309187822168658, -0.005730355193894302, -0.023242033080614573, -0.14333739190472447], [0.34950013560050586, -0.00478198517022603, 0.005549286428904703, 0.05347730515374203, -6.84795825455088e-05, 0.019587070902950362], [-0.5111213733002937, -0.000519134572824767, -0.0066543788330939825, -0.04142653563157876, -0.0017002873894775515, 0.035289209727264195]], 2: [[-0.26747409280655204, -0.003227718216386702, -0.007100565540953364, 0.005747571047259515, -0.0158712045119616, -0.10883065663807114], [-0.14353713914636543, -0.023700369010401314, -0.011955171081078728, -0.19477297718248737, -0.029594556491009415, -0.0044234041099349836], [0.15951622838359722, 0.028182160412657954, 0.013810228913618276, 0.3141197851076884, 0.027209489605346377, -0.03728338779124467], [-0.05626274195372172, 0.008786660737880295, -0.00045325738817399283, -0.29749092750565936, 0.0014578664441385343, -0.041782362239227634], [-0.30299056776907635, -0.015892643612221594, -0.0070401716800763455, 0.01869792808099262, -0.0002953724824865442, -0.10446732068527981], [-0.3633385679013549, 0.008202451100867059, -0.005295040484667807, 0.13577375265343097, -0.0027386785883195134, -0.15958040805309617], [0.2856958113335546, 0.026779216758244146, -0.0039221627475438, -0.06899831853407298, 0.03416976854025092, 0.18311028782416988], [-0.21528802907496683, -0.007752134050856766, 0.002360674501278702, -0.11182787229160414, -0.008738315769456976, -0.05405265664772557], [-0.30647084131873076, 0.02209427818066617, -0.011357501480717078, 0.11329370677640373, -0.013250932819027604, -0.143617895807004], [0.50535672189, -0.0023755865919177486, 0.003895682716562542, 0.06341901828295805, 0.011471216619879519, -0.009023719584150845], [0.4778117280033331, -0.0019956951766662177, 0.004759412446809434, 0.13973697469549995, 0.009691690886660146, -0.049677444188971694], [-0.286598099368564, -0.006019884745762667, 0.0033513858504073232, 0.015965717266409312, -0.00038157737373937454, -0.13182420829541694], [-0.1882147394706078, 0.01071169639460034, -0.002676816474721827, -0.1830000973115995, 0.004142268198193787, -0.05013564466919804], [0.1127245273795182, -0.017858709831275833, 0.016263974072999794, 0.37890651759557026, 0.01751766084446068, -0.03944651521726727], [0.29291766107555467, 0.01262123816378458, -0.0006517420684172076, 0.27383636527410526, -0.014997713929293709, -0.008064443938283556], [-0.07746308109624271, 0.00903719693572661, 0.02269113333947228, 0.3014533436797902, -0.02345900600097599, 0.22615636552317966], [0.13040589848857795, -0.025007643687121207, -0.02604301986517654, -0.42263517462618033, -0.003339749233521285, -0.017752919772232433], [0.1516408587865943, 0.059138379841095164, 0.0021746050224914115, 0.19490233261487036, 0.009161562429984359, -0.04855567839723216], [-0.20185865594826538, -0.01973312101419425, -0.020614965133348984, -0.26369704982050607, 0.004585482760634989, 0.12973315151815593], [0.4021822496073505, -0.07306292927843053, -0.006510869080405942, -0.23417727904262964, -0.0070002535564280734, -0.04883044245898265], [-0.27912357362434564, -0.017253790676383836, -0.008767221280397324, 0.02641040298036846, -0.01406272905743549, -0.11497165977037584], [0.38558307797426683, -0.007572807435021776, 0.0010752115716392428, 0.042754186829650515, -0.024975554565344713, 0.15787921895814094], [0.4210643155200096, -0.03455794094766888, 0.004182444421894468, 0.10798999588940122, 0.010808896562984165, 0.03236204324313255], [-0.26256341744385364, -0.022741260316056826, -0.005203686974306494, 0.0014996477313627143, -8.455875351783869e-05, -0.11789487239177554], [-0.2016155368970241, -0.005929404525110136, -0.007924390150556352, -0.31656348309836485, 0.0004702542610495368, 0.15306779850524177], [-0.20771646536311753, -0.014907814586026859, 0.029200394168594213, -0.12279846757210981, -0.0025866234104935924, -0.07678102323684605], [-0.13751876191076368, -0.020169824877856297, -0.00672559780665627, -0.22747471184069237, 0.011601319577661093, -0.003290518379789314], [-0.006810974836343204, -0.019544112344329283, 0.028763866947330154, -0.3062266074226262, -0.0009668912460008267, -0.07072194776469799], [0.08971371122375998, 0.0056245582506107466, -0.01531757882015615, -0.3648011162376, 0.0016905409233520876, -0.07644689405145395], [-0.26652638486816305, 0.008733488401843308, -0.009065507816578796, -0.08693016915675314, 0.006921417094972041, 0.05531287063039386], [-0.26980748980210056, 0.0035561053550298496, 0.0021167009962163546, 0.16959264775124042, -0.007654286588754936, -0.1533569029784279], [0.08916430112921159, -0.02225050275991086, -0.006177676079154024, -0.2252827939152018, -0.018982190043116264, -0.11866431293500221], [0.3932445415847527, 0.011398871659583476, -0.002017793364392321, -0.1373778508398661, 0.03263829151661086, 0.1642471537290238], [-0.20967316608606765, 0.0008446090517031208, -0.0060098247589190044, -0.13357778399675055, 0.0003528443681333992, -0.06641393348005972], [-0.17159524770472903, 0.005147829491012246, 0.028231483160245312, 0.16504462008303755, 0.011936633166136884, 0.40073991989953267], [-0.18571524223322072, 0.01846564253293272, 0.03263455948835958, 0.28547614334939425, 0.0010750939910351232, 0.29383491398260736], [0.2267004182930081, 0.0036455410382682948, 0.002577193585736651, 0.20531061604722473, 0.005081653725006922, 0.13284457731075608], [0.18288422006600766, -0.004507208717690451, 0.012268796629914792, 0.1494645412576559, 0.016270659749969323, 0.035374229109380824], [0.4260293217998505, -0.005773327835313398, 0.004303547892691548, 0.14867809786410657, 0.019598374729996072, -0.031819953845272204], [-0.13348198610836864, -0.01911556595343063, -0.0030186491548632143, -0.18296093776791617, -0.00021001869463888227, -0.06493379470173422], [-0.25433354652094925, -0.01819056785083719, -0.013970986900528134, -0.19071987287771147, -0.00499162556788048, 0.13879096515276695], [-0.2518000755660556, -0.014867573601118513, -0.005341120135161838, 0.13469232031009576, 0.00441494980089877, 0.09955718079731622], [-0.2619455557418491, 0.0025553621311254194, -0.027353704595113427, -0.1972962498368595, 0.0017738723242304542, 0.16462384224507637], [0.3984014799474945, -0.017779566030105814, 0.011041449996159008, 0.05885538355598712, -0.06540637041742652, -0.039696244295977026], [-0.18530577336271375, -0.02617016792260467, -0.00599332617813228, -0.04526154532651116, -0.0005626985528513609, -0.13062320823390686], [-0.2558091573670599, -0.0071419292571237745, -0.010746528015286342, -0.1183157840439136, 0.032020692381815485, -0.014924133525271615], [-0.21093947492721374, -0.007338858228165867, 0.014807990648232027, -0.09308372271925529, -0.0008486490656572607, 0.03358404365713947], [-0.013753677077022559, 0.0019325446386553146, -0.015870803426830334, -0.41919880352844746, 0.0049014082276542855, 0.10209540959735959], [-0.015054960447532283, 0.06525411121201423, 0.0105215999501257, 0.22838526528218694, 0.022633009995284982, 0.056920456409575965], [-0.16744227643736606, 0.00850134203758014, 0.03565343164767385, 0.3882144048659703, 0.027889413594679686, -0.09056702999425174], [0.4018787160443184, -0.01649467895859318, -0.02506027731582441, -0.07746419771811533, 0.02827997617556805, 0.16356808082026192], [0.33818880158302317, 0.005211896856109553, 0.01087199622148893, 0.15708978585990124, 0.017181388046349267, 0.025032798099795014], [-0.327620357613726, -0.014452286128505376, -0.005168658537134803, 0.10562173663128895, 0.0009740991436161044, -0.1644073849011609], [-0.13120112831272043, 0.01021233373001278, 0.012538201334026023, 0.314518296271604, 0.013948689850940076, 0.2999944474622683], [-0.22179002536920392, 0.013961224786265696, -0.031412138134050326, -0.13899381174849007, -0.0035158155498539643, 0.14912832395183973], [-0.18647007820968495, 0.010420988780424141, -0.005542439821573006, -0.172795835733384, -0.002402694879080797, -0.057883273470034566], [0.445587722613515, -0.002521392166104498, 0.01989413511634916, 0.1264630274414072, 0.027908884025396468, -0.04006631642450408], [0.06564336578392774, 0.008231847179813947, -0.012245885230014332, 0.030478014331338205, -0.004674026591077528, -0.1413041036513279], [-0.24634200449632127, -0.01199907409413205, -0.005578038916835675, -0.09029131150466292, -0.0017431896196055852, -0.023076857558919552], [-0.24523763413087057, -0.012103117378543884, -0.006701457038536258, -0.02055258604147144, 0.026398948112815845, -0.11921558209482265], [0.5425176954379259, -0.0028210832377396295, 0.002074648321095557, 0.06108439750344976, 0.009967160827021657, -0.03472342491236365], [0.3594268730270078, -0.004291186539986709, 0.007475086894396846, 0.2005438090840383, 0.0113734417552699, -0.03482635755406016], [-0.05858193723499945, -0.012905995543394323, 0.030748990607762254, 0.1791265370246805, -0.02071880609617496, 0.30020351282942787], [-0.25206216301026024, -0.014582813580011587, -0.0008117143269738564, -0.025345790495617696, -0.02335152381391324, -0.09226932810655568], [-0.28219674011294343, -0.014629997271656925, 0.03810567919798959, 0.05018835636236774, 0.004832742083394262, -0.1612610720051829], [-0.18626221628111794, 0.01603798867763606, -0.008735209834391671, -0.13467085898948467, 0.007972516102880144, -0.06674942397659668], [0.32314579322048453, -0.0479136403621829, -0.012737373448477356, 0.13428433176989216, 0.0035407441750280105, 0.13250284305795063], [-0.13619206077371077, 0.012348274239276407, -0.007325864196915179, -0.18135225233554256, -0.02631544507860886, -0.06625265185449888], [-0.20657726164609508, -0.0012625856350134514, -0.00798505659363619, -0.01100752741175253, -0.012782347849036207, -0.13894120725902454], [-0.1795730539121596, 0.011245268915411995, -0.00906674448777003, -0.15511156480125254, -0.016455953621974474, -0.06321128542558913], [-0.13657974567822595, -0.02366641514944141, -0.0077937158526024605, -0.19436078348697525, -0.0464593988375729, 0.0947157276978259], [-0.2898640733615896, -0.01391200220966225, -0.0021832984986005478, 0.07854544792722533, 0.015650906493914145, -0.16504884182314797], [-0.22379253931377624, -0.012738352307628806, -0.004950741998600015, -0.12402686579665013, 0.012812808827025273, -0.06197764274370292], [-0.2611505607189949, 0.0029373450643484142, 0.047391220891610436, -0.16424670851113013, -0.01997090101006508, 0.12378293761756319], [-0.16595345653716895, -0.010097755908146272, -0.004880643837357035, -0.17229009993291924, -0.0006230580694828843, -0.06082831904825958], [0.4673793902014215, 0.01906086863045274, 0.06851883019792644, -0.14642013848439095, -0.028981538878184977, 0.1159041756343587], [0.34101657109637, -0.00516381709040765, -0.02833496633102624, -0.2888832177675574, -0.014682413802061239, -0.1274935449942084], [0.43505848907119876, -0.00931660705888799, 0.004038108764923287, 0.1291129705168313, 0.040188223815121944, -0.025780159468163484], [0.5295344608062293, -0.0025038115633558237, -0.002099261246908607, 0.07811135387772972, 0.014695817587087188, -0.05199522612745138], [-0.12512516738759927, 0.029587852003648867, 0.002057770037665792, 0.08488997561469873, 0.014104927416342913, 0.33856821894286193], [0.5035708314585996, -0.010142383130223462, 0.00439851425075932, -0.021233321790245244, 0.005660309242377688, -0.005260616697937236], [-0.2687050214114422, -0.008259925094725801, -0.005713712497657573, -0.061211935744575306, 0.006117414191065027, -0.0646977718236164], [0.409969171440642, 0.0077880561214652855, -0.006224517897591498, 0.18368342226827736, 0.016290373134083765, -0.08631273111587003], [0.16912606118193033, 0.025544361932022405, 0.012878944553096143, 0.12264997073982781, 0.018057756392691228, -0.04364471384718638], [-0.28435520967705047, -0.002107416662456207, -0.005866814026635034, -0.056854452621317914, 0.0025283886172865274, -0.062462273407604484], [0.4374671653148095, 0.010776417686553283, 0.030615558856995052, 0.13433811751349387, 0.001883081888464464, -0.060575103165079996], [0.46412570149698434, -0.011233380585395967, 0.004128397375875637, 0.16480638031874198, 0.003756468842672794, -0.042756900782214155], [-0.1779708151766928, -0.0053260194990928066, -0.004883075087569418, -0.16656285474110635, 0.0052565573753550815, -0.060187126204226477], [-0.15970762075301134, 0.01682784120357339, 0.010464283428034531, 0.24404816011734512, 0.0037382814129258714, 0.32681992227417045], [-0.21190150511662684, 0.04514108844298675, 0.012183231567561442, 0.4108730556564302, 0.0029105851815338078, 0.01558968590425432], [-0.29941486906579445, -0.005235113339466862, 0.002903789267775116, 0.03530719490368699, -0.006251089236449763, -0.1277751990079791], [-0.2841711785585482, -0.003911036222874955, 0.027740325997022502, 0.013533225358931481, -0.0018134775859064726, -0.14313452565529], [0.31739870576021284, 0.01740665986549573, -0.0008001381163931498, 0.058528325258285496, 0.011399985429604525, 0.15295662053295322], [-0.2560464741016745, -0.020530330984804068, -0.005300128344004258, 0.05891365554679113, -0.0015891446901607187, -0.14916276649410698], [-0.3019315843991224, -0.014464145218748078, -0.006836376313063523, 0.10275787840466342, -0.0004536343895334397, -0.1738486405664118], [0.2868741900116885, 0.027915218383044015, -0.01834112689463631, -0.3117155605955819, -0.0122926067511733, -0.05338786751109702], [-0.1939052851248374, 0.019011698265097365, 0.0178108266344299, 0.10173530774759941, 0.02203279175733978, 0.34898259722830577], [-0.1890111867505909, 0.0025087055959555384, 0.016126164726075472, -0.2284380219445128, 0.001135913966433966, -0.008661575593361778], [0.5160886507699566, 0.01835361537482444, 0.008095572848493637, 0.04090584933266583, 0.012932105045944164, -0.0408646028956984], [-0.23804781179261905, 0.013600289513089373, -0.00706844021175022, -0.11662598493563218, -0.00045066889922214714, -0.054414050340531935], [-0.30167558767769936, -0.0010254552516848082, 0.04560501014567121, 0.043048312409691376, 0.0016578107064175553, -0.1457080268403316], [-0.179621092682442, 0.009759396370320247, -0.0016511466093494394, -0.28246787595783074, 0.004229011879420224, 0.0471617069998802], [-0.09852420656551865, 0.010342994378642455, -0.005604212868823138, -0.20367279694641469, 0.010847750006729032, -0.05530492483001173], [0.4147972586799482, 0.015390607121343493, 0.0031957944283071012, 0.16037655843302207, 0.01961030946592192, -0.043344586099558324], [-0.16713102884656358, -0.018659469434072196, 0.051598362885914906, -0.15564096129234262, 0.002374225967690638, -0.07629779594729393], [-0.1647517830411179, 0.012953946887630323, -0.006592627046598198, -0.2561137777422591, -0.000452423193880076, 0.010521426040986248], [-0.050529263649516785, 0.04672755774306037, 0.007871668209582613, 0.10692577326844348, 0.013604290133261168, 0.10243990040354367], [-0.2579209096168196, 0.008087511807072185, -0.004107171584142426, -0.09787380192918109, -0.002304451515148339, -0.052498954939558425], [-0.3069218482433643, 0.005710723683917439, -0.0010003441099336048, -0.049762708618387066, -0.004927056271897129, -0.0024056570759001337], [0.3153960822584717, -0.0802883939134046, -0.028260290793235006, -0.21635655278474183, -0.010735947379674553, -0.07840442119694103], [0.27165006749521015, 0.009613891595347318, -0.010536306950617112, 0.19417803170722533, 0.0021094407761296516, 0.10253773251956005], [-0.20350301191225437, 0.003575063028868783, -0.005024006311495414, -0.18500581112984438, 0.0004979375963686334, -0.021713504604976307], [0.06515548770159153, 0.0050709520189203584, 0.010076638427233038, 0.4467488131985368, 0.010232936658558402, -0.038828166811630524], [-0.2789078488096458, -0.010187386139189108, -0.006776893372306124, -0.09428794177088035, 0.01355372510745509, -0.015364607396385142], [-0.15510951390744537, 0.03145422361155391, 0.004240118868902546, 0.17806508495942203, 0.0033572403179383636, 0.38577288583216474], [-0.16999293245846414, 0.005136168822145691, -0.007624078996712296, -0.271438297606221, 0.0015862697298298515, 0.12849287050942027], [0.42063658358916534, 0.014896934819496251, 0.0006701721291186866, -0.1111817661781833, 0.014207631535490158, 0.15570019518716108], [0.5384407190157813, -0.021779554704545376, 0.0029896452870686488, -0.1140000903925117, 0.013472269900970528, 0.10664848275470352], [-0.20254278233561598, -0.024359952008640374, -0.020724259586861443, 0.12062457920001042, -0.0198174198002526, 0.08474500957370909], [0.5455521873994646, 0.009744504934942598, 0.006915656498199409, -0.15517897724343338, 0.017845001087365284, -0.01737908696225562], [0.46857821552493045, 0.02330594368790352, 0.0053329458829440325, 0.11887552317230288, 0.01353987733144366, -0.06084155321857234], [-0.14875409985262789, 0.008823547319213036, -0.004447121536170586, -0.27357647249492056, 0.002528859990924593, 0.014001953240247065], [-0.0997861013982219, 0.026295127921448523, 0.02667798701931, 0.1858964480651086, 0.008196209062680957, 0.3458942182185609], [0.45244237104976825, 0.014890426718645376, -0.0034867321615512322, 0.13997876455252484, 0.012013583417109423, -0.04555462386210058], [-0.1923968951882126, -0.014501876451216426, -0.0021758959366054934, -0.13444949334623452, -0.0011498155180716275, -0.06410649975013433], [-0.17343219074016608, 0.02896532379047505, 0.002256081108735376, 0.2733471209123705, 0.01864345706102742, 0.39088020786755323], [-0.2622547797202696, -0.005152153871738093, -0.00612536358463653, 0.020273987630431273, -0.01880565722731437, -0.13760936655980485], [-0.33114819845833354, -0.004479270007845251, -0.007862428865308932, 0.13609719153804617, -0.0009986077928449363, -0.1696110240760515], [-0.2378486872227508, 0.004450633015081036, -0.008079185763272418, -0.12347168787239357, 0.02037549732297069, -0.06526096723873827], [-0.30993475348923494, -0.025090654920746545, -0.01989614069540433, 0.27607431951230926, -0.009556814269419227, -0.09684071804226807], [-0.17098886191052698, 0.003084167412219392, -0.009255020147615726, -0.34070567239577615, 0.0012788378151545063, 0.14040591069840433], [0.35543334241921554, -0.03605998595696692, -0.007925005373216925, -0.06700736971601637, -0.01656647053340193, -0.16009456134466502], [-0.04409301658403896, -0.018531608701758108, 0.000906845041651675, -0.2910658309558187, -0.0009028225153934079, -0.0522368996179765], [-0.18470991190404054, -0.009824888221554934, 0.007150825329218471, 0.312685652491389, 0.008809901375751635, 0.34257818283399405], [0.5788365334183144, -0.0038250117674719225, 0.0025123450082207193, 0.012600924260731028, 0.012308819961718665, -0.026084216942121905], [0.36895565181565154, 0.017540677075814373, 0.026095412269680474, 0.06020573094436824, 0.014579981144840229, -0.02893596601778145], [-0.3031279279382196, -0.01863084913045653, -0.0009503550612612446, 0.06303223027554847, 0.014477188392205485, -0.15747361987114944], [0.4445381481054441, -0.0218435534048361, -0.017985074736255206, 0.10053793720387971, 0.0324210061306345, -0.05417512996553568], [-0.23702353929609854, -0.00040979074344084306, -0.005395078041143158, -0.10311570302189652, 0.0002165926296520212, -0.06894581486040575], [-0.14917763355375768, 0.014391189081704867, -0.002905472168918186, -0.20630667870903868, 0.0026091905467304816, -0.06703392853005441], [0.4215653265015883, 0.019850730057313772, 0.0006578399811101567, 0.120627556886103, 0.019626985000173982, -0.005973993981845583], [0.5117548466929109, -0.030059315823916775, -0.014448791990149283, 0.09485374106385557, 0.012521158231305464, -0.05134259055496047], [-0.20978131925664048, -0.019896713456058476, -0.020902958248279736, 0.12868893867526415, -0.02703574639787783, -0.04938839179259809], [0.37829812443699634, 0.004765564758892578, 0.0032416700082628087, 0.21881034476536562, -0.013286960157305999, -0.04056953746300757], [-0.07696079589357109, 0.014712056569284218, -0.004597697268234709, -0.2784132629822756, 0.00581243499599044, -0.06314273542119206], [-0.270669346762889, -0.00525064101733732, 0.01669619635190651, -0.0871784955293302, 0.0008309150036673364, 0.040677304493665266], [-0.1696862783878257, -0.003993509952176571, 0.0038375084720173084, 0.321307353943777, 0.03210517831324303, 0.32443749197186283], [0.3948230154540639, 0.022195653480059314, -0.00703618653516436, 0.07917375652622774, 0.025010584149615062, -0.04291094190283454], [0.1755500158556834, -0.019701208866021502, -0.020124140985799926, -0.473577249733635, 0.007414507927869812, 0.09503855199237535], [0.5633494984225094, 0.029047479266502408, -0.0025523018075896933, -0.13456086207780046, 0.018559744000545227, -0.017641891137507047], [-0.30919329059788153, 0.011502907603199759, -0.006528600297779656, 0.05080438058719064, 0.00173279828314563, -0.13710961867584895], [-0.28153174805035985, 0.010186258415332345, -0.006252701356498011, 0.03969104844770227, -0.03329694375783625, -0.13301748450848241], [-0.17343186817577827, 0.007518642016262246, 0.031071229660117983, 0.22589520597378995, 0.01644627157932799, 0.3769878998986588], [-0.2371362717437092, -0.0060749625785190735, -0.006017181556426106, -0.20323790604396272, -0.002359600442659218, 0.05064267293735849], [0.4504869937832623, 0.023399998237524186, -0.009197489780236335, 0.11628792169874358, 0.020780700628602558, -0.029997978369067476], [-0.22462066506399414, -0.0016548555946514158, -0.007744260952690462, -0.16728672346986753, -0.025255485745286174, 0.018834735924528494], [-0.2751755291528238, -0.023315998737400524, -0.012289253605403888, 0.020481599705245417, 0.0005021636838843555, -0.11029298189350038], [0.5238653299900912, 0.010626434941289675, -0.004101638332209722, 0.047712879202273364, 0.013314758477653593, -0.03366735988452327], [0.5037889483469525, -0.006631620872018044, -0.00920964215203198, -0.13160485954637888, 0.009150140952008523, -0.02768415720472613], [-0.3289493737462349, -0.0003197162615425654, -0.009771388598866254, 0.1402391148228651, -0.006773345397786252, -0.15976447493896356], [0.3819968936027563, -0.03226756727280782, 0.01722898795567271, -0.3779124261090157, -0.011225595148452018, -0.09095791207577748], [-0.2897776259665843, 0.00995100142501527, -0.007386703809663431, -0.02109050271074502, -0.0029554277853703244, -0.10192700552046874], [0.5615152578713197, -0.0030785871354092028, 0.01973258233693318, -0.0910050447932445, 0.015609704059021455, -0.002721055195767981], [0.4070978934556264, 0.01102122407170566, 0.004140525831381906, 0.19458427962844277, 0.01495688596075932, -0.05058128513839536], [-0.293014426326078, -0.007941814455661203, -0.003147556181108604, 0.08736649889134511, -0.01727930339476692, -0.06340559686226996], [-0.19846784437334994, 0.005926590022715674, -0.009718844875464356, -0.2999263042575824, 0.0026381260202755974, 0.12439478539991174], [0.14908761359950778, -0.0013641904629170874, -0.053716837229366085, -0.06865460084551274, -0.00435122060993369, -0.043662373398386654], [-0.1655548376013146, 0.007804569784693875, 0.03803641674205542, 0.27684212189257645, 0.01907825445843419, 0.3471558556759308], [0.5754001570496534, 0.032416483249469975, 0.0036411522063084798, -0.1003590853569328, 0.022677359336992123, -0.0621160664854965], [-0.06479681804834615, -0.006421499941586932, -0.004606758026436419, -0.29008323721382595, 0.00268760115118458, -0.03939802355317356], [-0.31071695589403064, -0.002543130872473635, 0.00503755563894892, 0.03901339969145227, 0.0034635753494755067, -0.12763989845882653], [-0.15353067687885616, 0.0226632374324363, 0.01869659941952125, 0.20091979311565047, 0.0074814052272649005, 0.4044296416839809], [0.3964642407058043, -0.004925982392872636, 0.008628095187702784, 0.21154992353179794, 0.012656416371846394, -0.04279602673761453], [0.38438370506699227, 0.009458603714475133, 0.009858341841383305, 0.20572274847700447, -0.017169683626759868, -0.059726729284681965], [-0.34252672427030284, -0.01596406408764164, -0.006636000705586142, 0.19361838203329457, -0.022632122109385314, -0.04532447086038106], [-0.17611208185226787, 0.0031541073914013516, -0.005968985906139393, -0.3558757814015131, 0.0038206816649664394, 0.13550480520158925], [-0.20477668567979537, -0.010404036644337429, -0.011076158661605922, -0.1856223285339882, -0.005655539407674681, 0.010884465239456602], [0.41685882357591847, 0.00417917493994949, -0.00861011043775071, 0.15707500656835027, 0.01737861391883494, -0.03998034113058778], [0.14926782418361365, 0.003566034917639664, -0.016623107771055263, -0.34313109737646946, -0.010150038249761557, 0.02165787347352057], [-0.10223350359076848, -0.005966150055795205, -0.003379400289251419, -0.22963096996390434, -0.016133895270415217, -0.03732941416319825], [-0.014308310572584972, -0.007321116662664769, 0.014789340303946233, 0.3948326400117425, 0.020706115186328575, -0.08147914445724447], [-0.187129798541804, -0.008010893870358481, -0.005639632197055899, -0.18549818732374954, 0.0002086868207655188, -0.01854398441160775], [-0.13767753898638618, 0.017594927160909327, -0.00903181976499783, -0.17251562484594793, 0.0036402421795621847, -0.08700494764790113], [-0.2854361326070358, 0.012251010659852044, 0.000695489786140728, -0.015718338850095822, -0.0008180371677574857, -0.1041711346782465], [-0.35125960836192416, 0.006157005393091367, -0.009955563093848525, 0.13699932109020868, 0.0007439020288884965, -0.16828078541431088], [-0.17391970213850375, 0.003497443660631504, 0.0005846581149016792, -0.17673589880748927, -0.0036842001639251915, -0.05741563399894778], [-0.09814358280211029, -0.006053220245833646, 0.06937274047522646, 0.19707401180076276, 0.009723118196991369, 0.3034964563844849], [-0.22044094171879347, -0.005702697485451138, -0.0038984101931831745, -0.12003409977266918, 0.006484815337357417, -0.06524866616725997], [0.07147755794416753, 0.010823663539954939, 0.007894744493691272, 0.2808973217495797, 0.027773855259446603, 0.07150497822528137], [0.40955210526468405, -0.02763578888177221, 0.005586814707089524, -0.15792009418968272, 0.01355995800504006, 0.07495252096765621], [-0.33172974800703187, -0.017339343945378814, -0.00981834973740617, 0.1429080423074042, 0.009036036445000829, -0.15827163706258915], [-0.2268673877179696, 0.001944053749050447, -0.004819022515517745, -0.1117832984373498, -0.0011278896079740342, -0.06743645547023933], [-0.18865559438415527, 0.008071313233241062, 0.03378411502893654, -0.1746875173377784, -0.01650556766937142, -0.05666619331531733], [-0.1618576922576412, 0.01618343010522366, -0.0033776297480344244, -0.20178893291916106, 0.011865108324979888, -0.05811428350536753], [0.5038126511176738, 0.008769369819609284, 0.007738504474002281, 0.037849134215014034, 0.026527921264545652, -0.025931520284786916], [-0.32221047366291244, -0.012298714533428111, -0.007191247628025463, 0.12405559227669238, 0.0010401938333796626, -0.16465201695237294], [-0.09495293685444131, 0.0005108829625506134, -0.004170420122063388, -0.23172924155019714, -0.0019284087833606398, -0.06854046388778216], [-0.2052092496279875, 0.008021171846746424, -0.004170302370585657, -0.14410960515419705, 0.0001576468266449608, -0.06686299485395382], [0.2691211639066176, 0.02251718505963023, -0.005709060780695781, 0.2233888195908352, 0.010064224707281138, -0.015650903912240326], [-0.20268804810560695, -0.0016035493262703557, -0.00024587680882514403, -0.14788044506165682, 0.006713074712324018, -0.06877241031192514], [0.3955056157977418, -0.0048311558756752755, 0.007272287666344242, 0.07456330786432047, 0.01954209655248522, 0.04314951466145068], [0.2348820668629907, -0.07733816273098469, -0.03668675251120252, -0.016723993196063845, 0.004094868781079148, -0.10232993196772344], [-0.2835902115094065, -0.002776802994168142, -0.009109766852971006, 0.016266808553222125, 5.376791144460439e-05, -0.12013457483521676], [0.5581793200295055, 0.020830036757608146, 0.004958638800790489, -0.06304732619151517, 0.013092419117158657, 0.008313578153113383], [0.5954233010517482, 0.02402741960981821, -0.008167415335374134, -0.08863237558925251, 0.021897126951750192, -0.019118215418854594], [0.2996922955673183, -0.04549473957145921, 0.005867674070183638, 0.13599199376500928, 0.010903530839650176, 0.13905638818643906], [-0.11935985377847154, 0.012339756140122567, -0.010603741183868259, -0.26110655110905573, -0.0020700098510456674, -0.006527695455777065], [-0.18225806246358597, -0.011509465423803454, -0.007101427763989312, -0.13137086833481199, -0.01180370792216087, -0.06384408713926683], [0.009396248045336798, 0.015238354908037327, 0.007741879566493096, 0.34978543441703547, 0.009505196941041206, 0.17812383850300392], [-0.06468159620384778, 0.042366201702835074, -0.011411807738159346, 0.26558678445778644, 0.024968417062168086, 0.25484517871769], [0.05571843840738959, 0.013684871992521486, 0.0034036639630622883, 0.32332623938713484, -0.003691028620443262, 0.16730710058461604], [-0.1679785416265047, -0.05072406963806584, -0.0010893117430957355, 0.27035673827197476, 0.026691718443521018, 0.32741140280010483], [0.23763691199768464, -0.0415644482156658, -0.021517633523219087, -0.30954018977555503, -0.04745172966697544, -0.1076592133372801], [-0.07436368664061446, 0.025754001927175176, 0.01910342220863761, 0.15467143381035306, 0.026784071096258754, 0.316510360772791], [-0.19191656045273298, -0.014970872403396633, -0.007279605514132221, -0.12832982010694682, -0.01571918839964886, -0.04883823883742754], [-0.16208998587522797, 0.009171166255113572, 0.004570527612111684, -0.20795697765335477, -0.0014097520053958515, -0.04987497833324626], [-0.17716810529474836, 0.002065068794140094, -0.00643079455155615, -0.13959065538519794, 0.0031200220839449344, -0.08363315469420116], [-0.12715226361575982, -0.015900214245153695, -0.006821663120311381, 0.18047908965514667, -0.013776154379138303, -0.17839035514134483], [-0.3080323639363445, 0.013883377874632995, -0.007908143728716819, 0.03541107241272275, 0.0024553265530019314, -0.1383467240193036], [-0.1911587156474704, -0.00961774914849913, -0.007218063607239863, -0.12459886353440691, -0.011118211933836337, -0.05000934850949922], [0.30890548331219114, -0.016280734802159687, -0.031073729462384504, -0.1523962834727217, -0.015472756476627766, -0.18728970684157112], [-0.2571621426132127, -0.008362881380031339, -0.01596287731263058, -0.08767708147223774, -0.006611339377932617, 0.13443778868468548], [0.20721596987054552, -0.006354073101798828, -0.02049942983918057, -0.4077474871873676, -0.040756480226893294, -0.08360610821096057], [0.40563071727801864, 0.0031395779707211672, 0.0017828670667526904, 0.1281820871512993, 0.012396115129695151, -0.010763031263156612], [0.41108481688688386, 0.03239470157273847, 0.013999002795994614, 0.059326438388642114, 0.03763302101097241, -0.05367256590868918], [-0.21832282295050023, 0.01291851076958454, -0.005322064233597732, -0.13784751449903984, 0.0022339300048457656, -0.056250039091291594], [-0.20828248483247971, 0.008256756824658186, -0.006030362328273389, -0.1537962180865886, 0.00784751674479806, -0.06266854165544794], [0.4244438381523998, -0.015456349051863499, -0.008585713048952429, 0.1909791577783553, 0.0056958704130921954, -0.041075895152123613], [-0.2858303032573562, 0.008219692656167002, -0.0037790058538904546, -0.07567571294648093, 0.010174422122760209, -0.04956020383230986], [0.2863142693790204, -0.012450032396178631, 0.0011973909156523907, 0.16912161867434672, 0.005222670119615526, 0.13008741664087478], [-0.15894795136127285, 0.020882072383990428, 0.017479750726687483, 0.2995822332473609, -0.01588591590964833, 0.32401754728278725], [-0.33117426997370514, -0.009863352023320848, -0.005659499225654649, 0.10391733558234714, 0.010584457989155735, -0.17260300568215586], [0.15510100079969416, -0.004780779126494317, -0.01557805190422572, -0.17391478794987725, -0.00990451925748699, -0.0049781403393869056], [-0.17335924883096634, -0.012189091524127504, -0.006591968782642465, -0.17220828871982197, 0.007460822006492142, -0.05778555748226601], [-0.1310538753254264, 0.02029859451200435, -0.012544888310298524, 0.05300397778631842, -0.02054389219844055, 0.014521908932668887], [-0.34007825351239274, -0.022251517356633412, 0.003922512074505122, 0.1677663238422842, -0.010471474310249041, -0.14490616216608496], [-0.165425957587152, 0.0014176133651796223, -0.01255549268571335, -0.05914297793212122, -0.0016106566725895808, 0.028897260261228967], [-0.11764895263056466, 0.006671590139520044, -0.005356818684219496, -0.30497131973198355, 0.0036503924920341336, 0.026231775081878985], [0.09340243973375056, 0.00022972619166581666, -0.00846518723326335, -0.3012471590325261, -0.01940315513299564, -0.09773486145757715], [-0.26942645299236984, -0.013914069504454021, -0.005228987051575858, -0.028776939441439015, -0.0015045775902703686, -0.09230378823470294], [-0.18672420092250372, -0.0020022210904740044, -0.010009080240752697, -0.2516694377127397, -0.003936709057912166, 0.040574323442985824], [-0.19545485067840968, 0.0002360687549505282, 0.04821262773917508, -0.23397356239409223, -0.005117170495236092, 0.0029235537402778336], [0.55663185997605, 0.02053955808501602, 0.0035785400299946507, 0.022680711615868588, 0.013209143887614815, -0.052146480261214635], [-0.23115963641524548, -0.011082886016464759, -0.007710812846573658, -0.10785732332426143, -0.004742867910832497, -0.05211980681995539], [-0.33736408582521255, -0.008422772963886785, -0.012761016792508864, 0.15349820051377505, 0.005026484034606141, -0.16835847563344136], [-0.132754287656246, -0.012415196338812563, 0.03118567075936018, 0.1625254207906157, 0.011725185015299493, 0.37797627691644203], [-0.18033819748773822, 0.03790096420793186, 7.023890205072768e-05, 0.23547779173270802, 0.030310320146306795, 0.3128341205939755], [-0.30498700277124213, 0.011474479848900345, -0.0008313843295076668, -0.019039927838666332, 0.0016114106003031108, -0.08173238281201922], [-0.28344990479661175, -0.019147461084435376, -0.005296018740301809, 0.021767832091321937, -0.0038549014598458155, -0.12469287934345996], [-0.17202308664670904, -0.006840514678525684, -0.020080978885542063, -0.3170195081227114, 0.021706170257997665, 0.12271518698305149], [-0.238174913477853, -0.010575126086555267, -0.005385309462011435, -0.1552771495168319, 0.0009284874085049845, 0.0038106778014122687], [0.28600688862869633, 0.013534518772135767, -0.0034932754806114154, 0.14622119429474056, 0.006358133752941356, 0.1333063495559051], [-0.03226422002272937, 0.016709304975798103, -0.009406783203537737, 0.02412277708358957, -0.012915398132011122, -0.14189795889381063], [-0.2661267936349969, -0.011671813943060383, -0.0030871445110895257, -0.0684812015671326, 0.0074457096186748475, -0.066085422629061], [-0.3201078455125057, -0.019611829143890157, -0.013496320315179847, 0.07418904243743518, -0.00973381051756951, -0.07624446930629311], [0.29333536289691, 0.015515179553099424, 0.00276723493484858, 0.1271741533430294, 0.012550145951019605, 0.1306512566544255], [0.2830604455609374, 0.006585182934195638, 0.0020010601069068877, 0.1785511036805985, -0.01081581038099279, 0.12594468476501994], [-0.031190331839660086, 0.023452980513596563, 0.008612632733707042, 0.09871008869243067, 0.03637660145175153, 0.08778136178150822], [0.5794529423073245, 0.02823228223489718, 0.0007972902965925126, -0.1097633909855856, 0.01643529254316007, 0.002096853444875306], [-0.18802440219214836, 0.018475084580509202, -0.007375048014112322, -0.0824792739094839, -0.007549377489567993, -0.10509211831075424], [-0.2795436986760208, 0.021642543782363355, 0.01028084473190858, 0.08733542433553218, -0.027854473699909604, -0.14109808429988818], [0.4178101382956178, 0.009614369060358983, 0.001906149226676579, 0.15036089986534087, 0.01066057144604638, -0.006692127894040385], [-0.12963312808892394, 0.0011548420441428826, 0.026872258764402784, -0.32837649340794195, -0.028139085953724555, 0.11444827330871025], [-0.3086510185104223, 0.015429295124310946, -0.006344244266665222, 0.0373152871609884, -0.0013628804561141013, -0.13709362730671076], [0.15429681946314866, -0.017438515506170343, -0.018871912614440257, -0.2681324062339816, -0.07088745940574054, -0.07362658969679424], [0.4545213966044966, 0.016658604817695006, -0.0068493837969331095, -0.05713851246386182, 0.028295700569112407, 0.09064144030123507], [0.01563100943913653, -0.003311853352936486, 0.019392146449835154, 0.24905623675966765, 0.013986586524186263, -0.07593719941296237], [0.39179222156093463, 0.009417434368311522, 0.004695336740668559, 0.1559042728641751, 0.0045270771777296435, 0.014946206307787063], [-0.24436411038885464, 0.007171344378650577, -0.008619298189779126, -0.1907966838559521, 0.005276213851662472, 0.05523424755627923], [-0.1904814603261663, -0.02961180069263318, -0.003777452224092585, -0.06954113170851409, -0.008530337595251476, -0.0324573412628662], [0.294100431017286, 0.0015281198566601101, -0.02369493818737565, -0.16882575794256596, -0.10422159702670125, -0.10011352322956721], [-0.12067891808267789, 0.0023549265320779098, 0.00702970166140289, 0.44872945974663914, 0.009087072635635072, -0.0852213694772042], [-0.13538172615264413, -0.040814782531055326, -0.010613530203134996, -0.002672547247096861, -0.03672676664960168, -0.12506715515297442], [-0.10723513643403018, -0.023432683492014626, -0.003890984698542326, -0.21340788985372708, -0.0010723064698233051, -0.05480099905186211], [0.351644901623783, 0.002981727248954266, 0.002334790755633043, 0.23999591177731056, 0.007968520854899718, -0.038411419183003156], [0.3326918516454565, -0.008346599220174911, 0.005850910346776834, 0.262621855240343, -0.057885644419409986, -0.03783622640684612], [0.3080111051691059, -0.00778868097172371, -0.0006632182789232713, 0.171194771041309, 0.0024370104894918415, 0.10380234588407036], [-0.30336391179422284, -0.01578814535804293, -0.010865305913336941, 0.18671888123777688, -0.021252200269162566, -0.1380313813950752], [-0.31311443993074617, -0.007254245825025488, -0.01822673859685752, 0.13915332962723967, -0.01685846395322726, -0.00906487422181778], [-0.24265307085379362, 0.0026438384093906307, -0.0098974826643509, -0.19785670301598188, -0.024820598164362658, 0.12779260583360996], [-0.052474402613901835, 0.02288826164082361, 0.07342601664207199, 0.013548025942879696, 0.022184281215389096, 0.2657068647917843], [0.49518674267992707, 0.006248240295342396, -0.003247093023824291, 0.003026893498547326, 0.015717175875546288, 0.026491929563349607], [0.4551449003995746, -0.018086887736645497, -0.012175951823675323, -0.014637805652006846, 0.021977200136579344, 0.01718746242509054], [0.42088581346257803, 0.007487712979601118, -0.01642382113422139, 0.1465497188195048, 0.010291313654280277, -0.018880737781744497], [0.3154744907299788, -0.0409185951437327, -0.024096922679615763, 0.042051110649428555, -0.06084145713780423, -0.1535823606550323], [0.3837896204422623, 0.012585667582359286, -0.00033214938764431607, 0.1558949981025198, 0.004728416059884646, -0.0028815527993841], [-0.20806589953795268, 0.06426413173825866, 0.025799258967968332, 0.24980824311777092, 0.022989023673172848, 0.2236405180932298], [-0.2237692098141735, 0.006682122154205288, -0.00418238082691826, -0.23000329608414, -0.0069016581771522275, 0.04654995430964052], [-0.11695046974884697, 0.017920615389623, -0.007777917501151112, -0.21817973808245011, -0.012145779203355267, -0.060456710853819186], [0.29401711150943904, -0.03911734917226974, -0.017704236048926018, -0.3034755227048501, 0.017910745347116536, -0.09464777655974421], [0.5125692228647811, -0.024839558013998102, 0.008471268193020723, -0.009339561881734307, 0.007886158781832877, -0.04944467280104977], [-0.14528873499069422, -0.008263989596964933, -0.00940246577832424, -0.010922487568785989, 0.020763832470708513, -0.14035710691689143], [-0.24733051884105986, 0.009352928256727474, -0.0063368032882624836, -0.11016570366559132, 0.009629219534119651, -0.06303674104355178], [0.3006132464337991, -0.00757662119820378, -0.01763824229583098, 0.14238514780367095, 0.01020211054457154, 0.1422815015691341], [-0.2543742043604674, 0.007841558696617734, -0.011539977377314797, -0.088528299418144, 0.003257277254718318, -0.06882968812874256], [0.29752880319463726, 0.019933876517601968, -0.002479969600860379, 0.12809382246851383, 0.01150018738073455, 0.11344439115048305], [0.38389563110120195, 0.01089808209766285, -0.004400484305643735, 0.12254514896341684, 0.022379868826342228, 0.021663181888448527], [0.011674673931401735, 0.009919339028871514, 0.01786951709147474, 0.40581330331024507, 0.0289957710963574, -0.057118065873713104], [0.45131082040079434, 0.055599458473581075, -0.0006517153408657374, -0.1410226933903221, 0.042563760136463714, 0.02576909987907696], [0.11734407713493669, 0.0174662619699223, 0.0053000359308095065, 0.3668771321318739, 0.006337867848803524, -0.09376061311158422], [0.45279017864347876, -0.0020785817751451693, 0.01927940189557888, -0.09981117484900119, 0.011167440830938097, 0.15179487811128853], [0.15890580901628193, -0.03771798832504179, -0.014245212550113275, -0.3685137042391951, -0.007347864207468594, -0.06525734921827608], [-0.268062093389949, -0.011194211872462856, -0.005734573597216751, -0.030914762112814936, -0.005686230601863725, -0.08956294324050636], [-0.07743177398749415, -0.04199978593966052, 0.009321214122341315, 0.4138303753890752, 0.001397407578134315, 0.15888780093283844], [-0.3211211678547367, -0.020441882617840138, -0.007011885777007635, 0.1019137151611632, -0.02118649468887271, -0.1389909747891689], [0.24614385347652076, 0.008145989636727533, 0.0016921341557318247, 0.17650946617367488, 0.019406084245004437, 0.12572080564567403], [-0.3243631493303429, -0.020768645174359928, -0.006462859182648149, 0.09793832841874542, -0.011408800025887343, -0.14020742308082765], [0.5596249119559035, 0.0019708208793076058, 0.0029538077014630033, -0.07841208067313427, -0.0002731763386615712, 0.001962383141783696], [-0.22027208796808537, -0.02522608636392149, -0.005183175567069608, 0.012862239228236635, -0.0072847877156047855, -0.12968036568281824], [-0.14303589912242848, 0.030653839208640635, 0.0051190979385352725, 0.16243396668450402, 0.012724086935328104, 0.41880806311732344], [0.4148731779096417, 0.011415222107042112, 0.003971010886951547, 0.15569972198023435, 0.005561272561075973, -0.011193738778281575], [0.432729356760462, 0.022105236996976172, 0.0034578807530858415, 0.05535976382135382, 0.00899849123213732, 0.02772480501409468], [-0.19915443342544178, -0.002611534205814207, -0.01259384179386256, -0.211554107140234, -0.036380965255102565, 0.09782988182045405], [-0.2107363441826481, 0.00024607991457785736, -0.0027013310781051733, -0.22377081154579848, 0.000480532015597365, 0.05917362544845856], [0.4157702847923174, 0.027148051163143668, -0.020825230617613648, 0.13214377645868172, 0.025322035637505604, -0.03645732256326882], [-0.29464087218943524, -0.007991859449170248, 0.006089226286805625, 0.04152776427435431, -0.0013613882057821444, -0.14115334690724854], [-0.20468597517408846, -0.005541660178742979, -0.010590837593219726, -0.1581933613298043, -0.00456711723192523, 0.008405618174445654], [-0.3080732039355897, 0.020003837038689888, -0.008934357510086146, 0.07418165107990671, -0.02003490958480619, -0.14391834839478698], [-0.1155931396961862, -0.03998930004125707, -0.004762242053120605, 0.004953237440939713, -0.005812273254451824, -0.10664007027471192], [-0.3179383346023256, -0.0020395837458225116, -0.008959882174354025, 0.10447491679505728, -0.012732425189309055, -0.15263892777025495], [0.32576987577147987, -0.008242020814987892, -0.006172024737440448, 0.2649101309990991, 0.0038675178395244413, -0.10677109810529453], [-0.22565067462297178, 0.015879887775325244, -0.007700463324538225, -0.025880228952415273, 0.004768607379781452, -0.11427907841455451], [-0.1114096603329045, -0.006264431165509873, 0.0293271256538928, -0.24170742429157127, -0.028365560948668107, -0.015598970483867089], [-0.227754283886727, 0.0020193434058214895, -0.0064775419867534115, -0.11936768959273078, -0.0037750236919586195, -0.05556813758098492], [0.15836260082999784, 0.006179372426907502, 0.001061171409976991, 0.2589162647401836, 0.01626794307817629, 0.13720598084809071], [-0.29524488244890434, 0.01761537423424793, 0.005075560461805243, 0.04077480010487005, 0.009016120103479908, -0.14111863912216507], [0.29963895295509846, 0.008166084647088586, 0.0016937888968117662, 0.2873033729828412, -0.02396203263864277, -0.04576945255748661], [-0.16481180878495438, 0.02620074816852972, 0.0037570869787665164, 0.25858390998883557, 0.02634610253849088, 0.36741332618969463], [0.14313490959856745, 0.0034234328087840243, 0.03018792551621143, 0.24412353251684588, 0.017635434021330283, -0.015416663033167988], [0.4366136893033735, 0.005939438331027939, 0.0022362828558411233, 0.12704989867473293, 0.010825462462295425, -0.00710000972251033], [0.46386731854713364, -0.0006737218299827785, 0.004675086846937999, -0.20151409144550902, 0.029498695280632987, 0.1456344543035241], [-0.08848701784476556, 0.006474595329452941, 0.0011514140921609168, 0.19969722836041845, 0.009570954015968419, 0.35235780440174114], [-0.1879485885971318, -0.007897078322537318, -0.0067308978419630275, -0.13243228077722838, -0.01295880180011262, -0.06337235266102624], [-0.2859402018043946, 0.009825684459895177, -0.0069824622590639145, -0.05117967849037769, 0.00027841764754177984, -0.06725842622026794], [0.49413567244268863, 0.00834696795591477, 0.0075082923682764785, 0.11204533725605853, 0.0055703740645506195, -0.06391976236705943], [0.5581254036058462, -0.014735849659761706, 0.004828371024404088, -0.12717704647389236, 0.015345955579413439, 0.09868983259065142], [-0.1662422715602356, 0.035066409930315255, 0.008657581714607399, 0.26656395373253194, 0.013638788077334017, 0.39201448236531966], [-0.19841348399735595, 0.010864837417613456, -0.0026288503511240078, -0.16301288578917825, 0.0038115372105396016, -0.060294487823828194], [-0.3300856973618604, 0.004638954419430943, -0.006132797095518726, 0.14534795595790767, -0.029641340639712722, -0.14263436037610602], [0.1142492613851703, 0.0030730226379047185, 0.004512245291149718, -0.2909753518681792, -0.007321895267283302, -0.07425858469977217], [0.3250564567418588, 0.0027566047785225725, -0.014391362591966977, -0.2099156784607214, -0.06639584422283447, -0.01745216037184464], [-0.1528471839120596, -0.0074820732664736276, -0.003537780473637491, -0.2450484310825796, 0.00038952033313843223, 0.01856094840160995], [0.3010423002338351, 0.027100557844886304, -0.002731160149508296, -0.13952751993911067, 0.015596141657831956, 0.16272893453990717], [-0.16426359111716096, 0.003219020725730577, -0.012995275219008392, -0.3463426109578849, -0.0009377058566113684, 0.12726712255782383], [0.20038572069594957, -0.0064370074952440235, -0.009038229499911126, -0.2254987070335361, -0.06935953557892309, -0.09436110555719862], [0.04842930741104901, -0.003448797600155984, 0.00565351985209319, 0.2981859154531195, 0.01148279291605012, 0.19515488101545925], [0.38183271677448716, 0.0035996008337103812, -0.003078443176782471, 0.047308411326842444, -0.03628565004073982, 0.1467833642824811], [-0.22022883709264976, 0.012411525848963994, -0.008720433877830492, -0.1312173212667746, -0.00045489496474584236, -0.06096337198029618], [0.39233579238356, 0.010345841179364308, 0.010419725248280082, 0.08256174527570956, 0.010201223175867762, 0.023129006070548887], [-0.15431036074478346, 0.018688593166976828, 0.006624159203424936, 0.16127026158457314, 0.0058384282285170765, 0.35863295661625794], [-0.1903391447704374, 0.02958686951331294, 0.02155148224080224, 0.34981272601553687, -0.008649636689744989, 0.22543183067465253], [0.5059815058731341, 0.0010915609816128154, -0.0023880697526145714, 0.13179838925507195, 0.0024430994027002735, -0.06282204131546078], [-0.3012363009972347, -0.02456753228615493, -0.014206822811223194, 0.022028017344249428, 0.01777446227023284, 0.0025705574325114037], [-0.19714010463279377, 0.023483457223182742, 0.015837987495332745, 0.2447870984893502, 0.030007881943445114, 0.09651376606156828], [0.5012131032117944, 0.012163337653128162, 0.0019010437872349444, 0.031076634018498576, 0.01060744890952542, 0.01819843241981826], [-0.19288683581210045, -0.010228836553406397, -0.023254262428107547, -0.06293178562334809, -0.007045557668504945, 0.17145197152608432], [-0.1599271567792714, 0.017215569947675743, 0.005291973391990043, 0.3224625886734428, 0.014693379081313483, 0.36758435997055916], [0.2262820325364689, -0.022376662058163254, -0.01561691242285646, -0.3919958360437882, -0.05943630628976634, -0.048482030007613204], [-0.3197598625691688, -0.0014522934700436868, -0.010529378604086986, 0.10724457849682564, 0.00019325268938531791, -0.1773772056338209], [-0.3049653888370887, -0.009905706773784759, -0.0013518621839472846, -0.03740647941652587, -0.009275886216760879, 0.0021605615233463668], [0.3047993581774808, 0.0033030963143126564, -0.006955439037021831, -0.0021161750472887273, -0.044417609749196944, 0.16329676934171386], [0.48310672228049023, 0.01937170611082208, 0.006018803856830859, -0.07001801662602358, 0.0015846646382304255, 0.06962777558380322], [-0.031639235958519335, 0.0672935025855035, 0.010053127142143214, 0.05588907669827052, 0.021138027148305503, -0.056328483874453834], [-0.2181238804665811, -0.007398576646660075, -0.006160394940575507, -0.0571301154089422, 0.0027008882709284236, -0.1137332118134606], [-0.2649202544224223, 0.0078122586419992505, -0.010342433787486886, -0.17964016062574778, 0.005548918729780179, 0.05126714765435302], [-0.1182921100045777, -0.003923788358358618, -0.0065463880827537945, -0.2035638694745629, 0.0009096595223756792, -0.07493575850408246], [0.09188497052048798, -0.013319201647886506, 0.018820141002976715, -0.380749386892662, -0.034471752770695106, -0.044796436878892905], [0.44941010451915764, 0.00682424592350902, 0.04400133125757002, -0.10268314452861277, 0.010472972440471389, 0.1408636570545664], [0.5377035365195022, 0.013507237037826955, 0.029851273924576777, -0.04672531030912698, 0.014488377377224894, -0.02111749550238908], [0.41205099159591807, 0.022208435570699604, -0.006971852789642472, 0.0687221357454811, -0.01826161588491245, 0.044078572429122186], [0.38999843466103046, -0.005045396297292122, 0.003244256744895395, 0.1867203834603225, -0.005296112576683859, -0.00038097991168866547], [-0.086908128531699, -0.010379294967291248, -0.003139410768343673, -0.22083097370671803, 0.00036416341734488747, -0.07461302210995807], [0.5814021344996358, 0.0007500367851012345, 0.004767276382796496, -0.11302185926710467, 0.009109977139842958, 0.04222819203548266], [-0.1480386273618089, 0.018790861127043305, -0.00166788650759068, -0.1928068903284787, 0.004316171503994439, -0.06851696176649294], [-0.29650992717179264, -0.021916387730935943, -0.010386693981151347, 0.024054115114570996, -0.0071151789932841734, -0.10079926057073968], [-0.3174518116506852, 0.006465358227593252, -0.004945902376173394, 0.09266218011710287, 0.01333935454841602, -0.16480730610133776], [0.17673346322450986, 0.025516329546137136, 0.012815618722245225, 0.31057037732555, 0.029294677528511847, -0.06954120367414413], [-0.1834984073368105, 0.03091369974533071, 0.006442299596187804, 0.27028726466052727, 0.016987285895996598, 0.3707600002959071], [-0.05757473061524755, 0.05020751497063596, 0.021984576344388, 0.26211441830104226, 0.008397773029325014, 0.0006558560059411767], [-0.25219206700054214, -0.022243801468479405, 0.0037589396357964582, -0.07094654404862111, -0.0012759733311451863, -0.06885722045367446], [0.06969415672324536, -0.009750901064206148, 0.009264804636158879, 0.39214373909882455, 0.015454950002288774, -0.08391081866038139], [-0.19059855814806834, 0.006272853468924105, -0.013829423672630367, -0.20973301716214046, 0.0077642042338036356, 0.09822009907656236], [0.22530505448628274, 0.01696111777079363, 0.014929988032422946, 0.14188171773829297, 0.0009342565699320248, 0.15737974946024852], [0.1604515597334538, 0.02774952108417281, 0.009024364390097853, 0.3644303704187072, -0.014613643326335166, -0.06636601583751445], [-0.3360528973786684, 0.003742776300701745, -0.00703763221170147, 0.08470571701263983, -0.007319434511854903, -0.1292747661966784], [-0.17255700230121074, 0.009980907566722125, -0.005349689323218537, -0.1726140345829252, -0.011513347197728195, -0.055834453209258464], [0.5361928621017321, 0.01087529720593554, 0.01343984384938907, 0.05109286989208364, -0.03586922160447469, -0.03955549498208337], [0.4867622418888143, 0.0013805929275774336, -0.011002702434617168, 0.13814490756252312, -0.030031785647858977, -0.03949404794723358], [0.33203929821652417, -0.004930703258313405, 0.04883784923871512, -0.23759374764038393, 0.01275817238068749, 0.1879598453484824], [-0.3248985507947476, -0.0010971023384796193, -0.008022080652222957, 0.1036496946637155, 0.001649699028666679, -0.16666784464588963], [-0.3230158543235441, 0.014711288356992694, -0.010621139510985088, 0.0632112387002747, 0.0010242079261035765, -0.13441336476897595], [-0.3188980099085688, -0.008869735775370462, -0.015043716556042063, 0.14928336332655384, 0.0036346841280244947, -0.17273987525788806], [0.49410029796739385, 0.009769019173451014, 0.008011412821181922, -0.10607757820085738, 0.016517921496854975, 0.10953535531340017], [0.21302642249292372, -0.014580448639102572, -0.0022748392059814644, 0.2218587655316885, 0.014247815382706632, 0.1337989511044319], [-0.28598608089104804, -0.015572854891504813, -0.004891184860769795, 0.02973597919555413, -0.0003595876200380634, -0.11051627093219268], [0.13931262122523205, -0.013467688954812063, 0.015900892803819674, 0.37754753221021775, 0.016034658222126737, -0.041144205982777016], [0.23050514780648737, 0.015545170463654938, -0.0027963208590924926, 0.3284940334426319, 0.008738942120156648, -0.03657697297383893], [0.23866649003007612, -0.02289173890212882, 0.005664497416510779, 0.2603137337287615, 0.008265725527227659, 0.05490319696145702], [-0.3025705202506331, -0.012842968182908211, -0.007350256952740672, 0.07160867848438189, -0.002028035828566384, -0.1506453478036959], [-0.18485938066401233, 0.04834395252755887, 0.010692837480456524, 0.3927325530121678, 0.016294661706290816, 0.029984374716534788], [0.1994163810579529, 0.036145988424934265, 0.002406998631926767, 0.25910525627828285, 0.01257295031604657, -0.05345883907695859], [-0.2333790446424589, 0.0025004820875977034, -0.015213495936386342, -0.2821391138530664, 0.006104935507347768, 0.12352335062187411], [0.5017332562493594, 0.00456521653220567, 0.0015421967112939697, 0.03984415910622041, -0.03284592632065743, 0.039601143073051036], [0.3065832823765051, 0.007459905949470617, 0.0007603623397796721, 0.22172260668303156, 0.01367519584692632, -0.019243734148094292], [-0.31971754397840146, -0.010350513857742476, -0.012849716302331273, 0.10833687923728964, 0.014637549574744075, -0.14255358108048408], [0.6043446649679437, 0.0226404649568321, 0.000983870385098252, -0.1096201157431075, 0.023304291587102902, -0.01089198567768408], [-0.2111459779970724, -0.016141124248353057, -0.005142965580448089, -0.2837545831440635, 0.005691321599089101, 0.12938571502336801], [0.577737678844919, -0.005879768625444065, 0.00359995811149292, -0.03314126970335933, 0.014881194927838645, -0.07611030437796083], [-0.3266456144822338, -0.003922499983701412, -0.0008076677150053645, 0.0856591980221774, 0.002214293341777257, -0.13238257917491353], [0.3929138579685506, -0.008190491529189102, 0.013194515554846515, 0.1934623511505072, 0.024330610252821443, -0.040384176730871434], [-0.1570795359490227, 0.032106157058356205, 0.0006939362304683902, 0.18914778122383016, 0.020185344528626963, 0.3980949081775794], [-0.17568651565856375, -0.022523558529825634, -0.005741064679824131, -0.16756781567574688, 0.002396347206147143, -0.041050725995520565], [-0.06482858456011197, 0.010962316157521364, 0.012065603486567147, 0.358158817880558, 0.016336012477657914, 0.04732694566891482], [-0.18898339510225162, -0.016319607683397956, -0.009735623457671524, -0.29693904519984393, -0.012502847880263755, 0.14848499754024577], [0.4641332820157625, 0.0066414260740646726, 0.0048686525047169854, 0.1438026087890798, 0.00317228556173211, -0.06488682637392998], [-0.12417799906854628, 0.032390772120130716, 0.008731395461271758, 0.29729983397574583, 0.020807773815526794, 0.2038914362562567], [0.10233659084207644, 0.005946608694245911, 0.019547281819147784, 0.36698245617140973, 0.005690896440843313, 0.011941880317990193], [0.20382787587812248, -0.05253638043589744, -0.04950988037850042, -0.12176541565227543, -0.09947839054437088, -0.08157261406188339], [0.4084679935062652, 0.020557011989212177, 0.009941918962528993, 0.027938093034272474, -0.018908078765873738, 0.04544083905137127], [-0.2589987182744322, -0.022493328026463956, -0.002900781281847019, 0.006660482672571146, -0.003278309640591308, -0.12599601211590397], [0.42320988650839575, -0.019476247879509287, -0.03285239238458289, -0.10521119317064452, -0.014142589255054327, -0.09269260462761897], [-0.28855130319165834, -0.014000114321713834, -0.01140270674294934, 0.027891943645558023, -0.006690027642645195, -0.10608779174659025], [-0.33648224404656674, 0.011988078288915044, -0.004445278430598387, 0.1387093511853774, 0.0026953438287461474, -0.15882143767225698], [-0.19527423201992783, 0.007165254438990333, -0.012094729796354578, -0.26881535668716044, -0.025774282652198343, 0.10132834671665067], [-0.26420972329516706, -0.002024041487367834, -0.006758622092007155, -0.13144814154048903, -0.00032107597664140613, -0.005965650510288105], [0.5961626486944583, -0.02505574742204749, 0.004627008992700559, -0.08974557790772278, 0.016699970648833187, -0.023718779196703475], [0.5159850072873807, -0.0038083363579674585, 0.006159945608973678, 0.03772241219173449, -0.034871556114052536, 0.008000305161705202], [-0.24718438963646128, -0.002912971715995923, -0.00481559822361189, -0.01318625712404283, -0.00912562859106072, -0.11178182137549329], [0.36492679458202704, 0.01278849151151371, 0.003798255014304617, 0.17374344131984706, 0.005772276484104209, 0.01503106788558799], [0.41720193877259815, 0.03257479378927531, -0.007715515844964399, 0.13414301856385627, 0.014069523598550735, -0.053528099328140656], [0.5141599506066157, 0.019711339127421262, -0.0003125583866277816, 0.06459966553238544, 0.014263603194652807, -0.031678666741117384], [0.4231847371181929, 0.0064654286991368114, -0.003333242085385478, 0.10595753454232461, 0.010104614917786471, 0.014449452839046691], [-0.2624683675550211, -0.045317450199773714, -0.019452250250009358, -0.00477849043071277, -0.009047885298179384, 0.04298290238031094], [-0.1751579615529458, 0.03400606672616352, 0.03673908665029194, 0.2674780199527914, 0.005813018645122841, 0.23741396780670365], [-0.24797792148579215, 0.0025384119871136638, 0.009060496664830386, -0.0419969963072772, -0.002252777306689406, -0.10786411609251102], [0.2450060538441209, -0.017727539661017307, 0.012252055540227314, 0.24097200185430712, 0.011003653045033339, 0.05715377537732817], [-0.04189929284288194, 0.030505822026004183, 0.009437979927885209, 0.4481902595382961, -0.0018807359816159113, -0.06757498504864184], [-0.32179003161988545, -0.0029042995161745956, -0.0106327216691517, 0.10996583513584075, -0.03368916289882508, -0.07417912674988358], [0.45371029108382954, 0.004468819044249886, 0.004940098410812984, 0.13264860139004891, 0.021137693112969782, -0.06378428370569286], [-0.0010538315052275346, 0.0702642329454718, 0.002123371413362964, 0.2646038317932669, 0.02277882974160616, -0.04630143263890175], [-0.2057217938978234, -0.004724785898903344, -0.013709662465500773, -0.2937509934506863, 0.004976621019581949, 0.1392415803427486], [-0.30103700933635275, 0.005062494334347222, 0.006053575220348049, 0.03577388105117752, -0.006473836548705904, -0.1328808452791063], [0.2950120616937776, 0.03042847564585782, 0.001621895955929931, -0.17446024809682553, 0.01640670519153797, 0.17906470989832352], [-0.1548818948457254, 0.024713053170119053, 0.0019245981398721707, 0.2702231038273104, 0.016880536046352066, 0.395619309826707], [0.5822617135600799, 0.01586479034691467, -0.0014219340735962642, -0.07670991092004292, 0.02496107153220601, -0.058724301874137894], [-0.10676023939012236, 0.04266479378501636, 0.023330363905164138, -0.11627951469999434, 0.023201132823868983, 0.23503645545498775], [-0.18426524907832278, 0.009697644243657143, -0.004078704753173915, -0.18350995018424474, -0.0027049332464818064, -0.04731214031476672], [-0.22451099445262865, 0.0031458908800371397, 0.0018981606208141675, -0.13981840895599587, 0.006215412368875458, -0.04874064869639594], [-0.2641320897841926, -0.004615622357232469, 0.008157985697252637, 0.03709153636008791, -0.018529598587495096, -0.11627054466175309], [-0.23465426198071887, 0.0048838128976954895, -0.012805984503835192, -0.291852629670329, 0.0023426832571049492, 0.143208572412931], [-0.16094321544245807, 0.0168960747605584, 0.0009930235548751584, 0.28603411801915773, 0.005293771615326875, 0.3642850370163444], [-0.21590311465983072, 0.008431656513213903, -0.006184033404338638, -0.28977841671410637, 0.002323458102798287, 0.1381573549241669], [-0.2624301865491314, -0.02676670360965206, -0.00859195369251867, 0.07681729774732933, 0.01216099260085934, -0.06578702225446216], [0.48435884919116573, 0.02355022978539408, -0.010404388168851093, -0.09983827729098335, 0.02852311258596352, 0.1188038072306428], [-0.17151725139765542, 0.036304240769366766, -0.0024079536748966425, 0.21135351904916752, -0.03160453832574765, 0.32105182484960415], [-0.12927211412917258, -0.009413645442191344, -0.011180887126164536, -0.35475971763340614, -0.01166514924958872, 0.12591579929480706], [-0.26194327664283623, 0.006104029238420043, -0.013680749983574615, -0.21197303387419725, -0.0007328333732322077, 0.11320082322755237], [-0.2414694266371605, 0.0027246297300432785, 0.01258514646480159, -0.2661727748218078, -0.0018920865775220543, 0.12864046422259875], [0.2065120520056126, 0.0009355516293006227, -0.0015368192361927293, 0.2232549588860697, 0.006470467870362288, 0.14219045551151432], [0.3092735063248656, 0.03702856611358486, -0.00037782489822540055, 0.21782187715256848, -0.03197126136066579, -0.07593691033274662], [-0.04470171760969136, -0.02334557643969584, 0.004708364130043834, -0.28077435132180917, 0.006939874132744453, -0.055380878605877856], [-0.2714328437885285, -0.0034455795340400373, -0.027909410403303487, -0.09983401653420154, -0.009424706667436831, 0.17888017257895655], [0.31169013683559293, -0.057642266810445215, -0.030839355543524346, -0.23819837322235543, -0.02119655507359522, -0.13065048059561146], [-0.17334075768747717, 0.029946250055394138, 0.008250315094945854, 0.2811031928835719, 0.018839881420168476, 0.3785754039476792], [-0.1233009784247247, 0.02377424156146179, 0.004371043708760678, 0.3749637223915157, 0.01566611410177217, 0.18401867239619998], [-0.16479251542129084, -0.009207228770514456, -0.004162082247099916, -0.15546378685248902, -0.007306003106479951, -0.06332505026879175], [-0.32438791939692546, -0.00627186753012204, 0.001832889456931773, 0.10699343541804765, -0.0012470424263909025, -0.14938901359383072], [-0.18499797002768448, 0.011571722418859474, 0.0039992089971201015, -0.1792683267984035, -0.00081490796072073, -0.0512463932958373], [-0.2933872928331362, -0.014887221737617168, -0.011543438178642023, -0.060854367375501095, -0.007960587965197986, 0.000417908090095369], [0.4240048258000063, -0.0046811274696976005, 0.01944949963426275, 0.12364645189003186, 0.029574241784704015, -0.03266830722372535], [-0.3579228745244764, 0.0031942568972738675, -0.023298536756607764, 0.16407547481319498, -0.005290781656593528, -0.15199833242358782], [0.09268571907274867, -0.018630290048866027, -0.011613044579117967, -0.3354027904116262, 0.003501135892428331, 0.08268871451887494], [-0.33639087826189756, 0.01117800414855817, -0.00432728638354486, 0.10930322542982807, 0.0037117985616489844, -0.17743781757055216], [0.05168386074414678, -0.01864987239915436, -0.018467512557376498, -0.2969793918951198, -0.0023526526284552195, -0.07399989710255206], [-0.05444428654721025, 0.018310990614181906, -0.008280918483426686, -0.2847910754865155, 0.001777214548899087, -0.046684360543365964], [0.4997522254907602, 0.01452922190404868, -0.0152564469776437, 0.09056339097318039, 0.008290453736199905, -0.05670694036464283], [0.46399969019460174, 0.020613089945386465, 0.0017658642543830281, 0.10381832773250654, 0.009926621035051219, -0.044555690606968484], [-0.09007595910505803, 0.016940378250033344, -0.006390315096307676, -0.2598409620388067, -0.00242466505172426, -0.05663181029146983], [-0.12979983233295264, 0.03916347132769575, 0.05932075877656987, 0.2853392116506313, 0.01965557174832885, -0.06337861340470499], [-0.1641273024748927, -0.012387401405091297, -0.008715020880987145, -0.3435849695849447, -0.0023579552402801644, 0.13032968943966394], [-0.027445268848811797, 0.022309319453150605, 0.03891454489621634, 0.04108066063689801, 0.020071225655393217, 0.27972753408016865], [0.44699705253768934, 0.00018536105621759993, 0.003309384969405959, 0.17785927617360958, 0.0061056743625052205, -0.06593563798831846], [-0.14598899938237497, 0.01203851402351988, 0.0016232423310689283, 0.33800648455316434, -0.012537501947722973, 0.3190496457037231], [-0.23225009105678956, -0.017792707528722983, -0.007905640982898613, -0.09279673200976535, -0.002121102489749419, -0.05733615979450871], [-0.22064372694050824, -0.00827787404729346, -0.006690802301300243, -0.197264518641755, 0.006191655051133373, 0.021701649858446897], [0.34366824095789983, 0.012334740992067364, -0.029519535181288, 0.22354667296409167, 0.006464421988520949, -0.03486466437641643], [0.36774572320255744, 0.012515781047521797, -0.0019242138366127013, 0.23521316876073226, 0.0025225232907354228, -0.06468192601073393], [0.40304551626283225, 0.024637670736268883, 0.022562672216378973, -0.04689672293732493, 0.023867174619382075, 0.12604677351804583], [-0.23799730997020963, 0.0017076722750237062, -0.006937078439892444, -0.20695230455428132, -0.004118071087524253, 0.05346252655949262], [-0.2314999838251698, -0.010832295374876885, -0.003947902128667622, -0.11142675480699447, 0.002577327030955225, -0.04829372422857933], [0.424787720868268, 0.02532400271651413, 0.0006692267561519833, 0.00741692953116298, 0.03734940205283819, -0.0178513116228387], [0.36121351467522456, -0.013087500388360821, 0.022079851893238026, 0.15647479322465083, 0.0021065822545800364, 0.03070929680220597], [-0.2569550413469459, 0.006781313916915467, -0.0039054617020397443, -0.09952514897315867, 0.008256316358363179, -0.0591586449198012], [-0.259693501771299, -0.011826446509951863, -0.0056085384163630375, -0.07049327744639024, -0.000986147754518636, -0.06356542143480932], [-0.33405705735285757, 0.018683257731182006, -0.010209297357691683, 0.10826899612820094, -0.022137395689582444, -0.1335662590083505], [0.15340093328569124, -0.023941651652520973, -0.009906814548938802, -0.41736863285467324, -0.011067838049554418, -0.06078932951334045], [0.26607967637202545, 0.017326327546897346, 0.02215473457854806, -0.03845481874528984, 0.023403651979545678, 0.19607674500709107], [-0.17349220504033708, 0.0026831271164765532, -0.005092264192247775, -0.18150021351888457, 0.0009380074030071062, -0.05820978510134779], [-0.27648435021343026, -0.026752790645137015, -0.00518806998208143, -0.0016218764571309758, -0.006204505039289941, -0.0014527588111903254], [-0.07864440835512772, 0.009339993554482832, -0.011472776274451642, -0.41265093067250097, 0.001977023662790115, 0.11635043917007448], [0.3654160173349027, 0.015152617428192648, 0.03442598396966692, 0.14793955394921182, -0.026671053349140383, -0.00984915107886535], [-0.15818450065298686, -0.019452604078136353, 0.01599086035524855, -0.31681991158294726, -0.001318135219340288, 0.12021734082354944], [-0.194256209828052, 0.007470836217068658, -0.002936606334703492, -0.22306353193580233, 0.006829532090843534, 0.007449313123977657], [0.35109782407232826, 0.004085478011330715, 0.03869051109930588, -0.0023586739616489553, -0.06532172125028034, -0.024860798923417922], [-0.23451757404970727, -0.03973121885129854, -0.008187116569433875, 0.22385814977827437, -0.0086989262779206, -0.15599369790306805], [-0.3191227809261021, -0.009645202027752589, -0.004478290728041588, 0.1358722145481297, -0.01337829760618188, 0.0006658581405072552], [0.11902606376136964, -0.016970042307824992, -0.020920878476239548, -0.4313777922607312, -0.05125754935196362, 0.10819591292110207], [-0.19460100692342386, -0.011118384369905329, -0.006287864161733394, -0.131220325048047, -0.013951972462313124, -0.05374378036790962], [-0.2613361041172985, 0.010820889595150896, -0.005935115718842287, -0.07059880898790495, -0.01444592556552492, -0.06359493520557997], [-0.3110837100584845, -0.0037478081426045506, -0.016388157012140588, 0.20737434343350233, 0.0024867141734668963, -0.13932229148464856], [-0.2420619151029801, 0.010223752158268692, -0.007052940592470692, -0.1185468829128288, 0.007845956399468685, -0.06258130328279066], [-0.2183090443349958, 0.0031621503658857096, 0.007716361821952363, -0.11274878188682295, -0.02016673359073612, -0.06495228570861647], [-0.16141611104363987, 0.012345756250877539, -0.006383912518337533, 0.29849411935993586, 0.004080413536312098, 0.3847850278810707], [-0.27512208589861337, -0.04606034092393113, -0.015334786533681456, 0.0018069653496561651, -0.024927437244278924, 0.03753358022083079], [-0.3145839353284272, -0.012584810341344798, -0.012134710884072053, 0.1087394862341686, 0.009190358044820114, -0.17297396348272082], [-0.12502068690485005, -0.0215972230661572, -0.009923527103188567, -0.2528408028563393, 0.002604890018890176, 0.04945282379945849], [-0.2408196486554721, -0.006024698314205529, -0.005669093195870409, -0.04495892287241442, -0.002850892838633655, -0.11001674412340375], [-0.31301339253402255, -0.006845294495495291, -0.016078815335522526, 0.14761897754415101, -0.004781188559850842, -0.16808119571017122], [0.15648488032711125, -0.0076894599296858535, 0.01379489596822272, -0.026704268692926594, -0.05283348781203615, -0.10094659814266513], [-0.12189027301696972, 0.05017101061338118, 0.01148498989680848, 0.2684613017721616, 0.031169017082317303, -0.0650859272777758], [-0.20765253907294315, -0.03764311057386216, 0.006004443485922738, 0.04163746461994529, 0.0058926019124866495, -0.14993600322869127], [-0.20634711394272703, 0.012375768741224697, -0.007333510149717729, -0.1256134529828893, -0.01539216370240743, -0.06694619463014886], [-0.31793893085300673, 0.00674549352506509, -0.014035349911859186, -0.007918729911972254, -0.0015545070939875177, -0.04854286242155132], [-0.2674246721282895, -0.02548569373614708, -0.008744471294544478, 0.1076599140590268, -0.030742922477316248, -0.14818940050997487], [-0.3324941137640551, -0.008815449655699645, -0.006056327328081091, 0.094285639844702, -0.00550697089081982, -0.1552513438973833], [-0.296098388573546, -0.01741100256998056, -0.01423762039304786, -0.03468299354113851, -0.007542179811240753, -0.004938515465654805], [-0.14816330976626846, -0.013118291387499167, -0.01396194878732155, -0.3522832556252084, -0.002436511838138223, 0.11980832718271581], [0.10635734144819614, -0.023418157428670695, 0.015521101715216236, -0.409352037453138, -0.005442592561670724, -0.05230922714850732], [-0.13847627804106397, -0.0068755250983634246, -0.005071753404545815, -0.18451387670959632, -0.016073258574770558, -0.060329308171659345], [-0.20588985185963166, -0.01872983090032419, -0.015011212741532397, -0.22685016115516024, -0.03696200604239121, 0.1240784932968086], [0.454470736247555, 0.0279391132942326, 0.0011740936638445045, -0.012696267379467599, 0.024524142601875302, 0.03333189646817332], [0.1394912833009435, -0.019615757530730044, -0.0073419121700049736, -0.4159924494998443, -0.00920355205162566, -0.06836808823921857], [-0.3260368669864568, 0.006834108998178937, -0.015662250888370498, 0.10909543923216505, 0.00047601367048495533, -0.15101501182476573], [-0.22118135495636898, 0.008085755833441201, -0.004546795345713639, -0.12963583469454804, 0.007620079772224734, -0.06648577217766181], [-0.14878306826745658, -0.02655823001087867, -0.010907255808273649, -0.30737634681903453, 0.0019353937930806362, 0.09240196852265672], [0.4891070662611712, -0.005956024064976431, 0.0032808386057651497, 0.060452496591903535, 0.016874475227348265, 0.012692814045452855], [0.48712886996566124, -0.02080668331175556, -0.009419417342397917, 0.15119934060798784, 0.009115331574234312, -0.057438393874683843], [0.4005072061629588, -0.01847827142276621, -0.0007621275447390298, 0.22738296507918518, 0.007179937043710005, -0.0656697093183502], [-0.14843843226256148, -0.011977049115409205, 0.0033800945266663192, 0.25965514668173434, 0.016945914195104095, 0.4024824765835455], [0.2783774858524714, 0.0053921864255187255, 0.0012330271641811492, 0.162699561960252, 0.009194374998524763, 0.12718003026571867], [-0.20530709378247242, -0.01353695179930988, -0.003382595723919675, -0.12741721334881387, 0.00017210201597440825, -0.06270158069479208], [-0.32366715187784145, -0.019239463870548017, -0.008939862762775929, 0.1905072778138232, 0.0015872004919553376, -0.16549438507599779], [-0.08481159350324075, 0.01273542418512822, -0.002153768798939989, -0.27592940523849774, 0.008744471354306888, -0.06084179466542217], [0.44676041592002325, -0.023317771870977663, 0.007180544662729561, -0.03365019161049896, 0.035480649996798806, -0.025419639161569783], [0.01459280702526516, 0.030711317902404882, 0.006386728370692888, 0.25889176454210605, 0.01980858202483617, 0.21939380013469395], [-0.1741185314968937, -0.014635754779251327, 0.0006720855626343179, -0.30160045916632877, 0.010874359351248473, 0.10410409527202001], [0.573657499658125, 0.01874058352628811, -0.002210123436164105, -0.0922847890555467, 0.019779451978852752, -0.023457146481084438], [-0.21521020925512113, 0.012047444794202453, -0.005213765309236394, -0.12313178701468999, -0.014420197579642085, -0.06582815230217864], [-0.27273851646634817, -0.0129940948530943, -0.0060014888984098475, -0.0635821170837417, 0.0054571950029514765, -0.06481431103468942], [-0.2184293734956953, 0.029401622493223156, 0.010515247843239256, 0.2996070192367829, 0.022154628942384822, 0.10868105700026642], [-0.2879001967813945, -0.00581942729665197, -0.00713632385669126, 0.02085928984620536, -0.0016679735073979653, -0.13300870173740215], [-0.28403177770861043, -0.0024570483657923563, 0.04661457736280496, -0.09006382754924762, 0.0009412017590444691, -0.036831220736294634], [-0.14188466275943237, 0.0093316956736705, -0.008443054048066582, -0.34350687395773194, -0.0017545478648099589, 0.13152855406747888], [-0.317398592254392, -0.005218530548403523, -0.006321308985556406, 0.07017779102383027, -0.0016330717145411888, -0.13654550256292422], [0.30071394930301615, -0.014621794942653438, 0.0055008614715690895, 0.20584296319495393, -0.012163386851893962, 0.06534574115833956], [-0.18376810327656465, -0.0012527282485263113, -0.005225110926347455, -0.1934552565215775, 3.220183628285251e-05, -0.028504336196600914], [-0.23046655537966557, -0.00011842190409994066, 0.0018556201736695502, -0.0617410948364841, -0.0030354834394811557, -0.08675073128060555], [0.42629631976899157, 0.008094863601373433, 0.0058586942677927994, 0.12025042136322822, -0.0030468390787193114, -0.0052573522395000126], [0.32142935406184997, 0.0143072148731892, -0.018973025640930567, 0.24758696787528692, 0.009365918831904022, -0.039681430001300526], [0.548692509428009, 0.022509691154757164, 0.0027203394242777575, -0.07950087767696096, 0.016261466335018154, 0.03524959860762021], [0.09430576672140503, 0.0009394937139879773, -0.0013192140842745426, -0.31699745179421485, -0.011266535864765113, -0.08343098026076735], [0.5109954328556163, -0.014073047101896866, 0.00263777455121585, -0.05743377148425934, 0.009469713024616265, 0.1039329457737525], [-0.3333101411376767, -0.010328491143489965, -0.011166392619518078, 0.14335291453177543, -0.013548198698361696, -0.1506007798151614], [-0.319388665995945, -0.009430663957066177, -0.017752644379825728, 0.13023139316649132, -0.010780230881820589, -0.08380373662570627], [0.503836835890533, 0.02484669816871053, -0.004705919968482061, 0.07347738444156761, 0.008919343392531847, -0.05692081589376144], [-0.03359534232414578, 0.014233516359394913, 0.0056971445488206455, 0.32105837414629157, -0.009726619312213287, 0.22121859945969502], [-0.09991703705289667, 0.002439724827699638, -0.005930658447076308, -0.25603517859396463, 0.0038416389927791133, -0.05907182305987468], [0.4222887355248286, -0.01626114846275171, 0.00568971835289982, 0.1497637512694718, 0.009262068232523861, -0.00377360110745062], [-0.09542645262991772, -0.025160375502440113, -0.003148862499769785, -0.22827877771358918, 0.0010665807667484442, -0.056892112421031975], [0.0032306424593647846, 0.03366519472827221, 0.014655910310415757, 0.3551059933567626, 0.017047638879214067, 0.0004129535993026083], [-0.07057050098114735, -0.0043028006172458454, -0.0066522930054457976, -0.24357298292694868, -0.033442681045954416, -0.042840758229980694], [0.21545481854083978, -0.02832046059123492, 0.08318730218406023, -0.20068924462647797, -0.0881186698746774, -0.12451445991822441], [0.44731504854880555, 0.020502639403081554, 0.0037551211497804434, 0.1282508925770546, 0.019271868796964912, -0.06173629511336918], [0.15167991234480202, -0.00390802901619083, 0.006331699067441776, 0.2771975368524018, 0.026635502452643647, 0.058390044965568245], [0.2261883730466908, -0.005255575695307683, -0.03156705298271306, -0.15789876843939496, -0.04006708483415258, -0.18311773788601274], [-0.15690986538193644, 0.02997419640632983, -0.0017366406844238679, 0.22628931641803895, 0.010474125692301993, 0.3600086222394418], [-0.31764930800558516, -0.017155740179106704, -0.018384225493410657, 0.08802267483708957, 0.0006611782794522493, -0.023215531819393453], [-0.3232703414591462, 0.01478204381986867, -0.005771727411696917, 0.06367466548346046, 0.005431909430805535, -0.13595490121183934], [-0.2284036301618745, -0.038315398591163334, -0.009150900034169818, -0.22934516745065045, -0.0026284607908563726, 0.1284028525212794], [0.3761810824199638, -0.032286211631132544, -0.00522542329020807, 0.1679512222259853, 0.01484158017170721, 0.03311567939661282], [-0.32732813775261344, -0.005865575936262103, -0.011890832665065652, 0.15413632018459564, -0.019827054673007067, -0.14232832459982578], [-0.21139577178746322, 0.003073234340388959, -0.004934597100165691, -0.1359813977436102, 0.000834345845888321, -0.06357306845699875], [-0.23543092370754348, -0.0007195424380554558, -0.003092688899036031, -0.11116161675129467, 0.0013995673417322705, -0.06118633616204847], [-0.26384770114833733, -0.023207987420372358, -0.009339387761592653, 0.0021420391215910897, -0.007145337973584133, -0.10953208065816009], [-0.19427784267859557, -0.01113043769458745, -0.002693655612002944, -0.1200312968218787, -0.01750044255041805, -0.06737299130918316], [-0.1525904288349341, -0.0001965609939690194, -0.01407100381368979, -0.36437204236368614, 0.0008370877183341561, 0.13033831243360128], [0.3273689852547654, -0.00029032336496344027, 0.0009706469964612601, 0.19947041797385362, 0.006189531049903207, 0.03783566272490022], [0.1368012260564319, -0.020038908745432973, -0.015893267146826207, -0.40887218797153674, -0.023782866858340926, -0.05808660402995007], [-0.2734116087534184, 0.010988161429626142, 0.010285833491876693, -0.0751006714645374, 0.014872371199604205, -0.06991853034759499], [0.3733241840045753, -0.018326879166126017, -0.008608823163772414, 0.23369543930590894, 0.016789868302580554, -0.07347767817205628], [-0.29336151915907643, -0.014800663741173404, 0.0037266036753820465, 0.03402057901566431, -0.0018744114985550636, -0.11905058829224187], [-0.25703274303220397, -0.02127495991507483, -0.018398135869809925, -0.21860929721643166, 0.0037262863149329, 0.15407274838575358], [-0.16200578491875486, 0.004448310334542186, -0.009486182324251056, -0.32439818476580695, 0.00793077262378132, 0.10601432301874078], [-0.2286459064050644, 0.02193943278243362, -0.012002823786078181, 0.028213371046077483, -0.0051295788734857465, -0.14491790198884075], [-0.0862641083179698, -0.01832504880095814, -0.00012006522491920565, -0.23348295830445984, 0.007723190359778021, -0.07291862875908889], [0.5144808183886931, 0.024826210270292488, -0.002879770617353162, 0.07037553106661636, -0.021973155713538996, -0.06829539422536049], [0.3758636057425583, 0.002709102638405434, 0.010540336202433734, 0.13001975566100898, 0.011140970177019885, 0.03553478030321169], [-0.2931792560042569, -0.018164270000954573, -0.005560584613905521, 0.02403067014211523, -0.0015970717385602896, -0.11353615445110415], [-0.2333750445044323, 0.008298694430608227, -0.012640968752394106, -0.2861007836633776, 0.0014508902640175964, 0.13081814061243674], [-0.17493629321724885, 0.016826876560189188, -0.00688133490187223, 0.33687988911718303, 0.013242525060941713, 0.32111184271275894], [-0.17302834940281872, 0.010408105186640555, -0.005986194159167362, 0.21341638582220956, 0.02046655148078891, 0.3899307699799082], [-0.17172661250540702, -0.0019119094791948635, -0.021650371067693273, -0.17409753764105462, -0.04935461052158441, 0.11931175550064774], [-0.2980438692573856, 0.019366332497200454, -0.009126883578111223, 0.03551267071005104, 3.716183787758099e-05, -0.1487961965233585], [0.548607668239155, 0.021573840388702695, 0.005920817200783781, -0.07424040052467196, 0.008833741430127178, -0.0009166191150552267], [-0.2516308235835734, 0.0024564659433540287, -0.006807197438539887, -0.06685909832180031, -0.023321695327393154, -0.06351098460538013], [-0.31615500248147166, -0.012676099556732417, -0.01523064833830391, 0.2623388605436531, -0.002178115360703698, -0.12953423290167917], [-0.22566596749186105, -0.014975704212112506, -0.005188800689655901, -0.11302756507708335, 0.008622187784627838, -0.06443748364724877], [-0.17502852747599115, 0.02053022314920896, 0.0903223826901638, 0.030781017120061278, 0.010703654369126863, 0.32082942475060394], [-0.17853559034326202, 0.007029380738971823, -0.0032131628242560973, -0.24933106330746116, -0.05387726379651333, 0.11193889000871061], [-0.19762475219885597, -0.01313371607437155, -0.009015566663694729, -0.12582805929035823, 0.0014951238199688874, -0.06056636292602121], [-0.31227700870134295, 0.014785200106641356, -0.009469353311887647, 0.0609060773713392, 0.010252955805459884, -0.14582953793687706], [0.5191002823051639, 0.006694753564304406, 0.0014636304902214145, 0.06662342618333277, 0.006010977575892774, -0.0557676657243389], [0.16702464588974092, 0.01751564713730582, -0.010224261641186826, 0.24487853934838377, 0.011809073678434277, 0.13848968892065572], [0.3240418126096402, 0.008197487712592426, 0.005327623141830541, 0.1644909373423721, -0.009558765277858108, 0.06847042828094418], [0.281101320192009, 0.00849608156705602, 0.02046354803869557, 0.13114855227935132, 0.008001458442661837, 0.1273776109087963], [-0.2435679135305753, 0.002144794490029075, 0.014902832437836785, -0.20159089715887643, 0.004956768026327211, 0.05387393954478304], [0.023490734672936837, 0.009253603147320225, 0.020339375052495914, 0.2738684909073454, 0.012479721477693398, 0.024288172324364286], [-0.3175975860529308, -0.018470026191861246, -0.004933781252073615, 0.09812912554481748, -0.0061754722281967345, -0.15477988275014867], [-0.1798552114378432, 0.006135469258653364, -0.010066199936092556, -0.2498619635479257, -0.006002220237199259, 0.04088280031901055], [0.4724064368033822, 0.010599293269166142, 0.004889847257576988, -0.03338424214334999, 0.008182979390848773, 0.10854496031414687], [0.4473041379349307, -0.042891535023856245, -0.01314839434867595, -0.30390323124297436, -0.02102214926933403, -0.11065898678025199], [0.3258869602382251, 0.018431744574279475, 0.007355953672008021, 0.2271940077904423, 0.027352428809774516, -0.03742902402348652], [0.2837337095142594, 0.012151269115136776, 0.004338619151851803, 0.249706456195279, 0.019738233750025376, -0.0336104990226387], [0.1945596696894763, 0.0015193273630998223, 0.020224925279070573, 0.25445129611762235, 0.02626415410927427, -0.0043891344633053275], [-0.007648947413253696, -0.03884640091920556, -0.005687863402339298, -0.08233177087242424, -0.0638965522494691, -0.08563896297880516], [-0.16852930773378938, 0.03243506735468136, -0.00231551556573183, 0.2229718175866907, 0.022398393396910254, 0.3529809408529271], [-0.3471819013913401, 0.007154535461591851, -0.0019840863051392454, 0.1091097798423583, -0.0017423585054665467, -0.14981051021577005], [-0.2320265922026727, 0.015020920580778783, -2.008568133711336e-05, -0.13605858298774479, 0.006955087470156952, -0.05076630273473633], [-0.14151876857779852, -0.008256949715886261, -0.0015270665641694842, -0.19478290871542323, -0.0055299348792264965, -0.05176603821416314], [-0.1549862089076372, -0.019398883983311795, -0.004999852948450056, -0.2112781401356183, -0.0023432376669722424, -0.005000343024677596], [-0.28492370978845266, -0.016396032202348313, 0.04904422917308087, -0.06473722066004178, -0.0011280042623045127, -0.06796116702183971], [-0.13333496141845097, -0.01784459269746214, -0.0029972058038893403, -0.18309986406769882, -0.021300967811969357, -0.04859574153386259], [-0.23987167409550586, -0.012314257018463986, -0.007378714204617981, -0.10698143951985668, 0.0036496292655864974, -0.049276877760475415], [-0.3132872284995253, -0.0013136284722493486, -0.008316371978456218, 0.10283941286922141, -0.007482551483604982, -0.148523869122395], [-0.29147628579993934, -0.014012624368867724, -0.003914740876123667, -0.014651939384702026, 0.0013887238560862034, -0.08750646675978492], [-0.11685764886557631, -0.046570571352029334, -0.019068956092142104, 0.008329065407956212, -0.0029150706602597883, -0.11225681843794746], [-0.2511838436887223, -0.016197024407073893, -0.018305630984222103, -0.2405491404062916, -0.008566800805204992, 0.1456286370333663], [0.5185958964870159, 0.026476126308836518, -0.002067441403992188, -0.09071509718657082, 0.019870992330941115, -0.05258083367909111], [0.4105767116889068, 0.011940589719917152, 0.01298174236605909, 0.13629114202994777, 0.01789782810250182, -0.03898634724066712], [-0.3001037377904291, -0.016599919633952163, -0.005673125057311794, 0.02446375603184046, 0.0028451036320559986, -0.10244586635599245], [-0.2821824337209744, -0.026454283991327005, -0.017160227190476583, 0.15085127234293644, -0.03178094907703936, 0.010417850070281868], [-0.3000833112991618, -0.006867317997980014, -0.01969749183364379, 0.15317863830595263, -0.015306721194541769, -0.15365671100211317], [-0.24187551105456714, 0.0029336355895471997, -0.006726768141403584, -0.10311363516194487, -0.003396532998581656, -0.05854844313501008], [-0.0994192683173216, -0.046710703928124736, -0.023779816912324128, 0.18252272919623275, -0.010342018645813389, -0.14314861980534607], [-0.22656973857134086, 0.0015109480805457272, -0.006758536089165915, -0.16197783288553194, -0.02385696920542151, 0.013312128670913932], [-0.33913355647486954, 0.006951348685726847, -0.007183876149456583, 0.17570282022554592, 0.005503836205004105, -0.16950795344433484], [-0.23714473376963438, 0.0036762719209207243, -0.004283685335495778, -0.16051822809098565, -0.00099474922651306, 0.009555559284315993], [0.47676308760343195, -0.014338189393827982, 0.0032560266801241514, 0.1483463184014341, -0.03455408888874398, -0.03679514463440932], [-0.1304966709293511, -0.027338971880199816, -0.00422712434369821, -0.19599611789704932, 0.009301781328238681, -0.045082896277940786], [-0.2530897227238725, -0.01535178602026283, 0.005408909235942952, -0.017835825699914414, -0.011806847888894894, -0.09758139356966364], [-0.3452641947569572, 0.004723859983290007, -0.015796915031338527, 0.13512798975793203, 0.0011215722173425436, -0.17033255089706173], [-0.21038714746973428, -0.014379194927328302, -0.003707290370720395, -0.11983203424929555, 0.0029405628470900538, -0.06389156249667807], [-0.1746312378254173, 0.013789222246128328, 0.00903175490671028, 0.30871561739294895, 0.000491257888520476, 0.3450848139625342], [0.5326799502995093, 0.02378976165621659, -0.005499672856219986, 0.04821156816549208, 0.012994597398935095, -0.05500079850147068], [0.19214179965417258, 0.0183738051768871, 0.01297455616659235, 0.2942726669525721, 0.013181886003654204, 0.0078273550116382], [-0.1665592345616439, -0.003178872856663704, 0.07006284889482761, 0.20714680754296363, 0.0247108883184615, 0.3708942293287196], [-0.07690036846251817, -0.0322246532717046, -0.012529644181397672, 0.07196694230659392, -0.02731159249319095, -0.14197267558334523], [-0.13509946758157781, -0.0069370267272133015, -0.006770604245361124, -0.20301737195107858, 0.007416176191447698, -0.06359837235288285], [-0.251624047360667, -0.010022762800892869, -0.009180348706397636, -0.07912102752756789, 0.005534621021251272, -0.06400976795905987], [0.23838796497019638, 0.015340318964851194, 0.0006224321817939698, 0.18836679944192064, 0.0019418316059845488, 0.13316731950192034], [0.44170958636632984, -0.017751894847863783, 0.01269427451621986, 0.027934454930943785, 0.017038594383195327, -0.016562057195869062], [0.5283295061640053, 0.01561996443058658, -0.0014422625330186229, 0.06055060281515844, 0.01949809591787186, -0.06019853837355347], [-0.18683055708438712, -0.02069808578464517, -0.010284238660471406, -0.21540917490279554, -0.0036802480553395993, 0.04745766789615844], [-0.23902570111183083, -0.013419074211205657, -0.0038441721648288093, -0.08666425256217822, -0.01075590816195839, -0.05367851083561618], [-0.3445435347028857, 0.010823622933450015, -0.0054974442751847844, 0.12866627216337062, 0.0016691864175005227, -0.1651600513403647], [-0.020642655451461177, 0.011584021836469357, -0.004766320584484812, -0.29126225132979155, -0.0014954664054562172, -0.04899542330337199], [0.5008677442175504, -0.009442545815636439, 0.006535386192449672, 0.12030928704205496, 0.008053865650681117, -0.04535604497941003], [0.5277781848199243, 0.012393028378731501, -0.0015150822145352575, 0.05475322217000654, 0.017277629603444585, -0.03779700564086946], [-0.1924632731310897, 0.02448031822205792, 0.01553270934591247, 0.007403313759539147, 0.016874317912621565, 0.31304543761693787], [-0.2637955460890237, -0.014778187504538414, -0.005215687088575157, -0.050471302031115904, -0.010923463438302843, -0.06948914718177548], [-0.2533205247181579, -0.015426547459364292, -0.005441166685550618, -0.1621160249479379, 0.026129937750200564, 0.045625642587419545], [-0.22447179734185735, -0.01045203826525503, -0.006954159617081421, -0.07551293640889854, -0.01930114660061344, -0.06362940324777568], [0.24106479752070673, 0.014510605760118143, -0.0002741543689331087, 0.18084363347504998, 0.009628947740677742, 0.1380806143168262], [0.08421900769214506, 0.01670396564056272, 0.013765143920076335, -0.0841775999164925, 0.02475756290105503, 0.13971533246106632], [0.4125210380839491, 0.010497825491869018, -0.017494994462409777, 0.18315723431540476, 0.013524795879311545, -0.07390304216526797], [0.3484175518517239, -0.0033108804036861895, 0.003313328255745742, 0.09183746302630502, 0.007423013585124507, 0.10293190463716725], [-0.26024321988525945, 0.001806629288973346, -0.009487717170373557, -0.12133841282987244, 0.028491323546472245, -0.019663299919636244], [0.4909132197968163, -0.003130808900790827, 0.0003905466305442119, 0.13092343244873555, 0.01633384616400616, -0.06712921049828739], [0.3978739192453189, 0.017338885918100462, 0.004285166179369768, 0.14262869007105058, -0.001863598154162008, 0.018162342902786635], [-0.08886261907440107, 0.0012172606027092874, -0.013131999038791437, -0.39258916058316257, 0.003537101053816639, 0.11807870275410984], [-0.19090644030691298, -0.00896972204058196, 0.004883619168462797, -0.29609173744385825, 5.8862647229285215e-05, 0.09676875130899287], [0.541601173118465, 0.01633282969662932, -0.014591520707757755, 0.02301990137820801, 0.015045615949454489, -0.02922130390830727], [0.5187321245036014, 0.02518227796518278, 0.006031780775799874, -0.2093904760410093, 0.0003815575731420229, 0.12324149424203978], [-0.15606371769772226, 0.013302896331663682, 0.00316492171352733, 0.3358973271289468, 0.012447916151784333, 0.3388551008162378], [-0.2772484791812054, -3.768411316974008e-05, -0.0036218437263669063, 0.11943452973728125, -0.0150296837403482, -0.1516036444922189], [-0.26919543923691847, 0.01617162257352539, -0.008081426905530112, 0.012812710982679517, -0.012880581282161995, -0.1254701741134528], [-0.23874693584154508, 0.01133723764106613, -0.007654746425825791, -0.12798034566562902, 0.004786378888380783, -0.044482126230854846], [-0.2409883330007793, 0.011851094343593141, -0.004815469079842915, -0.111212152475256, -0.002589842949031797, -0.06441863017201604], [0.19732000377856154, -0.007269958772542208, -0.016537462818646095, -0.4570754783860233, -0.0400641713611488, 0.025849696959380927], [0.175061814123786, -0.008592345676859094, -0.014802747088134732, -0.43788056976769496, -0.007920117017902021, -0.0014738917160552205], [0.3635189208815641, -0.011455261655009732, 0.0060295417228106735, 0.19545261578391965, -0.035984993332156774, 0.0007658432655347707], [0.38355263649848903, -0.01461006987534191, 0.012862793201433567, 0.22358369094647024, 0.0027675356410305886, -0.0692227768882732], [-0.05048765099591267, 0.008404901865729742, -0.007803436817406771, -0.2119433869764132, -0.034332028263536866, -0.08462940613846776], [-0.31205782895565365, -0.014857325629471656, 0.0025895932666537666, 0.09797091690457706, -0.019373235568704836, -0.12727121961927151], [0.351576270170881, 0.023789248473015737, 0.0019921251684645483, 0.13675430782070847, 0.022168192840599586, -0.03413800161652902], [-0.31104175090252856, -0.011865953725773409, 0.08269024956078644, 0.08201215866242079, -0.002704332120272119, -0.13253211176914614], [-0.21151332211443705, -0.008630028436503168, -0.006332025791108968, -0.1652944765612677, 0.00032473903251592414, -0.01822821946253266], [-0.16453876289300517, -0.025779357314632576, -0.01061708184858443, -0.3220446515536579, 0.0015875959192220704, 0.12052768828842805], [0.01439587067603442, 0.04289985093484252, 0.02364347582479042, 0.3677563782795554, 0.03428462292158245, -0.07678495120450123], [-0.07928762638570165, 0.01629992258726901, -0.00900110645700692, -0.33977899168012315, -0.0012498158790628328, 0.02355261781462421], [0.3805532579833721, -0.04115372535940885, -0.028983890415080718, 0.043304339078743065, 0.03588974845057832, -0.030896158309635293], [0.5545712289541332, 0.02030046152860042, -0.010718050031183894, -0.01573054901910573, 0.0086888771653155, -0.01824139512106366], [0.43263876471203055, 0.02506266224095129, -0.0009309547665472821, -0.05404148857842535, -0.0561709925526193, -0.034878510535911386], [-0.17690998574894654, 0.035541378692321664, 0.004750125472105929, 0.20880743073469563, 0.015912365771378893, 0.3908086850784408], [-0.2502576945845751, 0.0010387131796650176, -0.007524211431411537, -0.11530795587451996, 0.02758856440509934, -0.04950817708972428], [-0.3118354063425892, 0.015143348837314522, -0.009748421692118078, 0.0651174393304174, 0.0006704139221974132, -0.14647574704878996], [0.40423891994002586, -0.006171154524024581, 0.036206910196643634, 0.12977984611713703, 0.03582026213062579, -0.04208491373053847], [0.2538360951907908, 0.009713795589790148, -0.0006936350168584489, 0.18193873400756075, 0.008708942687872699, 0.12482273420750872], [-0.14434403542618568, 0.007515209140564825, -0.00571413774306557, -0.20527129207512873, -0.004167290076320015, -0.06269178715319851], [0.4692109837872009, -0.01926994780943186, -0.00931430646512942, 0.1272450407706719, 0.00766273386373135, -0.06070783748037811], [-0.26708384994189405, -0.01181404561195631, -0.00696441260018269, 0.04490409421657371, -0.00023481576190163004, -0.15236890859460442], [0.5451039534634318, 0.01064642964547521, 0.0026314806203995593, 0.02066015654421823, 0.015358193534442482, -0.05350211856987637], [0.5925888855193615, -0.012880817422622426, 0.010836646825434872, -0.11731093820908048, 0.02201511103432282, -0.042457935366466464], [-0.1483519731669482, 0.01705244016961448, 0.00756096068008605, 0.3114984430565093, 0.011537331824493338, 0.36096398791243145], [-0.2802197329304939, -0.01235523459546264, -0.005204191945167215, 0.03907479199593171, -0.02204508145123316, -0.11709055107357252], [-0.1946643667061471, 0.0026211037030075294, -0.007223772764245115, -0.13337608104065848, -0.020235267667036316, -0.05076553709354755], [-0.31055593322994474, 0.007076704578762322, -0.009533675376246603, 0.06611720164611462, 0.00421632474177835, -0.019135589422388185], [-0.05871729290580038, 0.014801566395371807, 0.008099933574960068, 0.37580745029807705, -0.03545786648239525, 0.20993573292930529], [-0.22869238365260044, -0.03341477160821259, -0.015055310798119455, -0.06599674460730413, -0.0016184959694603063, 0.02823967201404619], [0.527943115144789, 0.021165024138124853, 0.003454470057367112, 0.05270136292997654, -0.03784023166911092, -0.05646385397983374], [0.2743998346428929, 0.01433308207495246, -0.00196009365802551, 0.21269416440642025, 0.00877944507669944, 0.06274690079039431], [-0.23446819746077036, 0.0001020197300741837, -0.005860431384760485, -0.11725699163165428, 0.001789065048740463, -0.05878271920359004], [0.37153167750448607, 0.006565980409845332, 0.003839985599922333, 0.1759271926297325, 0.010320783377703724, 0.012141047144975775], [0.22077194823851287, 0.03740076432430816, 0.008030852186718701, 0.29749519610969793, 0.02239767585502532, -0.03329540562055671], [-0.23507159363507218, -0.002332524828576446, -0.005518627833888714, -0.15057815301831445, -0.01078386659740299, 0.005640844344626622], [-0.17433873889976878, -0.016506786990505937, -0.005299716743052036, -0.14927603852015572, 0.00237989029944564, -0.06338194247929713], [-0.23734388642969728, -0.0031164422313688695, -0.019343583879989263, -0.2546154224065711, -0.0038327905581636056, 0.14039551421010651], [-0.003447263355068444, -0.00014370112410150822, -0.0018810714866937608, -0.3310911318373521, 0.005483927443451379, -0.05747504535452346], [-0.31209371147231024, 0.017893372477198122, -0.008507485505937242, 0.03155744853894929, -0.0020746162338260224, -0.12372223708417168], [-0.24531896493896257, -0.007888503053633544, 0.0021752291704708027, -0.13340628161809762, -5.194540511097218e-05, -0.015016200821332178], [0.4379299442393311, -0.0023976315893281526, 0.0383158619525309, 0.14406378861119945, -0.02671428884528635, -0.05794676527753986], [0.559652876949367, 0.03794738371318991, 0.005486480413748038, -0.1263771007594953, -0.010782490004022052, -0.03447548364612688], [0.3104930540683817, -0.008968839776012118, 0.000818840290373109, 0.13794720494582355, 0.004312837522260225, 0.135723569615839], [-0.26720417197654595, -0.040224243860694, -0.007323415250343731, 0.14495463025672908, -0.01832785360529929, 0.010328187631459469], [-0.23954606484351548, 0.010909912083245432, -0.002417300391124279, -0.12188985905322222, 0.024110904583850457, -0.07216776553940722], [-0.32376079165274857, -0.009426737673680758, -0.022244839609486197, 0.050197748271471096, 0.008933757111162328, -0.020658184065767513], [-0.23099558778107368, 0.017087014418309298, -0.004491190404051312, -0.018991348893616384, -0.007955660089189309, -0.12179782495152723], [0.06182626035541116, -0.023631837645104802, 0.0145910554721566, 0.368766570912191, 0.0035631026825069115, -0.04812710530256432], [-0.19389179406435045, -0.01318012596281073, -0.005968917979299787, -0.13289861803017705, -0.003167063122541126, -0.06556681417415385], [-0.28696289873118985, -0.0030212208155856977, 0.0034285197939601577, -0.05504895522574165, -0.004046951622689478, -0.0025243068908154163], [-0.12733540030645163, -0.015972443336527913, 0.004272081158428014, -0.23280960574604528, -0.0030220287111035788, -0.011639269724967066], [0.46829840064383393, -0.020839823442977606, -0.005653165071996364, 0.1252318156106669, 0.016492673015630198, -0.04414157352683126], [-0.14300429973433315, -0.01884704582873098, -0.01109761982083478, 0.10871141757527739, -0.06583395186659857, -0.043715692021972355], [-0.16089230006112318, 0.009134354359330651, -0.0041138173204172904, -0.1867994154797142, 0.001258554171129898, -0.06851070900254], [-0.19951864246417492, -0.039202275423441076, -0.017322852913697683, -0.2219106176140243, 0.00020247224117800534, 0.1133352418719979], [-0.3282898718833669, -0.006645146898191881, -0.011219254321742807, 0.10021388371355684, -0.0032640206703318355, -0.16189255899260668], [-0.03498684599016113, 0.041370402533912165, 0.012444511691530498, 0.36344084946797356, 0.015002456011293354, 0.00924128329027863], [0.4838084733168819, 0.01669117272170173, 0.0041414446103383105, 0.10788462029252045, 0.006986623697600251, -0.041221382258092125], [0.30974766596265774, -0.0734435578270001, -0.02479120492934322, -0.009809081973288674, 0.00046808223906310496, -0.12291125412143944], [-0.1246364952675813, -0.0037768789482605534, -0.013660024047890929, -0.09933875314466135, -0.0027051126759668523, -0.11705606924897091], [-0.09693164569216116, 0.02447756759085203, 0.01572316136998676, 0.24399001733343403, 0.014059168163841089, 0.31425442964674316], [-0.3027223142823444, -0.001485650442707421, -0.007283793618080882, 0.025746297598486567, -0.0002777453546655487, -0.12846494204883618], [-0.11880920925004447, -0.01767832999777936, -0.005423443153782973, -0.20651214112421795, -0.0019247516801123457, -0.06432545812739598], [0.3059725208655394, 0.005932837033179114, 0.0011995406398442473, 0.1162181600429515, 0.008104350204603797, 0.1395242578805467], [0.5036875130889132, 0.0057668573982224205, 0.013578752698208506, -0.12125530632862531, -0.03504305526310547, 0.10882594186958919], [-0.19943535413435798, -0.0066642992006604816, -0.003627411475713918, -0.23270460889366687, 0.003462076906052036, 0.057343364914289055], [-0.09546729766122292, -0.02277455514675988, -0.017728028301608665, -0.3635634340391307, -0.0019364876978412698, 0.0988671382062373], [-0.22104545478075688, -0.03923104694099205, -0.013646082302168606, -0.09218195858921877, 0.007612823139298125, 0.16964776854627112], [-0.008039606346668834, 0.0029550341609668306, -0.0074096640939974945, -0.3268826630454043, -0.0051254247689299, 0.024300602482310616], [0.14912022988407891, -0.016174885472410912, -0.008079520767063714, -0.3976305231053025, -0.037719743676331206, -0.059387302894720065], [-0.04366752557413645, -0.015409662726767446, -0.005202203964940265, -0.12967770831579312, 0.002992089930005344, -0.1133077094060877], [-0.2213065162027848, -0.01929329213948216, -0.006503337676219127, -0.11579915741002206, 0.007210800414912941, -0.049981830319739275], [-0.1935044793834724, -0.02194067796154153, -0.007387546292269154, -0.059886860114331815, 0.0026395550356710577, -0.11552189604596168], [-0.24293920102875363, -0.009276849778416849, 0.013362705689856529, -0.26418329583692957, 0.0018422666052624172, 0.15549723149183894], [0.42662235767849577, -0.011059516184373674, -0.0015003291591012856, 0.19756828832586174, -0.03235727169226745, -0.07499917110425848], [0.3257893894851646, 0.006258623451859088, 0.0016502357433726205, 0.12308452374350524, -0.040712460172709394, 0.14500635441547413], [-0.11386369070668946, -0.021474321874710735, -0.009514671573328803, -0.16569079401965126, -0.02345513025035664, -0.06942472490859558], [-0.2663638127269903, -0.00999828114802411, -0.0270212111292758, -0.09676483097685278, -0.008286787065786002, 0.1292865265822814], [-0.16523108959720118, -0.0007835649840269288, 0.0008586475328606817, 0.3063559841738167, 0.025395190265377454, 0.3520049586595889], [0.35769432015875735, -0.014004150035972936, 0.009401846308192397, 0.23107380636733854, -0.028128824477853236, -0.103144855463319], [0.538002904350441, -0.027089671635709685, 0.014636766192625937, -0.13916274392645048, 0.007758843432924894, 0.05239810072036336], [0.1660401428944277, -0.026186767695469813, -0.005421715689219702, -0.32791193874381686, -0.011278620342758562, -0.09086979089935551], [-0.2261091553228325, -0.013923547993948305, -0.015404943820962127, -0.2746537359043605, 0.000335056957989502, 0.14108358568621254], [-0.16976293623001978, 0.013278116097496729, -0.013561850558536917, -0.17970057400141415, -0.0005744377726555114, -0.013744508011061621], [-0.2417419714054628, 0.0011002834581953923, -0.004446911467101956, -0.10280404274864355, 0.001225428423890422, -0.06633945292754383], [-0.317938914604191, 0.00501348051433431, -0.00734447265797743, 0.019382100860834302, 0.006468374956428566, -0.08294748418330182], [-0.004866160236575234, 0.01121039373026066, 0.008155833754076235, 0.328315178263001, 0.022487722206268986, 0.1873247557359829], [0.17916203610759562, 0.009328343077448755, 1.473508702901328e-05, 0.24920207380035678, 0.012195664244107587, 0.1254238143501292], [0.48853923675391325, 0.009818550109188792, 0.004137754149857108, 0.009172565228337412, 0.02542590616762485, -0.04865334862826063], [0.495647111691516, 0.025732369641053233, 0.007963447360630566, -0.25184162621350625, 0.015135096946535264, 0.11797093102109756], [0.5851797629925503, 0.020804994600292778, 0.002171262473091684, -0.10056122726187229, 0.030930966930694956, -0.04778242640142845], [0.42206709239074774, 0.03415598253636966, -0.011513468513671887, -0.0638265900189326, -0.0008059401293553535, -0.021003364976448908], [-0.22853423817204158, 0.002791869500536149, -0.0048503172881320476, -0.13266252984705437, 0.00042723455100789385, -0.043732606979609755], [-0.1828777777937093, 0.009068882949717843, 0.0037446562633966815, 0.3245836214831164, 0.015694076764324983, 0.239936102099885], [0.1369382303769936, 0.0059814328378763075, -0.009229157625647461, -0.3524383885171726, -0.0033526568952691213, -0.07683927702660041], [-0.26446475611563736, 0.015077728173964902, -0.020531156095140116, 0.13114091786590465, -0.01680867293311306, 0.09357927243735328], [0.14660507604470668, -0.0025027463895461125, -0.019618419828566613, -0.4132379889928431, -0.0026315706305887515, -0.07387101686983318], [-0.30932362482731157, -0.007824692323748857, -0.0034181939083761646, 0.0970649496025365, 0.004938072705153339, -0.15255032077206374], [-0.2885600396692793, 6.169671775189005e-05, -0.008773964589926644, -0.01761684010779118, -0.000996852406542571, -0.0912873332775444], [-0.17762430420727104, 0.036110215641762596, -0.01069494605936993, 0.32347613663350266, 0.014318391783152633, 0.23992533593404838], [0.3479429006164491, 0.019610265087497265, 0.0030352066521297785, -0.22860258809191059, 0.024298217335341304, 0.18769039233988616], [0.2375538855647252, 0.03021576935178785, -0.0068292299712055925, -0.12041874653677101, 0.04694146117121842, 0.2045103524837375], [-0.24674405399158728, 0.015444588813435259, -0.004828889893109086, -0.09651110332465465, 0.0016232353135322603, -0.06505195152079095], [0.5479061616373939, -0.006972203477446226, -0.005897091119648467, 0.01461682680304722, 0.020772330402253274, -0.06333348456306424], [0.5373317172369034, 0.006860866068870785, 0.00450145995356075, 0.00812627098476505, 0.016052034762931624, -0.018537745832430817], [0.53508160604434, 0.0007035610478276559, -0.0043284217343525794, 0.01744385363624263, 0.008673829938879486, -0.017112841631354007], [-0.27566360908364124, 0.0029434152416495593, -0.0073181470887317595, -0.05919517907106145, -8.363065580984891e-05, -0.07167099749055363], [-0.17564948755555013, -0.021536229625301058, 0.0007009388321146998, -0.24520410209044902, -0.003354506504788647, 0.047938036066779224], [-0.3166167925455074, -0.017521932351401513, -0.00634282779823036, 0.10685883983044059, -0.017550412488414963, -0.14975268642546918], [-0.3153008263472842, -0.016443399170418586, -0.01997859476248607, 0.2799782299966601, -0.019092461972377635, -0.03412993187108064], [0.47387009313095135, -0.016774089013471317, 0.0009790339988914952, 0.13534589993255552, 0.0070994330007824865, -0.04213601207535288], [-0.03345477719338314, -0.0003318161232234513, -0.007291992015070881, -0.32319983080269726, 0.009262498119000822, 0.02539020372965759], [-0.06688900903013084, -0.004402123629793227, -0.006927465524127794, -0.2223690992852476, -0.029520945509010556, -0.07145194525698388], [-0.2968119499206485, -0.013962318220026965, -0.0039194699695024055, 0.10220379945168223, -0.023366507874979608, -0.011548644952328735], [0.3924078644123803, 0.006114269471991069, 0.004641051038531484, 0.13871694992672856, 0.015586766139579631, 0.01796582628351577], [-0.2679205697472668, 0.0027954677066728715, -0.007747305030891814, -0.05395871738939903, -0.002235762882175154, -0.08560644599027303], [-0.17343408151212955, 0.0376746721680848, 0.003056047123653558, 0.2807702149474138, 0.015373237148841708, 0.3779937196479409], [-0.18082595257246611, 0.009978801782957092, -0.004811591069878635, -0.17887382273848662, 0.008131108468307021, -0.06410521053709926], [0.15399597499330075, 0.01073643646506284, 0.004825966387619653, 0.23132204923658609, -0.021253624261768352, 0.16941414956015216], [0.4676326157060575, 0.02753695259814147, 0.002196476589801079, 0.1013639597558975, 0.009364264256225562, -0.034311719886516705], [0.3612164915303502, -0.07257792502766623, -0.028048834529906252, -0.2456227136401286, -0.0257364635014937, -0.11706730807791067], [0.5220748801248472, 0.012222076809036222, 0.0058248984278844885, -0.1466754325272798, -0.010798843218740202, 0.015016388638217509], [-0.11144201569118692, -0.00868137131958783, -0.011903734557295799, -0.014416199974357677, -0.012947923392158686, -0.06290981080553774], [0.14855367731802716, 0.013784311740219928, -2.5952515659178892e-05, 0.3371521198387829, 0.009523776952774747, 0.01215175854260264], [-0.3004306214188538, -0.013361475839845684, -0.010763201693135825, -0.0455918491244023, -0.008913827921825304, -0.0186956906686039], [0.3145718640835193, 0.04144550998721539, 0.007280184261704968, -0.17500448034835606, 0.004330648993955251, 0.08518744823563618], [0.004833037535274535, -0.010998486149301587, 0.115636296959161, 0.09186622620760156, 0.016211215098680246, 0.10811690285111675], [-0.27069788755311497, 0.0053309977408752774, -0.018461290630068905, 0.20778622138761252, -0.002071012223751004, -0.15930832242784665], [-0.3094369014737407, 0.014003401181120725, -0.012785699212018794, 0.0611968065730171, -0.002798061752226483, -0.14590663914601684], [-0.1588183152475117, -0.0006420374388792745, 0.041761794136293204, 0.22882391859976878, 0.023135394611986815, 0.37837940406849796], [-0.1970262240737532, -0.0464729490657293, -0.021043336124762384, 0.22642512792731442, -0.007028606214444655, -0.11100607020345094], [0.4842766263369427, 0.020392188870132607, -0.0013538173410625623, 0.09744671928646666, 0.009676047297534657, -0.03732853666452695], [-0.2181575450249322, -0.019998515811442134, 0.06079071820308121, -0.12420210431055503, -0.007155245118803519, -0.07336730793734793], [0.20151539361111892, 0.032999062479970816, -0.0031608003540405357, -0.4236260165393641, -0.0001488910399939231, -0.0674947554837011], [-0.22937082831868047, -0.00035613324860096334, -0.013226428850799665, -0.28746292782330174, 0.011307802949955403, 0.1382367692596802], [-0.16014404032119384, -0.012777736084785146, -0.013285208685100853, -0.29474536579474825, -0.006185446238667249, 0.0899947116646801], [-0.3001360781318604, -0.003396179757734162, -0.000334962382547271, 0.10684299248059752, 0.009697966754313397, -0.15740158481257085], [-0.34097032085688506, 0.013393224949452616, -0.005647228469071369, 0.10069054738136919, -0.005739031427041163, -0.14901248191966748], [-0.27476084959265384, -0.01633078576384259, 0.003045016560897913, 0.029835650850535328, -0.02268347539124695, -0.12465388999702179], [-0.2079315829264118, 0.005529993681444787, -0.010418285847626397, -0.3070325363012561, 0.004145776873096657, 0.1292968293259473], [0.11422810089390499, 0.03459995021224458, 0.014464329577299567, 0.368252993165638, 0.021016887893112374, -0.032469591844431794], [0.47964417186599406, 0.015316537101279816, -0.008468776680638434, -0.05138289297188934, 0.016557344422177, 0.10858885435831385], [-0.0782164390936302, -0.05376385957468105, -0.027667387842321502, 0.13541488989422212, 0.016084800431496283, -0.08313878252657023], [0.39565482136350955, 0.01959603555829814, 0.005950918777732087, 0.062377279531292656, 0.008399233587730094, -0.03957424119951641], [-0.27594536061637137, -0.05209185613718383, -0.01224063233205193, -0.00627712092951086, -0.02404573322622591, 0.031156066649863996], [-0.26017595588529313, -0.018829876509305725, 0.021198848854521047, 0.07991334138406159, 0.0017427421524476155, -0.1366176714250022], [-0.23814089367967636, -0.006220622638417909, -0.006400912704387576, -0.11379992320207113, 0.0019486839574855822, -0.05186358663489278], [0.36782277918833817, -0.0028257907920403846, 0.0018196671307341384, 0.22764083989928055, 0.019254140705080414, -0.055807588512345845], [0.2390234249622235, -0.03129201970373847, -0.008793160906340573, -0.3722955136898897, -0.052801704318365845, -0.01124934932525809], [-0.2191896971075779, -0.0025884735372962457, -0.001860351059884384, -0.12303997635836893, 0.0019730846707531643, -0.06977184150958593], [0.39639668581985715, 0.007592584233146986, 0.039662460564381725, 0.16238171679175928, 0.01204597414754154, -0.07185989774716431], [0.17784757486058483, 0.0051870449757913505, -0.0003011863296751664, 0.2090985384801183, 0.023031255965663782, 0.16358843871418485], [-0.1571043370304833, 0.02178017902055101, 2.805814395525815e-05, 0.350728225100875, 0.008344628614319555, 0.2947165794841097], [0.2902012573365303, -0.004800645730240049, 0.02537290665425327, 0.12054528528868007, 0.022903756824088847, 0.03796124915049595], [0.45700576191217585, -0.012745601439654353, 0.004375637619444392, 0.15904234072763457, 0.005514719969354883, -0.042866192122290606], [0.4423849787065761, 0.012387938442928665, 0.013470959859405694, -0.33289410466747965, 0.0072124291094520345, 0.130269335345648], [0.5795781661894039, 0.02164871058819513, -0.002921701863132142, -0.12592150015059905, 0.006256078828622543, -0.024733721846461804], [-0.3177094656806139, -0.005174298335998149, -0.007368022074298733, 0.10292792597517623, -0.030386384192031542, -0.1320957420867921], [0.1521074384443102, 0.0008931239633339593, 0.01129495803203642, -0.460961822957724, -0.006206016781243391, 0.010986073878038282], [-0.3067640916632184, 0.016443939189167787, 0.003996472793522711, -0.02034519950117114, -0.004966411484276457, -0.08748248711180207], [-0.2809954466004535, -0.0128807262466331, -0.04213591771378528, 0.2059094399371793, -0.008669936517454095, 0.024265402876135884], [-0.2618906789009407, -0.058936643010624366, -0.026946661966232105, 0.10353917320768215, -0.02259442929783607, 0.06793817480504195], [-0.30531997518372556, 0.010366273943126534, -0.01644134625331691, 0.020314228508924014, 0.012150374428502198, -0.11212694674785931], [-0.2719288018664269, 0.011354378498447941, -0.006758960595478598, -0.0809990272231175, 5.0078237354774064e-05, -0.056307667050779775], [0.44882842626289426, 0.02484883118497751, -0.012097338175610825, 0.11623412877963778, 0.022356808012986156, -0.05431158070256717], [-0.08727008642911246, -0.03199691491866202, 0.009761919531404636, 0.2649659020886824, 0.02133832576920136, 0.30078907524699877], [-0.23265866247662792, 0.01295537498083957, -0.008916426711359894, -0.11930343472448067, 0.0024119186373192355, -0.0633287697056894], [-0.3311523444307409, 0.007029403287628375, -0.012079427377102199, 0.0675629967630329, -0.0004618565179715311, -0.055699763766034784], [0.32808787733306616, 0.0016288073440356532, 0.0025487921471097227, 0.18280833691608483, -0.01262659874478196, 0.06356992786162566], [-0.13561174514010715, -0.030773670603074885, -0.0044467966542280565, -0.1035253410513337, 0.002292109091406331, -0.08263662460817917], [-0.1375410533274831, 0.004950408384072817, -0.009601772021645107, -0.005696823271055532, -0.04510410604483928, -0.03381292356032052], [0.04919838494178877, 0.08124735699627993, 0.01427123608552899, 0.2152478834754596, 0.03272090106719749, -0.04660797721929793], [0.31313142939021293, -0.0017136035139540362, 0.001995736193735386, 0.1400156675892346, -0.03801882699433586, 0.14649959733510726], [-0.18208847241347328, -0.015925224605945018, 0.002781107953314818, 0.2822847027291174, 0.006318867927882302, 0.35414271515095525], [-0.1163750204789087, 0.007985159117935467, -0.00977604013552955, -0.34477048195847154, -0.019993231277990242, 0.11686937663772365], [0.5011242910678625, 0.009959115369002444, -0.009809169807538943, -0.06740110098087208, 0.016865056037221854, -0.01836787422536247], [-0.27168386437098696, -0.006720191586415603, -0.008691586774559938, -0.09336451249959617, -0.0011673850866748667, -0.02112317396748062], [-0.22965347457223945, -0.013434385990315567, -0.015395113387406612, -0.26716994221132, 0.007337232161173331, 0.12428918667390514], [-0.11915410246694136, -0.01592331140925797, -0.002796725843513215, -0.21540775520436825, -0.007988846271479607, -0.04631925880443963], [0.40812651956731844, 0.0029021622052333666, 0.05085404258506089, -0.34061876314713757, 0.010340476598196063, 0.1339424669532295], [0.17993985662444223, -0.00554570311435297, 0.004622909824612413, 0.2625804670423869, 0.02046814171224145, 0.12159432791066968], [0.48076948014333926, 0.016427857325061183, 0.07201095991624562, -0.20259385650413492, -0.023483011763653687, -0.03008650848194046], [-0.3172107252573314, -0.015399742527229823, 0.014686349246221796, 0.1506065669336842, -0.005713862649757589, -0.14473608361585666], [0.371418547107176, 0.014687990609342968, 0.02752082199167997, 0.18489147117514124, 0.02582345090074558, -0.06067943499319622], [-0.16302782134999305, 0.01131948245512797, -0.005292132060631079, -0.14981863982988503, -0.029529744859322388, -0.059157811021962915], [-0.31261888859977605, -0.01868995865510518, -0.003919517996278722, 0.08507450022777531, 0.011140521792867905, -0.14926845571128133], [0.37615733352609027, 0.021022575061579236, 0.018236785564219656, 0.16359494916883582, 0.01724869398915913, -0.09149319445274232], [-0.21365523457680896, 0.012413654485470308, -0.006065134827402408, -0.13937508002908378, 0.003598922054446084, -0.06825712710662056], [-0.25192656404671077, -0.0008195506255279853, -0.013891123680427003, -0.03794526088636165, -0.018005036512504614, 0.04684924836640062], [-0.24891296973138038, 0.0027383400004709806, -0.003985091427630559, -0.1324546560184461, -0.004903971408416303, -0.01626446187211364], [0.06438734186923577, 0.012206624449056035, 0.020830332982626456, 0.2861781659847445, 0.005180729929958063, 0.014992638297271514], [-0.14756389234746173, -0.02985615366642461, -0.015665208152529194, -0.2922294340069343, -0.0009107323419222406, 0.10945557160993599], [-0.11958252297771384, 0.014840352995620526, -0.009885515076693885, -0.07852667627626454, -0.0035829047186429356, -0.12084185545697793], [-0.28976232961023646, 0.0034755065268355853, -0.006309252095684938, -0.022981062749936835, -0.0017723962637873252, -0.08932379914052098], [-0.3282822556157157, -0.005698584408692618, -0.01292893830992579, 0.009500307186701484, -0.010695733013702887, -0.029130663381084977], [0.06986035818755922, 0.02505171630107781, 0.012656819146331695, 0.3540283225342187, 0.008482589095739308, -0.0010254815934337574], [0.3969991002468018, 0.009220727079894124, 0.0206905231881654, 0.010918679240388737, 0.012127743455535633, 0.13465560774159294], [-0.3391333989349729, -0.017987558450250952, -0.012850855292529289, 0.14675763978637826, -0.01174538851800381, -0.145169047572001], [-0.17604374094453853, -0.051041904835927084, -0.01201611242709714, 0.05181091509893931, -0.0293441083628786, -0.11279433424278336], [-0.33236039550578245, -0.018886552068031297, -0.010107960815666641, 0.13528785048137118, -0.006290242718132282, -0.16632812653695703], [0.46996305841597297, 0.0028182905064843443, 0.00013678218159369018, -0.040443013296000245, -0.0031964072763303653, 0.10800520721719663], [0.22118866664623607, 0.007394072663469489, 0.0046729650067475976, 0.1997228158015224, -0.00328999273134684, 0.15020335667134307], [-0.26592247098835303, 0.008264654872298086, -0.01857001913989918, -0.0783092755042724, -0.00982515378276709, 0.1476194913055503], [0.3449259186166996, -0.041924195311239684, -0.018910523853570442, -0.422801120846938, 0.001190201254197844, -0.07982623224010484], [-0.2214319944764344, -0.0024533294988981877, -0.0037306136881540683, -0.1257305277479552, 0.00270760629755392, -0.06153447421944518], [-0.12716006600336255, -0.06681061841223168, -0.0003788575307815719, 0.11793254570090128, -0.015027910283885846, -0.10720259663159], [0.4191911387242027, -0.005698036363369287, 0.0301264562623807, 0.15943531671959058, 0.006789814100366134, -0.03326802277650588], [0.2748431760566262, -0.07317776270107537, 0.0012108123568071757, -0.28616399216936284, -0.01612772916097177, -0.07388283771535793], [0.4821688383499336, 0.012623602135666946, 0.025456769619911064, 0.08505215650395534, 0.0066960677379069205, -0.05057552958546926], [-0.10029887056862846, 0.013108755944598592, 0.00820971098759087, 0.2554433969430715, 0.02342386601863597, 0.321943245080128], [-0.2566158161497242, -0.024891242446354333, -0.006895355934465216, -0.03459416906433282, -0.008591817024163685, -0.049494158904770526], [0.1558644458726832, 0.01500083377941859, 0.003788820451780003, 0.22722987535871117, 0.017266147005743814, 0.16617654419832936], [0.45672409179700096, 0.019278553731166364, 0.0014627930547499906, 0.12645827309347196, 0.020596536936250735, -0.05995263991698834], [-0.29321493844998936, 0.009459762588651333, -0.005648034522743683, -0.008601740842466217, -0.002513481569917211, -0.10579283157135087], [-0.2351907244319979, 0.009515521808217771, -0.003616828303470381, -0.14471555282462173, -0.01538128565866636, 0.020132202743871973], [-0.33105089635001733, -0.017130783362603895, -0.0033864742649288436, 0.11182145408855086, -0.008457525514991808, -0.14979980445280935], [-0.17884320341713178, 0.010822941253263847, 0.04435947509172657, 0.20215560565541912, 0.007102742390177511, 0.3247291056932103], [-0.09296971754398081, 0.015253710505020003, -0.002248180193448214, -0.24557737202174565, 0.010588759083543647, -0.06513719982938868], [-0.2524105150445757, -0.1021829444804462, -0.011081596028073327, 0.04281125690418205, -0.01647584106835075, 0.1819307174114984], [0.3406535956797019, 0.0635234079380919, 0.03700649720220417, -0.22317987318845942, 0.025769731526371558, -0.014439727234279345], [0.4278740717970442, 0.004061698444153152, 0.010698584350419248, -0.14809424608433075, 0.020244255558386767, 0.032988381032365025], [-0.21492794796732506, 0.011476932995605916, 0.03786642474338746, -0.19530390068130543, -0.0015766664569617783, 0.017125157366597935], [-0.28184146377130603, -0.0015141566785213398, 0.00822092728764509, 0.024181362905332668, -0.02411719572213502, -0.12370569851081051], [0.05057912248683455, 0.027517320103548654, 0.0010758496801616811, 0.21246474315569194, 0.016591814642421807, 0.24062722836271383], [0.5794803272826032, 0.01412773918270588, -0.0009581254855876486, -0.03524974951943458, 0.01691081750905463, -0.025758151826489127], [-0.26290148500161975, -0.01040734346680594, -0.00623998048284145, -0.06609264563800711, 0.00748285534503467, -0.06829383461819462], [-0.1926362020680822, -0.03289118641325601, -0.010806615490937064, -0.24787401252234667, -0.0018795045358579086, 0.11836580877110593], [-0.30820524399399196, -0.010762019354885095, -0.01279408785139712, 0.11105720336630671, 0.002649879802717983, -0.16587068867870838], [0.2208974292262427, -0.025758914383121948, -0.008517128780537505, -0.465474252469254, 0.003401308016062143, 0.030944891723939747], [-0.2361666415882404, -0.02336977900893649, -0.007761612876826193, -0.11408694217580245, 0.004074303380404668, -0.01524807773059977], [0.45713831059535065, -0.007901337932934285, 0.014002293800578654, 0.13681489443787587, 0.006868994954868797, -0.0395293896219756], [-0.2804574337951244, -0.0399136505782747, -0.01627618947269522, 0.11182850447232301, -0.01975832528328751, 0.06788965629370673], [-0.16481820882336815, 0.017878360827198413, 0.007153902871617394, 0.3084133358952032, -0.02503673108169269, 0.34264076888246425], [-0.11150191242921163, -0.014968626250659952, -0.010443543769540353, -0.27556166082711686, 0.001213555691545178, 0.013213854251649003], [-0.21529648856793934, 0.0005944554016042089, -0.005940585910635997, -0.15059851219453696, -0.020958793774715975, -0.022277329855737223], [0.39984068084185137, 0.011758692812236064, -0.012135433707936608, 0.18019103947250364, -0.02664409528711978, -0.03195243535808662], [-0.34488523900908347, -0.013098827676678767, -0.0031886744332009402, 0.1504428419065411, -0.005730567705188295, -0.15034767549330383], [-0.02396119307912859, 0.00956983596863668, -0.01053921051456502, -0.2841687370796491, -0.004022966358292998, -0.07385498383896205], [0.1325683038279765, 0.011460214541864806, -0.04113407009305512, -0.13541119082341643, -0.00783210360074276, -0.13780154032605652], [0.47376316413566927, 0.005154173312149285, 0.025705233256896182, 0.1215009170799073, 0.005779790505214629, -0.06828494495650457], [-0.1876728957691417, -0.013103576381866681, -0.005581402595652363, -0.15385196858994382, 0.000921530830000521, -0.05288502082672935], [-0.04855527432157849, 0.0913388221390343, 0.01788327186497069, 0.22148753008679964, 0.046636334105056645, -0.06747404381478612], [-0.1805003503024097, -0.011557639779107677, -0.005169202222976826, -0.17013077741702062, 7.04322826365628e-05, -0.047385795894454616], [0.2945375884005253, -0.010461521722978834, -0.00019032855828823614, 0.13460286714603, 0.0037145530289661856, 0.15229017503907635], [-0.2198915001600597, -0.016239147240955494, -0.009858051671865391, -0.26056724867122333, -9.218653158270308e-05, 0.10037828537035108], [-0.27195648191151667, 0.009865569171559027, -0.005133693676948401, -0.08114257811565485, 0.005962151179893919, -0.06851829998066583], [-0.18924075178689403, 0.01607766236430013, 0.0011657058945612083, 0.2630160853969557, 0.029185656646181415, 0.3374734986277501], [-0.25373485230139475, -0.03099819286290806, -0.014638788883639588, 0.04641335224146239, 0.0005586009223040602, -0.13632721294568936], [0.2977289227899628, 0.014083633676907124, 0.006512663200881125, 0.1008126060834438, 0.02336643746227885, -0.04464056627576064], [-0.20609094912820164, -0.041155740788155794, -0.0058189748190724845, 0.13430337095806189, -0.03296107562386797, -0.13604153902367228], [-0.190821942165081, 0.011098626958023203, -0.005941462777441116, -0.176378549956978, -0.0004299618948206277, -0.04845004349703503], [-0.2687035372587718, -0.01563181540966299, -0.007949919364596904, 0.023626062650325096, -0.01870858304318793, -0.1200555409074372], [-0.16140349892781547, -0.023875816693283536, 0.0056306379559399496, -0.28278814231809557, -0.00920232805334421, 0.09007219768198801], [-0.296183142931009, -0.02256115036968698, -0.024121128115690383, 0.16230898518378495, -0.010744474165834227, 0.017081203864657283], [0.024012217186984687, -0.001554463612245858, 0.02100371681729109, 0.28699123974309126, 0.029595619973527137, -0.04704386949263646], [-0.16854393093224357, 0.005122273707598054, 0.012094735063577021, 0.26384583324471045, 0.041271733590482036, 0.32392887913539575], [-0.16877608990074008, 0.021178116359142754, -0.004530008819588552, 0.29638281250063875, 0.009220578740995215, 0.36464851608347043], [0.1915640432950715, -0.011932121521250411, -0.0014239128655397627, 0.28155417097965235, 0.008821072763013044, 0.06914817592048055], [0.5010516966031537, -0.039406015993380575, 0.008672824970536485, 0.038404585682187284, 0.019198594657024576, -0.034737876395715184], [-0.25179402815568297, 0.0010598943064850039, -0.005802292159741432, 0.015266955834108404, -0.01885465675759134, -0.14032471660499285], [-0.31214033727734336, 0.01746268815980247, -0.010192135957991599, 0.08390622370657268, -0.03963109033642593, -0.131793585771423], [-0.2507390255672676, 0.01683318955649052, -0.01666359931086677, 0.014533927026821481, -0.00427529672031287, -0.005413365804355762], [0.03461480047958462, 0.04951850650732323, 0.007306619642427631, 0.29190974606082476, 0.03516032828156925, -0.034369351275980474], [0.3037884273888498, -0.05118439330449684, -0.013425588289081958, -0.4433113954013229, -0.006651786513615275, -0.08369018451525698], [-0.322308911491215, -0.016251398053666648, -0.011545025404153031, 0.17703569171415912, -0.00461326769282301, -0.15706185097706313], [-0.28611047428730413, 0.015730607188412737, -0.006715768663840436, 0.01751606095041009, -0.0038331653154278166, -0.1350105932055833], [-0.06614207150864904, 0.005630202869069285, -0.01097196143864075, -0.33859891399406444, -0.0014560650020393102, 0.02575186290454966], [-0.23675716614105177, 0.008177288297116789, -0.011271683885863994, -0.2924050021447122, -0.005042214187934372, 0.1338647858143818], [0.3490955769024286, -0.004409040150944475, -0.014111403970529733, -0.3925589618619343, -0.059178826761009655, -0.08317734415801326], [0.47032839584823727, 0.023200272098692402, 0.002084729172523262, -0.187599946953969, 0.01011451586379821, -0.030514593013412392], [0.4296913012580044, 0.010116142294532924, 0.010363921083836663, 0.051539930986635334, 0.025432023638257417, 0.03076668073873278], [-0.27879264082840916, -0.0013521596844066055, -0.0035306358846875284, 0.22996784047239346, -0.017090809884093502, 0.023732026265052043], [-0.13207461435722592, 0.03523232325361392, 0.02063099382477107, 0.22696736042092452, 0.019491670336487747, 0.36117777878693663], [-0.29362117215743294, 0.0171927040563917, 0.008114486170351945, 0.02557705407354463, -0.0005074385374465096, -0.11617896693874231], [-0.22712495427576118, 0.010497908302360488, -0.006097899784471513, -0.13190064601149787, -0.003732047189656035, -0.054315694374305876], [-0.2551282972023444, 0.00400035522234834, -0.0036029744976496846, -0.15241827307238018, 0.0007450645635997915, 0.0031768700844647625], [-0.24659369706004136, 0.012345815563611041, -0.0024013453449289166, -0.19631364102295318, -0.008330037915205538, 0.05192438972193156], [-0.24756716617592975, 0.002669306786393082, 0.001123660379709133, -0.15107000547568733, -0.004958814567290037, 0.011724923814710108], [-0.16408453342075366, 0.0027392691427661764, 0.030105023925648698, -0.13583137235838394, -0.0012827060320985332, -0.09573568125718011], [0.39693629635084005, -0.010935173540134437, -0.016026361516045615, 0.04179503833477038, 0.01783623122281561, -0.022985713391931097], [-0.15546744676504792, 0.009909670469731333, -0.010572222844749783, -0.14868158401831294, 0.0025419217517907225, -0.09455843383150743], [-0.21401575015151397, 0.002592258733838324, -0.004731549467705755, -0.11018179901698845, -0.019303425510282152, -0.06883698948930837], [0.34590411288581835, -0.004845735031632656, 0.03310494126208879, 0.14037989570435377, 0.03900564265316342, -0.034389939724873066], [-0.28245828094732894, 0.0051809421019731626, -0.00506987125385776, -0.0560826152487303, 0.002663646605703123, -0.07446271014664797], [-0.20291819979258238, -0.007693145134700604, 0.0007711877237313734, -0.2046904791472295, -0.006524772985004679, 0.016965409335785355], [0.5162555698190758, -0.0030380902556903424, 0.0066720846106329895, 0.08800555797726599, 0.010917311090191529, -0.06896303930208289], [-0.26355471779878936, 0.002598060501518882, -0.005622031718843806, -0.014321520402584956, -0.022068555395016117, -0.09373007872369837], [-0.19279100712921646, 0.006140594260804777, -0.012497825404144739, -0.3041029247999072, 0.000713903204267213, 0.1012936037417132], [-0.2198995275488306, -0.008362616412735387, -0.0036172780402498445, -0.09885530392688512, -0.021956536005666178, -0.05386302377991757], [-0.14264207095860457, -0.09084947548640884, -0.017880736141547445, 0.10994300897108318, -0.008927588579761312, -0.07658234415396639], [-0.04086175612607632, -0.0029357372272134715, -0.007056820064874964, -0.24311616212773954, -0.03559174085110132, -0.07166503850495569], [0.37659414523001733, 0.01215983554872093, -0.014678846251370904, -0.2676489357667758, -0.017629163024970026, -0.10667275002133807], [0.5080830888240997, 0.02016886024739713, 0.008814100404951918, -0.012997432434056445, 0.014871763693244559, -0.011280380735640538], [-0.3162097714155889, -0.02302272463556074, -0.006134136184103964, 0.08790314972532995, -0.0031066454877450287, -0.14231605674128747], [-0.3584106506091089, -0.01026813356490779, -0.0024027298037832376, 0.18634542659548678, -0.016614274170292605, -0.09281900352676249], [0.07818305732454126, 0.014929962974591806, 0.01306030201607227, 0.29696314073182534, 0.014450137797897323, -0.03515634088537665], [0.1100657145073529, 0.006525556248514058, -0.0030926731909333817, 0.34759738384725675, 0.007302331959112266, -0.04413712289511287], [-0.2162124425929291, -0.02836874488911937, -0.007838522509062525, 0.013360648468410606, 0.005111363953360435, -0.1362613500497077], [-0.2424847829406072, -0.011906852417115952, -0.004894574165077378, -0.10425790424290499, 0.0036290251831398264, -0.054758244750767716], [0.31150409963645426, -0.010021782102483709, 0.0019593659007033485, 0.1609103384177989, -0.03309991781746936, 0.14240789596499667], [-0.26102552449668576, 0.010946887356605393, -0.0062539089825864895, -0.02999749617748294, -0.007906206991097104, -0.10071259424616721], [-0.1570483209305691, 0.008597384874531015, 0.008004928703575212, 0.21514028697845874, -2.0440081531174865e-05, 0.40803033338786227], [-0.23158110834030393, 0.011930448129379843, -0.004891079451775539, -0.12105855911664855, -0.0036485608204890915, -0.06167447373349568], [0.5281977687590597, -0.011034233309390247, 0.0036542073664477138, 0.1017383310042675, 0.009503679901069876, -0.06477475372145766], [-0.28000295200449743, -0.011609187395829102, -0.008620045415515762, -0.024916781634796603, -0.008226080926012928, -0.03847685738525232], [-0.31356921038533764, -0.02064323177796064, -0.014342736559833875, 0.008317728658206076, -0.00820307874084544, -0.03525066167041888], [-0.24957965193237744, 0.0027433746852475304, -0.007754094666842178, -0.016901174548338175, -0.016320106596047147, -0.10844501360830881], [-0.28546859662917, 0.0017603548167615553, -0.007541557376727888, -0.04529869550573427, 0.0002973610995568357, -0.07842219973802014], [-0.16741329865303856, -0.02346546683055564, 0.03702843835969572, 0.2350030019979512, 0.027489254198677742, 0.3665121185463164], [-0.22462714230992106, 0.0014579301559850208, -0.01994205124920121, -0.23137078615896442, 0.015312506495576519, 0.12418092452819969], [0.5160153730735134, -0.01609972467953241, -0.004827592484971772, -0.22883673591792672, 0.014951355505545098, 0.056933514979557086], [-0.06982246550859589, -0.009708306991840523, -0.0014033553065990346, -0.2740076571597222, -0.0004677144380381107, -0.05926383392853772], [-0.09594801237094888, 0.030640334383456733, 0.00532046018890855, 0.1724944921592291, 0.005371485885837009, 0.33200236294556873], [-0.2761319529752149, 0.009468493440997455, -0.006537322816051212, -0.07967330018025831, 0.003813104747779878, -0.05507664126487225], [0.22213898900547063, -0.02254780698912067, 0.037692993120320796, -0.445036581412659, 0.01755249297065485, -0.032800800980382884], [0.5424572175630135, 0.007767737855411038, 0.001656477479120868, 0.07660827233639882, 0.003479137226036847, -0.05443629344037669], [0.2523589682167059, -0.024410389696765566, 0.01645308047190767, 0.06648688209540236, 0.0059661724385731895, 0.17365052456941488], [0.4764037417996971, 0.041238491899271375, 0.021701228143837594, -0.022461834329986618, 0.028727073204539846, -0.01514685739938926], [0.5291289705774563, -0.03683071976757988, 0.003891615293232565, 0.07567558350348498, 0.0035021879985360064, -0.0436686765661732], [-0.0894639135483119, -0.01584474383690573, -0.03386276237404069, 0.2324995837493653, -0.011069019346235314, -0.155519959939686], [-0.03105551153090894, 0.007597468807485988, 0.0077918784212849665, 0.3450761923791584, 0.02187455894622749, -0.05898135952918288], [0.31869962949265285, 0.014641515193256176, -0.024172605657655825, 0.08336403881761152, 0.009818748814415421, 0.13416581619686066], [-0.019067654680949207, -0.017617353100585802, -0.011637694212763944, -0.06653655913814381, -0.08335231205185431, -0.10994560368645057], [0.28067332074426915, -0.006584449407248734, 0.012720873986674477, 0.1328471106873019, -0.04441150017034107, 0.16212297749267826], [0.47527830909826796, -0.010189184494571638, -0.014264386308216682, 0.06908250608709461, 0.013535374712394571, 0.0257411904288387], [-0.16717101862812314, 0.04215972244632619, 0.03887701288164717, -0.04018923793724792, 0.024992391725999562, 0.2664833282848463], [-0.25971862827884606, -0.023092417433387705, -0.004504011323226307, 0.008313035725499495, 0.0019154197004892953, -0.1245033983905289], [0.22135605921440638, -0.038506530436167766, 0.005759621074313448, -0.4022000110994506, -0.008841814425254179, -0.09712161004213701], [-0.3154357023274354, -0.018979377572127584, -0.017265749611322812, 0.005265055178380757, -0.0007572465506573762, -0.01576420133905966], [-0.1655394197212269, -0.012094074359761944, -0.012617171786266099, 0.06434345334718208, -0.04819760918457644, -0.1451872758010198], [0.4691043471621879, 0.04311409970561082, 0.02193252143708577, -0.02713645377394767, 0.021709077078995085, -0.01669515449149771], [0.517822831516984, 0.009615959854855383, 0.0387764559158175, 0.01853929257961981, 0.022765986741791647, -0.06900446600301013], [-0.21332023945029135, 0.001454340740468709, -0.007749926929777818, -0.2042871382605489, -0.0018702590816248176, 0.024825729984575433], [-0.22396107551954472, 0.004664558850668574, -0.009295454210485582, -0.17567302215355085, -0.0006010371499528869, 0.001026030182864961], [0.2496594401940073, -0.007901025436855644, 0.03920987831494366, 0.13285312322060638, 0.016979945230420675, 0.14660863847687783], [-0.16693536953080035, -0.018112037641311158, -0.0059260927529134555, -0.14887486627929014, -0.010397663556731107, -0.0644273035722884], [-0.18551013301660463, 0.010706849988397497, -0.005485619314191559, -0.1767859833123289, 0.004030978278991428, -0.05862942595759754], [-0.22385062742002015, 0.008215880442700866, -0.0060966897618315585, -0.2802999095134207, -0.0029773885508468014, 0.10473369339555068], [0.5546230886448748, 0.015178980767540772, 0.004205923244933413, -0.09692209794120304, 0.01162969039148757, 0.03447417679712296], [0.3443407313679959, 0.009509259838772021, -0.004126037020825359, 0.1684813735811339, 0.01265836117064056, 0.03853440630037664], [-0.11444790252327033, -0.004551237368302537, -0.00926317642093548, -0.006034397122753943, -0.009849500019690927, -0.12865647103404604], [-0.17121801107176643, 9.231048111477679e-05, -0.006167689852659098, -0.17643042748164803, 0.0038709071066249223, -0.060653755848333886], [-0.2104314338816189, -0.0015605003283938887, -0.0038033076875215043, -0.13821568149626934, 0.0005346298998102676, -0.04897481761711682], [-0.3311858844930045, 0.010026944157264812, -0.014056696910264689, 0.13489716966237222, -0.0022226382424265167, -0.17293598800380763], [-0.08829496498120917, 0.003064223105853894, -0.014784983359639656, 0.17700864649295953, -0.01299411035858446, -0.13617809661366714], [-0.14111494434171692, 0.020902804505705565, -0.004594390875266653, -0.21930598814400173, 0.0017026696420960716, -0.04137555308566715], [-0.1559432249424556, -0.0016581195328485706, -0.004326810335870386, 0.29714107606458356, 0.030917778324187403, 0.34908847409186755], [-0.2235102216751061, -0.0013142339445041554, -0.002875033491824873, -0.05139920895983402, -0.04513863262697064, 0.2074563584760177], [0.466485219372298, -0.01647964150686743, 0.011068394507109232, 0.13830687026248856, 0.00932034493895309, -0.04254118757398478], [-0.21904971644754487, 0.019907203374059396, -0.00393064005815191, -0.07039072479042476, 0.0035189097885982338, -0.10027248284692822], [0.10931246050859383, 0.014272401674940847, 0.014297793763447702, 0.37929301974761337, 0.018913293532312067, -0.045095819906600644], [0.6015861511731354, 0.0037333584107511974, 0.006943593287194449, -0.11554426915969326, 0.026302707488518127, -0.009837731676099193], [-0.23355091927694469, 0.011548523273954756, -0.008434439316136695, -0.06679434225929269, 0.0017678705150375906, -0.09492081058367739], [0.4168866112125351, 0.0017184978747790593, -5.104641980399422e-05, 0.16242314204207503, 0.003485097969429939, -0.045009291926328986], [-0.1779820184924043, -0.005508194530173191, -0.004730894720307639, -0.22139263765623554, 0.007857569087788915, 0.0019161763113302347], [-0.2674864625865125, -0.01401320649739311, -0.006267207636288413, -0.006714922223242997, -0.005730096128561793, -0.10279477159466748], [0.5426602645689332, 0.02325198922234802, -0.010452127415704886, -0.010893903198719228, 0.008950916354385921, -0.045865076039185375], [-0.27556813477127734, -0.027935137861475494, 0.017818603223725443, 0.1043981322416009, -0.024844776723285406, -0.025107495633098716], [-0.1790569580647609, 0.02253475379396034, -0.008779647743760916, 0.28111565698346397, 0.014898566738637566, 0.3779294210095425], [0.16326562261712615, 0.04153505289376218, 0.012174834697783691, 0.31330804512985144, 0.0138360470785855, -0.0500704306878368], [-0.12766409274420595, -0.09455818137609924, -0.036936204367613065, 0.00387243203848155, -0.008173718188715252, -0.003970280954552649], [-0.3046890347295657, 0.01135820318690487, -0.00958485713229269, 0.02116388243726147, -0.004293869491057437, -0.11919416006763824], [-0.27785676060496317, -0.0004024358797609058, -0.027726069601733874, -0.1970272634207961, -0.002424956000224115, 0.16870083210927653], [0.3361053193314749, 0.01261003201437388, -0.0002625754754228412, 0.07602611155965888, 0.009545747632675362, 0.1467186982705715], [0.31604411835258134, -0.026532611124469134, 0.004547009231039012, 0.19500109643643818, 0.001361748000701441, 0.052196972437040724], [0.07063255252627192, -0.017710326578397464, -0.0009727532891124899, -0.2752058348894415, -0.0536588352194082, -0.09167480254991257], [0.08039635953228325, 0.02695617055298563, 0.005605014983982549, -0.10370984938121058, 0.02456628028119638, 0.19141745260218967], [-0.29109053799192613, -0.003064137180197722, -0.0032000876241479632, -0.009512120136579068, 0.0286304930281749, -0.10727027676199001], [-0.32297846337123776, 0.0019523716213943989, -0.002699137050525483, 0.10809834192912386, -0.01480272622583884, -0.16098208184595417], [-0.333261383881306, -0.008332838095395182, -0.01009509148519495, 0.025365692923570195, 0.005081566825613823, -0.02175866057300435], [0.4585638441521373, -0.006366482428027324, 0.039026037456081256, 0.11430023878069966, 0.005723852687503211, -0.03192082398173025], [0.26591727181647595, 0.011111110181478936, -0.000947965481336162, 0.1812236685146006, 0.007550842699873874, 0.11624951671335175], [0.48435413634483104, 0.015222771092325437, -0.0015632455094851665, 0.10825600337697902, 0.015241463232304082, -0.050952319013145515], [0.18852968061097644, -0.027847797832988696, -0.01088193744134493, -0.3224870092587341, -0.001221004154469662, 0.043399417282908415], [0.34925015975731777, -0.006208434015020002, 0.03665816305430039, 0.18234449736939845, 0.00147721935355247, -0.008956843614788], [-0.3161433504292783, -0.0006993208430170059, -0.015949543390386497, 0.1402546039947873, -0.012676902573151022, -0.1817447234459635], [-0.28958096512619946, -0.0006574044384451363, -0.006634424226395169, 0.023382871206081658, -0.0005125957735781331, -0.1359649326218555], [-0.14628818252916712, 0.0001915684733963668, -0.0049142288185433204, -0.20903400697942903, 0.0037810615829087226, -0.05215954506249946], [-0.1333498929579384, -0.025323192645383803, -0.00731589004228753, -0.3456749039858861, 0.0010632032701345228, 0.11274801172103477], [0.15422281967867688, -0.006538516007146197, -0.01670773670491788, 0.06284935816389291, -0.10504909230842946, -0.08300373758398087], [-0.10646434365189394, 0.009033563984037487, -0.01443932832023647, -0.2676580398601979, -0.00528502837584545, 0.029020277673410293], [-0.17252334895835517, 0.007059652611809558, -0.004299219343378157, -0.16243817908429178, -0.008074536030773571, -0.06439770252834386], [-0.19005919185515271, 0.0105512420705554, -0.005627589141532295, -0.15849363620744894, 5.3672889590116086e-05, -0.06359783108934544], [0.006803989878828765, -0.023754040068195013, -0.012395079747063616, -0.17462055506733984, 0.015634644162907933, -0.10950895915913816], [-0.14329187486541647, -0.04398582109755732, -0.009245213119845693, -0.007314032281546336, -0.027003913452441325, -0.12250079630984403], [-0.2985499651753061, -0.00755699749399446, 0.016217029319412945, 0.15664677843370578, -0.008531899411570466, -0.16939846807354156], [0.42812533567307426, 0.010845998917153894, 0.0017647932193818317, 0.10238587948866108, 0.008973517115366123, 0.01955257082445827], [-0.21867953365642892, 0.010473879323564443, -0.003282719764539627, -0.13212550296730763, 0.0035435106984595083, -0.06710296696708155], [-0.2942490972433097, -0.013886315700574664, -0.004750217651493305, 0.05397041652796321, 0.012616584392382271, -0.15504137032496715], [-0.2998839365963336, -0.030717467715748823, -0.003343506066855528, 0.020413362966255393, -0.009698620191773453, -0.018734832395544108], [0.41930454785551313, 0.020726317857575793, 0.0029688305344903547, 0.17078168316528128, 0.01996811384914003, -0.06204075553409212], [0.2795572127710816, -0.06570066677245567, -0.055165051111284634, 0.0176724548970056, -0.10539763694704257, -0.08698867214466338], [0.49121524273477857, 0.007421949390466123, 0.003189634415624688, 0.12322105010478873, 0.01113423214200371, -0.06962267804762472], [-0.19292055198659366, 0.02371539900760313, 0.015905412701664304, 0.2865841995522677, 0.0185858384865394, 0.23724751797350627], [-0.274652617212508, -0.019956948324134853, -0.007573009206251762, 0.022836407399766563, 0.0004092214912658133, -0.11923638748147043], [0.5280543687554455, 0.009588471628369549, 0.005360755205829717, 0.04379445629267186, 0.01494154751827558, -0.06297353879453373], [-0.1732542504142191, 0.01026296505458933, 0.0035421274771375725, 0.26634536311768475, 0.017249311421251665, 0.3458375459019102], [-0.23619814530224276, -0.004854999432739679, -0.005821742050225729, -0.1063600755373304, -0.0010080357155087363, -0.0602342568639129], [0.08976868165368473, 0.007511492958930467, -0.008800010135717856, -0.3929922726801147, -0.0030415531232917442, -0.04815401266616819], [-0.07760677138778553, -0.007917613948039337, -0.005920435076135619, -0.23838294054071582, -0.01064599702066471, -0.06553290869332667], [-0.19878546555205148, 0.006348389845495645, -0.004800628758220826, -0.12986752372290752, -0.017159741524542145, -0.06332503028777324], [-0.16358977398831395, 0.020035010007126066, -0.0003817716542866473, 0.27639255083260705, 0.026501947818306518, 0.36942822746074866], [-0.26458577000039646, 0.012256125573318408, -0.005176312872307268, -0.08242872602987837, 0.0014364778341943195, -0.06954814371127874], [0.17813642808721056, 0.019236816118428517, 0.0014001407735203676, 0.15632982912551227, 0.014108923639140502, 0.17137841781174384], [-0.25509715817761297, -0.02107392964480129, -0.009270440494788182, -0.2303322920794545, 0.004261876674300554, 0.1447214289419735], [0.2584431565195559, 0.012707371922063261, 0.036653015279751566, -0.07378147307847194, 0.032621310743401986, 0.183848959883541], [-0.22426598399596895, 0.009563677768360758, -0.00957825623956165, -0.2915227296946011, -0.0031697619273013604, 0.12564322850767531], [-0.09190213337076007, 0.03387780147942954, 0.01732769962730052, 0.0278099809310423, 0.016601591250143603, 0.2713409668246806], [0.5602500930068198, 0.015415330421938874, 0.002485235888400839, 0.018975023362128585, 0.007848003551817731, -0.034811826866673445], [-0.27794226386901144, 0.011043141075319765, -0.008605902480489295, 0.057716392574919395, -0.02723164982155727, -0.15376189434992943], [0.40022025544982476, 0.0153560177160897, 0.033549268866828096, 0.13150369050691024, 0.01463386573008283, -0.03086267681379933], [-0.28246675854600534, -0.019058986314924697, 0.044921974264634866, -0.022838314985706303, -0.0019286456667721745, -0.1031359354178928], [-0.19945120375969116, 0.0021758438719917212, -0.000732127689837268, -0.14766294552340054, 0.0011409833542133157, -0.06285816930089458], [-0.13282778957469488, 0.027667150227949956, 0.017002077736709496, 0.25643810780416487, 0.02226318983778981, -0.0711383494661045], [-0.1813435151187119, 0.011612451426435251, 0.0038519934539351145, 0.0702979477181565, 0.017789182525337743, 0.3473951285291227], [-0.17829614322713738, 0.027356279861930646, 0.010660281171839193, 0.16693340134582346, 0.015773857740693988, 0.34877016741123723], [0.0028692146645281236, 0.013203208220766347, -0.003211625923006315, 0.3554775613406223, 0.011082682102280307, 0.17870884754998756], [0.0016507996848278265, 0.024898361051680504, 0.0005452091892908764, 0.3110315396043162, 0.0036622778904562697, 0.19681228876990048], [-0.15559759516767477, 0.031905491155271845, 0.005309645123799742, 0.24658304912018075, 0.011910109277294446, 0.3883588243006494], [-0.08390174470065331, 0.042171166473426454, 0.003840281163654616, 0.14559292941882565, -0.00929433701486762, 0.3052944771045578], [0.15925069603436567, 0.01515800487198013, 0.011700666365119905, 0.12824200885599843, 0.0027538217338332442, 0.208459564043464], [0.4673166973668152, 0.01651070862206745, 0.0034230238753798245, 0.11697265965387961, 0.009931837412470675, -0.04263381581950213], [-0.2757335933880311, -0.007907002436331918, -0.00736833790087404, 0.05343685885490301, -0.010672644981093242, -0.15375030842494186], [0.19430020603095166, 0.017419479164465606, 3.2769931775766798e-06, 0.22277789865151806, 0.008414937394570998, 0.1317719795430948], [-0.15286730962073838, 0.00788320207604431, 0.0052513478515435015, 0.3620600272338491, -0.008798754944210598, 0.30202685081202807], [-0.24852151346164386, -0.032166103965783496, -0.02056464107155285, 0.02076425075814741, -0.028588376686006035, -0.023061994499200624], [-0.18411777981552513, 0.0028741963660802988, 0.01979677660784918, -0.17920191503976968, 8.050831716421236e-05, -0.0706051197691322], [-0.22182917413007194, 0.004292764367225303, -0.006670098869181297, -0.29218866418269873, 0.007609127855462346, 0.13975323450174593], [-0.23222888743631853, 0.024423328432839352, 0.01881463993142952, 0.3171408998292401, 0.009502602328241811, 0.0992657141728628], [-0.2991716648386406, -0.0020652803123169567, -0.0043629542731204885, 0.10021395826083376, -0.031051178316116722, -0.15909505739138813], [-0.2822675430026574, -0.009088967225452056, -0.008020325873451806, 0.05074019630354195, -0.027303477383451326, -0.12635821615186088], [-0.16469793943006386, 0.006731193671749973, 0.0031045122701684924, 0.04936597748883518, 0.04543314373499727, 0.32307579083574056], [-0.21949405365973626, 0.008523760155627025, 0.005038029165016014, -0.3142093781352436, -0.00249804875422957, 0.15268064360951722], [0.12137743126543737, 0.028401617068946613, 0.013558099593657787, 0.38371918681388906, 0.016610314626824246, -0.08297976764832694], [0.410016061480312, 0.06890667818551836, 0.06313395491088805, -0.14092537644058434, 0.018110897048245412, -0.02161396121612921], [0.36158554255652264, 0.016300364864619266, -0.0034943361720826925, 0.1697243781548315, -0.04187876910148709, 0.03348569333189917], [-0.04708594362768327, -0.0006937639319203395, 0.0016780785161735784, -0.3690453740662212, 0.0007725147779176068, 0.11269520261744578], [0.4672115397140893, -0.007346590638601773, -0.0014044489037141064, 0.0013020805829814166, 0.009696178161906686, 0.07782624108333472], [-0.2085883888505693, 0.011535913551786665, -0.018893593873746152, 0.17876949700407968, 0.0044385735224269, -0.05111059378328354], [-0.17420277405317025, 0.017273694856353376, 0.019512962694724047, 0.015358390533461283, 0.0595470632686919, 0.29699426638383586], [-0.29074865829902746, 0.0097452204579832, -0.010824218552089091, -0.06279797640298454, 0.0005992307824130019, -0.055527883700581426], [0.4837914032985825, 0.017936922227661863, -0.005152005355589328, 0.05163334598868064, 0.013191036658523848, 0.010625239211125573], [0.39281440888097235, 0.07317158881175988, 0.011929058534064348, -0.21780452132416156, 0.009026719733544226, -0.001902468311396065], [-0.20749734216220322, -0.004058754418082979, -0.00869025396157848, -0.19982275380935458, -0.0075494801172571535, 0.022218300780530647], [0.06085312787904711, 0.012109545625558818, 0.017950875672822456, 0.26700324480588933, 0.023159088615770825, -0.06006252762073395], [-0.26212187166952133, -0.012986746219269447, -0.004645510865570852, -0.0682923437733834, 0.002379511276486446, -0.06900637208207398], [0.4852401446291288, -0.018262603407102496, -0.00931846759323012, 0.1511401630542067, 0.0039208527673714405, -0.06110175611704331], [-0.3219661197989234, -0.006100082663532833, -0.009678954788499977, 0.12991744349474954, -0.018410750155578567, -0.14662704629229703], [0.28800917690545896, 0.006601358125031221, 0.013588982972743056, 0.16146079410721256, -0.03524604336651875, 0.12141239792273956], [-0.15141449133485776, -5.190733686316322e-05, -0.0021665138137769717, -0.18960522440703492, -0.003810597548464381, -0.06262459889233642], [-0.2111885437360791, 0.012498004053600662, -0.006571716614746506, -0.14254242183375498, 0.005249533137795571, -0.06795152167348258], [0.3740939242787138, 0.028187840741762577, -0.0036450771322482675, 0.19736623642586062, 0.013432977177120161, -0.03401331567885265], [-0.3128477386982284, -0.006474098876825532, -0.005728358139178732, 0.07627690244319911, -0.002224626952953678, -0.13885662458627113], [-0.292458248108901, -0.006265829768102203, -0.008707736531722748, 0.199736858382592, -0.025664617417831773, -0.17450616470677416], [-0.2338085775007671, -0.005767770072884994, -0.007515614155540851, -0.16333532065089587, -0.0013626363530203603, 0.007449918733109063], [0.2130750175952568, -0.006059199024813914, 0.004435930648341508, 0.22695976845274302, 0.0043033076836789304, 0.13927850797812644], [0.28939282191588866, 0.00514031333259119, 0.0010564536962144723, 0.14906476019276513, 0.0038590163430382773, 0.12459107896394608], [0.08482400625836371, -0.01081637239004723, 0.014390646949793055, -0.3936991588298892, -0.017287179665326734, -0.019662656608613188], [0.4559675810017614, 0.04185971077790693, 0.0002954922517086846, -0.08495746417440769, 0.020118713531488665, -0.022495065134490823], [-0.2875130460365009, -0.015554922628437993, -0.010963503600341762, 0.0310159004986089, 0.011796642311296077, -0.12466868959224277], [0.0109794733677589, 0.007869757791526709, -0.0034331280646001717, -0.35264586889253696, -0.0031440641684902714, -0.04701378908127978], [-0.12627931888777846, -0.015005579920556594, -0.007045704715792836, -0.20928593502524323, -0.006053806706510033, -0.046669654744118896], [-0.1794464704325819, 0.017003558145609526, 0.011884369314954882, -0.17509678003334528, -0.005455542678428332, -0.06439580098287641], [0.3921800002620766, 0.02095367526061913, 0.003278275599564293, 0.08383397489039651, 0.004715012314330583, 0.028330014053964132], [0.5229676458874861, 0.013370882000411932, 0.002799438107024895, -0.06553642913016049, 0.015374317682483927, 0.0838467536778582], [0.3043819192520197, -0.05127892748660318, -0.008151374384003792, -0.22194708719395595, -0.014736140294575723, -0.13051238047462035], [-0.21661665433534916, -0.012602260882112118, -0.006290776946031109, -0.1078592945167852, 0.005048257970706607, -0.07260260462376256], [-0.17858670621324374, 0.012746332984853913, -0.0003338025796856705, -0.1775873887157958, 0.003494624632600592, -0.0646563934420634], [-0.23972697230227513, 0.0007907227131691455, -0.011375314884392801, -0.24141823048177477, -0.015121389012476665, 0.15192685857092442], [-0.0805288682665265, 0.01649019894282365, -0.0058710985954171895, -0.28869447867840736, 0.006648313800914795, -0.03705073387005402], [0.45503517833046053, -0.009940711272485706, 0.0058145357128513205, 0.17507553599044914, -0.023699056711023177, -0.03752663645140882], [0.07614730758567163, 0.01094763336129046, 0.01595580645147397, 0.3669385672426945, 0.01630257173721327, 0.049475256478797025], [-0.19017515974922589, 0.01044461297724698, 0.000596674796907788, -0.1770289176867112, -0.0006652190097281266, -0.05784532466182303], [-0.2826095375415853, 0.004180596873188638, -0.008493739461126427, -0.11629712433206676, -0.007773479177510265, 0.01179217252798835], [-0.22800907303415327, -0.03738914371702291, -0.00997865534480079, -0.2147780754888263, -0.0011260473323270652, 0.12752356837204104], [-0.26633717795092676, -0.0020230322309422633, 0.011395867896906059, -0.059284696696027005, -0.004937391653946759, -0.05424977034545513], [-0.3105040992968043, -0.02979141818233848, -0.01514012066795736, 0.16019852759508038, -0.002963505755382449, 0.010076220626452975], [-0.2810046083884478, 0.0068154058342783765, 0.027694200252593098, -0.0845628818047412, 2.8682101431537842e-05, -0.07664413132844704], [-0.19322166446392672, 0.024269720360845605, -0.008789424441367828, -0.08130787993328323, -0.0073685279326514246, -0.10154402559184071], [-0.18323489070625038, 0.04530293933203359, -0.010368167443068808, 0.41668454572394603, 0.011850796963953168, 0.039095411050018763], [0.3850444372174532, -0.019245181670703604, 0.013031070595826444, 0.07410376974493818, 0.019082796256972036, 0.011861361823768202], [0.43844705929495703, 0.014997291369718743, 0.0031238234064281745, 0.15532360109244595, 0.010700360586174249, -0.047999340050801254], [0.33740262325253184, -0.0137576471630081, 0.010976969346780439, 0.21446631535857508, 0.026327443829020723, -0.06770213319532714], [-0.24591272357801172, 0.010178795682788462, -0.0046946104895431605, -0.09226816842708065, 0.006293120841442555, -0.07035308069626126], [-0.2549544027078708, -0.010630637954437893, -0.00744760308640929, -0.14691666743805357, 0.006878429843043239, 0.004468976581822915], [0.3352740721504418, -0.02531559654330172, -0.009713223645250048, 0.07858706182256653, 0.009852165129403073, 0.1606540925147089], [-0.22977796429101943, 0.0029920851654842494, -0.003727369412245687, -0.12869835374682725, 0.0032115050523737537, -0.04777477671734463], [0.5274395759303825, 0.020750843185113728, 0.0017049987035705285, -0.0992858737909244, 0.010499334066461703, 0.01663346317523402], [-0.2645277840889029, -0.013626058946978907, -0.007428732076280422, -0.07673143418558412, 0.0075168311422671935, -0.05987615517785296], [0.48027264438523426, 0.019098264899282593, 0.004130393816692946, 0.10038582320130453, 0.012069192741965898, -0.03882012856829111], [0.2773219149983576, -0.014550679182876407, -0.0029092846527673157, 0.18244636670648814, 0.0017200663202592353, 0.12429828247720291], [-0.2877209753067251, 0.0016599655254066641, -0.010980327531871031, -0.11215443498693753, 0.0072615268925660965, -0.0016001990368839014], [0.07488218846798203, 0.027891356280225096, -0.004566013521591709, 0.22591177919221128, 0.013025508351462803, 0.17947514854997176], [-0.1904603430577031, 0.006653626603871652, -0.009643187771643634, -0.30820115485530675, -0.0017393543174136106, 0.10749037199032715], [-0.1317425720926016, 0.021956144012655037, -0.02242872329886935, 0.18555807480283443, 0.0018912199356352062, -0.1691155740385088], [0.2965966066461452, 0.002517327630047942, -0.0241558590161771, -0.42137449622414574, -0.007393443579719862, 0.036397353721335145], [-0.001550060793956372, -0.027332445462860223, -0.03017778912173171, 0.21276347888487054, -0.009765235395969034, -0.16441485287225763], [0.5147964034386076, 0.02274894430140363, -0.0024181173705544964, 0.04645988184588535, 0.002774584106008553, -0.03039390062242867], [-0.31052109676025735, -3.216831762639579e-05, -0.0114735232441919, -0.04232005610762913, -0.003364675836735737, -0.004916644999666516], [-0.24419121666093557, -0.008639740011073404, -0.021987770752786507, -0.17738555316349225, -0.018212383362617882, 0.17381253732227525], [-0.1554688682830498, -0.0011287458603124628, -0.005097790355892561, -0.18646139627532993, 0.010251943142752747, -0.06682239727012888], [-0.20769032076831373, -0.012135737293117949, -0.006314065560126126, -0.13428691492316716, -0.0016524799020582097, -0.05259381488654923], [-0.26448128674105925, 0.0098494379678019, 0.006789480604956911, 0.1259550494059277, -0.013668450901134964, -0.006841770019032913], [-0.276544015444328, -0.015456833663293087, -0.009682184108597301, 0.019727263693912582, -0.0011781165899245026, -0.12703944722110244], [-0.2695401765393112, 0.0005877718298296193, 0.002714140451391001, -0.05424875430142662, 0.015436357806542641, -0.09212267258035778], [-0.15035999311265247, -0.020252020124660243, 0.00667315888814052, 0.22217561789393087, 0.013350961545605111, 0.3972448939572541], [-0.21988356375602125, -0.048710557955822444, -0.014520951677579505, -0.004755059621672147, -0.012364669033116933, -0.00016869001928054256], [-0.20979215482272923, 0.007835550762653295, -0.007041201565774554, -0.11041206506268221, -0.022264108846014874, -0.06596994203407976], [-0.04378269232260042, 0.03775666938728921, 0.01688539825060035, 0.4715537094894128, 0.02347248396817873, -0.04137335202411345], [-0.16915899058015857, -0.018021172608466828, -0.02092187900053169, 0.1474533278147699, -0.01122431737814885, -0.18371101586651206], [-0.24258596283368286, -0.028929677833642066, 0.01891253249431606, 1.7495395192420107e-05, -0.024914975608588717, -0.010201778634871152], [-0.2559186224738951, -0.012725932091568635, -0.010087866822762727, -0.14828377004295934, 0.003178195095007969, 0.03364604598156676], [0.4606643608326657, 0.012556899327858306, 0.020576341662509774, 0.11644966214648936, 0.00830464273387701, -0.046475240036734214], [0.3031184009915443, -0.05113461888910974, -0.008360414979772926, -0.23937878179550365, -0.00047883398813995947, -0.11546919671716868], [-0.14843111994362237, 0.00374098558415513, -0.011174942353351435, -0.31495780155637815, -0.015321944274363083, 0.125577008690745], [0.4715354663018956, -0.008246334729221325, 0.0036365567196317873, 0.1510278229783977, 0.0011424483008459832, -0.03776929290488583], [-0.20375589488864893, -0.024845794568345722, -0.005658540167209341, -0.020983342805078027, -0.003651875163996122, -0.13618793865010742], [-0.2135706475386059, 0.0010097865902412033, -0.005003449460278365, -0.1266597308996368, -0.002245712208160522, -0.06820357981689237], [0.5226674977981928, 0.005871827588355424, -8.70029387687951e-05, 0.06283488895127087, 0.009246074098283666, -0.05245661883066859], [0.33170482166589055, -0.00484350689036225, 0.040706188277342466, 0.18687407763972322, 0.011241876959005238, -0.008664150292290848], [-0.32532825390865594, 0.013943060318716486, 0.003662108363342204, 0.09071008586645703, -0.003575062685700295, -0.1583408156323609], [-0.31035900647833575, -0.0036927761606723327, -0.0049791088302361805, 0.10624225493234764, -0.015697921136099666, -0.1641433357382404], [-0.18477961830711964, -0.012988553634040589, -0.006776717513110912, -0.12688253197301602, -0.01904567303591538, -0.06295023887012975], [-0.3293843347469206, -0.008995975933158764, 0.03377996823509981, 0.1091963259918939, -0.0008984358271844895, -0.151704034011217], [0.4558339991071705, 0.007792138490657463, 0.007347677324890465, 0.04496088074320577, 0.015652154663389792, 0.04776759411513043], [0.47638702164888047, 0.0119053847692294, -0.004019819303721739, 0.05685603555396555, 0.003880689974213423, 0.01839064195996327], [-0.3417016135034277, -0.005936049835832133, -0.012677717666899447, 0.13952602852829793, -0.014446495226250843, -0.1328176221307018], [-0.05074517616395207, -0.01889487506897501, 0.017087658744998502, 0.30964609036706403, 0.0210952280047667, 0.10476834302365857], [0.4563365536864903, 0.01155925331598098, -0.0030401809215866655, 0.1366563461831534, -0.043214268068753545, -0.05084441415199454], [0.40012453674727066, -0.037722190371543, -0.01278984795753153, -0.35441547390665645, -0.001261004140861721, 0.005755725661063566], [-0.25073533997906405, -0.004663437843251895, -0.005740856349176115, -0.029467503030736048, -0.004238882703457988, -0.10566064676098018], [-0.10324881690605499, 0.012407672854876272, -0.0032426482138792575, -0.2244889395842324, -0.03274574889022647, -0.03765877416244394], [0.15752166139470286, -0.010815534151536329, -0.025357907825110885, 0.22982121770764097, 0.017163234281588665, 0.17560113811652361], [0.3814133473659591, -0.01587183565320047, 0.00114948336671872, 0.14813955044551433, 0.017065179115887664, -0.002846835751994958], [-0.28955203846212485, -0.0002571299336339077, -0.007742612546625191, -0.05670459066239763, 0.009528633891405755, -0.063093743768106], [0.5831463376185363, -0.017739166726697127, 0.008663682477877445, -0.08765364453528607, -0.0017867540548943653, -0.045133153192238026], [0.042503006076690934, 0.01786690730744268, 0.005912274006153779, 0.31735036156479035, 0.009941270520159114, 0.18053225895613223], [0.09492578186911751, -0.004611151277428263, -0.014280212570314765, -0.25038631669255196, -0.08140231844582481, -0.06633069316595168], [-0.20141341195005583, -0.014557937967069326, -0.0007709740299899669, -0.20963836018664392, -0.018447997497172262, 0.05296411222870252], [0.6126460706649443, -0.009256338791097423, -0.0018441659781458708, -0.08491540323355909, 0.022550770911231807, -0.01594950500195062], [-0.24023044004960065, -0.010974867739883388, -0.005045519463159877, -0.09365538615759374, 0.0014287636619087253, -0.0661958835850037], [-0.20435641772585114, 0.00464922158172977, -0.003974202899224049, -0.144061575163237, 0.006072566265356855, -0.060973513627401964], [0.3492335655668032, -0.007409296661898006, 0.006845227590676733, 0.18467878107888533, 0.013266967891039148, 0.025919754534491895], [-0.16641854791242153, -0.0002959829664042417, -0.01296459547048233, -0.3625920152988831, -0.0012609669974384414, 0.1386189497308983], [-0.15510022692987874, 0.006969039394023156, 0.001723002028480906, -0.19188872540213073, -0.024335862630500787, -0.042901670904438347], [-0.276575280042542, 0.010391197421427868, -0.006304378876075178, -0.062162424529836086, -0.011621650753412469, 0.04085912408202462], [-0.11585420642134332, -0.049727735754565364, -0.009745231268245794, -0.010105877186425416, 0.00522179087536218, -0.12330069791673896], [0.42527049376909815, 0.016761325285989694, 0.0006346092409689099, 0.1892980410870524, 0.005693873373022256, -0.07108935658101817], [-0.3198578084602065, -0.0031967113586992193, -0.004940160824309736, 0.15318861513054444, -0.011083409600628723, -0.1466352377641841], [-0.021208798335061275, 0.0032758485755637334, -0.02712059427552531, -0.3856106075887399, -0.0009852499975803108, 0.10719630638324346], [-0.1188817162421059, -0.008193967086682876, 0.007878666867229548, -0.3821409082781408, 8.00252018762433e-05, 0.13202428251654325], [0.4874274260998633, 0.002856413088802515, -0.0063291263185799185, 0.07056478704859802, -0.02827751479543318, 0.021123600957334288], [-0.2344819010391325, -0.01138848730703947, 0.012291638998830028, -0.11877324110669889, 0.008029076157583519, -0.06410041903687604], [0.08868874260024058, -0.0011911342462554016, -0.015395733218445905, -0.37798457311396694, -0.003824849155746112, 0.011470111236735882], [-0.16160889321210928, 0.03291740325246316, -0.0012014989472837105, 0.16788311076078402, 0.006941094055985705, 0.3819197563123812], [-0.284416230078224, -0.006201342116149039, -0.006321896408302612, 0.025699418414603027, -0.02508633453935697, -0.11318028193923609], [-0.07617879819713108, -0.022283427587622293, -0.004935808504713967, -0.23527781864826192, 0.01881084370125927, -0.0503916574301974], [-0.25189392700730556, 0.0047039982821909234, -0.006984999666427442, -0.13410012642698094, -0.014445502153306762, -0.00036944302817030133], [-0.13116302719727624, 0.014542820068200839, -0.010738125291754228, -0.24605268616448905, 0.011976582218953301, 0.029295374317302645], [0.41133784526468514, 0.018027525069356513, 0.015710567282206163, -0.03162502061181778, 0.01602802505794297, 0.03955010555667097], [-0.16414373094807594, -0.0008812545587164715, 0.033613667985327535, 0.2210841085285582, -0.049223249351396846, 0.35183149009033166], [0.5769474808219863, 0.03611601016773777, -0.0020741372310389998, -0.11962734839966886, 0.022527594156648657, -0.057574837610909854], [-0.22720898224031827, -0.01379539093649466, 0.006576637742133423, -0.10147539460741822, -0.00992906933024464, -0.05750780062765707], [0.5609588041667972, 0.014247902843488425, 0.0022866197556725243, -0.03111241138003019, 0.009814407048677344, -0.0006295684663573665], [-0.2971450488193313, 0.0007674702362110743, -0.0065205537312857595, 0.011849910250577944, 0.0018879950520313165, -0.11539055730192882], [-0.19638627425892893, -0.04443405141254362, -0.011375200519821847, 0.004145666198867258, -0.004660285473655074, -0.1168353897394527], [0.40794420351055444, 0.003062177938377777, 0.0026476340447329407, 0.19766473573946397, -0.0015095883328697687, -0.06810749623359513], [0.316074385935126, 0.005576497458946276, 0.0006498522363519815, 0.22277244161745002, -0.026195717772415184, -0.03837987444144453], [-0.1661815068644289, -0.010102792471040702, 0.0029040706961511115, 0.2613097154382549, 0.02773224452414119, 0.3304106063392576], [-0.28638264852157175, -0.013437813219347306, 0.004556435156089484, -0.03282503528805079, -0.00040119644981778934, -0.07918307501063451], [0.5408483270892714, 0.02158011435931978, 0.004178995463575972, -0.11667086469945549, -0.0248804387754739, 0.054071038279930206], [0.4542089081767132, -0.028632471242883255, -0.007561133362425571, -0.07766145240533223, 0.009574648728049327, 0.15208566677254265], [0.5878513045391781, -0.0037025979724678863, 0.007532064464705604, -0.07732207369234755, 0.02273198366828885, -0.035656871483552094], [-0.03639698510884556, 0.03547260845697796, 0.0026996786526370053, 0.21103581715960357, 0.01962375163211249, 0.25548703396941885], [-0.16660818695036525, -0.011219573079728842, -0.003204131502551608, -0.15949289621723708, -0.011992206044104574, -0.058406339539345646], [0.4936948684400015, 0.0155662157575892, 0.00043950361876522276, 0.1055316548970542, 0.015003300959641877, -0.04678387700638704], [-0.18205841423632596, -0.004424159618750747, -0.0010920623838661689, -0.07543371146142816, -0.008067722477745863, -0.09780804746894149], [-0.33084884188400515, -0.02141590202418302, -0.008425517329823898, 0.09993336169633645, -0.009440806039537107, -0.13935514582440942], [0.08020421499808328, 0.004516161006036218, -0.006881046619405521, -0.3048254614998128, -0.01236160737668841, -0.06428850592945876], [-0.2170694427868863, 0.012958100820253442, 0.002204142512680721, -0.16028271461537108, 0.008448712138404781, -0.04571784568812865], [-0.24519556021390643, -0.012972358869359215, -0.0049707188321470525, -0.07268801526573843, -0.0009905386930946722, -0.0745228081257542], [0.4646356358097612, -0.013254108429492668, 0.06772827078330859, -0.007647557087874268, -0.02892704437353576, -0.050500196702169375], [0.3020724622381937, 0.006459898618406113, 0.00159006543667117, 0.13324410411618118, 0.007541385432709462, 0.13191875082450438], [-0.27987159805673334, -0.02018197773962558, -0.007223783192620905, 0.10126346789029807, -0.017490268101360826, -0.15499674415363165], [-0.22353499700028356, 0.016943695211204004, 0.08093873868930641, 0.2570840837113299, 0.02303871284059202, -0.08899078900770284], [0.36547757551218424, 0.016900285659522413, 0.035668785083092264, -0.049829308040827136, 0.046428283574612296, -0.026535224963189673], [0.21402478426692967, -0.005075494572704925, -0.005636256632418886, -0.4551384565973071, 0.02100505598001672, -0.05863272768261378], [-0.20281562333495332, -0.006600773319902379, 0.0020410234402550294, -0.15971287726228206, 0.008970943953925052, 0.03573167160232187], [0.514878689284864, -0.0018258776240640605, -0.0071910629266657596, 0.07921283405534771, 0.009210075865453122, -0.03723789629448988], [0.42246844115745236, 0.008127786924177418, 0.004141531925118022, 0.1458522731190627, 0.004962215422217506, -0.011773200928981807], [-0.3032569221557935, -0.006786094054216539, -0.008901617547703439, 0.02991774573589822, -0.008007810150055847, -0.07488117833257706], [0.48436729895145875, -0.0029419557801973065, 0.05292999309979155, -0.031526020829336164, 0.010725823836250684, 0.018241765483935756], [0.45110628554042304, -0.02128667413694371, 0.0033564527469661203, 0.1725280694469725, 0.010853797282263317, -0.034299446031198416], [0.3780244666966555, 0.02370608732887313, 0.006856667716603912, -0.20745198638395185, 0.02447734003867793, 0.14568234523806087], [0.2610685285753732, 0.0034454199760228722, 0.02615174959298289, 0.01912060455816423, -0.025892400402405306, 0.1903216532554176], [-0.22460484376228926, -0.012356051055987666, -0.006136208321894361, -0.1530560279896471, -0.0016942109376215575, -0.016825991265892605], [0.37120061646846053, 0.019666927643473987, 0.027150433392800235, 0.12098104061121291, -0.04929754484864663, -0.060864602519002484], [-0.35182465930268875, -0.009216419491759052, -0.007408516084750414, 0.14318152661244468, -0.011220554377362203, -0.1497123769096547], [-0.22278554252218838, -0.022682881461299726, -0.007448816604116414, -0.060393322749419616, -0.0038172263151119677, -0.09397411510976897], [0.43903678068837976, -0.010553048215394649, -0.005430191940282516, 0.06034007200626898, -0.04305860373766859, 0.05403332453202954], [-0.1677198213992929, -0.002858735367042958, -0.005571032222492826, -0.20255809864777338, -0.019024270499724768, -0.0169413751970064], [-0.24017126114100404, -0.006605771635231804, -6.36987374768935e-06, -0.054594127261441985, -0.013145957116062697, -0.07805857646457524], [-0.14991392321051156, 0.021729284560758396, -0.0007600641848309832, 0.29814604500237496, 0.013021621185238271, 0.3777108461707759], [0.5274964673652737, 0.010747467905258285, 0.0016452060789161145, 0.0166304349252316, -0.03265176450877017, 0.0031000113633391016], [0.27924883353447355, 0.020287375180001465, 0.002718068774962442, -0.01911833295336201, 0.03150413453690108, 0.18683720086930442], [-0.2774239600194588, 0.0009209981392333587, -0.010962369191503679, 0.14467266135500675, -0.013923998138242437, -0.16908883319986062], [-0.25161905604284185, 0.0053027920550255715, -0.008001346615008656, -0.12172952933108507, -0.0008668320789792804, 0.01346559484419181], [0.0827506289899019, -0.007673423122300942, -0.00025779834507982415, 0.3534631403059424, -0.008815454961294995, -0.03574360080367804], [-0.020258041111292202, -0.02025209133353455, -0.00798972620515212, -0.2765074120948589, 0.012048457359123915, -0.0673811866142866], [-0.3379562283635334, 0.017676430005077293, -0.005303844943585583, 0.11125204033042325, 0.0012172665962812993, -0.16231597561152744], [-0.3490142383466877, -0.008902211739431808, -0.015556376807706683, 0.14350340467360995, -0.0056511021532439105, -0.1542219540717909], [-0.24254595289494002, -0.0243762855955897, 0.009315365462810527, 0.1438248006412015, -0.05672669818268228, 0.006464405489832882], [0.13850651358357707, 0.01176387517885734, -0.010097117590529957, -0.41973046055108143, 0.0010165809383110421, -0.06798849412323835], [0.5318264589928751, 0.019834706118522408, 0.0002675216972620803, -0.09473415724874944, 0.011313174460842276, -0.058941950052502086], [-0.26467744205065175, 0.0023334916372059937, -0.004446618727004655, -0.0246121954518035, -0.0005668551204397937, -0.1098703802873061], [-0.33052134949106843, 0.011637435423913657, -0.008654231387070919, 0.1102822448881766, -0.002906511214594977, -0.17368972683365932], [-0.15663548492235893, -0.0026970466724919647, -0.005934689827014619, -0.2316125480594345, 0.0032693006258087817, 0.0005498806201956796], [-0.19714641615451228, -0.00953149402706582, -0.009643554574158877, -0.16468197756957942, -0.002103197702687176, 0.016802354313715875], [-0.183546478564153, -0.011036783384083532, -0.005676801627358422, -0.1586188017936052, 0.010679182091616397, -0.05439031672241619], [-0.24301289818393307, 3.518484934377727e-05, -0.006050967723223219, -0.1710516896592179, 0.00486888715290252, 0.011488196916135208], [0.5368327744203198, -0.033985313411210416, 0.00871999985888444, -0.13627571647027056, 0.00470064442366119, 0.056417611178609925], [0.2721827423648674, 0.004491411087933996, 0.0035407033494902268, 0.2747285005350727, 0.018407392612094575, -0.07998836899707983], [-0.23321772892302678, -0.0056694870517805624, -0.01272597031158652, -0.2737297025118413, 0.00734050817447954, 0.1347394114360789], [-0.07262129814322026, 0.027183651702959364, 0.012803550584630136, 0.299102388037029, 0.016679408718959066, 0.2650734462857878], [-0.3144000310564112, 0.0015057435898405062, -0.013051019712893991, 0.0036346420239964697, -0.004401514205924564, -0.049144160507888154], [-0.28711605599762513, 0.008535026037604213, 0.08409774170416727, 0.014944448522401496, 0.004256875672244517, -0.13204216292292023], [0.5447185096625622, 0.002875186795413676, 0.009502486452175387, -0.07160800815651604, 0.009702972269717147, 0.036831948214738626], [-0.33383491212638927, -0.0030658538074711354, -0.003393815629272981, 0.09748622307703672, -0.012512301868513551, -0.14861667561406902], [0.3823001104788546, 0.020462192317946767, 0.043769946802828516, 0.13251011809823324, 0.015419222520832573, -0.027903039494058864], [0.4580879884259421, 0.032667345535924014, -0.010677519938677823, 0.11208934867378491, -0.02195472848959562, -0.0584112777447928], [-0.23070182748749124, -0.025136326387954447, -0.013587915390339243, -0.16865207809603172, -0.0008914671490426283, 0.04827497791937905], [-0.25491371317885886, -0.007980572209343695, -0.004699064631307941, -0.06911959805055737, -0.0013425073914105144, -0.06643269268666968], [-0.317637956559689, 0.022996468457976843, -0.007217780556686251, 0.0973440573347329, 0.012693284370321899, -0.17099119132622675], [-0.2721064264459213, -0.01884079113763009, -0.009186101223755402, 0.019527854762670292, 0.0007206030512647277, -0.1347884723399613], [0.31622622813251167, 0.013244446657819615, 0.007934471174639074, 0.13495072947774736, 0.017973747948638687, 0.001401805180072592], [-0.22382986978370978, 0.0029938901947338674, 0.004540865561873913, -0.19013940375066407, 0.009927086962127993, 0.020256366389867123], [0.42518541495101103, -0.013238647970287497, -0.005417668407290479, 0.20080441253079884, 0.0035894067445656507, -0.06503672737260864], [0.5336924845962622, -0.006814480206991096, -0.0102454732102281, -0.04420958633438836, 0.015652715763117386, 0.04541767272555739], [0.27387988165653626, 0.017912039332786227, 0.018625203410457924, 0.08669949819659052, 0.010529326062424469, 0.15724421007136344], [-0.2947876407367484, -0.0011376317400292153, 0.006135404875897155, -0.02056872069266888, 0.011609464335015834, -0.09884087604146558], [0.34201691179903293, 0.013827123060116254, 0.0024733191235693556, 0.2453813149860191, 0.025963195093990844, -0.062174483110348656], [0.5881735725656758, -0.005553943291837337, 0.0016077698156724482, -0.007876838837873771, 0.010456789144450323, -0.029207955456696123], [-0.13622768144984496, -0.0025649358411510073, -0.004644119517389394, -0.14204037794472435, -0.024644358560269693, -0.08072778594587923], [0.4141717875691843, 0.01601084134091518, -0.010754110113747169, 0.14650089404367225, 0.016380832363372008, -0.03770490303491001], [-0.2575794533817893, -0.0014341739531088386, 0.05740184034775492, -0.11750061601771611, 0.019773106408031225, -0.0536608765633449], [-0.29161094629640666, 0.000431944452845077, -0.003275150337481656, 0.027287813921023512, 0.0033325709758608435, -0.12405385176346065], [0.23165518600429694, -0.023577414363613887, 0.0075031126993580916, 0.3236815946920231, 0.011712962708776932, -0.04230948935988911], [-0.2412454720513021, -0.08784489283705844, -0.014218696855514232, 0.00392224423374451, -9.492953669042391e-07, 0.04442994343501248], [-0.2736445468657045, -0.008412764071582849, -0.007053845301959486, 0.026297598483571258, 0.0004460759073247518, -0.1434333024653732], [-0.15643326360738402, 0.0019713315126679717, -0.004513477742612927, -0.1776456594329161, 0.0029004932471566867, -0.07520275731024528], [0.021034716700028686, 0.010982505456647192, 0.005098054375586679, 0.34028825454910316, 0.007913652609109947, 0.16969995916666228], [0.4067147601799494, 0.01857349868560389, 0.002179678073952861, 0.18850967051755288, 0.011057270915002117, -0.04504154503872906], [0.494870462047424, 0.003410286653884948, -0.0017137253662338214, 0.12131617353995237, 0.015735831054541023, -0.07443214620913965], [0.2809596035932004, 0.01756319420684124, 0.0011011211064433947, 0.15970186996556865, 0.015267150292897092, 0.11073372750171466], [-0.19428683642839897, 0.008198135338131294, 0.015639029224104897, -0.07999964902152056, 0.003122712814301635, -0.11012354518332346], [0.5187794516286766, -0.02318912695991422, 0.014309001155539385, 0.0946591217679388, 0.020462676932346983, -0.059480172143638994], [-0.22409927593599097, 0.007362454744378668, -0.01545924018054349, -0.2806440482562661, 0.0008826966401081661, 0.11728193868001281], [-0.18022232676937178, -7.613792560314476e-05, -0.005289596959422757, -0.22146772847327276, 0.007854137452324253, -0.004025602226616113], [0.13844208033174493, 0.0008226671004580948, -0.005278982613588657, -0.42335840947183384, -0.0029743617474034074, -0.02828949781979758], [-0.00508815400109716, -0.05825174052946557, -0.044194077049704066, 0.0805900978096093, 0.0016487276667260197, -0.12658652056273426], [0.5356355525039852, 0.021759032658831362, -0.0018310070350558915, -0.14304226815321172, 0.014773647752086009, 0.04816157906989739], [-0.2605349315259824, 0.009936806749907228, -0.007174038436143638, -0.08130441590254206, -0.013466701341132646, -0.055503068750454444], [-0.28402843696702024, 0.003988593139363657, -0.0062264551164794586, -0.035796645583205784, -0.0163981287463271, 0.011817785291809697], [0.2897720023630102, 0.0046559488239086375, 0.00034117890995979705, 0.15024069718765057, 0.013408793602936108, 0.12690804577919926], [0.4366985493630887, -0.0142749544181839, -0.012328413822623847, -0.1571939547229729, -0.03132943845259039, -0.11712210540703785], [0.432076828833009, -0.020692315458939407, -0.0022245908058514076, 0.1978988426910352, 0.009600204198949283, -0.061082302791536695], [-0.32331612818917105, -0.020775652270551233, -0.007390405269997754, 0.10613025247431468, -0.005102456436050106, -0.14391767028337568], [-0.2289970782981075, 0.009881375348109707, -0.005791957069058724, -0.1366491026938907, 0.003139728999699597, -0.0512562996200849], [-0.34346994890879273, 0.00662738013728554, -0.014963207998809238, 0.1254017973225694, -0.005989760835470287, -0.1608239184246372], [-0.28943702866106397, 0.0014265401606398608, -0.009178492427361778, -0.056601380945406975, 0.006812001136851646, -0.06263941704143655], [0.5418826751401211, 0.02383683188923499, 0.05089439542895311, -0.03011654423017264, 0.010323134731771686, -0.03846604851546976], [-0.258818743890569, 0.0025820704743514038, -0.009631290750628851, -0.0892828751959651, 0.011103744567263978, 0.01090664544914256], [-0.2962525028107488, -0.0016314713533291168, -0.004893944661731235, 0.02187181644334517, 0.010606748614662658, -0.13209137086987913], [0.00013303418357512675, -0.012012986389415316, -0.007460219843371472, -0.16648548543325714, 0.0013627458205769302, -0.10076597722699654], [-0.14481546580241847, -0.024021016893858736, -0.0068326357638475, -0.182040882759099, -0.00452583185009376, -0.04579956922953354], [-0.12943470734474158, 0.0150907264865808, -0.004321242316311247, -0.21208001357579054, -0.0004189787153267203, -0.073509117867744], [-0.14140543245147438, -0.03366842453003806, 0.008864831485973527, -0.0005180391918359813, -0.0001189170616589128, -0.1405237801557278], [-0.3016810536456122, 0.014837241528602031, -0.0037975643025998022, -0.023257244419932065, 0.006361143277929518, -0.09127378680620338], [-0.30353908250966793, -0.028846916162228665, -0.007777112937153956, 0.07939326567105395, -0.008275253834349032, -0.11585497385549781], [-0.1773466175563752, -0.023306822672904955, 0.0020985763045627036, 0.1609020939759889, 0.01935236195732995, 0.3806563666380135], [0.23820754296269986, 0.007822907109893753, 0.001197998479263199, 0.13852004058969467, 0.016400403336080956, 0.16559444085570058], [-0.14835755266009054, 0.030754647349340348, 0.0031119911175936554, 0.2559346669042238, 0.017474306408326166, 0.37135665949532204], [0.37517255618340123, 0.023911101007733946, -0.0001325541128808975, 0.22116639160115, -0.03173793370859265, -0.056834640335893465], [0.22862918932985743, -0.01429121944599628, 0.0003795615992806167, 0.2103371847793156, 0.003297750870868899, 0.14530753286667175], [-0.030539384695759442, 0.018307988303296524, -0.011530360320761322, 0.36133556672384054, 0.008901434190556261, 0.1752437620100032], [0.38359181282991167, -0.009411123190021435, 0.0034133375921626898, 0.19350288932309798, 0.00912296503532319, -0.0021154371460309156], [0.3359977076417205, -0.028021089730511328, 0.006492343333776673, 0.26325585674323987, 0.01556733384382646, -0.03953565111055228], [-0.2290052380293707, 0.03351136763607658, -0.012907035371051884, 0.07423929995168904, -0.043175349679521774, 0.000175526920750266], [-0.308181206033532, 0.014513507963210085, -0.007304638589129431, 0.04516560704063919, -0.0046543317074381986, -0.14033727200708263], [-0.3069759719954327, -0.005134236449413974, -0.004238796873866682, 0.20341276711372527, -0.002783477218414958, -0.17840166985798284], [-0.1615639093137928, -0.012817652998498841, 0.003325946399750056, -0.32601128159683646, -0.005461425300057801, 0.13197625340720587], [0.5477110274381668, 0.024201192484706608, -0.004987329568691714, 0.02958439923031778, 0.012131746947717102, -0.045203258754443254], [0.38392177230704927, 0.0058023799139413305, 0.022811730241157173, 0.2072730700904759, -0.04010865179576421, -0.04317161388817462], [-0.04708341806298754, 0.005866322292354229, 0.006229436361024918, 0.2810979415745149, 0.008333158429542218, 0.0881510832150743], [-0.15816786030131394, 0.00395434705457268, -0.01311945696423303, 0.35180032090917085, 0.002612513007753392, 0.2818182315321384], [-0.296562582280503, -0.007329705343191535, -0.004726130032050599, 0.028292764255257088, -0.004973543653043357, -0.12687413627980132], [-0.2682084157535272, 0.009528825522078196, -0.0051675589886562056, -0.07929548154179489, 0.002348780680269399, -0.0607048800770989], [-0.28715346446551654, 0.008021230520883013, -0.004379721667730104, -0.08448412717216712, 0.008232944830502548, -0.0030451160142262843], [0.4993095634764567, -0.0090417074177335, 0.03977335233445268, 0.08423593871441037, 0.009659266049826818, -0.05594307982408439], [-0.16196729089848566, 0.01805992518927252, 0.0075962733073999525, 0.35025100860249275, 0.016786835278114243, 0.24758766028590926], [-0.35309225102740116, 0.011808287310357757, -0.0067619791089583315, 0.14356820179795762, 0.003310433173069395, -0.1748128742844332], [0.4632452038560479, 0.03675057669387629, -0.009902748544005078, 0.06256684120523717, 0.013180422517175055, -0.02176795806599519], [0.3466836980522325, -0.01201079093602222, 0.028730303398631984, 0.12125690606364435, 0.013889670441499335, 0.030512990757790685], [-0.23978381520141231, -0.00389887032127765, -0.00973406814054319, -0.24668333763407702, -0.004197767393454634, 0.14012232508931663], [-0.19687794527347843, -0.019461539175660982, -0.004860476476857619, -0.13850990066817667, 0.010945701269067326, -0.05140917300822717], [0.5012547049849546, -0.00010785195747638497, -0.0008339623116265694, 0.13253573828901938, 0.009030742082744138, -0.06677492664317225], [0.3577733457361057, -0.008587544190318291, 0.0031334059638400084, 0.25679661668991877, 0.004840140925591872, -0.054760250839424256], [-0.2694769289788598, 0.00300555309370254, -0.0029038522283474128, -0.14414574296435892, 0.006655113981026879, 0.011055268861542103], [-0.011810698780157634, 0.01939860762420919, -0.012349521353780256, 0.03801974454665456, -0.07144505160619925, -0.12327869885991635], [-0.03911159929244566, -0.020643306173422984, 0.02213187193015234, -0.07954745169364606, -0.012362506108444577, -0.0217990721542562], [-0.296640821729354, 0.004491850534753819, 0.051393121644587286, 0.111055516486993, 0.0016095266194044923, -0.06981665387384574], [-0.27922236629986324, 0.011093451060640035, -0.00821662906158836, -0.08914940034905015, 0.009254954530744677, -0.05093334321421574], [0.005691936099287165, 0.002103747243208936, -0.006277156299410562, -0.3957266936502997, -0.002711606654244915, 0.033604828632441255], [0.28556142897195835, 0.004941501744385969, -0.00022974561425213692, 0.22208268005587295, 0.0027027556666706504, -0.012529573205589827], [-0.24808158230625177, -0.027303851619017567, -0.018208803338592202, -0.16017065297809532, -0.004881591522896716, 0.16137493414580514], [-0.26154294403923994, -0.011879973651055174, -0.004109827176571712, -0.028127875181965075, -0.0027785685187840313, -0.09934128762285986], [-0.30390745047653633, 0.009480811024019799, -0.012411322986469159, 0.07152083204461539, -0.01122741752914007, -0.014523265929303279], [-0.25808927445984153, -0.01823586973103642, -0.013150324644743508, 0.04988516821905221, -0.0022626405042444828, -0.021785195525149955], [-0.1177161512556641, -0.012094577507003004, -0.018733465619774993, 0.07083416446981877, -0.010505853729214082, -0.1026960625279336], [-0.19183355133491403, 0.010774815872047261, -0.0055982595150016615, -0.17639015387114015, -0.0013026698481449926, -0.04574018130284527], [-0.049072636773497484, -0.025942966470803793, -0.012908959428705909, 0.12436058620772003, -0.012563707012129756, -0.16681284562311235], [0.25604351642879947, -0.006942194393344635, 0.008704521606805904, -0.4491573724160507, -0.007850588495110092, 0.0020783871101657816], [-0.330830768939654, -0.0027758173616517783, -0.0036715107098614936, 0.10764639899828271, 0.013664295136678578, -0.152949470344076], [0.21801194045595662, 0.025224397816175057, 0.007281645033712763, 0.32002133840283026, 0.020877545400509907, -0.04115998152514204], [-0.19758637331815093, -0.014169708244031017, 0.0017268333717099357, -0.1405888234553814, -0.001431426139521979, -0.06262383554795829], [-0.28936737241806476, 0.002326240851464666, 0.0002389810581565583, 0.04244577222247755, 0.006936602840499931, -0.14533689122119955], [-0.32310487589398446, -0.009895356838941442, -0.007958179820164585, 0.10385427182647597, 0.011130193181922121, -0.16987362821288443], [0.5078065408450696, 0.012903153000944273, 0.004788420852838504, -0.030474716241014697, 0.02003558124217603, -0.061001162239699695], [0.1752392640366619, 0.018355621026790003, -0.0006231929485723898, 0.23264109578189313, 0.00890468808179535, 0.12904728592619436], [-0.012129868145636824, -0.0034132214705743427, -0.006500657184863707, -0.3169889566853023, 0.002746646359825403, -0.07255394287344863], [-0.31072641359947245, -0.004950514398980787, -0.0009449843977570422, 0.11104722112080251, -0.02242702488785828, -0.1615304607074831], [-0.17533655427295744, 0.031121015604599456, 0.0045201491063944724, 0.16493448963212046, 0.015552288398377226, 0.3680372668654498], [0.3426360130741188, 0.024545139093277685, -0.02552906107341382, 0.03057176577841117, 0.012047840539043033, 0.14186846131872027], [0.29763127731645483, 0.004801723162008424, 0.00044226426094238976, 0.13906559100767593, 0.008935039178945248, 0.1344507717406384], [-0.34167502625444224, -0.017122756770019777, -0.01717729649628902, 0.1557764781085315, -0.005408936537189139, -0.14439912871725796], [-0.3115762994354442, -0.007117970147651652, -0.007445157437129689, 0.0782247175808358, 0.003978960487564635, -0.14046558730228187], [0.4555002222737773, 0.003957074854954672, -0.005314197657263925, 0.14827238816373048, 0.007763932454210254, -0.03810275342274481], [0.1424562007653748, -0.008509819524970463, -0.023073557705083934, -0.38759191950967975, -0.006782999885535733, -0.07542123747344251], [-0.31700972141169553, -0.013245991368347169, -0.006852779249522869, 0.10015429767412626, 0.030848645422819006, -0.14728802249595235], [-0.28000408591486575, -0.01351846762857731, 0.013422638406491195, -0.014468768969218518, -0.0014229173075283059, -0.11034839858630034], [-0.24055728309897914, 0.0043509665237349195, -0.002618842319292629, -0.10806569662420089, -0.004313992376579669, -0.05452240700664279], [-0.16635584126637631, 0.01050886120206673, -0.012779352735464778, -0.35394645318769546, 0.005269656426919344, 0.12397900257642207], [-0.055489636846454464, 0.04157223211648978, 0.011867103929262478, 0.2886716723012887, 0.012594667006228641, 0.2160835954099906], [0.28552651093412873, -0.009066246659793182, 0.00010004985932126535, 0.16141169247461262, 0.013285841646902212, 0.13206881841149293], [-0.05830187868916736, 0.0009176507749867732, 0.0018413353958998161, 0.34755400401347747, -0.015232621004053412, 0.05793034955769441], [0.42182082819971245, -0.013690795889243625, -0.005705841916274972, 0.2108239199136924, 0.004794152586488072, -0.07171559622770973], [0.09927353595982549, -0.025097601176140932, -0.01731457663257953, -0.2940303672913628, -0.01932572541206377, 0.025330459190002423], [-0.32697660124973704, -0.00900606254516018, -0.006661926073903703, 0.13497422701179465, -0.012386424275109859, -0.16366273094017528], [0.13580491050320126, 0.026399794609708382, 0.010730632048081919, 0.3497058163224212, -0.030521149211685944, -0.07243421062093645], [0.08318109620548864, 0.034076667402223725, -0.0001531609181946046, -0.005755214368743364, 0.03417978767944506, 0.21800365949761488], [-0.18016230270728245, 0.02483156300993184, -0.007243339376633236, -0.023955127499648832, -0.004817528717190119, -0.14972048145794503], [-0.21937656587030427, -0.011667516749114003, -0.007613395157335312, -0.11559874202354237, 0.008101199143713895, -0.06601831267675073], [0.43032563896409975, 0.01266116145293785, 0.004353603913789304, 0.15448814262873442, 0.016418617789044746, -0.046738854456748025], [-0.19064676127718222, -0.010039577639169767, -0.0060953933044269265, -0.28978587478015777, 0.001108480377110043, 0.10543927771849124], [-0.2950587245418625, -0.004493850549862867, -0.007500343339602987, 0.07650064339697847, -0.021634947472219192, -0.1324087298743831], [-0.16484300586914508, 0.011208056940673639, -0.0010668362508180597, -0.19807522932926638, 0.009736001020343098, -0.056215653178453465], [-0.22027541155891747, -0.0009065672166851258, 0.017199543797911, -0.30694333435028587, 0.0011013941418590851, 0.1455319942337355], [0.343226913417735, -0.004969175582046393, -0.006282315513479325, -0.4036497875726894, -0.06009232223852578, -0.07848997917766466], [-0.14906101716479983, -0.021860181030609636, -0.007043803195887657, -0.09411507718767229, -0.002050658736109694, -0.11572116744682613], [0.3796334145307856, 0.06396268339415855, -0.006604683630079501, -0.21785725227624717, 0.04736605241168197, -0.018149738239825963], [-0.17766994964536634, -0.04349816188309398, -0.005993067945542528, -0.021060582269630663, 0.003670084862001974, -0.10013187356536718], [-0.030771997116962692, 0.03333516399420444, 0.015141852596132536, 0.4011798311759358, 0.011060376613901314, 0.09460564575265913], [-0.19191328179681658, -0.0315744532029402, 0.0038633876415787188, 0.0025091646288092693, -0.012054308373744979, -0.14741058826196474], [-0.3255854069227141, -0.008579686400597314, -0.01312195319257236, 0.1721668231074041, -0.034488933411889466, -0.15555624000502932], [0.5440120097037987, 0.020374788429108108, -0.0008852702192205502, -0.11972863837743358, 0.017981346539049165, 0.09532243059135953], [0.1717784193303793, -0.009258310467338791, -0.010405461790354967, -0.31689597195150127, -0.005189386912240113, -0.027601431066089074], [-0.18850738700269679, 0.0008479839388060269, -0.0035793252341151695, -0.15046337531994758, 0.004055977951169884, -0.06224779590184457], [0.5352292038913606, 0.01717906267468975, 0.009006768461840717, 0.03963416500688542, 0.017405250657590975, -0.04262071296445963], [-0.2088494512839401, -0.005011522276983631, 0.015326584228456803, 0.3568078364054533, 0.0038240986013849025, 0.028520787658958907], [0.14533490602044966, -0.0063436230017105515, -0.010871501674685851, -0.39067935272094584, -0.03827013184722038, 0.006537322271728057], [0.08987625511342986, 0.035657777360346345, 0.014113554716774829, 0.270171584711884, 0.014691131710397605, 0.06771107956112705], [-0.2191130086977563, -0.023696312931418398, -0.001972484799287146, -0.2388724531481386, -0.0016262992838522589, 0.12959706715245767], [-0.2869078960350484, -0.014279416998104566, -0.005659789363449754, -0.023888418202878337, 0.010431223330783515, -0.08603570273130068], [-0.2239830115244992, -0.00522566080853949, -0.004987787900034948, -0.18863052523909438, -0.0031617608572578385, 0.02475512930814739], [0.38892915056984, 0.009417653657309365, -0.002029082365108546, 0.13781848859597295, 0.008675099439658237, 0.021723690102326713], [-0.1431482930078613, 0.03998303297777728, 0.015343890240449916, 0.09916029804707394, 0.0037227790581441608, 0.3638795426844116], [-0.32679490513063164, -0.0029905441358265496, -0.004142615371585686, 0.12093095737306951, 0.005480816754557527, -0.1739402596061345], [-0.2784691880878802, 0.002817998715922856, -0.004517104979148375, 0.03085405357936106, -0.011590974143725591, -0.12969668984643437], [-0.14943379754189046, 0.008708783587506232, -0.00949574237238262, -0.34484293309628483, -0.015313840281245933, 0.14809360739852925], [-0.14594260300289483, -0.01726546262674351, -0.009964061051305265, 0.20862852450859204, -0.0010695456321188582, -0.16658002134441094], [-0.19246061972043071, -0.001034844121828731, -0.006867001808358884, -0.21808270945939592, -0.0033558156406718054, 0.009627657417351916], [-0.1726957537000136, 0.0014205452018983542, -0.004956159256591768, -0.1713689533350777, 0.0015247414523479057, -0.0635977536958975], [-0.23794603109995102, 0.008554090343648528, -0.0062968884706031064, -0.11311156649644347, 0.002259397878108187, -0.06429900215475813], [-0.22115998048437735, -0.0053699057741948475, -0.004131289218140199, -0.13040934249400218, 0.0006964664184715859, -0.050769870016384416], [-0.32388469546491133, -0.013973915993305434, -0.007874638798601603, 0.1492123063541029, 0.009151098933922129, -0.16547989529094684], [-0.2847531445747556, -0.012094805105971902, -0.005474080800325106, -0.020961771086839977, -0.0016158610415319852, -0.08977367072390684], [-0.16166992571587996, 0.005090696774965546, 0.012578783323604873, 0.27407348450697955, 0.022793677151541708, 0.3680373315778313], [0.475522668032667, 0.018095104997852524, -0.009624850122614598, -0.06842746463584287, 0.012095308615183355, 0.12398624609976532], [-0.288045385325219, -0.017409164111450437, -0.007665910201682328, 0.03363793362089131, 0.002579872137501991, -0.13152067945337398], [0.33696543511299987, 0.02172583411134886, 0.004863671846016109, 0.1414135789379967, -0.024615968235031177, 0.0585574482266686], [0.3826542441334872, 0.022904659828255017, -0.004772287111827733, 0.17481023867743448, 0.023835243853331158, -0.03479594229793243], [0.27889378432383216, 0.016862561208779606, -0.0006210398714497874, -0.07661244338308067, 0.041170594378645356, 0.1886986862004159], [0.5813170473559123, 0.014062111001246538, -0.00674994728848253, -0.016562064288034518, 0.012268410870287686, -0.01903378275915937], [0.4208087526273253, 0.005425900553119704, 0.036995086904276386, -0.01644151243896807, 0.009890133580049077, 0.0893983054408616], [-0.17811565603183685, -0.017507768265590296, -0.014208123460304855, -0.3089172749554353, -0.0034978237141943353, 0.12462046221683327], [0.3708268680805844, 0.016973354657512055, -0.018954839805017238, 0.04746062825129539, 0.02285989884151216, 0.031517899497921206], [-0.27246225950118425, -0.005061005215922109, -0.007109287095368412, -0.07120950602030263, 0.006683314142095055, -0.05579236742042909], [-0.21729770911175553, -0.018610758429008535, -0.011181764478034805, 0.07942168481788302, 0.0022443317062674248, -0.13745601686335399], [-0.26909000014571083, -0.01294428273341215, 0.006679076922043993, 0.020930914077023254, -0.01686892725651863, -0.12389980284144586], [-0.33393805736378235, -0.003907945822342906, -0.018543192310817594, 0.13772608702901631, 0.004219074338291375, -0.15490354162794243], [0.5063092188231985, 0.008327205832455721, -0.009750716283145992, 0.07710987642597267, 0.009095301121737431, -0.03640815864749253], [-0.25531385788810973, -0.010503974289567872, 0.027810884568261862, -0.08903067927256876, 0.006055408278381212, -0.07352444806306276], [0.5377368662435053, 0.004603584970876995, -0.0015141852384424398, 0.055436978434678826, 0.014318741242069593, -0.03406592504663014], [0.4035035291294811, 0.022360110317186436, 0.002386505387541685, 0.1705706925398374, 0.009340274170328346, -0.04227792601506718], [-0.19030709778179433, 0.003912694301947218, 0.02288000514379068, -0.18476001195057062, 0.008701756735280747, -0.06676734644865384], [0.524859546001408, 0.022901975360645317, 0.002370119396322803, -0.08463546618686184, 0.030278086134589578, 0.01907423135738344], [0.5936286217773769, 0.023734121072913505, -0.005827988944663922, -0.07926136973350421, 0.023323336341033767, -0.04809941892585928], [-0.2925763982945824, -0.004351108459565677, -0.009665396917987465, 0.05287497355983577, 0.005854145683615419, -0.15673379132888968], [0.34571003337265005, -0.025256844977169596, -0.001224633722493502, 0.12708524705542332, 0.0034962031778044004, 0.11768332842711632], [-0.29327976833787267, 0.0011115922850975487, -0.015182081311959804, 0.01340543286054312, -0.030242008518406575, -0.04750299175579802]], 3: [[-0.014873166186869846, -0.00040496922567174093, 0.00035624668201779245, -0.0006497391066454122, -0.01857916479870765, -0.0010492073641232245], [-0.005072777433518667, 0.0009218877737337886, -0.0020120891526234966, -0.009324507345280933, -0.023070367113744928, 0.0033578532714342163], [-0.0037128412257682736, -0.0013772524573282957, -0.002247631149042489, 0.010208893247804373, -0.03323573354171538, -0.0031687682072833044], [-0.005498804261655039, 0.00017716233337258707, 0.0011810972296845703, -0.009846584708564855, -0.02058051895764858, -0.0006323516351887757], [-0.01507208381038648, -0.00010271842861487538, -0.0009950937757310797, -0.0005546586200729751, -0.017549747707640784, -0.0009256976575538701], [-0.017295430364871387, 0.0005210000807004406, -0.0010737722661456916, 0.0072843114119346495, -0.02373352338970073, -0.0009025854719173388], [0.10228618593703602, 0.024224809527815867, 0.0016103458263709868, -0.04893412064717706, 0.27373599669226034, 0.04097797313988435], [-0.01369922624418227, 4.315428583974097e-05, 0.000894605595775006, -0.006021225538427016, -0.015448798477300072, -0.0009685096217054326], [-0.016637895949747034, 0.0001671980208637477, 0.0005368955870659308, 0.006177619076945764, -0.0224085219885766, -0.0005352947465518601], [0.04061125597096948, -0.0017076376675774212, -0.002580217546322317, 0.001086603451009114, -0.06826267957763991, -0.001847324630438992], [0.03604195453169658, -0.0019220638703218093, 0.0021450567826079136, 0.010840862965917847, -0.06819196316400528, -0.0006138472458951642], [-0.015309148119665916, -0.0008060060378555268, 0.0007003301633350233, -0.0005950321973074873, -0.018495922107802966, -0.0006942217007031932], [-0.012740248676221113, 0.0012738889593579663, -0.0011296604912771925, -0.00799797150286261, -0.013761564640977208, -0.0008444436480198807], [-0.003905344658928152, 0.0014277107549727059, -0.0008198218574594644, 0.008823082378233005, -0.0357098362619921, -0.0025157903548259706], [0.022868551372267606, 0.005904172299050645, -0.002793890668672219, 0.024747770833475263, -0.06441774449400156, 0.0021300295467691663], [-0.01605715108821362, -0.001080315652114854, 0.0004736483033947607, 0.007367600581938054, -0.026712767068196783, 0.0008089849231923814], [0.010452016975282825, 0.0001878795118387399, 0.031280070696696236, -0.015448703188244036, -0.027825370824702086, 0.0012275195275409645], [-0.0026767028856697628, -0.0014071542888022901, -0.0026065816905967184, 0.0011985962908819398, -0.02812046810642974, -0.0015876893193834635], [-0.015268572472845895, 2.4380508808233034e-05, -0.001794510011926736, -0.007613337691624523, -0.011767167651911825, 0.001219207319500723], [0.02794285190913109, 0.0008058169442923871, -0.0009931458753179275, -0.012927041672691698, -0.04513869427579246, -0.001556453696288011], [-0.014724149799737055, -0.0001267421296269798, -0.001660413072421818, -0.000860940067442653, -0.01687595248209758, -0.0009518024486739709], [0.03567795503870336, 0.0018069021508706588, -0.003098460612217027, -0.0003699322606936731, -0.06487351116004801, 0.0006570468433846935], [0.2917625971918412, 0.003678334515664904, 0.022302315525454808, 0.07064432563155813, 0.48055188840085916, 0.022479586353668594], [-0.01445887234336659, -7.657678686691202e-06, 0.0002060160787224546, -0.0008457721532769935, -0.01794169043819193, -0.0009020234652002888], [-0.012625764535170938, -0.00015581362116874983, -0.0012777397591183738, -0.0072754721230114845, -0.014593180462441414, 0.0007279705009109123], [-0.014199291776481635, 0.0012578902225384241, 0.0013426984114180108, -0.006238911358376789, -0.01672649453169699, -0.000635890967401084], [-0.06660997768758137, 0.003793997260385146, 5.757572156097626e-05, -0.0558602245075368, 0.11148786449235423, -3.352099346802671e-05], [-0.00152301740767242, 0.0018855255609308479, 0.0010482555738060657, -0.011970349625891523, -0.023786377030019058, -0.0008540370711540163], [0.0067938099374717275, 0.0007743505702801883, -0.0021240605816272287, -0.013015225118559954, -0.02715323021341229, -0.00047564459415253474], [-0.06695464503637785, -0.0025269163419075753, 0.0027632895435169198, -0.00605877176440025, 0.066163322896265, -0.0025029459637629973], [-0.01812228519543158, 0.0025636791365829152, 0.00227662166466472, 0.009993293192508454, -0.025871951664030574, -0.0010393571342940033], [0.00943761124880925, 0.0005873425040340409, 0.0017841777910307568, -0.008276788166869211, -0.03258038812110776, -0.002401955255897076], [0.012673420513610322, 0.0005962245070803999, -0.00022468356160539922, -0.013060769212600902, -0.034675498736035835, 0.002824639822884733], [-0.012047903244078626, -0.000658121495310371, -0.0011304663183210751, -0.007140982172307893, -0.013857066068469436, -0.00036546070151264763], [-0.016842788541949396, -0.0003630682638101801, 0.0011623236546646585, -0.0003478283805070355, -0.019650144571014116, 0.0008415061026160016], [-0.01801616200028499, -0.0010287081081361241, 0.0005933978059993098, 0.005140469482960494, -0.022061858844483598, 0.00017286166394484996], [0.020741004575857782, -0.0005838862614103307, -0.00017884385574513405, 0.01888246853948614, -0.0707886998825263, 0.0010612902176711427], [-0.0009641308238470301, -0.00019328983008656077, 0.0011496641073494977, 0.00027840747862370755, -0.0353196204909344, -0.0001510304411051879], [0.007772853470648674, -0.001007472381248609, -0.001365316843690927, 0.0018847660611968895, -0.03557829435323777, -0.0023232026203349313], [-0.005786430290020631, 0.0003291395946891248, -0.0009152513161808795, -0.0074782038463218955, -0.020428380928754004, -0.000920873213411743], [-0.01380044558956277, 0.001199639307073087, -0.0009407125838264981, -0.006007180461416537, -0.0162364322409843, 0.0005851315687169694], [-0.1337398967716858, -0.0064535933789577685, -0.0037348712292642147, 0.03600393038195786, 0.14874207864442424, 0.0016787809249543717], [-0.09990576352670624, 0.0025050508507639944, -0.0010791457371117695, -0.02564371982769296, 0.1123807874450285, 0.00354279079571838], [0.012266901459423234, -0.0007961745078110316, 0.001113915983391442, -0.0066436757005422006, -0.03930915100702125, -0.0005818162274402145], [-0.012159025982353636, -8.169471447955251e-05, -0.0009070303082891822, -0.001676072551836633, -0.019310974373206648, -0.0010652020698343916], [-0.08816672842639954, -0.0009417452359485981, -0.0003715355134222416, -0.03203250784165277, 0.10306511708908034, -0.0012406953097525571], [-0.015771985618572945, -0.0006579629147385977, 0.001988549929101312, 0.00041292287633288584, -0.019590014572329045, 8.515696687296465e-05], [-0.02077187999582937, -0.0026576737844032972, -0.015948366927285004, -0.08775175127235113, 0.13146011715235167, 0.013941777049739213], [-0.006969953476337355, -0.0015688979407817677, -0.002097137003023948, 0.006690367282228383, -0.030554109790963128, 0.0002997309288777244], [-0.021351898998338312, -0.0005635450820995638, -0.0028024231685647694, 0.024140389077932697, -0.030479477445966102, -0.0008097110496306989], [0.01236712243566482, -0.00014756285762432462, -0.0011515390001517432, -0.010145175284994, -0.03427104085197148, 0.001481528892410013], [0.010516341243285146, -0.0011514729826437536, 0.0015547076089840912, 0.0020677415158441353, -0.04999145332380531, 0.001804135938335618], [-0.018069125748920504, -0.0001351953298477685, 0.0009351006733057421, 0.005785762346495976, -0.021784603974676756, -0.0019319379663567747], [-0.13878596826364886, -0.006314791520336089, -0.0068695739820751325, 0.034034432775782275, 0.147815536815057, 0.0017259197307764387], [-0.013208436001621079, -0.0008224351578380606, -0.001098697376727365, -0.004004242614315015, -0.016602511362464985, 0.0005363225129664695], [-0.011839745414337854, -0.0006917716656457273, 3.588522001484276e-05, -0.00718986264808491, -0.014461455548976133, -0.0010530499429702567], [0.17923778056774656, 0.010764148274667793, 0.02446142499976105, 0.06099210566713231, 0.36226477545148256, -0.009080949246504495], [0.002525382663942453, -0.0001740130610370343, -0.0007961794508348991, -0.0021451022548120667, -0.029554195188485146, -0.0025558927087733553], [-0.012954091305169842, 0.0012419084663182003, -0.0012565078494390209, -0.005810170424843487, -0.01626267012467035, -0.0001584687621955256], [-0.09307592049623227, 0.00035211071556248593, -0.01180568499897604, -0.004845338954147246, 0.10362646966636352, -0.008713540694475335], [0.036531013919469946, -0.00070296832906713, -0.00010762146580499488, 0.0012181804992765877, -0.06884075721378421, -0.0032978474100900596], [0.009073990894362545, -0.00042082619415387815, -0.00012159763143281582, 0.0013980838815869344, -0.04268937548441424, -0.0024402754659485545], [-0.007071916401728918, 0.00045902773064840087, 0.0010098163083407346, -0.00101495558039755, -0.030479659287625097, 0.001897687230762377], [-0.014679620939608616, -0.0003507883765439816, 0.002570491245053949, -0.002103358145069826, -0.017425602812878648, -0.0007111209709529362], [-0.12321923085452009, -0.004073514040114536, 0.029407598183772594, 0.003520109306210379, 0.13081847787011358, -0.009594457781479007], [-0.08109114415052257, -0.001955781324637503, -0.006610836065923565, -0.034841563900112615, 0.10418897563484925, -0.0071396501936530615], [0.0441604685607366, 0.005874163617084895, -0.005086807913486991, 0.006386992374843778, -0.0708011286159964, 0.002063931024437208], [-0.010330077013198272, -0.0006854480936870284, -0.0011555027158466813, -0.006516392666081013, -0.01576438113617218, -0.0007481983750148213], [-0.013227929933636853, -0.00039015321679069294, -0.0021196673197021506, -0.0009473268997080823, -0.017762699120466675, -0.0007522235096955957], [-0.011688784951774004, -0.0006081401699003445, -0.0018330160283000223, -0.0071681395275433185, -0.013426818740555421, -0.00047510058192692586], [-0.004844714127745686, 0.00047559234823299837, 0.002239785440626253, -0.007980246604361833, -0.023482611988299405, 0.0017255282648809904], [-0.14641345550067325, -0.010071283990691016, -0.0022609316548525544, 0.016384403494498097, 0.13474356630359413, -0.005707298651875203], [-0.08864506835338569, 0.00015078453743598173, 0.00317888454098332, -0.036597958189950416, 0.09419778559876721, -0.004151094800517171], [-0.014349467127776709, -0.0016422449651629979, 0.0009708206460523702, -0.003391100123095493, -0.0170761165497042, 0.0002881081196869649], [-0.012234649688306572, -8.576858511871297e-05, -0.0008411472099825955, -0.007201656405435244, -0.014201236315767382, -0.0006355417953895453], [0.03304108422621282, -0.0009178233720364534, 0.010317932504224386, -0.013418282100595723, -0.0580056811492049, -0.0005029443943144001], [0.016546685484818007, -0.0008616581326106248, -0.0022027713231224984, -0.013506673274462767, -0.03428421923124026, -0.0008913635233818798], [0.0606082540215571, -0.01791513051399635, -0.014751438231515155, -0.03480132669677035, 0.23640647594675504, -0.02873492976412554], [0.041815383546773895, -0.001701170702417445, -0.0018977740478815033, 0.0017368712555013116, -0.061082298824981146, -0.004071011226995251], [-0.013677836415814713, -0.000323710190635745, -0.0020775502137103437, -0.0014397855429687915, -0.0186207835520658, 0.0009396659151953535], [0.018919977039240345, 0.0010612224665543922, -0.0005220002214398801, -0.010368985700332843, -0.045620611091878524, 0.006747064174523112], [-0.0808262059830623, 0.0006734490900026051, 0.0005946255670762154, -0.02056157500513122, 0.09054815521874679, -0.002295115554299036], [0.01867890956393243, -0.0004055745481565977, -0.0019224617727030924, 0.011162377207313057, -0.05655975118317338, -0.0030022387630108363], [-0.0011373971007417002, -0.001091469273872535, 0.0011965404826889802, 0.0016826856463906608, -0.032264755216751456, -0.001918937871047309], [-0.013178136363418877, -0.00035396304295641354, -0.0012221327164787207, -0.004959929621529268, -0.014924055192524984, -0.0005617830630918155], [0.03294461552354693, 5.6199126352112036e-05, 0.010695054451814222, 0.008769728402348926, -0.0767408330213567, -0.00096047876841985], [0.03489732149851401, -0.000920813675631415, 0.0004087414095740234, 0.012306576672685709, -0.07806796257807228, -0.002573863327069973], [-0.011918997556016782, -0.00022686890425207862, -7.17036789208024e-05, -0.0071077176995902516, -0.014789997259990641, -0.0010847149012295125], [-0.016921596058900625, 3.328455974947191e-05, -0.0008893337723469836, 0.004288154558245607, -0.022453548535421065, 0.0007430392486735024], [-0.02470687261849264, 0.00028060563684409586, 0.0009576191119246699, 0.02378065736811646, -0.030209873796837764, 0.0005311976317784789], [-0.01574925334616943, -0.0010054689457625005, 0.0008799166356928198, 0.0009600026668246686, -0.01918561590589751, -0.0010995811046881241], [-0.016762257014830372, -0.0009338593582902905, 0.005567658597772571, -0.0006384749258591913, -0.0212142516706738, -0.0012188156281189718], [0.020606149755495438, 0.0013434828353180105, 0.0019007216062029598, -0.000560533662603907, -0.055219572726011516, 0.0029797521915988713], [-0.01602488362087697, -0.0005997507003898982, -0.0013103913453617144, 0.002529055218586106, -0.018774182307413522, -0.0010198472445440451], [-0.017073797849443517, -0.0007480638061488908, 0.0014798557493611861, 0.006136415930480477, -0.022299766085178988, -0.0026946439390703562], [0.01365443049860285, 0.0027703439922473505, -0.002258931752373528, -0.01461294536275033, -0.029992774201110534, -0.0004744088889015281], [-0.13785182715888314, -0.003535731866438947, 0.002735336114450154, 7.77760456968991e-05, 0.12041372756266107, 0.004286116127910805], [-0.013687815831926199, 0.00046078046076709987, 0.0011141141173056086, -0.008363592345059994, -0.015907295401578744, 0.0011838090004921695], [0.020922567840431425, 0.0004474560456704673, 0.002066178000704354, -0.002984152992814367, -0.055180790688114115, -0.00047125820587784554], [-0.011927041877410683, -0.0007168358515597181, -0.0010254830789385634, -0.007098382664236305, -0.013686707165664742, -0.0007455493621900464], [-0.018132617806771376, -0.000879523381330593, 0.0097846085124811, 0.0011607271770999001, -0.022715286958434607, 0.0005820924569554836], [-0.04954105741567712, -0.004662724972340483, 0.0014163062495627114, -0.02524531857826648, 0.04828842503744811, -0.0012889636540600261], [-0.0010705353564215811, 0.00015414715931004556, 0.0007156218094132849, -0.008652938615865465, -0.021002512367838497, -0.0020104492952645182], [0.021512526401199814, 0.0027184472703502467, 0.0011547485896128103, 0.011600557138931683, -0.06030306431966022, -0.0009308341280533398], [-0.014031713306423749, -1.6505530986417493e-05, 0.012144008338084164, -0.004842430040829057, -0.020679814828049266, 0.0008931220348709501], [-0.013010507375964245, -0.0008006322499294783, -0.0004838509567391063, -0.00770213727102006, -0.01666333516851177, 0.003460463022164619], [-0.01322784202324014, 0.0005703019481603391, -0.0011345605489824731, 0.0003532276180331678, -0.02320678361152295, 0.001445656617552], [-0.012742744182421979, -0.0002726935366519019, -0.0017208340536278162, -0.0050353531359169385, -0.014585070543159027, -0.0008433045482223893], [-0.016427948388030925, 0.0011895103313466389, 0.0010778376945955507, -0.0022811427933459197, -0.019622485472604957, 0.0008642286280395553], [0.014436527081569008, -0.0001768422431115536, -0.002502536781002412, -0.012407869746889198, -0.03317774275156733, -0.0013715355589985058], [0.03227146820354401, -0.0018345114527470688, -0.004543859926308145, 0.018552090138822187, -0.06855307307721249, -2.06853146699144e-05], [-0.012112389108636285, -0.0004329630160409308, 0.0005995675354293703, -0.008094057393554038, -0.015200631159506506, 4.047314230835182e-05], [-0.003993865322243354, 0.0011153947652173676, 0.001239089673022614, 0.03284775948324966, -0.043312673741579204, -0.0028099905719527964], [-0.0942397808154204, 0.0018079252725050454, -0.002631678017546304, -0.03189554347651603, 0.10461390895354962, -0.0008548319165723045], [-0.016603194343580485, -9.84524738298736e-05, 0.0004697602515753551, 7.06462175760831e-05, -0.020150354664544033, 0.001111595012802882], [-0.012357524721393618, -0.0010407236668198282, 0.0005810701068790828, -0.005970515724570264, -0.01737096146831503, 0.0009586554742196269], [0.014654067406607794, -0.0007979626421064788, -0.0027057560832753585, -0.010317638144539754, -0.03683835808725438, 0.0008056475505681389], [0.03659087504996925, 0.001750762902797984, -0.0021807006424105187, -0.013389834104948267, -0.05406102983413766, 0.0010899266287291158], [-0.01594536514838219, -0.0005022798233306954, -0.0030019517881687326, 0.004881460841142699, -0.020917196106635845, 0.00028533202537470517], [0.03162878962237706, -0.00042442596613794886, -0.0010952729528709876, -0.015419468566010572, -0.047680076843513594, -0.0022095452938439277], [0.02028718062929839, -0.0012214761703358327, -0.00021875226231282295, 0.002715856532011556, -0.052761140120295286, -0.0015016686083660729], [-0.012567796038461265, -0.0006678428285275606, 0.0005041174440326467, -0.008126781656865216, -0.015165266809656129, 0.000823569889477491], [-0.015928350277104656, -0.00015596749465919215, 0.0008534984105608484, 0.004821708639718448, -0.021550170200219248, 0.0007592809217037312], [0.03657596591983324, -0.0014616122016083784, -0.003546296981211629, 0.007467564060449824, -0.06680753626795323, -0.0007614178628431576], [-0.01227640948365721, -7.010261369124704e-05, -0.0011470158023198318, -0.006847200929866938, -0.014098059500457907, -0.0007612116700069103], [-0.017581974834939267, -0.0004586054534476805, 7.582812447906736e-05, 0.006152367260905597, -0.024179297486553206, 0.0007916823895554354], [-0.015964305518718566, -0.0005144395273172402, 0.0014953061393465365, -0.0005557247174076956, -0.018387609004777317, -0.0012732273711257825], [-0.016967601718778338, 0.0005933632363104269, 0.0011172028197441698, 0.013272936342114555, -0.025270444838211854, -0.0004454558411790134], [-0.0821008973675508, -0.00168348936478335, 0.00024799165528434385, -0.04334711082274691, 0.0962370322971996, -0.0021249549688313225], [-0.022642626453700083, 9.117309934464779e-06, 0.0012477580526919234, 0.021264791465509136, -0.02835983280178919, 0.0007807924273537098], [-0.01289836513659387, -0.0008069177961563746, 0.0006173226858018603, -0.007409253067229388, -0.015152358676085782, 0.00044957199026352776], [0.013652119377398547, -0.00015602693101747123, 0.0011255647485023325, -0.007828552483424494, -0.03969265335286229, -0.002300451358596674], [-0.0049670156151578075, 0.00212242693266631, 0.0017498212027333554, -0.011455905001765637, -0.01998934630326253, -0.000659981215213769], [-0.01860628822194496, -0.0006692488365690169, 0.00034920101236765203, 0.014745057372944667, -0.03193744548764871, 0.0009187241608503008], [0.03734767357400896, -0.00025349069623503977, -0.0030420776215962747, -0.0008046310872600652, -0.06291622611961767, -0.002197914715966551], [0.00887759453208264, 0.0009503924599047921, 0.002018689086842845, -0.0033793653179269175, -0.039058043752765934, -0.0021092670081374314], [-0.1361000692077, -0.0069971726629100995, 0.0008767972300797146, 0.0012023159081459736, 0.11690658448199497, -0.0060884557496105034], [0.08126436029781056, -0.021850384375939294, -0.048049184418922496, -0.030619239366051845, 0.2319796446468586, -0.016064085672644494], [-0.0133438086284199, -4.836943704467264e-05, -0.0009867806469143479, -0.004881349336391474, -0.014842443249958466, -0.0010972487012712124], [-0.010679422118600648, -0.0007004833759847285, -0.0005795078027434169, -0.007879035613514536, -0.014767867097669942, -0.0005936839914867565], [0.013548192852441656, 0.002073915487595499, -0.001212922358357556, -0.0010785659602187867, -0.0482305199865991, -0.00030010003486179523], [0.04150997528735942, -0.00019667625861959358, -0.002154018687004465, 0.002650070487502358, -0.06552768664349531, -0.004398330852409009], [-0.018722986829448885, -0.0009537702185235933, 0.00024287367410544727, 0.005197769828195298, -0.024696423143766286, 0.0037325366894379844], [0.017728656204370977, -0.00190452888571915, -0.001019935155931705, 0.01929086717566156, -0.06548200405553377, 0.0002345637647711148], [-0.05185983528897411, -0.0060374345126507, -0.005335902918216552, -0.04054406638039388, 0.07769839051934903, -0.003746151419113751], [-0.018981874869555652, -5.465379443026789e-07, 0.0011864545463077727, -0.00045737994935568315, -0.01676564749749808, -0.00018100569195410803], [-0.02212841946503458, 7.512467485354813e-05, -0.00022593289975909842, 0.0054336202587243546, -0.01862692842955176, 0.00027253586076753805], [0.006481074018583475, -0.00024914988295859756, -0.003376626050520697, -0.002947901388773281, -0.032751185944473314, -0.0023562107518576366], [0.05467149624681539, -0.0003411324818123641, -0.0235237184830377, -0.12543353570063112, 0.18267703937143695, 0.009743898666276252], [0.029686410519415906, 0.0021056984391106593, -0.0032599201883039683, -0.013707894038415812, -0.048919307858021746, -0.001104986873785014], [-0.01588412108668527, -0.0005386591703887927, 0.00011146399462844951, 0.0006248575519002062, -0.018258441681264028, -0.0012550996081906317], [-0.01573564982149497, 0.0006856934123136464, 0.00013914018521442138, 0.0010306016844773998, -0.019984558944900782, -0.0013352265156097956], [-0.01816366953246807, -0.0008035809997263559, 0.0016501045924056591, 0.006625389175815597, -0.022811496058013626, 0.0008032528219867362], [-0.012660272974439725, -0.0001925419643737305, -0.001566943297572885, -0.0063807888219733645, -0.014584517733486098, 0.00018506479184574902], [0.01576922699177881, -0.0015652452834594432, -0.0038641720277389053, 0.0021507642335186904, -0.046314260635799825, -0.0013763132782993197], [-0.01218951946218952, -0.00044199351579933197, -0.0017925522348486264, -0.00731227964654289, -0.014333303544826802, 0.0008696484042071296], [-0.01628969103093036, 0.0019600547838360427, -0.0019780856955421296, -0.0001142850985873301, -0.017572956116725144, -0.001205036842051127], [0.0627995951322451, -0.000655783055267249, -0.0029274019259593466, 0.0011921740543464302, -0.0615460920852858, -0.004086301643888478], [0.03718340810566333, 0.002370134017970966, -0.0035727445806275704, -0.011658470941653057, -0.0435252682687879, 0.000669608334100898], [-0.016935160493163678, -0.000505413741310189, -0.00163454489602952, 0.010838489716699927, -0.025682913560021912, -0.0005661713118889887], [0.03423942448228068, 0.0022092183360785784, 0.0010567826533097334, -0.012050522495289227, -0.05405747581736068, -0.001597427159019145], [-0.014026649384040628, -0.00030786034156769613, -0.002090125345388596, -0.0018307506282312796, -0.016575664802169817, -0.0003689494986020207], [0.03230795429837731, -0.0008616082328616022, 0.0030947664531582432, -0.01264244956896468, -0.05586879093820931, -0.001229872011499932], [0.02129237038182178, -0.0022154743498984516, 0.001165065597143284, 0.019414669153972917, -0.06495400488554762, -0.002235959230825214], [-0.016460177661891638, -0.0004746693274214705, -0.0022912092075198223, 0.005014325194460268, -0.02124436696972901, 0.0002560979721016051], [-0.08129187793845755, -0.0055311749555134624, -0.004030401797702837, -0.04706131997255213, 0.1010454313613214, 0.0026693433029045853], [0.010783604972002675, 0.000815887675105158, -0.0017030333176928723, -0.006189601143450513, -0.03493004982499362, 0.005219620210457622], [-0.029615283379899625, -0.0024363158785845086, 0.010301958466018144, 0.016968955414663235, -0.0173139973339006, -0.00010531728829667582], [0.03067932200462653, 3.442688184382712e-06, -0.0018723505324687386, -0.013476222780698698, -0.04974559215710393, -0.0007885992225395065], [-0.004010723234017734, 0.00013152139931278973, -0.0015678233313550624, -0.009431946474548548, -0.0190835036794464, -0.001237524679945088], [-0.01609905862036735, -0.0007254597419907858, 0.0018003481480018443, 0.0014001211605249623, -0.019551240714611742, 0.0004752897684430057], [-0.01688092254687408, 4.772858750948682e-05, 0.0010081047885514682, 0.0009483278822311767, -0.021280368180357296, 0.000957129468939159], [0.017379624579733657, -0.0005212097455147476, 0.0010675616947435562, 0.011331349213280648, -0.059545511185424144, -0.0021176969097602273], [0.01719459440944582, -0.0015249550365949297, 0.0017581216280161981, 0.011226130239429772, -0.060569975329438926, 4.941742247530924e-05], [-0.018338911189494624, -0.0012640493057630724, -0.001437518362628847, 0.016417252951395803, -0.030859528362456944, 0.0017113256975190476], [-0.09182841481023486, 0.004351864702726451, 0.003688731139717393, -0.04430882424132746, 0.10286991621684406, 0.0025267269922745225], [-0.01382427838965443, 0.0006404425492313623, -0.000681373401513017, -0.008053495506938749, -0.01728875125142029, 0.004007456000295076], [0.2006172609933752, 0.01859629895084215, -0.0058222301973921186, 0.1298022632404116, 0.4195106278449927, -0.0023680396931069023], [0.009494352798502078, -0.0008059800696150842, -0.002152920863680662, -0.013357645776152748, -0.02814908047107034, -0.0002287256179833189], [-0.00998406194046862, -0.0002193058458944996, 4.946374448418587e-05, -0.007936990797194256, -0.016260672548470717, -0.0008484326124561347], [-0.015365446238486872, -0.0012773249779951255, 0.0008919201995167956, 0.015551874090643274, -0.03310825255056462, -0.0018927705231135252], [-0.011959514324015817, -0.00018481277786968345, -0.0016145647739937185, -0.007082691093578032, -0.014303473047438783, -5.4943983104010606e-05], [-0.006531488377479901, -0.0008189535932537215, -0.001251498950287842, -0.006805248309322569, -0.01827928736023038, -0.0015135234094256248], [-0.015957388291539244, 0.0010192895328768024, -0.0012224590874124264, -0.0008277185512421394, -0.017864963643283552, -0.0003467599593995236], [-0.01735617858929682, 0.0008210030825127533, -0.0011312842977188177, 0.00623159745837385, -0.02222958327503582, -0.0015355543788351674], [-0.012150517039255989, -0.0007226752446907886, -0.0007972348441256147, -0.007020241838334393, -0.013975900093915881, -0.0005334309396773705], [-0.014046376222528552, -0.0006748682698735115, 0.005874368521810335, 0.00023389393992328075, -0.02741976417427639, 0.0008327462049447716], [-0.011951015707646957, -0.0002671115924818284, -0.0007219796098792507, -0.007101617686684087, -0.01397121567662158, -0.0011870597266863453], [0.026455050590478948, 0.01668235678464115, 0.02237639362687201, 0.10389820544148265, 0.31787909626928457, 0.044717230620574075], [0.012240772178591819, 0.0009365848801276795, -0.0013669355375844404, -0.015103461904577384, -0.03395258456508359, 0.002045624948525887], [-0.11124569504966351, -0.003215552347030414, -0.002463337031238723, 0.05502628928042644, 0.11577949095242067, -0.004081195804914296], [-0.012038896189004816, -0.0005818769605589292, -0.0008441537760297866, -0.006889589709204256, -0.01384757491929052, -0.0009979084459117308], [-0.013074825608297178, -0.0006208269544812967, 0.0028682575405711156, -0.007263382131192454, -0.016415904844530364, -0.0006933180020699195], [-0.06633552238806895, 0.0037567845668332383, -0.0037368386496878982, -0.05163644485197079, 0.09901042722185929, -0.0063417392322982526], [0.18336687610224145, 0.01662772387414758, 0.040808723685442076, -0.0013892297070404848, 0.3496296792551873, 0.00048836964716615], [-0.0178536734737869, -0.0005388013402887889, 0.0006233204593538357, 0.005469625385397114, -0.022194066094480318, -0.0007064049361950207], [-0.004091643924733847, -0.0005626913458246007, -0.0017592362319837406, -0.0099768859300306, -0.017981844005076277, -0.0008276985623509801], [-0.012181640855334597, 0.0006463255245690973, -0.0008548641919609072, -0.007734522663764763, -0.014435577113277937, -0.0006397207002309301], [0.016411964188803075, 0.00014056906731255262, -0.0024873916271844648, 0.011951712429496168, -0.05604906514039703, 0.002808401558160023], [-0.06134754666204744, -0.002218845160853901, 0.00066065792259967, -0.028179124778262198, 0.06102021980769214, -0.004135361129128181], [0.008227718518645174, 0.0003025724279489679, -0.0014425632988845566, -0.0042832258921560356, -0.04140084079448273, 0.0033963390389291806], [0.03216386449105203, -0.013937139255532471, -0.03054796626858057, -0.05546444029603579, 0.18485546679285794, -0.018781690225665837], [-0.014647440561564435, -0.0007039772160265928, -0.0008349093358595136, 0.0004649729968035017, -0.018572875132533428, -0.0009057707508196191], [0.041907899134259956, 0.0002929772861681875, -0.0013710339789742884, -0.00940119473852029, -0.05623211816856878, 0.01202013713230185], [0.0316823329130688, -0.001909483683106772, -0.0022804930169053415, -0.011526485723323002, -0.046416120080253365, -0.0017497504094803333], [0.020367480929551783, 0.00506838020852091, 0.0006384835090520656, 0.0017848956739186489, -0.055017256939278815, 0.002208016618235314], [-0.005692228607745735, -0.0012315886106153413, 0.0007135999437623756, -0.009776185016901437, -0.019367047974071926, 0.00015345026557201454], [-0.012108353506505411, 0.0001316477620626281, -0.0012544057852831051, -0.007306533760632947, -0.014024406153455748, -0.0006379485561854595], [-0.0038632568495940186, -0.0014984622034892381, -0.0011741672564172469, 0.00889246753237749, -0.038090387271782496, 0.0022004727155721664], [-0.10343118060270823, 0.0027036411951231482, -0.015123211614198296, 0.013766439287697095, 0.14356889062341366, 0.0045376433328948645], [0.005318643515183502, -0.003107799404886922, -0.0010935174926208035, 0.03753908281554382, -0.050678431085437625, 0.0001553549855513548], [-0.02057561560740783, 0.003157677162312318, 0.0002955500663560277, 0.006714793861609087, -0.025415619035119744, 0.0006232135522500921], [0.014876965538489731, -0.00020976092789512505, -0.0010434498709561981, -0.014016482757423575, -0.0344669634470376, -0.0003403085351772318], [-0.11763374229002375, -0.0010220618470808947, -0.001392594419211916, -0.008006731320314448, 0.13258801892416214, 0.006148063333421107], [-0.012050199697815516, 0.00012822277386890146, -0.001623774165590397, -0.006775647891268177, -0.01407928179279623, -0.0007993192263986141], [-0.012780085284737104, 0.0008099068696720899, 0.000510139997023224, -0.007703517086288352, -0.015731976011086578, -0.0003044684845833474], [-0.010618895725352004, -0.0004943273758349614, -0.001801352887476996, -0.004602021472145042, -0.015738870454240484, -0.0019445320849505118], [-0.010954823392209594, -0.0005529609300018513, -1.8278035160206866e-05, 0.007425598387063209, -0.027122125567861553, -0.001477410461830079], [-0.017465494075870804, 0.00530952333580872, -0.0008180446586620946, 0.0019277534096839946, -0.018804384233285235, -0.00034935377767465644], [-0.01188757247206657, 0.0003109030513444002, -0.0014884902947019832, -0.00737016206711329, -0.014015817367551645, -0.000748860849910946], [0.012037400575915288, -0.0008021443743028384, -0.0016398729569936119, -0.010413236494366263, -0.032618223624957886, -0.0017639231252946576], [-0.014558393240270417, -0.000899133363035816, -0.00210368720970326, -0.0013523500869057814, -0.016717829589107457, 0.00043139348902268695], [0.010859827665801658, -0.000802837930467624, -0.003155855337672442, -0.012575896476574235, -0.02780923788027782, -0.0017160000408095028], [0.050102121655177634, -0.0020620838052256537, 0.000751699296837581, 0.00576234885522156, -0.0724206657208403, 0.012440389242638664], [0.017013257198479115, -0.00029045900988643343, 0.0030725858325415995, -0.007358273635423053, -0.040772675114335265, -7.87209856616933e-05], [-0.012364777907246527, -1.2799303223042872e-05, 0.0003115506146504874, -0.007244146677724844, -0.014852889258117405, -0.0010369374683387332], [-0.08339202141580984, 0.006312684770495431, -0.005070674322868707, -0.0414064159242149, 0.11071365797027176, -0.0013572310778734674], [0.022937480384193885, 0.001233534008725373, -0.002525944066865875, 0.01148272317610803, -0.05820381812480073, -0.0023298577303018566], [-0.09477364641044797, 0.005750221830652539, 0.0019104737653669603, -0.03356963925594348, 0.11996124388719161, -0.0036453204834863847], [0.03322578503657091, 0.0009918578282691382, -0.0010501446199756796, 0.007513729004158303, -0.0769976756598996, 0.0011164484108770604], [-0.018068878889382998, 0.0008876495880284576, -0.0019099110911525111, 0.010190306870482951, -0.02634434257260226, 4.517609462631156e-05], [-0.1455854930513283, -0.0053398879775363095, -0.0015307389683897339, 0.023673468252317014, 0.12478957535560539, -0.013349780753524991], [0.008630276452363994, 0.00029700041301058717, 0.0005914221841927419, -0.01125413428944699, -0.03577220382957002, 0.003974305736116303], [-0.051586133104814094, -0.0005816223588249013, -0.0037998735338504973, -0.02392097943684404, 0.05107754310316399, -0.002888934668830432], [-0.011899918914824354, -0.0008575053257857387, -0.0011845131210311793, 0.0004644444336835113, -0.021890821392241144, 0.00016831432019883406], [-0.018453818933306852, -0.0017540800100954844, 0.0007323834728363246, 0.014645004782627874, -0.029360710268709966, -0.001008779043351973], [-0.012256379074339597, -0.00025246610644394363, -0.0019000102957069513, -0.0015526541127723224, -0.019380045104542595, 0.00014155469380534825], [-0.012418219155790878, 0.0005050780101947674, 0.000468453951047057, -0.008497845561717055, -0.016007680237309342, 0.0007502129935754194], [0.006595055409520356, -0.000963677743430341, -0.0016416134184576816, -0.008119224941162604, -0.027817701389182787, -0.0007528379172869898], [-0.014181020795276915, 0.0017889747366480982, -0.0002581685863239266, -0.0029819527708106574, -0.016878037769638924, -0.00018979481459772642], [-0.011659007806028705, -1.2131235792586925e-05, 0.0003755958343179992, -0.007827949175545296, -0.016282065917215562, 0.0002055583002641114], [-0.01438851943130669, 0.0009726513621535868, 0.008383156179869272, -0.008077618593199712, -0.020606826310909217, 0.00018382346005938624], [0.038812154350816676, -0.0021951839128655204, 0.0006506110494042233, 0.0007437888212684354, -0.06371192798121543, -0.001082775660741575], [-0.012332214602875119, -0.0002852360875939504, -0.0018168433607192022, -0.006445311727114029, -0.013881271672031079, -0.00043912254966664967], [-0.1300487163097938, -0.007705300396152193, -0.014549108955936231, 0.09329825546696538, 0.172767601765382, -0.0033972553799890785], [-0.015811976249990572, -6.50165478256612e-05, 0.0009561795882690705, -0.0008230234890810109, -0.02064075560886642, 0.0011845923074945259], [-0.08299617955726515, -0.005799459836139471, -0.005909566055402324, 0.0035994935207241986, 0.06336171040885044, 4.400151923215777e-05], [-0.014740619637050903, -0.0003740152697828196, 0.00106783512813213, -0.0012086200847099566, -0.019201417903632747, -0.0007431622329557662], [-0.015205318578854338, -0.00026974204278590287, 0.0002521683079091466, 0.0009132969371660839, -0.018018542203162957, -0.00037186242027206805], [-0.08279214985938758, -0.0019823787028622653, -0.0005531341949970345, -0.04188594523033544, 0.10971162133481806, 0.0016829390337166023], [-0.013029826849558826, -1.2211488117592107e-05, -0.0004472679163333261, -0.0077343443874834175, -0.015544609300057896, 0.0015682599415510077], [0.03503910951495933, -0.002232869448109614, -0.0034512270004945075, 0.005462822393549759, -0.06842914378690343, 0.0009113083269984774], [-0.004133650067326633, -0.0008388458412749871, 0.0006298304595068817, 0.0017201292316291806, -0.030286809452190696, -0.000623987663677111], [-0.09190130537588767, 0.001544408209583473, -0.0035013498149393473, -0.024974032335465485, 0.09591427761153276, -0.005281998294823904], [-0.017024359775836505, 2.761807443005754e-05, -0.0014810534417435293, 0.0026851612386512747, -0.019500504821631272, 9.313872612988223e-05], [0.03711422859995285, -0.003925158799573879, 0.0012302927377398575, 0.006572441146714428, -0.0718609919445859, 0.002335854926419271], [0.0343341954837527, -0.001962590764802772, -0.0014160722123118955, 0.014471185724090108, -0.07631953412043614, 0.0005737682706603127], [-0.01363956227919785, -0.0009857383260611864, -0.0013439718364496205, -0.0012062799728163123, -0.020667118740947257, 0.0026426711554721365], [0.035062008209112934, 0.002597289709415175, -0.0020605813727824886, -0.013649953323902946, -0.05175017248402961, -0.001648590737813083], [-0.012340207997617491, -7.921860744002621e-06, -7.806484496102033e-05, -0.0018697984893253441, -0.01990529613650124, -0.0009987106708509658], [-0.01706846814268286, -0.001049857075724804, 0.00076378631997645, 0.004374322064792756, -0.0219382952773998, -0.00028148788896179775], [0.023371248237200926, 0.0020286935445420003, -0.00048678101583661293, 0.014510347388586812, -0.06559913111902116, -2.437703547201235e-05], [-0.010777322741335159, -0.0007299712057756245, 0.0010171303198836484, -0.00866624623716746, -0.016916567518226452, 0.0008729773826209947], [-0.015347814544151022, -0.00032500414233604637, -0.0012251722533409117, 0.0007700044742723872, -0.01842238085678752, -0.0006496326776569549], [0.011350340057145712, -0.0005686274409177432, -0.005620794958080978, -0.012184047962606392, -0.02647053665118507, -0.0017063330443555476], [0.012975849787875804, 3.714227343259919e-05, -0.0027088330920922135, -0.010392408946298287, -0.036815440969039416, 0.0017036909461215076], [-0.01498481151245034, -0.0008612946705482614, 0.006785339067319031, 0.004146617235557425, -0.02815464657348154, -0.0021312035463964176], [0.03330354117556959, -0.0022086696754807836, 0.001949576397163526, 0.012362387139624967, -0.07274185974382204, 0.0013016913736114248], [-0.09268261832782496, -0.001588862050245455, -0.002748680028465812, -0.028319097245833046, 0.0952618641775097, -0.0010749874775214372], [-0.014603761225374788, 0.0019249463596404041, 0.0007149280153734537, -0.0029542403264633467, -0.020183758467785883, -9.811435538986993e-05], [0.012477074574272386, -0.0008474965231697949, -0.0018090258121559238, -0.0120004714567072, -0.03222507717417116, -0.0007950036080683205], [-0.02338420377877499, -0.0014763585446463995, 0.0013417538667874894, 0.022772329734606033, -0.03129370885209451, -0.0006598124258776289], [-0.01263191491718262, -0.0002785234976660339, 3.342544581444953e-05, 0.00034837217038626586, -0.02071608949516098, -0.0019552697061911276], [-0.005854563578799375, 0.0003981404336031089, -0.000421684553621957, -0.009431435794553628, -0.019056665870288734, -0.000833790636339471], [0.024766210155939525, -0.000458912251773792, 0.002057144231867644, 0.0128083960181492, -0.05921534180983753, -0.0017051153919641134], [0.015423482529346255, -0.0011983323241977245, -0.0018069962485615047, 0.012763964238104898, -0.058378221500984304, -0.0020038966937076922], [0.03410875822953789, -0.0011404256584115558, -6.139454114814114e-05, 0.012204973450163237, -0.08054776403613355, 0.0010691858893255757], [-0.019970381012119492, -0.00116559901888593, 0.0006828301344130983, 0.015448774484758836, -0.030123864689069402, -7.175989909711654e-05], [-0.017502577212875434, -0.0009007194802904468, -0.00096552253661479, 0.005363698817474243, -0.022458092804690848, 0.0012632132169972182], [-0.012668455321393704, -0.0009295052406640364, -0.00187557869796395, -0.005158566077727985, -0.015122812501416378, 0.0005549178391660102], [-0.003150013859744764, -0.001364865297290071, 0.0008720389877039684, -0.003999606557661664, -0.025105481192104998, 0.0012979279190974964], [0.019152257508287664, -0.0007111515760519313, -0.006345903266808109, -0.010670109504913778, -0.03139046022486343, 0.0014320337310161355], [0.011567276651193156, -7.160937671712394e-05, -0.003589815595307872, -0.009228021453283473, -0.033192466784312466, -0.0006853634415722232], [0.03231157016462547, -0.0026337723702596384, -0.004572510789945483, 0.005896724078718242, -0.06539520330354635, -0.000806807779592264], [0.010671150861239407, -0.0005483967020558858, -0.0023500271693499787, -0.002718598629378136, -0.03772026899571184, -0.0025338593647436115], [0.03637323375861592, 0.002829746508242272, 0.0010385123987726032, 0.010577284966831486, -0.07577877569707603, 0.005843331397947154], [-0.017351509116007306, 0.0005022710279960395, -0.0014825237913725898, 0.006421268574581234, -0.02335173864425015, 6.223194905271336e-05], [-0.013194353733669258, -0.0004327597298576316, 0.0007882247423270385, -0.00719486053743303, -0.014991052880153784, -0.00017519786121338518], [-0.011091122124397872, -0.0006869362927179482, -0.00014533149281354172, -0.007802103686356303, -0.015005771343893998, -0.00046873505982037437], [0.01542516749527188, -0.0003405571125774086, -0.0010584137305584493, -0.01402267758771757, -0.034519150751816906, -0.0006843683126015635], [0.029149880929246384, 0.0009126019912412105, 0.001425078108027725, -0.009135370622572176, -0.05580418212666338, -0.0017480082792798407], [-0.07335597586056773, -0.009980327552910834, 0.0018112882376819578, -0.017441315233911552, 0.1486225056868236, -0.01120141337235343], [-0.08036674598254337, -0.006258842812305075, -0.0040785150567788555, -0.0306077473544455, 0.09650412518384943, -0.002892273977776741], [0.03585753387816148, -0.0005339296779412718, -0.002472806620291939, 0.005891365696031638, -0.07008457037566684, 0.0014201848774848397], [-0.012588917379681933, -0.000695217384702995, -0.0019389778654871125, -0.005438170255019468, -0.0142327259551301, -0.0003059911599784321], [0.013750963704243583, -0.0017232125039559757, -0.002769913724932692, 0.0023649727769111924, -0.04698010848903534, 0.00015729823676913544], [0.009720058843508248, -0.0009088204189540036, 0.00010985252218450221, -0.0017669103323943373, -0.03914091617918493, 0.005120068898173814], [-0.014371574543859102, -0.00030110745661072645, -0.0012077898135942904, 0.019863081060108104, -0.032332764330850944, -0.0001831782485264445], [0.016288180128543044, -0.0015920970923960856, -0.002733979718588052, -0.021311924366291455, -0.01693546125582683, -0.001414717695440634], [0.000407582079514662, -0.0019591305754750573, -0.003007199442732135, 0.03033837556819721, -0.04625989557550831, 0.0001493155650511894], [0.017867744897513256, 0.0013048250013125194, 0.0021854622349514055, -0.008876221786678913, -0.04661831478966636, 0.0014365044425679826], [0.009626530894039117, -0.0004187875790830155, -0.0005603241379918213, -0.01392434370258068, -0.02959012083793468, -0.00033295463644895907], [-0.014020488153430688, 0.00021802200366136159, -0.0010889985222251606, -0.003174158755258885, -0.015978700061177045, -0.001155676511569636], [-0.022372674673596903, 0.009122281448513143, 0.0006672700959268247, 0.031143798316132447, -0.03686908571086447, -0.0002249228094442978], [-0.017161838708265, -0.00042294369829798254, 0.0004400484723081555, 0.006024995674751075, -0.022614294895835264, -0.0014659668446610683], [0.01915150572587655, -0.001508514245726198, -0.002035179897170708, 0.011200108541360975, -0.0565197783873956, 0.000345191596388208], [-0.017344871317584637, -0.00029535305498687816, -1.6023108728222449e-06, 0.0054633217262168065, -0.02188468515013891, -0.0011368098926336529], [0.04334272226052492, 0.00042118439805546507, -0.0011380945702173827, -0.006262621162998521, -0.058695782335951054, 0.013465924743919892], [-0.014387288032819993, -0.00046667698772261384, 0.00020222897553281744, 0.0001485182718368429, -0.019459906735107127, -0.0012368754917199871], [-0.015770381946818, 4.109720467308432e-05, -0.0020320779946689372, -0.0007724009157356777, -0.017534338210561277, 0.0008681018631107335], [0.033111385800351334, -0.002175689081345795, -0.0023079436140692605, 0.014217291387401758, -0.07573635003473499, -0.0006420277909364075], [0.012924779905695571, -0.0011073965385398133, 0.0010610912060753889, -0.002579886819121438, -0.04777388873062857, 0.0022753009765187644], [-0.012344215915429406, -0.0006782713027572682, -0.0002008819867586981, -0.005866799791728896, -0.016634399463019, 0.0005245684596931904], [-0.012288965992987557, -2.3684861777620573e-05, 0.0002365767823534974, -0.007171249488049387, -0.01481665062101934, 0.00011397418148035535], [0.10592985893180418, 0.020658632922003376, 0.01522531987740237, 0.02840352160722034, 0.2189473022379964, 0.012520285058493901], [-0.016027483586107786, -0.0008434759676709874, 0.0013729583812932896, 0.0003692935110911026, -0.019294905483805257, -0.0007763868548004172], [-0.013170745907248312, 0.001828963679161752, -0.0014341140410269976, -0.008323044600618033, -0.015570528010583793, 0.001469468880315364], [-0.015043011487138474, -0.0004542299486907014, -0.0006754363795361962, 0.0011572235887206761, -0.018557453690663302, -0.0009128063684063925], [-0.007773381538110461, -3.884234323798438e-05, 0.0006649607760132374, -0.000624721122913684, -0.02588295929552096, -0.0015450564762301798], [-0.015904637493773882, -0.0009757745527988023, -0.002218743206724281, 0.005255991234447467, -0.02020142787652135, -0.0011554081046292067], [0.018308040560480892, -0.0009138030083327724, -0.0008107516837637652, 0.012000932003424631, -0.05585324326965049, -0.0026370569550997966], [-0.014090788650622741, 0.00041084025745536594, 0.00022945559123011027, -0.0002829524776190696, -0.019624306707532835, -0.001842248012910893], [-0.0036948817605014, -0.0011456179276877999, 0.005766072399229541, -0.010194472190828786, -0.02208771444247821, 0.00015661392226658377], [-0.012059069628156033, -7.431355720176967e-05, 0.00018750283182798698, -0.007089288181911297, -0.015121526786188295, -0.0010433046783706415], [0.010773909820467132, -0.001557821941985951, -0.0018018703919578, 0.026040063208147866, -0.058482719714760836, 0.0001617723534228545], [-0.12203178988017553, -0.0008766512231471992, 0.00308710219435599, 0.001348540741855856, 0.11714407255231137, -0.004204607718533831], [0.010648666968357154, -0.0016581574785913462, 0.002718702616163247, 0.021104464324360344, -0.057317156942401026, -0.002363186154555102], [-0.13140410666421043, -0.008044383984805059, -0.011958226635206256, 0.011144031658582932, 0.11852292448158507, 0.0061230944773870985], [0.0029153215150058225, 0.002072814885550876, 0.010377300738499113, 0.008835382029010744, -0.048546109430822514, 0.005740528357994051], [0.039724788762291276, -0.0013401793536697894, -0.0014604016678543732, 0.009084016677700736, -0.07179490879808183, -0.00312760133467176], [0.024937102437524426, 5.808544154347089e-06, -0.0007122022750898038, -0.01696605333157975, -0.037821929576502794, 0.0013572742014935325], [-0.012651236506010516, -0.00044852967127415814, -0.0006669621558575676, 0.00030940686161923616, -0.022740759600073816, 0.0009980810715967585], [-0.012156479013246804, -0.0002831234341434832, -0.0009440641803298412, -0.0071050930916034686, -0.01401183744403706, -0.0006994028366394024], [-0.013363426250092517, -0.0008309238403220512, -0.0007191164223032274, -0.004318113249641064, -0.015345074342131714, -0.0006233458955094952], [0.05414831127992588, -0.003360790440625269, 0.00014977424996742044, 0.002616484335434055, -0.07202418761577191, -0.003979591808930069], [0.0365726750398247, 0.0017005107939669648, -0.002210712656066953, -0.01401774910262119, -0.05328072666802853, 0.0010360025929248907], [-0.01925331913138243, 0.003925357379625609, -0.00046488024307148854, 0.009326568904185982, -0.023219501294624217, 0.0011524410519331576], [-0.012223937738694112, -0.0005506833163627901, -0.00029985198600195, -0.008062381780569425, -0.013794770786046566, -0.0002683743923251903], [-0.017877403432308942, 0.001229525959328588, -0.0007274180233079046, 0.005662164050860083, -0.022605508195368954, -0.0008813603592029444], [0.0060942461621682825, -0.0006775199160619995, 0.0014946716398134775, -0.007249578497291248, -0.030949199821537317, -0.0014126195670912172], [0.015800041622880456, -0.001532253739074575, 0.0011852442924990575, -0.015003605352498567, -0.03757092790329382, 0.005254834412820723], [-0.012471014500352571, 0.0006887738428457801, 0.0003288906024092656, -0.00880465720400004, -0.016572934221626322, 0.0016309414807238291], [0.00694529552992932, -0.0009066237792177239, -0.0022887571857395715, -0.009306967186916355, -0.03094998432392531, 0.001307036945869614], [-0.012762574609401962, 0.001161769047355534, -0.0018728954724012428, -0.0074414938949441305, -0.014731741274075094, 0.00044693620346687127], [0.007662757372901262, -0.0009960572114643478, -0.0007789032953671137, -0.010593819016046526, -0.029946714556887296, -0.000547263293136011], [-0.0009890088361335187, -0.00012561801032486157, 0.0002253980139569553, 0.008182238755733293, -0.042359980841257736, 0.00153363758469251], [0.019765459110981255, -0.00012594605586307915, -0.0017952958365981585, -0.002953620653770358, -0.05195275669020613, 0.0018621601254563364], [-0.012435504381397617, 0.004749228462327196, -0.0019001872236650698, -0.007064288547441018, -0.013615254802682686, -0.0011839935071408384], [0.019655725635499166, -0.0018242640367372215, 0.0015343290125806837, 0.0017348025339223305, -0.05534596194900783, -0.000954631196257358], [-0.015840177476178833, -0.0005291914142276691, -0.0010523247404581756, -1.553310425663456e-05, -0.018461363928248008, 0.0006985906633692607], [-0.01769771728278664, -0.0017468624894279648, -0.00020994092215051587, 0.01472147018213039, -0.030271515526989805, 4.566039224418459e-06], [0.03397920002153903, -0.0014339186910728033, -0.0038877702920444984, 0.007007479237301039, -0.06893075739801617, -0.0019342328777064842], [-0.12490617261354323, -0.001520280915943463, -0.010121933299423161, -0.0002869088851352924, 0.11537455944791417, 0.0024274029327977977], [-0.017352796605049146, -0.0005712374537521318, -0.0008660202644363742, 0.005935710403925597, -0.02399533061879354, 0.00164967453810554], [0.03500403653199308, 0.0025022045595760618, -0.0007731605350588866, -0.002433506383958012, -0.06720236857018606, 0.0024527943976337817], [-0.013808577321702988, -0.0007170399539972106, -0.002512118653983697, -0.000968293213369378, -0.01776941587234079, 0.0005754450153940177], [-0.018727052881262623, -0.0008191930982350196, 0.002557124639703031, 0.013223098449040334, -0.030095116457445045, 0.0011611393481992633], [0.011845571925671664, 7.524801164480957e-06, -0.0029586535496105583, -0.01305916606296711, -0.029786954706125798, -0.0012483224081326892], [-0.016523471910621194, -0.00034525577118331647, -0.002189698057457607, 0.005473307375210241, -0.01971735584615967, -0.001897525789788494], [-0.017255330748962683, 0.0014379100133636807, 0.0008779475723352985, -0.0018296235412642537, -0.01945585392260264, 0.0010249506271305206], [0.00953331033323811, -5.113024242215241e-05, -0.0027730995377789535, -0.006364787623573558, -0.03822449414691733, 0.0026802012174538064], [0.016745861191159145, -0.0008955366267641679, -0.0022316361963177324, -0.009673430939274025, -0.03922787066626937, 8.261323746610379e-05], [-0.010116735283136323, 0.0008111837914984661, -0.000556058309865843, -0.0008388954313151938, -0.0230542698605146, -0.0014452249066665695], [-0.013631746855915339, -0.00042860584407498756, -0.00034073607174928643, -0.0018801587136955463, -0.017577230704019994, -0.0013415218105448962], [-0.08383333733031859, -0.006805796887449682, -0.0024962886321638535, -0.03288497680056592, 0.1196687261707633, -0.0005149931869320436], [-0.005469326248498302, 0.0005724556595186793, -0.003053433884241128, -0.005173295507852062, -0.01935483936730977, -0.0007215606516174547], [0.006126887189243861, -0.00028013023927346336, 0.0009495729245685367, -0.012266559183165198, -0.02820500918173501, -0.0015247615096388178], [0.027929271515240515, -0.0009652619998321271, 0.0012827036583354615, -0.010528884488573089, -0.0486205169597264, 0.0008693549412222508], [0.29062861047065647, 0.016117904451019887, 0.052257986609150965, -0.03766805317695936, 0.4145163065480348, 0.0036748785757293673], [0.03311264929803203, -0.0025754998042197205, -0.003262626365720073, 0.0024655626969779056, -0.06110797056931323, 0.00016788474424305304], [0.03312522080937251, -0.0008491483219884682, 0.0005081880595956772, 0.014210246643640012, -0.07914033894013035, 0.0006958317495106637], [-0.0037445680469408806, -0.0001387132264871371, -0.0006038407891799558, -0.00812863077568033, -0.021668178937354354, -0.0009160682243574034], [0.035466817497284434, -0.00037086898634031373, -0.001627392080616143, -0.013055812371776417, -0.054595964863968706, 0.0014832208054171023], [-0.011740256284164701, -0.0007417092806378769, 0.0007531131279062177, -0.005949274784141391, -0.016468244770088663, -0.001053628008873637], [-0.01488933711870072, 0.00037077828798481643, -0.0015562925320527132, 0.0002473684382277413, -0.016913187754512896, -0.0007926626542796185], [-0.01837176944344065, -0.000161647926014949, -0.0011047202496877121, 0.005665662496453204, -0.020222191281526355, -0.0010053335957835709], [-0.0023489537267885325, -0.0009661525728417422, -0.0012885394586110487, 0.008008456497181017, -0.035518435245952225, -0.0030863754929875043], [-0.01779723102696687, -0.0011060528236795494, 0.0008495644640048087, 0.0065856764871920145, -0.024536277713873855, 0.000804320613323411], [-0.013964169112425207, -0.001124163457250492, -0.0012138542839501031, 0.006941042888243229, -0.02618600352976856, 0.00034714749515106296], [-0.014301523662644885, -7.659676077144849e-05, 0.0008416254963512184, -0.003754241096561886, -0.01724260921744528, -0.0006666547589277452], [-0.001906499956809836, 0.003256367865733953, -0.004843588635395113, 0.03367059730094251, -0.03120541220959024, -0.002671464364881381], [-0.0737145634482143, -0.00429076610833958, -0.006699240870684391, -0.035857910761505934, 0.10932413839081491, 0.0012605650201514863], [0.017511629991578005, 0.0005248962053873574, 0.0013018242975161827, -0.0018895916760818011, -0.049501466288159574, 0.0018527074697596941], [-0.003366359221303739, -0.0008289287306182846, 0.0010733425710953391, 0.010192435752001561, -0.03889874734937788, -0.003371743021796995], [-0.01662554343015475, -0.0010971024802248514, -0.000799218979360769, 0.005268742906233093, -0.02098571696043628, -0.0009611610560565202], [-0.011632457569674938, -0.0006411571296429314, -0.001545267887608589, -0.0069065149026178, -0.013537825494009772, -0.0009367770164460213], [0.0422857080534431, -0.0018352574376570472, 0.0050854691371009595, 0.0021226039800730763, -0.07052852493321636, -0.0029966654664101947], [0.03422089472424935, -0.0013697559486052962, 0.002672194279724527, 0.005770108071566335, -0.06705473442120374, 0.0008390710720466568], [0.013907070063528206, 0.000185937231244881, 0.0033464521259942456, -0.014323150826126207, -0.036468056748278126, 0.001485081486970307], [-0.017495757712862676, -0.0009836284487008261, -0.001001043277031923, 0.006242842891266842, -0.021212995404604012, -0.0007494180480674821], [-0.015139896929978962, -0.0004209322197841278, -0.0006977097247274683, 0.0009731792025820937, -0.018620781837610314, -0.0005795727761956061], [-0.15285548695622586, -0.004514962221790934, 0.00036211027683226893, 0.02298243557604316, 0.13132120342247544, -0.0037453000973339833], [0.29125397984062984, 0.008570000037681574, 0.03433574863108995, -0.03636242555756416, 0.4094216802456939, 0.01402744537389661], [0.15993328134949472, 0.0049992438572662385, 0.025895145272750022, 0.1315533092349643, 0.39600190681758746, 0.023357589658412795], [-0.015360535386876664, 0.0003174783193158126, -0.0008691396446777716, -0.0011028118507128268, -0.017570605288886595, -0.0006143861481619854], [0.0514956132227662, 0.0419070436370477, 0.010920269400900273, 0.15484452535643, 0.23561720181504467, 0.005068917996383004], [0.02185256224853214, 0.003454566210033491, -0.005555424946304904, 0.041248845564004814, -0.05465927575467106, -0.003219844750165948], [0.010772614020883071, 0.005188998012914803, 8.71520030314093e-05, 0.01532189126080548, -0.0626374483246499, 0.002316793027015084], [-0.01681274364441929, -0.0005588900642269086, -0.0004969174599051267, 0.0025446888705706995, -0.019281787166801553, -0.0005943505352179221], [-0.024266169312854264, 0.008009642081470246, 0.0006811436611954332, 0.03041706013561733, -0.03520301943309337, -0.0002553237990020476], [-0.001991408758493196, 0.002759834158761051, 0.0013293069773528108, 0.0072836196918368455, -0.03992317386975266, -0.0009081781997049156], [-0.014715210063550112, -0.00047245451170765815, -0.0016557819385464348, -0.008616803325192893, -0.010175700861337357, 0.0004359507003344066], [0.03588782841156892, -0.00030558420811107516, 0.0012696207722495305, 0.0007351215198670202, -0.07148879348442636, 0.0028684736555187034], [0.018025311800513883, -0.0002830165895689905, 0.0010365428398285816, 0.011403694546190746, -0.05906601373720533, 0.004397766854526737], [-0.01915300790792182, 0.0005794479366072993, -0.002294605368521685, 0.006295058369073383, -0.01883761814926405, -0.0017892748799731876], [0.034331716720908456, -0.0007402606332098387, -0.00282215100079111, -0.011035138643007484, -0.05040725333586594, -0.002026913108034074], [-0.058940182135705864, 0.000712524395712709, 0.00021799705765043016, -0.026391694256957254, 0.05284077431867233, 0.0006939139539609922], [0.03908005368756881, -0.00013328575436504777, -0.0036500021008830483, -0.005513648417960147, -0.05894837756188879, -0.004368073185805181], [-0.017052407352161965, -0.0010239299210117393, -0.00020684248058284554, 0.0045208781719402675, -0.020206835047673097, -0.0012308633705107389], [0.15332003270256936, 0.01526463342312603, 0.02629016230176995, 0.11258018490038228, 0.34423750244808443, -0.007094896728312971], [-0.015974733072171482, 0.0004230247979024122, -0.0007942104855467624, -6.047597806660291e-06, -0.019446143709296676, 0.0005981100669190953], [-0.012482922601485837, 0.0018912751759304976, 0.0004247241573820362, -0.008207105982630412, -0.01592662326284979, -0.000899347486346558], [-0.016115864975468616, -0.002136550478135286, -0.0014678202719988276, 0.01669733489137741, -0.032746270558358326, 0.000569171392583573], [-0.012219129970729681, 0.00010198106148788473, -0.001566414068675144, -0.0072925469942477375, -0.015044891729265097, 0.0008210017014297367], [0.03584291417341847, -0.0016704364703285614, 0.00034314528737089015, 0.010112246602467434, -0.07130308374180491, -0.00031050013683756966], [-0.015719793803302535, -0.0011462306058228782, -0.0023736903366031958, 0.006002452888226215, -0.02216903793981458, 0.0002062997973168874], [-0.004926919194878691, -0.0015014093640384237, -0.000851366042444111, 0.01017294463134844, -0.0376204772315941, 0.0007772272016069033], [0.011624124534266356, 0.0001534988886567874, -0.0037686113781806333, -0.010978061982678719, -0.02943215671760876, -0.0027987933444550855], [0.015583381592692225, 0.0004483039638359344, 0.001052022801033129, -0.0024021392044797393, -0.04980956767162919, -7.20014814524631e-05], [-0.014980975277710041, -0.00016976807545526478, 0.0005805347908132362, -0.001208935148798665, -0.018557575157485025, -0.0008632811313642872], [0.1028164715700841, -0.02122866504619271, -0.022306280189363106, -0.10193556789961904, 0.23658788069899472, -0.015225108975173542], [-0.015547756972622282, 0.0001347602960645889, -0.0021136416158745214, 8.338856347669194e-05, -0.017136973076598082, -0.0006197771944464366], [-0.016995311378235116, 0.0016661005554795857, 0.002480243745224779, 0.013469423618542682, -0.027232634286972705, 0.001412177745960675], [-0.0120224607598339, 0.0010438367469413233, -0.0010474288856928374, -0.007404362065440439, -0.01497279014569787, 0.00045320510972367755], [-0.014900414691115098, -0.0005067075289044218, 0.00039769556488662, -0.005261884529677247, -0.016459949502582165, 0.0015312606873922243], [0.03547857020359344, 0.0001233712769594759, -0.0016969834431964101, -0.009284037653697626, -0.05505229258518329, -0.0037686277984756455], [0.03514062158710669, 0.0008702377028140671, -0.0027616109652696262, 0.0011157047583545577, -0.06496014462257509, -0.0008548084604305977], [-0.013993666466562145, -0.00038562987267112435, -0.0021254763899774707, -0.001355192672729482, -0.0164923419262435, -0.0008476926718163067], [0.017763882148754535, -0.002203976651562672, -0.0014105053230242855, 0.012658615811995726, -0.06159954103992576, 0.001972477434714779], [0.013774149301619304, 0.002358652701536701, -0.003947201489649539, -0.0012168698802625304, -0.039967007895387825, 0.0007982772621438633], [0.034635108863256964, -0.0018433960005813934, -0.00128426140134896, 0.0025762812019479607, -0.06653041894967401, -0.0027533137136004585], [0.019275243653842984, -0.002400247708983614, -0.003018025104625973, 0.0025170907834468513, -0.05324719677002185, 0.0016731351463414703], [-0.016465683175877145, 0.0004862752022633183, 0.0005305477974480925, 0.0002210335282213918, -0.020392819993601336, 0.0004206466415455931], [-0.01694560702233627, -0.0008795914011542109, -0.0014651710857657083, 0.005216546574400061, -0.021116108453954736, -1.0068611189174167e-05], [-0.014352881503348647, -0.00015677824888148275, 0.000721916156007484, -0.001180368195727403, -0.0194574374251871, -0.0007744507828629334], [0.01613236559894498, 0.006055788406994531, 0.0008273645800235926, 0.020764385549870278, -0.06328810781198121, 0.00014153700948114572], [-0.019856659556141285, -0.0004382801249715375, -0.003119504175952279, 0.02188556563328793, -0.0292716428505793, -0.0010661455923101847], [-0.017308425275108442, -0.00011610410138257149, 0.0003216825345394998, 0.0053636445369963015, -0.023657524539370332, 0.00019672684432551485], [0.0194358633560691, -0.00027417874198374493, 0.004474203924882477, 0.0018784513869265173, -0.054582399297296226, -0.001965273961931579], [-0.014247673136962105, -0.0009421446135291654, 0.00028600597085924964, 0.005932114432267715, -0.02486378798278213, -0.0013645146698536155], [-0.013828155767508659, -0.00022264698848902315, -0.001874997263967936, -0.007525131844347783, -0.012514610758860649, 0.0007655426231740071], [-0.015719690951143027, -0.000540721423408973, 0.0009363385172508859, -8.596312058827221e-06, -0.019322511898812015, -0.0005448179318281122], [0.010358149370776747, 0.0016173914091035797, -0.0016886459600724242, -0.013217513035889824, -0.03414164986956844, 0.0018722680856503198], [-0.020150524415717748, 0.001978516522812124, -0.0010103449506353051, 0.00682353644422428, -0.022464592452035255, 0.0006234088513518703], [0.036552414282305816, -0.0009222812396539925, -0.003992826320318077, -0.00870708800718594, -0.05171043366200434, -0.0016697850531434341], [-0.012027581540753517, 0.00010591118900288201, 0.000506838413219659, -0.006098026254379939, -0.01873095000255196, 0.0010438081954628079], [-0.01256162583803686, 0.0018150782018568012, -0.0009595157233981724, -0.008122524057872926, -0.014853588843744658, -0.0005178237388042169], [-0.012582432904149534, -0.0003582096328496684, 0.0010241510939595327, -0.007359153060637031, -0.015114868097623092, -0.0008094873987002853], [-0.015467200135866478, -0.0006740965934308328, -0.0005819059506342349, 0.0013665330959288218, -0.01790647587426566, -0.0011035212083983473], [-0.012483215933433978, -0.0006227887665416149, -0.0011381906672506767, -0.0072714880139207695, -0.014295641363152926, 0.0006113247442999179], [-0.017298880748802786, -0.000313745374106812, 0.0004580899118428186, 0.005583753077782312, -0.023725251508868547, 0.0013460346421529442], [-0.012664339242680553, 0.0001329668859492495, -0.00044773696335088405, -0.007576385212599517, -0.014973211927208794, 0.00032870645989048237], [-0.1300991627097872, -0.0016190347359536159, 0.0010874833233481423, -0.002986024456727646, 0.11991404606401514, -0.001997307484895084], [0.030741750089317812, -0.00025805068495213896, -0.00299862678491255, -0.011729529472313633, -0.04756893318219047, 0.0007800567017175411], [-0.017031844718790167, 0.0015644924243329692, 0.0003490779663119891, 0.0003497866915649522, -0.020902628420675657, 0.0004711160572558286], [-0.00956220970306916, -8.874466330271742e-05, -0.0013418607968028473, -0.008534196516710777, -0.01670855923764786, 0.00103557091753331], [-0.014480594771245445, -0.0006454914531770314, 0.0003575876664306576, -0.00491741159778226, -0.01607152289599176, 0.0005574330517657797], [-0.01299459695765212, -0.0007803789960829684, 0.0008119158533046243, -0.007196185668835672, -0.015382188523507846, 0.00034143429277392316], [0.013620524901707768, -0.00024518780758970877, -0.0015919717461145547, 0.011750176978105896, -0.0582702262096507, 0.00203668388354126], [0.008765457382098557, -0.002353765699564025, -0.0023039693495153653, 0.013406357046138406, -0.040542050119280686, -0.0030053625932102727], [-0.0023113891952363645, 0.0005003829615353584, 0.001307169654613544, -0.01088084152500844, -0.022081402980335823, -0.0017339189155683489], [-0.015112895257043686, -0.00038474839239263886, -0.0006986858591530067, -0.0020608591635338405, -0.017684965488850894, 0.0007421541609739749], [0.015995294932550076, 0.0002409453938249147, -0.003129514629001428, -0.009703782384130609, -0.03616741099051151, -0.0007688656560648133], [-0.018489771738351687, 0.0010798784960106157, -0.0012751000705158524, 0.006546791871806107, -0.02417110013982517, 0.001109301580875958], [-0.01647302453610922, -4.268558137170932e-05, 0.00022616545797258137, 0.005863348672384233, -0.024871497819579022, 9.769380670307054e-05], [-0.012537297198529999, 0.0005439553725058613, 0.0003063810843163358, -0.007768122376927902, -0.015004839081196602, -0.0007400778001677449], [-0.017148810488964425, -0.0006885016870225396, 1.871144321681487e-05, 0.005518998032663467, -0.02055565568022899, 0.0009885917136689807], [-0.012508499411484077, 0.0013058000534982902, 0.0004081750816362915, -0.008127089404843139, -0.016075927706489957, -0.0002024586123174802], [-0.015309145456209193, 0.0004869298725107398, -0.0009415419616637536, -0.003285047880274932, -0.01718262814645246, 0.001031433572089524], [0.008710403340219443, 0.0013364973262323765, 0.00041743572409311686, -0.0024460002566132717, -0.03547673428120195, -0.0027416018527297373], [-0.018561195150167558, 0.0008450796937344085, -0.003055762978956832, 0.014539452146565468, -0.02791370838516545, -0.0010538653260100867], [0.03135062038936682, -0.012869455555510806, -0.013790991230950157, -0.11043149015598207, 0.16399896998953584, 0.0047090132302066556], [-0.017707732552028, 0.0012382607378409163, 0.006014921403745294, 0.006917767229023781, -0.021743575320786836, -0.002002974831128558], [0.006482324910594509, -0.00012426097271497926, 0.02882524458242708, -0.013166829266356195, -0.024583150013988146, -0.0033932498748829518], [-0.0026953395749882824, 0.0030758328287704673, 0.0005943287857565712, -0.010896476242028596, -0.021201877380567953, -0.002409801750275642], [0.027944156814278143, -0.0016286383715615603, -0.002261571672145831, 0.0013766012042173107, -0.0501459365607726, -0.0012901669695710542], [0.01725624067367691, 0.0033889865223563458, -0.0011072580832233402, -0.0011629824291809864, -0.0476629666946063, -0.001626305703308307], [-0.009633822841781181, -0.0007130128990013468, -0.0014442525427483565, -0.007791823663138279, -0.014499209945340196, -0.0011178781079906768], [-0.01802090566398303, 0.0008727631206513765, 0.026750119722313302, 0.003945198336457405, -0.021912997781921933, -0.002594098368437799], [-0.01295259600202602, 0.0002206995500686312, -0.0009304346765704805, -0.007405935953696269, -0.015071427581119609, 0.000939694663343705], [-0.0072639844972114355, -0.0013132479033292133, 0.0009397835947263602, -0.0022346924306836363, -0.02682008708266022, 0.001492228319158088], [0.03388161812874195, -0.0015420429344568834, 0.0011819302508748089, 0.012217604639025315, -0.07578974064465341, 0.0002672972271348623], [-0.022401149140204732, -0.0014503737100748074, -0.0010822466815666322, 0.021872673156926894, -0.032787284926196286, 0.0006483813011155565], [-0.012522011794747481, -9.127291870651474e-05, -0.0007941250639264657, -0.0055946788737972755, -0.015490888323107684, -0.000707023025714622], [-0.0857542494519764, 0.002843203450055868, -0.004039470086074538, -0.037534771482549335, 0.09945447732751742, 0.004080810243026957], [0.01875918118084605, -0.0013612779563551862, -0.001391537581751465, 0.013564719365678051, -0.05377681380097424, -0.001672375782606635], [0.018959467829745478, -0.002249055424413776, -0.005613314210905456, 0.01243591361625835, -0.055379037621958185, -2.0640855393161477e-05], [0.011298800037555366, -0.00047676341095053, 0.002138528050614649, -0.011719038642923112, -0.03748053846120298, 0.0010390124269065398], [-0.013046204183168994, -0.0010128675988828172, -0.0005561160385190309, -0.005590509622348741, -0.015010097263243332, 1.5794706162870463e-05], [-0.013707687811689023, 0.0014610234243962474, 0.0005075916753978147, -0.00610467512735799, -0.016515669222349108, -0.0008405829383979923], [0.05518139659292045, -0.008419661432935691, -0.01804995097546427, -0.08877514570206568, 0.21171600179782257, -0.017185973613610978], [0.013342935183814197, 0.00015492683475236817, 0.0003076113444968663, -0.00028710518333775796, -0.05154385865992378, 0.002825490480198022], [-0.07847572054789735, -0.0016380231066589173, -0.007260834670642722, -0.030013764038398782, 0.09979456319720142, -0.0035228875002704008], [-0.014970377307899282, -0.00020842707620883922, 0.0016513018271926967, -0.0046247479718905865, -0.016283772895241827, -0.0007639765759522312], [-0.016487583969158448, -0.0005075440517140808, -0.0015925345814050771, 0.0057828155261518315, -0.021392030897273557, -0.0010031220266007378], [0.010887784746587576, -0.0004029539014882801, 0.0008731761787186641, -0.014292031942557853, -0.03065762133198944, -0.0003583537492707123], [0.010363810058825737, -0.0005526374561751565, 0.0021105027230511973, -0.009526861423356672, -0.04017792153844131, 0.002583107636096145], [-0.011722193311679084, -0.0004181897942215114, -3.941339484813701e-05, -0.007398421203036847, -0.014483229900011173, -0.0011385523962032917], [-0.015759594298377606, -1.6368910712089837e-05, -0.0006646804533883325, -0.0008578888710205539, -0.01918071743439829, 0.0012792499678967921], [-0.0053968901094109956, 0.0002723162633989291, -0.0015832849549327333, -0.010281940214713266, -0.0189523049965542, 0.0007421040122122216], [0.018309185134899722, -0.001036087969076101, 0.009363286611987822, 0.01080838788824806, -0.06393554503071536, -0.0004949409210584344], [-0.01110204433083705, 0.0006161625551620852, 0.0014776536481120887, -0.009050656904535614, -0.01770657490008316, 0.0005654599321816133], [-0.07719952996060789, -0.005069599165480931, -0.007907765184047217, -0.03410595190226484, 0.08918321892098374, 0.002399627291417001], [0.008533496067713729, -0.0002917531977029071, 0.0024893996395399976, -0.007781998921168328, -0.03376257466254641, -0.0018865689258360744], [-0.017683777497465237, -0.00046658813695942767, 0.0006343398229335218, 0.016384914483030054, -0.03047493601331438, -0.0035939526582246684], [-0.01821361451686652, -0.0007008413484253196, 0.00043015158952683954, 0.005549624552455124, -0.023519332518356126, 0.0012540122416659846], [0.012184158643313353, 0.00019814593169737585, -0.00524806383660344, -0.01393243950305448, -0.030537625645922304, 0.002135824410569408], [-0.01197111846717405, -0.00033140372605113477, -0.0017265995535392451, -0.00679525174526726, -0.01375269812721591, -0.0006229283807524266], [-0.0130762670194905, 0.0005284754677798579, -0.0016349748215071123, -0.005439840353827996, -0.015212050662862452, -0.00036534261009184944], [-0.02308139564686917, 0.002943300029290352, -0.0024749265958989497, 0.02460850235598283, -0.03161038143661243, -0.000585098705892721], [-0.014634739580274428, -0.00050566514609697, -0.0022062072831627744, -0.00757918706680429, -0.009139813475429305, -0.0011343874482322546], [-0.012741314544711408, -0.0003805289666450993, 0.0010483577092490317, -0.00711017591165514, -0.0150377874275447, -0.0009785508586927553], [-0.017791434317035063, -0.0010512960099456237, 0.00011646290806536045, 0.009772280178045349, -0.026605434186706307, 0.00035942142757620774], [-0.015629396415130545, -4.2125632178258215e-05, -0.0019332834980119887, 0.00010969114209454918, -0.018026338618179864, 0.0003214530214060428], [-0.13326950304598376, -0.0011787968131358722, -0.011394920545764723, 0.02649835329715992, 0.12752303598825593, -0.008271026023388639], [-0.006686204841689214, 0.00044474091410027095, 0.0019287271677709994, -0.009454676421157222, -0.02332084582560742, 0.0018882590065825973], [-0.014709642966192087, -0.0005315069518699147, 0.0007699373052818695, -0.001284847671781191, -0.01884349643463896, -0.0006004432807997796], [-0.01815307207625406, -0.0007857361808817345, 0.0004864019284817854, 0.00743914138957193, -0.02433042155631996, 0.0001436864954019661], [0.01336291019895578, -0.0007585473368377768, 0.0036393949055054785, -0.0033152369373169255, -0.03969387103065813, -0.0003096497996483669], [-0.01647392274884907, -0.0003701203459853463, 0.0004731329479096498, 0.0050056356557003625, -0.02292033974249888, -0.0009143857662767626], [-0.016803572210885433, 0.003525277605514074, 0.002348819595157591, -0.0015594240947927893, -0.01559089712745643, -0.0009535371008704297], [-0.011872971009831168, -0.0006473316199721076, -0.0011011802894507908, -0.007067570508829426, -0.013576897075214374, -0.0009340494967021753], [-0.01785568640078715, 0.002396056616826547, 0.0008859147004189961, -0.000930568942637428, -0.02019544161929391, 0.002166392312139518], [-0.016689352994785793, 0.00010840452566762358, -0.00047607983079286465, 0.006670326756897691, -0.0229466101782245, -0.0018666882787622646], [-0.01736894930882356, -0.0007719032840360549, 0.00020176285881487644, 0.005917744373957439, -0.02122812021676736, -0.001950534423145408], [-0.016976195407554368, 0.00036557803635496713, 0.0005189160776879015, -0.0008515567742160521, -0.01960395116758261, 0.0013472092353101009], [-0.012454717973587421, -5.059157837812688e-05, -0.00021334526125196322, -0.007829874480228696, -0.0156776847654329, 0.0010262140588790617], [0.004881205273117263, 0.004011182147812328, 0.00023292112979652498, -0.013716631685843058, -0.02743169541088636, -0.0015103147873301059], [-0.011466858618714068, -0.00025940358693498977, -0.0014252877256756842, -0.006880997435687969, -0.014042737239308507, -0.0011247153936788045], [-0.012956838413707419, 0.0005127729153883358, -0.0019494973018832132, -0.005595379440929458, -0.01616119490358928, 0.0009501371447209784], [0.013966371113311914, 8.19503137416298e-05, 0.0014336232603225634, -0.010452143710338493, -0.042340622213959615, 0.0021108212369220117], [0.009930901690270328, -6.419183858789189e-05, -0.0002619370104313495, -0.013132038366380674, -0.0301446467377732, -0.0002780877370972573], [-0.01636971073759408, 0.00023105146777641132, -0.0022613992317871883, 0.005460155515516578, -0.021293340011223075, -0.0009667570026887061], [-0.012552714210716798, 0.00015371069467080836, -0.0011049604218987526, -0.007651430639545448, -0.013507160072663235, -0.0005374453498466112], [-0.010227602669803935, 0.00046051528423422095, -0.0013355499144263728, -0.008456373081434498, -0.016241758314160082, 0.0006007686955906332], [0.3274913582079197, 0.010017627349002222, 0.008152259235179636, 0.03513624758858887, 0.4646174584408693, 0.03131758886098007], [0.0558069670070481, 0.001322065100465159, -0.002856437163632173, 0.0037626209387527487, -0.06979710314301496, -0.004354779406285339], [0.03505756722579936, 0.00150864116450054, -0.0011069725741300516, 0.019716627523228423, -0.078948282699814, -0.0005942473062508149], [-0.01825082308125093, -7.355636481012025e-05, -0.0016857746836267035, 0.005733835792936687, -0.021854821797183708, 0.0009311401339346858], [0.031046705108678314, -0.0026765984177305863, -0.0027047929755305254, 0.01202468086628479, -0.07244230849528141, -0.000447686086420526], [-0.01255554135697052, 0.0014113893035797994, 0.00019927909795399752, -0.00788199642283865, -0.015298952900502597, -0.001074177721222079], [-0.022411862789081274, -0.0007846186349867977, 0.0002877168495798242, 0.02100682964319793, -0.03144293078517795, -0.0018551342835317407], [-0.060379754222578776, 0.005067419786030926, -0.00399027263207918, -0.05875626261432595, 0.10282805180600627, -0.0032469599008311044], [0.012879346779123228, -9.896941507490559e-05, -0.001176142804583767, -0.011238996142962344, -0.03483334113198334, -0.0007318972845188596], [0.0022421166024814095, -0.0008997374968148876, 0.0006374162535223748, 0.00939788014909878, -0.036021621111808816, 0.000860612270187777], [-0.07696726463495587, 0.001329645854496974, -0.007236683866474342, -0.03529092995240977, 0.09436178647839395, 0.002770112787615737], [0.032019158306857844, -0.001695230698864695, -0.0014581879637127104, -0.013294746125497916, -0.04499206454396247, 0.0004710710251799357], [-0.01213794046436967, -6.381037865030125e-05, -0.0009959516522927167, -0.007030862508327875, -0.01401634158981548, -0.000955093406544014], [-0.013497673054733915, 0.0004899240573513544, -0.0010131551184438723, -0.0019757811561782377, -0.0160170439363278, -0.000686270791667576], [-0.021514519423131744, 4.6364731013922245e-05, 0.00154772022734541, 0.009452712691765846, -0.02362003606634796, 0.0013877578393544667], [-0.014674122052836434, -0.000364989553491807, -0.0011717059782350024, -0.0008113201036408134, -0.017041199893392683, -0.0011366624184033147], [-0.01485554047251127, -0.0005992142501288788, 0.008849357777076832, -0.004587736634535208, -0.020548480697422503, -0.0005417190558123859], [-0.01199125820719927, -0.0007877902662454513, -0.0009795404111448078, -0.007432129546614285, -0.014622205737662595, 0.0006129241688663835], [-0.016841621533258116, -0.0007960942440892432, 0.00039218548351658503, 0.0024605492202539846, -0.01934713081717222, -0.0010678881092510873], [0.017968298250570083, -0.0011744191956802594, -0.0018474514544754561, 0.011878094042458076, -0.06165511524297197, -0.0003694063999005691], [-0.012160076209666233, -0.00045981587284494484, -0.000939387263641446, -0.007466857924339076, -0.014117955954854013, -5.590677465433726e-05], [-0.01430598183021397, -0.00038921453996086175, -0.0014207917215126834, -0.0007557873232375572, -0.017103141576295826, -0.0012250830087791667], [0.017807732068224114, -0.0007492107703061815, -0.0011686015454078224, 0.0010499028157661399, -0.04879515744187457, 0.00040533487359826066], [0.00992301851838003, -0.0007262130021362888, -0.004112329313333766, 0.009694643281863235, -0.04789707648261139, -0.0020820430021618124], [0.04511378503188034, 0.0010054096956874721, -0.0007373532784393854, -0.012199120863292844, -0.05597510594827607, 0.002949528219583227], [0.006407927563721134, -0.0005744584551150658, 0.0015825533357162092, -0.01293890929092607, -0.029034320183512315, -0.0006427929698839875], [0.05859612860411339, 0.0006179418012812007, -0.0030028349129425586, -0.008759313693465921, -0.061890463901919504, 0.001762351626742754], [-0.016704306504049627, -0.0006187980140337982, -0.0008282502461811199, 0.006877086945189274, -0.02213886235537995, -0.001072584111259107], [-0.018689682554747396, 0.0016213523975118155, 0.0002237246811626152, 0.005312201892857936, -0.02391622962223563, 0.00024863320545060987], [0.01963174807101884, -0.0009077001604218993, -0.001621437925901724, -0.0010075027763978663, -0.05108441514177376, -0.00021069206652363383], [-0.01286399468999142, -0.0006581856565336557, -0.0005835224926730083, 0.0065888859304604275, -0.02838975687847725, 0.0007065737872148469], [-0.00968092438944097, 0.00012412416121452795, -0.001264422310886903, -0.007736235518770431, -0.015416733677300109, -0.0012258082648161405], [0.2744725479590276, 0.006573172936142535, 0.019620537366310795, 0.08985023098989814, 0.4765300847923844, 0.0033843783371885045], [-0.00589442209017303, 0.0018248501671370059, 0.00022251938686655102, -0.009839695017516924, -0.019871153502100895, -0.0016420989442127545], [-0.033349542741185276, 0.022667900779898435, 0.01964837985401517, 0.1805495627062229, 0.3011485098662731, 0.024881221280807546], [-0.00030362185821461413, -0.0006198818405717094, 0.0006686983383183152, -0.0099437535956047, -0.0204924219629284, -0.0011756857476656248], [0.012539349771563825, -0.0006544648469570156, 0.010751245170639146, -0.013219302969854876, -0.03889574635736887, -6.795053736510345e-06], [0.23516617355293548, 0.019289213820023295, 0.014891479275582405, 0.1099670009265066, 0.42790254550418777, 0.0021589837461610693], [0.06812274788398388, 0.014925715230110864, 0.03236170565661372, 0.1273681225415259, 0.36178541169380063, 0.04866486842253622], [0.01009400277750284, -0.0003793339165125514, -0.0016878014278351539, -0.00742616341862108, -0.03036421005620031, -0.0029364939583337484], [-0.016295541606441005, -0.0005028206925201508, -0.0005633953405723795, 0.0015167456759096086, -0.019010435697413217, 0.0004887809943704194], [-0.01771181405380802, -0.00048203444221528967, -0.0013479964034374592, 0.002662838221858951, -0.019473541645038308, 0.0011525483226400653], [-0.015437653818511803, -0.00023298165290974244, -0.0011044953872281272, 0.0007861052239216973, -0.018351923358340332, -0.0008590510069317875], [-0.014117783314367412, 0.00021687420267166042, 0.0007557127821138285, -0.005357543632935749, -0.017937714197561627, 0.0012404541600792623], [0.015150337618413017, 0.0012275921853735073, -0.0025935911411336787, 0.007857284381747848, -0.05520108905731746, 0.0023594660129166527], [-0.017334715867534472, -0.0007566630026131925, 8.517779387240395e-05, 0.007384973657675236, -0.02365327026522381, -0.0009255023161762575], [-0.012327816286971771, 4.88242191689179e-05, -0.0007514336665843939, -0.007204130906178421, -0.014428633889519631, -0.0005368094699147437], [-0.012962793661959624, -0.00020927969045066066, 0.0004072553149965639, -0.006918181398771066, -0.01500934822521825, -0.000507652338597017], [-0.014430934615404466, -0.000101593805355211, 0.0004550240671396003, -0.001358058041497103, -0.01855435066646212, -0.0012100869384207573], [-0.012901873784444877, 2.3126008152254908e-05, 0.0004966583289794981, -0.0071800702893191634, -0.0150047666714897, -0.0006330735918780609], [-0.01192392755945794, -0.0005734570222670739, -0.0018046549315506107, -0.0074124094677806285, -0.014161381020585243, 0.0006758300016414603], [0.017828300285901426, -0.0007710365570236022, 1.5577068078292868e-05, 0.010645876182912595, -0.06471683636627078, 0.001798119386402023], [0.006112796957316677, 0.0010161746630867818, -0.0024959633517274865, -0.013078441617676281, -0.026064080874327378, -0.0006904857766724157], [-0.09356321380916925, 0.00019427401817882428, 0.006326415206765596, -0.02656543467699949, 0.10287614544087463, -0.0025515195129837843], [0.015162743276918873, -0.00040541730717729086, -0.002605200565958049, 0.01221411683670805, -0.05166916882402518, -0.0037304067497997604], [-0.01635198852912259, 0.0009900432069003574, 0.000699345127503427, 0.003657049507494812, -0.01945437029052246, -0.0007400790222535965], [-0.013913408330350345, 0.0003688040352586364, 0.0005138162560815717, -0.006028798873984415, -0.017052240415065548, 0.0009118273280600506], [-0.06367366167994201, -0.003788845112021226, -0.004473719172366761, -0.048063003800956974, 0.09480688537004225, 0.004464566617466817], [-0.01437769869852279, -0.00035382393470590937, -0.0012178672393719833, 0.0006551797820936558, -0.01878542416780815, -0.00040608002739917013], [-0.004202012410014783, 0.00017251143618747143, 0.0013044320684449265, -0.009787007331565149, -0.020789538176405904, -0.0018983855866466076], [0.03676488948933193, 0.0006786960553093933, -0.0015926372396910503, 0.007072499234397759, -0.06811346560678463, -0.0040099819325633244], [0.01028076823243013, -0.00033618716238798706, 0.0016815998868959306, 0.0010320484806670813, -0.04765754671998646, 0.0027159839490479227], [-0.01525140413952091, 0.0006037544156294645, -0.0005039599998911289, -0.000795491904171027, -0.01867534499237836, -0.0005775533796680845], [-0.013496219677924698, 0.0007299254272816494, 0.002358994721608119, -0.006946368532725234, -0.016487616886996, 0.0016412849487561316], [-0.01837269391380924, -0.0017210394911190662, -0.0026127840891463036, 0.01451382683134209, -0.027181721446937854, 0.00017441210967032616], [-0.14716996713123606, -0.0016842178824585101, -0.002869659681030308, 0.0012331476541283335, 0.1241783777199984, 0.003771049479328657], [-0.01117880841300229, -0.0010235668379755868, -0.0011295762048918595, -0.004821719205108006, -0.017610917086039034, 0.0005645877470167376], [-0.01530702238044947, 0.0006047422813598448, -0.0016990553802288186, 0.0048326153545000085, -0.018365171162428997, -0.0012661087127526388], [0.2800458485764162, 0.01372050931575707, 0.002109119734627513, -0.05616347147860946, 0.3883563911258662, 0.018063562432902035], [-0.012477906403918895, -0.00044693265036229514, -0.0016425432221297287, -0.0051469839092668085, -0.014535970696032149, -0.0009496631182901797], [-0.0222175045441684, -0.0013706340357697432, -0.0031589561262916556, 0.023702548918769977, -0.02560511567645241, 0.00011632813057885128], [-0.015826536209994444, 0.0024023106753481727, -9.307554668651362e-05, -0.00874404764148838, -0.011061869836751744, -0.0018767814404271456], [-0.018173515803345676, -0.0003596041062146241, 0.007526131018823039, 0.00026198557216096553, -0.024811204917261336, 0.00035620823583753104], [-0.011646289750989207, -0.0006282881520473646, 0.0003190656367265642, -0.0060500274691121965, -0.017719315872776635, 0.0005248556081988097], [-0.012267205814007138, 0.0005666274709912367, -0.0017231315737137412, -0.007233751394406906, -0.013883553175154719, -0.0006589855137087604], [-0.12173201648503658, -0.003776409677970688, -0.004910875863171499, 0.0008775475869757031, 0.11062459655461108, -0.0034018897344557045], [0.3192968179577031, -0.00021268414895435017, 0.009504431319999618, 0.03801639419794093, 0.4497122319329867, -0.0013941753866607367], [0.009680981248334847, 0.001887919158277524, -0.0040833002452280885, 0.011033168195277606, -0.05043655827727967, 0.0011225518253797056], [0.03389784903342432, -0.0016508843575351328, -0.0007817744970034906, 0.009312186070067723, -0.07491928981315435, -0.00022475310246571558], [0.2699697377879083, 0.02031674001111137, 0.018737324270823863, 0.06668979928160387, 0.49459297953294235, 0.0119869256091166], [-0.013200499712509048, -0.00047505470576555687, 0.00075378948544086, -0.007446890483788462, -0.014807964284737359, -2.3380298640492838e-05], [-0.012318211990310877, -0.00042451045227071216, -0.0010363720879682879, 0.006562460061148393, -0.028558999578405884, 0.0005756340478073015], [-0.016671201521793435, -0.0002696815532785343, -0.0012528866453638788, 0.010427425827135428, -0.020911915167862283, -0.0025217409388373587], [-0.011414124147568186, 3.124171223013707e-05, -0.0015499307755853027, -0.0072570185288951304, -0.01517397579043152, 0.00016380753024995823], [0.03494512937688756, 0.0008920718815528749, 0.0032523812345215675, -0.0048162805672714495, -0.06549644627427584, 0.0010231443485852008], [0.03522683070203198, 0.0006147909327960184, -0.001964314849675261, -0.011821136628609627, -0.05172357628777557, -0.0017825938687675443], [0.010753724840849613, 0.00019330432443166997, 0.0008475556651459942, 0.010287656666872137, -0.04730634623455166, -0.0024758952627478045], [0.013176007082886979, -0.0023290385826619654, 0.0010266014182493202, 0.016725205987458, -0.04520368107280622, -0.0033450948331261447], [0.09962606010809233, 0.019746460979789087, 0.041356235585511276, 0.11848235054368139, 0.26798457188161423, 0.025250605172596346], [-0.0013020102371144931, -7.731432838554971e-05, 0.0003639532213650304, -0.005254638492769841, -0.027206136836432592, -0.0017238533266625755], [-0.1391832038142817, 0.009226723779143374, -0.008000638941943087, 0.006808713090521618, 0.13280163839699907, 0.00039676748956096205], [-0.017362180725110952, 0.00018063537178144936, -0.0010036389161858226, 0.0063572724286968295, -0.022718734585949824, -0.0006533535732317259], [-0.013245620676666138, -0.0004946328594968255, 1.8976590083299022e-05, -0.006950041601241924, -0.011050595837565153, -0.0009780856151132772], [-0.009626637786891521, -0.00033816105575317927, -0.0010652923123343008, -0.007530154997815633, -0.015477075646008783, -0.0011626782011966152], [-0.01145402281324363, 0.0014489029002584535, 0.0006371282007227668, -0.007457908882476454, -0.01844896033629087, 7.486093102970315e-05], [-0.015438367815082659, -0.0001935230883359982, 0.011626091234161532, -0.004001962119878046, -0.020467427935135774, 0.0019418563909375489], [-0.009843182306721673, 1.5160972076711057e-05, -0.0011114628753536712, -0.007857984037682921, -0.015498118689876597, -0.0009044130624418752], [-0.013552874760200563, -8.589089918235022e-05, -0.0009227110646930418, -0.0014424953103355416, -0.014913120336713872, -0.0009495742955413434], [-0.01714958500012175, -0.0008668702520035712, 0.0005938237922478858, 0.004904595129520847, -0.021860734229846846, -0.0008212294397966179], [-0.015047191550703517, -0.00033616932532515395, 0.0006275653581736244, -0.0011335599755735086, -0.01827698947454419, -0.0010336550320273388], [-0.005298287213358241, -0.0003017543753137608, -0.0031685115184445518, -0.0007786765546628156, -0.024566472964115398, -0.0010862973741052626], [-0.014121856383266409, 0.000857822379358488, -0.0011725414638457177, -0.005733373027435339, -0.015799915073429833, 0.0007698635686187636], [0.02686553421869376, -0.0006851246894294708, -0.002996182598559721, -0.010359650264166742, -0.04744829805866833, -0.0005762786078695273], [0.015551737509578719, -0.0012651796567003705, 0.002748983697616889, 0.0020283350691319424, -0.0488562627632967, -0.0020742805229972024], [-0.014839719545328483, -0.0001654495759243716, -0.0010163670185616652, -0.0008890670411406422, -0.017196646765653134, -0.0010927500533917565], [-0.0199407706623402, -0.0005124052709749343, 0.0009008099026328683, 0.005982238457450595, -0.025415677577486567, 0.0037858051507182], [-0.018914184462884516, -0.0009092933149853586, 0.0018699926974655302, 0.007219514995005253, -0.02460642294951977, 0.0001403930349188173], [-0.013419608362900992, -0.00030818146706244973, -0.0018189827978185517, -0.001083735890638008, -0.01487135526779012, -0.0003648028804565885], [-0.014614170145645835, 0.003354666570611429, 0.00047659881802461203, 0.00617912929999149, -0.02709607727308898, -0.0015001472698928024], [-0.012921842280146758, 0.0005089189843336832, -0.0007167337697257166, -0.00794997222141808, -0.014858105341304821, 0.0007377346282616658], [-0.13820631892654994, -0.003687145484556633, -0.000457445994179924, 0.08201648712384402, 0.17137432831778782, -0.004448238369679097], [-0.0143425483637389, 0.000160223435584152, -0.0007976172079954676, -0.006966257299974801, -0.017099055360499504, 0.003845254796624441], [0.03752968537297755, 0.0007148416972116192, -0.0003458170279419052, 0.006969103050976337, -0.0745953363226338, -0.0021391434372564043], [-0.05720659957981774, 0.0018555004870036205, -0.009173706179083706, -0.050832466278024006, 0.11792733204952117, -0.005547838277377224], [-0.015721275883422522, 0.0004941663463170096, 0.000909307404235932, -0.0018120848434502945, -0.01857514101537463, -0.0004949720083055591], [-0.017600739422537773, -0.0008152885065835452, -0.0029034063094750375, 0.01118291033097808, -0.02146801317728399, -0.0010954629150977755], [-0.013324106006467537, 0.0023564151622701567, -0.0007308459179099519, -0.008161968195627943, -0.014372196515984282, -0.000967298526280494], [-0.024129338387202437, -0.0015090226402545482, 0.00023543194101399825, 0.020590467250244153, -0.030902807324464755, 0.0005152691606635642], [0.03288942105249296, -0.0012117940052247045, -0.0032789289047057265, -0.00032203067840159364, -0.0607978455588047, -0.0024788219053561537], [0.0011409913228616188, -0.0022119891607655065, -0.0013829319877134809, 0.010554747198005112, -0.040513392007566186, 0.0005459079685118044], [-0.01838695554963853, 9.323952777364164e-05, 0.001224962874800039, 0.0004137698458946831, -0.017621316893659297, 0.0007429668614961052], [-0.005248236263519798, 0.0002988954590210215, -0.0005631824861344276, 0.003254750431912198, -0.02765836391344829, -0.0012838632278307515], [-0.00999005242175358, -0.0003693158712367693, -0.001466086555968099, -0.0078773041346953, -0.014205760099561615, -0.0012914809167846428], [-0.013467212915932917, 0.0003773356181339653, -0.0018113516976955406, -0.005983605928456575, -0.013674250831586215, -0.0006409142444627508], [0.02064023575514531, -0.0009386175548692705, -0.0006896198105447427, 0.010397685063724906, -0.06473622553606566, 0.00012654208260940233], [0.010127696235402862, 0.00010733253422802606, -0.000660768744284912, -0.011133542956810865, -0.03189107888836655, -0.0017496381801685985], [0.03553712944521303, 0.00391788670391199, -0.0009119731924742017, -0.00013785894310917561, -0.06494723491350518, -0.0036579491000363045], [-0.01363272182589979, 0.0021685885422599352, -0.0016890008706340943, -0.007498670950912187, -0.014731103847460235, 0.0001829089526463133], [-0.01398204148147877, -0.00011533696106392412, 0.000651501459505716, -0.005059514531539497, -0.015731840943893707, -0.0009627675415298917], [-0.01693526620633027, -0.00022221995219063516, 0.00027451096133322765, 0.005392613581158812, -0.022034523924351377, -0.0016751144596198087], [-0.0018945169767077819, -0.001198715000923001, 0.0014310079709775184, -0.010365481344349175, -0.021880942056753897, -0.0012913525922437186], [0.03423095933922683, -0.0005675078596904917, 0.0006042220252635006, 0.0055184258790423905, -0.07099365980425039, -0.002742439579591736], [0.04180371890941551, -0.0006971211819062665, -0.001004435032277218, -0.0005921623161282832, -0.059620438727201394, -0.00508956165190239], [-0.014386111753338655, -0.0005294899428380865, -0.0018455117890953828, -0.0028184270857636398, -0.01638314229540409, 0.0007626828664397923], [-0.013687492605575226, -6.69760862858072e-05, -0.0012440118627614824, -0.0037337309461557447, -0.01586413370167706, -0.0006036547975447413], [-0.0957895193566906, 0.003317155358838086, 0.0017992291441356187, -0.02607336290163099, 0.10748108455869891, 0.00011303224426786197], [-0.012549626209789654, 0.0004436975167335192, 0.0014563349561581651, -0.005969676343214153, -0.016095615024441547, -0.000818448228779701], [0.01764454075012002, -0.002893987678135675, -0.002760337250144927, 0.011297926469676008, -0.05968916367048755, 0.0012010213789719797], [0.0008966643271287417, 0.0033363856458440517, -0.0009588061820912725, -0.006801302008413758, -0.02683556556346303, 0.0030792904476619307], [0.02229843755122819, 0.0025172991091040845, -0.006937150161927657, 0.021636661413509135, -0.055848597432683805, -0.001580936193515663], [0.03726919101211266, -0.0008165562099847405, 0.0019252766869133479, 0.004477411726463826, -0.06894421582073566, 5.555927189721205e-05], [-0.0757747170449574, 0.0008309402939637793, -0.007440561649666298, -0.03475595141759659, 0.1015417163629516, 0.00129143059816199], [0.03506176200439273, -0.0010735107641665685, -0.0008195543960852186, 0.006232253115197818, -0.07113472540803581, -0.0034662245513028735], [0.021383520923782155, -0.0013717251536716198, -0.0010980212119959484, 0.010762898633046785, -0.06442231049412132, 0.00154563730295986], [-0.0049966477508192495, -0.0008301635594596987, -0.0010433714933469319, -0.010290551543102492, -0.018884770817972095, 0.0008455051647003941], [-0.012350598794069025, -0.00022566825033530966, 0.0006229202843155766, -0.007670878229474673, -0.015961939898904588, 0.0003861648884679808], [0.037505585676907434, -0.0008291598429032024, -0.0014352395713579766, -8.527920297826396e-06, -0.062192621069821645, -0.0011567039391933312], [0.033783314382718685, 0.0009615861437321402, -0.0025209351857397996, -0.014964827433916427, -0.05158461801490819, 0.00037548010811353535], [-0.019163444440957918, -0.001748460431328713, -0.0003897429193599771, 0.01675268415313105, -0.031171489802395293, 0.0005204534409107572], [-0.016129586200721992, -0.0005050481962134423, -0.00127116630437766, 0.005958849900952596, -0.02202732874581433, -0.0012257204538252452], [-0.015031782427999428, -0.0004029162822751066, 0.0004524478862789569, -0.0005089441627450264, -0.018555116145616685, -0.0011536888676427774], [-0.012731380987461391, 0.00038891794766356347, -0.001094107951578272, -0.005947661852190161, -0.014774869094746408, -0.0010408980616873954], [-0.014500414390750284, -0.0003420284108190157, 0.0018126308992322082, -0.005224494250735526, -0.016040760473714135, -0.0009049333732133126], [0.012583378738128047, -0.0005883300122454315, -0.0013430100785433317, -0.0142441412981591, -0.03316869309149277, 0.00156079574231257], [0.009456664380421994, -0.0009725180342105322, -0.0019554962565618617, -0.0148397969189372, -0.030785853055979185, 0.003896999885266724], [0.034170708093786535, -0.0007411886188682175, 0.0007862488283671638, 0.014622642792168117, -0.07811431501597793, 0.007909237253857794], [0.019507478011427885, 0.001143182634538528, 0.0015273664446225487, 0.017140547420400547, -0.06155292721980672, -0.0030881963107906784], [-0.0031339385723019256, -0.0010642501167822825, 0.001671172562925917, -0.008183660664509627, -0.02369693035274396, -0.0007923928565881513], [-0.018407811457840494, 0.002634989699082427, 0.00027747133594369804, 0.005684833895743172, -0.024169688201348603, -0.0012197952715803059], [0.20195074352796236, 0.020936034922947194, 0.024159346859981404, 0.089674060782711, 0.43106202861813464, 0.003550141820620134], [-0.018256716100480345, -0.002014391530249516, 0.007462680879416232, 0.0034612784112369554, -0.02469743033728588, -0.001155421322637527], [-0.012359158005794152, -0.0001378527149594925, -0.0007439198729288823, -0.007395841046685969, -0.014460831190255336, -0.00010239716937622466], [-0.08449792472449201, 0.0037082835654406333, 0.0018453172375846878, -0.042160804629879346, 0.1032637204685867, 0.005224741416092799], [-0.017071386038114714, 0.002092115048932994, 0.0003595973288734755, 0.0069796198547414185, -0.02720909147263896, -0.0003508547217942972], [-0.003927318749925505, -0.0014448324991140475, -0.001877888816806604, -0.010295312756461714, -0.017628651973964737, -2.5995203727400063e-05], [0.016271131818282762, -0.00025244825977834755, -0.006564958859875302, -0.011450388708280907, -0.0008830303234455296, -0.002296496143093193], [0.039680502408926434, -0.0018588740251641036, 0.03830424957440529, -0.007165506323386643, -0.0514052144645285, 0.0009849221948270647], [0.011300831334712389, -0.0008802859651834295, -0.003131588428038821, -0.010058427341892143, -0.031145878663129876, -0.0012846509364680878], [-0.016379315750872044, -0.0005243997643739463, 0.00031301225567848743, 0.0009422207908058763, -0.020241359226383464, 0.0006898416951450357], [-0.08661160022679747, -0.0035867488075997044, -0.004177132950041967, -0.03441466215310973, 0.10972801993474554, -0.0033759710352918907], [-0.015971815241478088, 3.031163804031315e-05, -0.001902426084875967, 0.0006927050975299775, -0.01791085549947917, -0.00013791990973711988], [0.0721604319170357, -0.017651596710016218, -0.005506450645301746, -0.021892207174061352, 0.24562916255763162, -0.027242911373859664], [0.2533229206267932, 0.0065514878257129415, 0.014700774578956056, 0.14357708560738167, 0.47626380696682835, 0.014919638680042289], [-0.01085098279339057, 0.0005138873644535676, -0.0011661995122158701, -0.008367131991084005, -0.01501912632032495, -0.0003104467474381973], [0.014560918048512663, -0.0005969055715694851, -0.0023885765813672537, -0.0015087741565894795, -0.04292709707723576, -0.0010895646617506876], [-0.014633046015838425, -0.00044356176357801384, -0.001369939201832892, 0.0009861841239094567, -0.01835867092404526, -0.001380966218614912], [0.03730023825733515, -0.0025277699545789875, 0.0017339184938240544, -0.0006325857554047495, -0.0640953573507909, -0.0019784436903845185], [0.035607120541958595, 0.0011636101940289702, -0.002353524631723129, -0.012621927381133413, -0.05253379470950208, -0.002794817346962374], [-0.01736293110806805, -0.00024207079463381905, -0.0005009818747218852, 0.00560788308007485, -0.023359902390479432, 0.0006580030878282584], [-0.015821091593949704, 0.0001999315678873828, 0.00032701902376373263, 0.00010634294640250886, -0.018970312527980517, -0.0010418894161234877], [-0.012300431804307642, 0.000450404799424097, -0.0017319973498975824, -0.007009744931172362, -0.014189840228694586, -0.00041839048535196574], [-0.13225266589253526, 0.003684765276288524, -0.004573668789774398, -0.0009789090005706898, 0.12223547232813502, 0.0034350060784568964], [-0.015660687609798572, -0.0007822136124004728, 0.002467937554872752, 0.018288124393754643, -0.03473553048103316, 0.00022236975460473694], [-0.01467824027740391, -0.00013308789334710305, -0.0007696310628471864, -0.0010866503037135309, -0.018943391742556836, 0.0004110012798684857], [0.035607559714997926, 0.0010301307230102215, -0.00018173522559989465, 0.0011218251928586413, -0.06936260257343596, -0.003415177831830869], [0.02017014446581928, 0.001866192932095934, 0.001455264926414178, 0.01698640118819406, -0.0665162181110853, -0.0016617854014382017], [-0.012682416345467858, -0.0005680850995913221, 0.0002704690709181767, -0.007181457413191577, -0.01455603822049893, -0.0004824719921685586], [0.23480040045000225, -0.0168004019460843, 0.0028087060779884722, 0.1105080609387422, 0.4749985016322529, 0.015818066180432026], [-0.001441982300758342, 0.00032340091919537314, 0.00045026053926682414, 0.009559393321429122, -0.04234666589815479, -0.0017444065809782332], [-0.013972446285848381, 0.00029998029976778014, 0.0002964623466493814, -0.007706301269287149, -0.015588044463283449, 0.001470349372001743], [-0.012864218400686793, 0.0004519924409102201, -0.0014556422744065813, -0.005419548289867519, -0.015392034639502764, -0.000520548836446576], [-0.013119413120627475, -0.00015249284387063394, -0.0010623128239078388, -0.0058034082632399744, -0.0158465794195873, 0.000784206471233151], [-0.003083376871294952, -0.000517015401388458, 0.0020019901792979445, -0.010192139671495907, -0.02181157021019748, -0.001597888024921194], [-0.01589296015054308, 7.391272127708646e-05, 0.0006298909886595336, 0.0006001987878254257, -0.020306824097117402, -0.00030421825010161686], [-0.013273553860417722, -7.622956741918035e-05, 0.0008454268309783905, -0.007342127249011229, -0.015229587070890295, -0.00012392908324000947], [0.01718462122627766, -0.0007735483805011892, 0.000988611329085032, 0.0020956109377202603, -0.05314134905200578, -0.0015539460605760726], [0.032110425761572844, 0.00036429365182833915, -0.001853186492220062, -0.013095750306935819, -0.0519769344400171, -0.0007488481742281498], [0.038010020249324164, 0.00024301791271876683, -0.0009048743988851814, 0.007229034384848197, -0.07916323483630684, 0.0018860366883010785], [-0.01841386084868611, -0.0006930476555658579, -0.0009855981141926407, 0.006013313760348395, -0.025039218847751236, 0.003918411705847399], [-0.08617743397333068, 0.0012477436547097985, 0.0006385780950538684, -0.033006088482783386, 0.105604477834122, -0.0021144199849144588], [-0.11765245108707163, -0.0015734072758428915, -0.009756119145501183, -0.0003007380689355349, 0.10074959014571032, 0.0029997920983075217], [-0.014338939142955007, 0.0007387112579383501, 0.0002834302555120578, -0.001257543230590437, -0.01947662793574935, -0.0011490312041556891], [-0.013727488675076463, 0.0025617149063232175, 0.0001368366254740799, 0.015194590581808465, -0.034673345060165575, -0.0013589750450304443], [-0.011828906242429058, -0.00019872016772362315, -0.000724781086437892, -0.007313176071474419, -0.014370248548937484, -0.0007641678829975703], [-0.01621504215872573, 0.0015459868767567421, 0.0007412743715580662, -0.00033031319137044704, -0.019588789021268104, 0.0011468831230493964], [-0.006074777421610883, 8.180707947815082e-05, 0.0007641345677804636, -0.0077742346015411325, -0.02221337695381218, 1.644732970555216e-05], [0.014846958798789415, 8.161014133636991e-05, -0.0009570177905504756, -0.0019579527265305963, -0.04175151744269555, -0.0024620809803491475], [-0.01371633814708725, -0.0010949793744410399, -0.000464494577735532, 0.004876119014713581, -0.025175776146634105, 0.00037546923118427313], [-0.012343508953467275, 0.001198009798384882, 0.0008210595902621103, -0.0068981108293452295, -0.017758252693864707, -0.00021919691196983584], [-0.010838790490735677, 0.00021308400968664536, -0.0014714473667483258, -0.006061817992283345, -0.01790845216350499, 0.0008674240035856548], [-0.01767210936274484, -0.0010620144022450582, 0.00012230670323353258, 0.005556196201408434, -0.02182018172213196, -0.00032419741752018235], [-0.01472080330631925, -0.0010980564159238724, -0.0013326943871749186, 0.009121805856527032, -0.031294317680884524, 0.0041240659337754525], [0.034826731142023656, -0.003520848246298955, 0.0019640623075008684, 0.0031182362316467846, -0.069234000095141, -0.0023541813397312916], [0.011014341199448628, 0.0009222845282612179, -0.0008069416655682808, -0.005314027100077058, -0.0368235228570985, -0.0016921341049660288], [-0.007318772215323137, 0.00018457037140594571, -0.002594710251785473, -0.004136997624439826, -0.01962858387519758, -0.0017055064046599237], [-0.018043722760103315, 0.002072638759746916, 0.004168721794230224, 0.005609600271737543, -0.02807286209688355, 6.562403127213696e-05], [-0.015396166236406711, -0.0006441641412492176, -0.0006075452397099978, 0.0002359438344711532, -0.018146182946119492, -0.0006418852709857939], [-0.010687360648403938, -8.718456522865915e-05, -0.0016685423846865856, -0.00744182167096881, -0.01451747451675877, -0.0007976162139532457], [0.04138056583364537, -0.002670536794381883, 0.00218522345836333, 0.007376127533729874, -0.07634094042855531, 0.0020362270638652684], [0.03358873188407101, -0.0019288943941722276, 0.003802117837784508, -0.012695030298844832, -0.052658942587653426, -0.0003079824411850234], [-0.012425149036344108, -8.306965009053526e-05, -0.0007497764112761614, -0.007399381351957337, -0.014672528700331717, 0.00012990514999979355], [-0.005522988769716395, 0.0008779585508691232, -0.0018938396907365743, -0.010872431720196444, -0.019424233829745705, 0.0016355354595259497], [-0.09887150321345747, -0.00010122037898256894, -0.012323520675699515, -0.027084249010307777, 0.11500037748474054, 0.004430115793706508], [0.00766084847799136, 0.002195322582260491, -0.0012591944423576004, -0.01707168850138097, -0.02697184299559917, 0.0039965548790858235], [0.00984336191764092, 1.2412017365207541e-05, -0.0012533139951668366, -0.014547266256863452, -0.029169724909897173, -8.546877307875039e-05], [-0.002384055342795339, -0.0002648170737365973, 0.0017052069821926854, -0.007625356681645547, -0.025617505102651737, -0.0010134727813634903], [-0.012407010123971681, 0.00036706820965590686, -0.0017919404951149851, -0.00630144166771332, -0.014444871009015228, -0.0006218049138407015], [-0.011073863143670391, -0.0004315433072498573, -0.0017069663584396095, -0.0021300743871563135, -0.018942077226023826, -0.0009154755774600327], [-0.014650672942543718, -0.0008219683078508706, 0.0011810360016315281, -0.005359640428308153, -0.016152603465113987, 0.0006038491421851464], [0.029467908762614778, 0.003768336134410515, -0.0008891193399314321, 0.015569009013939516, -0.0764116121150534, -0.0033711891226466404], [0.03739905935675273, -0.00033513882165530853, -0.002288426275683855, 0.00883852542397672, -0.07430353552476633, 0.0029895158413760794], [-0.00485551496414387, 0.0001274597451134719, -0.0021168185432186497, -0.00788861680568073, -0.019798324573010416, -0.0006681848590598601], [-0.01528977100673214, -0.0007972376757017174, -0.0012633242936092267, -0.00033424313778174415, -0.0177857481191469, 0.00027032423297169126], [-0.1627240944319862, -0.006679626503269672, -0.0008999837811521624, 0.043003811741517746, 0.16407374681630216, -0.001390520508078567], [0.015828642337473855, -0.00025122528411580965, 0.00018528631116533813, 0.01204439188691376, -0.05507161796917438, -0.004602143948929524], [0.057241268277435385, 0.0013998309729308521, 0.00266970405431578, -0.017295456281514892, -0.05891765730679829, 0.0020832626645834323], [0.009612632105447546, 3.4908211107661e-05, -0.0015322253796215778, -0.014554336392900397, -0.028402166946486288, -0.0003588115975470055], [-0.012625891284129144, -3.854781970448049e-05, -0.001049611930846299, -0.0070499501553088, -0.0151254225309193, 0.0006894237209079583], [-0.008936892502594813, -0.0008491287180841929, 7.67503210187224e-05, -0.005605592753767727, -0.016871870974067286, 0.000320067960828574], [-0.01297688428237443, -1.225300736553828e-05, -0.0017781674462027336, -0.004798308545643033, -0.014699292060826397, -0.0009350946575879052], [-0.11671564750253215, 0.0026790847090133554, -0.014072443998783076, -0.007264870909358454, 0.11102693619815174, 0.0002778938844609491], [-0.0063510428638443375, -0.0007903737452513372, -0.0017011538667162183, 0.009832281629435696, -0.0330867518240636, 0.0010637073371064555], [0.010631382382422892, -0.002664050157523011, -0.0006051050380648221, 0.010041351426707758, -0.05325759331585313, 0.0006540147023102727], [0.10202328733133299, -0.008745499933603235, -0.021956204499533942, -0.08863535272639964, 0.23860656981784734, -0.019284466656310107], [0.03188744187762496, 0.0013206313180291907, -0.0022892695197001193, -0.015594771070239346, -0.050584103796096055, 6.007119038134343e-05], [0.04293918549205237, 0.0015619228291131827, -0.0037281033889572566, -0.0185221681162925, -0.03782494814633742, -0.0012092220029117418], [0.01468808056091369, 0.004482274597334103, -0.0019076883493071373, -0.013309703534565644, -0.03537452957666386, 0.0005072805880031621], [-0.01204653591399969, -0.0007014704365499428, -0.0008332280298545576, -0.006919373045699115, -0.013926219021160516, -0.0007731735527362393], [-0.017313042111966166, -0.0006910173154812109, -0.0007332983384086978, 0.005132050890011889, -0.021694922636903657, 0.00010022951274778738], [0.007065454421203607, -0.0007732222873947303, 0.0008566928433793523, -0.01307997615265406, -0.029151398629572958, -0.0001175501949612583], [-0.016852917255562056, -0.0013754243851752334, 0.0003115034513356846, 0.00514330721307555, -0.0223546634154678, -7.180560820619127e-05], [0.009364191166734556, -0.0008682588637942669, -0.002764061674673169, -0.012811112198708792, -0.026485520249559778, -0.0016352381799985497], [-0.1253563371655478, -0.006628021032954546, -0.012588155666800981, 0.01938364094739854, 0.11403372153686568, -0.012187705761817578], [-0.014595874237115765, 0.0015094423403884017, -0.002087030508704112, -0.00033562784566459815, -0.016413822507011386, -0.000777087241892592], [-0.01756343529724983, 0.00029394899133064554, 0.0007185654960403793, 0.010473882392342357, -0.027753738681618737, -0.0013692229008448527], [0.0127800853101205, -2.3375948925689764e-05, -0.0027042549483084375, -0.014335630734806551, -0.03267777280698093, 0.0017609491289010725], [0.025365935946420164, 0.00677095342205952, -0.0227894699800394, -0.08372297614496677, 0.201298328998363, 0.014537942043877533], [-0.012959058479241328, 0.0003386484811266024, -0.0002566670867179015, -0.006058840140649414, -0.01588703597494602, -0.00037704679957199013], [0.3066236405364061, 0.0033544513043151556, 0.0010733728480257462, 0.013133251079883184, 0.4429756408444117, -0.013955594708279716], [0.019711057689489102, 0.0025816647798845567, 0.0004002870230074779, -0.00777670733713741, -0.04270634278572621, -0.001909959369517464], [0.027252893358459207, -0.001004344734710599, -0.0037494961802830304, -0.006794755633001142, -0.049026602514980255, -0.0018776942954841678], [-0.013059085240379608, 0.0010988738722058783, -0.0009060581702718457, -0.004857012872267179, -0.015258450057729233, -0.0009682675315580846], [-0.014063792534085373, 0.0018693156851967848, 0.0006892240698958291, -0.007768073223310548, -0.016010425414306508, 8.375141660976417e-05], [-0.016849485682540574, -0.0004982256359033886, -0.0014600889033367307, 0.006156409648827916, -0.021398494562910084, -0.0011501148641372067], [-0.023838752027177695, -0.0006746253510215332, -0.0030052864530512612, 0.02080363073183971, -0.029826954473436713, 0.0013419875728474652], [0.016924787020836815, -0.0010274347402186561, -2.1548087582483004e-06, 0.0021026977272449467, -0.0509392392694833, -0.002258655929621665], [-0.022647137085581175, -0.012752189440591128, -0.011896506362541076, -0.07751161897714541, 0.13082299377629386, -0.005054827624720746], [-0.0025368790368082133, 0.0010561534692412273, -0.0008413262943096532, -0.011536562422859194, -0.02041051492154153, -0.0009308707937227018], [-0.018278052818444267, 0.0008811371971370357, 0.001702330708982186, 0.006660682033292142, -0.023211484314728748, 0.002045387193761609], [0.0149223090179948, -5.468200882001782e-06, -0.0015816209347935125, 0.0022432725519876706, -0.05339730275247236, 0.00261881031816528], [-0.01324616227183011, -0.0003946574558011749, -0.0012149578007411787, -0.003640471337371785, -0.01559497137013158, -0.00110877976412423], [-0.017406504669837942, -0.0003895789656996679, -0.0007935758019619069, 0.007336070994868624, -0.023351485598938158, 0.0010717407082356878], [-0.07094996663128593, 0.004382531228980122, -0.00792532108012429, -0.031509860633277834, 0.08551948179529491, -0.004800198012920349], [0.011169126197725655, 0.0004190505884854335, 0.00029076002197168043, 0.011892593231169304, -0.05912625260520812, 0.0030118654229989096], [0.01910668149127354, -0.0012457607559768983, 0.0012464410079822796, 0.0017584318700352502, -0.053250012482344254, -0.0015657811309699302], [0.02729101302242138, 0.0017461130938156396, -0.0030334610773154773, -0.012994253099249793, -0.04386854444893333, -0.0010075341574050875], [0.04199603192493324, -0.0023956351375590272, -0.0005517418695755739, -0.01625262676640952, -0.05061775040703245, 0.010931246065167068], [-0.007952005720775174, -0.0006325051988609951, 0.0005645673563888077, -0.0019131475285445916, -0.02897488987633904, 0.0037079809681309786], [-0.0007952707506350447, -0.0005214067398975452, -0.003023802898963354, 0.01109173240743845, -0.041232105097874415, 0.000530853079931892], [-0.01686166617549465, -0.00011106611167791461, 0.0015461958717150903, -0.002961060653304986, -0.01862393416166171, 0.0018115312304241137], [0.014901638852698535, -0.0007613092912990199, 0.0012959151552008991, -0.013557080858824457, -0.034191005347669653, 0.0011118414898936512], [-0.007822880430455896, 0.000886645633612934, 0.007934989317952489, -0.001562724861030485, -0.03376927620018134, 0.001633246540102321], [-0.016700579908486782, -0.0013235821591197862, -0.00011381645690018503, 0.014930292983813164, -0.03152418494331933, -0.00046812951598715726], [-0.016763065167041465, 0.0017771892788636748, -0.0021995390130027225, 0.001107027452721862, -0.018727629666907383, -0.00039398288463402204], [-0.019720265849952557, -0.0009518554230512525, 0.006682400712246651, 0.003493303918216325, -0.02511915330784914, 0.00041556995038994965], [-0.016793031621581713, -0.00012281964424610943, 0.0004597757125092263, 0.0054583056985022735, -0.023138534339161305, -0.0010636958060224618], [0.03325223461932468, -0.0036190001864588673, -0.001212253960929735, 0.004732289077693098, -0.0665832414516202, -0.0017700280980088503], [-0.014572992538622606, 0.0014915444553229907, 0.0074276905155971525, -0.007759500427285383, -0.020477319244891268, -0.0013094227601209436], [0.01030027687367665, -0.0006351253103529531, 0.00048497666859812134, -0.013146847559446156, -0.030561875330128354, -0.0016414053423473365], [-0.07709065941190499, -0.005107263109910978, -0.010233398144806198, -0.031048770774065586, 0.09429733173933856, 0.0035660930346824446], [-0.007991463584275263, -0.0003245086904495988, -0.0004526286203755557, -0.00797112811333495, -0.016396472236957716, 0.0004362012453930428], [-0.01828536213718856, -0.0008795914725698506, -0.0005778668440143083, 0.011454278260201192, -0.021427622618558895, -0.0014838351878696387], [-0.01833351047855264, 0.0035272146363946166, 0.00046225655054720865, 0.006671619769115131, -0.022678782138009588, -0.001098798339494787], [-0.016238164007602026, -0.000507775262536533, 0.0008345848501446747, -0.0001316579942941909, -0.018420727144556302, -0.0007362604411556757], [-0.0122114287807192, -0.0009425826353343018, -0.00035438548562735856, -0.007643657046277495, -0.01484329368490965, 0.0007953476328679802], [0.00378580988551059, -0.0036449475253441314, -0.000763521468570211, 0.02715659494978942, -0.041858576356459974, -0.0016610737706400033], [0.032164215588004155, 0.00022260088283355945, -0.0029906970993104297, -0.009206088041222846, -0.056132752486891584, 0.0007427211565871156], [-0.04256651643672424, -0.00029862783474349236, -0.01696284352215529, 0.048009774971854914, 0.20981205378101916, 0.02031867430326388], [0.02319551810761133, 0.004709532299659089, -0.0003923364308912148, 0.0017464664335801062, -0.05265093288703594, 0.00659651438183845], [-0.016768622875141644, 0.0010711531679510674, 0.00047062536941348533, 0.001195727463478958, -0.01980006975541534, 0.000297853296380071], [-0.018261081850471185, -6.94333599374305e-05, 0.028978753848234857, 0.005235060002803401, -0.017870198452103643, -0.0031396874901133977], [-0.012769069632264122, -0.00022955850327891373, -0.0014089987034882042, -0.004561613402226548, -0.014661909065216602, -0.0015688506935256667], [0.19480576532713556, 0.003621021819428727, 0.02430872770919548, 0.14119030004920932, 0.4320501610761906, 0.007982754177570947], [0.011682948691885225, 3.611524208373084e-06, -0.001861572946676256, -0.013936099025731856, -0.030500413052475188, -0.0005884751912103366], [-0.013519469303025385, -0.00021936068588213704, 0.0008274637048179297, -0.006954263174058652, -0.014691632719284793, -0.0006427378225670282], [0.020851699762899058, -0.0028108280931412387, 0.006722333010170144, 0.009805329765389425, -0.06505348785984444, -0.002754262271747562], [0.01049237087754962, -0.0012978732647188423, -0.00242115514003644, 0.009636237671735404, -0.047194003373809815, 0.0005844232292799942], [-0.023430362355696868, -0.00144591781008037, -0.0028375217421395533, 0.020344874945635208, -0.028735999969082788, 0.0009049269313643225], [0.010335536142146317, -0.000989301591324321, 0.001579365868833505, -0.001446984641326631, -0.042719537378274865, 0.0026242549332793643], [0.031524410821820295, -0.00042526286729656316, -0.0021610816198903444, 0.016031892256273675, -0.07452143930065247, -0.0018985192902545149], [0.036178222721390006, -6.940804030942839e-05, 0.003350880554784287, -0.014394668203791516, -0.05711809242096035, 0.0010197320555536832], [0.03451431039467869, -0.0002478491404849856, -0.002321506746831953, -0.011277219698196145, -0.051311809559572014, -0.0020559252495936198], [-0.016883898438478005, -0.0006974305509202935, -0.0009718125489603636, 0.0058527133710891575, -0.02137685603109095, -0.0011227158016396278], [0.011169981636401937, -0.00041420350404765364, 0.0014513417522493642, -0.014592898432414458, -0.034057680304035394, 0.0012434588518461756], [-0.014778869610234675, -0.0002080249529370263, 0.001163777674005508, -0.0022869735193726775, -0.018351962560774378, -0.000737947030686816], [-0.02028361438099221, -0.0006663154919248903, 0.00013175635054209423, 0.0053054951132071645, -0.023880668334574945, 0.004193346743742751], [-0.01754321490027731, 0.0006601321275589098, -0.002154609206157656, 0.0026729285411437754, -0.019180999846982235, 0.0003457632847144391], [-0.08487431441198376, -0.0016656600430257176, -0.012644437019087252, -0.001720518238950939, 0.08227682478347632, -0.0011254664990002117], [-0.013020125253986993, -0.0007218348319089701, -0.000873420867315158, -0.005292060948867463, -0.014905457058817654, -0.00038710103910383204], [0.12369574476245297, 0.02240900253282398, -0.04649549743435836, 0.016564900337636083, 0.29711502419165076, -0.012536793437824214], [-0.013553763020572419, -0.0001396309980562314, -0.0016506158077156097, 0.0060751120390330786, -0.027247992824030693, 0.0013168906113417958], [-0.013828399282437887, -0.0007456157807364946, 0.00039103161614440846, -0.005487867520551623, -0.015241663035071453, -0.0002874859973470084], [-0.016225713953506485, -0.0008021339793611523, -0.0009488604204621493, 0.0007326419987191993, -0.01966742194596511, 0.0017114883005756218], [0.03489250284525716, -0.0011618651936952226, -8.223491627819856e-05, 0.005553799963212784, -0.07282122971418699, -0.0015809729843094973], [-0.007627846578121888, 0.0021151917500744554, 0.0010581583958536801, -0.005384877463347785, -0.023812207652123687, -0.0015484184523347807], [-0.011618771837892517, -0.0006146578056017879, -0.0011895797736039535, -0.001074503500883581, -0.021006267297729385, 0.0003037802157111925], [-0.07404289625909344, -0.01391517097240448, -0.005657164349596002, 0.008831495970473998, 0.11658920845033596, -0.01186658395082689], [0.0426007776615793, -0.0014028089611109419, 0.0016730983578387684, 0.00998075040881823, -0.0741148595520961, 0.0018963754183040024], [-0.019683545381453584, 0.001385017555253814, -0.0006439676765449677, 0.006655445432957979, -0.023979108452567774, 0.001066158522354493], [-0.005178208925856307, 0.00043443325433649844, -0.0012090909009968482, -0.010605683623683441, -0.019293360420208197, 0.0006519106164082404], [0.016071987608461374, -0.0020121904001600055, -0.0029906733019473383, -0.009966134320218974, -0.0350136740763193, -0.0012893155098157704], [-0.01313737648156662, -0.0002795194290363595, -0.0009611679864251202, -0.005699535839530226, -0.015007277127362577, -0.00011512313607917445], [-0.09239673019645805, 0.0013998179058566846, 0.0026160548708125133, -0.03768383642201206, 0.096713319281778, 0.0030323269409752132], [-0.011284874099199113, 0.001411138632884019, 0.001409037366706132, -0.008677057060349597, -0.015942879088191118, -0.00044869908518368914], [0.036772398493150435, 0.0019133370648804675, 0.0013210896983210825, -0.014286458575375922, -0.05313867349364918, 0.0005516401460063595], [0.00947723121120419, -0.001168575733586986, -0.003416047840200395, 0.012584059266732929, -0.04136266600315777, 0.0011859990990079538], [0.03839741460706427, 0.0004925594676808369, 0.01539539109337562, -0.01116217048690299, -0.05771359233672539, 0.001354683369793359], [-0.017676998538453614, 0.0025737916345158983, 0.0022929163910639125, 0.01126636956010835, -0.028796475058997524, -0.0003596039882371709], [0.1768932533142031, 0.024700760674913873, 0.03224603858576358, 0.11261810278551485, 0.43091322818028016, 0.010547664078372335], [-0.010616044267038686, 0.000614692985625597, -0.0017913382425938714, -0.006195731413205071, -0.016616960991863723, -0.0005946180709242732], [-0.08981643918028702, 0.012818485164837998, 0.0015627247302542088, 0.01650963619591356, 0.08253479308018634, -0.003725866657571709], [0.01613817507148019, -0.0018335849765090371, 0.001696556588671434, 0.0028850956252919275, -0.04925634052452979, -0.002329901784404808], [-0.011741545868531328, -0.0006659599076596226, -0.0008783687966566323, -0.007534150090272087, -0.013933243661750604, -0.0004467316751297546], [-0.016262475662355065, 0.0005521030291879543, 0.0006057469069547335, -0.0003792111219410535, -0.017518531862781617, 0.00030236871093495816], [-0.01226063525933318, -0.00044684139828286265, -0.0008941089928999932, -0.007313744204264921, -0.01425594741267486, -2.872273254423044e-05], [-0.012298407504005496, -0.0011004579611017349, -0.0014309974071326744, 0.006561577772038091, -0.02731854514401678, 0.00038683024421850914], [-0.0065469019552947694, 0.0008246793864781804, 4.828678320801991e-05, -0.008988642215395318, -0.021619761582654322, 0.0010823395836581538], [-0.006086499433618309, -0.0006250792295753935, -0.0024442064239428123, -0.0029459185196432365, -0.02263705890563061, -0.0004612374875896563], [-0.01477495431464086, -2.0735734906636723e-05, 0.00027755662312416895, -0.0011332064388031207, -0.018398030602827995, -0.0011506295319456581], [-0.016098941776111236, -0.000480846095285991, -0.0009981872965283106, -0.0006154937248405402, -0.01818412633478411, 0.0011775952275501103], [-0.0004789604286133948, -0.002270406771526378, -0.0016809775816154495, 0.0190380054074639, -0.04618295850680882, 0.00554196454776678], [0.06312988289107264, -0.0014566232208130076, 0.006151374269160245, -0.001033685387837552, -0.06698004737880262, 0.0007033845415059391], [-0.017143771400130847, -0.00030916261942610403, -0.0016731109056053203, 0.005335036812214821, -0.020197114041396336, -0.0012118778456562743], [-0.015471637212816044, 0.0018591817584015728, -0.0016028364632541012, 0.002232878788264217, -0.020543292487050717, -0.0016742943835450017], [-0.017076168321325136, -0.0010618162917776157, 0.0005097868820959184, 0.00703616170943669, -0.023204349290160145, -0.001403614688269768], [0.03706254035312113, -0.003812996152847018, -0.0032643626401414553, -0.006111594497078557, -0.05575145648890284, 0.0016778694258487937], [0.023407766220736197, -0.0002832429803247506, 0.00146024290495046, 0.009939934059029508, -0.06155615076018469, 0.0019504981748408778], [-0.015785443073210594, 9.301462682132151e-06, -0.0002599751476420698, -0.0009136249562819985, -0.018793209572313566, 0.0005429512867659989], [0.03056853960911262, 0.0009701559087600079, -0.0034300877374569437, -0.013201928728875932, -0.04894827717689141, -0.001158401874648351], [-0.012908702889615311, 0.0004563797263529074, 0.00023310019565107003, -0.007113848890054047, -0.014735661485921795, -0.0011312666564128864], [-0.013268378120154611, 9.372505106262524e-05, -0.001483527932451915, 0.005767159842506657, -0.02459884338729077, -0.0017101354536720374], [0.01987421070365544, -0.00130746210640642, 0.003541906333410343, 0.008778576354329626, -0.05865587016089044, -0.0021372434770398022], [0.013810171689408136, 0.0005889141655576582, 0.003869845838317656, -0.014363373768560844, -0.03388799331223711, -0.0010508979458188755], [0.03730910408068867, 0.0014209403659147149, 0.012502502465423254, 0.0023566888540511935, -0.06717388269737316, -0.001198686402038048], [-0.015499957316104285, -0.00041722482799250204, -0.000588025850130037, 0.0034417597236181367, -0.023452375931500347, 0.0013158242021089536], [-0.01361015220112287, -0.0004113854082362381, -0.0019109710907718779, -0.0011986926097745268, -0.018136741974620632, 6.794328452612245e-05], [0.012453302814054873, 0.005160504012081276, -0.0015184487383722988, 0.012206537885280323, -0.05556112149137665, 0.0013449398040467273], [0.27901356884732553, 0.020795873853762213, 0.015268165666348855, 0.0971648193636092, 0.4643147065250933, 0.0026654847914796874], [-0.015436031647427002, 0.0009745486629227486, 0.0003832370738736228, -0.0009231759105775507, -0.019162742995587345, -0.0010358351832045513], [-0.014264811027137015, 0.00010469146445025811, -0.001761548031841486, -0.00579455907664583, -0.016974946119845236, 0.0034911727910192443], [-0.017775238001953266, -0.0005751708697777186, 0.00028979691936275117, 0.005511684002565884, -0.02142329555286059, -0.0012277764973371655], [-0.017921572436657873, -0.0011073018759203022, 0.008696268132705012, 0.010300390542532314, -0.02392660074849204, 0.00042548305249950083], [-0.03720981153089652, -0.004601178538530318, -0.009152476857227672, -0.0637890781225685, 0.10684559272012324, -0.0032374921153447422], [-0.017004191660591115, 0.0014062037351785534, -7.688620956956508e-06, -0.00022004203862941888, -0.02000665723921783, 0.0006323758242167194], [0.010830397861353935, 0.0006846206513942176, 0.0014951631020439938, -0.014032882995463195, -0.03320194026166873, -0.0009753583576602499], [0.012519116212782119, 2.4215880767880922e-05, 0.0004057463278501558, -0.01416870008656153, -0.03346936075455614, -0.0005110175802824796], [-0.014328987433110044, -0.0026297265195351755, 0.007688518284912142, -0.006674687873052756, -0.020043925373181275, 0.0007888089139670307], [-0.015800968060440436, -0.0004029245614729624, 0.0013701182102322123, -0.0008148732053859006, -0.01915622156524334, -0.0003951308176896517], [-0.0025935209133768925, 7.854781871118033e-05, -0.0006362888657348702, 0.0008300555014414145, -0.03435685698389668, 0.001478063442855819], [0.035970756760761405, -0.001976458553258774, -0.0015448580575342205, -0.006956018681207641, -0.054617127309335184, -0.001909627492758896], [-0.0898203519877433, -0.0002125958858382676, -0.0029113921870238114, -0.02862681684426521, 0.09359287822153399, -0.005221721316663611], [-0.010913291356970322, 0.000380545500256244, -0.0008135031667216106, -0.007026764567071251, -0.018257346757271295, 0.0014303603477782126], [-0.1488491293386752, -0.006146868348331013, -0.0029372511982113034, 0.023360743472956812, 0.13011180322377383, -0.008239297811512985], [0.011878359308660437, 7.77184034460004e-05, -0.001349710380076396, -0.014342121018335686, -0.03317930169930149, 0.001715055385607136], [-0.015310054429852725, 0.0031354942277031514, 0.00018544723568634982, -0.004450989878678662, -0.016033849605721604, -0.00022604754913654933], [0.03361024100148172, -0.002036594256950301, 0.0013777802682909766, 0.009591908275247186, -0.07400272947771612, -0.002490605810353351], [-0.0175715881099572, -0.0008967456276998286, 0.0001439083343568174, 0.0037225697618579346, -0.020733797323756437, 0.00013565296519867833], [-0.017833656597032562, -0.0006808158439173487, -0.001793221707052999, 0.0057274662755620685, -0.021188530352935414, 0.0005687582253762013], [-0.011563142966326058, 0.00035415875779803833, -0.001606879852371495, -0.00790358354877659, -0.016179479372365184, 0.0016989269820412553], [-0.011921833154818389, -0.0004343967414309204, -0.0013857002695119376, -0.007381708746176622, -0.014091236749076691, 1.4875661014509522e-05], [0.01708355537700872, -0.000745806060688645, -0.005027875761692364, 0.010019075364217627, -0.04978051244719052, -0.0014543188245960674], [-0.01747761764402295, -0.0009751796139115222, -0.00041323302687095624, 0.00982061663583837, -0.025467164493652203, -0.0006874218573808286], [0.004318870010374488, 0.00040240087579022894, -0.0018323055125542152, -0.015256399049272205, -0.022331453288944425, -0.0005011130353939655], [0.00707351215282039, -0.0008555614410230525, -0.00194936118100764, -0.007331459013784499, -0.02730988449242557, -0.0023272460245796693], [0.021370654640442955, -0.0003025982393578926, 0.0010991496161406322, 0.0021273772689889385, -0.057470292292247385, -0.0020242909939673895], [-0.01188565546114487, 7.400327562727216e-05, 0.00012225975589965415, -0.007427647593847847, -0.014966591445463716, -0.0011163685310705297], [-0.013015759868803555, -0.00023289884721578688, -0.0005613681031950347, 0.005302337305651533, -0.022090720412213747, -0.001101590074223454], [-0.013030843772556248, -1.0667977851211294e-05, 0.00013055021830468097, -0.007023155020657724, -0.01457617943079374, -0.0006897040164458237], [0.03298119625738617, -0.0005974651343294615, -0.002254945939595365, 0.0073252321605679, -0.06993253898822516, 0.0012785216441959506], [-0.012624845295146061, 0.0002794359443113155, -0.0008343598450343952, -0.007318496727838075, -0.015269786316621785, 0.0005680522403289423], [-0.018214166206227695, -7.314949940550938e-05, 0.0009160093997959171, -0.005412376903094825, -0.011361172815467436, -0.0010551439756005243], [-0.11535824800061863, -0.006744972884709586, -0.0025818873900567796, 0.026007213837803078, 0.10219839388886742, 0.002987833882047938], [-0.014554573143920804, -0.00045078820084687893, -0.0020148382479018845, 0.00092795999658023, -0.01769923511580085, -0.0014085252881098557], [0.0048487309392341825, 0.001878323676706835, 0.0010315856461798809, -0.0021051837285630207, -0.03314474616805136, -0.0022087103655065296], [-0.016606676429154112, 0.0002228888120180853, -0.0010757658749753808, 0.006801100040521209, -0.02374568446439267, -0.0007958620840172301], [-0.012246128986799366, 0.00039249928322084086, -0.001310933732167868, -0.007289780501572184, -0.014228183789082883, -0.0005174722735985954], [-0.014973184505220314, -0.0003957007305487258, -0.0008440684892127209, 0.0020025418693321705, -0.017939373046023152, -0.001383548431660653], [-0.010392845289172068, 0.0005678908444336462, 0.0015472978522494505, -0.009405974640747106, -0.01793033163522341, 0.0004139628684594651], [-0.019253451023296283, -0.0010064536251860648, 0.0005440258134816665, 0.005893649803396049, -0.02515150230697343, 0.0037737313385780074], [-0.012892625820757564, -0.000699383562006705, -0.0007655888964689603, 0.006099901661597397, -0.02527621889177832, -0.0016660844905859015], [-0.14517237810644484, -0.0092490315882767, -0.00889200398284499, 0.01986654542050714, 0.13124626862380767, -0.0017494003667480774], [-0.01790107363728934, -5.784157561459845e-05, 0.00012099684803618578, 0.011845364059582758, -0.029210283400392827, 2.8377056777892104e-06], [0.015446498500856106, -0.000900637634032873, -0.00171795779959607, 0.008202608404848566, -0.05300666227071621, 0.00010948413197374696], [0.0185127854845521, 0.0034129990058478403, -0.0008024521359978405, -0.006841547892141367, -0.0441197225860152, -0.0028620618762455977], [-0.014267276015732128, -6.053465660600458e-05, -0.0019141541268847813, -0.0007292783256688803, -0.017008459692173974, -0.0012202971829342892], [-0.016618358874769378, -0.0014618404077688346, 0.0005543286550103515, 0.0049983400949360456, -0.021386205395084634, -0.0012862640723235875], [-0.017823171482773494, 0.0004557328143331781, -0.001060992145488466, -0.0007388342402462083, -0.019699572936076225, 0.003666837990251146], [-0.014951034967924716, -0.0006746204878316008, 0.0013454640804141209, 0.00599139471340518, -0.025635502990687384, -0.0012757003473756138], [0.02991571771865754, 6.484739488718512e-05, -0.0035502172849271956, -0.012885392149497061, -0.047503917529390836, -0.0012410381497296376], [-0.016851681603329022, -0.0012632266649731199, -0.002390460985527141, 0.013904606894637144, -0.027814451245430916, -0.0007847863953770296], [-0.015147913478850982, 0.0017173533223653619, -0.0008528339727344096, -0.0012371590598106437, -0.01908981312468519, -0.0005896336862842264], [-0.0033224173616557646, -0.001475206390638041, -0.002325141517552604, -0.010042314361980878, -0.018002397622795856, -3.252274537688609e-05], [-0.012522763794002122, -0.0005191232769523839, -0.0009264153871612244, -0.007469627510503188, -0.014508291481357277, 0.0007462214499761754], [0.03133920436055762, 0.0007661618085645788, -0.003389028867277311, -0.012900157570073673, -0.04918531952087007, -0.0018308602109011203], [0.020164791421519623, 0.00017203810791861047, -0.001566060950002008, -0.015329305087128806, -0.037231306315278614, -0.001410157177028823], [0.16906968481792767, 0.018961607866213377, 0.039871557655528474, 0.0078111441271703184, 0.3722442425080677, 0.027907239215569486], [-0.019481431285282538, 0.0029658370053790408, 0.0012820124283955397, 0.013658444717183024, -0.030945761452808276, 0.0029161366823712027], [-0.01705493304836134, 0.0007351184628187693, 0.0007199697795134113, 0.000985033157527085, -0.02166646973102072, 0.0010812813795227215], [-0.016199898968440662, -0.00047927447074158164, 0.00043088510783050293, 0.001382447086558075, -0.017913576951803976, -0.0015872484700691087], [-0.01272651709157098, -0.0004263031464754347, 0.0004255240197528955, -0.007522002693453117, -0.014651482386668076, -0.00029921870158534924], [-0.01275155398455132, -0.00038995790691687105, -0.0013331930324231751, -0.007508367560028996, -0.014709228880224724, 0.0014923013641450437], [-0.012996333548628919, 0.0001394115621883073, -0.0010732422992351383, -0.00649191483788586, -0.014766943589083184, -1.0977287355236445e-05], [-0.015273142059782006, -0.00046693073524249634, 0.0013283535556376947, -0.005053094409413446, -0.018487703933321355, 0.0027525175821215323], [-0.010900274064103259, 0.0023365571029453466, 0.011673613036405519, -0.007339369373045565, -0.022232974990689658, 1.2448288487574897e-05], [0.012695966256032384, -0.00016377089265622694, -0.005676529543806376, -0.008346016182469801, -0.031802542836836166, -0.0019071068002638998], [-0.010712312055929539, 0.0009967687686984127, -0.0019387248831453944, -0.0061545295873819145, -0.016248577175270923, -0.0011426250669706566], [-0.014393145555474692, -0.0003442068729565633, 0.0009528068958253588, -0.0047478210395716465, -0.016128554571586767, -0.000539078856235767], [0.02624020957567658, -0.014271005554215128, -0.008957805555822781, -0.03335085105855281, 0.21556635595674953, -0.029403093840026113], [-0.014427580006505696, 0.00030037536391017394, 0.0003535234106691658, -0.0005706567413831108, -0.016819646015169942, -0.0007026826781873291], [-0.013593450667349408, -0.00022987348153211424, 0.0007178699285266024, -0.007404341438621823, -0.015519227628020403, 0.000829023286997077], [0.020975287356123504, -0.00021841791809043359, 0.0009739700141954748, -0.00011547432096188177, -0.054518446333051286, -0.0022969187982154592], [-0.014365286466899727, -0.0001837190100234721, -0.0014542926349040623, -0.0012787626706698895, -0.016896916580006194, -0.0010210226374967241], [-0.011953794329557435, -0.0007579917794040463, -0.0009763044106496731, -0.007524411442702017, -0.014227644997830677, 0.00024014696014382152], [-0.014623709485774128, 0.00033888296611625957, 0.0009069015969849859, -0.0050963293370649785, -0.015753062352654758, -0.0009726833876074504], [-0.015398065206155475, 0.002977171195651848, -0.0016306308504655852, 0.005972455024703567, -0.029235235536024973, 0.004614305372290506], [-0.002068111308873166, -0.0005249402690463359, -0.0019121620821490445, -0.010668030721217747, -0.019387976736441177, -0.0006387788822726215], [0.027812166417079023, -0.0007178430785284612, 0.0013898323836319308, -0.0160930245975504, -0.04456921037608681, -0.0007719207485452712], [0.03164092532197744, 0.0008171080746952001, -7.906123855057984e-05, -0.009468362081304532, -0.054350864718254496, 0.007490254641436868], [-0.01749549771300895, 0.0002886067372504011, -0.0015250826185055338, 0.005826881725989815, -0.021600841461111974, -0.0006940666706138691], [-0.018326163456672918, 0.0013572855322431805, -9.86605584810765e-05, 0.016336126392837023, -0.03210904317573374, 0.000973788599140777], [-0.012450084444059097, -0.0012350443633151432, -0.0011332431680001651, 0.01159193147013006, -0.025682946633973136, -0.0022906128607826214], [-0.004128263116906843, 0.0004948170817013454, -0.0027854032575094713, 0.007971138034187262, -0.03098428639841753, -0.0032680023430548483], [-0.01417356521571003, 6.748101855857403e-05, -0.0012451297329728632, -0.00013199511297107202, -0.018885857306743477, -0.0008309336501611885], [-0.014270515164952691, 7.775795959309121e-05, -4.326160897405635e-05, -0.0014426475675136637, -0.015149232420707816, -0.001038767864111603], [0.022640086977567853, 0.0010497492764873072, 0.0028549025800045807, 0.012874155435389927, -0.06824090390664667, 0.0023720096371970555], [-0.01507659894806332, 0.00042931322974514567, 0.00034077231365103064, -0.0007294643538807364, -0.01950669793394157, -0.0006573243075106237], [-0.016091589098982676, -0.0005000595956077593, -0.0014383222389666914, 0.0006762643099449456, -0.018658934786041102, 0.0008126414096532149], [-0.01280270812663892, 0.00047179622582894595, 0.00047426907089374854, -0.007833304955096718, -0.015205220400974934, -0.0003048318140121695], [0.060565502409284176, -0.0006426203400668302, 0.0016492498918676763, 0.004379288026717938, -0.06972692628737949, -0.0022340175099472477], [-0.014516846807571338, 7.029375012895548e-05, -0.002030129714297341, -0.0016627846577556978, -0.0169988817439219, -6.165082658271981e-05], [-0.016297583755980734, -0.0002011702542320522, 0.0017973450711244558, 0.00028608079897816046, -0.02122946934342164, 0.0021114641501983954], [-0.01423507218766869, -0.00034643254210116477, -0.00244670637434146, -0.00046655151157294595, -0.016556500875544, -0.0011487365087717914], [-0.013040540535324607, -0.0005310505226452862, -0.001859191756122194, -0.0036104523411521435, -0.015233286754955944, -0.0009254780897998826], [-0.019083278186908085, 0.0017093400313412076, 0.0007752374454224248, 0.0020584795747509823, -0.02096371413977189, 0.0003039352751653002], [-0.082646197611099, -0.0027901908430988725, -0.006089426504869327, -0.041270772973386514, 0.1232203018051616, 0.00434057184157802], [0.035032849166784785, 0.0008552676627291168, -0.0020474395962300765, -0.016299959285484486, -0.05308588307947841, 0.0015951651316789739], [-0.005579828164679189, 0.00012899997491209116, 0.0014745622388405715, -0.010102274539271048, -0.02034657497455595, -0.000774884535246553], [-0.013902232689408853, -0.00042129556097777354, -0.0007854705979086171, -4.483265100812004e-05, -0.020847966030114753, 0.0008017975294180378], [-0.0131364641679738, -0.0007208438076755297, -0.0011706683555910834, -0.004830002787693062, -0.014921911440094329, -0.0004201094409722579], [0.07732275696888398, 0.0005068224625314838, 0.0039100604948327155, -0.11883321580505138, 0.21073258832663347, -0.01035885371767188], [0.035106208072363254, 7.328902245568907e-06, -0.0005691268735686705, -0.00045755768925530294, -0.06665959990587465, -0.0026272525059101923], [0.009673216452791211, 0.0012457373088093152, 0.00023294960262000396, -0.001503965486239785, -0.04398622368871522, 0.0016382858107343696], [0.01341521166097291, -0.0009509756205634774, 0.0023911217970337523, -0.012571449142898204, -0.037178259287507155, -0.0003056494070378683], [0.039215337898787415, 0.006177981990395658, -0.0001713521640488356, 0.0025341271806490987, -0.06681575025027513, -0.004390344655508236], [-0.024836732327071284, -0.0014732317659351517, 0.012990493215784856, 0.0076973280673745675, -0.018285683230719284, -0.002125507292767069], [-0.016237794128258615, 0.000252197418103467, 0.0015897161736410624, 0.00798738616696517, -0.026040514851306918, -0.001084324112477554], [0.28681603638022807, 0.009367365841698274, 0.004856825442520427, 0.04248376090697474, 0.4001678810405155, 0.01910813038806265], [-0.002614099257960029, -9.982214532732792e-05, -0.003507655066484848, -0.0016024626701404536, -0.02539305773780784, -0.001982903122279554], [0.01718468540601199, -0.0005706868907451541, 0.0018359752721579634, -2.9286260986513597e-05, -0.05522016576126183, 0.001599478234823432], [0.322387501485221, 0.007329528713795017, -0.014120465557293606, 0.04755328388581105, 0.4411693877093214, 0.026385525667907278], [-0.014248675026896803, -0.000563648505834035, 0.0010766428465066645, -0.003638594143679934, -0.0185164992570426, 0.0006907740869466739], [-0.015411926072483808, 0.00045571960500956425, 0.00017275251695776823, -0.0015150215679072098, -0.017259800918482872, -0.0016417235630934843], [0.013064084631271367, -5.556387415388759e-05, 0.0022952029314361345, -0.015038075411486477, -0.03471800011786226, -0.000747648159204893], [-0.01665393419783549, 2.318492703289501e-05, 0.0008962653960944028, -0.0009944416204655943, -0.01976923129305441, 0.0012981567882280871], [-0.008702846557050992, -0.0009354824198718754, -0.00033526650726623805, 0.002508088190462978, -0.020849600615725154, -0.001051558757215428], [0.01339219310538886, -0.0009409024969122887, 0.0023092481453750448, -0.012046707273499786, -0.03767440666860856, -0.00023942481174328933], [0.2992773374177246, 0.010748865293382573, 0.029714448278025708, 0.034017566600102994, 0.45173537118123297, -0.008114764816646221], [-0.012542359855515094, 0.00046865494439654186, -0.0017937116849319466, -0.007478767120835529, -0.014789653627899354, 0.0009358373447853427], [-0.012448614270684945, -0.0007817991012343734, -0.0016264500169446468, -0.007464612319378066, -0.014269863916803798, 0.0013913396250458097], [0.01449024418400117, -0.00011780767118384824, 0.013805015415016341, 0.002815405898513406, -0.050710512013519005, 0.0020176541871718694], [-0.011964527161139795, 0.00020463999754753454, -0.0008872015669730211, -0.006416302342761644, -0.014837345642701285, -0.0012992632839718117], [-0.012583446076622224, 0.0003878966991437668, 0.0003327226329130514, -0.008297239820630818, -0.014724089428115698, -0.0003158440066881026], [-0.012932225661990942, -0.0005598596756364991, 0.0005825468499992841, -0.007325986207997712, -0.015097918018315976, 0.00013344271394180035], [0.043103375357683896, -0.0014591768866097014, 0.0014636691965384978, -0.015055002809132998, -0.053869089282051724, -0.00027663271928518183], [0.20061964495906917, 0.012727467873030014, -0.006775696945969731, 0.12858393511849173, 0.43118970897613235, 0.026097797162103864], [0.00498163633402054, -0.00046080125073761984, 0.0035936042957566638, -0.0016836523597443906, -0.024451482833285982, -0.003012637519342597], [-0.011203355307621362, -0.00038400416789661725, 0.00044365014440282186, -0.007777970268919329, -0.015076211800068816, -0.0012021085998967413], [-0.013328939253461802, -0.000459424550250699, 0.00037152316733787084, -0.005119484572449194, -0.015669592510323058, -0.0009940822808531928], [-0.016635450333797454, -0.0007401720141568557, 0.000221320108920944, 0.0062067581827839925, -0.022650117628376217, -0.001602338315374444], [-0.10416725522587966, -0.0009807224557678686, 0.02270899795260237, 0.03179618208845513, 0.1753583745946919, -0.023243499032023964], [-0.009657123831114126, -0.0007036882237937302, -0.0014767877778639582, -0.008029421840116123, -0.01490593924250767, -0.0004270390846044316], [-0.13722513876775227, -0.0029874914384233108, -0.007534861523507621, 0.023384849752000048, 0.14160470457200527, 0.0006293659771065548], [-0.01671580230864956, -0.0003110528947706508, 0.0007694989403034915, -4.270311901658306e-05, -0.019663824781459043, 0.0007638841635922426], [0.03698476619824372, 0.00019375554739270825, -0.00040189154816805726, 0.006316766148082641, -0.0731885466854605, -0.001771516326757111], [-0.012855565462850485, -0.0004985360457629681, -0.0009044829613491601, -0.002733294783817557, -0.017659687669155853, -0.0005484330770640312], [0.002184395834321514, -0.000602498575147443, -0.0012209087514599164, 0.010608164625497804, -0.031888525876526394, -0.001363960590019001], [0.03717588631114384, -0.0005667897884037373, -0.0001440891991571387, -0.014526552395611772, -0.05448198581697334, -0.002656469110997904], [-0.013886794927482314, 0.0015059706414565623, -0.0001553947514101588, -0.001351924576731121, -0.018388372061195524, -0.00042348432463749447], [0.2638466356934471, -0.009184817959891853, 0.016309989208381653, 0.10799475775296379, 0.4458659559754244, -0.0072349016227058055], [-0.08301540017276283, 0.003845092010651476, -0.0033731058076784254, -0.04112278360050908, 0.09446653145735205, 0.0041663327796135205], [-0.014223452570341352, 0.0002764694636041305, -0.0020452774969625413, -0.001520254557484293, -0.016746177819140107, -0.0009413070196758902], [0.0323728764445298, -0.001077034904640105, -0.00045898147020612877, -0.008125512918380737, -0.052552041902832754, -0.0020259719151367156], [-0.01934504666408398, -0.0006506868344072004, 0.00036735451975594203, 0.005222595675516325, -0.024552203277473598, 0.0037579865806924716], [-0.018272614983894753, 0.0005063615260995752, -0.0024399678424100023, 0.005384126076512777, -0.021104510505748738, 0.0007266057294411522], [-0.0034844029987547392, -0.001278181973353649, -0.0004202618601754082, 0.008792149516056724, -0.038061472152673956, -0.0007478305310989453], [-0.004457053238827501, 0.0016066414272550055, -0.0035463976844635524, -0.002883992467592999, -0.026413253662383156, 0.0004940556260121543], [-0.014329112953085869, -0.0005161893956726048, -0.0017817089877669588, -0.0006822498888871525, -0.016943451551870888, -0.0009472872227165961], [-0.014006480679738899, -0.00036321518867414045, -0.001035582939446101, -0.0051900111571080015, -0.015528431990551744, 0.0009237219555188166], [0.03963960468682483, 5.501951711655879e-05, -0.0029049635732381254, 0.004640177252350151, -0.0719098527915714, 0.002780014908518015], [0.0354569493219003, 0.003150796905429101, 0.0009306883595157887, 0.016914476889664024, -0.07775046641676661, 0.0009308882735908327], [0.007236170578155751, 0.0009063215213858057, 0.0017554937155680294, -0.01129467484062958, -0.030794962420811138, -0.0005083485536689055], [-0.002677475557997683, -0.0019897463858668724, -0.002438771718665495, -0.007503499231588206, -0.021437520899237283, 0.0008470137933555005], [-0.1198326890959669, -0.006193876594150384, 0.003007504185079115, -0.00334107415646345, 0.11646123491205287, -0.0037891944886464545], [-0.016660806524129792, -0.0003819997071398192, -0.0012943337182766007, 0.005532166908306885, -0.020709942527974333, -0.0016850844307863926], [-0.01838444807677464, -0.0006347110397314626, -0.0012589415440998827, -0.00013127437059068275, -0.015997796410731284, 0.001207171441927911], [0.03966520710008899, -0.001599044609609643, 0.01177353706381392, 0.005859280712892119, -0.07472433615154647, -0.0034246441156388], [0.018400042858636986, -0.0028261341583441534, 0.0009204634730670644, 0.010294922836019984, -0.0611477375534578, -8.224122588812978e-06], [0.03701697735184479, -0.0008802231763803435, -0.0009904929550562855, 0.010321479312692622, -0.07029354906389947, -0.0003741914692012261], [0.009877417193150946, -0.00024061948578825573, -0.00023451221504147725, -0.013341131857637424, -0.031293180258668084, 3.202662398423851e-05], [0.017256822998893962, -0.001971879142996466, 0.007917019868291889, 0.015835134734291906, -0.06906424839283688, -0.0001728500656443992], [-0.016401980975620054, -0.0005746866987317677, -0.0024467580850012147, 0.006894135874124055, -0.021096876193394034, -0.0015738339213770109], [-0.014644012881742456, -0.000450685128651757, 0.0015740080711579845, -0.0008279969041766027, -0.018446809854797748, -0.0007378366351228307], [-0.00969135993899814, -0.0006590351422850332, -0.0008859957716915356, -0.00815659379951994, -0.014733540157587764, -0.001073475189917622], [-0.010746775861435324, 0.00032924115515532156, 0.0004067023288700967, -0.008607719666289614, -0.017941039854362257, 0.001359591898061733], [0.0069820264442522735, -0.00017871250993770904, -0.0018637317731405938, 0.0014519051223200467, -0.043365781597999586, 0.005345722885934114], [-0.004215598231825916, -0.0014639602768546955, -0.0014905348351743227, -0.0070848120716475805, -0.019196672999464377, 0.00025157841496686097], [-0.012548148155439453, -0.0005195579857078169, 0.0006666180794985565, -0.007103716457828335, -0.014810675080643808, -0.0008845203998792087], [-0.011953215263653671, 0.00023895604117447444, 0.0013627943634830061, -0.006823774135374172, -0.013455479590286272, -0.0008192814153433957], [-0.0032822387408992892, 0.00200803293548689, -0.00790300365198164, -0.07064872864728092, 0.15385791648304337, -0.01686293075932091], [-0.012581197216392717, -0.0003319674990149757, 0.0004345224713217819, -0.0007111403005012728, -0.020387934327780254, -0.0016222831276326113], [-0.0166694948629506, -0.0009029559665623994, -0.0018171261428611848, 0.006749150779228096, -0.021047224215523262, -0.0015123495913306978], [0.036611151681990035, 0.00018040747651690533, -0.0016072929056502898, 0.007167327667531135, -0.07252271046375691, 0.0023044498767024716], [-0.012371474789500379, 0.001189981971025716, -0.0015649506641835851, -0.0063238392300661635, -0.015719617681222483, -0.00041009960605313794], [-0.12161655207416708, -0.0036560748168361036, -0.006441033610948256, -0.0008137267031187529, 0.11832166929901428, -0.009077615427277459], [-0.0181582549884685, 0.0017287133398120437, 0.0002798353163661126, 0.00011056650246515149, -0.021050825396931008, 0.001889965226756129], [0.21453822345555373, 0.03560405109266928, 0.02530956637092202, 0.1451399223820857, 0.45847320989165274, 0.012937407759497392], [0.010106694616979805, -0.0015478350707603002, -0.004537006277938565, -0.0033357994296174366, -0.03482044676683547, -0.0010656070718280453], [0.03712316421450941, -0.0010517320790583169, -0.0024649245284329518, 0.006610455111994537, -0.0707678063841858, -0.004649156334826821], [-0.01736337801955476, -0.001031128695050536, -0.0007600929905997385, 0.006097273476894632, -0.02232993237272859, 0.00018725860103892587], [-0.014749502955969246, 0.0009684613226785439, -0.002048532867232276, -0.0014136674277532588, -0.016676616361984635, -0.0012801417097391716], [0.01988842707431845, 0.00036108340137363913, -0.0016713958243926648, -0.000976350406701427, -0.04910692165329503, -0.0020281759246363173], [-0.15325252266925454, -0.007789564463009246, 0.0005138929764731542, 0.03345557037930195, 0.1621152230552154, 0.0022931150069880404], [-0.012728259944312275, -0.00023431009596663923, -0.0017774429889318758, -0.00518216121804947, -0.01462465654707839, -0.0006531692056613927], [0.005721463125532494, -0.0010471477926096253, -0.00185846207060236, -0.011694669018111984, -0.024754290374660665, -0.00156689386954791], [-0.009499924960399255, -0.00017658733420255533, -0.0006765176109200346, -0.00825365865239059, -0.015752069294730657, -0.0008412421473569533], [-0.012200343264965729, 0.0011651986578313021, -0.0015453912270183133, -0.007199004662904482, -0.014407019809281373, -0.001013439693661436], [-0.02810642499405081, -0.0021508143693376707, -0.00019703773047936705, 0.01369389477826214, -0.01241268772202903, 0.00047307003763471465], [-0.01304711135536794, 0.00185684368150583, -0.0018351903062828916, -0.006140412689543393, -0.015566942427582208, -0.0004671869027294404], [0.13309001666632156, 0.031145295716623802, 0.02633934926459612, 0.04791590801551113, 0.363695302809563, 0.031280794194051054], [-0.015835009134585017, 0.0002725235490340193, 0.0017337183834633652, -0.005762198593229976, -0.016635483334239986, 0.0010264491295575242], [0.00520789490868431, -0.0013916070795642378, 0.0024775482153283623, -0.011000327889490256, -0.031458492065401446, 0.0009649839104432038], [-0.014263177084375077, 0.005000548141874968, 0.00035788432035835496, -0.007934037983422822, -0.015335859426117647, 0.000724642031682164], [-0.013447584522413566, -0.0003793211089731905, 0.0001243903348376749, -0.0008646689654550375, -0.021410355788809073, 0.0007775400508131331], [0.03688994709928127, -0.0018547092630265675, -0.0007677358230196437, -0.0016705596399777844, -0.06419679555806088, -0.0011001468151962887], [-0.01603946403813716, 0.0011481583747528367, -0.001379012006492551, 0.0010896808842056694, -0.018753228082755688, -0.0012661351315731823], [0.016898898094534084, -0.0010566275582480546, 0.003505611691632928, 0.00354095345612475, -0.04769132696730275, -0.0016475087167409734], [-0.01690061706119041, -0.00045737029833004526, 0.00949346591418573, -0.0017448798158101006, -0.021923015346987297, -0.0011675833918679622], [-0.01259248758865787, -0.00038883951745088556, 0.0006785378077787481, -0.007089035746964068, -0.014816718598528353, -0.0009914563561776338], [-0.016147975947098603, -0.000860228295878451, -0.000372182269893389, 0.005382631641238206, -0.02178477994355077, -0.0014174651848170806], [-0.01632297572939687, 0.0006077045453281919, -0.00032197509061001884, -0.0006483970113818974, -0.019459442330949296, 0.0009450856170097894], [-0.017873278516971083, 0.0005190442912351954, 0.0005625977684789973, -0.0007266335136458112, -0.018278332493443537, 0.0005966024643461943], [-0.004507705985033441, -0.0012818895290225333, -0.00250640138838037, 0.015023114393610591, -0.03835868700217363, 0.0016696647490945767], [-0.002575855483438077, -0.0024909571622622074, 0.0015311870660095438, 0.008479036875126912, -0.037324145519490794, 0.001180734224054631], [-0.016127542810217475, -0.0007561667171464024, -0.0007459020293520229, 0.0010421572472849648, -0.019443436960502027, 0.00083089126993289], [-0.012728007707946682, -0.000614369256422812, -0.0002596685217278575, -0.0010105204583587565, -0.021252193331160952, 0.0006647592756170121], [0.013295491042357431, 0.0008228336436818716, 0.0018342093104526634, -0.0006847113378370798, -0.04969889927513875, 0.0023560766164837737], [0.017761178680885812, -0.00022679600466804273, 0.0012005474962718836, 0.0029122945809827287, -0.054889495633728386, -0.0019577291197440633], [-0.015048963281824826, -0.0004582354637881516, -0.0014497438126329512, 0.0006471038043705448, -0.01806955878254114, -0.0008206024635835174], [0.015271603638601382, -0.0019832266500318576, -0.002041632847766785, 0.021399560507869297, -0.06457704007154416, 6.40687562054445e-05], [-0.023539125930783614, -0.0002925656928117169, -0.00240592058820079, 0.022395875017512352, -0.028914328741325778, 5.606593560954271e-05], [-0.017660623256978862, -0.0003782914000726501, -0.000667857571884296, 0.0006455095016826645, -0.02085701026436044, 0.00371827299161349], [-0.01318567478653561, 0.0004709014074526522, 0.0011413053546429663, -0.007855935307376586, -0.015087657969111701, -0.0006829386990717649], [-0.013742940531831735, -0.00017248308363688057, -0.000973162366226154, -0.0077488366978525226, -0.01304669746890458, 0.00048412014845183323], [-0.017775939128362783, -0.0010682904841495995, -0.00021236046428788503, 0.01495325220488582, -0.03234452666738953, 0.0012478645393038865], [-0.016894311065859455, -0.0008695564945970087, -0.0007614334741600891, 0.005334215439485048, -0.02025356219216099, -0.001755352212707578], [-0.015554078268691287, -0.0004907120913707552, -0.00010899116505286958, 0.0009060935253007417, -0.019099860899176296, -0.0008524511010096209], [-0.10057276433701688, 0.002021078703052593, -0.01174469918049281, -0.01706568518936921, 0.10279592651356739, 0.003199476823591851], [-0.013499416491352207, 0.001051062822389566, 0.0008862729060313812, -0.007089225419348476, -0.015870222190252657, 0.0005715283725323441], [-0.0030452522551286942, 0.0006158111279805725, -0.001798826100087929, 0.02077908756217441, -0.03896310171989674, -0.004454385281708315], [0.011749608231150411, -3.92145168122661e-05, 0.010878520414571565, -0.012913212980179658, -0.03887158656216389, -0.0014207812532328517], [0.017888162992632697, -0.0018408697515956414, -0.0026650826717811316, 0.011112355091281043, -0.06102750983866208, 0.0013329441781250086], [-0.0024769340955813337, -0.0016754339081834857, 0.0014184038137950465, -0.010862489883603882, -0.02216961202957715, 0.0005660661031507482], [0.3277546333338424, 0.016092852686086918, 0.015473480056534263, 0.01869029943350537, 0.44439703763871125, 0.014284553994176781], [-0.01546927220615922, -0.0010005943186388682, -0.0026551894571519128, 0.006286550561377581, -0.022563657370281775, 0.00020216279085416027], [-0.10051094361770786, -0.0028670016856743195, -0.005853276575457256, -0.017902616042032516, 0.11195411874440536, 0.0037178144145616375], [-0.01259828224882812, -0.0004042363130570349, -0.0019043556490110876, -0.005038878896830097, -0.014443730823682045, -0.0008105160685916692], [0.3262107145903519, 0.019759846397870037, -0.023683689896267603, 0.042794030058394605, 0.4600014708042193, 0.0237811201089237], [0.012123265509896512, -0.000906682521774987, 0.000919753760219294, -0.013445911629657406, -0.03263575958756799, -0.001254665531115459], [-0.01287645363672044, 0.0008468917975492909, -0.0013429229974037198, -0.007847047164858767, -0.014901480524462168, 0.0009210125258957781], [-0.05697015948936749, -0.00886618147833007, -0.008339769493241323, 0.04525074885732744, 0.17994063088742313, -0.025822412140954584], [-0.015251420219421498, 9.699505648895786e-06, 0.0017148327272087332, -0.004938307877749132, -0.01599290808841034, -0.0007418960472767151], [0.03201703133039987, 0.00018871455070685093, -0.004730961720521418, 0.00772824541587647, -0.06894604680219264, -0.0014569827742690346], [-0.016950389239457027, -0.0006351817828819849, 0.00011978858133108134, 0.005090805863166424, -0.02157464501780797, -0.0012503784043505941], [0.03330865611795169, -0.00361774051141431, 0.005976628151952322, 0.006969402495546155, -0.07492852730223536, 0.0004249143815328197], [-0.011648232646925038, -0.0005097802591468799, -0.0011205056897142594, -0.006861646644372622, -0.013954127447669268, -0.001105707312171952], [-0.08133462441147446, 0.000449269809775101, -0.007112333124357115, -0.0315607387344437, 0.09936811376763036, -0.005259687307130208], [0.01211548540034966, 0.0042389897638738655, -0.0021416123526385162, 0.011489397595291127, -0.050532260265498466, -0.0020366668080443887], [-0.017695480330844133, -0.0011184377196873925, 0.0003984982525263978, 0.004054118158540475, -0.02014997580495359, -0.0006887225555818126], [-0.022899063866161775, -0.0015381735933771537, 0.0008289924747247369, 0.022277127261539038, -0.03202410805785943, -0.0018447742188655285], [-0.012782069939655511, -0.00015831726831979818, 0.0006867121474401359, -0.008269589043978386, -0.016279096986909977, 0.001602361091423487], [0.013298253361826138, -0.0007770366810308878, -0.0006081384023874599, 0.009162377443500324, -0.05859337932440434, 0.0023179236024962334], [0.017523150761755788, -0.0008841044188173647, -0.0016917980005478146, 0.003062313144538372, -0.05398595113191753, 0.0007763896449884193], [0.0065047190808108805, -0.00022459553470505715, 0.0007909340169448004, -0.01319412458745965, -0.0287736780589767, -0.0003032549166143772], [0.016180710470339903, 0.004310101081586436, -0.000829544044563626, -0.011392205867711376, -0.0367929773308024, 0.00010962997686537779], [-0.1099007822800046, -0.0031262915218997363, -0.007576000580997645, -0.0006442994161590406, 0.10289913305186187, -0.008101759252800983], [0.0016961760054706265, 0.0016497019777103406, 0.0010516525020252173, -0.01420009695952741, -0.024887362351446802, -0.0005100711742320916], [-0.011370802338024327, 0.00015578365804263017, -0.0011876510752335937, -0.007305224230016566, -0.01438019158055083, -0.0011119144342173301], [-0.012536542926909124, -0.0005991336692444384, 0.0006575137400139223, -0.007395816966018858, -0.014738690282661141, -0.0005873298951804382], [0.01893353186822256, -0.0008492875609328566, -0.0018933110699807757, 0.002621690619711041, -0.053244679214327725, -0.0007679446426923996], [0.03548302214284082, -0.0012254472704968879, -2.0929182785641324e-05, -0.008617034487443779, -0.06150083060077939, 0.0006812193986648542], [0.013002399361657665, -0.00013015218195520088, -0.0015726760014039924, -0.011103105173565235, -0.03495899088561245, -0.00043747511912075785], [-0.012994897640595322, 9.586937908633426e-05, -0.0020246550678318853, -0.004946513980439466, -0.014447532401217881, -0.0008822702890017792], [-0.012539172560742806, -0.0004479219436761413, 0.0008512859205175142, -0.007662573763525883, -0.015184529859314302, -0.00021708779325845348], [-0.013280803570114916, -0.0007869417268691229, 8.560422257799563e-05, -0.005657207980597982, -0.01635102814922742, 0.0007903772042314017], [-0.06331330148928203, -0.0061826211897076655, -0.006480320916225041, -0.041073912864247754, 0.09100921891396192, -0.00415906245449941], [0.05510519243003872, -0.00016235544942573446, -0.0022205300725302184, 0.00687851704400694, -0.07099756464539025, -0.003779449782889793], [0.045150138449391, 0.014362871676337374, 0.024536542292139845, 0.17771961002306313, 0.3660376766217166, 0.04882451014370116], [-0.012913487158432012, -0.0003615304621660058, 0.0009151681604628844, -0.007198364978742622, -0.01478994711942042, -0.0008518384417019055], [-0.01366721547062485, 6.149986678261385e-05, -0.0010602873821979736, -0.005893381331815564, -0.01554680079403554, 0.0009061851118912603], [-0.012981637402930553, 0.00036506856384189616, -0.0018264284953441782, -0.005453188698845882, -0.016512592972790237, 0.0012087790060689083], [-0.014703508498211337, -0.00011038220535587904, 0.0021734661709992935, -0.001697689128935382, -0.01913017952980003, -6.504014203007513e-05], [-0.020445551983947687, -0.0004606296160576316, 0.001308478256721429, 0.006342307346904645, -0.02309883587347859, 0.0011542318698578254], [-0.015749420634193042, 0.0030460011400832237, 0.003921324239447425, -0.005716605977236347, -0.01858870100226391, -0.000445931099170779], [-0.011806654868507842, -0.00043517504914298595, -0.0017046817314894732, -0.001138567481533834, -0.019222794421285702, -0.0008921264480402357], [-0.024994195947548137, -0.00046873015427302236, -0.004061221925683668, 0.021913068736476776, -0.02978919380893172, 0.0022002730999597406], [0.01074278955604968, 0.00012771728473790476, -0.0015940745324603588, -0.004822035872633128, -0.03974507707536431, 9.06806396701892e-05], [0.2547818119574579, 0.004030775613191055, 0.020994932820630384, 0.1077853963137379, 0.4950244823127087, 0.007515934315607423], [0.06385073584685709, -0.013782883502341855, 0.0073143623453601906, 0.018917477013943803, 0.19286076622434917, -0.00998148967420042], [-0.013078291535351769, 0.0004300146864437867, -0.001190822066001383, -0.0051662546192219145, -0.015107873691023609, -0.0010867727748451572], [-0.013767072993361619, 7.392389485273769e-05, -0.0008134291343757656, -0.006443394781591988, -0.015822449736708745, 0.0015724227511853225], [0.04005088624528098, 0.0012243391746829956, -0.0037750704394836406, -1.3247716089738449e-05, -0.062178882561131854, 0.00199197529674116], [-0.013441886447493176, -0.000733009650080038, 0.001387617896975869, -0.007050820238775734, -0.014511933565595972, -0.0008499679950310162], [0.02809429172170567, -0.0013518362045461974, -0.0016413096508467913, -0.013468912338588894, -0.04631976541756867, -0.0005124681101551765], [-0.0685482742106063, -7.46696409243672e-05, -0.003271817059294101, -0.022740809908149258, 0.0633220114516906, -0.0028864406327164605], [0.022296335402975917, -0.0013049146952655605, 0.0018113678591453106, 0.0030778703401739714, -0.057898431517775725, 0.0005677726107460052], [0.0337665539596737, -0.0013938492383901867, 0.0023214709573082955, 0.01551434249542061, -0.08132562458403027, 0.000917106410017975], [-0.09744800503235997, 0.003804634164249273, -0.0020844703766177675, -0.030082387044676848, 0.10483469911411764, 0.006882672032430559], [0.07852979616324109, 0.02569145711107929, -0.0018090929928569232, 0.08969745436945177, 0.31947231329516235, 0.01864069110154143], [-0.012627524832361875, 0.0008132197067261018, -0.001191710891045105, -0.007791057991720464, -0.014563332458848088, 0.00016040646724941162], [-0.015232617744222762, -0.000848583566254354, 0.0022966881194274847, 0.006477820950805944, -0.024795399116452142, -0.0005979086433041972], [0.022107173429539478, 0.0015596206224853357, -0.002365842845960006, -0.0167024494360999, -0.03865587446791206, -0.00014262730205293796], [-0.00536684255546604, -0.0008650858680308432, 0.0004900563549860221, 0.00753666621840717, -0.03587276992552493, -0.001122024224371413], [0.020754235413374764, -0.001464368839016724, -0.0016772971512166112, -0.0016431422474482025, -0.05073479143287303, -0.00043463574282022677], [-0.015835108357329954, -0.0002839283381381832, -0.000888915663942052, -0.0011646966353021954, -0.01817217397254909, 0.0011448229672613728], [-0.014365873169800392, -0.00022569334329008848, -0.0009379450597070103, -0.0036355034330727756, -0.016763533086997233, 0.0007285480928674085], [-0.06831672025556655, -0.005966034375439096, -0.00041204692439082084, -0.04969195204549994, 0.09779395452381645, -0.004440534256253292], [-0.012704994622830338, 0.0005563444621044674, -0.0015995003793527883, -0.007029884112042975, -0.01388173588916938, -0.0005402294587090359], [-0.019296928624090107, 0.0006013588309353998, 0.008575998539216426, 0.005904787231139474, -0.02968383125493336, 0.0038771867063035373], [-0.014881458932452995, 7.084242606724093e-05, -0.0020571007862515696, -0.0006122801501895019, -0.01696107166898332, -0.0007589308881898979], [-0.10575238121445704, 0.0009641662105627538, 0.006264075866775416, -0.008727275762260024, 0.10729979712562808, -0.002921001273868542], [-0.01660207489999685, 0.000870934222198324, -0.000608656938473221, 0.000441815391474193, -0.02004062570623597, 0.0007386079310334302], [-0.016002093970517668, 0.0019743056725649, -0.0005962622124142459, -0.001681585004397373, -0.02109321280765064, 0.0021988483224149656], [-0.01218035978822278, 0.0004303404641395153, -0.0019481791173267954, -0.006744407011610339, -0.014222075342246026, -0.0005353192047335956], [-0.022416500553803127, -9.350475311869435e-05, 0.0011340077337401665, 0.026017726431846813, -0.03432852241528067, 0.0003201268899487808], [-0.01543982643339832, -0.0010093436399867747, 4.4880053057533767e-05, 0.0060972993158126135, -0.0241203031341077, -0.0007727061613774309], [-0.015968238596033177, -0.0003136517256872724, 0.0017930678400616595, -0.0003078260394388697, -0.020026238956276036, 0.0012895541440403181], [-0.013643158567929026, 0.0005267323471308842, -0.000157442703194008, -0.006267991343801017, -0.01677370363340441, 0.0011155639011975029], [0.039133826179299475, 0.0005420153328817907, 0.014074421084069942, 0.004298172358925132, -0.07316121903067686, -0.003087215924499391], [0.015243170217644863, -0.0002544145303196278, 0.0013468219110770552, -0.013799278347167042, -0.036930758984906734, -0.0008055402663285408], [-0.009466051085652866, -0.0007835374448718448, -0.0009585757473343776, -0.008615738824953341, -0.016471163014642097, 0.0010950661174544802], [0.034642736165268966, -0.0013440941066232247, 0.005743550290410989, 0.01659116489354903, -0.07658769661906945, -0.0010789939568695676], [-0.010340687733893754, 0.0009412613319185465, -0.000538020253036052, -0.0019752107420342824, -0.0192239605478552, -0.0015633820550992926], [-0.011916073482903383, -0.0004649708969838987, -0.0012202685291510076, -0.0056487269499469975, -0.014232459984269709, -0.0017175001567450597], [0.0391844656018862, -0.0016116935852487275, 2.134789389519554e-05, -0.00165350216649822, -0.055073065481402964, -0.003150885595964766], [0.018811628213059494, -0.0007876681537271391, 0.008955394069269451, 0.010812474606409439, -0.06407467908349378, -0.0005838163181841471], [-0.01751090359400338, -0.0005214370997063332, 0.0005190932849859393, 0.00507226984679805, -0.022497728351731867, -0.00026129408634246526], [-0.0169778391806787, -0.0010924334521268061, -0.0011867637457478276, 0.006002735712902036, -0.021431094458552853, -0.0005146048757959173], [-0.01176447668484914, -7.44100072831802e-05, -0.0017409118742516088, -0.006677644276934908, -0.013770300594009558, -0.001172256562671632], [-0.019925970139595767, -0.001098618593710441, 0.004974558314934592, 0.005363579822445312, -0.02326836939301295, -0.001245180011060871], [0.016086479157857856, -0.0005203526896035798, 0.0021055192001467747, -0.0027424038092936, -0.052084592024950464, 0.0019553501658428956], [0.0351521453289526, 0.0034578835541401237, -0.0032602608884319557, 0.004986764969448371, -0.06766643773430292, 0.002129904770193816], [-0.017205755252389588, -0.0004913557599143656, 0.00023472597132510078, 0.005140695681838695, -0.021906736791167187, -0.0009715738496927087], [-0.01328188118535065, 5.2219290575715707e-05, -0.0007429942923173021, 0.006530726810902976, -0.028231636357655267, 0.0004735657338444503], [0.018320026095771883, -0.000958096264109787, -0.004047139551620414, 0.0024622572226750874, -0.05010993955650967, -0.0008671079462071372], [0.04340098559157705, 0.0025401888304456706, -0.0029036908691215226, -0.020224209151925854, -0.04816849070061295, 0.013405216299637458], [-0.014453913557570025, -0.0005040020510237945, 0.0002570527848810159, -0.0008949480555358764, -0.018345385850631883, -0.001258803270119505], [-0.00651850881525639, 0.0012848344865968142, -0.0010913707489828505, -0.00916843972249719, -0.018656214257915364, -0.0010503009419450778], [0.00944355900898283, -0.0011677858032027452, -0.002452162996897804, 0.01004147273230679, -0.049676234760630114, 0.0022222629305522373], [0.27012745577921266, 0.004488497293267226, 0.014297035193133502, 0.06736213711878267, 0.44009061606080024, 0.005319789690333539], [-0.059986092150419106, -8.544134100704865e-05, -0.0013041187358003906, -0.010818249911329578, 0.0642245590010325, 1.934313752369012e-05], [0.04013202442189447, 0.001763670412103896, -7.186467797404242e-06, -0.0018680986975851276, -0.057480808863022224, -0.0022396008055937766], [-0.002529268817749421, 0.0009036580639006711, -0.0015419855561688322, 0.015042907341114768, -0.047910409024469094, 0.0008350979933718436], [0.005708714301050922, 0.0001835239946389572, -0.0021899102966440958, -0.008836183403608838, -0.028467839793605485, -0.0015983048018315387], [-0.013255630216571772, 2.620855071459524e-05, 0.0007732186404124554, -0.007190456593169306, -0.015658197667021694, 0.00010485728563566146], [0.033682431062771356, 0.00015739270326388488, -0.0038382780832341232, -0.01077689588503841, -0.05174747627641896, -0.002677173521343765], [-0.013213388192723873, -0.0001340868627961906, -0.0010388805953508585, -0.005419611426747581, -0.014837098286307217, -0.0005569346360743462], [-0.011837732790620045, -0.0005123434758080887, -0.0008732539642539604, -0.007544355515511636, -0.013848325252470126, -0.0005839890013361797], [0.017641280518019553, -0.0009730914877726052, -5.3683348609571224e-05, 0.01225953338176763, -0.06333189043085656, 0.00259118470078489], [-0.01246867500514888, -0.0006193473084869355, -7.779636572596831e-05, -0.007558631460618346, -0.015130922695178106, 0.0006553728351581938], [-0.011253373478845734, 0.0012641505344738197, 0.0008378839205792725, -0.008384340782303601, -0.016872476770692604, -0.0007918434232112116], [-0.017512272770348676, 0.0016701003819001594, 0.0011065498584768084, -0.0003549751436529783, -0.01997501922549676, -0.0001343831008786365], [-0.00890363726036823, 0.0029494661326747726, -0.001017951035764483, -0.0006796698856195682, -0.02210008623571714, -0.002114788381872055], [0.020730779514075613, 0.00010726580499857512, -0.0007804571595370305, 0.021016560516530697, -0.06834872674762602, 0.000407911404891496], [-0.0172551779750967, 0.0012957817550417203, 0.0007195496026891274, 0.01230889820141323, -0.02704488590294599, -0.0010574990144348534], [-0.002666918559928963, -0.0013742255361645687, -0.00154274807792113, -0.010881218828147318, -0.019296008808107872, 0.0005611198102697693], [-0.010884571324774325, -0.00017594568101403515, 0.0008086428400082727, -0.008306151912062154, -0.017341936015791946, 0.0006999620936341708], [0.03514307191444658, -0.0003119720082880673, -0.0035219303242578033, 0.002120855589415463, -0.06710731018152555, 0.003477285010209432], [-0.017473650081270446, 0.0021295589944991303, 0.0013565115045744967, -0.00745870069524403, -0.010983522479184809, -0.0007701972433744081], [0.009098577172448963, 0.0009194424260466654, -0.0013928557286956857, -0.014221467643495774, -0.030058089777942953, 0.0021210602183053826], [-0.015064342806947555, -0.0005593335236350369, -0.0012182325096471, -0.0003611618604255654, -0.01880771596171925, 0.0008107866623744531], [-0.015599186635458102, -0.0007858552516445033, 0.00018981706110734645, 0.0004164950639849993, -0.018634501857969898, -0.0007867683800199399], [-0.02685005670057701, 0.0015679640250970625, -0.003581881249247225, -0.07932863783060237, 0.12638534389965744, -0.014922494049089931], [-0.01346554737787826, -0.0007029158172157854, -0.001061552260463504, -0.005981641568011172, -0.015403561092161072, 0.0014152181157297574], [-0.04205091738453698, -0.010818024348353375, 0.0022475602465595003, -0.06281258265349667, 0.13681858687663395, -0.0016917655939493105], [0.07984787818150224, 0.0009092687680317272, 0.016539387676237447, -0.08707433316562936, 0.2354981412564746, -0.0006762950975686964], [-0.018845833893194115, 0.00017960386244149538, 0.0009067190430361934, 0.00382997494242265, -0.021974320057780734, 0.0007038561030744939], [0.03366988104239421, -0.0010741438371249402, -0.001997591770441382, -0.01439861777527029, -0.04942230731048294, -0.0019772203490745852], [-0.014380244050148346, 0.0007401392310063855, 0.00212050887944402, -0.005689897377138747, -0.016161937514364864, 0.00017143083120148598], [0.03668898364408962, -0.0022466939472998606, -0.002876679908403507, -0.008042314507062675, -0.05605091916077121, -0.0026723761205523383], [-0.014219305940091605, -0.0007255470617023404, -0.0015660181724463287, -0.0008972430112827009, -0.016782982551506256, -0.0010089032629708359], [-0.012395774984972872, -0.0002925485081109348, 0.0003272332949133028, 0.0003451876585410525, -0.021692596698412496, -0.001491500761958111], [0.023468514223046077, 0.00010036486133210275, 0.0021156392426662737, 0.022339035222812092, -0.0666598168869437, 0.004317215718039566], [0.010119010904319832, -0.0013017600810406864, -0.002403422030800381, 0.007738331659788581, -0.04704681319248251, -0.002305347259784898], [-0.01858864012268866, 0.002077168136517393, 0.0016841693215861415, 0.00712167552996173, -0.02327868705034433, 0.0007843141849676991], [-0.014408291088522934, -0.0004196749086403211, 0.0011006984599056867, -0.002934866015518549, -0.01762186543086161, -0.0009160010163623334], [0.03474151401542822, -0.00018677180596763195, -0.002436190504470003, -0.013056868953497568, -0.05000317250267557, -0.0005085102488174668], [0.017181860003726453, 0.0013401026317616572, -0.0025584617883818643, -0.011272539319571456, -0.04120711265045182, 0.0013161511229170138], [0.31163314765715755, 0.01627454305650104, 0.03410772291674606, -0.04330159156979251, 0.41705325555607203, 0.0022829223833148557], [-0.006558332265837227, -0.0013427001592545137, -0.0021642742536178337, 0.00088659015112483, -0.027788442650225512, 0.0017671591778101981], [-0.013306066643956938, 0.00012588505347695236, 0.0005995645058916991, -0.0071225829677773, -0.014984833772987908, -0.0005119661746465736], [0.03418485013386518, 0.002944030271175904, 7.355222430453381e-05, 0.009089156481569338, -0.07637493392561243, -0.0013666551853024342], [-0.012644926124657146, -0.00035962729353225086, 0.000973721624442466, -0.001737826668538622, -0.020237277550125526, -0.0011940639875889856], [-0.016943382776215037, 5.6017537951446405e-05, -0.0012774731635908647, 0.005997837074154464, -0.021731341775470682, -0.0013016568968294224], [0.006749897808684019, 0.0023917630169048394, -0.0015214133167156954, -0.011553546531558219, -0.029852249902091732, -0.001414451075223269], [-0.08138239794828533, -0.0015869509347125777, 0.005243736474285435, -0.040010638416175436, 0.09547425095776686, -0.0018546667995454306], [-0.012970909394991402, 0.0010169372199293342, -0.00035254675365007347, -0.005700176512216498, -0.01603830885658597, -0.0011549957024854383], [0.03247886371455486, 0.0008447292820350118, 0.008699354976429185, -0.007663393007947315, -0.060093066518529456, -0.0017879170179709103], [0.037330195206736555, -0.0012500183935960007, -0.003010656874207355, 0.0075418855006113734, -0.07461564594731628, 0.0038042405077717578], [-0.016885074716308263, -0.00016314676094265107, -0.0007421917454040681, 0.006385263228460202, -0.023036758656737218, -0.0007580913490680845], [-0.018004257857215922, -0.002334647754314764, 0.00617795645203999, 0.013805349003633057, -0.033680625508139606, -0.001163774336002825], [0.026600253318597878, -0.00333711265937713, 0.004736323019153939, -0.08753681721172385, 0.18437881643379958, -0.02173789147187885], [0.057044145000817004, -0.004575007991618633, 0.0008909193437386889, -0.11611959420972272, 0.15990733143447378, -0.007403349133243946], [-0.09788655147305086, -0.0024096786046881088, 0.00542375619323832, -0.03717342943697176, 0.11697072350402404, 0.0035061321984008577], [0.03278891051546179, -0.0018422559140926279, -0.003305237179311394, 0.0019047909671529551, -0.06278644773768516, -0.001959760651525581], [0.03335141613616808, -0.002094756982217738, -0.0014387469552708844, 0.00945879274365834, -0.07015548602316832, -0.0008212189191694872], [-0.015562650044092397, -0.000535570729477138, -0.0006803582716599449, 0.0007745028004640729, -0.019258750394562266, 6.28266393275873e-05], [0.040184643606107545, -0.000389054471752491, 0.010755512305095551, -0.006517784026543912, -0.06368348127600235, 0.0012954019583335885], [0.027558439826782796, 0.0029392699748739177, -0.00015773752451665504, 0.018472152485528078, -0.07813072297677152, -0.002131401785896584], [0.014548746679497319, -0.0016215075444393288, 0.004784622145687414, -0.01449083778802018, -0.03553950790976245, 0.0012851510837038936], [0.010119030954816498, 6.956568668621232e-05, 0.0018727989123689518, -0.004671491542128501, -0.04219712384920417, 0.0021072198374609283], [-0.01278660237316177, 0.0002838349695109769, 0.00042757753589316226, -0.0075757878718880735, -0.015469421185909143, -7.960107444522028e-05], [0.015269902589490982, -0.0003240787950489746, 0.003947886486764543, 0.0008082356405060832, -0.045177376660323836, -0.0022245692613888203], [-0.017901478808536247, -0.0012309006523895198, 0.0009229684625972512, 0.011019401776903246, -0.027205247401764335, -0.0008047433768105319], [-0.013231008552883892, 5.10621760122276e-07, -0.0015649135079748501, -0.0017986299160222465, -0.01738694733767999, -0.0012190113071991662], [0.014258113691051762, 8.571540561762816e-05, -0.002918520615876744, -0.002426500081504076, -0.04680742629620551, 0.002608617896916901], [-0.011947956153700248, -0.0006225455722569078, -0.0009173904333180218, -0.007488578533792732, -0.014157453436672466, -6.607587025968095e-05], [-0.01364800636633442, -0.00025749261297855665, -0.0012544052560762645, -0.0036881896924305445, -0.015483223758517134, -0.0008686823136631189], [-0.01782858878510056, -0.000459646448227789, -0.0019992540357191804, 0.004869154836546007, -0.0205110743025277, 0.000729408735029214], [0.03609929075350556, -0.0033213751014839385, -0.0020196599395957814, -0.0033810630491477455, -0.059613877453613094, -0.0012966485429982404], [0.1207663288227328, 0.02319593162554872, 0.031197439192826824, -0.021236311555237686, 0.30201949276632306, 0.03844442073510772], [-0.016083067621201413, -0.0005414846466348191, -0.0024228770006660856, 0.005401143717894775, -0.020814783497875494, -0.0007389309515169976], [-0.015159032263217694, -0.0004459923394858509, -0.0017117878275469635, -1.9130279087279877e-06, -0.018182425982538792, 0.003634484774031301], [-0.005725166830627229, 0.003387741584008536, -0.00010574487927758458, 0.00966436280714113, -0.03760113766724004, -0.0023200550140048556], [-0.014007133251861753, 0.002009741298999807, -0.017162810591020802, -0.07472396582064277, 0.12833542680160867, -0.003179036214860853], [-0.017355843848258914, -0.0004101580577759157, -0.001265121808687952, 0.006265345323907533, -0.02227005010390734, -0.00016417150527746024], [-0.017351907952994947, -0.0013019785909712844, 0.0009242287886622514, 0.010673061416737387, -0.027453986401123096, -0.0006894172603104076], [-0.022451419132836166, 0.005283502135473082, 0.009701266592383842, 0.007347265381665144, -0.029436982204187948, 0.004951605322740106], [0.008388453130636698, -0.0009341627707592488, -0.00215016417291506, -0.012246673720591485, -0.026356733683142033, -0.0019007187832288863], [0.031607013887481834, -0.00015981992022162515, -0.0013953368979633097, -0.008183021934785303, -0.0537777625403651, -0.0016244059274798598], [-0.014460703023325045, 0.0007834485283984169, -0.0018994266170312823, -0.002813327647025671, -0.016020567504878375, -0.000789423736138089], [-0.016421856526071026, -0.0008325502109647054, -0.0017849939471057347, 0.0053156367500552795, -0.019847261659301074, -0.0016289744066127986], [-0.01252770381483186, -4.6916537214618e-05, 0.00023547821996527452, -0.008452343466598504, -0.018397254282783066, 0.003988739881462744], [-0.014101792069373877, 0.0003338802067389802, -0.0007141043979504775, -0.007647321177500714, -0.01703289870835268, 0.003962236146438699], [-0.07158596674637398, 0.0011952941404483944, -0.009435681053031032, -0.034349019905587946, 0.08473413078976591, -0.00450875722522124], [-0.01470221328025069, -0.000536210103192929, -0.0007036444684806045, -0.008475447971561663, -0.015037918075148754, 0.004255433898634588], [0.03784175029150917, 0.005707228777074737, 0.0008543402977802413, -0.01553153253225594, -0.05435089475936801, 0.0002791079252597569], [0.008359927903418694, -0.002194139347726669, 0.0003106450374284478, 0.015461628260764746, -0.05086177595171937, -0.00377628590216592], [-0.015038707107220438, -0.00012712348562084, -0.0012190083807750859, -0.005991520473748863, -0.013383812714898061, 0.0005601721622632288], [-0.01448562296979798, -0.0013713404356132685, -0.0004302578793001383, 0.006830101533850727, -0.02667168836381817, 0.0009288081146787566], [-0.016363268267475535, 1.6276168831273883e-05, -0.0008892083159204279, -0.0004550478219477265, -0.019453523417405014, 0.0019447716539173245], [-0.02285268883428639, 0.0031812874573829905, 0.01709214070812193, 0.0005952931924032255, -0.013849934538682752, 0.0010505686817275986], [0.329407631007456, 0.012968887167271036, 0.021514658041501947, -0.023851169409384133, 0.4088414955215321, 0.0168530214811462], [-0.016887405979180554, -0.0011490395793900053, -0.0012435627527176898, 0.006078929978799303, -0.020976399501306385, -0.0010225221662047389], [0.010001712964724475, -0.0002052552686433707, 0.010314566064194685, -0.0017695279683372222, -0.046445034069082035, -0.0025131283895232152], [0.01805445910954395, 0.001122073424838409, -0.0037469132956609603, -0.0007038341678798265, -0.046777030893646765, -0.001148754177194846], [-0.014240118560193377, 0.0017984829681282522, -0.0017720807902158315, -0.005547096290935231, -0.015702080379982643, 0.00026289305319876196], [-0.013124048854844271, -0.0002745166533093964, -0.0008934334046533568, -0.005065699581576946, -0.015155494227614119, -0.0006868072780019776], [-0.09414261412964103, -0.005373866544232886, -0.007287170500583799, 0.015268905396562197, 0.07593248530546058, -0.0068834538132793175], [-0.014554208752453134, -0.0004589953443199818, -0.0010860162250315424, 0.0004431383583781559, -0.018076475157226036, -0.0014674428793475041], [0.005879244647060269, -0.0014738214313437551, -0.001854423240117633, -0.0014606588228749096, -0.03649917047111055, 0.00020882931838662425], [-0.09355862410726178, -0.0022986444204856522, 0.004685609532897411, -0.0373344750125908, 0.1055825473038388, 0.003973586703602195], [0.034442426308574824, -0.0034278071073584934, -0.0035225707043151956, 0.01630193463706598, -0.07137938116260584, -0.0026146019713611437], [0.03868239411246801, 0.0028940634876020456, -0.002818686462047853, -0.009319510191698249, -0.058431454836028, 0.0017931938897039774], [0.014859562898910427, -0.0009096155974382024, 0.0044861796157830685, 0.00026668232418418136, -0.04994986785717443, 0.001047058615734807], [-0.07454443980504631, -0.0010887414828114343, 0.0016067049213495254, -0.007461575919245463, 0.06260328781973298, -0.0011485688673127188], [0.16913739337432776, 0.029252920311157272, 0.01049051837459677, 0.18804549827600034, 0.4374840384769794, 0.006897567694875141], [0.03722047160593886, -7.777015597059577e-05, -8.872440429459974e-05, -0.0038895172385322845, -0.06468340419812287, -0.0024310556090185054], [-0.005171490406059474, -0.0007255077878986864, -0.0013641808304843083, -0.0073321506820861046, -0.01962501698914462, -0.000981653304326846], [0.01228291963885826, 0.000495621185834752, -0.0028457874693928282, -0.0033230124874689946, -0.036722227717589914, -0.0025875131502413324], [-0.08626669743375592, -0.001374717629709892, 0.04051755506671745, -0.022074559964881557, 0.12814618279594883, -0.001562986499542384], [-0.014745475503788151, -0.0007726480787599432, -0.0018597289076243706, -0.0007559963125824791, -0.01659548692853826, -0.0004706642687068448], [0.12036255384897789, 0.01397387920305314, 0.017361013830624526, 0.200242123880293, 0.38395437864272314, 1.9145832423671728e-05], [-0.016271448084210344, 0.0018593547377065243, -0.0009843592573127585, 0.00016267195335244522, -0.020307535777175895, 0.0003413164276399449], [-0.01481518453376102, -0.00025074807232613837, -0.0014548641757654108, 0.0006002541831363031, -0.01795950091909434, -0.0013199564821894376], [-0.00995430866554652, -0.0005431365585151091, -0.0014539429466844855, -0.006471145246664744, -0.015555224311530851, -0.0012222422710583126], [0.00033843359878815547, -0.004101664187494051, -0.0009740512484347995, 0.024934308112663256, -0.04416607789491199, -0.0007547579044201076], [0.020954027416785922, -0.0005557797598427602, -0.0008833497133297877, 0.020314173710240146, -0.06520841357190639, -0.0018206580819471732], [0.035596607040829335, -0.0018736046899257485, -0.0010273652490640094, 0.007029581886751077, -0.07143312578320875, -0.0034920932053818356], [0.022402198074859107, -3.0104879942921078e-05, 0.0014866276226179985, 0.01061995186513018, -0.0646669314026694, 0.0008215920533383635], [-0.01290863343966484, 0.0008127209781983826, 0.0009438562593153769, -0.004167510989339026, -0.01916209115606945, -0.0007183416524405063], [0.04834344957082486, 0.00026817373336505816, 0.0032051845936293664, -0.0007983860286953911, -0.048712629607333115, -0.004255792261790847], [-0.012324152317740084, 1.8180668485304592e-05, -0.0008762796544641428, -0.007382182337553589, -0.01525017070136681, 0.0006146043426392712], [-0.014652578383460528, 0.0005458179707338708, 0.00039661291556038344, -0.008668361468916787, -0.012397183842706709, 0.007909026142123037], [0.011362210352612444, 0.0011118802642751356, 0.0012199013018474084, -0.014464840607601935, -0.033173046264348444, -6.10504678462714e-06], [-0.0029426454735495514, -0.00029990280732119856, -0.0003325975224805679, 0.0019631751862880143, -0.02745787767291509, -0.002519040598910559], [0.029642511068840145, -0.000872035930349442, -0.0033830765446051145, -0.013085331866693086, -0.046960562336957536, -0.0005415043902350643], [-0.013422579193379039, 0.0018117707918826464, -0.0014205859932259865, -0.006532517178372292, -0.015312462324236102, -0.0003236261026692765], [-0.015108385627696078, -0.0008119637878913842, -0.0019781155467190575, -0.0006339069407434865, -0.017564544116979212, 0.0008969160200291636], [0.25610183559791055, 0.004650112955220258, 0.019183507863810456, 0.08204889603621815, 0.39178673547473697, 0.0161955787387703], [0.031607403438324515, 0.0015421591746722942, -0.0010539555644846095, -0.010274569221995367, -0.05443298911786762, -0.0015880487086492054], [0.02220063627417825, 0.00242157451004051, -0.0007025895099614623, 0.010532030078170706, -0.06200684152565427, -0.0023506921797149874], [-0.017452626313361076, 0.0030626682794355794, 0.0007992226607166311, 0.006852372723695193, -0.024743997616176974, -0.0012176397343094656], [-0.013108926171292947, -0.0005368637395864834, 0.0015036723286999537, -0.007301654454522559, -0.015064909842323676, -0.0006913181209743585], [-0.01699647541096553, -0.0003826749542716705, -0.0024030763572273576, 0.005359552106043597, -0.020581387863349194, -0.00019593752022987887], [-0.09169464887803626, 0.00029438449246820973, -0.007381217957221624, -0.01728312901915223, 0.1019548943094407, -0.0028402829474988814], [0.04096195829286196, -0.0001262742950324039, 0.014580516844091649, -0.00523215107395568, -0.0637931383198098, -0.0011742447814890797], [-0.11187437608231714, -0.0010755014495490048, 0.003637529834901047, -0.0192750498250365, 0.12202665611398464, 0.020785955083230778], [-0.019791520030757515, -0.000255090968169609, 0.0008212416156069915, -0.0010530186453145028, -0.014241295099750233, -0.0006803168716152133], [0.007411409563086714, 6.300494130268925e-05, 0.001128429057043993, -0.009379122095799875, -0.02857083390461118, -0.00085288756102239], [-0.009562806115034385, -7.740846339399948e-05, -0.0017115722317246902, -0.007725419285557406, -0.015237642774710418, -0.0008851511295791221], [-0.005726962095171294, -0.0010183130473032443, -0.0010403404301659097, -0.008359794737535516, -0.01819030574219583, -0.0008642839476282429], [0.005879630872799343, 0.0004657692610123861, 0.001896131461414855, -0.0033351614082906106, -0.0252465816533546, -0.0021097885335813853], [-0.015153051228564262, 0.0004520211062708757, -9.634618357148062e-05, -0.0025692821821072603, -0.01729722444737158, -0.0005361170646563369], [-0.018399236261931143, 0.002279575356439717, -0.0008269890897822555, 0.004214202373451777, -0.021303939305972793, -0.0011636130722054176], [-0.01653107020990183, 0.0010870848565352398, -0.0009541945261746279, -0.001136649504840042, -0.018627349569566968, 0.0009621789539481598], [0.010228530129777563, -0.0007917299220827061, 0.0006530947975868348, -0.0014137790687551632, -0.04651854882368725, 0.0026424328871606398], [-0.01647341458133395, -0.00045479639940320245, -0.0004967020220216416, 0.005788600150187086, -0.024489169541127235, 0.0009254823936988731], [0.018032847032687156, -0.0006742419675456747, -0.00324804796889195, 0.013158690510566165, -0.05433925246538775, 0.0006200048585719861], [0.01985643608528284, -0.0004947055844076188, -0.0004412981465279238, 0.018498730465002357, -0.06530011717408674, 0.001014287688070377], [-0.012645568656266207, -0.0009172907066179321, -0.0024744165989312008, 0.007969151261215373, -0.02770332269981288, 0.000571447400412775], [0.018914324360383625, -0.0005947177427578157, 0.0002185716802319973, 0.01154349824806989, -0.06497001093103923, -0.00031166561488844293], [0.01835806223215051, 0.008363156380867133, -0.0013366877091923336, 0.026055493950435545, -0.06470251039805575, -0.0027708477895384684], [-0.018164654244222228, 0.0012218524611272285, -0.0010920030425694046, 0.0006900375008359497, -0.021378554830590455, 0.003523322155418838], [-0.015797832251363633, -0.0004661990885794897, -0.0004596234641672206, 0.000564838734007618, -0.018951989910870746, -8.919401902659241e-05], [-0.023504089376528276, -0.0009368161886224322, -0.00014191439565927204, 0.02235806441543808, -0.0324771057940491, -0.0004981386605790886], [-0.013713009737906345, 0.00014538096508623697, 0.0007388411170924814, -0.007277057167671622, -0.015664894843846887, 0.0005707396672460916], [0.032929719733234504, -0.0016135125907616422, -0.0028538618588487747, 3.525296388917518e-05, -0.05897325054205048, -0.0013910143721294216], [0.018613483659920594, -0.000883905517680774, 0.0015374028561593945, 0.010597789384047129, -0.06137714765738048, -0.0008935050780070708], [-0.016891335134348342, 0.001362219802322254, -3.299376757365615e-05, 0.0075435691057727985, -0.028756918933094233, 0.0030040303554925364], [-0.023547443183199963, -0.0015169225631320809, -0.002793216790625104, 0.02043056698792278, -0.02857698751628341, 0.0008040030653177212], [-0.015375817546682857, -0.0006263820624096824, 0.0003492294421811082, -0.0009708857323585265, -0.017800398205161855, -0.0007757458955682363], [-0.013051191536965162, 0.0013380070876799065, -0.0017641101838566946, -0.005688630254501821, -0.015364268357472522, -0.000669806754883751], [-0.021102792384136333, 0.0005096280610849818, 0.002703425708538527, -0.0059323675133979535, -0.004651865298554029, 0.0016073047597980753], [0.03712046797844763, -0.0004831314161002145, 0.008503637673352972, 0.00022410175680393155, -0.07296977904664882, -0.0031310112315697173], [-0.02047777250909692, -0.001497112589589546, -0.0025506803568444178, 0.010674416176091417, -0.021235968434260524, -0.00011288228630003924], [-0.01641542192248896, -0.000707521848308356, -0.001941914446263099, 0.011629273602266553, -0.026112433037432655, -0.0016519823477735506], [0.014053069120528722, 0.0016418170948530636, 0.00048333609022097865, -0.0018492849867756277, -0.04145441630372157, -0.001546743237327746], [0.014237849382048773, -0.0002257386437771905, 0.002300035582847785, 0.0004604417483246613, -0.048753224349339076, 0.00011396961322827558], [-0.013697360768533227, -0.0001751979109858785, -0.0016721090313355203, -0.004958076725713479, -0.015324975167936333, 0.0006277196045043805], [-0.010058134612945467, 9.162274169745535e-05, -0.00029249494993518674, -0.006077835405047287, -0.014969581343830607, -0.0013935764299389491], [0.03854394739506188, -0.001861298448728275, -0.003321218530195174, 0.006748085970912588, -0.06824547273528052, -0.002480710318437117], [0.01146150359720329, -0.0014625136221405651, -0.0017484817524883113, 0.015783499823883957, -0.05830989725662805, -0.0009241107898304172], [-0.09077707364991953, -0.0007101764021348031, -0.007706227535646707, -0.02842249554343839, 0.09525249149262538, 0.0029134816385140037], [-0.003477434871193253, 0.0005468458500364295, -0.0011084639332200638, 0.0005770216462663207, -0.03155398096948806, -0.0001839877224013793], [-0.0038482597528486174, 0.00039738341128327676, 0.008331524200107198, -0.0060936166043360565, -0.032351854041335386, 0.004198156120462897], [-0.019759882465534475, 2.3272945992188443e-05, 0.0076390965054136395, 0.006032111433698645, -0.026179531716755088, -0.00045506670281499816], [-0.0808716612774688, -0.000572909143608739, -0.0018763338677925973, -0.03800945254992466, 0.09708379031386324, -0.005620100141735147], [0.005311424984022191, 0.0018595348528567378, -0.0015496078222095705, -0.016427898450197067, -0.024149496246436458, -0.0002439573180358997], [0.017574026628111444, -0.0018083615362271136, -0.0016673483790881518, 0.014721607314767281, -0.05883840152145444, 0.007556572731986067], [-0.014876466563044068, 0.0012494974417511587, -0.00104573604291699, -0.004374366329207045, -0.0171938837323772, 0.0010409552257940975], [-0.014156420293628251, -0.00035995556764075, -0.0013495466089835256, -0.002144126339139892, -0.015956477515505017, -0.0012334736751026221], [-0.016056873366713298, -0.0008887122631116032, 0.00043982408998105766, 0.000995761711391274, -0.020656193979371972, 0.0009661938078244649], [-0.017814517374320035, -0.0005600820235516322, -9.135907345428472e-06, 0.0005287779790080396, -0.021044602129641748, 0.0036995594558507355], [-0.011189023488115379, -0.0008426664508142589, -0.0006624155806006604, 0.000479779631566899, -0.026502235210470744, 0.0035165610984340527], [-0.012116144850351288, -0.0005365747858558973, 0.0002036435355381949, -0.007335085574899483, -0.014677741945360525, -0.0007380963790710504], [-0.006444827757592803, -0.0011878547104592022, -0.00022131900409818922, 0.006271549154522756, -0.032358322232340184, -0.0012592254500324311], [0.015188897606893513, -0.0008437720070086015, 0.002413236439963933, -0.014728452239912121, -0.03868216591568815, 0.0014522561157514007], [-0.14537367304823623, -0.00285753522794929, -0.00204835525994505, 0.036770177872313015, 0.14486814117922048, -0.013284945991593375], [-0.0005799041873681005, 6.220967960101002e-06, 0.0012272047663985217, 0.008198707392992699, -0.04048998802945026, -0.003562240910532985], [-0.012435205980369653, 0.00359207560145148, 0.0008871569064943665, -0.008681704800508575, -0.015537724827618792, -0.0005245968994488512], [-0.10685603037416566, 9.705551361700071e-05, -0.003590263808007472, 0.0018682766772788776, 0.10127199987810206, -0.004741037886824651], [-0.1520368743235585, -0.00523301066508205, -0.003355693820350889, 0.019485789241549128, 0.12930400904528389, -0.004614219477841561], [0.01790271396350387, -1.2953035288897092e-05, -0.0010680043486646566, -0.00880546551426331, -0.03812914037844035, -0.0009204840201799922], [0.011082319281492723, 0.0035705964207902864, -3.7865229289873395e-05, 0.011004540363124702, -0.05792566365591543, 0.0008560728197975464], [-0.0021272072064752474, 0.00011685616615122489, -0.0011139307319282075, -0.010925262882374862, -0.020288951961075725, -0.0008615033842972518], [-0.017497393200209307, -0.00045493117086012406, 0.0005873316787562833, 0.004540294476240079, -0.02099348864874491, -0.0013818131351820524], [-0.015543800325511637, 3.0491224719278528e-05, -0.0019475353253871862, -0.0005435217476558556, -0.017964429745015064, 0.000768795918850382], [0.01340430146328893, -0.000755017713623786, -0.002158934025642806, -0.0012656787517227844, -0.037488942125769906, 0.0013976044868036683], [0.03616153288476387, -0.0007206807156754417, 0.0019036070577340088, 0.008142801216520402, -0.07797315099266292, 0.001285890549320152], [-0.017958097362473262, -0.0013838790715427986, -0.0024821249646769616, 0.015085006650887516, -0.026367037791524276, -0.002093867460670332], [-0.016401517960462995, -0.0008621081188883126, -0.0001828436087555022, 0.0026152120062900314, -0.019689953182446242, -0.0006787891357370844], [0.032301893435299746, -0.0011261229339609036, -0.004651589545235529, 0.0092286263887122, -0.06802439696208432, -0.002095077049397768], [0.009591259670509637, -0.00040431645368618573, -0.0019092257995837203, -0.013353044501709673, -0.0269345109538421, -0.0009401619616880343], [-0.14213225989559639, -0.00889600331916999, -0.004862781481185711, 0.038856684569468204, 0.14433141660939666, -0.002526818387674744], [-0.015354712863191507, -0.00035084087558232685, 0.002216357224358772, -0.0006981269715688616, -0.018586909448342025, -0.0007591003990074388], [-0.01199840929651676, -0.000619599831195918, -0.001490252914397306, -0.006550116616927831, -0.013701595416107843, -0.0008400259248543786], [-0.012360291624990232, -0.00060267300938853, 0.00036728933646986654, -0.00818973485327156, -0.014841856548604863, 0.000427266699785288], [-0.012894231411881894, -0.0011634342616658458, -0.0006081618707030902, 0.005829704237120281, -0.026900216972455798, 0.0005363402795862743], [0.2704226846439845, 0.009120297890369496, 0.01410023733377295, 0.10107053244162718, 0.46553931403335, 0.0207433622283241], [-0.01595989043006424, -0.0001924897853626074, -6.522122012204278e-05, 0.008391718866328841, -0.028973965100003572, 0.0030284190977949723], [0.02323443303286608, -0.00038702645855952786, -0.002915353998171002, 0.01171920677619342, -0.059289822711308816, 0.0007718966923131773], [0.00714331323943336, 0.00043846893892025, -0.002105482191596864, -0.009292270305768697, -0.03267545292603132, 0.0029580899117099373], [-0.017810455474679226, -0.001308465267689247, 0.00016258043173917968, 0.006691081308063989, -0.022845105892372156, -8.963510506260725e-05], [-0.005482771307202189, 0.0007253543634030478, -0.0016383700560906682, 0.010421307565124962, -0.036560043213315636, -0.002665477351919555], [-0.04311633894004839, -0.018981647888496683, -0.017410791278345956, -0.04651732477339645, 0.14483540272357695, 0.001782366823376834], [-0.01248329527527497, -0.000322388599460932, 0.0002915906005613561, -0.0008659564177359402, -0.02034636345811815, -0.0014735868499714244], [-0.014096504011650876, 4.679922836871392e-05, -0.0006876030825987547, -0.007819114852129008, -0.01033297468954837, -0.0010606025924417214], [0.2721547697202211, 0.017533239778045325, 0.02393647674658811, 0.10334409748521876, 0.4943566154549688, -0.007656151565994242], [-0.012200344933731625, 0.0001324592588469798, -0.0009332086504331091, -0.007658022925151818, -0.015095179245470072, 0.0005542964959396016], [-0.016615643555186854, -0.00099483307793216, -0.0014403340817107063, 0.004339578703597698, -0.019502403243736553, -0.000986364745031489], [-0.013505273459191294, 0.0007663385398170844, -0.001635927795739615, -0.007529437107517651, -0.012152029091142292, -0.0011436710862262492], [-0.012938082303466044, -0.0005930757403700124, 0.0007060762782195868, -0.007345207512295379, -0.01542827799250757, 0.00039856727041936], [0.02988921465409179, 0.0012923556955676287, -0.00031325665987782375, -0.01409522751380802, -0.050286280418548726, -0.0016868057574248138], [-0.006871250589027189, 0.0003780616706856396, 0.0011882696412385551, -0.0016513126529502401, -0.02373399558481486, -0.001176439151798597], [0.05612550642456105, -0.010166863419977355, -0.025806969074477628, -0.12122865450415556, 0.18045534950685782, -0.005804559408999129], [-0.013148144662162681, 0.001840373044156787, -0.000465615119481953, -0.002159116265610625, -0.020025601540212958, -0.001241895456688616], [-0.009695980709528422, -0.0033279787059578452, -0.0004921901649737448, 0.02692950239196638, -0.03523381472602868, -0.000903347609287253], [-0.014725959973131147, -0.0002508866653964362, 0.0008899031910621697, -0.00015690801727091924, -0.01952816767632692, -0.0014279808589368], [-0.01694985330192678, -0.001707955495153897, -0.0010702332829059882, 0.01439550549947027, -0.028649172850938397, -0.0012182905685453325], [0.05946499704262937, -0.0012327121567816675, -0.0028631964803270743, -0.013685865692019112, -0.049769433719641466, 0.0008743062442350636], [0.009503935495681242, -0.0014665012134998635, 0.0011569917375848, -0.014152480644038097, -0.0336104977653786, 0.003368552389650461], [-0.011695495314647749, -0.0006369031761944635, -0.0009057035261232975, -0.007287821221578633, -0.014076114116362653, -0.0005979626450932604], [0.32621905016242286, 0.0038042645496091224, 0.029055733873996063, 0.033016431300451655, 0.4593653994791025, 0.002890707936004801], [-0.024931091811587067, -0.0005990787879577027, 0.0035679839487000593, 0.024261583843443315, -0.03090826495537392, 0.0009088677627753067], [0.009326801986607798, 0.0013649218924888112, -0.0014301214055981966, -0.01589437696878628, -0.03258263500178385, 0.0040154094970716765], [-0.005792199056552992, 0.0005620073167884221, 0.005229158202754908, 0.009111776845589219, -0.039701173109614796, 0.0024737631343685618], [-0.013617158743109793, 0.001126385977325692, -0.0012037579794686938, -0.004138271184714795, -0.01612647045793776, 0.0007592723879052989], [-0.11048046500509143, -0.0020703289144128156, 0.0033142607678092483, -0.018780758613922987, 0.11193130763250507, -0.006197349200220585], [-0.012901578904244685, 0.0015121385602131234, -0.000744906744043048, -0.00814845333291657, -0.015468410591768348, 0.0005512110127594984], [0.0373858277925601, 0.0007631362403119418, 0.000988130344252794, 0.0061540197663120175, -0.07822282053316951, 0.001231706389732702], [-0.015358104443719617, -0.0003225123308580987, -0.0016030661928271997, -0.0011008313277661268, -0.01786686082296848, 0.0010513751181394572], [-0.018650326177747367, -0.0008917899035175221, -0.0008574205312200999, 0.005556694544507905, -0.017662759565245306, -0.0019801126524919594], [-0.014457595563739777, -0.00041067965552321406, -0.0020720609683947742, 7.90401702352648e-05, -0.017191170531223314, -0.0011475334513542377], [-0.01254891434549953, 8.388521185175547e-05, -0.0017888613610295512, -0.007112748703501158, -0.014616769522747671, 0.00078340872092611], [-0.015011116590564996, -0.0008043333940580799, -0.00029004015866954824, 0.006282977600306553, -0.024920934746852117, -0.00045655271016187835], [-0.01333719265570901, -0.0004383016349902983, 0.0003399371457078081, -0.007637973385130673, -0.015070769093706617, 0.0009442996238287301], [-0.012386336826410941, 2.1313426302747298e-05, 0.00018847071285788212, -0.007455701372379318, -0.014997497428439296, -0.0005702485119311245], [-0.011993312558307717, -0.0007413788372400496, -0.0010564933894570698, -0.007129845912275433, -0.013351357138768902, -0.0009276121639508701], [-0.011585967741642349, -0.0003010919140496218, -0.0015568898581288374, -0.006923404724152144, -0.014099769065428291, -0.0007328766965987839], [-0.01846947856733469, -0.0011582059950901807, -0.002289069027343681, 0.0077148295221904275, -0.019849036292973878, -0.0011490396394480465], [-0.014464997120238915, 0.00025139193058297695, -0.0009769086661951191, -0.0025818802270067926, -0.016508854710233098, -0.0009187512069091098], [-0.1563696815809549, -0.008537172727633479, -0.004859552034414667, 0.023415488600385954, 0.13305007942447752, 1.7504984806240632e-05], [0.02687677061529764, -0.0011203900922929395, -0.004050586489585022, -0.009093724262546461, -0.04812122938599308, 0.0011424929484530897], [-0.01568709390515134, -0.000299207800154866, -0.0014311374467818902, 0.000537057769918496, -0.017656891949197156, -0.0006627266686332775], [0.01903374799935739, -0.003811230553002107, 0.0005210244740175837, 0.004701429450824071, -0.05348220320320906, -0.0009127681679880434], [0.18757905586657875, 0.03349074326309337, 0.006195053191841089, 0.14142448760251325, 0.4253024078376468, 0.0024451570002318327], [0.005113030646024988, 9.664450431815479e-05, -0.002368276137401804, -0.0104928454306468, -0.028844322437701332, 0.0012957688554067414], [0.03551824616360449, -2.9100328597383586e-06, -0.002367291683082148, -0.005945987315776712, -0.05523507659022583, -0.0025003138749933504], [0.017435144677727516, -0.0019591751403831214, 0.01227322851217231, -0.005819831695971536, -0.050148771382917075, -0.00031392830396153253], [-0.013069651506867272, 0.0005849966283429669, -4.165495002021471e-05, -0.007787621851207408, -0.015520507795619067, 0.000634439475370956], [0.07273884275832587, -0.002643034828155833, -0.054633719507378484, -0.034145336225422894, 0.2238097719847123, -0.005427714658271558], [-0.012975233724697265, -0.00026680885035278113, -0.001483843566842775, -0.005384324363074314, -0.01431355718504904, -0.0007762323099838813], [-0.01548613890602641, -0.0014599354664592225, -0.0016575326841063993, 0.0045287669550423005, -0.019405845934862845, -0.0017193139635874673], [-0.01521140200717838, 0.0013852194525994402, 0.0031803194708880543, -0.0012213934476603613, -0.018592825917299365, -0.00023991755134945941], [-0.08950185996243301, -0.0071759480870620925, -0.007878263012795152, 0.022330142343405922, 0.07348207758936146, -0.0016585298228580438], [0.020375962553263138, 0.0010685105237979767, -0.0035624597782495772, -0.000706539872986422, -0.04690414674815568, -0.0017213266776694642], [-0.01470412227307607, 0.0005951747087562648, 0.0020769352384749995, -0.005215040945264314, -0.015639145146536127, -0.00031380158235481346], [0.034684765583514125, -0.0007241867809064551, -0.0014593287539690048, 0.0007541623320378329, -0.06459576411182626, -0.0026096482688500707], [0.01886192863010456, -0.0015709609427999338, -0.0020453028633270515, 0.011070854293450503, -0.058464990452611366, -0.001384861998150065], [-0.08657341187486511, 0.0062734039705014235, 0.00483530015613851, -0.04209032340370021, 0.10014314781170633, -0.000942878564542514], [0.12422405253978552, -0.010187417733484158, -0.023001524508439307, -0.12263461700638731, 0.2364643608476015, -0.01389818747240988], [0.039094850847878615, 0.00023081247149742636, -0.00468334773006473, -0.005915732737955324, -0.05402777622411669, -0.003648806627239297], [-0.015866020327022896, -0.0008064546000477519, -0.00010654285027162538, 0.0003682931182777624, -0.017816305078680997, -0.0009729702622545472], [0.038316877026614134, 0.003393240788598334, 0.0007800123190123168, 0.006527199440310642, -0.07504181480217263, 0.0013244852276371676], [-0.0169993068756503, 0.001289334912281103, 0.0005009992787595089, -0.0008064835772026536, -0.02071887196360901, 0.0015343282254212493]], 4: [[-0.1711663868825359, -0.0011358215061465693, -0.005177145567012405, -0.0026194430348388943, 0.05182198826380732, -0.04501984506992747], [-0.21363666864475492, -0.0032721322335316346, -0.0014460020043704526, -0.0033528838329731953, 0.002036542332012712, 0.025587811050284615], [-0.1791547821511898, 0.0018625093772266443, -0.0027693159865829478, -0.0012989890458635128, -0.0020963045378288654, -0.027558071563157007], [-0.16914813086176464, 0.002313209315628495, -0.005944976339295268, -0.004165026358996081, -0.004214538907042288, -0.030507203515196587], [-0.1657631143105206, -0.00908013058361753, -0.0015799673670945707, -0.0008927954270717377, -0.0013991656919191773, -0.032907888635280394], [-0.18514743560980226, 0.0056534496550346945, -0.001568074099544985, 0.0019080788632683345, -0.0026707672038787543, -0.029798313620580812], [-0.21735505919583728, -0.0032546259202983294, -0.00614593871034966, -0.003374869486569462, 0.0008546922681686312, 0.02088385965294472], [-0.1501343812363162, -0.012470111918255148, -0.006086528657211088, -0.006055486085522162, 0.01593644979814331, -0.04202327523417183], [-0.18405805573263018, 0.010547481396989848, -0.0021665043536192043, 0.0030354656320103132, -0.0030003906072521157, -0.029155200100725295], [0.7302179135420701, -0.007623952985439776, 0.006917597292665892, 0.001977380819942901, 0.0029160272554109713, 0.03484503407535011], [0.6039339951405124, -0.01793937289248051, 0.019286304975195425, 0.01638923249461392, 0.028924828557626656, -0.08538471695763808], [-0.1724517624959646, -8.559156984723045e-05, -0.0054623273094013365, -0.001393735300560963, -0.00021021640243064268, -0.032019428937298984], [-0.16938773328683032, 0.0034727286525201447, -0.0034458780222180367, -0.003079603400477776, -0.0007922835022661842, -0.03656314880807452], [-0.15539353893929028, -0.01506099115300062, -0.0025603620954462464, 2.4983575602551284e-05, -0.003219206763735279, -0.03541394663963375], [-0.2055361711969583, 8.062443083292238e-05, -0.00435660847951462, -0.0012318098731603402, -0.0043228268246275564, 0.019366791943427622], [-0.22520647045463407, 0.006004138471326336, -0.004743234463785736, 0.000616770659450317, 0.006002032268527829, 0.01807676351911578], [-0.1693292567575132, -0.006877412383745108, 0.01402790993687983, -0.009013852027730208, -0.003602755185788141, 0.03473584260837318], [-0.1880841330939518, 0.017288906708221714, -0.0021485841544327763, -0.0002819860128472403, -0.0011943057026099948, -0.026496564411046784], [-0.22212956841748902, -0.0033707424429810307, -0.002119123849402556, -0.002366882241309937, 0.0010255745973658506, 0.02164022953330415], [0.4087594678937209, -0.1589188277781161, 0.001213505210090714, -0.024059435716523445, -0.018336329180754556, -0.20286477577725462], [-0.15916838004537698, -0.010775947472032747, -0.0037658435038256163, -0.0015250496408534303, -0.0027476428189598685, -0.03364019853445541], [0.7185009711928828, -0.013306733705729441, 0.0034071302074530417, 0.004410562117289368, -0.003487070438350149, 0.0704751406264523], [0.7141001447552889, -0.017701504149734062, 0.007153881932596549, 0.0072636165603322655, -0.004734331081988211, 0.049881992888481025], [-0.16446533294691562, -0.008136002982780185, -0.002831066100156085, -0.0021089834951651884, -0.0008881232429167027, -0.03319355324756992], [-0.21404765311343305, -0.004639157465993538, -0.0021470579587552366, -0.004128208136982262, -0.0042379552021932585, 0.0175333652106911], [-0.15403213602504223, -0.015409944884910091, 0.00537398858182839, -0.002148627810615328, -0.0029983937279592933, -0.03803294814880485], [-0.21796992692834008, -0.009094257114237066, -0.003828492530717682, -0.00409215273144421, -0.000606391083044232, 0.025174553721116707], [-0.15068903228556094, -0.012432548415357566, 0.0025581415647881406, -0.0021742114348517806, -0.001567051518649403, -0.03569529791036808], [-0.17677648915792749, 0.004826356157881577, -0.0021648274450935313, -0.005453860306980435, -0.0012237964689701592, -0.030669967812923377], [-0.22443987234164012, 0.004736897402414565, -0.00283167202845532, -0.0010983937464584378, -0.0017155964264287356, 0.016181970473901637], [-0.17816655554264121, 0.0020851044448056805, -0.006349087423026422, 0.003964330794858256, -0.0005868427494957895, -0.025903344873337505], [-0.16659375854457956, -0.006051018993097636, -0.005638178761632103, -0.003924159604538233, -0.002683995904078688, -0.02673195020757759], [0.5627114046078153, 0.00663954591830224, 0.0032321994263583872, 0.0022383480456471274, 0.006967229101847532, 0.1943907600795163], [-0.17227062028899795, 0.0014228539841844238, -0.003090705164734417, -0.0051769894032454656, -0.002333032836721555, -0.03021817295715127], [-0.21620974748388144, -0.006903056339481145, -0.003704825445353448, -0.0017214772258592771, -0.0013471969751416752, 0.0229220177554315], [-0.22194599290229233, 0.007320984437613272, -0.0034148545598422426, 0.003743590097527024, -0.0023730417296294226, 0.015335981323290313], [0.6050973447894177, -0.005558340229807258, 0.006321970626543099, 0.017598026003346433, 0.002637464959076267, 0.13192235624524648], [-0.21942195978525975, -0.008250138367359374, -0.002052970398981801, -0.0008267782086835266, -0.003468064008706002, 0.02401991076899007], [-0.16239973732443866, -0.007673631285984455, -0.0025440227213854357, -0.0019397045065335553, -0.0006626532533986, -0.033903312923763196], [-0.16369778769378981, -0.007787120627108795, -0.0020028805421482944, -0.003494423131389102, -0.0014069660413805738, -0.03323388397968721], [-0.21379610335465588, -0.010501232283940225, -0.0020206345837467388, -0.002639608022890478, -0.0023488786218602152, 0.021614580116393556], [-0.21977616006373157, 0.0006201616595001463, -0.0029739390600011894, 0.00024243580159762708, 0.0009170611056683157, 0.01835139293791907], [-0.22662245252743532, 0.002226757547189941, -0.0017298295476785733, -0.0032208368020113795, -0.00122426279627564, 0.018903957459544093], [0.144013029067148, -0.03566007765491343, -0.004254421692478481, -0.00478417272533932, -0.0002924900784397686, -0.22865088131243338], [-0.21108927369373057, -0.00933538556039825, -0.0020735253598464726, -0.0021563629543719567, -0.003959140483853058, 0.01694702138553363], [-0.21928118724907655, -0.006005612695610592, -0.0035717359728763085, -0.003120588625473937, -0.0012389338309651565, 0.022139627001453957], [-0.21951818278190846, -0.0005782416987135389, -0.002769336612447467, -0.001119942910229783, -0.0032640719215205165, 0.017583109258153127], [-0.21485895524113005, 0.0012192232549702571, -0.006543555699028068, -0.003987377837729018, 0.006880314587295321, 0.018123684268954946], [-0.22406948816778272, 0.0080047874769583, -0.0018735017606706672, 0.0015112362932185087, -0.0039025439662442314, 0.01782951012452042], [-0.15705428236983413, -0.009370202285584628, -0.005641487213035706, 0.005489567667737017, -0.0013818186906133928, -0.029458443775335373], [0.5672512564700605, -0.017281823107445117, -0.00028070976943371756, 0.001071448003587603, 0.010902800922586954, 0.17455082838562075], [-0.21835205858879717, 0.006599539387360294, -0.006756329487121881, -0.00121006787406928, -0.0026996321135338034, 0.016085215342828675], [-0.17608435090843455, -0.006526731256673699, -0.0019215337549813804, 0.0053783204401301864, -0.00023858349686764935, -0.028063516372009777], [-0.20918536155721104, -0.011933971049074633, 0.021077684139977182, -0.0023322980122641936, 0.0010573357917106872, 0.01893565830590973], [-0.22750354932353936, 0.004876068645777476, -0.0014732317097359913, -0.0022143669225927892, -0.0002951036392214369, 0.014943516282645345], [-0.1944765946242923, 0.0072389693342166175, -0.0005820429634712443, -0.0065455796846735905, -0.0029940381184435194, -0.008265713943335988], [0.17813732372276922, -0.017640797167983942, -0.015972255068133715, -0.015075131966128693, -0.0007258050006707293, -0.2648463965353564], [-0.20059563129636762, 0.0056775385313099095, -0.0010152676122116816, -0.0025405757788515963, -0.0004659376479534848, 0.0014772887700609666], [-0.20904587272055714, -0.006546712766923365, -0.000845242393204307, -0.002777427117683158, -0.0034571775619229683, 0.030172432560291226], [-0.17833510704486746, -0.00373970840782775, -0.0075584286143528185, -0.0015407789711836496, -0.0013947867015597873, -0.004639523593541811], [0.7702992935742545, -0.0033641281421582818, 0.014300185299587514, 0.0027911073432341153, 0.00905241935276325, -0.04371990306870677], [-0.16162273451953835, -0.009176317110502378, -0.0033375036675105394, -0.0015157661029236704, -0.0019746322122771057, -0.03399610840275173], [-0.21421908451563776, -0.011702164991442552, -0.007760547673088188, -8.430773556210201e-05, 0.002547502961534651, 0.022885268620863087], [-0.17461190857870143, -0.007343963228574729, -0.004299580754518341, -0.0023172007426054153, -0.006097140243025687, -0.016953268468077945], [-0.1918548221658605, -0.012054161758986734, 0.010681508434606772, -0.0043351362539406964, -0.0025315370049260673, -0.005155851250892806], [-0.18510787365331588, 0.010225685210706848, -0.0017794300947715158, -0.004704808013221525, 0.00148198702731603, -0.02923862249221786], [0.7205682097771031, -0.017283334943962142, -0.0020289085075148846, 0.00786820976146332, 0.003410289055044736, 0.056476039059544375], [-0.22235208209436916, 0.0067823109022083355, -0.0030178229217465175, -0.0035608613732622135, -0.0009545105852330789, 0.013436299405736514], [-0.1904497981571376, 0.001183471996790937, -0.004674054184641034, -0.0007351815936070377, -0.0032911242428526563, -0.013048267725948361], [-0.18726568360790935, 0.013069407969509379, -0.004821969540255701, -0.0063954413539780534, -0.0017746401481405752, -0.013728339985891798], [-0.2138205045253673, -0.005628504340442511, -0.0023528415546228056, -0.0033906074280449124, -0.0011759156152734384, 0.018451706797084466], [-0.21715539033208844, -0.003692127281346465, -0.00563270656056858, -0.0003143852795625599, 0.0023694249460889214, 0.014591851174143947], [-0.22103379152356892, -0.004127867813818817, -0.0035864818143632924, -0.002595897654040821, 0.0023426113889544447, 0.02466809408350437], [-0.2178529632007607, 0.003815796005047294, -0.0006009428248138587, -0.0024148646191280306, -0.002768514974395433, 0.015154822947384418], [-0.17281936702997278, -0.00723330401556632, -0.0011377907168232952, 0.0009024792987526645, -0.00025762103810519904, -0.020996063164951423], [0.6552014574673279, 0.008136729265158444, 0.05717305435269861, -0.0035263741470723405, 0.007730691708025248, 0.02220110802052734], [0.6514842913722959, 0.04512883878779734, 0.010021136807439076, -0.0024696516256282213, 0.01769660475552522, -0.03473026771647737], [-0.15939954379799454, -0.008694521796033099, -0.0034673523825796623, -0.00010206249254798024, 0.001859500111779236, -0.03681908165812789], [0.7662478066119384, 0.0009815387299289275, 0.006209607256418395, 0.0060724233395409825, 0.010574712875859772, -0.04110989833749559], [-0.21735499562682753, -0.006774616502775128, -0.00654084788641132, -0.0011014058723889531, 9.934514973858963e-06, 0.021941418552916408], [0.6252405527016024, -0.02047568517116715, 0.007691197077471435, -0.0004412231203768219, 0.003319146807248179, 0.14397215082753745], [-0.15772504403576132, -0.009631214819401062, -0.004176255089079352, -0.0033303134422323676, 0.0005084370721048906, -0.03726867170113443], [0.5037278345837346, 0.0344089423086218, 0.02023366629028826, 0.027631686677250966, 0.01650205784721415, -0.031271787171138876], [-0.1704714326196252, 0.00011063113167462696, -0.005238046106996238, -0.0019985470250889355, -0.0011551209437176014, -0.03287054645175038], [-0.1678629152467004, -0.0041881269093539605, -0.0021444761051333623, -0.0005712537961708836, -0.0010500455092053323, -0.032056244448939904], [0.668065794253892, 0.024177878297220453, 0.056938060221173654, 0.01160912556394876, 0.0019114695148827188, -0.007202327851118724], [0.7399644259586374, -0.008848511719005739, 0.003637777817010256, 0.013514048048534634, 0.004592665548227481, -0.04127707232007154], [-0.1869502868636966, -0.0048026755057950085, -0.004024977121323773, -0.0061681581269892915, -0.0034003552195353477, -0.00548688049599341], [-0.22161653730816705, 0.0003440773787674743, -0.0011342931597216022, -0.0004523906401305592, -0.004463632013849111, 0.017930883851208664], [-0.20668290371927064, -0.009981078602060562, -0.005918055693481745, -0.0040098044712675255, -0.0027652022985908157, 0.017690378118005053], [-0.1700025648476053, 0.0008972605269301409, -0.004727492003382031, -0.002489445692400027, -0.0020439792439226247, -0.031590174088457275], [-0.19659648599060914, -0.0005613147303843508, 0.0008546138755017408, -0.0030051194153147454, -0.0028162682995133215, -0.0029587587730135626], [0.5570940888462522, 0.01021104230200462, 0.004046306939396892, 0.006411717384434503, 0.001876018239666018, 0.1995274929549122], [-0.2089207402783619, -0.005221504136259133, -0.004531791255884347, 0.005161244502251383, -0.005557649758065687, 0.021570440926319545], [-0.1944770690462499, -0.006445725704905087, -0.0004553910599297051, 0.0010616442901558493, -0.0022715303300500835, -0.003078594815687559], [0.18035317768918627, 0.0010688547541803607, -0.0018546117067219647, -0.007820501308021347, -0.010846174603112274, -0.2600552490514443], [-0.22542801118114553, 0.0015866446901741057, -0.0050384295450850115, -0.0002934998070451621, 0.0008482890088542159, 0.01999167350091426], [-0.22004810246321654, -0.0022553320301621407, -0.0019047291310254682, 0.0021881854310745606, -0.0011786681327755218, 0.023031979659438387], [0.6287383267192075, 0.07506694582480797, -0.0005632929330302495, 0.003955877284787729, 0.011034637203331627, -0.08173055611460961], [-0.18528828869864866, 0.01181296285687068, -0.001359029177836883, -0.005353141795027976, -0.002194746531574304, -0.027617756653782802], [-0.2042698490850211, 0.0012501537025111755, 0.016618382735373624, -0.004130450829741707, -0.004622529676710097, 0.0017376264869213552], [-0.21765222802915474, 0.0035218104757843856, -0.004398597181182079, -0.006470221111786734, -0.0009772462522596218, 0.016809815431932443], [-0.1737931377523265, 0.0036913239306880534, -0.003995880273752862, -0.0032758029658232057, 0.0018738949567250122, -0.034490806849789654], [0.18082356104811997, -0.0029630883010262877, -0.027556452510571906, 0.010682529893985196, 0.0003649329330195745, -0.25920300312891703], [-0.18589665946782588, -0.0112565637070773, 0.028309599564121414, -0.005610699094943008, -0.004607733520680789, -0.010354610440260887], [-0.22150241264054507, 0.0037837980457866164, 2.957058038454282e-05, -0.0034844873704712133, -0.001884362755049232, 0.02166900525100567], [-0.22616376205403896, 0.006561893140562834, -0.0012323393595270432, -0.0009920511894237012, -0.001514408769889339, 0.01832128838735457], [-0.17637543155171814, 0.003975989783742215, -0.0024867584666527494, -0.0031377852700928623, -0.0014763049449779942, -0.03191868993315095], [-0.22480993670053062, 0.0016713093053691422, -0.0050006013458364605, 0.00012163378241703946, -0.002314223655361772, 0.018998485280609306], [0.2580608600669841, -0.11339308354955989, -0.017653374433257443, -0.01898278392148133, -0.003279997790430235, -0.25282523183830924], [0.6679321790556739, 0.031198747696101623, -0.009156101841768806, 0.015391414329275245, 0.004074084505064407, 0.05957158101755604], [-0.226479069617598, 0.0018857588712336247, -0.002426195379859363, -0.0039107074617874635, -0.0010806073714196335, 0.02284415429276418], [-0.16986247943030505, -0.01033767029017024, 0.001731655018693225, 0.0005278357888243954, -0.0022959905631566614, -0.02843001719055228], [-0.22369566813991396, -0.004205740167469449, -0.0020133250094192794, -0.004013638328491652, 0.0024928192368589694, 0.02202378770255325], [-0.22115392461408223, 0.0003656997214918279, -0.004525307688949207, -0.0010025015058291154, -0.004657721311560098, 0.01930708873226232], [-0.22280988525522177, 0.006944720796805682, -0.006953128818986206, -0.003831731226997476, -0.0035829971268238722, 0.01856635496455737], [0.6137323582880174, 0.013060135715027746, 0.004044068941721059, -0.0017391458667849276, 0.0007782975178287992, 0.14702904730895133], [0.7352158466952133, -0.017396853357131623, 0.004986990864091725, -0.0003658816559404146, 0.0070915037075117955, 0.05052791755577694], [-0.2157825554593998, -0.0068155537466392906, -0.005440630592309793, -0.00149228125307712, -0.003055726529373192, 0.02092008091413241], [0.7633642876463088, 0.007628420219592791, 0.010839222599230263, 8.970287797360632e-05, 0.015373712655406713, -0.04096201266517972], [0.5690281929212407, 0.08794995748026424, 0.004480622292013847, 0.022871251649844243, 0.014173283356026287, -0.031070226891308866], [-0.22119228970315505, 0.002239787955616133, -0.0024331047076692914, -0.006462984900562529, -0.0010570539727975613, 0.01723897866190178], [-0.2224439261726418, 0.0016250316878229207, -0.004422628266435292, -0.0010073486400759403, -0.0023798647731555237, 0.017295402831152384], [0.660545292185607, 0.09105642886923462, -0.0022979799953024844, 0.00860221922206237, 0.002237537118018661, -0.05607748008360448], [-0.17338364862060207, -0.0075287622996326456, -0.005492395695232305, -0.006443664623994573, -1.3783845104635532e-05, -0.01880441158210032], [-0.22908861476585063, 0.0037193430085698575, -0.0021622529250876817, 5.670466452833514e-05, -0.0014569102160073763, 0.01807901705555275], [-0.18980376145033384, 0.0007573294796382005, -0.005078510660045563, -0.0011734578859126886, -0.0014843345644583081, -0.01105059825222117], [-0.19950942031238747, 0.007140540679547708, -0.006282414238734926, 0.0031679330740077925, -0.0006462779557087656, -0.013870361246724364], [-0.2210527754256162, 0.0007882651697027832, -0.005253164422157988, -0.0025592403105663044, -0.001115579581847071, 0.02152582790381854], [-0.22232047919167938, 0.008381028623021857, -0.0006743959315886833, -0.0032038487697138743, -0.003968464992792696, 0.0180361602627524], [-0.21638174636565932, 0.015104648865179296, -0.008627027455197778, -0.007293033695324635, -0.0016108317679812598, 0.021099657085650717], [0.4245704357647029, 0.01257116405652082, 0.014114398768252838, 0.014675156140353115, 0.013657560088739031, -0.03844188942174334], [-0.155822936660567, -0.01584266689537075, -0.004429473925044629, -0.003951321547487672, 0.00041322715893664987, -0.032033494797133015], [-0.2216868035555301, -0.003518810819598145, -0.0023401490741421327, 0.013999856806535508, -0.001528183697729495, 0.01794173739928762], [0.7727903256840681, -0.004244280591765032, 0.009667245761894457, 0.0038919869941408643, 0.012146084432512099, -0.04329898132846929], [-0.16863515210941993, 0.0028130781234368, -0.00667881589405372, -2.7367823392206987e-05, -0.0014036108524105243, -0.03415377849367749], [-0.21239767762830772, -0.006791817018123502, -0.00626288767725411, -0.0013274846258039244, 0.00042056628645607266, 0.016359300663033354], [0.277849851380491, -0.06283047551689687, -0.013730524494105093, -0.007718841679883063, -0.005939509074867834, -0.13315634040801974], [-0.17589264117482112, -0.0036291765560272277, -0.002222157656502989, -0.0033493894402701198, -0.0015282425983719338, -0.02500145458951039], [-0.22048242691786232, 0.007422565815941368, -0.0017465530743763908, -0.004446518963831961, -0.001696335038594267, 0.014615934845390493], [0.5446665768457409, 0.009650352546189095, 0.004724142842336481, 0.0027286583801314633, 0.004890709489070116, 0.20442289322986473], [0.7163994130558684, -0.018853199895136064, -0.004744447559320409, 0.004648870613465224, 0.005573079131831085, 0.05795247512947943], [-0.22016074155657028, -0.0008151155852441042, -0.005150546733011106, -0.0011130936076215795, -0.004368428805488122, 0.019941259621268748], [0.48463784078011063, 0.11706787764393942, 0.0034502695365215682, 0.06960354729278867, 0.009560707253715198, -0.11101414582597893], [-0.21691471319966962, 0.0029431340766544597, -0.0029233117340408416, -0.007731452172737679, -0.0015066747510087037, 0.01446635111413567], [-0.2232034924865439, 0.0016525733359131012, 0.002979453219001742, -0.00023798321040887602, -0.0005185832609106374, 0.019328032402948652], [-0.21962719462638483, -0.005592834270912059, -0.005607912973005556, 0.0031697170885181774, -0.0013497119382773917, 0.019841270053395282], [-0.1719234021808574, 0.0016400579637247455, -0.007650453478700689, -0.0016041829262502154, -0.0016680229273447454, -0.029808950357967295], [0.537865651508843, -0.0014119578681927852, 0.00011801226277770915, 0.005907108057443101, 0.002443539269219278, 0.1857333244255855], [0.765917360595375, 0.033740921476754275, 0.004527184570522615, 0.0005486112180864245, 0.008030286431684344, -0.03840072792878456], [-0.1958329318916149, 0.012137454112027537, -0.0005830818007321485, 0.0001356350860534706, -0.0015419961795279154, -0.02298174599287294], [-0.19121392126703934, 0.0036604917699977167, -0.004089483934925057, -0.0009867764851783489, -0.008441406689531389, -0.009762236726657141], [-0.21833730532569287, -0.004001362884377059, 0.006234507856044267, -0.006324795938587924, -0.003122875356628807, 0.021718498315909253], [-0.22167113499738106, -2.989468959843246e-05, -0.0031128790557660513, -0.003638982409268679, -0.0016363746856825048, 0.020922599171030173], [0.19469241531626114, 0.04772126612455657, -0.016453389083943636, -0.00519283951061423, -0.013971896925327475, -0.24717001479539064], [-0.2179846420503684, 0.000663704434777855, -0.005601244888152662, -0.0020012229108721897, -0.00707629116776898, 0.020333029915718174], [-0.20972688151516644, -0.00711994759942348, 0.001015517364969041, -0.00023382356039685365, -0.002226199450614739, 0.019458001427299258], [0.752199847450942, 0.025264856004717992, 0.009339871284706288, 0.005587720354662211, 0.0044497378578122245, -0.031452755563564055], [0.4098288589876607, -0.1011832820847257, -0.05786932728965963, -0.024150178984049808, -0.0378253895884367, -0.1953562827388327], [-0.20163352866235754, 0.024492566681286444, -0.005454983459235288, 0.003324364353226396, -0.00033627800620761626, -0.014320712335283765], [0.7120963238389543, 0.0013536170494337812, 0.014744298052990308, 0.001906157187888408, 0.002351042511499338, 0.04433427564494627], [-0.23145903358876088, 0.003399011298162711, -0.0023378548298904804, -0.0015939600041059915, -0.00168234231369412, 0.02200751277162191], [0.720627787516383, -0.00016302993276753278, 0.006753326806056237, 0.003989279021202538, 0.0033954080217081994, 0.040813895234083124], [0.6534782902967239, 0.0797717162962327, 0.01211383123082256, 0.027180986351896575, 0.006307510427996946, -0.040893460144799006], [-0.22002783186605998, -0.006201542475363406, -0.004120384724138952, -0.0005910068575211495, -0.0053175691417665805, 0.024591668398183417], [-0.22438492992989761, 0.00017062667675229212, -0.0009961964329343572, -0.003739053643916675, 0.00042099714125584664, 0.016861889522074066], [-0.21787731995853726, 0.0009397865749867194, -0.00012186373544869585, -0.0017755401345486935, -0.004297565271458638, 0.01771583585833965], [-0.22476351304160744, -0.007688912899556475, 0.01119245883710144, 0.013475685594137103, -0.0035333293388088763, 0.01965094418206729], [0.7292648304240125, 0.011750648553696702, 0.0035690415330948467, 0.002565082360982338, 0.00493824698603392, 0.03524548347551287], [-0.16167556699588628, -0.00728561851932652, -0.0027667061800354612, -0.004067866734732479, -0.0016363956040935026, -0.034234512632592494], [-0.17324541378559108, 0.0007434697471846596, -0.004990716260846162, -0.0012632468061957822, -0.0015337392633665604, -0.02933341564668874], [-0.22187500800199017, 0.0011308415732285504, -0.004183943426662111, -0.0002372263964457222, -0.0024279661404674625, 0.01875996905900355], [0.07300332487583451, -0.012983814407417296, -0.005647989883601983, -0.004753205398120686, -0.009169785785833265, -0.2094882580830318], [-0.19940971495358856, -0.00422119470134109, -0.009967135533747966, -0.0023810183679751486, -0.008652133384335736, 0.027964530274322358], [-0.22800969089209924, 0.004473436969199125, -9.681396399576297e-05, 0.00786830491887238, -0.005545538552678143, 0.017726968187368054], [-0.2205207949289104, 0.002337639093025215, -0.0007334133289098447, -0.003951409156346229, 0.0034648677311864416, 0.016069777256621745], [-0.21243757183822423, -0.007416736596008728, -0.0004389120174653398, -0.005562612251542698, -0.0022877721618139698, 0.02434458525721233], [0.29160150982201655, -0.061720378948645224, -0.00924112988687024, -0.001140004363464971, 0.0007107447796077241, -0.25245101953536375], [-0.2116512897099087, 0.0041350879171771085, -0.0034549861464833417, -0.007611800148591718, -0.0038797939719466463, 0.015796115393086996], [-0.1589848359618554, -0.008136278270916055, -0.0038905747242564863, -0.0017281751398298651, -0.0017465620601742134, -0.033846907176301524], [-0.17771824260785843, -0.010692976472570974, -0.002611152447896373, 0.015407153521715481, -0.00322822024130003, -0.02329941889494649], [-0.21300647979951812, -0.007515874594000295, -0.007045997170023369, -0.003727750981166616, -0.003397001636197736, 0.025281339475024197], [-0.19155103725356903, 0.00810035136352111, -0.00033403248372208973, -0.0026522501852159204, -0.003075974795593962, -0.02211011866092401], [-0.1817086242674317, 0.008084222846874161, -0.004078191553280332, -0.00046680941911569175, -0.00236550161355712, -0.027550743043006604], [-0.1839484568489119, 0.006951378287045927, -0.003030021461004356, -3.5963279469649064e-05, -0.0037945956682638375, -0.024432069711566574], [-0.16912310547482434, 0.0027379709010250716, -0.004252501437050888, -0.004155005581604603, -0.0033654665780782156, -0.03163781019681372], [-0.2413136985635857, -0.004202355372784921, 0.07849711163322852, -0.005158223807325935, -0.008559338241747148, 0.0034587265744371594], [-0.19216157648285245, -0.005693475140703819, -0.001975731225680906, -0.004918064210752954, -0.0005387430234631334, -0.005545743249879842], [-0.22403987799042907, -0.008067476414146546, -0.001926768126457143, -0.0006624352066648636, 0.00011784425558923815, 0.02475820066159568], [0.5624149631341855, -0.03893322354464766, 0.0050467859226828, -0.005799720137284168, 0.000466066429025896, 0.18832893771984724], [-0.1732192900505927, -0.007210802127874323, -0.0037572147359922004, 0.004424900586792567, -0.0004947940878269391, -0.031409466251172774], [-0.19974941893159634, 0.0018702008829503958, -0.000538262922678008, -0.00500159979070842, -0.0023884652170515297, -0.00285912068758243], [-0.17489960976876634, 0.00800519291118955, 0.00806468204435951, -0.003014105406714263, 0.0021058369905900057, -0.031678663437325165], [-0.18881688926960608, 0.007409409531179295, -0.0016163688863409817, -0.004470997059948853, 0.0024162362073004815, -0.022590370905434424], [0.37021087950025394, -0.007461269074573059, -0.016829525671698837, -0.012251888906075366, -0.003649805138033412, -0.19086169082061527], [-0.18375562991520403, -0.008805936151458946, -0.002632741939749359, 0.0016732648897554525, 4.235515985083555e-05, -0.018144374058697565], [-0.18849932522772708, 0.0014061181638991339, -0.0024213119320706067, -0.004587643260004678, -0.001694448810816043, -0.015870055599947037], [-0.21990402557160535, 0.003395210938615342, -0.0010896172738512821, -0.003217255408274302, -0.002453726365790815, 0.014936080347573173], [-0.2131480838830413, 0.002593944411434576, -0.002971450452211892, 0.007455689820083583, -0.0017164650734119576, 0.02361969851048016], [-0.1872265404946878, 0.0011809680926439185, -0.004646224235597361, -0.004436352299224599, -0.0003979964058495478, -0.01614052132395116], [-0.21967520330690296, -0.004380331051220187, -0.002116378801424934, 0.0003699952356454038, -0.002305508971437536, 0.026440760228673466], [-0.20227709375486297, -0.014981992460504467, -0.003437956062074603, -0.0008270692918524161, 0.0012561833763839656, 0.019101261526244157], [-0.17894675562240273, 0.0009797852466562626, -0.0005015877581971116, -0.0011070329246551578, -0.0018817869893006135, -0.029681812999862556], [0.7432379643440011, 0.008848213158256053, 0.0028386793078121627, 0.0029979313148703703, 0.002983668359185913, 0.02742687684920596], [0.7398858444184149, 0.02918196038274518, -0.0099595546876515, -0.0019361378252769786, 0.0004894013241055132, -0.034690734391558134], [0.5405162380252228, -0.026117880450203353, 0.005209202977111136, 0.005577261346807825, 0.0006500123239356631, 0.1958734991104583], [-0.22631503187817192, 0.007951333934087052, 0.00041093900687295444, -0.0032328377839146304, -0.002284647200097293, 0.022220243921224233], [-0.16750008054076856, -0.012970772568093173, -0.004305158338342404, -0.004043096149905595, -0.00334442344031291, -0.019503135629243613], [-0.22454244488522068, -0.0008880892245124682, -0.0011381625628095133, 0.003580734797824309, -0.002324821162388717, 0.020504224478548605], [-0.22423878138740622, 0.0017704467869725713, -0.004440335579174085, 0.005714016558571261, 0.0003090712945002583, 0.019199535814907995], [-0.22709651690312177, 0.007186137999534934, -0.0019801219070061877, 0.00994480169850497, -0.0035127492546560756, 0.010875115033410323], [-0.22281189019180367, -0.007530252754133039, -0.0010916714130815002, 0.004021741531018004, -0.0009535524479705481, 0.020865625275970863], [0.27930228282117436, -0.051891319657560935, -0.007593234355609216, -0.035367563322639586, -0.014786830350683748, -0.23238555735690317], [-0.22381555425692606, 0.00248140536344638, -0.005424903039612892, -0.0002483408943601482, 0.0006259215808783935, 0.01804813791324116], [-0.1538053477737071, -0.008590511003680513, -0.006184117611156407, -0.0025012919745745496, -0.005204331544717116, -0.03533746210766807], [-0.168959481014299, 0.004312599493101999, -0.00575152976789109, -0.00557474847922893, -0.0023812192359151997, -0.029978954329101], [-0.19576150599321993, 0.00303689047944391, -0.006684305219704218, -0.0034369123996193206, 0.00040623576775616226, -0.0028104026346564416], [-0.19420969435739288, -9.10769409195676e-05, -0.0014646684178851435, 0.0036786491417329535, -0.0008577129492025376, -0.013678558491836447], [-0.2314878595008528, 0.004468664376370103, -0.0020631147403801934, -0.00027493122935325526, 0.0007834848138293646, 0.022721043102092175], [-0.14884719162290258, -0.015223069478045205, -0.007229953393197961, -0.002253983147271693, -0.0026629232175800687, -0.035449545807669215], [0.4030172516771397, 0.03146648579851479, 0.026167093634990327, 0.0030438170001648975, 0.013661019711638269, -0.03378369812547854], [-0.2232327966101015, 0.005600405286532522, -0.003802262244198301, -0.001514349121842267, -0.003247666287458538, 0.01619666897706762], [0.4663641377659563, 0.040677528261344784, -0.002284546097163503, -0.005660841946818317, 0.016994897494013252, -0.0394834914946496], [0.7328869435179607, 0.003620813026574847, 0.004562636628954958, 0.008840920683234742, 0.004015519439408429, 0.030406500037199285], [-0.17777419338896197, 0.006763010461872978, -0.006944040235020957, -0.0035729995719909783, -0.001004208469535928, -0.02846563081186671], [-0.17990486279074752, 0.005083669577572504, -0.0031490594562393736, -0.00690007451967544, -0.0019081254777863093, -0.021554880666457006], [-0.1770951246249207, 0.004360455411056026, -0.001957087125722053, -0.0067347103167615845, 0.001648607665668619, -0.02835139271000032], [0.2731036321274252, -0.11971229894687521, -0.0355843693185682, -0.0021745046661877164, -0.024356779658612573, -0.23664874155268514], [-0.1761183639971125, 0.0034684327255495467, -0.0036702451426610123, -0.0033542574757423104, 0.0010313049694468679, -0.03277585146233125], [0.7223324491652633, -0.01876787275243699, 0.005506442144162845, 0.008015317170996144, 0.005269848585113257, 0.050620006163092195], [-0.21830338067204907, 0.0010458331481632273, -0.001026631319943204, 0.0024509309052005005, -0.005945079816261827, 0.016540232516794567], [-0.18973480448816107, 0.0010442097238662024, -0.0016408958443455096, 0.004050249500566087, 0.0020879014268345856, -0.007699517461617184], [-0.220917392867628, 0.0031438588594479254, -0.00010164945082496182, -0.0004258301375828192, -0.003503508751592433, 0.02038785568151377], [-0.15886610211209834, -0.004530059266510005, -0.005243555283598506, 0.0004080083892524777, 0.0017068183452635189, -0.03764177673897557], [-0.22798577517940324, 0.00570675900165961, -0.0018045606384501707, -0.00041555355136484797, -0.003229901268631859, 0.01606236496952341], [-0.16709444468907952, -0.006169039719803236, -0.0006778580719638175, 0.008407630583096445, -0.002364060415908, -0.030435561019675095], [-0.2198843389788679, -0.0037742378020456544, -0.00516248387432398, -0.0005967630072187862, 4.46591637481105e-05, 0.021036522645937354], [-0.22105983515761918, 7.46982890792323e-05, -0.005089348137450319, -0.004153252443403088, -0.0006565147321306841, 0.019217585514857482], [-0.2239878725994115, -0.00038682060768362566, -0.0038609179357653556, -0.002893070262365132, -0.0030595770787649364, 0.022521591817323963], [-0.15679567622543464, -0.016428457561769035, -0.0017234302072595485, -0.0016259885876529494, -0.00040298009886217685, -0.034646529334525576], [-0.21839209659075806, -0.002620781164770527, -0.0010807876683910486, -0.003986538300161336, -0.0009705559345452478, 0.018230246838113733], [-0.22594153362951225, -0.0015867085887731058, 0.028297614868722922, -0.0005357323103872424, -0.0060617562367962395, 0.017578115896745676], [0.7360525851845136, 0.018415878505328668, 0.006202185268519403, 0.008422018625983463, 0.004646136421483285, -0.01823627875330427], [-0.15499703797584033, -0.00981429733343813, -0.007568344566634834, -0.003531301229581794, -0.0015026646062390166, -0.03420941630376965], [-0.1872666471275912, -0.005376045921253309, -0.005090750188688324, 0.011130529604887652, 0.0005898322356705795, -0.013177394793501478], [-0.21621049735626674, -0.005597292288986473, -0.004707426429786287, -0.0018952609376486542, -0.002829862622058823, 0.019573672968080127], [-0.22493173251457335, 0.00598423245918839, -0.0012374080217388882, 0.005586230961852122, -0.0016455074161531376, 0.018744184531424568], [-0.18458691699647764, 0.016648236421882442, -0.004959778893013966, 0.0006234681367045364, -0.001154671821842357, -0.027237003513919764], [-0.15570891928926844, -0.013641291051285516, -0.002792253556212358, 0.00022676508434442844, -0.0035387967993241974, -0.03366856640375774], [-0.210767182453531, -0.0033215669499233476, -0.0023761169242844235, -0.003725186632769024, -0.00037895491896914907, 0.017235674546143952], [-0.2198665008224603, -0.006319503154698025, -0.002110716029723741, -0.0019862909066961903, -0.00019681651399634881, 0.02608088323451317], [0.7279434970756379, 0.01141835826296141, -0.002622576379736845, 0.009257328461307671, 0.0037181361234950336, 0.03167414534522215], [-0.19432799950802532, 0.01224475673537171, -0.0010388619977899824, -0.00026232087337539454, -0.0015640764877959572, -0.025860606395516846], [-0.17882535386685194, -0.008161197844239219, -0.0009139642028501583, -0.003435279068667527, -0.00014625766283946786, -0.02014100937005524], [-0.22216885556946822, -0.004660253063024362, -0.004117840136991854, 0.003498575816251632, -0.000780587251059984, 0.02822896020429264], [0.7273674216820728, 0.007251181731508733, -0.0033798068048984595, 0.005432674653720581, 0.002836689131730243, 0.025889126427570425], [0.712377400674014, 0.007916877206694406, 0.0038436850174942984, 0.005627432993708835, 8.823605101172815e-05, 0.027757479168187202], [-0.22970168489925427, 0.00014893099621222353, -0.0037603455131964064, -0.0005195855285698307, 8.198256476342238e-05, 0.02441406052727398], [0.7343050077610614, 0.012385945157341845, 0.004905375559511118, -0.002884404197162761, 0.0035287039907003414, 0.03092603839521329], [-0.1846741706758845, 0.008969318880418164, -0.0034810239898397507, -0.0005450486820111948, 0.0017981122505405646, -0.026636678370155733], [-0.1822387459880769, 0.013236957492664328, -0.0055820322082660795, -0.0008496146373094249, -0.003724269835023174, -0.02834229482398864], [0.6629672634464778, 0.020275421150387093, 0.006171791168113502, 0.010874505731726332, 0.007066932347320605, 0.07986630837819834], [-0.21468179076801028, 0.0045835405856977105, -0.004820597751305733, -0.007998820317363077, 0.006370670021499872, 0.01838033156281537], [-0.22634449584759816, 0.0070734807045559215, -0.002447429208440221, -0.00017159640723606937, -0.0032061167492436235, 0.013429490841295296], [-0.14838094826252646, -0.0074890799175984755, -0.006666429460833873, -0.005232416826010581, -0.002012498567306023, -0.036008355647894716], [0.6115265298530731, 0.01207664349551015, 0.004891800982541337, 0.0014660221234533849, 0.006626961964009665, 0.14716295733232843], [-0.1774494989531385, -0.009744539766816794, 0.015302355176905702, 0.0037918283810711127, -0.005259502932419159, -0.027807308572269058], [0.679553324173807, 0.027718912970818386, 0.006274755217024925, 0.01276013419622937, -0.0011433994575424248, 0.06025293956632795], [-0.22667337297451443, 0.0022745718996822362, -0.0012325272880708631, -0.002761812639166175, -2.1093603884812706e-05, 0.019308007499727057], [-0.21275046995293054, -0.007292576367465984, -0.006095996026527368, -0.0008577936197140801, -0.003825054835512261, 0.026446890802150427], [0.4803763888690396, 0.006428278188090458, 0.0033602682072881905, -0.006874707597401816, 0.04302618320008553, 0.16828081135511924], [-0.17705908701521045, -0.008307638596390677, -0.002614000153137863, -0.00041393234728976557, -0.0028598080369479605, -0.017912200517689648], [-0.16761177308905095, -0.005693808750325571, -0.0047678274162454215, -0.0011080613147058627, -0.0006249891911312414, -0.02764993558737796], [-0.1588882920523042, -0.008544867864618444, -0.0023943840124842587, -0.004449298670523232, -0.0035645094293626353, -0.0337817099862109], [-0.16375725779894232, -0.0058051109843132035, -0.0005664692171412092, 0.0022429619624014423, -0.003689666215324971, -0.03671418642885005], [-0.1542604840561926, -0.009902965398172522, -0.003952769028617723, 0.002538841407540768, -0.007704975877612265, -0.038340709062449546], [0.7287413247669784, 0.000586779918444261, 0.00814048659391281, 0.011905010249007588, 0.0036180380996772153, 0.03334169370531225], [-0.20830194296862675, -0.0027917633830128533, 8.67621106219617e-05, 0.0022512982624392393, -0.00609618991990782, 0.022601835898486453], [-0.22870057402337599, 0.007785825162025814, -0.001990504221617037, 0.0019321182158588942, -0.0024231411997511568, 0.016710229555230955], [-0.21940895452698617, 0.008277602752979703, -0.005209831011981874, -0.004404115796113761, -0.0028111048053213682, 0.016889736720756936], [-0.2224622056535993, 0.0028129982953425557, -0.0029987596198988472, -0.0014233645787988841, -0.002984780968174227, 0.015389445858461909], [0.6614180910493722, 0.014859836781442757, -2.4491600497928885e-05, 0.0033304906131766923, 0.006034986143290348, 0.09056057419270513], [0.5443449409780599, -0.006749371439395526, -0.001968500548347453, 0.005656688743597118, 0.010013616280116567, 0.20075911524834006], [0.719307207726207, 0.017379081688733918, -0.0042394698887727005, 0.008948209533200855, 0.003833943198848085, 0.038771027741780345], [0.3225311259513485, 0.030919397326164725, 0.01839880350736233, 0.014472934475753775, 0.031944215595931795, -0.05937957209465543], [0.6897636912704873, 0.018200134374074337, -0.0017560705725932712, 0.003462792727342872, 0.0030947490792320284, 0.06556803645478945], [-0.22584771773021403, 0.0014461812820146939, -0.00041604443736019257, -0.0001274796347528129, -0.005112994160532661, 0.01839138801417791], [-0.21935540955293195, 0.003301427194445696, -0.004227266877542368, -0.005805362451730463, -0.002901994391571727, 0.01732193941266434], [-0.18690377051124898, 0.01444198922624139, -0.004513959168741959, -0.006799374872165756, -0.003381524282470677, -0.01584336039161376], [0.29658052255130285, -0.06269854450743799, -0.009837344817166626, -0.03059881254917722, 0.0034741064559639084, -0.2340984985620562], [0.646235011059896, -0.0407619667140129, 0.0029216481943584193, 0.00187732112287386, 0.008235775917092028, 0.12874221041979214], [-0.17267219452561647, -0.0004964438152994473, -0.004365521371161686, -0.0017849814617053213, 0.006401276818679458, -0.027586150041352817], [-0.18933054149768677, 0.014414762055922292, 0.00016520312846131642, -0.0030066925887714175, 0.00020863970609236594, -0.02661803747068423], [0.7237816761574015, -0.0024038055372445225, -0.005268386521389402, 0.008737526201164686, 0.007526305071044491, 0.049577665021178476], [-0.18356334367636634, 0.012445201893481287, -0.005713459641408076, -0.003611988052217587, -0.0028846611479910545, -0.028338416042164492], [0.5412605475018236, 0.020381203177652955, 0.0052101928069761445, 0.00548835893664895, 0.004157746463809712, 0.18100195111308817], [-0.22590624597381254, 0.00022162303657911978, -0.004792728647952787, -0.0009421080423079247, -0.0006431859346145973, 0.022726003709337764], [-0.17794286070745746, -0.0065005616476074305, -0.0021333179274307316, 0.013595946645411071, -0.0015149347463606182, -0.028586639698922695], [0.6123962829458944, 0.02945786096104616, 0.005276442522945792, -0.005505265447830111, 0.008236712561021682, 0.12263796645692211], [-0.2169533221824464, -0.002654018706415037, -0.004035629899142832, -0.00584629816015478, -0.0025967315799843823, 0.020419333861476627], [0.6209365795970186, -0.007401803955727033, 0.0017762455744091132, -0.0006884246670179542, 0.005823226634443692, 0.14616131967401633], [-0.15670466741018851, -0.008100683117959084, -0.004299484644231625, -0.005624127281258432, -0.0008106427174764451, -0.03612706149555252], [-0.15647811212772142, -0.012105476603173715, -0.0011729197614130757, -0.0019978046891039915, -0.0038069056170790157, -0.029811843217012614], [-0.2073770855655266, -0.004868734803854326, 0.001853562657435605, 0.0004143638944247252, -0.00428071044587159, 0.017591937596725664], [-0.1626994232035835, -0.0066658695472300855, -0.0029539986890611512, 0.0019598960311436333, -0.003545039152006977, -0.03605196078809886], [0.5592311212221844, 0.012027410566738603, 0.005726963837335089, 0.002211760811090993, 0.005443524003426105, 0.18727257770645422], [-0.16445414234642097, -0.0065961371662313535, -0.004174081401278742, 0.0027062845771276094, -0.003139042293003569, -0.035965943385696655], [0.7362332158033336, -0.004765166210744061, 0.0048689013103992125, 8.574988537621076e-05, -0.0027155301158969794, 0.03795949599419685], [-0.16597848012531735, -0.008782093527516415, -0.000385820170895944, -0.0011844315636944523, -0.0006427472215563338, -0.03464948940652337], [-0.22086089950837917, -0.004058544550996052, -0.002661561049483188, -0.0007758879948220445, -0.0028719815716918813, 0.020170316116813712], [0.7233585819013811, 0.011205034237030363, 0.0009843168290340695, 0.00879292760715714, 0.0032008633141570137, 0.02954160944457201], [0.5468798131364281, 0.01601616380600635, 0.0033846974424988044, 0.007489067003228729, 0.0029593137631671175, 0.19343761151533723], [-0.22402996694891286, -0.0030192617694152248, -0.004646045448569357, -0.0019365174201998814, 0.03914832236450837, 0.025872358111477887], [-0.21780941815048524, -0.003013068805572346, -0.0014956554763395872, -0.005352257518324558, -0.003428535813100151, 0.019432269097155418], [-0.16838134476887237, 0.007205662883174754, -0.0009738521144421796, -0.0007001493235447818, -3.7208941962048205e-05, -0.03738480129549854], [-0.1797927243128816, -0.008579499198630736, -0.00419719952523059, -2.7720207878336322e-05, -0.0025298478232993003, -0.013996070947583492], [-0.2104205760664155, -0.006611856362203798, -0.0006355873034099817, -0.007682498491355462, 0.004668677159105236, 0.029931841064279895], [-0.19253116093996545, 0.015138198043532789, -0.00019328674824393462, -7.201068758554306e-05, 0.001780698916329047, -0.027455771917400555], [-0.16597469740347376, -0.008431307396630786, -0.0035323721445473383, -0.001841597783366718, -0.0009849802098394653, -0.030858107077645917], [-0.17161138799845377, -0.0005553290791632403, -0.005357075436345504, 0.0016214293423184084, -0.0015905138224245447, -0.03288018502143469], [-0.1549228894119849, -0.010012785350041884, -0.00011609917055030313, 0.0021030764227555165, -0.00295477570087435, -0.0031798601226370555], [-0.17314449964357803, 0.0010602728059762752, -0.0041506457745805235, 0.0008949666590360066, -0.00277174298144243, -0.031011413080915377], [-0.21888871065468146, -0.0014354003486049855, 0.010863465208054713, -0.0006914359206268206, -0.0040372408502795986, 0.024105989232805394], [-0.1731722138027947, -0.004409833024851377, -0.0010193328846572884, -0.002914365837220793, -0.0015610759062624976, -0.02854624055971721], [-0.22504376254151695, -0.0020019724242910505, -0.0023158804659957058, 0.007425726148821385, -0.0010315424292233052, 0.01421743171220553], [-0.19114059489091464, 0.014490606720659042, -0.0036301679395896016, 0.00021795328657398243, 0.000740049194024642, -0.024427846370753094], [-0.1735478583629273, -0.0004885346842098438, -0.003372084391634657, 0.017212528440310234, -0.003652066167290187, -0.025104365786629005], [-0.22587475545999336, 0.006225155177345167, -0.0009483510058667019, 0.00530131166995177, -0.002078686140332548, 0.01820865909222875], [-0.22035483276655268, -0.008824343484839256, 0.009559488670118117, 0.0002115362693217212, -0.005056361513242658, 0.015297846158528087], [0.7177990916519714, 0.01258353801018422, 0.005095598592779729, 0.007663058848472479, 0.005548395584323729, 0.03151865064559894], [0.6564944919830314, -0.02356807095604102, 0.0047932497368207375, -0.01182246872930877, 0.010044951136813824, 0.10347542924626583], [-0.2174958058920786, -0.008780824815797327, -0.002031704159580795, -0.0011000223723851294, -0.0038993837698239535, 0.021641074342999007], [-0.20642192501020007, -0.0058648617700064395, -0.003204692659038538, -0.005085841087619654, -0.0010307149727363115, 0.01577470216626774], [-0.19246275289200007, 0.01707654254810225, -2.9065926741098426e-05, -0.0023920261634340517, -0.0016658937079933613, -0.032193470524600434], [0.7403516817015858, 0.016182651882116015, -0.0016716189802155495, 0.001667198785690835, 0.006658791718383294, -0.012938705107561324], [0.7323743528886133, -0.011386385809940728, 0.0049289956955928, 0.0012076646431904094, 0.004258669976454934, 0.044311147050532365], [-0.22880913142773723, 0.0024879610478206952, -0.0005986867187132007, 0.0002874425441501269, 0.0006972379952231666, 0.018766673907277324], [-0.2240064147781403, 0.005746750017004869, -0.0026033766583880786, -0.00545798062267123, -0.0019077489044938415, 0.022312104280021856], [-0.17778953781885454, 0.005414535479048631, -0.0017948567820782765, 0.0008015906355777607, -0.008134798655923198, -0.030119994873274242], [-0.16752561766551594, 0.0012051093296985003, -0.005851983762633159, -0.003653449229896232, -0.0013385197939731039, -0.03279193422651714], [0.5848379826908838, 0.022125713134923276, 0.0002476686903103272, -0.014066303660846468, 0.009635592823710163, 0.13530267965435266], [-0.21553634616514733, -0.0074065246294562365, -0.001423932678523098, -0.0037700597638696143, -0.00096862368744655, 0.024522153591109846], [-0.2216668783360687, 0.0011513779619171376, -0.0031302131726080253, -0.0029593594238013243, -0.003291509220734609, 0.01822991552462892], [-0.22133661607800928, 0.003431438341391103, -0.0031529013399441462, -0.004119788723959924, -0.0012036748645432084, 0.01839908652471432], [-0.1918845117628362, 0.012196995142868967, -0.0037688295398663504, -0.006793203601067403, -0.00529816024185759, -0.014118956663907898], [-0.21467375538779726, -0.006531523783289112, -0.004404434258038169, 0.0025195981413893564, -0.0017831492233434949, 0.02537326451107897], [0.5819777729305333, -0.013501929846059243, 0.004079809594441979, 0.009781953971128196, 0.04992942885582392, 0.11917939306555994], [-0.19104963560038896, 0.007427212382303845, -0.0031611351519424936, -0.007537543869793525, -0.002241177358110886, -0.009771053735401165], [0.6460710857308092, 0.034287756740846984, -0.0054155391305688964, 0.009147808340341871, 0.0043545478011364665, 0.07605434051743347], [-0.2133122314741742, -0.007167191977687909, -0.002416904539156912, -0.0008660734383888717, -0.003729299867936294, 0.020825034630677638], [-0.22030463342828613, -0.0026041128757806394, -0.0034208299707843472, 0.00866246103595764, -0.00516059137945831, 0.019269148059793025], [0.7345323259393644, -0.003396543020415966, -0.005876234720168268, 0.009796151543624988, 0.0036234332736694262, 0.04040420031725642], [-0.2155114618776374, -0.0043883262911525054, -0.0007052156745310458, -0.00036629424145708894, 0.002442995428718, 0.02715877022770426], [-0.2259410323600672, -0.0008378255791021418, -0.003508682530315199, 0.002499122621927393, -0.003397658361398348, 0.020127517650396817], [0.6943751243035512, 0.019871293044996362, 0.003662175647903008, 0.00541576045622874, 0.0063213540853709295, 0.0573718363215992], [-0.22517563240744623, 0.007688520113052828, -0.004682239612040494, -0.0016144906972895444, -0.0037746116804338015, 0.015891787617490355], [-0.2261129248307531, -0.0019348168687839415, -0.00057998453657398, 0.0033222634947365783, -0.0006936523225049647, 0.01660722317198752], [0.28698323782679713, -0.03679474156086212, -0.026082030192691376, -0.017978329620848586, -0.011181678031257764, -0.2405258235005022], [-0.17999279450364541, -0.0010183351714855781, -0.005641040623672325, 0.002635366708538495, 0.0009497747585850427, -0.02522269985049048], [-0.21482992886123012, -0.00822264588396682, -0.005598480668064677, -0.0008633476681028908, -0.0016978842081210982, 0.022878953956152352], [-0.2086696441841303, -0.008536356681425779, -0.004891859316294724, -0.0017096496210888272, -0.0053662194459151175, 0.020840395915521864], [0.6767789712235412, 0.017769753289621378, 0.005573949572967783, -0.0014832720441624123, -0.003993393356675402, 0.08368732464804038], [-0.1807259629164295, 0.005812749011877979, -0.0016342518644576099, -0.0004559114960801584, -0.002902882318583703, -0.03151272079917773], [-0.16967138560752423, -0.0008054405389293485, -0.0006431954697003077, -0.00035716193589033213, 0.0005590824316747318, -0.03287162756180102], [-0.22638337057069863, 0.002134061466305042, -0.0006739658128226448, -0.0028032270574701963, -0.00023588467976030822, 0.01629571998778001], [-0.21393977518434948, -0.0036775370777506686, -0.0018795930941916802, -0.004833379003591629, -0.0027597729922966933, 0.0170900573521802], [-0.1515976787735012, -0.009264657842360833, -0.005037848004255945, -0.003105427416641843, -0.00432767520685052, -0.03583337942305625], [0.6770522056669384, 0.015821013957578792, -0.000735319944941178, 0.003054378852599531, 0.006038219039886501, 0.07139450242793824], [0.7340606392593692, 0.02000464529170462, 0.011259722633162649, -0.001017699335092831, 0.0012975012801529292, -0.04087083477032143], [0.7024128988623464, 0.02130658334111963, -0.003114589731705149, 0.007752309007542777, -0.001776706241261606, 0.04532426666671767], [0.714105413523576, -0.008657637613771887, -0.00011923950498486801, 0.010392547063292897, -0.004435396612466825, 0.044408757588797663], [-0.16504097585483057, -0.00848905524329119, -0.0019954304528340097, -0.0035174838817706162, 0.00033117296779588485, -0.0329112895505734], [0.7394599516685078, -0.0034336364326559097, 0.005609099765524792, 0.0022121891992056227, 0.0037052005334238755, 0.03531320833788786], [-0.19042693563249724, 0.00441216022226317, -0.007232797077362781, -0.005090141067481702, -0.002237944039766349, -0.00859100907182171], [-0.1496467505555571, -0.017480176145273592, -0.004016910525372396, -0.001552321082941501, -0.0015018590037800706, -0.03742504470257891], [-0.2259789501436882, 0.0032640233186008215, -0.0009818149471310352, 0.0009845056914049974, 0.0011747781471310064, 0.01568474475538804], [-0.18539109146692495, 0.0016616217141897879, -0.002810348163307671, 0.0020228179432008955, -0.0016890177129571396, -0.02480893622159648], [-0.23282916196556916, 0.006262540485014977, 9.902695473497396e-05, 0.0045908618216880925, -0.0034468624889400245, 0.015323595193070876], [-0.23421057275120932, 0.008340137831436192, 7.076106023321866e-05, 0.004385968840021263, -0.0026614695278812254, 0.02365850788073275], [-0.15871294891501966, -0.011825089888989122, -0.003887867815199026, -0.0029353000852207116, -0.0018685722006329764, -0.03239328311044237], [-0.15793109589455417, -0.012169895897792698, -0.005238307018453016, 0.007192765571495939, 0.00027302380705179584, -0.030662204853461467], [-0.22074139284585012, -0.0004281679009312547, -0.0017627400307744677, -0.003007741930788014, 0.004590292419050984, 0.018849750289292755], [-0.21314024962830333, 0.0015700564278101304, -0.008368891141172922, -0.0017243448053400868, -0.006998969388838591, 0.016995731869178323], [-0.18258331503004635, 0.0011016288615606435, -0.0007695145805126302, 0.0006564927100157805, -0.005113280289897338, -0.024306965578515492], [-0.17656995293757427, 0.0015407356400105475, -0.00159547371790503, -0.00022816517766677898, -0.002117560204219464, -0.03196469301775375], [-0.18585958243247572, 0.009360818652909646, -0.0012773168811240734, -0.002436937553322397, -0.0030538932514281108, -0.023399755201226124], [0.7548220366799425, 0.009943143604472593, -0.0003532431790775882, 0.006787637916657984, -0.009964336330090936, -0.04269357202523903], [0.7172024781027421, -0.0007996654896917355, -0.009019605854686231, 0.012876605363517217, -0.005360418060562498, -0.055643441680367174], [0.5795450469805885, -0.0038034670710048015, 0.00582257318970408, -0.004015533675451633, 0.0017913727459648097, 0.16402905544924645], [-0.22458283898751422, 0.0011321683809498335, -0.002693578961960515, 0.0034467707739019512, -0.0009351162485462022, 0.01696592837650252], [-0.19392361791161986, 0.014868120148518734, -0.0009760333870919466, -0.00022228738206908196, -0.003025067176170161, -0.02838778095823456], [-0.17170916977415662, -0.008337289467856307, -0.002370772983293446, 0.0004862997125290815, 0.0006469196241975674, -0.03033904912692397], [0.7021355542396394, 0.014471506802292311, -0.0016843439048160476, 0.000894506914249589, 0.006423712418091373, 0.06109239686387672], [-0.21099213618121035, -0.010318170443371116, -0.002565861740260624, 0.0031082708042943236, 0.001659983049273211, 0.022808894903431683], [-0.21880775139192804, -0.011853437692803327, -0.0009486846251572938, -0.0004969384814372049, -0.0010796092357822167, 0.027353088093774718], [-0.1648086055709463, -0.015079083959783997, -0.0012161372585843151, 0.01729129424330388, 5.2266905748629165e-05, -0.03457306769307105], [-0.173326267043869, 0.00409411305112178, -0.0027810803294449553, 0.003075082935346975, -0.002826523888594017, -0.026985324724560337], [-0.21636555059966087, -0.008091978978908046, -0.0044554367026289916, 0.0027065618515467083, -0.0030000811330196365, 0.021289818896004314], [-0.16662518139834645, -0.009601891085417324, -0.0013250436659361657, 0.0013216565564517725, -0.0019322109954984445, -0.03346039142675702], [-0.20726573168665385, -0.0116470674642296, -0.004953892492363819, -0.0037739960433524674, 0.012116116989098202, 0.021191237364168492], [-0.1751311977492572, 0.004519966685689076, -0.004881202120577709, 0.0006356321284713088, 0.0007740298664858379, -0.03152225570528959], [-0.2270540622064182, 0.0015599098347577646, -0.0012153953048647695, -0.002199390018174531, 0.00024714140242947856, 0.018841283471757713], [0.727998239409332, 0.003586149233835631, 0.0050181638425669806, 0.004530526743428615, -0.0056975619278101415, 0.051647816031977746], [-0.2187202893298414, -0.002946436749424714, -0.004994776196337624, 0.0010960337930602955, -0.0031578678270852553, 0.020485241071533693], [-0.1561675435978473, -0.015683579634701746, -0.007108082946461619, 0.00348667903352787, 0.0018673329364206918, -0.034684534473107834], [0.770276567372003, 0.019335431489985636, 0.004840659152174977, 0.0015840842042514126, 0.007776589666567394, -0.03862826694991757], [-0.21410012876748932, -0.007396834515833217, -0.004571001040785589, -0.003946523819315746, -0.0007576910932933639, 0.019105512570050918], [0.7497210556426235, 0.007259016892156042, 0.012157099754787104, 0.00605337743347738, 0.00926507745657285, -0.00628896051295114], [-0.17565008114194414, 0.0015824102554919785, -0.001799370611734584, -0.0008402428891777701, -0.0029651157909483324, -0.03134255372908282], [-0.16212590020767387, -0.008130069410561465, -0.004924536449887186, -0.0017280427284887356, 0.001348769186821006, -0.03272994907238028], [-0.22466916268400428, 0.0016312023512411694, -0.001108892347647686, -0.0006532588760214239, -0.004733812094015787, 0.01786725698378137], [-0.14948242270980652, -0.015978018178806102, -0.003129324203318371, -0.0047534388305054565, -0.0021665231100202443, -0.03611333498304707], [-0.22474107501012291, 0.019415966008690957, -0.004393782776730273, 0.003139599203363501, -0.0007665049908295471, 0.015202940422771025], [-0.21359344597711047, -0.005695627862931962, -0.0027771744672741893, -0.0047194505425054, -0.004238274287555759, 0.019357306470711627], [0.7391277289760791, 0.023633315272575155, 0.005689703380382241, 0.01105799102128919, 0.0038645313201225813, -0.008373269970449933], [-0.22393060569781645, -0.0003757603369014465, -0.004338263067949169, 0.0023658898718457097, -0.003992512486538936, 0.0192126931588013], [-0.2291276168068638, -0.0025497993559693553, -0.00146460503546929, 0.00029696778703205956, -0.0008138403255547547, 0.022825560403491627], [-0.13687232195207247, -0.015365326521893666, 0.0010093392390341897, -0.005496953537882307, 0.003602067711718335, -0.03986891457345576], [0.6147941799112937, 0.016631335139004135, 0.0030021791632744606, 0.006911015586281986, 0.0014489306266774547, 0.14054569290680216], [-0.17406272361829936, -0.012378414931828919, -0.004264239545735933, -0.0020162548835412093, -0.0012180753490498538, -0.015183353687048354], [0.6787889783733013, 0.041769971485215565, 0.004144056472347438, 0.0012931140674460246, 0.009761714280569666, 0.04215883198778656], [-0.2188382364497617, -0.010462572155922499, -0.004766496629273575, -0.0014376314485459961, -0.0013659447318953126, 0.02687088141539897], [-0.17724778609888758, 0.0018115455366516148, -0.00618231938210422, 0.004846363841829463, -0.0036055275805334714, -0.027122276316955792], [-0.21756262265049653, 0.0015389962809845663, -0.002223396998740254, -0.004858021965881803, -0.007188752705887491, 0.018627131373355026], [-0.2272618449644296, 0.0009313022904836739, -5.541914143896827e-05, -0.002287544927326506, -0.0026684176452720634, 0.023008591054650248], [0.75651498175271, -0.035439054860021704, 0.010053038396244017, 0.0005132142604508737, 0.009865248882526536, -0.05456512073960212], [0.6774811483986354, -0.0065562819891962125, 0.0014946493714321663, 0.005031581804278131, 0.025773544927009506, 0.05648567494815533], [-0.2073403525208614, -0.0066038639847725965, -0.004840418876356746, -0.0007823299989487212, -0.00611697989954549, 0.017683945280485166], [0.5489473018423112, 0.018645461203793488, 0.0038811454435578856, 0.011493829349583835, -0.0008562985499810465, 0.17288856071073416], [-0.16002917484718482, 0.008221974687360686, -0.0032300997009825404, -0.0010150693639810887, -0.0041834073195431455, -0.031183203838519788], [0.7509739399236435, 0.02551152324732539, 0.00668653989967324, 0.006255991032292699, 0.00543454203459917, -0.03882643335143273], [0.6596099987820171, 0.034670838817527826, -0.002555570468967004, 0.008332581376902768, 0.002736448389065201, 0.06318784596059614], [-0.21735724982696975, -0.008377828575524876, -0.0006890279807711464, -0.0012643067909426852, -0.0011552344655591862, 0.02314078187807851], [-0.22226943976826696, -0.0012446660696897856, -0.0021872461609060818, 0.0002296116229196185, -0.0039706552219064386, 0.019442395597849275], [-0.2149436138717698, -0.0007613533627495524, -0.006667474651421843, -0.001174455333555952, -0.0024354669387965495, 0.014315697491627104], [-0.20098795723610124, -0.012288054478101253, -0.00944067456909029, 0.007590390691192888, -0.0030838822890238127, 0.02279351121445736], [-0.17598692673257402, 0.0015103954157151323, -0.004098199730002381, -0.0008200965379332768, -0.0010298355039275841, -0.02499200357794421], [-0.22399540202654578, 0.004311202903448727, -0.004683716463413787, 0.00032563957861002314, -0.007558593386510258, 0.02193420272774471], [0.1902481342404294, -0.003102385472314531, -0.017447710250943674, -0.01438590570166624, -0.015564037122616265, -0.2435259196131546], [-0.1869479464848708, 0.014522208918376026, -0.005315253990753345, -0.0007855627497995061, 0.001741795151060409, -0.027817954022307065], [-0.2138468063719632, -0.00639144909335613, -0.006310948708755951, -0.003993003325631546, 0.001321698356500388, 0.019399996322693916], [-0.22442458995339057, -0.00029564238913714534, -0.006311195489463814, -0.0019093089708468042, -0.0013398925190283948, 0.02261396265520031], [-0.21974518086995523, 0.0018408459315196026, -0.004188070569412701, -0.003118824215910519, -0.004855842116162888, 0.01840040517325534], [-0.22781472827492616, 0.003449823347021202, -0.001223890493917326, 0.004128193723872618, -0.0004502752233969513, 0.01974237426936761], [0.7342241983528451, 0.01002703712877529, -0.004076368928607705, 0.004820694335757073, 0.006102610033499819, 0.03623516241106333], [-0.22350206222587218, 0.0003632470016729831, -0.0021996978496444624, -0.00292912341812045, -0.0028810073968155497, 0.01948197722211301], [-0.17538222604441106, 0.0047695189225900984, -0.001315823043232465, -0.005307590800250487, -0.002373010743620388, -0.03185345332508915], [-0.16814848999858356, 0.002193964368372081, -0.0055781326520286684, -0.005017129446722351, -0.0010994870288433903, -0.03097997694287412], [-0.16464313890902446, 0.0013487098727909294, 0.0009812925461385085, -0.0005098350628317216, -0.008756404247403135, -0.03371035288184023], [-0.22587232959320572, 0.0009145507830076919, -0.0025109781526102583, -0.0033042816359531314, -0.0014610904431584515, 0.020567462375253285], [-0.22497568765526788, -0.0013626074483639673, -0.002473230447759409, 0.0027460552866175525, -0.0030244375760547326, 0.020531349282269764], [-0.22206264974695522, 0.0028343766054773917, -0.0010476764756265718, -0.002250104409807964, -0.004036600451945735, 0.018229321145524526], [-0.2238294370245473, -0.002473948805960074, -0.001981976738022583, -0.001550148659715934, -0.0008537662926205574, 0.0256892775208665], [0.6665028773937992, 0.007699559015614116, -0.007926649020405931, 0.001455896602004826, 0.009888688336641641, 0.0923805434232616], [-0.22505186353253603, 0.0015886725848275082, -0.0025222584548562762, -0.0008932804820942752, -0.001800853747082162, 0.020762916965074617], [-0.21404253307457238, -0.006618096348063149, -0.002701599767041954, -0.002402275015817813, -0.002227683773987366, 0.019658854646149866], [-0.2247454505851049, 0.000825744848079724, -0.0014634038530876008, -0.0019137789583520634, 0.001346122673486392, 0.019284099208311662], [-0.21696384926860024, 0.005294109293215482, -0.006292604701938208, -0.004714082677159652, -0.0012681583669633602, 0.017277919054779708], [-0.216358721946368, -0.006728248458975149, -0.004169860644021859, 0.002641821625683878, -0.004119402791180422, 0.01790107888152854], [-0.1958874795721366, 0.00888524396724049, -0.0025394435814245614, -0.0020298046389594184, -0.00847725473186104, -0.008284594776192216], [-0.16054998736754622, -0.0051827202866550305, -0.004668497631489524, -0.004736752472971308, -0.0003745394121096711, -0.036154169495894894], [-0.22636204017853495, 0.0004739124088675195, -0.0012362731190535834, -6.50777978633304e-06, -0.001658877548183263, 0.01760699051776579], [0.2777259286457707, -0.08710460058323125, -0.03860937871988191, -0.024392734654420052, -0.017763950575845983, -0.2393057070802765], [-0.2298184546142628, 0.002209809278339603, -0.0012696161329378744, 0.005680006834975695, -0.0022211010119402652, 0.01875268897915852], [-0.2240899160482332, -0.0025622458720558354, -0.002291251321600476, 0.0008759364175220742, -0.0030738652512734737, 0.020082783517082398], [-0.2014661426125345, -0.010789719115271683, -0.00372331442202851, -0.003863813743356045, 0.00013232034408780445, 0.015544002882436792], [-0.1756060371459245, 0.00093276572813734, -0.004190165079373976, 0.00372527991012532, -0.0034102613601986317, -0.03182464406826877], [-0.17179388612783755, 0.004369488939851895, -0.006819543404486884, -0.004659818643316658, -0.0020574611559143036, -0.030457759991146856], [-0.21728458057514055, -0.0060956048670096795, -0.001937141194746588, -0.002099829998975143, -0.004577040073209734, 0.020915765336532856], [-0.15891948062282435, -0.007271542982427678, -0.005149296066264887, -0.0002002671738949396, -0.00012154087816046388, -0.033294267625265], [-0.18498595200858037, 0.003004516612921487, -0.005598736343787142, 0.014053896723455778, 0.0008000508974112849, -0.026797585405230147], [-0.21360668532310778, 0.01257312254764855, -0.0026537988723999437, -0.002335052356810645, -0.006625485353264298, 0.014925677135712362], [-0.20839088863961056, 0.0044348832426400345, -0.0016606406743734637, 0.0035554619688273966, 0.00011764215028414625, -0.0038523764151144665], [-0.16463415769375303, -0.007250225826002195, 0.017047497100777194, -0.005592620324767278, -0.004965788532999101, -0.009402323770874547], [-0.17364451650767831, 0.0044239461933583656, -0.0040733514162693005, -0.0077473768787599435, -0.003777302248582948, -0.026848065808734417], [0.6648684443346787, 0.01669694500610664, -0.015527153981038067, 0.005015141276478737, -0.00194728585802062, 0.08888200445988902], [0.21659657092176113, -0.0029929688526536486, -0.01672303280062835, -0.009144936765712456, -0.011802274237199968, -0.2521590537587045], [-0.19045327939840606, 0.005852449671238342, -0.002127337105307387, -0.002323564990908353, -0.0031503599157964987, -0.010922908260820213], [-0.16874886662904406, 0.005904841965029561, 0.023102170116662674, -0.0002601591867815577, -0.0074914053027955235, -0.031256580963070754], [-0.2181419229280883, -0.0011941160392060105, -0.0019284064294567433, -0.0002859118204418247, 0.001817391116518168, 0.019912453280162185], [-0.22078660806956352, 0.016577221258919132, -0.00922080379593516, 0.0006989023095931218, -0.0059263295778824625, 0.019490951208202073], [0.6800359666091554, -0.002437731615778019, 0.0053465777954029, 0.003460661103864068, 0.0051675814932044, 0.08215710334430966], [-0.2190147894719964, 0.008987325783101881, -0.002868964800513895, -0.001477724827473148, -0.003890440814804905, 0.014681260798353035], [-0.16452723053478227, -0.008877943904882755, -0.0012990392254668764, -0.0020592681990947476, -0.0012746083240367109, -0.031584971827240356], [-0.2180056912563153, -0.001892974102932722, -0.00024038712748706092, -0.004735556060132618, 0.0005197192519977991, 0.022122611768474933], [0.1422226144764038, -0.0031669567443882896, -0.008289169405887298, 0.0025979631152452227, -0.009257832311600581, -0.23374753828813397], [-0.22241733563916233, -0.003338681633495208, -0.0032211173418181596, 0.005941627188119714, -0.006033071367767067, 0.023010020235564504], [0.48954365817608086, 0.015139377856258778, 0.004449143829887461, 0.005186950213031124, 0.007767241821685931, 0.19319140588083233], [-0.2226623455865029, 0.004327931360138673, -0.002436110238764259, -0.0047080912466065826, -0.003209700170057356, 0.01702164921512593], [-0.15073062340705792, -0.014552367230820505, -0.004407619512990895, -0.004346851449199903, -0.001801054237754674, -0.03578454617767976], [-0.17673255088560336, 0.009850278483763191, -0.004672307564540671, -0.004011375339556378, 0.000808579903789332, -0.03086568661335574], [-0.20590882921305503, -0.010178966524857658, -0.008399081134305535, -0.00040803814239511946, -0.006615496695384641, 0.026093745043331486], [-0.17897627463479682, 0.010288958515886966, -0.0031635494629348626, -0.00409150587843101, 0.004124457508344561, -0.03180514806357246], [-0.1628082961104704, -0.008182348891239885, -0.0033975096429642053, -0.0034641631946522983, -0.0018176407701931937, -0.03195310340598384], [-0.19158091565987512, 0.014804172218601287, -0.0005429787947851895, -0.0004903997188887262, -0.00479764322260018, -0.028558901489118733], [-0.12502705695982558, -0.008506709471962994, -0.0029523180169526465, -0.0063892556655870975, -0.0014432950211901116, -0.0673480315311483], [-0.22077695449174986, 0.001096825312641732, -0.006243047317570143, -0.001967640256576847, -0.0025455101573784353, 0.01910299357730038], [-0.18186207073005645, 0.0026909173790351107, -0.0010779765009249094, -0.00023452413814027773, -0.003161264971430212, -0.024688414371816455], [-0.21710020295484855, -0.00762508733534033, -0.0020950852240205903, -0.0008358501354687703, -0.004292776729831848, 0.02028233571284342], [-0.22224228729055445, 0.0023535781716437273, -0.0016592520205182943, -0.003225342353466571, -0.00325785102779748, 0.019697821187359705], [0.5466574466913845, 0.022537899027406154, 0.06686796919504799, 0.012913578938473596, -0.000548492506673874, 0.08990493198769438], [-0.21412465870505323, -0.003506299326236045, -0.004634084607928735, -0.006261937501540646, -0.0021125045040521176, 0.018972817978144412], [-0.2212854068233015, 0.003822600592021672, -0.0029680221696576125, -0.007305474765021596, -0.0005557799540096038, 0.016625416453301827], [-0.14831193791975225, -0.008463922061717238, -0.007381442374613536, -0.004148695059027607, 0.0007137487161626194, -0.040364146649889326], [-0.18898396517132912, -0.0026113598782668925, -0.0023685449548801916, 0.020762019370677242, -0.0014676984126331806, -0.009695530318647139], [-0.22022911555096897, 0.0003102128287423225, -0.005893846557673196, -0.00038427716967515997, -0.0043929081504295, 0.018923267933338157], [-0.20188300944377463, -0.009584440441831647, -0.0038252885233312282, -0.002671074178736196, -0.0029625546901245257, 0.024468033944465433], [-0.15587111852064384, -0.00876799112381842, -0.0038210026825413676, -0.00423980001817784, 0.003614123755736778, -0.03341421141055533], [-0.17928315654631966, 0.007166390430769043, -0.0019778065817461134, -0.004011211867649779, -0.004147533284724091, -0.029369744165833215], [-0.15050173777858297, -0.01577887381658414, -0.006867040243330025, 0.004002601609307708, -0.002520504682249281, -0.032084445088561085], [-0.1949892284585164, 0.013231879517433333, -0.0013091245770521986, -0.00160466562425218, -0.0007584766406922564, -0.012915622312158192], [-0.18488311792731682, 0.0028538466049368282, -0.006317385949741245, -0.004956677284797336, -0.00470055271777075, -0.01249611272531069], [-0.22216246281317104, -0.00255366244708978, -0.004141099547485801, 0.0018838209909735225, -0.0022487683117769683, 0.017555505461883624], [-0.21673035469037555, -0.008306476153702393, -0.0032236718202993935, -0.0014171194372821216, -0.0028422550328859844, 0.02085321046787909], [-0.16203923203012519, -0.014624146719377487, -0.007099941265199667, 0.000491501216543604, 0.001304532430010949, -0.0296557756473557], [-0.21982624551204277, -0.00435092746237809, -0.0054895916540593625, -0.0033624655276974164, 0.000729260761921675, 0.02271663606092299], [-0.22406366986685575, -0.0027405365466160944, -0.0033061025214598913, -0.000996879131545027, 0.0007227399163483164, 0.02413444815012851], [-0.22716032274150819, -0.0018497586497001311, -0.0015146239211393277, -0.00021756071193433258, -2.0097147245669926e-05, 0.020345696504861115], [-0.21294762768153383, 0.004805377424146795, -0.009054781348173946, -0.0019515929666087295, -0.004561579072047782, 0.020793536977551094], [-0.1856622390313994, 0.013452159459430239, -0.005535239222058784, 0.0007035587220923963, -0.004111449719719586, -0.03001345687501161], [-0.16221810669203668, -0.015168184579243025, -0.003603854900356817, -0.0002523906632020836, 0.00026940961329749967, -0.02814993479396268], [-0.18966944692042426, 0.003298893295397746, -0.0012557532909831793, -0.005895193923471984, 0.0035355780854796166, -0.011561696293616941], [-0.23111130793957202, 0.0039307848220586075, -0.0005614746467025249, -0.0009532504750166997, -0.002755835326031711, 0.019784416898597192], [-0.1579495333271218, -0.0125226476911605, -0.0026865355077122007, 0.0016138822079282706, -0.006208053430818921, -0.03220350759995195], [-0.1941061679142732, -0.0025578745907682623, -0.0023401936098530274, -0.0014177898758035697, -0.0018602494093093564, -0.0033843912666590985], [-0.2213213261736558, -0.0014497667003478306, -0.003562004277212882, -0.0018871036938493638, -0.002542983610731533, 0.02159651778913058], [-0.21731971376373918, -0.004967831613825112, -0.0021596008703986314, -0.004704378395448982, -3.772482826549703e-05, 0.019368736651165157], [-0.14807043756469965, -0.01357633075469862, -0.004675387321654652, -0.004143842465664335, -0.0014927263092560578, -0.03720794225069312], [-0.17343684051801533, -0.0069955255416521275, -0.003266018897467462, -0.00277992117010684, -0.003774980863402727, -0.010163379676022356], [-0.21126677967050653, -0.005111475677749633, -0.006251357428126876, -0.0022098246775321463, -0.006905026443748471, 0.022577797230997412], [0.5482171921765564, 0.00616370758983418, 0.00015019085467658455, 0.0017053402175246871, 0.001635534388453871, 0.2025447014396197], [-0.15400685020857746, -0.00974417407646317, -0.00503284527458769, -0.0034713291173856524, -0.0023177367448300027, -0.033760397911489166], [-0.1728909255185986, 0.002494530616054954, -0.005644674205761484, 0.0015049807609745136, -0.005018299520635812, -0.03206867414753698], [-0.1914758863846138, 0.004558342187401685, -0.001916880981944537, -0.006059235918330064, -6.596758313856347e-05, -0.016502956353387967], [-0.21636365312533049, -0.0033928004047753717, -0.0016454363170899725, -0.005862333339194704, -0.0004938072724588079, 0.02038849803049376], [0.7283798074092983, -0.00026451327422111643, 0.00608159501253205, 0.0020228632718006473, -0.0014040789551154295, 0.03653048038185855], [0.7212342232368106, -0.045826610493973456, -0.007745369659945027, 0.014051912555356993, 0.001186653309484322, -0.021150808947733878], [0.41861473743793437, -0.1590624947551512, -0.016065592409981393, 0.004464757416029532, -0.02508865926548772, -0.16495798651858198], [-0.22309561483663018, -0.003231827292607674, 9.818287328982621e-05, -0.00013114970349878343, -0.0009082813728093216, 0.024065824570566812], [0.7232473112919823, 0.01399888374239889, 0.0048494491000862314, 0.005368986816691925, 0.0039857323098052315, 0.02896630340569946], [-0.17025808074670415, -0.010224077811622235, -0.00026022383287179146, -0.0086262303341694, -0.0028203259043079218, -0.0031860613703245913], [-0.17282511062989683, -0.0072453763544288305, -0.002716551790950432, 0.0026269890129152216, -0.002428015073476698, -0.02574526849749547], [-0.19102058505205435, 0.00809212456478704, -0.003585926866370409, -0.008394739133905729, 0.0032434544766145657, -0.015000994655737402], [0.18623519896426904, -0.0158193469650117, -0.00868181036201332, -0.013061369131646669, 4.2478071160942626e-05, -0.2811894030684526], [-0.22360829823763362, 0.007254576942718255, -0.005138679116474356, 0.00045017796518338207, -0.00017239616015832736, 0.014528572094736942], [-0.20907926283570086, -0.0075531969400729855, -0.002070090150818251, -0.00592834330131531, 0.0022109688892818363, 0.020516078184779917], [0.7418292429750131, 0.026941127454255694, -0.0018515705434235322, 0.0020917259181356443, 0.010663486531367207, -0.0315365664479022], [-0.1955767153496795, 0.00343214577121999, -0.002951668992069249, -0.005120471887739224, -0.0005594216597301761, -0.002640534548668455], [-0.1677048439404002, -0.00647958558195924, -0.0013253785845921367, -0.0017660751684307864, -0.0007504171526967905, -0.03193009492075794], [-0.222088965731027, -0.006606830873688377, -0.0018305632052817201, 0.0024282676578701707, -0.000912703991936939, 0.021461776536220743], [-0.1718245366672203, -0.004639722625315516, -0.0015182483389977219, -0.0013384618676569443, 0.000870337384378927, -0.027815287043549252], [-0.23223715857670024, -0.000325947369575799, 0.01568786288158795, -0.002486536643746728, -0.004292729269848766, 0.022904508978283573], [-0.21527144561198594, 0.014367747097821265, -0.0006724548359066903, -0.004293220578764124, -0.006309445297008184, 0.02086929541631963], [-0.17249120695737605, 0.0005364093349456976, -0.003976840270206563, 0.0006703262433439241, 0.0012810671404674579, -0.031142817506678175], [0.5608539759442522, -0.006357968498596855, 0.005819710480424921, 0.009197746416259192, 0.00016215247549786164, 0.1837410498488297], [-0.2240748785162707, 0.0009738240211218423, -0.0031648151950921925, -0.0025213646095586574, -0.0026602254379163894, 0.02144745973771647], [-0.1704714087005977, 0.00040078864020556625, -0.005002057698121153, -0.0008123552712319081, 0.0001691704053443572, -0.03542332842336077], [0.558177959382395, 0.007920868376421973, 0.00892041642563453, 0.002395515362215232, 0.00046586069434189704, 0.19006082120043244], [-0.16832596714169046, 0.00034705402695126453, -0.004203590830356274, 0.001998441616850132, -0.0036954100614212736, -0.036076922959170234], [0.7377202025305465, 0.00732525533254567, 0.002330541994951648, 0.003318040518274633, 0.004987348516405106, 0.025498098286763696], [-0.16911109099377286, 0.0010795141378869297, -0.005985705407391898, -0.004243958319502655, -0.0019453248098656444, -0.031416496622857566], [0.7320312609059653, -0.015677922818191338, 0.00470855379008598, 0.002267229227075419, 0.0036560717937569346, 0.05016908419676034], [-0.19066247905570058, 0.02028269510545249, -0.00116816115378492, 0.0007203663715302962, -6.907183010361951e-05, -0.033031920865964993], [-0.22933167994370346, 0.0018039659649418673, -0.005457709181154218, 0.00333128495474093, 0.0006115046546280357, 0.02126485577276928], [0.6079585502783592, 0.10663329429622138, 0.021359659875666973, 0.01645134067256108, -0.0003797533614011286, -0.08161043949119895], [-0.21981750474728842, -0.00546993866436694, -0.0021631768933054194, 0.002924876267448189, -0.003795626259534875, 0.02101281173848899], [-0.177336142343723, 0.0043515065829111245, -0.00041501016586296323, -0.0003876952943823561, -0.0026403575058039598, -0.022738967939805316], [0.6728553538575349, -0.030994253071106075, 0.005506526667738958, 0.011082783788239532, 0.005898579216312553, 0.11065100954127892], [-0.1567109171558643, -0.01467442528568579, 0.00031612012728159897, -0.00170509720229628, -0.0006335643888750094, -0.02825878276122689], [-0.22944264374168588, 0.003953781499930739, -0.0023157051740926122, 0.008598275452635798, 0.0012349443788349154, 0.014054680917710241], [-0.16277438064367195, 0.0025979909920475913, 0.00043383920391159957, -0.004342762373947243, 0.0005607150216322985, -0.030181797548809234], [0.3295670936677148, 0.022005316659646265, 0.15935896671319635, 0.006748540926298836, 0.014191051558948864, -0.053960255240091175], [0.6872785332288064, 0.023288214023246184, 0.004722805907404156, 0.005762296393941809, 0.004863925577534403, 0.05364817835743953], [-0.22337521042951808, -0.009047320880755947, -0.0026370710430213447, -0.0010044750805697892, 0.0009730721703968359, 0.023424338596801774], [-0.17955136396209181, 0.0041684818858144955, -0.0013076147660044271, -0.002395699759129404, -0.004004561459339111, 0.009924091394083515], [-0.22456080860783, 0.005369900429039085, -0.0030898246358104864, -1.6492447789271463e-05, -0.0038184619161087986, 0.014449020511832332], [-0.2258432620122526, -0.0014158980321535594, -0.002134108321334277, 0.0019144499939519537, 0.0010004464375392747, 0.022141730081478034], [-0.18768029933276884, 0.01117893225993443, -0.0019392776387494241, -0.0010236315806877994, -0.0028598452877517254, -0.028298940435480407], [-0.2180792603274055, -0.004336174380892429, -0.0058412857805565585, -0.00304520065931227, -0.0018125716024800004, 0.02144782608398016], [0.5076581618004037, -0.02334888440171202, 0.003997351852199646, 0.010723452641959834, 0.0066000117723947795, 0.20264371585856222], [-0.16496497801765408, -0.0034205786362342273, -0.0027101374017077656, 0.003120846166716169, -0.007135899778056565, -0.03317898101523376], [-0.16671763876839826, -0.0029160699825095666, -0.002442426618067932, -0.0041676587688386415, -0.0022457644620772505, -0.03151044140010803], [-0.16600134169626218, -0.004551769291022241, -0.004285764095329819, -0.003974951378357767, -0.0013844500995587932, -0.031424785454973105], [-0.1596499605029756, -0.014728469022058916, -0.0013130985277751972, -0.0015892614727548396, -0.0016682225470674675, -0.03267404994287179], [-0.16663924931074614, -0.011275759367418698, -0.005128189214177693, -0.005351484887050978, -0.003645420740038126, -0.018868672781786052], [-0.22006767297015403, 0.0012153138361174253, -0.0053698856418440575, -0.004584145501609762, -0.0005683694413992324, 0.019554246898376825], [0.5459649037120984, 0.006202593992876407, 0.0035887316960115405, 0.0025396194204614845, 0.005260712298266503, 0.19446724840409507], [-0.20662073707468553, -0.00380980550221253, -0.005210868968606258, 0.004953775703071254, -0.0042041922999374775, 0.027808494809037833], [-0.2256770703895921, 0.006390897476259823, -0.006395185880137122, -0.00464439745794539, 0.0014743456814628921, 0.020101410569952172], [-0.15239039307864707, -0.0174439992383436, -0.004761799607350265, 0.0037447302031931593, -0.0016821289838507775, -0.03075613797717186], [-0.17230056279578143, -0.014151336119764987, -0.004474031221689819, -0.0004561004639506267, -0.001956115115260289, -0.015784916299056845], [-0.2188573456725202, -0.008518242861705005, -0.0011444086550402857, -0.002814814034267374, -0.00024303376924510804, 0.023732122088231583], [-0.21721393465887018, 0.005936485322787392, -0.003978802066880756, -0.0055371179776834084, 0.002275411989015111, 0.013517957391631835], [-0.20339777645238502, 0.017156007878216653, -0.0031143909539431127, 0.001263620237576324, -0.0021275585616036285, -0.010196568814527686], [-0.18307179350674313, -0.003692095584943448, -0.0057927893270374895, -0.004018114212800095, 0.0001912932222854881, -0.010283167257428056], [0.7367417097158379, 0.02551005548778702, 0.00936915683621099, 0.005341583635612337, 0.002782145427589234, -0.014881014739401963], [-0.21469413106190566, -0.0029680305644563107, -0.006102240424197254, -0.002633223072652616, -0.0028273155277822527, 0.022558273984327602], [-0.15346555811695906, -0.017279527311874994, -0.0024396467016247593, -0.00044996122029579454, -0.003287555006658302, -0.034700813658090655], [-0.22859540232661452, 0.002118739587674343, -0.0011186145720031326, -0.0022062596202036236, 0.00019076783919958558, 0.01794410242528075], [-0.22383092198740823, -0.0010738108486307268, -0.004193442496067876, 0.007823521232225584, -0.0019448185166830887, 0.016552805949897307], [-0.22058002048622075, -0.006370605742672707, -0.0006822370307693577, -0.0006084075944986513, 0.0031549483521425377, 0.021099143014839387], [-0.22763446612061414, 0.00589650269000854, -0.002417615336495547, -0.00431642911346485, 0.003568549807171621, 0.01757012474006146], [-0.2011163812352487, 0.003621612826962431, -0.0011971466585312248, -0.002413155550544676, -0.002464925144568469, -0.004763337571402827], [0.7384691847129091, 0.008743461966509904, 0.0030933178672181644, 0.0012350586069925812, 0.0037733791840153666, 0.026531751508508235], [-0.17207420684457486, 0.002294862865363359, -0.0014326530449713744, -0.0034426209567004415, -0.00878307421556168, -0.028185369819058625], [-0.1629720372278863, -0.011232094462248746, -0.007693094823410046, -4.780629331748639e-05, -0.0007711369333638847, -0.027700496926439642], [-0.17279748126940891, -0.015139730097990354, -0.0025059063903716504, -0.005064862383569151, -0.0010271282668394392, -0.015131558258487079], [-0.24355568393254845, -0.002988761266460951, 0.08015404760229537, -0.0037592983056950902, -0.008022820301125265, 0.003561405092423056], [-0.22015372090100496, 0.004455118540145596, -0.0046809621494140665, -0.0027313003945871875, -0.00554569158308696, 0.018989889821281174], [-0.16721953750987062, -0.015397313879793024, -0.005808530893677776, -0.0037965230410586413, -0.0006201948294570513, -0.01628096186164645], [-0.19285414193944062, 0.014352569395878008, -0.0006210268649692575, -5.4016562997553084e-05, 0.001082914093507158, -0.02273963145531075], [0.7260229811029832, 0.01375450240314778, 0.005009887828492463, 0.007567407348929969, 0.006548926183571187, 0.02371534275192186], [-0.22452979062111866, 0.0020764730164074888, -0.005404302036880085, 0.0025156333175279746, -0.003354829129762294, 0.017844102275530675], [0.6913937343714209, 0.01370902887670243, 0.004493552739313602, 0.005532568066311063, -0.005802273733879986, 0.06961483112157242], [0.7028893443818864, 0.014679791373233778, 0.013129143567801652, 0.005895703494683195, -0.0007915300762861997, 0.024280880592015306], [-0.22222400592410424, 0.0011754535989510725, -0.0044211507807324, -0.0025959171954631024, 1.8448222847712902e-05, 0.020880505411834678], [-0.22189727318834296, -0.0058429526469045865, -0.0020875322847785896, -0.0007420246850243066, -0.005843185967418197, 0.02474630210580185], [-0.16837662832715866, -0.00990721094320798, -0.0017747717395252226, -0.0019902468818375806, -0.0011855644914413868, -0.028388639632332697], [-0.21852701281325865, 0.00021343699434044191, -0.0032409174499833785, -0.0006526552371757427, -0.004123883987819813, 0.02010881027167479], [0.7386532328514203, 0.010109526713107738, -0.0012353708723946979, 0.002147307870706443, 0.0011193136807127653, 0.032539323089780706], [0.7340210808186918, -0.0004802392534800307, 0.004580740350268466, -0.004538018298321594, 0.006485101408196554, 0.03105633497464186], [-0.17838682681585422, 0.007482949400923475, -0.006469176922400999, -0.0008326239424758162, -0.0014570574007019873, -0.03114637284662187], [-0.18612098095330967, 0.013237620867911033, -0.0037206754531405005, -0.0007717501121590958, -0.0030251986904390187, -0.031265682325529405], [-0.21380921449904697, -0.010346782920462306, -0.00010656448250037447, 0.0013059690836650293, -0.002468153686375991, 0.019174746504720744], [-0.15157009867622895, -0.009537132354714675, -0.0047846581323357585, -0.0029740998570081346, -0.000310531517918293, -0.0382798748106314], [-0.2222031092553177, 0.0026211846775602535, -0.002875398158250333, -0.0008210745691685009, 0.00031386842191279466, 0.013964528883263338], [-0.17996979518871264, 0.0020274490895582135, -0.0021208309734982063, 0.002834744468183192, -0.0025699627944032527, -0.03162058498397782], [-0.17762626493941924, 0.007509361519087292, -0.002332922879227718, -0.004137865472892385, 0.0027640521028609467, -0.0307994223459125], [-0.16157886339890132, -0.005426390096261913, -0.006125022823210881, -0.004057641721761714, -0.0020225987379100243, -0.029912545237458007], [-0.2137327689480288, -0.005770231753441561, -0.005334960581598721, -0.001527516465328785, 0.001290201778529318, 0.02924194263653566], [-0.173065344838464, -0.005807720712201976, 0.021335560677186043, -0.003799163912858123, -0.005705874553729811, -0.02879078999326535], [-0.15661582956152814, -0.008464043619337778, -0.0017178120637894717, -0.004463153550512332, 0.0002592251156725775, -0.03645478166934203], [-0.16435402144392194, -0.00799734013885175, -0.0021172421063292753, -0.003232032279690066, 0.0008172678095186602, -0.03473969385622938], [-0.17542642888957632, 0.0020379724154487968, -0.004251909904758777, 0.0029220146435631365, -0.0014917431023216284, -0.03166296717785889], [-0.15889697572194955, -0.009968717802778823, -0.004192698899371837, -0.001994051357085808, -0.0015381479513053484, -0.03503247028301231], [-0.15851477782401605, -0.00846464038614314, -0.0024836212583742505, -0.001490790904119396, -4.1569348459639644e-05, -0.03896099562772427], [-0.2161975554149088, -0.010977585489624537, -0.00243274171576796, -0.0032295115185525993, -0.0023252956487903264, 0.02349602312097772], [0.6314943523529295, 0.06884329526472177, 0.008176285147746392, -0.006007080622526389, 0.021059830409079234, -0.09082610820381745], [0.08402503570701696, 0.026096402612080183, -0.007541009124265001, -0.004982430415263351, -0.001498884801168602, -0.19953959016887668], [-0.15760022328114487, -0.011342339586524053, -0.002116570130532335, -0.0020523802863354197, -0.003267838096390215, -0.03524371063457708], [-0.22169301831937033, -0.006656594800387781, -0.001069285657578647, 0.0024654359592572346, -0.0017964224014698498, 0.022749885219549452], [-0.2214820815684048, -0.001352008946303509, -0.004491614559932444, 0.0006294521191689206, 0.0003614193959002173, 0.024918166892905554], [-0.17117697709419177, 0.0023610607009473122, -0.0052470529526469755, -0.0014395294969436142, -0.001960409410531893, -0.030826820428803484], [-0.17739713807920932, -0.00851142947026427, -0.004728155591798237, 0.0038482317025254776, -0.000591295034145854, -0.004411880193774443], [-0.22029039070266382, 0.002174394698101303, -0.0013798954839824035, -0.004364349159740088, -0.005811443462306639, 0.018005017443925235], [-0.22276649325605952, 0.005565221218184057, -0.002457411428807316, 0.0017208519774302748, 9.422467578190057e-05, 0.01314122586108987], [-0.221628038673024, 0.00045424863274488197, -0.0014882906839303088, -0.0034162764146885623, -0.0025001709913646288, 0.020245194796929697], [0.456688370208384, -0.12308391472040021, -0.003307842415386353, -0.012336057156259447, -0.034419386634914326, -0.18472772336041918], [-0.1499647977847665, -0.016223264366897885, -0.003072272604345413, -0.005343594774083862, 0.00044200470242820796, -0.03496113718783778], [-0.16029945530316458, -0.005902229831256785, -0.005134542871402816, -0.0017741380042176165, 0.0013089346506758106, -0.03282163065613793], [-0.19270925449941287, 0.012764012483146778, -0.005878701826441582, 0.007856886809879204, -0.001841933443859358, -0.02102434285664526], [-0.17256202173461854, -0.012968596885800229, -0.0022907263102633804, -0.004206623502124089, 0.00015480678327514733, -0.01724990036597251], [-0.21163148265399692, 0.00040362681411807593, -0.0007809175010278434, -0.0016124472423783248, -0.0021006024262966417, 0.01688848967624858], [0.7407451050402222, 0.04369592579946338, 0.005709740707827678, 0.006563128731155996, 0.0029880418009005888, -0.03400497238260152], [-0.2352444459847991, 0.0081862217032681, -0.002015558502263278, 0.005099042538348865, -0.0012171604372979829, 0.021164122904965436], [-0.21616166945691526, -0.004319393091990788, -0.001777081173680705, -0.0028412652686906573, -0.0009745953776891005, 0.018693051988014225], [-0.1664896752436688, -0.006905026490491837, -0.0009347599554332143, -0.0005578895563923652, 0.00030915807499389827, -0.03143676073640308], [-0.18011990166439643, -0.0059064792603012745, -0.007017473245688734, -0.0049883611736187515, -0.002621323030099202, -0.0030131282925621923], [-0.21729552345789255, -0.010710651579641208, -0.00563719185555264, -0.002428324138209192, -0.0012419081874180079, 0.025646932552047468], [0.6216627897090491, 0.025608714608876527, 0.002149128755980227, 0.013516730534055195, 0.002282017976673301, 0.11137784063758778], [0.12651768362196195, -0.039760412444036046, -0.005759830869239558, -0.0072065163886053905, -0.010267043671122177, -0.24786916448668467], [0.7508184403126749, 0.01966253625779615, 0.007173668237941575, 0.003974085474211639, 0.009253577717705486, -0.013074227192249594], [-0.20740392033168065, -0.005264144312749077, 0.0007594499217344506, -0.006792665859013968, -0.0026293303056688375, 0.023484887982832187], [-0.15825914941429461, -0.00795228030999684, -0.004394601251438587, -0.00385949629327161, -0.0031823917742749668, -0.03397514297222738], [-0.20033886709002688, 0.0031920366881236176, -0.004863074416970992, 0.0024802519536356008, -0.0021358870083281107, -0.0033344601264331396], [-0.18073952838819773, 0.014462093786573513, -0.005745712717351299, -0.007539070000224221, -0.0015723614531975335, -0.028032087894269104], [0.763704721086943, 0.0016639043148104424, -0.0012022123905378562, 0.009949151251976509, 0.003408664388705559, -0.035649228651898054], [0.772078434654455, 0.015325308137673133, 0.01061629579333529, 0.004435877187176158, 0.012763976531979157, -0.032039051639907494], [-0.22189065221273083, -0.0013192895023311567, -0.00041148392358723564, -0.0038540852434246226, -0.003976279546233906, 0.02228512376164062], [-0.16176551313732773, -0.007460518794864338, -0.003999595408037289, -0.0029620385009108104, -0.001645228400318602, -0.03379016777404515], [-0.2135638396463745, -0.004392643982154004, -0.004285108849109362, -0.0032576330509599288, 0.0014617765870226858, 0.020585067989194388], [-0.2043422215236589, -0.008764292113150975, -0.0029379967788053137, -0.003101211916464693, -0.0030226719985124257, 0.017049346711545208], [0.5520354283535125, 0.0255849546806529, 0.0072867063747738894, 0.010518046483723443, -0.0017886666874879036, 0.16344686412815826], [-0.21689409340122476, -0.010016195310460151, -0.002936832879994345, -0.0024560613328226716, -0.002907686366184417, 0.023544202624019794], [0.6247794461533068, 0.050857905261940006, -0.02702735534150852, 0.01760494618866517, 0.012176076528901283, -0.024985715761002298], [0.6884711336971412, 0.02126065266727086, -0.0026341235385039688, 0.006703674132841014, 0.0012957227076815466, 0.04690294033356861], [-0.22739279579747318, 0.00758915175107774, -0.0026062438706665266, -0.004943334528100244, -0.0023927173691609973, 0.01807927314765666], [0.7443689721947183, 0.0028884982682760786, 0.010715778989230169, 0.005362728354216268, 0.007719982597692772, -0.015032150880324571], [0.6822025492847172, 0.027900705084815257, 0.007136919949985376, 0.006931658478715328, -0.0031921590503883905, 0.06568699291882037], [-0.22109200653464242, 0.0006352543036823151, -0.0023129503694599594, -0.004826599785758016, -0.002749511056738412, 0.018679146776249714], [-0.21498429748477627, -0.004196481416779607, -0.004047136488459221, -0.0029725139677624817, -0.0024203077304342084, 0.01978740375487887], [0.7404892319492279, 0.016679359809392613, -0.01077732595467457, 0.0017490890077771352, 0.010309725188519983, -0.03955072935089278], [0.7371843326509788, 0.011239122700888728, 0.0023821407230586304, -0.0017771487665708397, -0.00012166611256430792, 0.03275988547087484], [-0.22173721475930375, 0.0049570869953913565, -0.001267948083455671, -0.00041513505050962476, -0.0024264540891853308, 0.011722998320396253], [-0.18176702527614602, 0.0020075318492322886, 0.00024830064924476046, 0.0038621249788458943, -0.0003948481899062967, -0.029329146026774312], [-0.1954186156912948, 0.0152269698311674, 0.0001813782169620088, 0.0010383018972683741, -0.002328117611507072, -0.016795154737834293], [-0.1824540924767897, 0.00626541095161798, -0.001432902622907133, -0.0028048582665748814, -0.0006241585820536618, -0.03036837938614328], [-0.18018046728644077, 0.004940482464821267, -0.006110433008187522, -0.00448598009400433, -0.0018906681919329722, -0.023895995899759583], [0.5649648985616298, 0.014263736082109074, 0.004346162721031705, 0.004378903060885859, 0.001980875193186042, 0.1861209799367122], [0.5643488203608823, 0.014089327753672753, 0.004432684641869938, 0.0011057531130214753, 0.004468424129109567, 0.1941661011125545], [0.7001437541756453, -0.001275201711666475, 0.004176434532849091, 0.011904144186580964, -0.004212924541160139, 0.057041571135527394], [0.19604939287299594, -0.056036744624425844, -0.011923045615209139, 0.006281044344570411, -0.004643668166125413, -0.2291436454784727], [-0.20137385864682306, 0.0009075376832211575, -0.011171834825955285, -0.013387606358590395, 0.10091006231100341, 0.01629625539269972], [-0.1550168124353314, -0.013118498027627738, -0.004438221547031527, 0.000682629050493779, -0.0032981801044662993, -0.03643397895154041], [0.6211075597156376, 0.02603784823242706, 0.0023070421622873164, 0.00901385223943764, 0.0035848037987706587, 0.11544889385144057], [-0.2076942826817765, -0.003672519974170644, 0.10652638378755978, -0.006002754605344439, 0.000421304546041915, -0.018459083453262776], [-0.21655047255797594, -0.006592213344061117, -0.002406991043553928, -0.0031754489390864093, -0.0027353865127119096, 0.024382081024840498], [-0.21287677734733187, -0.005801517776492144, -0.0029474073196001545, -0.004904003223811551, -0.00020299576028726243, 0.02152436809419004], [-0.18199945183508756, 0.0040162825960983915, -0.0017734778625128325, 0.0008878742582003025, -0.004468097639970415, -0.02828619153223141], [-0.2195452892660451, 0.0017471107343818145, -0.001117128351398404, -0.005521718516974926, -0.0021304893128252277, 0.01740084804619533], [-0.14464137261588372, -0.017489861798635268, -0.007494906672574397, -0.0018186744282123568, 4.4791563500845376e-05, -0.040223038063698785], [0.7304463911125975, 0.02791394102198409, -0.0020175971153079736, 0.0030885270124531186, 0.0053899453250125925, -0.04072596926150008], [0.31682862132485323, 0.013475316269609343, -0.017574539039286827, -0.021410486696309478, -0.022149506363384575, -0.22066746751098598], [-0.227177750419949, 0.00597230598266333, -0.002988279844463353, -0.0010394016983199456, -0.00273610588625263, 0.0163025651996546], [-0.17033741168099656, 0.0012377044463163867, -0.004552692627332484, -0.004406581671272951, 0.000547513345132191, -0.030407512194697197], [-0.22991729578575199, 0.00327580499783637, -0.002099372957156916, -0.0010199432648068506, -0.0026243454220135835, 0.02071848576522599], [-0.15294518315644334, -0.007663089510502519, -0.00014012707435890503, -0.00368003436314067, 0.0009043612441223097, -0.034765655821847485], [0.6473021260881935, 0.007595258367986214, 0.0052319305319737455, -0.029430523419493728, 0.0035839060218812308, 0.06019349288565075], [-0.22355417693106328, 0.0027410972444089706, -0.004195767042778525, -0.0038498240619949347, -0.0015625259125862374, 0.024004530037347396], [0.30509954336060613, -0.05120996297938814, -0.009831689764158366, -0.008181361498185592, -0.014871778113067483, -0.1456436860371837], [-0.19056636993273077, 0.0011742477708673483, -0.00019867853949070182, -7.184175930575424e-05, 0.0006555485166096233, -0.005576239389283105], [0.7253260758630303, 0.008597769366854156, -0.0031170144057856286, 0.005495409642525205, 0.0049547982401307675, 0.021867961293243636], [0.740761907558525, 0.0047910768140015944, 0.009281225700163882, -0.0038541671751318868, 0.0048690354438221, -0.013575268817571677], [-0.22120863808132868, -0.0049278903609007545, -0.0018794615143469385, 0.002080044333916419, -0.0005737943783192347, 0.021284514775754035], [-0.14962608771088876, -0.015453540880624755, -0.004264153646461535, -0.0011889950361988915, -0.005436631547724418, -0.03565365319360555], [-0.15422442046280985, -0.0019634887796297655, -0.006315273880587089, -0.0030874861296760602, -0.005646577401222416, -0.031679420012741134], [-0.22847173901130213, 0.0023071992380556783, -0.0008053779745490406, -0.0007307378025824545, -0.0005653347521311986, 0.017413277124214158], [-0.2234120853544642, -0.0035311351481064376, -0.002389218298591659, 0.008004118170988932, -0.0001596280571877494, 0.01542939012880255], [-0.2201366718024623, -0.004309607128566634, -0.0016887064573369254, -0.0009587006887206359, -1.413669707037364e-05, 0.02140495701246777], [0.7275006317495593, 0.013231018394655455, 0.002855460147658829, 0.0058243485135187015, -2.1913766818298497e-05, 0.02956878829475733], [0.5551302155475565, 0.00956747352927209, 0.0060445338153981955, 0.007488794612809248, 0.0068048351359814435, 0.1852141473589829], [-0.17369572625112087, 0.0015880912342998187, -0.004344617163060148, -0.0028006383581203713, -0.001559514612781465, -0.030810656864720457], [0.6812740537412907, 0.02855849603476552, 0.002802172140192763, 0.012042832289253552, 0.003241922905474664, 0.04858052288902224], [-0.1857719992707667, 0.007502768742927721, -0.0014995042264246833, 0.003123710552674184, -0.003377751707451385, -0.031600286106462816], [-0.22127663843557804, -0.0054131956853478115, -0.0053521328283286235, -0.004138914104466551, -6.597890512351823e-05, 0.024580193292178254], [-0.15346735823519975, -0.015170357617661393, -0.0035264165468709818, -0.003061644134376037, -0.0016858885356402411, -0.034711396945754855], [-0.22111688902761048, -0.0012332694885239688, -0.0023135070010845177, -0.0021552842803649968, -0.001906046559242155, 0.019558329690159516], [-0.1755180434881799, 0.0023923617995258617, -0.005720050552017703, -0.004065929393050118, -0.0016170629040906208, -0.024637942128854087], [-0.188619967767498, 0.009236596126560779, 0.0001238978145211287, -0.0004318569187383881, -0.0013820836472073124, -0.028307122706198545], [-0.21631470288410293, -0.004101811854423699, -0.006137057426213713, -0.003753296435100688, -0.0031671147743739347, 0.026807316707548625], [0.5721192637341589, -0.00995637129434618, 0.0014374637216574226, 0.010299210449263746, 0.0035966585672846524, 0.1156228224410299], [0.7638110314158124, 0.026263339779106283, 0.009214908874063938, -0.003712658276889394, 0.0011358941360819369, -0.033896498612157655], [0.728607259382575, -0.0006784135662818891, 0.006712332027557379, 0.008364104249077808, 0.0007418015985805426, 0.03833624964182353], [-0.22037511566041096, -0.0064663291723285185, -0.00034982519865367717, 0.0007055416279615298, -0.0031705776830543006, 0.02173963941981927], [-0.217351571645933, 0.005771617066050794, -0.00560032011315667, -0.004378911428425354, 0.0019652962375835727, 0.012927223217214286], [-0.22717020646145036, 0.0006348691739185509, -0.004505239502250215, -0.0001800497984819484, 0.0006529101430570791, 0.02074720362469376], [-0.17590889329060438, 0.003009156983681771, -0.004887117888961224, -0.0009242776774253259, 0.0009542886624936659, -0.03284818368366335], [-0.16084195952277533, -0.014698693793352488, -0.0030485781787073342, 0.006864255366139105, -0.001069659362155577, -0.033872031175814894], [-0.21217943417252982, -0.0066662851669505654, -0.0022599367795996337, -0.0032869490635555987, -0.0018870721390208324, 0.01761301065499021], [-0.2184782522595666, -0.0002069057588066947, -0.0019556805083959946, 0.00037789078173978215, -0.0015295228818562939, 0.022125803960219084], [-0.22005261777702792, -0.004069958830730117, -0.0040839061763435994, -0.002477867558793894, -0.0016015791472640494, 0.024904977109207402], [0.17010512335814298, -0.0635549965151853, -0.005290479482250016, -0.014299385762027158, -0.009449179204373591, -0.2592770015526678], [-0.22606960232336634, -0.0011031713818996072, -0.0041029030740754304, -0.0003505192942064043, -0.00018963929778738673, 0.02300727681277674], [-0.22382808445335345, 0.001659881336650804, -0.005807170489157889, -0.003408106351942378, -0.0025590678915173052, 0.022275881182653817], [-0.21760097330919914, -0.0034523649976522723, 0.00021464167654349078, -0.002704432607384344, -0.0026324088374203767, 0.02112651846726948], [-0.17513732818910832, 0.00035062055800739105, -0.005700510753864831, 0.0023909155839607526, -0.002705787103511122, -0.03082097211098746], [-0.22923454842036145, 0.005270813260851123, -0.0008336927557168448, 0.0021474531106679836, -0.004231443143618508, 0.016214751281510748], [0.7495508928268987, 0.02240913310309802, -0.002560671038843887, 0.008743224921389053, 0.0055429538689786345, -0.026907755903744734], [-0.12892245902701172, -0.021455866845530448, -0.002817414979775095, -0.0008204845572548576, 0.0004979351179966447, -0.058104771723928306], [-0.18418644895493838, -0.003156231891425702, -0.006568398948711523, -0.0026153408829276787, -0.002541095287838689, -0.010932484034157948], [-0.2253013705035368, 0.0033172405100937775, -0.0016172782031627852, -0.000998842725168563, -0.0011516981476172407, 0.018649235891096946], [-0.18040929955003127, 0.0013930236407253872, -0.001639422165243849, -0.0007583720807381586, 5.1652696559538415e-05, -0.029776773589033342], [-0.2073072464102196, -0.003963460804118056, -0.00435064631318414, -0.0036355760046094844, -0.003419163040858718, 0.017009425906323615], [0.737330910187665, 0.005465146881717591, 0.0028774488681935708, 0.005050519559798585, 0.002551391916215289, 0.029819820681647333], [0.7090616825180648, 0.008052807949377964, -0.0049398743285825235, -0.005438031474523975, -0.005218428602366514, 0.04648184393802978], [-0.21531302911291988, -0.005273215789541731, -0.0023432224391699108, -0.004783112699620464, -0.00284382014051208, 0.018889733515097703], [-0.20835279350076433, -0.007703928438675455, -0.005321950509873686, 0.003971283758514366, -0.0034284687616108795, 0.022294190785743895], [-0.21426184358251363, -0.004255626543435064, -0.0007478210258128375, -0.0014576798292531645, -0.0002409431737933526, 0.023118191250261703], [-0.2287752197398398, 0.0016434360322428137, -0.001638572253848909, -0.002083999495094648, -0.0019172604816606444, 0.021104949271534747], [-0.1063073322953402, -0.010611850960893073, -0.004569058117874615, -0.00401244680169838, -0.00559446008952876, -0.0722381850679983], [-0.16713355771752764, -0.0044119625701175445, -0.004418898351657997, -0.0026469586386098994, 0.00023982515203206447, -0.03125150988962288], [-0.15392458638012052, -0.014606836670964803, -0.002401903461796506, -0.0035204980106456367, 0.0006560662022434542, -0.0353253036942195], [-0.17722635793575606, -0.009110781369130001, -0.005178916728919962, 0.0003569455495677125, -0.0012714630104171147, -0.017192488520848336], [-0.22083960583044687, 0.0030271402584381535, -0.005719547853201938, -0.0038515030009028204, -0.0010646828439466482, 0.01711486593672722], [0.6288521708785583, -0.037810556915850925, 0.002233789684692357, 0.02337051533878676, 0.027973656289321587, -0.04064536892630043], [0.6620443555219011, 0.0030797582254857158, 0.004194412202128041, 0.010091675639082219, 0.049149233815984045, 0.04525700603685844], [-0.21333025151964694, -0.008154735334024846, -0.004640434210844831, -0.0036122962699021797, -0.006027665401616489, 0.024098716069369128], [-0.22629690536785094, 0.005580424136145409, -0.00017011026061556188, -4.2260851909729696e-05, -0.003223675946204739, 0.014152528290435387], [-0.21688227434453317, -0.008578392969615458, -0.0031713125666301715, 0.002520138524130798, -0.00046772553751720305, 0.019912900227498823], [-0.1637887663829153, -0.0020046971769476347, -0.0065966686758292216, 0.0021439798556860527, 0.0009671799927800807, 0.0080706390538927], [0.7233306777037728, -0.021404158006378862, -0.0011447296048910135, -0.006371536620350012, 0.0013907998411248836, 0.05342513716291189], [-0.1491534947408732, -0.016133797835216266, -0.003519851762641066, -0.0032632966244305274, -0.0017696415935764584, -0.03528297945876605], [-0.21895852733793145, -0.005432625046202096, -0.002449219131255247, -0.004117335050320184, -0.0018017865563796406, 0.021092826455422185], [-0.22232073681589015, 0.00949926799645549, -0.0025124904155703963, -0.0028907304873828513, -0.0013969564306948274, 0.02587164615308227], [-0.16717806817170464, -0.004791993755170717, -0.0043907529553871534, -0.003713896602426724, -0.002012861559417756, -0.02953548897139674], [-0.2322339063927538, 0.0030464482569731574, -0.0030087610831071947, -0.0003421418688837454, 0.0003420272191626377, 0.021343620690314064], [-0.22562371124773078, -0.003600685356595399, -0.0011135006227338709, 0.0019929622695704355, -0.00027353883360551083, 0.017951807124428537], [-0.22741812859128682, 0.010074126636155267, -0.0023348756195287227, 0.005536786754624977, -0.00373952155219201, 0.016214945705560296], [0.6145150909753849, 0.009171930864231664, 0.005408445209352267, 0.003761854569162173, 0.0036221011818096728, 0.1356039105333935], [0.7388880868551687, 0.011195823312858448, 0.0025197088880005323, -0.003156731580433347, 0.0026901256469394055, 0.03432497518155953], [0.743556399150349, 0.02142004999670574, 0.008854778394475295, -0.007266810657532077, 0.00520345694488479, -0.014737570798579371], [0.1849217069909439, -0.0030461606706293667, 0.0010238848789819038, -0.015664362498463546, -0.0037246423996691266, -0.26876985195303144], [-0.17160989530204868, 0.001460042572113643, -0.00218965739602671, -0.004010678288170748, -0.002977542587348383, -0.032338935665185566], [-0.2190410263678444, -0.004824582010664185, 0.00015414604178576717, 0.0023151669301991165, -0.002422696849368847, 0.021402325589225824], [-0.1763925218500513, 0.01801863880109411, -0.00444457742555008, -0.009678237513047278, -0.003961853500513078, -0.032708115178598694], [-0.22050147235637862, 0.022661770440437275, -0.009098932347618842, 0.0005722161250903338, -0.007353139368272171, 0.024052890840075124], [-0.1621070960543985, 0.0065590452460659735, -0.007583064032532232, -0.0050111651707190836, -0.004735700339530255, -0.012542938016232828], [-0.19010156246343382, 0.001910878932127962, -0.002333542024927313, -0.0008486283357552276, 0.002883224138361579, -0.011510370246372637], [-0.16087461408335152, -0.005511982460009283, -0.006796005146543865, -0.0017101666947195926, 0.0002221869401169911, -0.033968609603254485], [-0.22283008861128917, 0.003061189474970224, -0.004780976007829158, 0.004765912913736057, -0.004343393023113952, 0.017460688586859344], [0.5625332085759567, 0.016676621020379565, 0.003014471852322436, -0.005677359404133039, 0.00583068190092426, 0.18620570938788247], [-0.2183955639214061, -0.0040058731026662315, -0.006082551935533725, -0.0031984496477507588, 6.631441894490066e-05, 0.02179561136789915], [-0.22990736234095088, 0.003197607051592944, -0.002489735787059477, -0.00294205177598007, 0.00044952642151952074, 0.022108683097544646], [0.7589138707392762, -0.01310335982861157, 0.003572329601370865, 0.004206618982178118, 0.003011270314189393, -0.03441025361792719], [0.4306847175073138, -0.05698717633301009, -0.004042091506908115, -0.026702965188465926, -0.025251693814388753, -0.20362147172766337], [0.4085248164133642, -0.09782441160060153, -0.057771672601978685, -0.01519249920260474, -0.037045277738269565, -0.2038417950631915], [-0.1754825361048663, 6.929208932238441e-05, -0.0005049604533023777, -0.0021561252266809375, -0.003245100989261975, -0.02830363133071461], [-0.20930693734362596, -0.005974442424185272, -0.005009284815438602, -0.002692647583639383, -0.0029511846277580386, 0.022878941239092194], [-0.1640599654741305, -0.009798456041229281, -0.0025304744617710086, 0.001570766452360709, -0.0045724757776127185, -0.03223245671312091], [-0.21510925795720293, 0.00264709105039322, -0.004817436867062479, -0.005390605743075334, -0.0027222939840649274, 0.01872583683434545], [0.18440284699030537, -0.030650155646195377, -0.0018386706461289612, -0.011603208329765368, -0.012037724156916675, -0.26605289625854905], [-0.21819002053098258, 0.005203151296379822, -0.001677238387732932, -0.00609565411483897, -0.0009273292854379603, 0.01502042435594626], [-0.20960222625710956, -0.003795308417078197, -0.004510409163888171, -0.0033390629173485997, 0.00020552958853912523, 0.01687481050021913], [-0.21886600216431262, -0.003249663259787485, -0.005095721978127389, -0.0016388523450192483, -0.004983161321751234, 0.022166734402331398], [0.4826341192622492, 0.005636780279898832, 0.0019604203545505842, 0.004404298573482559, 0.007547040769082538, 0.20006403224129848], [-0.20009628816778582, 0.004198084075243394, 0.0027109356822590065, -0.002251460779829018, -0.0020052897798670793, -0.005055981030020441], [-0.23071532925302385, 0.0043938476279596315, -9.63947995907038e-06, 0.0025076627767409232, -0.0025885784364891628, 0.017412036764771124], [-0.19450858460110157, 0.003803839579952907, -0.00149060606923503, -0.003124804697392501, -0.0031318777982351715, -0.001839633080655109], [-0.2223694202505672, -0.0025293934049609454, -0.002891310007204737, 0.0018661708653833675, -0.004212700051908522, 0.019078094290699905], [0.4862621456742505, 0.14708166203695255, 0.009343821568205478, 0.02623172552022922, 0.0123688895374001, -0.13223214765594174], [0.3995048638956595, -0.1505288836916064, -0.024148662423588933, -0.023637927130875046, -0.029214143852989132, -0.1907054516692468], [0.6965221088123928, 0.01457364467877383, 0.0047743840557749676, -0.010181724474436002, -0.0006123851104026278, 0.0444954006093244], [-0.22477636529338718, 0.0004295409960851883, -0.0025936315571060686, -0.0009366220801081689, -0.003248695084436031, 0.019942977320027592], [-0.21909185030352904, -0.003032180154389642, -0.0061241113404406885, 0.0013125089185458524, -0.003482681718549618, 0.024359756039804475], [-0.217936144681997, -0.006577196902057989, -0.004472440206833787, -0.0020914712349943297, -0.0027359833417689846, 0.02273480499510332], [-0.21134423641526726, -0.0001373480217658046, -0.005689261549965061, -0.005852633559658876, -0.006737314272793421, 0.01809412715278439], [-0.23276100185086235, -0.009197499964771991, 0.08476809199335997, -0.0036500469066551803, -0.00616391411135339, 0.00583770417361582], [-0.20463655884344112, 0.012229191802902984, -0.0004224893558446646, 0.016046708516424354, -0.003026699438431559, -0.011559200300657333], [-0.179029137806973, 0.005353118360757575, -0.005435898545903416, -0.0005074032563147713, -0.0020904935915055317, -0.02889521205453921], [-0.22284394348033437, -0.007823857699667409, 0.019016914246205475, -0.00368596521478693, -0.003827392307018752, 0.016247577788935187], [-0.16001027400957382, -0.0078109371904407825, -0.003415854884632378, 0.0032671505532990645, -0.0018988604392595346, -0.03508761937822963], [0.7542277312831952, 0.03340833222633818, 0.007996019411117605, 0.006131623986515813, 0.0056433577104926585, -0.03151778722838384], [-0.21370884851815544, -0.011635505175579304, 0.03343262033783633, -0.0033244290886094846, -0.0056379452820497795, 0.01812410772655764], [0.23185404659936715, -0.021642570142421883, -0.008963287350936282, -0.015135812104723586, -0.013131155950725849, -0.1532609829553219], [-0.22156224462309093, 0.0055534928020561855, -0.004927647146173878, -0.002241434140854178, -0.0008135021072897862, 0.0163246685486858], [-0.20882889908371308, -0.0073537505139317895, -0.0040063591560024325, -0.005521494157753819, -0.0034808205652992557, 0.017524656810034008], [-0.18363631926859475, 0.002455405119249493, -0.000131529327449185, 0.0006860064538752812, 0.0006908400240691294, -0.028983383384000434], [-0.18368166338897238, 0.003392274961275387, -0.0047967565912322395, 0.0032554210757939324, 0.0004502197234686798, -0.029224522674812063], [-0.1757639822872545, -0.008301609536536664, -0.003411597443525684, -3.9249595028777474e-05, -0.005367742326795119, -0.014572214159696499], [-0.21938399887391435, 0.005354475014890221, -0.0016785219302936137, -0.008112236923675166, -0.0031278044211194003, 0.01528142046744581], [-0.19834453742357472, 0.01095815826280144, -0.002410942334884773, 0.01512672046855135, -0.0016182776984026068, -0.025615883179252393], [0.7209903294900422, 0.015545463402264995, -0.002516990778929173, 0.003098919789139881, 0.0076767049781830965, 0.03847311697894753], [-0.2142994116309571, -0.005287036812482744, -0.00340189095974697, -0.002903170642032981, 0.003046610979065238, 0.025356319018751145], [0.6243369741497721, 0.018002320511185377, 0.0049220782992319394, 0.005498839105329593, 0.0006141760928969242, 0.12912561184158391], [-0.21646755282692784, -0.01016027141328423, -0.001775195561889951, -0.0010830936502266597, -0.0027516453649832364, 0.023071092150645307], [-0.1545782438225665, -0.016164880900751556, 0.022828950722448737, -0.00190324881750582, 0.0011324647956398983, -0.03158885150107387], [-0.16440045359194116, -0.005758741489907385, -0.00029997481452905407, -0.0031133483098363854, 0.002700691626772397, -0.03241790210272881], [-0.13895387042826848, -0.010511678706152588, -0.0033772243464862068, 0.0014733897114915769, 0.0039062373238613075, -0.056659915569949405], [0.5993818954347351, -0.001515255696230794, 0.005621510649401013, -0.008358499206061607, 1.0526219135352822e-05, 0.16538760037679895], [-0.18173016619475943, -0.00439874425263449, -0.004845356185505973, -0.004843433025515962, 0.00018581633110324253, -0.01599117868819136], [0.5041132300835055, 0.027134831680561907, 0.10290815789874824, 0.02518406802312941, 0.012558220255683462, -0.0045016825448033525], [-0.2238329123285169, 0.00039251399651226446, -0.00034130660299262305, 0.0011318277866101247, -0.00038366640974401417, 0.022030235038693564], [-0.21825540178572098, 0.005834833315666006, -0.0018202238936809198, -0.0024372684381052012, -0.0038711471806936195, 0.011977779411105734], [-0.21922230937188275, 0.013904773822619789, -0.0041512165279964, -0.0010943767237157256, -0.0053789874889941, 0.021775449623302544], [0.7400122616448582, -0.016668521686906874, 0.00890981054837379, 0.019910006272349488, 0.007789285240194332, -0.03759173090775882], [0.7350756389864856, 0.0013812788531523807, 0.0016655280929801107, -0.004493258320063568, 0.0034095391479235975, 0.039294606572853615], [0.7649686449555573, 0.018561451180808486, 0.007972255831769364, 0.0068635269763188305, 0.0016892756718001705, -0.03110818491928528], [-0.16039936358647058, -0.008190107812516306, -0.00266298698751399, 0.002353339038105764, -0.007251584105672459, -0.035472358561436185], [-0.21160617870578577, 0.0011117298082074162, -0.007561134107432022, -0.006625628362054765, -0.00489825281254204, 0.017912797512940837], [-0.18130415462847582, 0.01054265182912563, -0.005114792725949458, -0.0008589165935355309, -0.0032563087588227783, -0.02913154113784593], [-0.22251839668023038, -0.0007215160303798006, -0.003741914135328348, 0.0010450303867029062, -0.00338268379793787, 0.020986146923840564], [-0.21371556542417255, -0.01094271148777612, -0.003221380454999273, 0.0034154016045968576, -0.001985423376960394, 0.023900659531468322], [-0.18484013493633705, 0.016665872252657893, -0.006616198548692503, 0.000930948835430424, 0.00047923801352055555, -0.028250677997531625], [-0.18411920993842412, 0.020827970550650745, 5.0898119881512384e-05, -0.003903741746660159, -0.0020380500818132524, -0.03444881928458702], [0.5683079557286684, 0.028583665212030266, 0.001004591727884816, 0.007789548692303331, -0.0048873211807082045, 0.09909441696267879], [-0.22133896704566164, -0.006730253526113691, -0.0020675005064123226, 9.202409815308337e-05, -0.00011013683203154615, 0.024084320991553197], [-0.22595219880960452, 0.00615540677741508, -0.0027329301274233533, -0.005083666863783345, -0.0030886017670079253, 0.02028532412373759], [-0.23225047168956883, 0.008807224693044355, -0.001784010067671209, -0.0006663595705778931, -0.0029168468525675737, 0.021727130154007325], [0.6770428584950291, -0.0024096581239979493, 0.006938516141725328, 0.002979267388384596, -0.003750710152810782, 0.08933861514055813], [-0.15464262986134272, -0.011097903484759983, 0.001777345925643189, -0.0021684494672105366, 0.00019341668134876275, -0.033351508475849015], [-0.2289076361221652, 0.005586964894011059, -0.0022589714930727456, -0.001309242201970585, -3.62219826718321e-05, 0.019175106905869543], [-0.19054074853646621, 0.013189034826449652, -0.002479684425596709, 0.000882195590017982, -0.0006521340065799551, -0.028970092019253234], [0.5945811044395716, -0.00705668101037977, 0.005354209101051547, 0.016086768049797542, 0.05506849774272463, 0.1071903080264412], [-0.2203476235643632, -0.00963330061844167, -0.001099820712666407, 0.003963578089292377, -0.0026820041732615574, 0.021893008514454464], [-0.22200650870949826, 0.0023398076178020584, -0.0021063842488755675, -0.0011936830435730326, -0.003611519632499522, 0.01935606579442225], [0.591563399241184, 0.13422085262801106, 0.016164179889642118, -0.01161580765470664, -0.008058785333759437, -0.08556496431149785], [-0.22104239898960917, -0.007676284046148013, -0.0024245501849769063, -0.0018828733368488318, -0.003402157521126443, 0.025349832706160418], [-0.21703983788230954, -0.005347658630132543, -0.0031274728944219216, -0.0038368211736963197, 0.001562947033509789, 0.019455510213717615], [-0.15102635080111151, -0.016416025291314293, -0.0019787073166637244, -0.0034586169061496634, -0.0015544758329093407, -0.03723249051851799], [0.7235590882562856, 0.002182968919085632, -0.004865961428982266, 0.0004462923723861016, 0.0034419332023084906, 0.03435472629796398], [-0.21922626783827542, -0.004773196470728576, -0.002796607138279531, 0.0018838567699272133, -0.0030713054374971027, 0.020066853448186917], [0.6374504216718088, 0.011314170011767346, 0.07731326884667297, -0.005152544341842195, 0.01115925011280547, -0.040882185348830966], [-0.16006275625236976, -0.013862651609176782, 0.0010455556502635326, 0.0022249318614570614, -0.0011950249626971087, -0.031435768973191155], [-0.17600767956779229, 0.0019567181730947457, -4.784868506844181e-05, 0.0012341214091970102, 0.0013564321449798938, -0.03243010370222276], [-0.17299620309984381, 0.006776231074463503, -0.006035150124334811, -0.0032093694562742242, -0.007473284618250465, -0.02868528579126379], [-0.16294615735526083, -0.015585194419827314, -0.0013129079694246435, -0.0006627796685716581, 0.00045545110593604455, -0.03157147370835524], [-0.16973335613709417, 0.0019040680644846406, -0.008893706150638397, 9.131860514484493e-05, -0.0005565609193461426, -0.01114509679588407], [-0.19513859413380027, 0.01598157702611852, 0.0007440499856215867, -0.006155740924446369, -0.0002953934377779724, -0.009302565182381695], [-0.22008726419677058, -0.0030511182040891495, -0.003940825201569478, -0.0009125548161258913, -0.00282722808028953, 0.0206361947999199], [-0.23054502994582893, 0.0002442866265265676, -0.0014108246916347202, -0.001156169563860098, -0.0011358545032477112, 0.02233692541137829], [-0.22554606630186524, -0.0007940271859624853, -0.0023049415480429084, -0.00036869095314965065, -0.004527778611433366, 0.022482946041894922], [-0.218366372741097, -0.0026515944470451394, -0.0015294264722526564, -0.0034678691671196226, -0.0023784127115930405, 0.0208446559312647], [-0.2302166328012098, 0.0028610341432286314, -0.002960107287042242, -0.0010136301693365103, -0.0022891134633804804, 0.02195178291107333], [-0.17137024837836978, -0.00035052710369605456, -0.002648397710686575, -0.0007490247906447886, -0.001514898292390657, -0.032728060977811266], [-0.2263563980924527, 4.914797747848918e-05, -0.0010594491630430698, -0.000709079666340165, -0.001899490039889387, 0.019625806618655355], [-0.22670994921480953, 0.0017337415480952487, -0.0016549610412375146, 0.004503815342019054, -0.0040483748654661245, 0.01550906156473208], [0.7287878884566228, 0.006988248615203177, -0.009586753925926058, 0.00339862683041695, 0.0023354246614526097, 0.028826565362231393], [-0.1613565977907686, -0.007782945337911972, -0.004763671247037204, 0.002735618113059803, -0.0015246535280614014, -0.03705581222478389], [-0.15195930348397807, -0.013087715022258419, 0.0011577452066895975, 0.0007379224997063249, 0.00015635086316348568, -0.03404472874549345], [-0.17239491423106743, -0.007429831463419646, -0.0037833882358290546, 0.004883743976694638, -0.0029889995503089395, -0.02490967251157339], [0.7174802215343774, 0.013866629534016125, 0.0010647387195660803, 0.004244875415423572, 0.0005580264786926703, 0.03218431784173173], [-0.21572031499549038, -0.000703257418838416, -0.007342517408054859, -0.0017647457992857887, -0.006402110494698051, 0.021516279449701105], [-0.22628728541160864, -0.00025452818130598033, -0.0012760038725709352, -0.0003568288210239866, -0.0021385805316925213, 0.019254668259643126], [0.7071014345653339, 0.0016099950162922713, -0.007077107899630267, -0.009141129404304855, 0.0057760361659424594, 0.04923077155636498], [-0.17062091152975484, -0.00353255637583597, -0.0040851842788530305, -0.0028791368900798354, -0.0023103099428448564, -0.02444496299813534], [-0.157370115153159, -0.012270081162199653, -0.001320050308841533, -0.0009210164119232896, -0.0013484388353910285, -0.033393360143989075], [0.17834103887950228, -0.021407202625781573, -0.0030134763887964713, -0.013688210250471728, -0.009982260319287187, -0.2794562846440029], [0.26142954437050764, -0.08374076251094065, -0.00024291212289083208, -0.016248741780263646, -0.017277724065181098, -0.23130630865313603], [0.6401781973608736, 0.05563001038962412, 0.020739482686539072, 0.00981467050622528, 0.0019074443025044927, -0.05447417032513164], [-0.224797243596818, -0.0035711092459552794, -0.0014816854508263777, 0.0036664895812018317, 0.000203847164575574, 0.021643059695051415], [-0.22175041466833506, -0.005250629394724286, -0.0022406412492571855, -0.001401603048542901, -0.002237187835662103, 0.026213809529855057], [-0.2294647695407206, 0.003004029178948198, -0.002633262407551407, 0.0026735647888921753, -0.00024351301220482864, 0.022061237814341886], [0.685580206306531, 0.021659113389935505, 0.005246559789171725, 0.0047307198591779115, 0.004682404821031305, 0.05641494932252422], [-0.18320228791426552, 0.0065625003493915415, 0.000584573305158937, 0.001221498829603076, -0.002577960418939794, -0.025187860089354766], [-0.22565176685323232, 0.002849926723323671, -0.0021874163099824817, -0.0034307862562745818, 0.006536315454094306, 0.023467060575405024], [-0.16643594529316866, -0.009172376200976846, -0.0012000541177564735, 0.004507269046930564, -0.0006387445614596783, -0.033683210889072494], [-0.22582566307882052, 0.01395475055778163, 0.018357387512155012, 0.0006332689390465144, 0.0016490544233585622, 0.0157312016464785], [-0.21643592000371806, 0.007551242554941935, -0.003564647505804125, -0.004680083088910003, 0.0024770769281110606, 0.015485664448712542], [-0.21755114587985638, -0.009524145033270969, -0.0027983479047131877, -0.0012553258058713337, -0.0015902895307180116, 0.02146925415442969], [0.12390951088391117, -0.005035528774851334, -0.011591625559322197, -0.014819000017102443, -0.007411573193217644, -0.2537184500060846], [0.5947202415890116, -0.0022148437405901267, -0.0002631898350453892, -0.004848694674290092, 0.005858201960133464, 0.16902606247855906], [-0.22128842583695418, 0.003983200273849262, 0.006398699295616831, -0.0013277191352783376, -0.004646734085025931, 0.01750597948779254], [-0.16567288115643156, 0.002246170908390613, -0.003698695786792449, -0.0009840306674678092, -0.0037535864524970023, -0.030260038860705486], [-0.2264091601164168, 0.0029909082618167007, -0.002104017323500689, -0.000997173677496985, -2.1142256772912274e-05, 0.019021205267408962], [0.7560474219085164, 0.020987921169156033, 0.011103578417945018, 0.0007906388449329302, 0.005043147416881821, -0.025831293616018904], [-0.1572662370332223, -0.015142742884393006, -0.0009075029992686405, -0.0035380671939332785, 0.000422199055020175, -0.03269071095970642], [-0.21847996537531006, -0.008295141014598264, -0.00165782005005678, -0.00114424345989219, -0.0011990003235633952, 0.02364381728224431], [-0.1787563977403208, 0.00039806885372429667, -0.0036533882276970553, 0.0013525249751419077, 0.0015630224166580046, -0.0291935589596768], [0.5461418891211695, -0.0020697681279731824, 0.002891667851580462, 0.002404638807587876, 0.0012584262316438233, 0.1938681094859539], [-0.21989617603167758, -0.009367694475435713, -0.003451340935772402, -0.0025601358045806433, -0.001524864749009847, 0.026383545329809608], [0.7476770759967711, -0.0002887167253257931, 0.006800009236397555, 0.007163354736605267, 0.003123730310902574, -0.04421950117439851], [-0.21771743726770781, -0.007071762629127888, -0.00354427102591969, -0.0012628102803574077, -0.001864028029905488, 0.019793642566351848], [-0.21954013400260236, -0.005905760363789111, -0.0037515279512220718, 0.0008240466004953194, -0.0008953910600624175, 0.021210208218622514], [-0.2133988868788971, -0.005640346646057893, -0.0019007450446928602, -0.001253168203315832, -0.002325368466587113, 0.027440083867002082], [-0.21809438235107306, 0.0007853560830866756, -0.002311495480864112, -0.0024505057103198498, -0.007432754721606957, 0.02350378218077755], [0.13877287430943272, -0.0027363939447440084, -0.008886198957419371, -0.001716259032734942, 0.0026903097047171654, -0.24209858457094569], [-0.1683039820559939, -0.008983867796164785, 0.0006255677305009402, 0.0077457999877961385, -0.0042727627141213465, -0.03014408848534999], [-0.1713550507188151, 0.004712983768961351, -0.0042816585064894795, -0.005476071483774798, -0.000455524358830077, -0.03260726373506494], [-0.18733822796661082, 0.004102451548837397, -0.007550231783406115, -0.003707463146670112, -0.0009668103206599225, -0.008502303365504142], [0.6397257922503002, 0.014730488755001553, 0.004574481984590675, 0.005871661480839234, 0.0044589165666152, 0.06537675420074807], [-0.1604282473823795, -0.012290336412287894, -0.002310043211442749, -0.0034292589749859675, -0.001036426788254167, -0.030922353897316256], [-0.1921125250881344, 0.013501557554938646, 0.0005753095428950006, -0.0003757462430472708, 0.002736616399521484, -0.027082687249229617], [-0.16440244959099717, -0.0027319050730503854, -0.0022287156312912554, 0.0004733956988671969, -0.0015612196945908252, -0.034215772375604345], [0.6556676299212757, -0.018096338868816106, 0.0018711907485589225, 0.008574332280537793, 0.003806665172924795, 0.11197670748688125], [-0.21775782338180683, -0.00495677176467211, -0.0016225588799844448, -0.0013881585595567512, -0.0036136319597558455, 0.022116722323553894], [-0.18606455509174274, 0.006121174494475988, -0.004461829545996642, -0.00396804177106018, 0.00013483826030800722, -0.02338464836148819], [-0.22743041554290935, 0.006358515148093649, -0.0018307872941394716, 0.0009398754522222308, -0.0024187247303715286, 0.01604820363377096], [-0.16427667757691647, -0.009464578575599413, -0.00643132462694967, -0.0013955325303408113, -0.0005779950120459564, -0.025310287026984513], [-0.1642889332467821, -0.005608248752856456, -0.0020193868873062825, -0.0012065050226630633, -0.0035592267401153913, -0.03433265325767259], [-0.1584799832643683, -0.008224224102929048, -0.0024422636481441517, 0.003223659949740402, -0.005010584401989758, -0.04006466654781258], [-0.17571322662173983, 0.004837063668118661, -0.0015172986731480027, -0.004860339288638757, -0.0020883577406050347, -0.028038793724939524], [-0.1672952274645048, -0.006022531585893187, -0.0012734629366033814, -0.0012041296100579598, 0.0020905121854888687, -0.02875155593726653], [-0.21277643016054743, -0.005289466370364402, -0.004366328937416631, -0.003945939615510895, -0.004280159505447869, 0.021491657922621178], [-0.22743371328132272, -0.0010125074189694012, -0.0019823079433833823, 0.002718106746931963, -0.0026859959117567483, 0.020158322570405122], [-0.16491329390627604, -0.0072809354876114995, -0.0009475740769996917, -0.0013536131303193795, 0.0004927354850321452, -0.03428704756599586], [-0.21519139549156086, -0.00686562761604951, -0.0038922174109936266, -0.000655056660914223, 0.0014417691782784106, 0.018495861334573427], [-0.22507537596860122, 0.0025874254546563846, -0.004810009628505281, 0.0049314760407683615, -0.0026913157622754365, 0.015057799863957343], [-0.21494144158807654, -0.007126720741404844, 4.620977369676177e-06, 0.0034894796287481588, -0.002073075140811002, 0.023147136864174862], [0.3571209427062536, -0.14547136218746573, -0.0012318086400932788, -0.009939759891434904, 0.009890252601579585, -0.21573949510251203], [-0.19373365065405057, -0.0002677976048654528, -0.00306795950386682, -0.0006898587080862214, -0.00047804291178033486, -0.004012690617350462], [-0.1848584522016502, 0.012110159482228799, 0.0007231488589467711, -6.917395805420057e-05, -0.008330291341887505, -0.026242057506250582], [-0.22719375188143096, 0.0029691286023584154, -0.001169881480942998, -0.0011827347175316378, 0.00023294249934888332, 0.018844296978197995], [-0.18606328204827502, 0.012562097231649099, -0.004460776897351098, -0.00033065973427702814, -0.0027002668599446834, -0.030673778358468184], [0.6764715922565031, -0.026555453518706133, -0.000637437009329609, -0.0047478716883618925, 0.004545513667362669, 0.10021532295919697], [-0.2112011707321193, -0.0026719680331986624, -0.007431468795496001, 0.015140778873077281, -0.005796689667129095, 0.014460518354865821], [-0.18239561798380277, 0.0044693223899865625, -0.0006340558099282341, -0.0006887100904024026, -0.0019150960836666047, -0.029440869316665175], [-0.21691384791315413, 0.003903019699425479, -0.004913994077394192, -0.007061041655658326, -0.0005064480200946149, 0.016325645300209503], [-0.22630989498594575, 0.002821701781657516, -0.0013655516379319271, -0.0030319675194222253, -0.0024476951578701975, 0.018666740852845986], [0.7218449320896523, 0.02606067570337641, 0.002035354958423318, -0.0007690816312854298, 0.007178868415479846, -0.012647827457725767], [0.6145002909585286, 0.01912071420896099, 0.019884601915751268, 0.0035764952513785233, 0.013765825137834989, -0.074702545077073], [0.5562076933696837, 0.01300244623300295, 0.00018760259488280256, 0.0069693559905186, 0.010745789885344604, 0.19305377859323306], [-0.2194784405185112, -0.01097403399502572, -0.0015216803148010425, 0.003006201558426638, -0.003217297886601773, 0.023435251156513], [-0.22470316439624566, 0.0028301015182713245, -0.004449082140761633, -0.0011733596958395881, -0.001987303889331661, 0.01781614193724079], [-0.18424834495089648, 0.01095327971556146, 0.012400611346399252, -0.0021740097888395815, -0.0036346741443459727, -0.02913019551121197], [-0.18434975300338488, 0.013167891010263633, -0.0004965284411388721, -0.006573556132927656, -0.002705414223155561, -0.026264861431878676], [-0.2301156938535103, 0.0003426839079301868, -0.0015108126856504217, -0.00224366360520976, -0.0003591272720151096, 0.024066100687942505], [-0.22641841600311438, 0.002949720505620592, -0.0012896692413785116, -0.003865301236155519, -0.001905834415100764, 0.018862833723461966], [-0.2225525028487675, 0.0015855712588926576, -0.0043936454277403725, -0.002453227930838915, -0.0026040645684605452, 0.02075120285024823], [-0.1742417682768341, 0.007732549513041091, 0.005804446014315345, -0.002813013084847027, -0.0025599599935812625, -0.027884839206107292], [-0.152195025802145, -0.006879994012644698, -0.0037452965476181947, -0.002907502350652641, -0.0015835072634853605, -0.03868673603895786], [-0.19451595461989007, 0.0060123917848606675, -0.004434405405214676, -0.00357689168041516, -0.0021372596644012856, -0.004477132115619788], [-0.18525312749744355, 0.00372333426883013, -0.005919738560194478, -0.0024287908150026154, 0.0009512637940192363, -0.014196003205712356], [-0.15818973189126234, -0.008608810795062158, -0.0006111111043598285, -0.003236292536266439, 0.0032517345704699057, -0.03244313597330883], [-0.2261348140757378, 0.0033028808597496693, -0.0004231716032511703, -0.0011484323366523113, 0.0006006686589798951, 0.01546953516357854], [-0.21221148361744316, -0.0061612192119045355, -0.004583124085010467, -0.0027657555883055054, -0.005658373071670168, 0.020301524201785458], [0.5612700962133512, -0.019570216442738923, 0.01730639785276103, 0.004249672993052337, 0.00736804348980444, -0.019933517915752794], [-0.16331178105515803, -0.004990217151915722, -0.0021795823555016513, -0.0013431949187476843, -0.006289791272097741, -0.03290038715397499], [-0.22176646882637668, 0.00527388749768502, -0.0016907636938791291, -0.005813592050310765, -0.002594627157985979, 0.016924897564200996], [-0.14853117663179102, -0.012052507943539462, -0.004884150116935656, -0.004206346864598091, -0.006207108300666971, -0.0357417721579725], [-0.216645693494147, -0.006078532255426706, -0.00029730180238594483, 0.0001286606102094081, -0.0031106360257619355, 0.02232948335966902], [-0.1654382718286187, 0.0020483552078699382, -0.0035563887957726485, -0.004061014614894051, -0.0044807929656503955, -0.03326188700293356], [0.6775510008908989, 0.023689619378531642, 0.00409637672078884, -0.011617002688588327, 0.0073842174625409615, 0.057562454902492816], [0.6905409722343157, 0.014772734542257956, 0.0044345713820111874, 0.00287282447548889, 0.006483232930974983, 0.06607515161443955], [-0.16103902312624477, -0.007404857447406336, -0.0008292706033862959, -0.0008608258554618564, -0.0007964511549728824, -0.03423430049469813], [-0.23279891050109655, -6.72714084024548e-05, -0.001583530506640273, 0.007445398550869251, -0.0017435264137307678, 0.017914506945666948], [-0.172522555747913, 0.0007311585308496262, -0.0014526706593583508, -0.0016336076051244868, -0.003974696288920291, -0.032162582136928866], [-0.1581484660209536, -0.009027099912234906, -0.0060537525594259655, 0.0009666123294291498, -0.0012279269051551746, -0.03688242894716278], [-0.15464134953397823, -0.01682914321223204, -0.0027054331755761263, -0.0007204579559096894, -0.002831228014863599, -0.0338954501229437], [-0.17031841204334883, -0.0060290744161573305, -0.0026827703093394084, -0.0033002203753767934, 0.001003862125711544, -0.02904644699699296], [0.6324786630770668, -0.01827004232362146, 0.0034558304857284197, 0.010665424080622977, -0.004817965306802344, 0.14009920109811705], [-0.17624896818014069, 0.0030844109467498154, -0.002891780933542535, 0.0028357368154618883, -0.0015507596332693264, -0.02768503436409631], [-0.21670066232259044, -0.007488294096480948, -0.0020173387802636922, -0.0012902498070405165, -0.0047886312428726515, 0.020618509582581487], [-0.22725023966865024, 0.0020912042815546305, 6.86773105930422e-05, -0.00195345626433064, -0.0014072455886005968, 0.024117726596100505], [0.7369710426513147, -0.00418824539556673, 0.0026918955516789904, 0.0011405392020049413, 0.0038189966979167937, 0.022065771292649965], [-0.2180878739128744, -0.008246272425715354, -0.006123849819402264, -0.0012363046772878947, -0.0009119244320902525, 0.02710622526737023], [-0.2211729482141193, -0.007445678952271343, -0.0014561832151269187, -0.0017012664402148298, -0.00360023797845672, 0.02370964813352241], [-0.1853561405713153, 0.000611383433176644, -0.0064866838123876485, -0.004805595702573441, 0.08686762033671777, -0.025818678921713432], [-0.17851648817967897, 0.0015748262586233275, -0.003087143649623725, -0.0020978048478581543, -0.0017086747595538314, -0.027787776837412452], [-0.2132651140348075, -0.010247166543961071, -0.007812236634884136, 0.0018000425494963173, 2.2880373825420293e-05, 0.020751594290331223], [-0.22066369733675686, -0.000498786880232107, 0.0010036103292979095, -0.003021471532400466, -0.0033050157398062, 0.01981869449323109], [0.7332757062217069, -0.008390441016719263, -0.002508178505098923, -0.005029381125542338, 0.0034647973939889143, 0.04606249703166232], [-0.16115560029344883, -0.0062968945091317055, -0.0017687828134630588, -0.0038282658502982398, -0.002854841185208034, -0.032428948681783316], [-0.22564547523095166, 0.0006648437555512323, -0.0010811080870547073, -0.000565159723090881, -0.0026950663018613205, 0.01813916988848261], [-0.1873461458579795, 0.012011610779764908, -0.0017510031152549142, -0.004096153176437734, -0.002673773367255756, -0.02731120192950364], [0.5402575193256203, 0.01092310853311185, 0.006525313287825438, 0.008981707329974232, 0.004052958686359747, 0.19121772617044158], [0.7498274244971143, 0.010930227886399012, 0.005381385547359312, 0.004631995857130374, 0.0009988157020934011, -0.018245074715322013], [-0.20835037322833333, -0.009165678967026029, -0.0052554304585372535, -0.0010616816715273288, -0.0029530077266863098, 0.021786172052110125], [0.16777768440680127, -0.002863582934670622, -0.018698415836389332, -0.0174049748302804, -0.0013990844912078542, -0.2738144502345187], [0.7153987516775019, -0.023766013835477606, -0.00029726365732850486, 0.003480180987650286, 0.000301576108370292, 0.05238276871928247], [-0.2184135634143891, -0.010070774659616347, 0.01663460799748115, 0.0006468074670746844, -0.0033232858483048786, 0.01774049417204028], [-0.16621180576971792, -0.00714998061181093, -0.0016346154043006267, 0.00039784471560227195, -0.002137544590137521, -0.03238696035513917], [0.7113368456873271, 0.012068223923247487, 0.003973657171858065, 0.0045197866074950434, 0.006596634488640898, 0.03219349232659443], [-0.158731344786169, -0.0026100848147837313, -0.003394079595249811, -6.297398659398204e-05, -0.0019294636243666226, -0.03281178187500732], [0.5562568626353571, -0.002882751395645291, -0.002604748684082435, 0.003589769742487032, 0.006065005330127625, 0.1942782433241376], [0.7257687415410722, -0.004317267729738356, -0.0032826329191862694, 0.002176583609241702, 0.004186450611463194, 0.04326525912545647], [-0.22453852149453413, 0.003378630946760532, -0.004715879585427608, -0.0015526393643283075, -0.0018435186840642316, 0.0176052615149272], [-0.16304149925293746, -0.016847652745396464, 0.00184466312825998, -0.001209960935932978, 0.00046498378777694836, -0.026500262663940642], [0.5517318504682022, 0.011815366171619051, 0.00287879221319802, 0.0024356812732352407, 0.00557791795842429, 0.18939372524865355], [-0.22104171616139087, -0.007194109261937555, -0.0008827587934012185, -0.0019676472560943956, -0.0013643705245173303, 0.020783935330674897], [-0.21373253478295282, -0.0001701200965694984, -0.0034309399907304244, 0.0033251714589076597, -0.007792047207530051, 0.017408578726983256], [0.16377701359811533, -0.006398489839518658, -0.012709795074617771, -0.01982261925615298, -0.014267031022769, -0.2696485689919893], [0.7141760641106011, 0.012368396361323072, 0.014605156317203254, 0.0038509998566260075, 0.0036202466432026623, -0.00982919662229093], [-0.2155297819688527, -0.0028726907627118423, -0.006180168846050917, -0.0043616475348390625, -0.002492722491860106, 0.019770344937647906], [-0.21932198310309547, 0.004300196735647058, 0.0002593021771219956, -0.003189979592745904, -0.00016842519101356858, 0.021870888974085737], [-0.20832429045133347, -0.007556057443580536, 0.01878901610527209, -0.003956452824008003, -0.001678740422357458, 0.018143191702674023], [-0.18063516230977653, -0.004233456679717972, 0.00016948824905770607, -0.004703152678909751, -0.002359878987686871, -0.010321170926299937], [-0.2244970595336357, 0.0013377890543735129, -0.002507859946299857, -0.004357158502776063, -0.0010492974643878737, 0.023156919726059214], [-0.2182492322987503, 0.004291573733738137, -0.0047274112877079355, -0.005241568978735752, -0.0031636630306667193, 0.01542363519545612], [0.7389414599502737, 0.009000413304992952, -0.007475206172451116, 0.002867636363917192, 0.0023856830227700103, 0.03161334686383003], [0.5470520559499502, 0.012099067077783115, -0.003830298726589657, 0.006316556348441265, -0.0006683560248999646, 0.17991986426420326], [-0.18415767074702363, 0.0011164961675822387, -0.0014107485860875531, -0.001104751582234564, -0.0008963885692738089, -0.024686127730724684], [-0.19558344771136693, 0.0019236267430172592, -0.0008122650577790987, -0.005399154113911559, -0.002631561526282184, -0.008330531667010603], [-0.16497566576760142, -0.0048721298832052045, 0.0009804220678694285, -0.003983833519891483, -0.0015554269555188077, -0.03388309462382313], [-0.19912227134016558, 0.007316948068890867, -0.005362865196004419, 0.002504646812594287, -0.0011303989968468658, -0.011872726015134838], [-0.22820138409275914, -0.0013660526071423709, 0.08639628116387266, 0.0018426889102017498, 0.0008026305261032142, -0.0121408305669426], [-0.18110671729351674, 0.015906643693768284, -0.0022570454911225567, -0.005177284838492092, -0.003422067986894803, -0.031860194750408584], [-0.21472852666549042, -0.0009998255541326215, -0.002059761733791774, 0.0005758960532478349, -0.0013685028872136354, 0.02411503451287124], [-0.22026134568792038, 0.0011305112498468373, -0.0047022414089092504, -0.0008726295056113285, -0.005088730545937357, 0.01812776923186504], [0.49380643648843503, -0.10871623864322784, -0.0007106588480290793, -0.00977604917682806, -0.005785829784136017, -0.17135357308956872], [-0.2057539330122768, 0.013588844391371837, -0.0010019125871722101, -0.000438035354017433, -0.002594586522667968, -0.011467043581903954], [-0.1765355200178248, 0.0015479146757953549, -0.00031520473307767883, 0.0012383182284230736, -0.0015616535675978841, -0.030388808493113477], [0.7690981478214987, 0.003607696117707602, 0.006155341182782734, -0.00605569123768735, 0.008343171399846703, -0.04612485576033973], [-0.19448123180629764, 0.004110783290796063, -0.0034766784784863295, -0.0016334199886640338, -0.0026558181235584763, -0.012861696909293317], [0.6645446595584574, 0.09160664890063373, 0.015251645198787116, 0.011581302247417352, 0.015602849220340012, -0.052211564000096085], [-0.2120197926677372, -0.0037439127914413615, -0.0033987028142967706, 0.0057219889261827415, 0.0005402028641843345, 0.030073751281643486], [-0.1519795084615867, -0.017110018682388414, -0.006006348841759864, -0.0016212500732147863, -0.0011720717264146552, -0.033733864230139375], [0.7070629260805552, 0.03739370940761658, -0.003760411010608961, -0.0013321217936282366, 0.0034817993116424317, -0.022000663900337752], [-0.21370413925723183, -0.006650825495871196, -0.001958037224140292, -0.003788901789852494, -0.0027637397995218076, 0.0181989768999511], [-0.2270061690997226, 0.00479873341139724, -0.004729880512891291, 0.0030172670837489107, 0.0012917538686118162, 0.021100517471077745], [-0.19147109467428733, 0.01158734652860447, -0.001008864833583349, 0.002502806034806024, -0.002989457944986757, -0.028429843637684776], [-0.2130385871078829, -0.00952055853851568, -0.002958419589603375, -0.0006909683640675687, -0.0026775255998182797, 0.022337039592044108], [-0.19091298460064604, 0.013511732486837414, -0.0023674166450644695, -0.0010893214472722107, -0.002472335179747694, -0.028336341280773766], [-0.22840706196195223, 0.0010821273841604335, -0.001385354548964919, -0.0010269257014854784, 0.0006401097971795826, 0.020109925543883144], [0.7346793322000515, 0.002043057760211386, -0.0008973361982075195, 0.001954381502745886, 0.0021440200417659227, 0.031684652801540036], [0.6511228957327664, -0.0434156902311035, 0.007905145197413874, 0.009048897723012676, -0.005188099663996487, 0.12456653378159029], [-0.15550825719911174, -0.007014144005881198, -0.005317744393304326, -0.004650356793907305, -0.0058826804889584436, -0.033249879134340646], [-0.22374998998409396, 0.005305204512668511, -0.002587536920229158, -0.0035284488736868603, -0.0031764199493777736, 0.016070524548052512], [-0.18896735532051326, -0.00011939738967292984, -0.005499792742938746, 0.0012271462507724065, 0.0053422098166232244, -0.012903491677393554], [-0.18093243613206028, 0.0011376237790479476, -0.0024643087790573994, 0.00017625800381188616, -0.004364640288913153, -0.02456745049022449], [-0.22791199181418884, 2.4510450240913943e-05, -0.0009838997319835951, -0.00048424200386568714, 0.00047111553541932886, 0.02038119904494018], [0.6813021682389094, 6.846507574107356e-05, 0.08938416021279304, 0.006738963482238949, 0.004224235548316663, -0.034557278272283935], [0.6189365994388133, 0.03148865204940109, 0.0035035627279890466, 0.016881309698816398, 0.004161862636347019, 0.1064446801152998], [0.7584060084813855, 0.030462470981347883, 0.007785291742741237, 0.006001926674167469, 0.016268496873172635, -0.036127557281759336], [-0.19822556697857346, -0.005566611451480817, -0.004959786368436256, -0.007042323450939985, -0.006872408705891668, 0.021000030288656005], [0.5462046416652984, -0.003022448971040281, 0.07653732279697474, 0.008811510738662303, 0.0012808745065705455, 0.11939643259686832], [-0.18473572412830155, -0.004055930296873575, -0.005778443308467163, 0.0013985514787870337, -0.0026181365565189616, -0.01046031718862533], [-0.21960951222277966, -5.851687100415204e-06, -0.002091443919928442, -0.0007074004890744473, -0.0034212659754138813, 0.014776915735738133], [-0.18164533592182636, 0.0020647362696999756, -0.0011505391242401042, -0.0034548503304623604, -0.001374834131014649, -0.02610584342882287], [-0.21743215054119533, -0.0031795015149436107, -0.0033355635982182386, -0.005102317262454176, -0.0016975912931551638, 0.01908045754329995], [-0.21768683948562076, -0.0016010883761118087, -0.002206403983496886, -0.0014180906755580387, -0.008149362324856682, 0.020003226287085563], [-0.21791464501532018, 0.004530835496412805, -0.004567853631595059, -0.004333463052978307, -0.005510473641682351, 0.016128933178496407], [-0.1846041357661111, 0.0005345142156458101, -0.00808856708111987, -0.014561721806091181, 0.07513859564652618, -0.017978209018373777], [-0.1873587422260765, 0.006964033933447279, -0.0033286200687969955, -0.00552704003125839, -0.0043267350257844535, -0.01285146801010232], [-0.16965863402831202, -0.017001065404710358, -0.0031451766924766042, -0.0030513287218011617, -0.001954352439962239, -0.014772776046070994], [-0.16491328514174633, -0.0074896350520465135, -0.004695568161930885, -0.0022463831754965468, -0.0029375834246996354, -0.02934060705958406], [-0.1788585007590046, 0.00019323792043969554, -0.003270699742219661, 0.0008563615836516287, -0.0006119592950696314, -0.02701483505663419], [0.7225512907129559, 0.014574261202950693, 0.0016416547698614237, 0.003746883024513704, 0.0021004635363450243, 0.03238361076805985], [-0.2274306697083101, 0.0025852738911894334, -0.0013650638834480694, -0.0031282583608365348, -0.0036846014663802582, 0.021356652861118527], [-0.18222481660233802, -0.012189067484017, -0.00025018516830898507, -0.0019165061252048336, -0.00014715523503062397, -0.01118893605176697], [-0.2171090429474222, -0.007080867800642407, -0.004923254026989216, -0.0014367769936877665, -0.002838714125759148, 0.02713865589450058], [0.4518267931011045, 0.060299359176667254, 0.008269458518138905, 0.03586012087346518, 0.030977139295168123, -0.13693756563310971], [0.3245934086338284, 0.00658291894300093, 0.011755848920741248, 0.012597786799167648, 0.014394068411146476, 0.12040334924449593], [0.7378483163968675, 0.017807419638952778, 0.009582329682402354, 0.0034723385371715028, 0.006169565117093785, -0.010147285389804701], [-0.22951079463415489, -0.0004917001866608034, -0.0014577915042172697, 0.0034124866238734505, -0.0027370796430719957, 0.019726320785672775], [-0.166357426821137, -0.017246723263563555, -0.005175580489021058, -0.0020417631029856853, -0.0012919702695723006, -0.015386536053720425], [0.5873323190517468, 0.03611403826456634, 0.01387851441821539, 0.008250458400491241, 0.014323102214398514, -0.009745369462484483], [-0.2159700242903566, -0.005960665439627144, -0.001619056544947191, 0.001651877549224707, -0.00145650130278606, 0.018354370028492502], [-0.1654010009117417, -0.004985178289910003, -0.004321074985498542, -0.0036323329613499903, -0.0008755375702369992, -0.03240793729676636], [-0.18104423521840277, 0.009739092675103566, -0.004514263025835701, -0.00734462348124254, -0.0031961007703626564, -0.025306536845926465], [-0.17652320711518196, -0.007392377963453032, -0.002563180386584385, -0.002489417641731855, -0.0012413132444436962, -0.016873836981938093], [-0.1793646035749863, 0.003711128731520743, -0.001592457530285481, -0.004904139521489311, -0.00131138959744138, -0.028161600522821968], [-0.2282023726731767, 0.004906798029230752, -0.0017819166846609227, 0.00025253888099104776, -0.0013028732942627602, 0.01821115907521175], [-0.17778246124954103, 0.005005937342339682, -0.0029134110715788265, -0.0027931243880328274, -0.0033822082505032265, -0.029757794398187255], [-0.21682284748834468, 0.00023757758959085402, -0.005991423561949234, -0.0008666475541144891, -0.002704485888664198, 0.016314493570148636], [-0.2206087431823539, -0.004766770621696481, -0.005452264858512607, -0.0024690620289276504, -1.6159036828500535e-06, 0.021631789928507183], [-0.22018215221316972, 0.005931531662327939, -0.005806205666887909, -0.0006302856642452854, -0.0018977686176274537, 0.015918213832936207], [-0.22554476425557302, 0.003665387979685722, -0.0027589351518145823, -0.003004275031633268, -0.0015906486910668856, 0.017983235150402184], [-0.21861776897380125, -0.0024087644362985024, -0.004690829881091012, -0.0011915886350221817, -0.0013832246084865961, 0.01804217653469948], [0.7552511142817886, 0.02787733578561407, 0.004216865721667945, 0.008064375888214672, 0.0033188604828333548, -0.02853158246314933], [-0.19547674069490273, 0.0008901271760421743, 0.00024164803505107867, -0.0024736546498476644, -0.002780407743185209, -0.004984305456491089], [-0.16580849497345032, 0.013529622761109866, 0.0036517647613372286, -0.001411790298614223, -0.004934796048369248, -0.04335963953534649], [-0.17263027563348823, -0.008643369602969133, 0.015023371997973751, -0.003326315610855377, -0.0033061258527505963, -0.029283951964576966], [-0.18898018397415256, 0.00269528842002766, -0.0074663222271203365, -0.0053936058971735526, -0.002893934911223388, -0.008794574743691011], [-0.17306994107038298, 0.00018288963966655122, -0.0013280526936443214, 0.0005618103142340939, -0.0051210226681640235, -0.03224063742910465], [-0.21959926237990357, -0.005606645053637719, -0.002116479009312312, -0.0016535084167167277, -0.004065591451112524, 0.021374819644016243], [-0.22516089660106692, 0.004106282557796479, -0.005241288222422597, -0.0009149206079530951, 0.0011095846217747973, 0.016677096502148256], [-0.22228221985204494, -0.0013306188087146755, -0.0051157475181218895, 0.0009124799749259308, -0.00033985787829767406, 0.016489297415586284], [-0.2283834608138431, 0.006080726077964552, -0.003912952828612755, 0.0034551535882178528, -0.002762926683137477, 0.014670747481116195], [-0.22972323246553322, 0.006393100304078885, -0.0003338855348108558, -0.0002748059891121072, -0.0005746232556198223, 0.015327400429368795], [-0.22293699624681546, 0.0023881402308122603, -0.0023384328203510055, -0.0006808568478435347, -0.003858087038796311, 0.0157595660563273], [-0.21875433485277324, 0.0033440101053213377, -0.005878897256551628, 0.0005435631935175492, -0.00447101901580058, 0.018550011159620048], [0.2507243233402243, -0.009449215354410635, -0.012151369900239028, -0.012009096461299192, -0.013788973867624468, -0.24128585338128877], [-0.21189109385709695, -0.004650288689964699, -0.0028493747975739648, 0.0005098533887238865, -0.002888594114370301, 0.018436164736948757], [0.4619068790759788, 0.01900500290505087, 0.005776123416556725, 0.014388730221672122, 0.004310446570848693, 0.16614059558766972], [-0.20715796525439092, -0.004732827858453921, -0.006722126970243217, -0.004585636506364526, -0.006061525312342484, 0.017593415235128577], [-0.21789785048238716, -0.005911764956291682, -0.0012886594285051614, -0.001042404286412913, 0.003779580607597676, 0.024527765212666284], [-0.21235696453711023, -0.0018529566373579405, 0.0008853771187280244, -0.0018796315351547867, 0.0007991610737145157, 0.01757168118384731], [-0.22586184884158728, 0.002621734285746792, 7.939179706850611e-05, -0.0019598280706691774, 0.003867222023122372, 0.018336662139651915], [-0.22406380033635387, 0.0001295418557054123, -0.0033600526427325964, 0.008440950717902343, -0.0042672685831157825, 0.01581207043003572], [-0.19607773160921343, 0.001368918414943925, 0.0007473528177991642, 0.005114795415714678, -0.004966068217239208, -0.00202060015533827], [-0.16164822565161777, -0.0064029375752631314, -0.004819775075424872, -0.0009457544867425427, -0.0005910184470198653, -0.034107242671327476], [-0.2114821769445134, -0.009847635415581223, -0.0045082165080335326, -0.0004729555572121307, 0.004699153925167096, 0.025908964738484055], [-0.21880440663182338, 0.0009682046838652665, -0.00445408812488821, -0.0072289867980926655, -0.0020361140693507555, 0.020222057606956664], [-0.20146205533077466, 0.0015315945525637509, -0.00033049989235075925, 0.011257163201835278, -0.003269109325824912, 0.0006657639374084793], [0.17154064000625735, -0.008323929580940165, 0.03099115862592832, -0.020171148381691446, -0.01386586350674545, -0.2554625238294747], [0.6121724030270009, 0.024169831519837256, 0.007221765114797014, 0.016202919487080178, 0.0013505239060518403, 0.1183825569452327], [-0.2161807131683768, 0.0035810890933821197, -0.004384453561178486, -0.006861056188613863, -0.0014273254281877608, 0.016439125919641886], [0.7262457686361022, -0.0009083725612379822, 0.0053710110337760525, 0.004846985112195334, -0.0023452709042031867, 0.029240859075523037], [-0.22662029961908473, 0.025910340113159576, 8.066969428908442e-05, 0.0013898123603273544, -0.0025359232728163006, 0.024215876914600636], [-0.21326753272507976, -0.00714097268614859, -0.004612263272284619, -0.0029335273205659397, -0.0005752829886359312, 0.019362912326048493], [-0.17731797847086397, 0.00843124803446534, -0.005066408786025869, -0.003287511541534967, -0.002420052141767485, -0.030962359109776495], [0.7240149970175671, 0.01476353184002569, -0.004828078866293056, 0.005295516783196749, 0.00560326927794265, 0.03716830780721021], [0.1721834748929106, -0.010381799187157134, -0.017077647972354684, -0.024537290382560153, -0.013508459536412904, -0.26534494448109264], [-0.2122311051994524, -0.007523882301545688, -0.0019618399473457387, -0.005037259040812972, -0.0034927081118850778, 0.024513251183675307], [-0.1786394045853159, 0.002612078874290974, 0.0021618678235909277, 0.003054786648622763, 0.0014795105167878535, -0.03054190129348039], [-0.18137921977041443, -0.006627424580805005, -0.0029980349376437926, -0.003816186867557515, 0.000338231049284864, -0.017140426908367853], [0.7119003772603525, -0.024107447094820836, -0.0009333429226845672, 0.006344297884744212, 0.010487420841876188, 0.054618217840054065], [-0.1677342995214694, -0.005624192675582082, -0.005308355147969616, 0.0006369087692509162, -0.000777537272406062, -0.03220747805921902], [0.7096723239709773, 0.012479766848699793, -0.0003287189046380337, 0.00919050975869891, -0.005679293142659769, 0.03502652258003157], [-0.19001178582587872, 0.0026981232174109397, -0.005132379118106663, -0.006172532662122947, -0.005084412363214709, -0.004963679914754186], [-0.20102670450751098, 0.0035598338663753964, 0.0003554695245578042, -0.004831081923153902, -0.0006177457397514381, -0.005856437887183255], [-0.17184772090422773, 0.00455264909473577, -0.00448141058858239, -0.0009352498408035177, -0.0026852869284892064, -0.03268862788214983], [-0.17128730175635112, 0.0005646040005583139, -0.004032602236499415, 0.0007175691925226589, -0.00017683448312074716, -0.029908496732613594], [-0.18796000932280418, -0.002065578678266966, -0.002488911834967601, 0.00394310259670952, -0.004165593652297514, 0.002731038510674886], [-0.22011848368381193, -0.007113724361875276, -0.0017326283072238756, -0.0028335175316547704, -0.004226731635421421, 0.02494665414743875], [-0.21226023842350916, -0.0063556097969248385, -0.007052844950552658, 0.0009404612480007511, -0.0025780910370042792, 0.021556322959990694], [0.5602365937231665, 0.0100497517629307, 0.005790676225626715, 0.0011057893442609134, -9.0877381508748e-05, 0.1935161744336324], [-0.20892305285583349, -0.0052681013388162275, -0.005061748158633592, -0.0022356548316720378, -0.005560965476753316, 0.02466857028075693], [0.3030959610370718, 0.01674560199166071, -0.016856539069007883, -0.03193149183860236, -0.011818948911658194, -0.23543293699668882], [-0.17037469787021423, -0.013947045132517709, -0.00409574822677545, -0.001948204634196062, -0.0004459929092595492, -0.01210497789370359], [-0.17156529524246986, 0.0036888353012538367, -0.0057147556200575, -0.005842463041872638, 2.581444550316746e-05, -0.03205472087637041], [-0.15255228129160295, -0.010494306760094024, -0.005596008541323069, -0.0014027360498830535, -0.0021853364070376674, -0.03276933095005893], [-0.21107737458172454, 0.0009936481276922114, -0.006145416449150309, -0.0068571525501020785, -0.0022279113672258946, 0.014897540153844042], [0.6246873871958304, 0.016133901958901065, 0.004003134567579722, 0.0033424090434759873, 0.0027903075680547795, 0.1327270701924747], [0.7366148821579248, 0.00721887433193801, 0.0020560321598976383, 0.00199824700964608, 0.00417769806333677, 0.032517599610589504], [0.2710820941936603, -0.05512553794916652, -0.002864374872732067, -0.03486297121899592, -0.014412837647128503, -0.25067951388622056], [-0.21514692763164422, -0.0049984890017469395, -0.004145653685781646, -0.002878853187542244, 0.00016679696592617586, 0.018336459874122364], [-0.1814244562621636, 0.010115658732335428, -0.004935674296657922, -0.006680847028243728, -0.002890456128914293, -0.02585089168302217], [-0.22322878646420857, 0.005891189239996957, -0.002558108123299667, -0.003340123063205154, -0.0047600074947543565, 0.016329169238804436], [-0.18363279349786538, 0.012430927369576662, -0.0003346167991831811, -0.008390772193414045, 6.558334550231778e-05, -0.026804994891282883], [0.7283913272950456, -0.02854769694078395, 0.008470421601746821, 0.011576438902794878, -0.009496859364338778, -0.05538966324049742], [-0.22520746558547264, -0.005294630240368174, -0.0010782062963063386, 0.008743896874121262, -0.0005765532737720658, 0.01853200614084545], [-0.18264157052003446, 0.010048115141452485, -0.005641492429496231, -0.009046430848934545, -0.0013263642469832151, -0.023058923762670738], [-0.22431449385038235, 0.003307781785970049, -0.001800729131004485, -0.00204101040418011, -0.00603033774046153, 0.019212122673391827], [-0.21581440718894906, -0.007818649329753766, -0.005324836547080294, -0.0013828393928458657, -0.0008698814976956838, 0.0240782610151482], [-0.22337101215119196, 0.00032676920878172664, 0.0014120749482634917, 0.00014836610277778862, -0.0018539685210403726, 0.023671103745742682], [-0.2276307101078261, -0.0009117728874701923, -0.004043282441415391, 0.0016475262191095299, -0.0005164509250939035, 0.019788023476029624], [-0.18095761658761586, 0.007674441352926363, 0.0008601072130122703, -0.004331394928217802, -0.0018127627754146007, -0.03055583629019405], [-0.19290468425812435, 0.012489058505572942, -0.0018691331396838615, -0.00026541653660971443, -0.0024241432737980096, -0.026648743312861004], [-0.2108555582339064, -0.010453492657278953, -0.0055578755246172735, -0.004672741911749055, -0.0011223962934868627, 0.022245397954371877], [-0.21160570056466949, -0.011776886696687024, -0.006655902911887844, -0.0006001335242757185, -0.0034629778730239373, 0.02743493490387736], [0.7163821096089191, 0.0375089658082202, -0.002791728556891003, 0.016584591409330333, 0.006864593413412187, -0.031762817397278105], [-0.21224496307308371, -0.012262462707655401, -0.007877784302879045, 0.0018136822763712185, -0.0023200228229972066, 0.02497488396357775], [-0.19676153094550172, 0.005044963836178132, 0.0011515447416338238, -0.003243743407467722, -0.0009026370588595017, -0.007252882880268662], [-0.22384223565532588, -0.005239969231029304, -0.0025163458624053855, -0.0022670735398088652, 0.0006216042460580514, 0.02401174251611645], [0.6651203059920539, -0.03757820684440657, -0.005199488611080278, 0.0032116774684211122, 0.005703702394686575, 0.12159866764815938], [-0.1687263084115538, 0.00239488935444862, -0.005126380056632609, 0.00048146629306727927, -0.001373785189430106, -0.02931654865656593], [0.6859756660882126, 0.02006561021598816, 0.004883205511136043, 0.000978777090348295, 0.001024102507917189, 0.0754059719197295], [-0.16659227883825395, -0.00728275204572777, -0.0010950406386527013, -0.0036988537922046432, -0.0001379575649873533, -0.032816179135677206], [0.5472791099560684, 0.1399691397586429, -0.0011786678918711871, 0.019909080737032202, 0.006237315484033287, -0.10848780344073103], [0.7001514920381893, -0.016290594197972297, 0.005481892240917222, 0.014386856114502803, 0.002906795030628414, 0.06836355877373357], [-0.22800424786359633, -0.0039260168838810925, -0.001548419934770911, -0.0013385446131678786, 8.274489165497508e-05, 0.02491397158324879], [-0.21771691432220128, 0.0014060125255625834, -0.002651619011069772, 0.00035385817451841895, -0.00013157254607756318, 0.013573568512600857], [-0.22209719858806976, 0.002084016236474382, -0.0026021881146863946, -0.004663847365656632, -0.0032307453171730614, 0.01884329648244476], [-0.20415868888671934, 0.013924616435508932, -0.003991405448300325, 0.0021463633034540163, -0.000913830871245732, -0.01117372119936402], [0.6815394671157615, 0.02647142879163845, -0.009270808709258513, -0.004547567147062406, 0.003960193712518998, 0.07531950845862237], [-0.22049994179822666, -0.0014449595097988102, -0.005968099815191851, 0.0032820917259073516, -0.0026140149114794124, 0.015578257642122945], [0.6650818094573586, 0.10737547898321308, 0.020155891572351557, 0.011855740290167092, 0.011863325107625335, -0.06051027571374748], [-0.22728758546227845, 0.0004728430456130396, -0.0017564509802531743, -0.00037046182646113243, -0.0021766052240534935, 0.020059701888874182], [-0.21739622516179938, -0.005733986475456812, -0.0028848672283097163, -0.0021256401932816193, -0.004442279295335609, 0.020916331687516724], [-0.18776847584480108, 0.0012809211229088246, -0.0036138313234472088, -0.006055227713955661, 0.0003815111426273997, -0.015891564049998693], [-0.16474387706344262, -0.008057683455435607, -0.0025622503828897822, -0.004185635603420772, 0.00024493625262107056, -0.03236215641409892], [-0.24649587388423724, 0.0018462836725765288, 0.07989675317883027, -0.0005471417950079863, -0.003312493975447039, 0.005751361692174353], [-0.1729748959012539, -0.009765643388951872, -0.007295680448654149, -0.001712557016802489, -0.0016921360803194905, -0.018182149179521583], [-0.18973725108177836, -0.00289362282888478, -0.004583107369669483, -0.0030922729527562394, -0.0017005755644016326, -0.00532650353584268], [-0.21264727790028265, -0.012241063295646452, -0.003463883989232176, -0.0019775363027718608, -0.004752660767554498, 0.02341575558882064], [-0.21882746124250782, -0.005096604461014958, -0.0016661695653964533, -0.00065150254193529, -0.0005348209673521393, 0.028859892111539973], [-0.18395180727070598, 0.0037832957204766713, -0.005132060657905153, -0.004128080692822783, -0.007960140561148535, -0.014234268553397584], [-0.17885247882785107, 0.0006449535048325112, -0.001963131796893334, -1.5757618912060973e-05, -0.0018455819637331688, -0.029634669964109257], [-0.2224934185834588, -0.0013693943643401563, -0.00578609657957775, 0.002080174138512427, 0.00018775304626694324, 0.01571431567593102], [-0.21279752584065317, -0.00409842646119075, -0.0004326490882898851, -0.0013947682406798402, -0.0028394662453623647, 0.021181883495223864], [-0.21685309347302215, -0.008085673908478879, -0.0037833035945243156, -0.0007860926406401226, -0.0024331423600782035, 0.02360797264341072], [0.7376636487046311, 0.017949423033553524, 0.012763808940988112, 0.007115176232631001, 0.009290125276831165, -0.03160162663307986], [0.5870221422884963, -0.006944870746828053, 0.005458171204430376, -0.003481364600895313, 0.006875643059609458, 0.1670980565729651], [-0.22251162108862282, 0.0006049015141448505, -0.0014434589529787106, -0.0027699352253779987, -0.0035937980329456293, 0.018047245119114128], [0.7485685336452419, 0.0014944438149363582, 0.006993197446076146, 0.022635309931660838, 0.007076670943001613, -0.03649037800313999], [-0.16170356754036627, -0.016511458780830893, -0.0013956539173635427, -0.0015781025547651026, -0.0001016300810511388, -0.030332649141126947], [-0.19983851834505953, 0.002615746292517171, -0.0022309509943876198, -0.005399284998617409, -0.001464219440325514, -0.0036827725141269745], [0.7313972903713108, 0.02649550121237261, 0.005138008586948679, 0.00951362657194687, 0.00469516110817645, -0.016064984676152846], [0.22476112001039966, 0.022325380756561816, 0.1588453003916214, 0.008031838484708127, 0.006709171837175847, 0.1168271885195328], [-0.18321094164688492, 0.008005338211053337, -0.0065562496089386, -0.0007908043258570579, -0.0017694502281833235, -0.026487000928320875], [-0.1769511871141664, 0.0005832623879430557, -0.0017987719218215325, 0.0024987396481536755, -0.004678948150574378, -0.030668048756929755], [-0.17284844102343627, -0.005598945584005363, -0.004432924671119831, -0.005260054297860295, -0.00350333738377, -0.012522963706475088], [-0.17396825011332323, -0.005990651774263665, 0.00023578335422843284, 0.002304128141668813, -0.0005788080996459315, -0.029875263524168184], [0.608284460362254, 0.006948256252923525, -0.0033292479555986497, 0.006112224875232649, 0.002056010105361392, 0.15849972493125564], [0.7372022095537677, 0.014016002564035296, -0.0035351682578677675, 0.003677682884252688, 0.003998290267190408, 0.032974316321953655], [-0.18806898007258815, 0.02070495644089133, -0.004740211645779391, 0.004748348882945469, -5.1239119244031765e-05, -0.03277144591479649], [-0.22154373796280352, -0.004080361186416828, 0.0010875016694452296, -0.0014387773652492293, 7.856874430272916e-05, 0.021230139434055134], [0.605121339653447, 0.009525303448878851, -0.004301044173826142, 0.01211667353144153, -0.004157384487474236, 0.13726654059896118], [0.7264391768668768, -0.016914985188531295, 0.0047054075574362465, -0.005945737677911232, 0.004633264852773586, 0.051766329022811364], [-0.1693837348481496, -0.008140648191615056, -0.0018213248160493447, -5.156763630608035e-05, -0.000764419840597011, -0.02854470001612025], [-0.167729001346524, 0.0035881863590400795, -0.0038903854212331834, -0.00411451044662664, -0.007273740017508858, -0.032203611142651106], [-0.2146553252921099, -0.0024020867751532423, 0.003593548665161255, 2.947922691374489e-07, -0.004165530388941525, 0.0265800793909313], [0.7058184447372157, -0.010123700891739326, 0.0009453404587581583, 0.0051302683892820245, -0.02125289669536172, 0.034897810108287276], [-0.18260332308002178, 0.0017767737065978455, -0.0015631243034136281, -0.0014437930600853324, -0.00018952812699065256, -0.02760006715159032], [0.740737086234514, -0.03253043105906489, 0.007235095409992026, 0.0021883182750404738, -0.004992858868279271, -0.04676816237315505], [-0.22609211348219357, 0.0013573812399828593, -0.0017652722262720298, 0.0021498180671229304, 9.749012911917212e-05, 0.01772491849446255], [-0.15462801327314896, -0.00546516764276575, -0.003661043075145449, -0.004283388951837428, -0.00908951816925786, -0.034495930903348335], [-0.2090950703100565, -0.005700966435013635, -0.003299478095460869, -0.005479329688623069, -0.0029281424234687305, 0.02233632028595675], [0.7723500802032032, -0.0065914140635689785, 0.003941276926412805, 0.0029020933359259625, 0.013363720748282841, -0.04485861429311252], [-0.22388747348161064, -0.006310545875884699, -0.002730389331904695, -0.0030248267907238293, -0.0006337411145966909, 0.024920309928054163], [-0.1926842348210141, 0.0015282338659858127, -0.0018009131501069762, -0.003351400107637409, 0.00011532864359564382, -0.01543007644632643], [0.5453086063910936, -0.006969638052603136, 0.0012853504123804523, 0.014405596856771495, 0.0029893179997171142, 0.19664743305930593], [-0.22142375288135896, -0.00022197883628000654, -0.003407770065781754, -0.004018665093542212, -0.001560673472222146, 0.019382840349185106], [-0.16549025527309688, 0.003847486128482345, -0.006339697409171768, -0.004965410558090714, -0.004202495433294823, -0.03447268947033179], [-0.22608145102728955, 0.0029421536190923194, -0.004524843549286993, -0.0001811660930381412, -0.002258145691907843, 0.018770119409096973], [-0.17447949594091872, -0.015185251220841618, -0.003066872242849362, -0.0018361366371652127, -0.0013694047522357553, -0.015729505872655924], [0.6618477621879565, 0.03280014631957703, 0.0027668894357190756, 0.019787565357542558, 0.00544855155496937, 0.04460106927121909], [-0.16181669155802586, -0.01223462675711499, -0.004188060816102732, 0.007932212276756185, -0.0021464017113317735, -0.031713098100847305], [-0.21619877817181785, 0.0049952828260739135, -0.0024015467252003716, -0.006293556594276225, -0.0029597028378815816, 0.015691634836435143], [-0.21343186649914192, -0.005231492210509653, -0.004346105884976153, -0.005320270863691122, -0.0034974769732338154, 0.020160545764886493], [0.7139834503220304, -0.0031791789601299924, -0.007541086622988125, 0.005382249898415404, -0.0010282139106465409, 0.06263277927331488], [-0.15812091512719237, -0.015548186653041186, -0.0039015524657007635, -0.002860938384913829, 0.0007409162933127201, -0.03193238567796834], [-0.22114891552399904, 0.002342532199263229, -0.003635666015263678, -0.0052736496616803095, -0.0011899979983295594, 0.022489030333342683], [-0.22613706202644046, 0.001242643206004047, -0.0026909485128190818, -0.0009133834518522485, -9.036875057763897e-05, 0.017736406357390662], [-0.1675358663360712, -0.002658467685148509, 0.0011104507413998562, -0.0030672564423096043, 0.015302305497395205, -0.03731783244193233], [-0.15666707495421633, -0.018121599579443756, -0.0014738270072854778, -0.003890832560649432, 0.0020509859678436307, -0.03352071388175229], [-0.22479520648057097, 0.005023731775043439, -0.0014173582486051877, -0.003324630703914308, -0.00274034155466811, 0.022170471879381787], [-0.214273844173762, 0.020014938852974142, -0.005393600587667552, -0.004133746233472649, -0.005206112931616525, 0.020159031740211674], [0.5418355733921488, 0.011062707823431172, 0.005824954148804298, 0.006127863442805779, -0.0001757069178255273, 0.1893722271582535], [-0.21326201023685035, -0.00199696914493787, -0.00447685619361915, -0.0038959872147957026, -0.004655532731343456, 0.02012068885488005], [0.7307107915124031, 0.01263654045799558, 0.005728867566639025, -0.006183452462176236, 0.006263630967454525, 0.02184362195768198], [-0.16142354441687679, -0.007005947764569087, -0.003866875121668164, -0.004179225466658629, -0.002285816631138917, -0.031194985947925656], [0.7310257235006405, 0.011009207032962756, 0.0009094182844130257, 0.001410525101569015, 0.0009078268110981276, 0.03198729926931473], [-0.17546045810894975, 0.0015235856582967778, -0.000219364635227405, -0.0009833809527758034, -0.0011227651781968626, -0.03075257069054268], [-0.16460978840240278, -0.007414715034795348, -0.0014615780645959844, -0.001907771902108988, -0.001411927734525749, -0.03481728087707477], [0.6590128637716888, 0.030505983613755314, 0.003284229365063901, 0.005642944031648734, 0.005658224380043869, 0.053854088171131846], [-0.15728592457771243, -0.006612501998453856, -0.006032325664607714, -0.0011457564505341566, -0.004383883401269574, -0.035554561814817774], [-0.21896016513014954, -0.00939485325366753, -0.0027265350997673315, -0.0008348828685053826, -0.0011794735700532358, 0.021429243255476745], [-0.1621982057398319, -0.007190862532972659, -0.0031549979627219053, -0.001496553492951188, -0.001270794686540594, -0.03381164760048568], [0.7234049392743054, 0.011747183865731868, 0.003795056627007992, -0.002326176733295629, -0.0017092083500816443, 0.044532649760775124], [0.5826956169656395, -0.043249403815163914, 0.0021550322781436347, 0.0024257296175315803, 0.002144589650068557, 0.1835427210180668], [0.7675622848493846, 0.0006072577501109064, 0.0058987934441444055, -0.001993937029391484, 0.009526075562258358, -0.0388504745765064], [-0.225354564648769, 0.019889654519017693, -0.0003318696674397327, 0.0022880249317558904, -0.00142128644081812, 0.022221707972919415], [-0.212120796169311, -0.005663343286386933, -0.007420513925117153, -0.00301846955325206, 0.0001661520572592428, 0.028390304210141635], [0.7674230300157191, 0.017886617190760103, 0.0038468251365415622, 0.008684704242685954, 0.00972777096817944, -0.033421660732181606], [-0.16267180514326485, -0.005369278232907717, -0.004900091414030914, -0.0020441455232234618, -0.003492322819110557, -0.03314541888296627], [-0.16368162903782094, -0.007044817656453154, -0.0007931540083026468, 0.002436871205706634, -0.0008649574033592567, -0.0356337084486073], [-0.17363334147548037, 0.003970005004393098, -0.0033501631995121696, -0.0036526388013469907, -0.001479704661867498, -0.03327313724903653], [-0.1841022939686864, 0.012630028311206092, -0.004799735482791392, -0.006746863146344204, 0.002585204505998017, -0.02456634021938204], [-0.1753752484926827, -0.011833332665943308, -0.003323532325949845, -0.0013942782081306892, -0.0014023089756383044, -0.002879632664988616], [0.5818855614524403, -0.03359319116273091, 0.08032657370049426, 0.0008133150401160933, 0.0025257121862312087, 0.10482774306916232], [0.7350647318724891, 0.005687093102635833, -0.00030915022430732305, 0.006626869663584332, 0.0029853392174423155, 0.03388655780959496], [-0.2154648764474314, -0.003935832392173975, -0.002620475701441864, 0.006300018680869117, -0.0031557369824691006, 0.0172102361759808], [-0.21379630603905522, -0.004074251471329914, 0.09797718582949146, 0.01093611648850733, -0.0044767210284080965, -0.018030309493491114], [-0.16235820247189572, -0.0006325560870914086, -0.006089876957201885, -0.004498828943469098, 0.0029515617122152925, -0.033911825934727635], [0.20581064058967521, 0.001596852159634247, -0.013688582196840497, -0.007957727496275258, -0.0007807098792194013, -0.2629685684150696], [-0.21880517653498346, -0.0009732968507411925, -0.006482492892265163, -0.0024536349241570414, 0.0030204080653749212, 0.01902752647010542], [0.47191306112219017, -0.09860856108174111, -0.04453304647062907, -0.013022244897008345, -0.0021100520490550606, -0.18912943475647706], [0.685637392479288, 0.029009500253628365, 0.004212461116751671, 0.010219455333090032, 0.004088120004621194, 0.055166404145951774], [-0.22610827400033764, -0.0003273304568737883, -0.0002165875903512782, -0.0003687153273545268, -0.0001434527149614936, 0.021105801531319936], [0.6235048952753575, -0.002492069801576565, 0.06162471788126446, -0.000473607490937106, 0.0013797156360903357, 0.07137301516646997], [0.7142424695635003, -0.05132828273431922, 0.005873525979836676, 0.025110411896243485, 0.006114847364125532, -0.05656852762494367], [0.5519426028138114, 0.010192946045241406, 5.400408260456777e-05, -0.004986651987303159, 0.004357784921412521, 0.17802264745756627], [-0.2099989294329882, -0.007935370264818185, -0.0054186642197980315, -0.0034930345140479495, -0.004284554313823156, 0.021963886078809246], [-0.2188740073103497, -0.005922734587407443, -0.002410376620870885, -0.0028800105478821738, -0.0017172260405651637, 0.025725923734526715], [-0.16730673886514477, 0.012202619969543358, -0.008494668924060242, -0.0009725517034874413, -0.006613191677365197, -0.03764880213281898], [-0.17074760112264883, -0.008769554896719893, -0.0013744219907195021, 0.005356792403552957, -0.0013410153721116628, -0.03229086568801929], [-0.16139784050238765, -0.00877322698745544, -0.004254189015097724, -0.0018353570233166242, -2.7799604969890497e-05, -0.03533464888227638], [0.5977499222336403, -0.011487444885946961, 0.0022295820087241585, 0.00844694823280056, 0.0030697406658480885, 0.17276902952271125], [-0.2204429000718694, -0.00038101414418800056, -0.002980794586166591, -0.0021714023436181853, -0.006448252523609696, 0.022424363669451948], [-0.2043446778246365, -0.005027853577945152, -0.003731211390840042, -0.0029386953470549666, -0.0067579601353043375, 0.01696706494244801], [-0.21876252330423185, -0.005167748734781395, -0.006415452189331229, 0.0020729170526663956, -0.0009458612479295743, 0.02316010986504905], [0.7221539198859813, 0.005591429508395435, 0.006111007039816842, 0.004025528297500989, -0.00959683566103374, 0.042131617596003805], [-0.22003276145644357, -0.0036960853027255243, -0.0049715440531720115, -0.002295717876986037, -0.0009895323563590406, 0.020318974379019774], [-0.22216993037092922, -0.0017519548386425179, -0.00357571138567104, 0.00035400027762312207, -0.0012977463344499899, 0.016774675985402963], [-0.22524195633264213, 0.0022774394415434837, -0.0041695731985457035, -0.0023087432917575806, -9.446362256541993e-05, 0.017870630337300547], [-0.15307916192530835, -0.012928292769302433, -0.00480528585397986, 0.00040550373262444856, -0.003604269474701307, -0.03594488905816922], [-0.2083103506499664, -0.01029039645482845, -0.0073768025977518446, -0.005830234350740266, -0.002632236164507649, 0.02652335355112856], [-0.19534212294204537, 0.014090334399975741, -0.0018588645363893453, 9.317426559007562e-05, -0.001002517088744381, -0.02639667076505315], [-0.16989221724706396, -0.006619541281767365, -0.0024743606952975835, 0.010161137459084263, -0.002574904338838991, -0.029433447229449693], [-0.21280832381916054, -0.014063759983380312, 0.016323317687381583, 0.002498278350165727, 0.005661583336379781, 0.019281761571471184], [-0.18100225625563615, -0.00048614119367786204, -0.0014018103156056198, -0.005320264283623716, -0.001228531487406191, 0.0014330511549970623], [0.6624516339338369, 0.03993135412994517, 0.01597898307226771, -0.0007069078465066214, 0.0021440858252568177, -0.020602179417831545], [-0.21275256986726865, -0.0034007347439346525, 0.00024005905741462382, -0.002700875413119747, -0.0041495496336599585, 0.01609700393390164], [-0.19866981626554173, 0.007676993041326416, -0.0018893660631608589, 0.0034796733358790936, -0.002026466656963448, -0.00523768405820626], [-0.21619286919149824, -0.004061866963299796, -0.0024220802777277736, -0.003430615730933079, -0.001112752596613248, 0.02322018476007239], [-0.21050332364756646, -0.001790696796934435, -0.00010790847076215477, -0.012775793082055497, 0.10875690928206067, 0.0343394635089089], [-0.15390579215464886, -0.016711578122334265, -0.004557343226804875, -0.004164790810657737, 1.4189532048993976e-05, -0.03234135188426962], [-0.226843212352604, 0.0010698014640956768, -9.83709252709642e-05, -0.0023934653141186543, 0.0033656071923219063, 0.02007912711506341], [0.7271322496043665, -0.02283099541371337, 0.0038670316144089234, -0.009430353866941297, 0.0030407037023720152, 0.050364221502363526], [-0.19815717864436969, 0.010744014458593252, -0.0012322340000746157, 0.003617406359106971, -0.001117732880396286, -0.016354275292859623], [-0.2196732018675145, -0.0071654212610781295, -0.002038522519607877, -0.0018959705171033216, 0.0011416503369968787, 0.019810953007794542], [-0.22811288084069567, 0.004797759835199169, -0.0018367485589520012, -0.00016956366844372527, -0.0025647744677011514, 0.016219541033926324], [-0.2274003815839794, 0.0011358877387807266, -0.001452157502737774, -0.0010177515873407718, -0.004162156048374896, 0.021838000425093356], [-0.21644200142674871, 2.407985708635074e-05, 0.0918970058342411, -0.003019777856246041, -0.006810525419073708, -0.018690447655925514], [0.7263697206000758, 0.00014024123401036088, -0.000966479367376385, 0.0031721001117831025, -0.000894052767839366, 0.03646418447506077], [-0.17957352854401393, 0.0012976024915744722, -0.0004928524333146451, 0.002383702635644088, -0.00420256320192998, -0.02917731485535565], [-0.1735339364594869, 0.007938203366266161, 0.041214366587927403, 0.00012901356627365701, -0.00448398740905468, -0.03349137283022027], [0.5368468948304153, 0.09411762934781266, -0.005451275373510926, 0.02250778688761912, 0.017663425262674906, -0.03342657058956383], [-0.21223105081231997, -0.008144875874684892, -0.005347804943111792, -0.0031084303468434655, -0.0013181682769879475, 0.023554607349401718], [-0.17704508781044953, -0.009220233221806592, -0.0019927743101982322, -0.003232906699336828, -0.003642917318587905, -0.016489142655124793], [-0.20288386926327587, 0.004911891380108991, 5.16654644030684e-05, -0.0010002998114637535, -0.001927521339352964, -0.009389961668514675], [-0.19020965911308355, -0.004501664555037818, 0.0030109386760369576, -0.003314851062462048, -0.0017315084063690043, 0.002080077794249081], [-0.2261149942383926, -0.0010488054620144693, -0.00022199804729093072, -0.0012550253446335802, -0.002444593338582428, 0.025569287398655865], [-0.22235469559509866, 0.0010821607385756381, -0.006174056261272771, -0.004243902774177405, 0.0013274747543022406, 0.020124923899576146], [0.551389053877908, -0.04224536423756338, -0.025019656657169393, 0.0534341735143908, -0.004831323328800947, -0.03495108951797237], [0.7241039057575422, -0.018208212457345656, -0.0018251172460929043, 0.004600439065650443, 0.00466723720898735, 0.050482691433377705], [0.48367597124498557, 0.020537193301200823, 0.0024013851287154686, 0.015270866615540973, 0.005073875478164343, 0.194187533628218], [-0.1718929220121313, 0.00030609678032540277, -0.005339375476795558, -0.0011017575466500235, -0.0008742480644718511, -0.03272085569578037], [-0.19057255613927687, 0.0005615744422085609, -0.005143969507035463, 0.014924129622150735, 0.000364489596927518, -0.016383668014974213], [0.7770540303985028, 0.0037636047017486755, 0.013041730611615222, 0.006930983262922613, 0.007427953233837587, -0.027801635541960724], [-0.21466567423214278, -0.0018840877637844504, -0.003230370045820944, -0.003382681338678762, -0.0012922601585968196, 0.01378840687235727], [-0.1662946651605335, -0.0002464354260575394, -0.00714239557512853, -0.001311006344294617, -0.0037050143413765208, -0.03231543706000496], [-0.18097604530862493, -0.0028850895075186573, 0.03964554127199425, -0.005266234090393246, -0.0006383123340679845, -0.02959414574567513], [-0.222526337616636, 0.0004921779109917165, 0.0020805188049311625, -0.00016812345033759488, -0.001857621639587363, 0.02447938599063777], [-0.15317248774081835, -0.01691367570924756, -0.004389055002866135, 0.008133687683628653, 0.0022051345354778583, -0.030149318051888408], [-0.21438795492128243, -0.008793365747944024, -0.0017938512545717068, -0.0008343049297806231, -0.004285193672389759, 0.021886337192634938], [-0.18856380435575135, -0.008888253271031665, -9.088983275188892e-05, -0.0034174383786889735, -0.002746431443621641, -0.001543182718154416], [-0.19929223196281362, 0.0022300991133702573, -0.001565650031088978, -0.0042305814116693945, -0.00032385458695988417, -0.002984447787504695], [-0.22644386450028633, 0.0033429801276347354, -0.0018062494102750331, 0.011448781783934572, -0.0016047549091145247, 0.013396440241439564], [0.6020049621122224, 0.07972621272208738, 0.0147259539818328, 0.049849290915898965, 0.01879880008911274, -0.06646182155275672], [0.7412084711633272, 0.01539735843097913, 0.010122867946836008, 0.007291808574864629, 0.006064605726511185, -0.012471475478882296], [0.6659983917187883, 0.021149213952436078, 0.004898676821273018, 0.006384765280324185, 0.005199644448947729, 0.07594516602850833], [-0.17979718158015992, 0.005504719507346998, 0.0025310588943506466, -0.002836887559228086, -0.0017963437007350947, -0.031066012611091876], [0.7206222770594757, -0.054142417291254005, 0.004009667074541766, 0.008452390513986512, 0.006875758597912811, -0.034401009287995096], [-0.22448470449518296, -0.00011909488365594686, -0.0008138101112795712, -0.0023078746826579416, -0.002908950220601326, 0.018967767726711212], [-0.21697372318674266, -0.0024016952666881634, -0.005511119263111883, -0.004362459996348779, 0.0009547748588405128, 0.019056127615956018], [-0.21178091637365584, 0.002576024276868872, -0.006123130230191335, -0.004014387756039126, -0.007030301566486271, 0.026206044982837382], [-0.1569657480003294, -0.013707678769804029, -0.003760028383046456, -0.0014211282556290147, -0.0019624301805622876, -0.030806048426132455], [0.7396163784081925, 0.011519459634061325, -0.00027767456495213276, -0.0027322185569532754, 0.005436734017664917, 0.029770654395318365], [-0.17332693679566935, 0.004032780241020886, -0.004717834737418457, -0.0036687395399320325, -0.0009954019794013217, -0.03169692920410349], [-0.22720692577494048, 0.004637278765515988, -0.002658243998104611, -0.0015016801910794867, -0.0022144763213399806, 0.017277380853282145], [0.7006938536636825, 0.012032235056108074, 0.0027053044729590274, 0.004644543734497842, 0.006608344824869985, 0.057482384914548684], [0.6928625742505816, 0.029706266948240592, 0.004592750186608729, -0.0009601795413047281, -0.005522064745002515, 0.048653986234209086], [0.35545438805756435, -0.16331651274546996, -0.00737694714921095, -0.0005982909559551055, -0.01795326610331074, -0.18027886169055066], [-0.15803264415762308, -0.015762032171180267, -0.002871092521466014, 0.002524211860430277, -0.002767011697408584, -0.034714493328256095], [-0.18419722843454028, 0.013593954924362854, -0.006668262556925087, -0.005092015267686656, -0.002517393564633834, -0.02295238843391044], [-0.19023604017016166, 0.014485923161525723, -0.0054599868040120965, 0.0015592836177299924, -0.0046287024092886, -0.027387144062459814], [-0.17723331035803527, 0.002917358692484053, 0.0013679680518574836, -0.0019012218855640488, 0.0009017483697322914, -0.031425604885978294], [0.6842670471721346, 0.017229379687172898, 0.05935482648228567, -0.006189022804032528, 0.008457449755134901, -0.027369680292695445], [-0.2191626368489995, -0.003807960999233568, -0.004690602237448005, -0.002626977490405721, 0.002157771402954847, 0.02227326331598928], [-0.17816362458500123, 0.0010847220245784976, -0.005085582322350439, 0.00012149599166192077, 0.00037917178326639183, -0.028709244907658932], [-0.16505455141492234, -0.007074306474259154, -0.00043338055308548933, -0.00353536930172214, -0.0016334943312382635, -0.033891959940276196], [-0.1596396755781185, -0.0062064115622979775, -0.0027027548436240347, -0.003444019175446797, -0.0008976880856468661, -0.032899179437036144], [-0.22699436139718288, 0.007114909763022691, -0.0018654053799232826, -0.0038194248925847484, -0.0003962058252628061, 0.014293821065264644], [-0.1771644402591983, -0.0083024186499017, -0.004515909063339739, -0.0013547675282751107, -0.0005719365163582661, -0.017213589998430485], [-0.1859478973624599, 0.006459977798546783, -0.003233913126031427, -0.0005776845218833979, -0.0002846217390644321, -0.027413923064611226], [-0.154403662443358, -0.014128237689681255, -0.0025288694125232616, -0.000812891787063736, -0.0013805301815753102, -0.03836887050130202], [-0.21406141914688917, -0.010156807400231269, -0.002013281162351301, -0.0013580684545162094, -0.0038578466253812727, 0.023114089456035943], [-0.2189817133686267, 0.0021774911025754943, -0.003015869320238441, 0.000628713359122781, 0.0007426693976862132, 0.020115375496147517], [-0.22889595316496855, 0.004235467093652357, -0.00010228695337237843, 0.0012885832301189571, -0.0029042386163561566, 0.017211761744258862], [0.486301324879859, 0.08029168796902403, -0.0035626031552947477, 0.03940927517170774, 0.06260078937514732, -0.03411190281187177], [0.5497024865499138, -0.001387730845887656, 0.0021846491268462205, 0.003676789494306603, 0.001457457076392918, 0.1837578051810606], [-0.2194894533406246, -0.0020877698013072278, -0.004911732049548762, 0.0003174576883882163, -0.0028694408868172286, 0.017982379831350738], [0.5525391078620818, 0.006451604835421061, 0.006318220251063455, 0.012477820497333028, 0.0016486014870716829, 0.20014797840036164], [-0.14860546508056163, -0.013230707659988994, -0.003417480457406731, 0.011937274398635424, -0.0014885932318065458, -0.03394502796887151], [-0.22432420110359827, 0.0038090561260713993, -0.0035362454831787456, -0.0022822579268474493, 1.589036936962402e-06, 0.021665392683949403], [-0.23203666754058028, 0.00527296825987543, -0.0015926147268664375, -0.0009240855108688507, -0.0030220697007226926, 0.020635802552495704], [-0.18151204770658794, -0.005551070876522962, -0.003752592237061213, 0.0011377790281110471, -0.0007550905254295669, -0.0142922029077344], [-0.21073677975941427, -0.0035383589776430274, -0.005360412517689689, 0.0033988446610645185, -0.0032199228772178913, 0.023623296137567768], [0.7495100176303717, 0.03542234224973008, 0.006844446465812427, 0.0047943954010152365, 0.0052510335815136675, -0.036488901995110797], [0.21591218705169432, -0.007956401343557276, -0.006573135147206671, 0.0032892993804507003, 0.005632981899385598, -0.24533077163404834], [-0.21585449616432428, -0.010336423002239931, -0.007026986451949998, 0.0010410701178310527, -0.001487235996305113, 0.025390261973178844], [-0.21657955609635093, 0.004103478425572563, -0.0030781101674104134, -0.003183684273995726, -0.0033717349068579138, 0.013121511780946905], [-0.18186023293258505, -0.0005320538657928322, -0.0017698293584951586, 0.0028220419181590236, -0.0009537429872702318, -0.010400673360948225], [-0.18342909813725664, 0.005474849960400587, -0.0018818758240893684, -0.002877212019529982, 0.0011059545966886255, -0.029811598959064007], [-0.22527698045026637, -0.0026064646150119417, 0.0003039614938887609, -0.0018318991677243406, 0.001487654659164329, 0.027269881926103377], [0.6686788980979752, -0.001874959075114383, 0.0776868669968296, 0.0036335691605066983, 0.0048699604267510525, -0.007791954654566752], [-0.2183735450584861, -0.0051876527544355965, -0.006385791119958115, 0.003840210481917621, -0.001370366220441444, 0.016810478004736867], [-0.20633361855478108, 0.011894869378060599, 0.0035229958170258126, 0.010415654862309144, 0.00022940117821458985, -0.003776921728447993], [0.20051238843479055, 0.0004012224799041278, -0.008203192317638419, -0.0077687147860517084, -0.01304219156230474, -0.2668414775831072], [-0.20531675975259264, -0.011180209716964654, -0.00807770421893867, -0.0010499000563254982, -0.004574302730785268, 0.022698876475606877], [-0.21757784684027823, -0.004700977791780466, -0.003733035235662831, -0.002223415476006037, -0.002564844711215125, 0.01913345338827604], [-0.16940851470074367, -0.0074425058234680085, -0.0029584842726391985, -0.0036490779870898314, 0.0014152539311545077, -0.029579733162717683], [0.7326329309853736, 0.0024452295213205494, -0.0043982585665343875, 0.0070038111644958765, 0.0037459318128016724, 0.03590368841587398], [-0.16356011773278462, -0.007091905492408832, -0.0036502404884123565, 0.007822462820617599, -0.002788518529246565, -0.032398347244431636], [-0.2280558897804553, 0.0028915080687768207, -0.002318000363361629, -0.001820155189893159, 0.0004977704915246843, 0.020822310633057714], [-0.22228800994460135, 0.0028196990486468914, -0.002599829682846506, -0.000748992121026476, -0.004512592339824294, 0.02332972503965172], [-0.2151537235658559, -0.004430905473777778, 0.0007614160300502488, -0.003034910177076056, -0.002035061756529799, 0.01847651827652263], [-0.24510839136806167, -0.011605673757675423, 0.08351739096985857, -0.0014124095992493417, -0.008513304331991555, 0.0184557214204522], [-0.18471157682026673, 0.008657255744908839, -0.0010724602403671876, -0.0044291730592753, 0.004450838718709784, -0.028267946359213286], [-0.2214051382742533, 0.0030427300541476044, -0.0013843154294405692, -0.002789852541110647, -0.0014261351494272668, 0.02042469964417787], [-0.1474115234147175, -0.007006925226820414, -0.0063997501636103935, 0.0005543398252558602, -0.003552054729809309, 0.03548258037636877], [-0.21465209793008785, -0.010920702701622213, -0.0035914168587041877, -0.0017872271864959597, -0.0037922754162530528, 0.02307705342649661], [-0.16543923929355117, -0.008062327208365281, -0.005394600596638613, -0.0003674645200461351, -0.0011102917137547242, -0.027582472016481115], [-0.22808956419924667, 0.005178084052705959, -0.0008494604562901282, -0.0007896411424689425, -0.0032116172819741215, 0.016095532360606794], [-0.2234956121128585, -0.0013112551970095611, -0.002283492079199865, -0.0007972675358335869, -0.0036393497010753398, 0.020468418067418063], [-0.2249416823997231, 0.00022078312703427823, -0.0004918498111453427, -0.0011350300653056816, -0.003193464477198201, 0.01848268506777951], [-0.17893112173895387, 0.011492075382260778, -0.00404901879464098, -0.007718423697681743, -0.0029221619663257507, -0.029538015851324918], [-0.22148781095726677, -0.0021389718077118756, -0.002506203841434032, 0.00023097653080259045, -0.0016665537383887385, 0.01638576811507395], [0.6072330390871825, 0.0148089797309806, -0.0027345646243554914, -0.0016557649647125838, 0.004222533139131105, 0.1555424442984411], [-0.18547117131462856, 0.0015935258442101375, -0.0017193932099120392, 0.0036292506823122086, 0.004451538877135138, -0.029974159833395967], [-0.17628498982175606, 0.0016605317597213692, -0.004018085034084886, 0.0001825732825084849, -0.00432530667440453, -0.028837785527488103], [-0.1535549945046614, -0.014828690845523759, -0.003946627487356598, -0.0041348442762605565, -0.00040838109521587715, -0.034793128457648224], [-0.22140486814836016, -0.0005703437211047789, -0.003028325844902664, 0.0007802379415775986, -0.0008454109611787431, 0.015068710733968666], [-0.1761398663725918, -0.0027398033970524797, -0.0046112144605840195, -4.088261336211166e-05, 0.001861124969947586, -0.029952420141860916], [0.6549525726208868, 0.012586653005199067, 0.002566502413890759, 0.0007841473944677204, 0.007430220641316668, 0.09934657059090604], [-0.2204805488269606, 0.0017947960440853263, -0.005407593996553186, -0.0013360328038240072, -0.0042222416975571005, 0.017984954614143282], [-0.18430090915390077, 0.0028775833781760317, -0.0023842801546722294, -0.0048317324890880635, -0.0025062878701101843, -0.01635437371040456], [-0.18782046381858952, 0.00017484975076835979, -0.008644602867592653, 0.0010863027331574078, -0.0019133209394246846, -0.005216098191651689], [-0.2205409118416736, 0.0005971024415824431, -0.0030772026567046544, -0.0012226524662751392, -0.005068279830978105, 0.01825338579549037], [0.5459119684884547, 0.011363419219089227, -0.007033706385371183, 0.003906254100063746, 0.000443182413404682, 0.1931549139103902], [0.7318586840092132, 0.007580967948301917, -0.0011545756475301408, 0.006242935937896382, 0.0041463403792695525, 0.0354923140395149], [-0.1665664214354662, -0.006084668081861707, -0.007781865894203814, 0.0238373394183999, -0.0004670820240429587, -0.02968333372885629], [-0.17526212441845396, 0.000911894099152399, -0.0030733390633958003, 0.0011927235578050193, -0.0016037723652593547, -0.03318033571724388], [0.73055475160422, -0.0012475138021665238, -0.01500523401132821, 0.015227897998932935, 0.005189550113118427, -0.048278975712300824], [-0.19932156853607141, -0.005102421730893404, -0.004385413894952109, -0.007929360205223375, -0.004439617207020778, 0.015345048240827885], [-0.21882458625303572, -0.0071956864554846385, -0.0029963875006902567, -0.0005299162485102601, -0.00235425323199101, 0.023150829689712046], [-0.1797347676784647, -0.008185221855147029, -0.00428480201758041, -0.0005347463601632218, -0.0011742555943905462, -0.015209268509757975], [-0.17292990976978737, 0.0019674403826804064, -0.0037908233187722556, -0.0034227855332320733, -0.0014436417812457379, -0.0320033419951468], [-0.22116855804680308, 0.0006369611909487785, -0.0012992586792058675, -0.0048631940777134584, -0.0010248473266677262, 0.016052230272774706], [-0.22769412374491063, 0.0119339930979868, 0.0018732251506500754, 0.0021435091847994948, -0.0015931057261063205, 0.018336502037580076], [0.6597248522427801, -0.02205380841211674, 0.006334036313054001, 0.010848949015178216, 0.007509353306948378, 0.10370804610558511], [-0.21641008794909278, -0.0024031296271612677, -0.0061426781408126015, 0.00311685282660658, 0.00015377359168729351, 0.025518602632106594], [0.3008310565328303, -0.09581457295286233, -0.04783564768762677, 0.008273747137915347, -0.022211175014894616, -0.2226323430467391], [-0.2193732927725371, -0.005525008417379889, -0.003539506484177295, -0.0034709203724286506, -0.002766642614525131, 0.02359693928849948], [-0.2213398601616911, -0.0022686818065651235, -0.00584847888548529, -0.0010234415928444959, -0.0017586340503899143, 0.022572429830309645], [-0.1822132460222481, 0.005475474561498265, -0.000999324632967716, 0.0038224204335924952, 0.002459122747687565, -0.02766750910306618], [-0.22559419908268108, 0.002903907738260888, -0.001329384941606524, -0.002664486206082987, -0.001078593060184817, 0.01609608888562765], [-0.20472315434358698, 0.019124635429146695, 0.0022130737854722145, 0.001990581017062452, -0.0012555852454713937, -0.0030301061981786235], [-0.1584215725188773, -0.015619556343011341, -0.004667083394036457, -0.0021898966467071273, -0.00021679784971181964, -0.03050815526315966], [0.6708551248340211, 0.0473548451107717, 0.008377977371866288, 0.01714046189223182, 0.023483431400795965, -0.054676126323974145], [-0.21437602779288353, -0.006043449040176355, -0.0034987393102906884, -0.0046957985834736055, -0.002771360170614848, 0.01971870823077271], [-0.173613515916961, -0.003512741141966052, -0.002488723279591788, 0.0016131959997556422, 0.05627776989839228, -0.04240597268961586], [-0.18413709585869606, 0.00666015466622699, 0.0007094613488902337, -0.004404268565895347, 0.00011100811314698869, -0.027024906753190105], [-0.21516157560497826, 0.004842158253479662, -0.006044112160970829, -0.006221620806161815, -0.0032010030715108697, 0.016119486723475726], [0.7326396408984887, 0.02303836140160219, 0.008430508045101141, -0.008133758400268216, 0.002414829037643275, -0.018805706523693632], [-0.17255386886456317, -0.015226837160843254, -0.0026787184000690418, -0.0019070639533620266, 5.339721000460695e-05, -0.01680997084667078], [0.4330421459330063, 0.11007388583288989, 0.011996179310740632, -0.019308826902633993, 0.022754669194724055, -0.12862352955920398], [-0.15585560657077688, -0.01451677122947156, -0.0017533273653236488, -0.002414998515176522, -0.0017162664002225835, -0.03536609193453269], [-0.23157155842040847, 0.0033436129442810047, -0.0006106374809785962, 0.009997092173001883, -0.00070443943988113, 0.013712596890651678], [-0.1860312804415133, 0.0011043926416072637, -0.006712706720746036, -0.00253206363598766, 0.0018869070708770102, 0.003784751085762585], [-0.1775366138367146, -0.0028923444663198016, -0.0028223156487533346, 0.014452170621577979, -0.006597247955610906, -0.028995540606070663], [0.7386043070200862, 0.01182257521840704, -0.00047680781109610406, 0.0033582643578320094, 0.005037085784417531, 0.028116563734447424], [-0.162509221658779, 0.003364816769095347, -0.0036181615556991264, -0.014544430314796434, -0.00478485290441627, 0.028341849664596266], [-0.19102222112032005, 0.001321552499262116, -0.002418586053090474, -0.0036309920249654416, -0.0012156106775696395, -0.014657204638819973], [0.7343858646223681, 0.02527876958154219, -0.0035501847633321345, 0.006947502852062329, 0.007181081131004429, -0.013651763582375695], [-0.21889613402703523, 0.007132102481650718, -0.008252857396714025, -0.005576068657070681, -0.0012594807284202567, 0.01768577166092303], [0.43557891148952976, 0.0071407556450255125, 0.0051977802199625585, 0.0010065674169520294, 0.006752763712289685, 0.18097599929401714], [-0.22691281257992324, 0.0027991999283228683, -0.001393617559085221, 0.0005974365413615212, -0.0037151813579444793, 0.019458308360601365], [-0.20678073730168287, -0.007241270220727373, -0.0012050063163672688, -0.003379620514047076, 0.0009066733489727473, 0.023949961003852226], [-0.1661239307303719, -0.008652829598339256, -0.002570739650348538, -0.0026066421978242944, 0.0007067499799230651, -0.03237566981854268], [-0.21191535510493148, -0.008810964097911354, -0.0017193657894359934, -0.004628875419516964, -0.0028955729123772917, 0.023611589906806372], [0.6986224515410417, 0.01842869084091998, -0.005151595844564779, 0.008677057855103459, 0.00027893089217782287, 0.056644464715321086], [-0.21986921573781165, -0.003795956667385587, -0.0016471690168010803, -0.0007917547633518788, -0.005401309738220102, 0.020446847365011785], [-0.20234573439931328, 0.012422490002714613, -0.000457722712474937, 0.0017551880180752297, 0.00028722106235372043, -0.019399537209450587], [-0.17263436096860565, 0.002431420181582458, -0.004252480764292725, -0.0017709899074267467, -0.0007624204751269418, -0.027550896748300746], [-0.21367086690657258, -0.0003987301107980279, -0.006290388080010871, 0.0021694985104136278, -0.0018874917737514772, 0.018689089471831052], [-0.17174443799212843, -0.00543287628979778, -0.0008690184065947749, 0.001019947469031504, -0.002895703610594138, -0.03109286507731183], [-0.2228886501845927, 0.00131812386683269, -0.002992314962916036, -0.0015417801776960042, -0.001824318025721214, 0.021262272817426667], [-0.18643017731903322, 0.0011675600204451367, -0.0025373210184098925, -0.004275803410597477, 0.0005975285535524798, -0.01518845349262342], [-0.19665752305681086, 0.010077526391281559, -0.0017790525587991816, -0.005357721244134646, -0.0006369058188792719, -0.017312990379324003], [-0.1604042964339208, -0.002337003401020614, -0.005621378011110877, -0.00423443132319166, -0.0022357944595635705, -0.034290158386696114], [-0.16593113504055168, -0.01145240683125512, -0.00409011241436339, 0.002679541427621232, 0.0013895727219072006, -0.03221852187886156], [-0.157579809191059, -0.016220184442083674, -0.0014342738356711744, -0.0016032306817246897, -0.0014402455908376448, -0.03334531827412765], [-0.2154617676902625, -0.00642595857606313, -0.0045435468334533964, 0.0020578931919874566, -0.0018457586552087213, 0.023302471896333922], [0.6785472979111505, 0.018943753000663956, -0.008760241007864792, -0.00010588989429543182, 0.0007464572969946132, 0.08617624174096884], [-0.16143811513777429, -0.011762769577509442, -0.0013453842785845754, -0.0014407560532071206, 0.00022722969436625434, -0.03586326666279411], [0.6051505952426433, 0.03985055260570208, 0.00424933789450584, 0.013776715818553194, 0.006602551864168405, 0.10727500847918933], [0.18833719626669934, -0.004654637856811661, 0.0033609573995051615, -0.0013787569615513976, 0.005913382004669174, -0.2522233117115511], [-0.22586465512040071, 0.0031797179784518862, -0.001962937044527242, -0.004220947779116378, -0.000531197282435731, 0.017733352581361495], [0.7633779084821843, 0.02013739576741733, -0.007644189623101511, 0.00381815263207138, 0.008283250268872975, -0.046192214497141455], [0.6209915387082519, 0.0355290167668208, 0.008077596507680343, 0.005243035533818924, 0.002449918524715872, 0.08179222729204516], [-0.21222646278356316, -0.006483015732716381, -0.0030236992620961338, -0.004677155904857947, -0.0009733411276336968, 0.020717008144201], [0.5422220627921631, 0.012760285075985653, -0.0029419882351741735, 0.0065950435099518335, 0.00218362773226749, 0.18778505947889643], [-0.16471611081723211, -0.002341524057241946, -0.0020760726937902466, -0.0028278495078997152, 0.0009896408554983483, -0.0327940029376952], [-0.16754397468384766, -0.005280178384363188, -0.004767957789353004, -0.0007065753480064695, -0.00313395189841582, -0.027690423911517423], [-0.15568754506771376, -0.01429332668934262, -0.0025227383912754175, 0.0005070317879045393, -0.0031439660732332834, -0.02898251758184336], [-0.18800808285306625, 0.014043410903004943, -0.00527167011587242, 0.0009522635447939176, -0.0019534330335545876, -0.031429155111971786], [0.4115531467269884, -0.06321070235258949, -0.04754886127094321, -0.016312110778666684, 0.0015832315920649026, -0.20890588741660743], [-0.1621789191641458, -0.012309642603084673, -0.0015119313981962748, -0.0029870532299887415, -0.00031884956935587257, -0.032316666050732176], [0.7498349253010895, 0.027412409322343328, 0.007573392236022293, 0.0037247085882521204, 0.006322748289535801, -0.05071603633509634], [0.17773717179550572, 0.008592569523122746, -0.005158531779382797, 0.008435524859353838, -0.014107708830549345, -0.26488543534690223], [-0.17464137197287238, 0.006023027090027537, -0.0008089100430631589, -0.009764379918901694, 0.0056834843427242656, -0.027121101198594753], [0.6822511802109293, 0.019219160997540227, 0.004536926987329572, 0.0007351037588049665, 0.00627307586657579, 0.06656788551215448], [0.7451929903738308, 0.026071723028720743, 0.0020394172895607998, 0.001809974773482435, 0.012070992560985757, -0.01584130293278443], [-0.22178223200271815, -0.000394960593929222, -0.0029464114067128755, -0.0003324968734054429, -0.00044472170724689134, 0.015968026885088072], [0.7344886438967916, -0.01559306622679173, 0.006941582235872079, 0.0028447408795656054, 0.004346263621535182, 0.04692281598518275], [-0.22612792075496804, 0.0017384562010623276, -0.0013388012600715042, -9.44886514364083e-06, -0.007056780218300812, 0.021127828230754937]], 5: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]}
data_raw = np.array([[11.40101972017505, 424.488625127534, 541.4791230865201, 648.178488329185, 20.6408564261623, 7.0], [7.8310220745672146, 667.6852364875508, 655.1736700860375, 544.2226963961857, 45.44245450424639, 23.0], [7.367297813930439, 426.9350595245576, 579.0215563243348, 748.8220962325847, 87.83204535628637, 3.0], [7.367375601432784, 382.7127424935474, 440.49778399861407, 411.9449357711013, 79.95591729924578, 4.0], [19.731647798996462, 661.1570615794242, 592.6573874665174, 651.2882207762274, 95.16111136508007, 5.0], [13.204647216004693, 323.1766548410095, 588.133028112369, 857.8229510777132, 80.91805512693574, 4.0], [6.307648242089852, 531.3413289412922, 717.0814014525783, 582.8940125238177, 140.94927398447112, 18.0], [11.694111035998883, 790.3496968578515, 483.3417122156807, 540.9018167746584, 26.18175245460435, 7.0], [9.861202997613862, 297.18572021570185, 717.8783379306532, 758.8274208163067, 104.07882894589444, 8.0], [1.9317015182843515, 592.1502072800507, 588.2603025480067, 674.5147640880698, 119.84031218490418, 14.0], [4.259513305584991, 578.931316733873, 467.0469994902005, 761.8170923085399, 116.04615375075198, 3.0], [10.28304438021449, 475.6657953343293, 442.94660255076855, 655.7747728542909, 118.61818964003554, 8.0], [18.71203302698996, 374.0526039740158, 832.1266233490236, 300.0, 122.93347725172804, 6.0], [7.40705961573834, 803.6992642235963, 580.8100772597471, 826.9870115638548, 76.88060340178328, 7.0], [5.993019401300914, 350.6078410582585, 712.3804940076586, 960.5039573717412, 61.95548729069129, 14.0], [9.126191118222122, 46.99027471759496, 395.0256726470244, 842.606151025443, 33.95954981915983, 18.0], [5.763740046198127, 660.1333979077584, 916.523386555668, 300.0, 84.12191498960914, 14.0], [7.098021162261081, 199.67416255804147, 707.1893554040554, 717.663231875835, 71.67400628135434, 4.0], [8.462996672712874, 672.9702862137718, 569.3471743013749, 579.9881284137431, 129.84826440971412, 18.0], [4.1136360178229365, 833.725811960861, 563.2600970573388, 487.3634369582436, 86.37325669546595, 6.0], [13.59368213924466, 657.4374695996686, 688.1517753963595, 668.409824245741, 64.08782788317326, 5.0], [3.611479382156933, 767.8528254718728, 681.2618024070057, 627.6579057650418, 59.34393638116698, 18.0], [3.3044260069991687, 1000.0, 621.9035136476926, 726.2986706419762, 154.66460161351384, 16.0], [9.603938779159265, 741.6901156522367, 601.3054509842676, 658.4870283958181, 94.33741055025268, 9.0], [13.00316423311724, 582.6433481563589, 667.3280500831099, 481.8781839666535, 71.5548661017817, 18.0], [9.460521819756936, 908.487045880912, 390.9955736303788, 563.5097334885544, 98.72125423316064, 10.0], [7.94315810119725, 938.6938862116464, 555.4470344862897, 539.3062165512476, 151.90667468495212, 14.0], [7.054618815573708, 942.1929458148552, 402.3763370048584, 392.5275234883331, 83.30958073101334, 9.0], [6.345469529212034, 376.9641212009993, 672.5510813643006, 444.2533034069175, 115.6103212834459, 10.0], [15.335154812944408, 161.495410375399, 518.9023234760017, 618.3396995172209, 130.59383111990974, 22.0], [10.341362320400895, 361.7019541412783, 411.41063055849867, 863.5631112960918, 38.2572832445028, 2.0], [6.6750856412014485, 675.9104039482938, 504.74561711772367, 569.6664380451808, 75.84762743991382, 2.0], [5.462283972306911, 539.7897363330264, 617.9227441758942, 536.3853837870815, 126.8325988447659, 17.0], [12.145396529152393, 448.3797537057882, 575.3725940933027, 472.80697899907614, 74.02089682692481, 10.0], [15.134264365532715, 640.1203497012339, 459.2212129160672, 654.5804742493026, 107.4387619780185, 19.0], [14.38353144875192, 132.29872958168323, 395.4157677923261, 826.8889001915189, 72.2455520262557, 21.0], [4.856278204950636, 535.2923246728853, 608.8719344436603, 950.4024149612889, 81.63344289464219, 17.0], [7.017360637516271, 779.6144112119692, 530.3078271899446, 692.809906508253, 77.97908583178682, 14.0], [6.266300933864867, 637.3289111395571, 569.0303148163139, 670.9990613954652, 114.10010461087496, 6.0], [7.599460591986949, 672.9524554584322, 580.837777501404, 568.4790339119685, 101.04740342598778, 9.0], [13.522394513347033, 899.3431974881287, 651.6078824610382, 593.5150067763483, 97.9657833503306, 20.0], [17.763539782195124, 483.46417878158326, 701.6335802123896, 744.3762636576055, 170.224857135765, 22.0], [10.566022505025565, 336.7270846624159, 617.5811041150132, 602.7687001905381, 147.190922213602, 19.0], [5.586648450114115, 797.3787606662605, 561.5444940288428, 625.6576329248879, 35.562691383386586, 9.0], [8.347487302068577, 792.7801577584818, 602.9716245714964, 646.1979087874363, 81.69770980344354, 12.0], [19.598820236501258, 639.5729186889491, 559.2915639288454, 561.3757833638786, 167.79391023820975, 14.0], [8.996334796672933, 459.907249854388, 402.8935975640255, 638.7531111383673, 77.76229907935874, 22.0], [7.064272616770484, 500.3449127519963, 731.033540554749, 347.08908068137976, 157.63237957625432, 17.0], [7.436124158281839, 180.80676196722837, 708.4126634985364, 732.3253824527693, 82.21019555395219, 22.0], [10.538789669214117, 778.2366325518008, 838.6000792296716, 1002.7657758458082, 87.98007288360311, 3.0], [4.967370369845785, 769.412827382286, 873.6599167773129, 567.19345772554, 120.90714077407436, 19.0], [6.170738683305962, 172.07931238330406, 458.8775841296688, 706.0755717502504, 92.29378474976492, 16.0], [14.266335812742472, 716.908020976534, 559.7520095545406, 777.7364480369276, 109.83873918722858, 2.0], [10.369109099219038, 601.0254137522521, 901.384751624417, 842.0322909076192, 142.24846097922915, 20.0], [9.050738370074914, 244.50647240230091, 595.1339261301318, 610.3517766889731, 117.76402801927308, 20.0], [14.661850353920224, 198.49994973188265, 629.3053083056102, 379.5024119466921, 99.24596396449098, 1.0], [5.36532340582405, 593.3733173504306, 417.46440579665045, 690.0847926273144, 145.20580525152778, 3.0], [6.933075995653091, 400.5739715232705, 618.0346357892129, 669.2943035981641, 121.30062427625298, 0.0], [10.176150550526655, 954.9872126374177, 831.8328449710907, 598.1500437985995, 72.99370510707956, 14.0], [9.971482447071386, 738.1495986622266, 754.3950972124234, 629.9731815673489, 154.4022570033206, 0.0], [3.0682444956389165, 618.4232841830569, 526.8018454177748, 671.5520667526597, 114.27096776150536, 5.0], [6.518555049138259, 630.7697510005986, 552.44203601179, 704.2642988155776, 88.58969693390415, 6.0], [7.717965032321437, 832.7167950917919, 464.2919649006884, 697.9714565359877, 34.375472462421584, 18.0], [10.82366106321256, 711.6295229318345, 497.8604231700353, 617.3471510748307, 53.67270859806, 11.0], [9.95117727525608, 624.9416096240057, 370.7745685456237, 707.6418252478927, 146.0040473119981, 0.0], [8.15019211392399, 254.9347059256225, 589.8216390408309, 607.2453727587707, 147.1899575591135, 6.0], [1.9210633859562265, 1000.0, 781.2132815916493, 746.3038252302979, 97.5585220619972, 18.0], [8.395531504499337, 224.20431067294172, 642.6518308912499, 555.8835746785248, 41.89267472073541, 12.0], [8.851956381716704, 421.8643157249589, 763.9486787958879, 660.9587288224344, 62.39074239048183, 11.0], [25.0, 86.31793089750062, 728.7899281111198, 406.4392126824156, 46.28694943851266, 11.0], [7.5670890772229, 634.6995396923155, 534.3830292512846, 594.5134295814279, 45.196662202587184, 21.0], [10.208075449983204, 537.7776093212019, 508.8592006312261, 735.1233081694968, 155.8040710681072, 12.0], [16.00668426822059, 691.31910959119, 513.4578531778253, 494.9397699937897, 156.7975372524342, 13.0], [13.09657095280027, 65.23359222389155, 393.9525661869541, 601.9849285474875, 61.37458539303291, 21.0], [18.16311271691815, 778.226416845233, 603.879627935469, 300.0, 94.73795072307202, 11.0], [3.104106339557374, 298.14114963259635, 251.4642724107496, 444.7888837202016, 56.45611520204592, 19.0], [4.333245049342012, 462.74870015903366, 670.9698611143663, 484.0288165829147, 94.5428586049974, 11.0], [5.902992334809564, 732.8928171702662, 608.625712286265, 665.2626576157458, 144.17911650107288, 5.0], [2.7244646899646745, 565.3166710329749, 656.5727839793808, 700.0113518589069, 127.36157152409044, 9.0], [8.896652179981443, 582.9856391667159, 726.8419539537763, 641.5916651089705, 119.25988296491862, 19.0], [4.574097990596252, 699.7947049763499, 623.8553506006624, 592.6109925668933, 89.17460788211406, 23.0], [19.4653215439931, 607.1976599846811, 795.7886085185708, 593.134737747688, 132.85901347989093, 8.0], [5.059892292249896, 414.72302436282064, 674.6334973142652, 757.4068623560742, 114.40006239749994, 1.0], [6.958589771296762, 458.3827179411928, 509.144778382074, 681.9229770324579, 107.11746743947158, 6.0], [13.529437329761729, 552.9696696629557, 571.4746749424945, 599.9081605541957, 91.53771985400644, 8.0], [3.582042520018989, 276.11097661341626, 328.6450767899836, 843.9949242861888, 104.55118818543998, 11.0], [2.588105301272659, 749.0021648471675, 491.73714714334005, 942.4962561494152, 103.6475473360798, 4.0], [9.534971100438698, 572.163482701871, 554.2804312879931, 463.1386324649495, 78.75705064777218, 0.0], [9.964686744722917, 518.6940669804922, 665.55283314692, 728.3765105315767, 71.75250398818201, 21.0], [13.305490450277103, 798.1241397578677, 509.9566765749827, 1113.4834411529578, 65.09279119008829, 14.0], [11.561683998736257, 469.9715891971576, 450.4483183278683, 692.0308311575292, 67.0141066461922, 7.0], [10.181230927962869, 454.39475071189577, 373.4961716390249, 667.3163954246414, 89.35691373364845, 0.0], [5.450250389377923, 378.8428089990089, 528.4298977688018, 641.639909015958, 84.41945303013256, 17.0], [9.805440333026056, 585.7667522314838, 834.1022162841516, 723.5272276236112, 76.37798760192331, 12.0], [10.158250960840345, 577.1125270828787, 632.532844187644, 763.2348749824824, 74.15655575794503, 0.0], [5.318295190163018, 333.4289498725233, 712.7495294826215, 463.28496887462256, 103.7564007834895, 4.0], [22.46766148039381, 455.57650572164994, 491.9934920802901, 646.0274557566757, 136.32822341522916, 17.0], [11.256590826736945, 335.0994328701918, 411.1860287877338, 309.5753203253886, 94.67252956779652, 15.0], [4.386192862525918, 323.7828865461274, 453.4707043094319, 618.6025075067749, 92.71007599877692, 9.0], [13.342496555409028, 67.52554206828825, 588.3442669010835, 514.5967109266891, 66.95339921859892, 3.0], [14.62238877520445, 442.2724365796112, 360.7045387824892, 700.8766368703226, 88.30790564627877, 0.0], [10.877204200739945, 74.44668347999715, 479.2091500308275, 401.5429209288461, 130.71009021828107, 22.0], [7.314195832705934, 384.2960784151222, 529.6367176834448, 573.1524034311358, 126.12383310526916, 7.0], [5.16954038879354, 346.8263385305979, 441.4889632724039, 794.9816375780919, 124.03921232823006, 4.0], [8.136740047445286, 762.973764316401, 345.1580379210205, 548.5500272833625, 123.07606487659528, 1.0], [10.443074954179856, 246.2662984225896, 634.7584437851709, 364.25963338731714, 108.10197666694503, 23.0], [8.601456213683214, 309.7034037963811, 605.5045733861477, 688.6299729693477, 107.03621490448988, 16.0], [18.60538455427127, 404.6189652021069, 693.4835841156155, 562.0206510361046, 102.3425098005432, 6.0], [12.238139803187671, 354.964087124306, 466.7489100897069, 622.1171845890656, 82.49344159842457, 16.0], [4.7921827948751385, 842.313699355469, 691.1907291656966, 540.5195620306112, 97.51098195846528, 5.0], [4.071432525194022, 215.3581567375951, 783.5165033384262, 946.1595116720044, 101.467990511176, 21.0], [11.310980060523727, 405.14887525353953, 631.4657678684226, 416.289089621973, 113.99505809948155, 14.0], [7.418065875655078, 583.0163122911016, 526.8076446664007, 1075.024341784141, 98.00769461135478, 4.0], [12.834599688889597, 720.2958357605016, 617.4096779411093, 591.4045203872972, 158.15484142917254, 14.0], [11.25284962137006, 404.9396110174492, 521.0655132845308, 677.5169864327203, 58.70563408826264, 17.0], [8.158389779063043, 20.475099874051864, 501.0859730785498, 563.7805373593275, 117.634270993121, 17.0], [4.829141028392318, 462.4739446036238, 681.9912467403991, 557.5049660216819, 93.7626888008284, 20.0], [2.812832246914509, 838.0071190451788, 583.3139616255362, 532.2755144851051, 95.06438846379626, 20.0], [9.70170101057658, 570.5468518576067, 744.6114784140035, 740.1886943065454, 70.62272653067576, 22.0], [3.538642690989815, 559.1044655899832, 546.9092412907715, 438.3800459559416, 125.58029801746056, 5.0], [4.697838705918586, 234.2455021057509, 559.0590503195987, 704.6162816364003, 114.58524602963472, 1.0], [9.362106373696353, 208.58210242783383, 532.0186805722234, 322.32608974222865, 120.00912190411076, 16.0], [8.841135991444363, 504.5104121585248, 465.5779873575354, 691.0293156712646, 100.07736214456894, 20.0], [3.845756527465893, 143.40526942964806, 697.7309878339909, 777.3713908898287, 85.25315556079325, 3.0], [15.53630601456224, 695.3405555906276, 862.2853997536986, 458.8221339038946, 93.00315354366629, 11.0], [15.309523782511937, 288.33843778340946, 554.1327313720618, 751.8113902578116, 108.0039695016692, 18.0], [11.517551227937886, 493.83567981712656, 518.3404867589993, 657.7078115936482, 34.83271481240778, 1.0], [11.509579836099926, 28.558270311439458, 510.5611819551492, 889.7639683036759, 121.18798602726372, 1.0], [11.517630487743231, 401.5495328891233, 518.4320126879544, 521.7190426533365, 167.785784440492, 13.0], [14.552353851509997, 203.53486321744924, 797.9740423488962, 1072.3978855584096, 109.41005140860428, 14.0], [12.409886982276369, 0.0, 513.782337628813, 300.0, 86.57220127376678, 19.0], [5.098450800949776, 635.4605223340793, 529.7679838932552, 600.5744299045812, 90.16294235962192, 1.0], [7.354677188226571, 910.2898847728934, 451.15748974188335, 395.21454340656, 116.22627052802508, 3.0], [25.0, 782.3683973359866, 635.2378528635685, 927.6581265274648, 87.25134010054644, 18.0], [2.0223981610278976, 670.4226724759512, 591.9324525418816, 610.7448509276245, 121.0602581401753, 8.0], [6.297155656676839, 396.4697080136136, 463.5385469036928, 624.2472383946076, 89.77577848609268, 6.0], [15.582316100651456, 701.4725571369565, 492.86044832183046, 716.4857147127819, 146.4255039987301, 12.0], [5.314665180502463, 805.963299643432, 783.9520044642669, 649.2792220509069, 145.59788408066564, 11.0], [12.600047972001484, 534.2054574178234, 603.9249307522522, 560.9357665397998, 85.09490996177176, 2.0], [8.773694657129246, 203.9099569219943, 612.2022474536757, 461.5029124056428, 92.569284897836, 12.0], [5.561154191829509, 375.1539253521572, 612.6784590031274, 675.2770103500144, 94.3289341436498, 14.0], [1.7226148943546116, 925.674122086984, 767.3550465853411, 706.8250449034217, 94.7310102270878, 13.0], [9.545218269012494, 481.3139579624584, 474.5742765215148, 754.3065854602594, 66.57712864817971, 23.0], [5.025788879119162, 0.0, 552.4325375788254, 936.9090275459783, 62.38217947176943, 8.0], [7.947701552457319, 132.7691340824516, 579.7241752004987, 323.9346643379012, 132.49272714643868, 12.0], [11.453543144413295, 393.8878242143796, 393.45890096523306, 639.8424203970943, 127.69361042612502, 22.0], [13.90400193226072, 758.9563096043921, 490.0630180102388, 828.7606753732141, 127.17562142531814, 21.0], [6.284305182318547, 431.719411898272, 725.6009792737125, 631.3216363341703, 92.91864899179012, 3.0], [5.554365994325899, 763.6079137433151, 691.2256195596141, 300.0, 151.76035309969086, 20.0], [3.4948146254994543, 368.4594116131277, 712.4480250122438, 456.34406377251184, 94.32593069976298, 7.0], [14.284450000492058, 8.926869441096699, 529.3091732962318, 699.2854774515979, 120.88628521699222, 2.0], [25.0, 377.1097103504789, 544.1165711857396, 693.8295942325011, 53.34133956395048, 1.0], [15.145489540532267, 586.2362503865712, 385.43214059543607, 750.8779547511531, 88.10794005506449, 18.0], [17.013328984382078, 565.4875873353283, 711.8321450779572, 532.8486584677723, 90.3253683327855, 22.0], [5.343812418425661, 186.22306268584111, 731.5982722519316, 688.9824852378947, 93.03537385935157, 7.0], [11.068998663975954, 412.2009495490587, 730.070515984273, 508.1205038155513, 55.23503096012942, 16.0], [13.25504171598474, 1000.0, 792.4909504172688, 688.9936990465536, 116.11642783472772, 12.0], [0.3306200644929243, 191.6827934257808, 650.6530312187887, 657.9818295655309, 125.85830933910134, 6.0], [4.221625728184271, 665.1808486159372, 786.825820385603, 526.8709969755644, 72.68421832631253, 8.0], [20.221755563701706, 0.0, 575.9430469010546, 883.7275446144031, 60.579754510360125, 11.0], [3.029381317527532, 715.9479561424155, 387.958756199334, 394.1546074756419, 71.93175125318233, 13.0], [18.497863329401223, 276.9186524528883, 703.2869612688141, 628.7568525484209, 109.86496007420544, 13.0], [3.014417945737856, 600.5220645507351, 383.5333048789565, 523.5641653513535, 77.10189935274937, 14.0], [4.363341086708779, 177.11859955794966, 620.4242459392038, 925.1303053784412, 90.97804251090872, 2.0], [11.183028488792514, 572.7132000441768, 714.6388562549107, 739.7597011430628, 62.79617765891915, 14.0], [8.90331141790046, 272.83742606464045, 638.9110555242237, 552.6034436944416, 154.3785921650439, 18.0], [6.67405949158688, 321.2379542583437, 873.3891684698731, 608.0427066019441, 97.19595884488348, 23.0], [19.793885160994787, 610.4965015987576, 364.0885207023626, 929.1886060970794, 129.4914598767989, 19.0], [3.5617168695742323, 392.403630900746, 590.0567192956902, 508.05865524049545, 109.7950950271651, 13.0], [7.764489292895789, 595.0953620540778, 602.0946558303187, 385.33277416541625, 93.74359835104792, 7.0], [11.991601174133848, 452.2896703391623, 438.7076620581104, 693.6510293695721, 92.19013025332168, 4.0], [10.087508692560156, 516.8734203118188, 461.55206511308575, 697.046055111173, 73.24268970086644, 18.0], [5.675449183981079, 611.2716178383504, 498.584566700808, 749.905186008479, 81.48610547348824, 3.0], [5.809211499739476, 412.4953863522148, 468.0423247140885, 750.8938641307928, 64.76593005204487, 13.0], [13.78744984094137, 250.98755349955647, 587.0406419803359, 915.60584761007, 61.49402084201165, 15.0], [14.150333131440275, 366.9219335175525, 527.6442119817408, 392.2793764435946, 157.59126252654056, 19.0], [10.241985655111169, 798.784794958359, 638.149006091481, 469.5795208032, 103.36737095245827, 23.0], [4.893055093430428, 529.0432710191375, 677.9316559443675, 749.3305000919281, 138.97151725112178, 5.0], [5.92802840068101, 288.7753866473816, 669.8134751662667, 484.156174742834, 71.17813228891934, 22.0], [8.424575760287945, 600.3901520969297, 550.7848505778293, 431.5118040110955, 57.45189222605767, 5.0], [9.161237388922912, 617.0788962347846, 634.1370755357367, 929.8766828427906, 65.66665367123447, 2.0], [11.72338825699778, 629.9997689494526, 724.215469941208, 407.0086802503569, 82.33784344730913, 14.0], [7.519767233402073, 255.5406999071786, 653.4677497532988, 597.3091944334031, 96.0949979547848, 2.0], [13.614159021129565, 329.40225142852626, 823.0773233332104, 641.0948823152893, 90.4719014946802, 10.0], [13.529383462848685, 330.7926959032471, 717.8614784120025, 846.3704962944917, 71.80427094540114, 2.0], [17.334585308137836, 409.83802198345205, 806.2088487178154, 369.6795412576294, 97.63145027953291, 9.0], [8.64356179319192, 634.7297962859336, 300.9188246483922, 712.3212727259373, 100.2615837361984, 17.0], [10.341851669342786, 574.0462011827534, 611.6011345368279, 516.4434412609883, 116.06829282654736, 0.0], [7.871167018784641, 593.8042392892906, 561.1280998857338, 744.1416818888675, 139.11296909037614, 15.0], [4.932017697312718, 838.5047855928808, 585.0024323622941, 513.9788410959711, 84.0831863147323, 15.0], [24.802080021871262, 672.2227331229075, 503.4048695875033, 885.8459384278666, 131.46441780903896, 9.0], [16.128500998874994, 464.54959375097815, 647.5439280461015, 514.8544972360617, 96.9781278028396, 0.0], [13.374750331937603, 204.02870598687517, 382.99722810991045, 300.0, 49.18043037746656, 3.0], [8.469496714159394, 321.3632478822824, 648.4124235445549, 531.6726125388049, 148.4981243118751, 2.0], [4.650970731591148, 434.5737330804433, 507.3946535783351, 610.8049834146658, 163.17609325699073, 4.0], [14.710440633203888, 707.2824568340284, 539.9536691855479, 812.3530809743463, 113.57289434154652, 11.0], [7.787710282726582, 493.1670164734767, 652.6202123478886, 461.2377754152619, 97.69354893257396, 11.0], [10.87812895800186, 335.9510112778892, 609.45332734396, 464.2030809328887, 86.8578481950651, 12.0], [5.913515931898835, 316.77677856819355, 726.4705093687744, 794.0866855563477, 81.08156091546775, 23.0], [9.878501379203934, 482.6054477696935, 483.6595792984813, 443.7162323056691, 131.3210375944255, 11.0], [6.148930915692032, 722.496890554275, 590.3153217339448, 623.1524837912464, 94.4818028585548, 15.0], [6.300969500879374, 897.7548264437793, 706.6527740413381, 626.2023224796482, 145.21819391734306, 12.0], [9.80427226571014, 448.5578504925831, 643.3439692501678, 682.6369304514443, 96.04587245987496, 4.0], [2.982614591697423, 488.8142048093505, 651.0713104155355, 537.52276844261, 82.62959919729778, 23.0], [3.0891845200040287, 196.51457346065428, 732.965143005988, 510.9508419988156, 76.96098606464801, 3.0], [5.302705665932204, 984.7981422748666, 551.2509061356567, 700.2341662628438, 72.68851744116506, 20.0], [7.465839781186251, 149.749290460979, 525.9802947453275, 515.6006220245322, 110.42437479218138, 14.0], [13.845506870943048, 792.9155779451376, 669.1085324318803, 446.0046214821238, 56.165199354259656, 11.0], [7.724550805032378, 408.6660117468823, 582.0682216786693, 834.1699981777529, 70.4498754999124, 18.0], [8.436956619768734, 350.15837653992025, 734.6644281237259, 723.2143072068774, 134.7344787752321, 20.0], [7.11205815333366, 222.49910702851832, 644.3728124538068, 915.1253593648008, 66.36352067526566, 18.0], [10.33325414848666, 1000.0, 525.1979183651686, 767.5951476825419, 120.8811476597005, 20.0], [4.899420292620703, 555.3729058195986, 640.6060832173371, 459.7784522552753, 33.10964565497554, 10.0], [7.977522665174579, 505.32010927601584, 491.676040293296, 664.4553860089993, 138.50261126527255, 20.0], [10.809344625025444, 721.6830898886355, 713.6314162028468, 497.6557617922631, 56.8930448291736, 5.0], [9.758728112217534, 337.29181859250923, 432.3417232142218, 402.2585604922574, 68.70818769575723, 8.0], [8.599305192025303, 500.4614888077514, 746.6478329035422, 582.9200910395025, 115.41027733958975, 0.0], [8.662055030823048, 514.0840717030302, 638.5051813421493, 797.2299312948682, 103.11224402624146, 11.0], [11.409161752094152, 371.9635669129667, 580.2526836452791, 691.2369066181261, 121.84962308284422, 13.0], [25.0, 799.9095197964784, 691.6000981120652, 441.3658803111844, 56.652121122272746, 5.0], [5.118783023435433, 445.336397126024, 661.3591972994826, 577.6487637180895, 86.39583771999384, 0.0], [25.0, 111.71698665779265, 759.6458022789673, 628.570012198703, 76.48765372368689, 21.0], [4.968081968105779, 424.6256591197816, 758.527408602416, 396.6331319043733, 61.6943360554273, 1.0], [1.5510654939699664, 562.2946679284744, 642.2748532999543, 756.9758547872918, 112.87692375561096, 23.0], [6.070258065782579, 276.6266915280979, 495.3875174350108, 615.6897460611872, 123.29154339820386, 10.0], [10.446686516842348, 296.35130170739023, 535.0189199464511, 441.4107708601914, 84.894013848784, 2.0], [22.571408197606143, 363.4352404684898, 591.4023737026022, 432.9239609388399, 140.6813389788851, 9.0], [4.646926258787154, 805.3089150167401, 729.8066033274113, 815.0995917382417, 71.42899337364972, 5.0], [11.370651551420323, 378.2104943385735, 545.233624639503, 592.8780451781421, 139.95401972411122, 5.0], [3.326553988728959, 857.2792546377577, 555.2471375483836, 828.5528043434617, 111.86717179274795, 19.0], [22.12747610871935, 346.2623904629161, 821.8020755120913, 885.8233219854073, 58.29298443574968, 20.0], [16.29948487145551, 581.2535340641355, 587.4841786346728, 803.3672409283886, 154.44572801248006, 1.0], [6.308924525756411, 388.4393314450622, 523.8321731680073, 569.9852380073764, 81.66567400426918, 23.0], [17.7433764581829, 688.520792572618, 693.1975234072977, 300.0, 131.15353218843353, 9.0], [7.948681364036935, 220.43146135868017, 593.2638860448499, 696.9494456802242, 99.36947365788896, 22.0], [16.7736225109368, 685.7148698943079, 433.24810038797216, 963.7364554667822, 106.23323673355496, 5.0], [8.560764035781313, 528.467350077421, 724.9096899368849, 644.6914882739262, 122.03620341476925, 22.0], [8.78119594322902, 340.0020458056377, 510.355281516087, 351.5276721801279, 120.37553553407012, 16.0], [6.390927401708125, 458.44462882246233, 605.2302718067491, 548.7682255354823, 84.73560310629618, 13.0], [12.219426022563432, 883.4637352580437, 591.8233063455173, 622.4400087849792, 110.94996787815784, 4.0], [9.17750967210159, 534.4611262201959, 627.6121408163835, 485.3017148870304, 108.69799546095506, 22.0], [11.513745165563314, 324.3179960218403, 337.0711112922403, 300.0, 101.40683599067737, 16.0], [2.7650621946108425, 198.82667929297185, 559.7880625440919, 635.3223844677312, 88.90050398510813, 1.0], [17.1026007181702, 626.3510683577492, 734.9804472138842, 532.268563046356, 101.34694499582744, 8.0], [19.47634892477013, 548.9613117526713, 751.8534623869984, 915.9037491349872, 154.0262223858758, 11.0], [9.152185168815588, 673.7551737720559, 481.0451542031615, 665.1550231974194, 86.14917943657528, 18.0], [24.46687697287825, 134.44381507753943, 710.4717891254179, 720.4368345347588, 131.19762259412627, 21.0], [21.385182349291203, 206.93488677712503, 508.62842946892994, 632.1370784755911, 76.41090728250607, 6.0], [12.369582462983784, 786.7197998129748, 555.056128873526, 664.059995476078, 68.5605601303177, 8.0], [17.86931724799177, 582.8354116948351, 561.3618760248899, 402.1551067456785, 193.29198035339, 20.0], [15.345215790432622, 633.3502128093904, 613.056445164649, 493.9750542168066, 111.2007259096773, 15.0], [2.2474517230370226, 259.3282419730117, 720.1639747549035, 813.2922051979028, 112.0254038386884, 19.0], [7.201526438426888, 254.7532004177704, 631.7501443234934, 681.4192954595295, 112.85683315502608, 10.0], [12.858910723164296, 747.8146645480222, 584.8641887611386, 590.223962317623, 137.21899119603765, 11.0], [20.186520052105237, 759.5944136912872, 716.0773479645859, 722.9256862198811, 107.2343004385326, 14.0], [3.162383157130187, 227.92012824273652, 503.68063317195424, 725.879286476089, 119.48665617671575, 17.0], [2.368415426638117, 180.15737603678872, 590.733906783846, 948.6341949416312, 63.24295165230432, 18.0], [8.246343304035479, 472.85175732188895, 719.4937474053035, 674.4599515828367, 124.51770611733356, 15.0], [1.64766436223796, 300.1628866559985, 663.5167454148688, 490.6238636734549, 87.89718325887377, 14.0], [8.076050106348838, 294.9557802384936, 544.0435227066105, 632.6455275626498, 115.15227446575618, 4.0], [10.595950796682054, 187.15543839315183, 418.4451880079679, 741.1348440682065, 43.54640135915913, 10.0], [4.503925138211838, 382.8915379299083, 637.3029828953773, 799.3488517143054, 113.9053877709534, 14.0], [8.209971522815819, 211.32631434178356, 410.55897577164126, 407.7452191808181, 27.74310935675514, 17.0], [14.560862860703155, 226.14977424769089, 574.4998362370602, 688.056007923772, 86.7225358869147, 12.0], [6.051527863418839, 574.5979616415761, 741.4738079084001, 515.4117608752825, 43.83345227301184, 3.0], [5.024496234305499, 486.2004414281559, 671.611037556222, 565.0052113070836, 113.9911017875158, 16.0], [7.986168037985654, 621.219670383033, 357.16152263854, 776.8759436365089, 97.07809643527624, 3.0], [4.186244341866474, 87.74164116267343, 624.054180520838, 761.9372809147849, 71.87637790879171, 15.0], [23.468082365724968, 502.4701524574408, 561.6996102476887, 548.2771308203662, 138.03224135104483, 22.0], [8.718935380766732, 1000.0, 447.15391191294606, 626.3236968439875, 68.10000516999855, 14.0], [5.37425506940621, 406.9318789286002, 640.2025554962523, 569.5877307318719, 20.0, 13.0], [9.701053052135686, 706.1872516430744, 540.9914581451127, 1083.7554484935151, 110.40905101595456, 11.0], [7.959710233569118, 704.0017121936058, 541.7359153632602, 691.7537249668044, 42.00203841758152, 2.0], [7.429939096206553, 716.2996520003189, 614.8854142112065, 524.0777613090816, 69.04971070967473, 9.0], [6.050515914021399, 541.6654217643307, 524.8938562918219, 848.3250719331689, 79.47960281182165, 7.0], [5.864164579069823, 641.5214956860535, 611.4883000639876, 853.6192048524948, 52.28018450911964, 5.0], [2.811974958629458, 674.7534050032514, 528.8010132444078, 918.75060657853, 80.18817241825337, 21.0], [17.85429105838455, 658.8615070785894, 535.384624725295, 963.199052376514, 52.84154338235332, 13.0], [11.914665954590967, 220.64545662165096, 719.1515723154633, 756.1572324299193, 99.8714955539914, 16.0], [10.265318852234238, 73.15387936400333, 731.3605963656332, 597.5222384332808, 58.137642327434335, 20.0], [7.88219182507763, 111.60411414629824, 394.961490342885, 619.6074848593521, 95.90439196346044, 18.0], [4.450688962531525, 432.4600019043341, 689.6487412615863, 606.1887047590185, 129.86023112813632, 15.0], [5.159764611679559, 648.4816836056611, 767.8309205551327, 589.436694923146, 118.4711831087649, 14.0], [3.050093552336009, 223.89718062210383, 786.8852180884292, 835.9905210932886, 92.54867923575266, 14.0], [5.580925164373834, 730.992614646101, 662.4455671062143, 650.1081360403964, 65.33788044740545, 0.0], [4.037528494236158, 374.4312063730284, 473.1122570707323, 795.7132797224309, 99.77531347767744, 23.0], [20.7343768566651, 376.8111972014803, 675.5977008417042, 751.4774832171485, 74.12842626598743, 22.0], [14.01925517470606, 290.9732510300999, 495.6897173740901, 485.20175553168747, 101.03370376104576, 22.0], [8.94404599257988, 28.80341024593156, 558.8441800612015, 392.4234995206575, 43.54144347641002, 11.0], [4.777519516795958, 557.3285321086618, 645.9698588453782, 464.5005549972402, 119.21783169006926, 9.0], [4.179734442349144, 928.1669282096424, 503.4288784057513, 585.9940448403817, 95.99649689039444, 13.0], [7.576847292976932, 484.3338691168592, 462.4319631938658, 670.1492150401813, 161.2555004002752, 9.0], [14.345799620369196, 37.34208957628687, 655.8927982188987, 552.7549508983424, 134.18184385528178, 9.0], [3.112857981205186, 695.0857029041115, 824.0669045174778, 755.4597720111652, 111.9856080926317, 19.0], [11.82618596119262, 93.80050247425272, 749.1881196575357, 570.8472208374561, 91.28021078877292, 10.0], [5.564917139403199, 197.7414007527036, 684.8623422060114, 688.5436157275917, 73.11186908895003, 21.0], [6.232542866842682, 440.4154569216073, 717.5022522707679, 667.6372520992829, 123.27227885102404, 15.0], [8.789406486525031, 530.7108596350495, 676.52633966183, 910.164963768108, 88.11590626040865, 8.0], [4.886007024166589, 247.8339997082929, 653.9880275291272, 503.71386123495904, 130.050938988449, 14.0], [7.344604052890197, 430.6709858585049, 707.8341407518376, 981.2628636732056, 107.02469491205856, 13.0], [4.615117601788239, 678.7906001901712, 435.7964289535173, 560.8475220890593, 107.2399138174929, 18.0], [5.860286969304856, 713.0447212020047, 623.9382359331496, 448.9914664458786, 112.83578380903884, 8.0], [13.120629824936977, 812.8641054535661, 590.2862152507388, 617.4627514360193, 67.4111918468733, 3.0], [9.06299788905548, 977.3546630733734, 609.4683397373476, 1136.020872295521, 64.33591268379132, 20.0], [13.092775716513216, 702.4723304152792, 544.6353342303853, 784.2807852527937, 63.29871654910326, 6.0], [5.517967141371355, 445.8158713989399, 662.4585413729078, 740.4446155161893, 126.42490744627972, 21.0], [11.875812848550623, 714.8284477807431, 501.8486388374872, 769.278029246228, 66.62835237444583, 7.0], [2.5410753937233426, 616.6790585012955, 535.9374084729559, 534.5609803246539, 70.3652087541399, 23.0], [8.787079353821772, 626.5917443815636, 521.534410361748, 698.9066564548789, 111.12551380919427, 7.0], [12.324235625316986, 544.3046962679214, 703.6735964723842, 650.3753450475684, 87.43852626095432, 18.0], [2.462718935929072, 271.98647446152336, 598.5931796148968, 921.3600122966616, 83.60617273724132, 14.0], [5.560228190650332, 242.97589804451744, 544.7001192243885, 633.6968425120021, 81.0033509351924, 15.0], [8.783422931220562, 475.72044973121433, 555.1884365980583, 593.2585183133888, 25.231049664759567, 21.0], [14.945884845755677, 537.227518961593, 600.4656740016488, 475.28009039463166, 65.99355422107945, 22.0], [5.859796682992189, 330.13976926707176, 794.6609902643825, 671.0021730313423, 130.44689410646274, 8.0], [11.61206489462349, 605.8354871521439, 451.61755963839937, 693.9427750278253, 78.81805695398081, 11.0], [18.50044836414138, 1000.0, 681.0611997215742, 469.7617717789224, 37.9853088194678, 15.0], [11.918029233802113, 63.91732018043564, 656.6234805637209, 717.9745406090864, 64.76017283394529, 3.0], [7.515672182246795, 645.5611193909401, 494.0441521863869, 684.9769285692288, 105.79561894264988, 4.0], [10.623428159430809, 483.50634477729744, 776.0247948185624, 802.0168639825646, 105.20103827617214, 7.0], [5.750288350269919, 612.6638888336637, 664.7136541056811, 831.8552176852855, 86.17158162315951, 0.0], [8.99385979566589, 392.3213248849272, 539.0720036124292, 664.1418940421419, 74.32387858744899, 5.0], [7.745407046093546, 463.48502873820826, 376.8704008864488, 494.5452673801857, 52.934120427318696, 14.0], [10.421869013582649, 532.3751127520247, 627.4976201398935, 525.4067629183032, 106.04555109236024, 3.0], [6.5569190905638, 424.59152911618486, 594.6305298292821, 914.3191494618636, 120.13611253003369, 20.0], [10.764993497376077, 119.86467065752532, 476.3144482480657, 674.823408968384, 133.66640006155535, 10.0], [6.517074150476259, 441.52124233888674, 474.7662707837236, 911.9230669326586, 61.682379239298704, 4.0], [13.942890397432036, 145.89159295095152, 704.2211840543406, 724.7907227730844, 135.96568941218564, 17.0], [6.94053426881522, 663.2982776450774, 365.3166111214806, 763.0603455767673, 93.3016373820663, 23.0], [1.6993755480105688, 33.613724730469414, 637.7139306087644, 790.8085424357239, 95.7971589021292, 14.0], [4.2751343754807385, 757.1635948639724, 577.5075629626966, 466.5291778385865, 121.80776121894384, 17.0], [8.629942318543415, 607.3907500508699, 659.6585794024127, 696.0426125168789, 78.77138289560801, 19.0], [20.633443275397305, 608.3471539334737, 564.6833988167758, 464.4501647956906, 30.085909959529147, 12.0], [13.64264127475243, 0.0, 635.1929958312378, 607.2631478183803, 116.14584644587283, 4.0], [1.4093345887769648, 417.8501404771109, 510.8670186565549, 746.0254454184576, 105.4571223288522, 0.0], [2.904223038400706, 788.390857164965, 594.0954100591957, 496.1999164201321, 84.19169459369073, 21.0], [16.014086785129475, 370.3480052424263, 658.6900653780891, 733.9005986791508, 113.67594031811582, 17.0], [12.285480735038552, 230.48446250907148, 618.1046245530564, 425.2446521801246, 119.80830381395587, 13.0], [25.0, 340.832476248572, 645.6368210881853, 821.4979735942353, 49.63981063815271, 4.0], [6.350952003087222, 482.9305943401655, 488.6494440658024, 550.3344226841979, 95.67538923193185, 5.0], [4.818242511954193, 287.15738742179724, 529.1190525738177, 495.9323863671824, 26.673121794527475, 23.0], [8.91279807400228, 808.4543080223234, 531.1992192940866, 434.6966126013379, 91.28920431464653, 15.0], [6.27275456542993, 483.8331175024019, 566.8212293295896, 563.0969250743512, 83.81773267913933, 21.0], [15.18439625611242, 361.10766301301794, 763.0299443541394, 363.8582289564801, 109.7572705228172, 20.0], [6.13603223871136, 174.66411829110518, 557.4985493357159, 567.7401291337505, 62.8083431112822, 11.0], [7.199342198073818, 621.1116513197345, 462.77286482814, 814.9230537999622, 82.35540392445444, 19.0], [4.372519572159531, 615.4165736778408, 664.9974825420711, 621.693260039025, 20.0, 17.0], [13.131752916909244, 301.94777345382704, 761.5972117721378, 473.3722076140039, 111.72511463101512, 1.0], [4.564652678922807, 0.0, 448.6545777777229, 683.9394974853212, 92.34800249617778, 22.0], [10.88122225251421, 602.6905473939711, 836.7372905661708, 670.684366381236, 97.61724313461828, 20.0], [22.577655068467408, 453.2991521847495, 649.8211922787686, 943.4914943878574, 50.197077918958215, 22.0], [2.204823760350726, 584.0177919814173, 724.066894688207, 768.6081527810609, 106.94186524559017, 12.0], [18.981404434860025, 989.2936334099096, 776.0485962051978, 695.8397190759267, 144.58249110542678, 16.0], [10.081531873121332, 493.2920390727351, 716.7141171249291, 785.7599822539321, 89.45261843371428, 16.0], [3.899703691716215, 371.540345789307, 586.7357283590782, 629.8428397729906, 114.50266724244932, 15.0], [9.11784442898544, 213.16950808791492, 746.1237773029698, 638.68802286407, 68.88136930622346, 20.0], [12.91548204429503, 421.8945521262377, 630.5519723845103, 898.8410455147348, 92.69431429858834, 17.0], [5.033305802431702, 540.1443374039103, 730.8581669889168, 404.6631459775628, 49.05978158125498, 7.0], [11.241735358562233, 513.3268551496901, 743.985058053682, 795.1918269711409, 118.64365414080387, 2.0], [19.85961011828917, 930.8663350167758, 469.0924092032678, 633.3812786468336, 99.7952173149619, 16.0], [6.131003628221368, 624.7713913664857, 695.3161358372407, 606.2765868119972, 58.42140842521812, 17.0], [4.336344033359282, 466.29978212319753, 578.1729697783954, 567.0656099459432, 70.94413469485755, 22.0], [7.892756681766352, 323.3598552916318, 616.5828764140817, 673.0511030846283, 96.83258517794938, 5.0], [8.827853058040501, 566.6681909458301, 615.2511298611357, 643.4705800243, 122.71595047892524, 7.0], [15.623267076844952, 273.2479604043352, 610.4364261090429, 587.7946419909549, 139.3858161055555, 22.0], [7.733995334002862, 544.9577698844766, 782.8230374674847, 544.6013382325024, 90.97623862864528, 12.0], [6.491363138035775, 621.8534817262108, 436.0262847727124, 364.2004321323597, 54.060539783918585, 7.0], [4.261903455646433, 445.88694430713815, 393.3187971756257, 541.4294383472495, 102.15395420971385, 18.0], [2.477956677182826, 342.42838898371457, 368.00519140013125, 573.1557280608542, 148.54647197097054, 5.0], [3.411687182295802, 0.0, 728.2973116723097, 672.6846608250263, 46.31754784032456, 22.0], [3.3971936321373315, 785.9591816050977, 507.1875009758808, 944.4628827022256, 66.36213052133206, 14.0], [7.247429012710436, 600.8250975613204, 615.8623763417863, 557.6065709729871, 113.32893032739928, 9.0], [2.8421184670550907, 606.9043203961955, 586.2339451050506, 496.4403719379245, 99.607974057625, 16.0], [8.252744111730538, 109.66191605567502, 493.6349831232024, 542.8255863045354, 78.64940815205597, 0.0], [22.30050160789165, 844.9596749350143, 702.1772685066838, 673.0017151403172, 101.60341619904356, 6.0], [15.877815303881505, 383.69578334548, 584.7044438139138, 738.2953189694409, 124.32635605562946, 12.0], [7.203944703463811, 427.8011743251285, 571.4717907553938, 759.2517819604315, 92.04990973422504, 2.0], [25.0, 232.67828614529367, 627.9445569288301, 774.1561997351399, 89.4197691712371, 18.0], [8.846539569479873, 53.26356716503796, 686.492650180586, 795.3845881417963, 112.6707250977715, 14.0], [9.61085823393902, 789.3365779308917, 484.9228507758663, 608.1674543790245, 86.88455354287412, 4.0], [7.800977634311287, 821.6493423494062, 704.7502016799945, 957.9710275611812, 129.2757068263439, 10.0], [8.363320306208145, 458.6179621616682, 773.9824223728019, 596.7847218261854, 155.80219035453055, 21.0], [5.996975425212439, 330.1538575484125, 428.8653157486589, 666.4010864511364, 68.21174095611013, 17.0], [7.269074460454391, 405.7389577164354, 633.3958774953397, 844.3740607659038, 57.49923401647853, 2.0], [19.052271097444304, 412.5962524911126, 563.4093345171207, 734.9637541981649, 106.1515771607013, 5.0], [16.841517576613075, 200.03506945510043, 683.9076696864169, 313.8729883965892, 56.43486430916293, 2.0], [1.6373025504913763, 426.9361961390399, 454.00089449677785, 650.8453745445279, 55.922675631393886, 4.0], [3.0890573264666608, 538.2754982335814, 795.472587077229, 779.294713412973, 33.523826890284745, 8.0], [5.079970149418678, 754.5393119121879, 407.0526232285117, 448.4794580705663, 83.53243257284375, 18.0], [14.4161082740044, 503.561041003428, 570.6862872268159, 762.603919269348, 94.0102909151825, 12.0], [17.051799582253434, 105.91207277514104, 652.1155291194432, 708.523091002018, 79.47911032573302, 5.0], [12.887296917699622, 588.3869657996692, 517.6395783678211, 827.5880725777059, 136.70999550596514, 10.0], [3.888668454374056, 436.01735958256165, 498.7918141851555, 546.2103016492039, 134.47661801138352, 20.0], [6.075401227615167, 842.6024234959666, 626.6682763956629, 799.3840123383868, 152.8653189206527, 20.0], [11.354557636412196, 857.3644505110019, 609.3311903989296, 673.9412371101247, 96.39585397801126, 13.0], [7.338093510854856, 866.1135780668249, 600.7125803298179, 914.9075456078432, 130.5145717553233, 5.0], [6.720712759826208, 359.60301615668016, 703.0426213758751, 1067.9074906719989, 104.2655794636146, 5.0], [6.782645935730589, 997.4637683343892, 552.6632767886374, 880.7495658578829, 101.25146937558864, 16.0], [16.13824006511723, 587.7356262239267, 638.3511179754005, 722.0778017162337, 80.97841774113616, 10.0], [12.022992843648153, 906.5951014171584, 539.2076630069022, 998.1505994092709, 27.105596277345683, 14.0], [7.015828485807157, 374.3450965420601, 498.080461700471, 740.5217062501976, 114.11874365041967, 8.0], [10.353538112747666, 425.3508155975934, 565.8289645811977, 513.8081429716076, 129.6240870958407, 20.0], [3.523047069323819, 568.5068381680213, 540.1481775367643, 644.1090039572637, 49.23146215105251, 16.0], [6.082892536985671, 537.969402403485, 546.1435438604663, 764.2508279442502, 94.70904013452176, 23.0], [15.27257458952566, 813.7029858672925, 739.8392901362594, 773.8931233430096, 124.74941505243709, 6.0], [2.70992796828266, 441.9676703628864, 569.5075855157335, 481.2102665288511, 84.21503670063746, 6.0], [12.107969490338723, 791.1537475775891, 520.4511993264163, 485.9701570310491, 130.79311019996803, 20.0], [1.6225031553980518, 683.0586267831634, 603.3694528857588, 598.8035230685227, 90.79150899927896, 0.0], [20.82247505497617, 446.6519453482129, 612.5073526954089, 730.1815420184286, 79.24524407905218, 7.0], [5.990805166202325, 663.6691543705783, 457.5742524487114, 749.3843824819879, 132.508607388237, 4.0], [10.793595486800186, 381.6837382917378, 564.1912473691388, 671.1957686157216, 66.98738791579709, 18.0], [9.14426482625288, 1000.0, 559.3578811081684, 524.8356899723528, 73.25287904525621, 7.0], [9.343090020187914, 0.0, 579.3426871549832, 910.5414414528956, 68.13362345848081, 16.0], [10.244950063984044, 668.2358011237598, 687.0634047645583, 489.19129072354406, 60.72214191538211, 18.0], [2.353359224846271, 58.8510226807382, 616.4006245395459, 845.3511117280642, 99.59101752123163, 11.0], [9.457707405709808, 443.5858241954828, 747.3524816862397, 760.8234925175548, 79.61196194664491, 22.0], [7.607568563911518, 478.5979776321885, 598.6007572933421, 842.0498455807592, 109.65839289249372, 14.0], [6.012127084233871, 876.9083897284211, 813.1304736456077, 583.3350060615625, 42.96078993405426, 3.0], [5.07816561701929, 326.96194698077744, 544.7069983522017, 628.6018702866194, 61.32355728486751, 22.0], [9.598195582455132, 827.0029789154445, 507.6525142601851, 658.2563835921566, 96.5473726857911, 11.0], [4.373827833255381, 205.2958399438008, 665.6763209732369, 578.5403359097727, 139.0904179161115, 12.0], [15.120151076415215, 836.6674309387165, 763.6898324958021, 683.7887680242648, 106.35506446752562, 13.0], [11.248827502563255, 381.4136395241318, 434.5436034497728, 886.8634184106811, 77.31828215448671, 4.0], [9.94237608930812, 518.3586298102797, 576.3611222142822, 455.4749229248041, 50.31405939859109, 21.0], [14.76739577357279, 415.905334222399, 526.7061665896955, 551.4182437683364, 75.56461854781719, 15.0], [2.075673949637339, 820.590868528901, 617.1421215590495, 535.1382021940616, 93.72547632371538, 6.0], [3.4412204959733472, 771.9349608084369, 601.2021508199285, 635.8372606795929, 24.269243178707317, 14.0], [11.53430433699446, 610.8754301461312, 761.4457497153265, 629.6710596531693, 58.59889789256256, 12.0], [5.274977130742788, 212.0329038533065, 586.991424855026, 765.8471876661059, 72.87332106059083, 15.0], [5.720086591351027, 321.2445329924393, 764.9740918764276, 655.0584654382062, 87.27774178445117, 11.0], [3.0913155621869226, 242.85586862346867, 562.1879596777577, 695.4324915078954, 93.36990560753888, 6.0], [4.382198574755416, 0.0, 711.1778250126259, 709.7672068268001, 76.67396665378595, 15.0], [15.308287368135971, 815.3295254625116, 627.4395633764063, 686.8106091808718, 110.17619546608752, 22.0], [11.379504578879455, 418.0441184143258, 807.5862478420133, 755.4969592186387, 98.74120709091396, 22.0], [9.536109268755844, 415.3765089269257, 422.7582164622686, 631.6514926800883, 83.8981014648446, 12.0], [5.864898780641457, 874.02555835639, 460.33707386882105, 896.9552594112497, 114.31326142813796, 22.0], [9.18691579358211, 381.0028494034081, 819.7946741590064, 1031.2429770078188, 111.858218495304, 2.0], [16.058007487014912, 289.88226607851726, 512.0081887816375, 755.9346462150604, 51.38604347761182, 14.0], [5.361016380740819, 443.8135948712538, 519.6235970308966, 684.1986745563592, 94.03917292524063, 2.0], [8.351800941754126, 216.83273703077597, 494.9131697400175, 743.6873463975866, 121.70781594401602, 6.0], [21.53457966094605, 586.0889423219891, 737.8135147840469, 484.8450371174534, 126.72467837906537, 17.0], [12.7917836606937, 406.5844804339574, 449.58508641298505, 677.0158201198122, 106.1272637190586, 13.0], [5.918664447075063, 373.18214735154686, 602.6940494733193, 540.3013089577087, 88.0013431476082, 17.0], [12.212523177205584, 351.52420699492444, 568.1253813863354, 726.4965375860495, 107.63959382697308, 18.0], [2.6769930992548043, 473.4446265394972, 724.3818700205321, 541.0461038034279, 119.23320293875352, 13.0], [8.150255396098116, 520.8702813799628, 532.3580805202026, 598.2833831961656, 98.47495847028694, 19.0], [12.204973531529758, 370.0062278981195, 649.8327094935929, 369.2043641715421, 101.85812436735262, 5.0], [10.678262698605565, 410.3085159762133, 459.4540591947276, 474.0131465078498, 87.50590536994042, 7.0], [12.138187334739538, 479.6192932951628, 872.872924641264, 703.2310704404069, 54.8964439674896, 6.0], [13.674936547472068, 454.3954661151153, 571.2489210415549, 519.6730882522354, 89.76907639243534, 19.0], [11.164744168463598, 507.7689917187535, 601.051618186244, 809.5660975678366, 74.68658162358398, 17.0], [10.980968560970297, 330.83229918799657, 798.570523062065, 506.9166367380032, 74.78235867010876, 17.0], [9.665281534566734, 751.1482272816227, 529.5992670833749, 717.9881751928656, 138.0150346748047, 14.0], [4.039088091714995, 524.2070443523793, 800.085881152634, 535.7186556542363, 123.546721331915, 19.0], [15.304515448664144, 374.0383117937807, 550.6234591034447, 707.8204662550289, 44.19502727494782, 21.0], [8.328299212727893, 586.0960543446337, 680.6037820585511, 419.7819006357291, 62.453636711709805, 18.0], [12.612946648802088, 481.4078502953213, 531.2782031808841, 582.0225536555663, 84.09313866342295, 21.0], [14.435429040746635, 66.54341137908301, 437.27562658263184, 523.0680072068398, 75.78103309592113, 20.0], [6.110783058458968, 585.2381725554065, 649.7513220648739, 782.484886296347, 104.52254820511374, 20.0], [6.458258795778405, 245.1234494039532, 676.7884081180239, 719.3393696681306, 57.69244766578395, 1.0], [7.017545266467287, 733.6069458078268, 442.0937109208513, 465.11359180767766, 123.75595883895843, 7.0], [16.921156925124073, 439.5625902766383, 563.1530017309545, 619.522889506868, 93.67983365174524, 20.0], [4.6602064388854645, 695.017574347766, 744.0910753844288, 534.1523235915748, 84.78463865866469, 10.0], [18.547838211910737, 342.44536170059826, 672.0142740955932, 792.5385790802814, 94.5704167185268, 17.0], [9.6022061661891, 517.4823167851599, 552.574199785914, 840.7286221480908, 82.47475779454905, 22.0], [10.006277825020955, 802.5608751267191, 528.2969158038653, 433.753228023378, 34.080722878216896, 12.0], [14.077475588443324, 432.7458633962944, 833.7893190161732, 774.0054461572221, 95.3577794231443, 8.0], [8.970577213139219, 352.2957253160488, 426.368151623657, 526.5055896113436, 85.37760834096325, 8.0], [15.399804542762162, 663.7601534454474, 592.8961710709677, 612.0624840921021, 66.3662896204174, 16.0], [6.105582462105421, 658.9153991877401, 424.2363751567267, 654.8381343591435, 127.92289494249744, 6.0], [18.21983438513941, 362.8127166462953, 757.7777573813137, 922.4909245333208, 115.0831404017968, 8.0], [6.41061180229322, 3.634744895559436, 599.1451665349996, 499.4225409453673, 168.16570119737753, 21.0], [18.60407012934708, 347.49425562541103, 621.3808901723322, 775.2124583964953, 120.0448550357736, 0.0], [6.520557423470388, 669.3398804216055, 932.267077155313, 474.27433877455735, 75.6969479890917, 1.0], [7.143918038525829, 306.241452644069, 528.7832789116328, 463.9109622738898, 67.82631864194639, 3.0], [4.288153115267549, 497.2655407459511, 809.829109687024, 654.2695172628733, 80.17083196140935, 13.0], [5.117110648302637, 335.628765508414, 553.1121702710174, 653.2594236484584, 75.00114152964065, 3.0], [8.353364960489662, 16.958209368487587, 571.9785050898122, 300.0, 98.26350091561898, 1.0], [9.816686848871084, 331.12970574001804, 916.581030794472, 823.9394434063247, 48.83607868523001, 7.0], [14.779687397449752, 668.2830051724281, 580.6691547285992, 329.1328813353723, 114.41859692138094, 17.0], [7.463250712303294, 0.0, 435.07543148457023, 628.1093845830047, 97.73589108114282, 18.0], [3.9082887101023815, 563.041837618894, 633.8302545751575, 849.1423177407958, 110.97392923366088, 13.0], [10.691690413039694, 203.4826921501467, 579.2715809799595, 981.3775950033018, 60.67560568917431, 19.0], [9.433030488468104, 652.110517368548, 644.0515010080633, 595.5425204467131, 102.4006631100817, 9.0], [11.374018802042825, 678.5797339684133, 583.9682648006676, 435.0576374574368, 133.97529742608907, 16.0], [5.615774761928437, 457.013016272181, 865.8653539445718, 843.5822338651938, 69.18478246699352, 7.0], [6.048456009074106, 457.5829603046945, 666.8083006603123, 779.7734825180394, 68.36432877568382, 13.0], [5.6171699399382025, 298.40932571592464, 471.00424467290185, 576.6511565632287, 116.35872400151796, 20.0], [16.376644698796863, 163.60414997375847, 614.9456162428626, 557.639381268619, 65.92846845387251, 22.0], [18.31667539263809, 924.5232265834333, 504.04442109995534, 542.3911754510852, 88.00148050528192, 7.0], [5.8848283740742335, 283.73214320721803, 570.2743388996455, 598.3120196398049, 139.27198036977035, 4.0], [5.960706898391623, 891.9490327991181, 490.7471147319813, 697.4102213602446, 68.78993324232123, 15.0], [12.286479844319896, 285.1249900499558, 852.8542227847228, 565.0978125894842, 141.33326296622795, 7.0], [19.44333299414716, 631.2975975844321, 517.0338461786134, 584.0629586656639, 84.92654774137607, 9.0], [21.77839117001814, 215.54821008659133, 682.5077222392964, 746.1918529115864, 61.36396019483449, 7.0], [5.715001462381046, 630.8487542255039, 561.1824693748048, 427.6378788411691, 113.11112465367884, 8.0], [6.663472325110743, 418.6031607117195, 460.9904192912652, 593.6096420129631, 98.23244403878202, 17.0], [10.11600767641174, 421.8709598310824, 623.7179163479408, 398.60884713513474, 95.39540782660067, 2.0], [10.403097509321098, 652.5057924461127, 648.0830204502848, 666.2618710029352, 71.93845218444903, 16.0], [7.641677350251811, 382.7146587302255, 667.0351839305416, 359.08918877871884, 98.01358851356343, 19.0], [5.0046095495664105, 142.96287613872562, 285.3337014550869, 783.5015190239644, 32.971083377349494, 14.0], [8.361393666418463, 690.3263908523631, 451.818514359709, 452.8043906311992, 101.56294210354312, 20.0], [21.765513675003213, 213.52485250441796, 714.9065662826262, 384.1132151957818, 133.9174760055816, 16.0], [6.104232958763904, 629.1293210994775, 489.16716448694496, 595.4580056147573, 42.0371390742288, 7.0], [9.756987954838763, 743.2491363332501, 624.3532860372134, 906.6660937515576, 82.79674351118332, 1.0], [25.0, 406.1265284119456, 449.0370365725457, 756.6788680132363, 68.76080557555838, 16.0], [6.0489885156222565, 795.5882110229461, 687.7921322590128, 361.6528841819052, 49.66837898261403, 17.0], [12.615596628839311, 620.445926965321, 711.7523515194689, 448.5312098533189, 33.413767389522945, 4.0], [22.278213915412056, 326.0346449638061, 687.829188740684, 590.6599440131439, 59.574922490235934, 8.0], [12.006933204941284, 850.3572505665187, 723.9068174824317, 985.7529742656292, 87.72870153840778, 8.0], [15.737034196073624, 214.5820445651524, 787.6680173879439, 509.15687780466976, 129.474111454056, 1.0], [13.578985006897351, 405.82489252410863, 459.1111042069231, 510.3672134445208, 49.31856470628483, 1.0], [11.468089402776142, 448.4212990951678, 505.3593682618365, 879.1144080320272, 103.8317827740862, 18.0], [17.985269607017155, 651.3059252545638, 707.7052284364554, 679.2360807893564, 101.58924759851044, 22.0], [12.391910157155465, 778.8583243452836, 735.6263726204977, 807.9500040948112, 145.15888880219825, 2.0], [7.689496354907894, 763.1978892256784, 519.0788455713689, 540.9350689736053, 112.13094393191442, 16.0], [9.323186261804702, 555.9370798935765, 510.7336721609299, 634.7546228406421, 113.85798479091136, 13.0], [10.6084827144076, 429.9327238841083, 521.7343812083393, 847.3447793282367, 115.979011991537, 13.0], [6.638523681131165, 43.50393959309247, 407.3886043147265, 628.27386848963, 67.27303954042418, 13.0], [9.57139638337809, 171.46353638619587, 447.43065813911096, 818.4579796289695, 77.81532429258678, 5.0], [8.778298520565308, 941.6638723228588, 450.3248825963816, 708.0900950651795, 128.50604486812216, 2.0], [12.370306878277685, 40.98200312436518, 564.708183143324, 494.05563823716216, 33.47600432017556, 0.0], [25.0, 303.8773391894726, 632.5003819000746, 647.5524181480965, 74.53066783378608, 15.0], [9.90397128040332, 786.1515705125224, 652.0167397970847, 797.207598841255, 55.075722144140016, 3.0], [19.715574282361267, 552.4283247993417, 538.9725296069182, 750.4124286780327, 102.02732126117812, 0.0], [12.120421950293764, 732.011267751423, 541.9215460462946, 643.9620652368758, 82.35174244066641, 16.0], [9.546090008701444, 635.7070878176692, 614.8393794181513, 366.2428416542751, 113.77911424966152, 17.0], [6.406591822656772, 867.3956343402216, 420.4782531420688, 380.6012777885426, 91.73014517865856, 7.0], [9.588278727301834, 573.8568216412212, 715.1328638499707, 412.50416155593257, 48.61723039088982, 1.0], [9.24720704777468, 665.8619322878778, 754.2462334817125, 573.4336689850173, 50.003970822124494, 17.0], [5.372737456716303, 318.93836698541486, 524.9303234774352, 593.4323189459828, 94.81854444082744, 15.0], [5.933191141975068, 646.5832438945342, 552.1876608781465, 428.68828983589657, 71.86595098299117, 9.0], [10.660153624635583, 400.96847550269285, 749.1974669814203, 803.3769190544024, 79.02098441667957, 7.0], [19.606829659502708, 386.7235330048219, 585.233809081212, 460.4346384169187, 123.15246265367342, 11.0], [8.19434088134266, 723.6808786617844, 670.5894535141956, 444.41614368994857, 117.77098030601655, 21.0], [2.9827972424672904, 725.405844014841, 588.9233527439071, 663.2353630828923, 148.93201521732072, 15.0], [1.4288178728722054, 922.1105389365327, 786.756886617633, 796.0857679989842, 85.50993365779806, 11.0], [4.2319462999005495, 905.8157963715504, 658.8095553901795, 958.228192971975, 75.76560096641401, 10.0], [12.795921696520868, 729.3287903396938, 665.718692630303, 737.3817581754765, 107.54566893286729, 19.0], [3.439146050869342, 69.88859744082185, 667.9567846480304, 886.9744578085429, 78.17616592280075, 18.0], [13.95255310014202, 956.920245737424, 533.3863516406745, 468.1915022410157, 79.15816660922788, 0.0], [12.401906158977734, 729.4919176665906, 544.4113035692958, 973.96088909915, 88.93695218204492, 2.0], [8.02550288735051, 308.5349774784244, 692.309476747197, 370.7773324166768, 144.58246867450637, 11.0], [5.404325530103211, 682.2272888906955, 615.1695395320342, 574.4241119168479, 116.1189741538588, 8.0], [7.333791066604551, 240.6835647593179, 458.6866790625873, 734.0978779368163, 119.2646120683355, 20.0], [10.508399082448276, 654.6575300274354, 824.7725309549736, 409.2174476456085, 143.23563010424738, 21.0], [3.5218133835759247, 170.9324897314752, 717.368624075652, 510.94248621119993, 122.07598880225974, 4.0], [18.519677340576177, 297.9453978033409, 604.5151588614023, 487.1116206065549, 61.05783169594888, 0.0], [10.45248451070858, 677.4822823001317, 567.1798821840042, 603.4668489125398, 121.62147092423402, 9.0], [11.149580565446763, 768.2216857482234, 623.8411029036099, 863.2374285009012, 125.42764571345772, 16.0], [12.351832980580475, 581.3592148584526, 662.1024118988913, 652.5237599431716, 114.55594558079288, 3.0], [17.533768846373444, 428.4578408280073, 356.0399117195944, 580.5404083648973, 89.13674378621347, 14.0], [9.602016177834022, 0.0, 853.4819651382904, 300.0, 111.38995118005252, 18.0], [12.800353687803469, 447.7447250871601, 505.2138828999696, 720.0132807257869, 94.3928083610706, 7.0], [5.464340120477903, 633.371269206132, 599.7713240413491, 768.5023113467269, 65.65734871254651, 22.0], [21.82783606250624, 420.8087235160882, 604.1840259334975, 394.99703482846826, 73.11099063994527, 14.0], [9.220410330010573, 494.3555147263954, 839.6220262426622, 639.9541460416217, 116.11168456697844, 6.0], [5.44627142125946, 453.5956330205198, 589.4251390791967, 678.9512285498071, 67.82539461731473, 14.0], [6.407671811369198, 419.5734455446416, 790.6208753224355, 794.690263595554, 70.65339369571208, 7.0], [1.978146154761904, 518.9693474899532, 648.8072963353337, 541.2847312631789, 117.60153379954076, 22.0], [6.381275237199228, 431.8263544618849, 484.5738165352861, 518.6959458372804, 99.01367188595316, 9.0], [1.1004750031848822, 860.8223884079855, 616.1648961723365, 581.5254052297457, 112.31579951563184, 17.0], [13.333456646816847, 0.0, 585.7789533942105, 842.6655579166076, 58.129461903404426, 3.0], [10.955266273788268, 363.5393535072306, 469.9115418114925, 806.5768163259005, 111.48288522307293, 14.0], [4.412081062150845, 282.43594412069365, 642.926732336669, 646.798204674138, 75.19915553267771, 10.0], [8.286434163882047, 554.7776500086024, 602.6523299819827, 802.9664790782432, 64.49348043356466, 21.0], [8.514521824006392, 519.2861964707815, 666.7706046550248, 300.0, 79.83965323399109, 2.0], [3.798663381771202, 905.5482753873572, 623.2023335517978, 768.7481309140288, 135.86194870782433, 14.0], [7.429964948670357, 987.7855120744488, 521.8676523583116, 482.18507557792304, 84.06637595153175, 2.0], [8.674599833991712, 306.7291361073797, 692.2000790748377, 915.0536741593792, 161.04172136863804, 14.0], [7.305762336860352, 500.12501625840736, 525.0722063695705, 490.3632932100164, 33.13465685755559, 5.0], [5.442659682228147, 549.0761042310888, 250.0, 522.0856752149533, 48.61166243055763, 0.0], [4.296484440038024, 300.546562915666, 615.438353754506, 734.3635595196768, 134.72555711436672, 13.0], [7.147094916667065, 657.0955090094058, 533.5111527319212, 748.8431202171555, 142.268136774672, 15.0], [5.883063621185071, 440.12315780876384, 645.8556605275962, 583.0501881849223, 69.0962828843482, 0.0], [11.888741542377147, 233.09128734975556, 839.8646083127658, 705.7106051634546, 97.06680850168564, 20.0], [18.13339413962151, 430.1079239333642, 843.328763177767, 724.3069722992731, 116.3656975550983, 16.0], [14.334396269580257, 250.58135681483785, 580.1852493333793, 708.9671984087051, 88.11872937905736, 7.0], [8.848744711423254, 759.1617073548139, 500.8181794504733, 589.3167623662406, 99.49601274163334, 17.0], [5.579492548498384, 975.3898137784176, 670.1619938301507, 725.7816536128536, 95.3279903972957, 15.0], [14.118096821818888, 561.2048434736349, 555.5729289760375, 860.9555607524741, 50.556590610416265, 5.0], [12.301411028887015, 527.8191920133189, 647.0131826816331, 466.176728117176, 84.66016142113945, 9.0], [17.43464705423602, 538.1086917185464, 520.6967085327775, 523.9909946687291, 95.37095285409598, 9.0], [9.608114111695398, 795.5585072345788, 630.0830899227303, 654.0883448540869, 98.09365502324168, 4.0], [21.198908164321438, 790.1747508992958, 502.5631017247829, 491.5663938997951, 43.08481674142728, 11.0], [9.60987412971928, 471.0866078629423, 742.1104099237759, 379.4894398659109, 117.61081698662032, 19.0], [5.56187045935835, 563.1474818447728, 529.2026771096865, 813.9684463822935, 98.88480958688284, 16.0], [6.246282302400361, 663.1616017942696, 679.3323042881963, 315.68176009606253, 63.33841335974799, 13.0], [11.567404587263036, 215.5781358623165, 448.2279260177377, 589.6425953713497, 148.9719270207096, 13.0], [5.872788668195859, 803.1520593551141, 693.740862793283, 763.8416479638006, 84.60246255143542, 2.0], [21.118297781888103, 933.7079728068068, 484.03391168781593, 690.0049684253108, 86.55057853235208, 11.0], [11.022475750678788, 791.2580262098643, 631.3412168369412, 591.9125951031572, 114.2512773557398, 19.0], [8.733468009462468, 188.6015220963613, 722.7841168017584, 485.3840635197561, 151.52003412709573, 17.0], [9.176059940506548, 7.499886145281437, 576.0020521731865, 701.5801719721906, 115.43118788133566, 11.0], [7.257813596115264, 677.1796683794829, 476.499970710488, 501.7408342299691, 119.7832951008161, 1.0], [2.446633411803642, 309.9526377594784, 659.6175008563106, 689.8898879477393, 64.95134403528806, 1.0], [6.249251094764024, 571.7833385307465, 499.0822321216143, 671.1376472352134, 97.60879106111508, 16.0], [19.683932887791244, 883.7871800018872, 551.4905280963326, 672.6549363582703, 79.03214821485273, 9.0], [9.760781713432936, 335.7612059003975, 631.8555191394872, 553.9521861230877, 116.12389544147828, 17.0], [15.257859358713937, 405.61133589776, 763.2811243329246, 955.2513664002184, 118.90897386529775, 21.0], [15.502278893358032, 569.180743147079, 603.1864024559362, 703.7309950805562, 143.99688549559824, 18.0], [8.3336819356412, 175.62259000444237, 602.7429011814148, 608.193661367053, 30.249742493326067, 20.0], [12.416137012235808, 310.5359509985309, 688.5015472347421, 689.6405859592707, 76.0826551305104, 0.0], [2.662141044735289, 506.2143759378173, 606.0446276007597, 533.8307534113535, 134.00746832756124, 23.0], [18.80826138770252, 424.8862827955622, 676.2636239799477, 592.054325228314, 52.95410459202445, 3.0], [17.967514179186317, 652.1019384498065, 729.4240255275537, 1008.7879898179276, 119.16953098681014, 3.0], [11.38401463746832, 1000.0, 560.6715419678401, 534.8928790336537, 129.0200150244425, 1.0], [9.969270890644404, 524.7845848545312, 250.0, 638.3340949603376, 94.1498046390062, 18.0], [8.40049387261078, 289.57201349049774, 501.848748769744, 581.5842725375351, 52.35605938530424, 20.0], [16.245556329988144, 854.560088689391, 745.0418413002363, 494.0549085762831, 91.35440965399286, 11.0], [13.234479177962848, 102.79538116994468, 595.9482262322815, 713.1627485329709, 138.7604401237617, 9.0], [0.6256638350560497, 152.77295724272318, 559.3043674886521, 655.4708264981314, 141.17052125766247, 12.0], [6.776035357814942, 351.2241403380691, 740.0141991843849, 786.6540963499891, 101.23641130333073, 21.0], [3.917772891574422, 429.2783198700936, 587.8410100524125, 771.5866880495007, 65.68286933855717, 22.0], [3.39831728246176, 345.23181587069786, 348.8960153101013, 785.1061578562994, 147.15191582148233, 19.0], [16.15020610526983, 492.1219188942443, 419.3637526789988, 512.2694940723509, 119.62768637329128, 22.0], [8.290486996088829, 568.6854756865863, 650.2377624980287, 746.1441198804644, 67.65470093969384, 14.0], [11.276765541749045, 767.2398838187845, 591.9730870086328, 749.7907259435142, 105.27556913421525, 2.0], [8.941100073995011, 522.7970684609455, 774.5587774595067, 496.8405502689716, 103.90245442934474, 22.0], [3.431488308737862, 359.5134380762246, 512.0077579434987, 602.2075636659141, 75.8961469535133, 19.0], [2.7100429591812603, 753.6613466098529, 562.8599085595447, 431.5831855281548, 105.44320444144132, 12.0], [6.4859906785587045, 295.5711310674673, 493.8596312327976, 750.4433948570426, 128.18560514371643, 7.0], [6.83801062032361, 91.84931357816924, 537.5286986783457, 749.753370416596, 86.11252403707282, 6.0], [6.835280920609669, 801.4890044761355, 372.9715318495652, 864.5544198677533, 131.24807274764163, 23.0], [6.960535100957273, 785.463311202743, 553.7178403139362, 615.6802238278482, 42.33936292039508, 7.0], [24.63600951503132, 301.92540194897344, 593.7757363476118, 717.7204379845695, 161.548009793626, 17.0], [15.128782460362514, 400.2252865447686, 583.6402989041179, 798.1602651537962, 96.72053809939044, 8.0], [9.842954552700846, 270.76811366751764, 857.9463916479372, 536.9162707167962, 127.19062984410868, 5.0], [8.164617240145645, 581.6560543912835, 820.0181492612378, 514.1052288023451, 101.55834459604117, 3.0], [8.340299747034898, 933.0123228420204, 509.8920634846697, 541.5672198385753, 70.29432149429068, 14.0], [12.962888178199773, 675.7204496487805, 354.7275966416885, 597.6056797692169, 93.90870129021486, 4.0], [8.118453882104935, 758.9311606457582, 588.0959576482132, 519.4168466530771, 37.715869462962296, 4.0], [14.273149818036464, 704.9320829227529, 645.1812165804745, 548.0258435058164, 115.65160792138428, 6.0], [10.475012848214188, 499.9702128602266, 496.74191119405975, 782.1372753549867, 100.81293735934462, 8.0], [24.759400385171546, 770.8351094778288, 502.6829239289133, 635.5480052758561, 89.25328524595265, 6.0], [7.824483702430303, 686.2576407281999, 789.9065963175038, 696.0047560489154, 117.58098713226912, 8.0], [15.732942151945036, 859.861428788284, 569.0938114625635, 567.3924817822137, 96.1256978477958, 19.0], [4.171104000132315, 385.1127647095609, 679.4700222834887, 557.545185622649, 113.20489616847723, 9.0], [5.671192185186383, 65.47209011365334, 488.3633162574666, 692.8333834310596, 123.86949736640432, 3.0], [22.951923500314308, 761.3127537182694, 593.1671449530248, 668.9093492005107, 78.52315581513804, 5.0], [13.935229588264594, 587.4423801515145, 559.6393975638584, 766.6980017462104, 69.93769152824525, 23.0], [11.394477490320376, 425.9884749196909, 518.9552113436055, 845.2389497005989, 43.32319775742591, 13.0], [25.0, 439.9617969231272, 738.6360493826031, 546.2881693585393, 98.67682396483576, 8.0], [8.23576917917477, 1000.0, 433.33475884403833, 779.1776806645671, 108.56291865352064, 0.0], [15.712076003365452, 339.45133686643226, 636.0068190613547, 488.0290748460256, 55.291538045328814, 16.0], [16.36062115070945, 286.67417329341674, 557.3854013411401, 961.0668024204924, 137.61139544085052, 12.0], [20.231233192043454, 521.3210583733379, 649.7789378443049, 532.0543920292507, 70.32993585941828, 23.0], [3.835398617986932, 795.1227194932403, 559.0108041337971, 753.5945854635139, 31.89499724823497, 6.0], [8.202542219462103, 1000.0, 704.0631674302728, 486.9139491883002, 139.8446869967698, 8.0], [10.369176343232237, 664.3412408473928, 487.2356882097418, 627.1863670779802, 35.809550000454806, 10.0], [20.048561566491564, 182.3978140051425, 738.9063830010374, 881.8599713230275, 123.92536320150276, 2.0], [11.129489188188298, 992.7160676794496, 596.5436766390694, 527.344377202089, 116.63358562627558, 11.0], [16.380308397106912, 495.13282523072576, 440.5796965181852, 980.6279251829334, 104.58662303659632, 20.0], [3.591591253848172, 290.5726420262576, 681.6258547083388, 649.2086604336431, 79.3167515535031, 2.0], [7.070926742784735, 230.1518857251615, 576.1380307778819, 796.3568806973999, 110.08391883576732, 14.0], [17.92990146328401, 709.0821798338641, 395.0594488632547, 710.4416860255404, 126.45454621113652, 20.0], [7.660507521783879, 559.7073465466035, 639.8961700510645, 718.5560098394551, 65.42568510573581, 9.0], [8.6150020250707, 586.2688442364627, 720.791485833324, 436.0829424285343, 124.2912101893845, 0.0], [19.871711913221244, 821.6257797679827, 753.8562888432157, 579.0720810532931, 122.92623535811732, 13.0], [4.828541739507226, 299.461352122589, 554.597774088028, 829.3964693725555, 103.79153272764664, 19.0], [5.610945757642589, 841.538657068412, 617.4381666390885, 598.3518197345004, 78.86075365756662, 5.0], [3.3185944925231983, 370.2106693015775, 606.0643611351494, 670.824812407672, 123.06372094546327, 1.0], [20.31108992305255, 1000.0, 778.7763157885624, 417.8866807446273, 110.2493886848757, 22.0], [14.972660034556553, 649.684753998285, 502.741370115202, 569.3634588498884, 64.52121010416298, 7.0], [14.76042579994647, 269.9403675559057, 544.240865735455, 818.4558946447198, 83.97932573695256, 0.0], [6.9459637804444005, 0.0, 499.83356286602327, 457.60558521095896, 119.571648152621, 3.0], [3.1369172977491995, 702.2311805697528, 426.1019372264536, 757.5618555494246, 98.43741851302534, 3.0], [2.4996926492169047, 406.4549310024541, 629.7344617262498, 676.6667207877479, 127.57948683335977, 4.0], [18.003433775621936, 461.1085763695411, 792.1557998133105, 614.6066770965232, 84.75059913054811, 19.0], [11.832292105053662, 691.5490022935268, 576.1818649781927, 608.011318651699, 65.003141232557, 8.0], [13.145269959296655, 752.603456562664, 494.1724054148239, 576.3455452883019, 160.80807846830214, 22.0], [10.088674588660684, 805.9275851673227, 622.4231642678482, 589.7043541914555, 44.57314751760528, 12.0], [5.16868583003926, 206.28967290426584, 670.6941814527187, 789.4610337216736, 80.00481207795008, 17.0], [7.149069419876423, 856.4736874061473, 581.2364235076113, 585.1934278632839, 103.39244002099952, 16.0], [4.339718190948086, 355.23866595796267, 768.4242398345012, 921.138548328846, 124.58251674433134, 0.0], [3.967995151266676, 0.0, 502.51657268549735, 694.9298754564818, 114.39348105626785, 21.0], [14.287799787858155, 205.38991328498852, 565.5636436185165, 563.5607106457592, 164.33777014828638, 14.0], [2.676327555206324, 554.5062730331413, 635.5185485811554, 796.0048928728421, 91.47204642025604, 1.0], [4.36007148704197, 255.29643847971516, 647.1556503141393, 743.5843914614865, 68.8105021594842, 15.0], [7.412496649996282, 477.98888270598906, 605.6673710757793, 401.8735339658512, 88.40911927350652, 19.0], [9.095910458326273, 584.5631218822946, 446.5896093171422, 494.82929644736936, 85.40070411946927, 21.0], [3.2882409663035888, 454.65364922267855, 880.5613685047596, 632.6869548196381, 92.95114558129582, 4.0], [2.622435290578561, 376.4114155358208, 571.683610194468, 412.4588098369704, 65.5253563130216, 21.0], [11.048223486050484, 187.32598425294344, 613.2321022301588, 963.579485295742, 118.5216811475107, 18.0], [9.757708965942149, 505.48595205383305, 674.0113311807577, 773.3348273059012, 106.12490628008628, 4.0], [17.830931383014253, 0.0, 523.6193564820564, 648.0000729445308, 57.08357512777667, 1.0], [9.188495036395564, 328.5004895701446, 606.8311705044931, 598.5940169580105, 124.04075227611752, 6.0], [11.698869572195283, 268.05397273480975, 518.7371714816047, 560.0504518461502, 70.65555389046486, 2.0], [5.365558912633716, 440.1557871227713, 649.8660889769077, 355.3951763043636, 59.51553493087133, 16.0], [5.516884192638504, 446.6675501726827, 566.8772821733317, 390.06895225315554, 97.61986111764948, 23.0], [3.5440956184838104, 739.9891686084363, 632.2462232765602, 935.1681096901413, 42.26469651802069, 23.0], [5.400483764988145, 895.8693193123999, 474.4155184217278, 886.0399458848877, 112.68509149631744, 2.0], [7.082983466013205, 220.55528994075743, 500.5300279192553, 563.6333367613939, 20.0, 12.0], [16.85514407326134, 998.2081563175076, 436.1831187771838, 786.1998734273686, 64.11911930034087, 6.0], [4.87018099369444, 305.176100119642, 514.7363904808551, 725.9631162495547, 142.25411963263736, 23.0], [25.0, 543.1316957307724, 291.1770907562232, 743.6431934973263, 74.40022372480044, 2.0], [12.292111408510586, 627.7157472870795, 563.1479753174608, 436.22333532489415, 79.65244384381106, 14.0], [10.883460302708492, 1000.0, 535.8911673644488, 378.773311620476, 133.20995162391313, 17.0], [8.764327114013453, 364.429415559334, 618.1093156350083, 827.3891538604834, 71.21409184783045, 9.0], [7.787982564612716, 50.81549675548666, 605.3529269321148, 353.1052545444755, 117.9031454416843, 22.0], [5.9390580886024384, 872.5948675482454, 750.8495722177285, 627.7709687926227, 130.22564495177966, 6.0], [3.190229938448474, 291.6566103011413, 911.3172221728934, 594.8446510673319, 109.3559094244971, 8.0], [4.984947360348993, 409.60285485610177, 736.5046815780762, 580.9773557485387, 51.65308031107221, 4.0], [20.28410198097736, 220.56258651845616, 545.6270892749894, 697.1676233435985, 76.8804242235237, 19.0], [11.865292418879385, 410.3240567377703, 540.7931100348133, 548.3611355332323, 176.4085933258699, 7.0], [20.1844890740587, 289.9021159486398, 704.2395650826858, 709.3603193697572, 80.2842904240528, 13.0], [5.840339597049198, 669.1104541236928, 397.9986366512601, 671.1733122036792, 184.58988126729284, 3.0], [4.2531845976032825, 520.1979074304949, 562.1756488603826, 964.8098531075638, 146.2689215608968, 19.0], [8.712681161226813, 329.7362098463303, 571.421647794737, 442.1885912272806, 110.57208618116664, 13.0], [5.109030204402814, 748.9679941647319, 670.0957781782574, 654.390347585724, 103.91702817492548, 11.0], [9.814226832470895, 563.7491876686497, 673.6639615368937, 714.7017649816645, 81.05529934618696, 1.0], [2.711712559034632, 38.48793738898616, 510.6527351528365, 611.5396166701556, 92.33262227289612, 12.0], [1.779711539739795, 729.9258285210755, 550.2752805953779, 449.43873381682425, 74.3382753042718, 0.0], [10.641039878679692, 546.2169709631587, 653.4125105799249, 838.5654170982791, 93.09120788541192, 19.0], [20.59708062757116, 847.6831218978857, 504.12952128974825, 700.5401495216078, 54.13415325852041, 6.0], [10.915571488498472, 530.9793565856612, 744.734212172914, 459.9654930610109, 52.526869608325285, 8.0], [11.796056967446212, 302.0726623581013, 673.1010280128289, 715.800194383739, 134.0304698796413, 16.0], [8.936699250918648, 448.9261666745408, 620.5811457833812, 905.2349105079284, 46.6307079206175, 20.0], [9.449289410323445, 700.9798327201801, 646.9128717937103, 644.552609987612, 121.58414460482396, 22.0], [3.0726228703760547, 307.3157350747844, 543.973125967104, 649.6680995794156, 38.16853412222276, 12.0], [5.237955028730882, 389.9258032231104, 524.8450293374858, 885.2007546564979, 113.88912809076768, 22.0], [16.759496997410793, 444.5778550002332, 549.0333340633534, 504.442402737312, 88.77246964570881, 9.0], [4.003468485293077, 212.6570974460318, 659.9960030711819, 909.1328063931024, 134.63948608410212, 15.0], [6.984405818891901, 310.9191438428564, 624.1217976850384, 795.9366605552356, 85.04016623359772, 5.0], [16.57174867110525, 543.1375479510475, 549.2742648368747, 511.3671554029977, 34.713798344855995, 15.0], [9.29551530188002, 800.619436913252, 689.9498091969186, 549.7596500233827, 89.35891142339995, 10.0], [9.57675680428552, 579.2867461776189, 579.1264293196615, 571.2780749476566, 99.78981696643775, 19.0], [7.017800568787459, 488.5346851577446, 516.8027105292998, 374.9653679019365, 91.04067073904632, 2.0], [12.7942255363755, 292.1823065119223, 630.5209811158225, 689.0707623606321, 112.25323462969313, 8.0], [14.57250642385726, 773.790102930058, 478.4364586052595, 518.157949512822, 78.0683570388283, 14.0], [5.049807427354086, 608.3763749867106, 397.5331500077159, 694.4826982146958, 57.09790512363181, 12.0], [3.295974370817433, 323.31755912019014, 599.3874509321683, 456.3585124512175, 65.01053347057328, 9.0], [2.0851953652541986, 724.6621146712251, 521.110344754694, 788.4519855341749, 79.52697966411388, 19.0], [11.472654729132383, 639.8942653776168, 569.501477679618, 760.6438895195754, 79.26954151254938, 23.0], [10.729199062321186, 296.617412658441, 483.9920498226661, 544.1181478909677, 167.18367074009814, 12.0], [25.0, 474.65543593395, 745.0718567643084, 703.9473736823173, 132.6531624974925, 16.0], [8.968121253204117, 376.8068179392658, 520.6208707286235, 662.438083896437, 115.60127613045697, 6.0], [8.026938646636333, 1000.0, 548.1600746195028, 872.3449229229368, 108.38037532185614, 5.0], [10.00826553480821, 632.9362207814147, 645.3459921099881, 495.1281548157692, 103.9931670680641, 12.0], [10.327195400864015, 546.69051371206, 434.806284201278, 627.155303225878, 104.15001635629186, 16.0], [7.651330399837168, 618.1657640191784, 425.9340272101555, 553.8974544938341, 99.89858159240396, 14.0], [5.162027766646375, 844.7906470493745, 645.7143402832096, 656.7227295650251, 96.20357247309737, 3.0], [8.681444919770822, 479.1549638903836, 546.6062843445396, 729.991040417539, 32.24407716347398, 14.0], [8.485769948816111, 352.5226755335609, 491.8205871727013, 552.2288932889437, 83.92618180782519, 13.0], [8.403219972902212, 711.0708070243544, 672.4047807536784, 595.3840667916181, 87.79097584555257, 20.0], [13.559650278957418, 456.263858499672, 502.56940090704086, 794.9061197086044, 72.53080708189864, 10.0], [8.587546159432756, 232.52616772987412, 567.0585240541291, 853.5986992889664, 78.31709191313593, 23.0], [3.350024706492416, 183.82839206098535, 492.4643438432985, 724.0274052804318, 104.02247051542386, 3.0], [5.728117621330852, 882.3241377098833, 636.4124623499789, 629.3055351885262, 118.0277023084988, 2.0], [7.593114622747191, 532.9111417298211, 759.8491431260129, 615.0354618444276, 103.97709385755324, 1.0], [8.832798297559114, 352.6081540018997, 376.67520848014624, 732.2189493090935, 117.2733271594087, 19.0], [11.611349910687572, 441.4162722394841, 613.5189226500775, 684.1650502107308, 117.80324092444071, 9.0], [8.874702956699716, 701.8611795020879, 702.790613026362, 402.65362431630456, 86.0088321941644, 12.0], [1.813497244749884, 426.6486210394887, 549.4226703233072, 735.8577848461854, 110.75922387506452, 19.0], [3.6444196151228527, 137.64506770564543, 464.2749150844937, 520.9145315866157, 42.6088256112896, 20.0], [12.067887705549456, 581.4602130369811, 650.5156232552885, 437.4595387286522, 83.47024319423683, 22.0], [7.465860243957676, 802.0869543933908, 720.198420405252, 399.7482667836912, 79.94445231601101, 17.0], [9.334020703586964, 785.3660794930627, 706.8626613174563, 617.5843443225021, 140.93980406942697, 20.0], [6.890286474711316, 360.16751546568867, 643.9091205497793, 497.5213456726096, 105.11261878201044, 15.0], [5.712545887384383, 817.2931641145949, 611.7496485130371, 301.0457660391204, 54.96637756541252, 10.0], [7.048333076539521, 553.2374934179057, 492.1184933538839, 596.2316739744783, 112.83751414326672, 9.0], [9.400228640132744, 810.2868359696818, 703.8293940412731, 576.4143747671201, 122.29350035303489, 8.0], [8.549260448433003, 643.4217575006385, 697.1948176023009, 627.514832801243, 92.31008235381098, 11.0], [14.519204443981966, 119.60078934183258, 463.433001761368, 555.6653068936434, 115.7076778736841, 18.0], [2.888175019370917, 936.6847343770868, 643.7375673415789, 924.3994495290916, 20.12877848652606, 0.0], [3.1981667998384475, 512.9459627997384, 678.3388650623742, 727.6005942025461, 20.0, 17.0], [7.411867499569346, 644.1984559784543, 705.8536526490608, 575.0580014567784, 59.63992291107434, 13.0], [11.68703792350157, 163.98884288203897, 591.0012457047335, 640.4559743828956, 99.47119101874992, 21.0], [13.14684922666458, 626.3316250195163, 497.0918911024627, 862.1964001243022, 140.9649130882564, 20.0], [5.861348382367535, 775.731933185665, 539.7460584163146, 793.4934217603237, 61.20716933351733, 0.0], [0.8922238048765743, 983.830084149601, 479.1300169852846, 440.5682278437452, 69.38967775136189, 16.0], [5.9853360880564805, 850.0999486988205, 593.8727201862022, 503.8564158760056, 97.65819172174244, 10.0], [10.748845417758112, 638.5561759865099, 570.3737524117571, 532.722968896499, 80.23043615062895, 19.0], [8.453580277642349, 113.90566166290348, 857.4043704922449, 584.0450073596287, 78.52427692109529, 14.0], [11.93252577370164, 533.0681976243553, 707.3612570925758, 560.6865195685937, 82.35575487706123, 3.0], [16.437688761462905, 351.87260869676743, 716.3457680266916, 655.1013882291617, 134.44365474557054, 14.0], [7.7314425763641665, 558.7444687040618, 656.5622988993275, 760.3581966528332, 128.48402866214366, 21.0], [6.420204568355032, 195.91854180122527, 555.9218823997778, 785.3990434073728, 95.70407966877848, 21.0], [4.694521458797055, 447.4582730520247, 566.9532049418311, 600.9803985651064, 144.6344082175893, 13.0], [3.18454336128398, 370.7042290484829, 577.9356840220055, 404.7959721868406, 110.27085062203926, 21.0], [2.5737939721014538, 362.9584110076884, 622.587992724136, 493.9385087192388, 129.61682403615472, 0.0], [5.492805575600987, 365.4409535022114, 646.7003772268138, 572.0337815554955, 69.18360282354114, 8.0], [15.202549065543964, 466.0363209029389, 600.0850353216626, 463.2448311122964, 75.38968838755827, 6.0], [12.588722812736505, 571.5486897620056, 565.1281827188241, 821.128848005242, 75.06853125512094, 22.0], [6.229212254960502, 0.0, 528.7061243414419, 473.0688412128908, 92.398332525755, 10.0], [25.0, 0.0, 432.8821490660096, 742.5594567102805, 96.79739711273784, 22.0], [5.743343653845487, 415.3271650195312, 754.0054998055982, 397.7179467466193, 104.72531925790987, 1.0], [17.303277514368244, 450.5577388235964, 864.6635481918497, 748.5801651944081, 154.19314604245716, 1.0], [19.57033957476144, 547.2778041509852, 743.9204137113901, 629.1096939998621, 116.63834387747144, 5.0], [11.169696146406627, 318.081911285944, 448.9121248883105, 876.8796053453826, 79.5562490150103, 22.0], [5.459205287368532, 502.3662985056704, 694.441988356907, 480.2656090180505, 94.9017686283253, 18.0], [6.763135617547914, 531.6308012545138, 722.853156194498, 565.3827799082608, 140.14826567345614, 19.0], [9.535232456677168, 316.3604757836321, 551.587476165437, 594.435109788434, 118.45639527812368, 13.0], [3.252122305478359, 753.3639554352752, 658.6712687566835, 627.1249894774531, 139.7534649027218, 2.0], [4.39648699336778, 544.2465778637652, 545.2436360972856, 598.1028753646264, 80.51950209057952, 6.0], [4.1484308312365386, 636.7402738658226, 783.4714793776488, 600.998215113652, 73.65015384271084, 6.0], [11.656067270398127, 517.2114328209632, 643.8445106658377, 598.5406184921505, 74.08865895921733, 3.0], [16.349928467436822, 1000.0, 489.9413410354742, 339.32301761293, 82.94040526591594, 22.0], [12.86860490133999, 649.0992672794134, 680.3769717611206, 799.0001752411652, 63.1015097068852, 4.0], [14.868230373519149, 378.0016709481937, 727.0536808188297, 1079.170485145053, 103.8557918512526, 16.0], [5.15810446229694, 705.2417210921001, 623.7757981363051, 694.7406656901483, 104.37161730234934, 3.0], [6.955769922421763, 90.59877063583208, 657.5330030091922, 479.6373728212559, 135.16333911263186, 22.0], [7.288552096815295, 529.9000656667761, 591.6295884925443, 494.8192053727569, 42.19212498259973, 12.0], [11.14417636781146, 579.7900725229977, 463.8863456224519, 741.9761002837351, 66.6761733061274, 15.0], [5.59296073282276, 532.7551905196746, 581.791948989539, 707.8204213772365, 119.1674142359218, 15.0], [14.358086596676683, 416.55528229027817, 662.7647559556269, 607.3564834169993, 100.50419356134702, 1.0], [19.271589907053013, 246.93166079074615, 584.6367758420115, 763.0106556318779, 96.3440736916382, 17.0], [11.471664475946874, 302.6632469872594, 696.0918512087509, 340.4172429868263, 132.91561519730362, 0.0], [6.764786333995331, 525.5187290618459, 552.9382126562899, 763.1093190846685, 64.7013739197152, 17.0], [4.856306396576628, 221.62019189300185, 560.8091007059007, 685.664174689234, 73.13811822037933, 7.0], [4.1552250119122816, 942.5419039118182, 699.7343997592685, 500.6078013825152, 69.60464466669167, 10.0], [1.8305032779201893, 0.0, 626.6223020163654, 435.0014107428973, 34.61275868319931, 23.0], [7.428585858393152, 419.140533113849, 550.5934102613289, 668.1990924118442, 98.98452600248795, 23.0], [7.167018347424655, 527.5974097769242, 724.0566451210349, 861.0857205151646, 79.3144223889114, 14.0], [13.865819992137302, 626.8552319583222, 517.3517138459366, 615.7860152004448, 73.01297574524668, 15.0], [5.834673873523301, 271.01881818439324, 512.7661249179181, 522.0972475650741, 67.59724621048092, 16.0], [7.68947864124699, 783.5736984720531, 260.4661426723111, 678.6932390204379, 94.4675992274482, 16.0], [10.0967894642241, 151.54223904676928, 533.5237741707064, 923.1307715185212, 71.04381456323716, 11.0], [11.36467786386934, 358.467852848108, 757.0460014706739, 714.6949719595574, 101.64702582824717, 9.0], [12.978346624842256, 655.186888819856, 351.32487765930733, 730.9102982314112, 108.7754050200459, 19.0], [9.514659540488728, 760.047141198188, 485.7695831724939, 833.4197953955834, 86.16267078500013, 5.0], [3.392238183927053, 148.99967590472556, 652.6882267068204, 720.8836099075734, 84.12597080721724, 7.0], [10.796968533298642, 1000.0, 330.61598713287265, 532.8545439259768, 100.56473000955472, 12.0], [5.562110519314121, 275.6271092081976, 538.7161663788282, 388.3152572152946, 94.13450839597196, 1.0], [16.77745016436425, 171.95438061805436, 736.9035858732896, 511.7520320332607, 134.90478066064185, 18.0], [8.644972963123417, 625.6787229433143, 845.1630962218787, 484.2910542353488, 104.54796612680614, 21.0], [9.854246899477864, 412.92663660463575, 634.8295878800438, 748.8962179266991, 127.86758000822928, 9.0], [25.0, 375.9869218650639, 488.823957811066, 769.7522077023661, 113.3153300707704, 3.0], [13.106367039830918, 648.9110143924299, 484.7845513622061, 676.3358092573233, 55.44745579620832, 11.0], [10.684092413388734, 118.1760773930477, 617.3889563197085, 459.69620347352406, 78.25205372583054, 17.0], [7.830076200780802, 74.46640000632442, 633.7155985398118, 917.9895535890043, 93.55707730872028, 6.0], [3.6013129933147923, 350.64415871419294, 822.9514512069899, 587.5684679240617, 121.05332931787795, 19.0], [7.487006605955526, 769.4754348633313, 716.4965823044644, 740.5221515452305, 152.83438992315715, 23.0], [4.815829406723669, 361.4861766468041, 610.0927785466189, 657.3987255049339, 80.4755860733837, 23.0], [20.572005141800677, 899.7891711321536, 559.7751296088697, 681.2995224874873, 96.6576829486003, 22.0], [10.76633202546002, 800.2536902929726, 961.7654051162652, 735.2276005010597, 118.9973384506516, 3.0], [16.43817404531464, 570.7365461268712, 675.1913759995075, 545.5969757598573, 93.26517475082326, 6.0], [5.727033209778564, 584.9309648436893, 519.7497943706946, 845.8103639397749, 144.3229033355738, 9.0], [5.053827529219497, 570.5006562542139, 573.0452491690227, 413.074187436909, 54.56639074362472, 14.0], [12.38903255897533, 539.929432170525, 494.705431199628, 494.2165540946173, 115.34059746400978, 11.0], [4.68509191281126, 408.4395756304551, 311.45405689795325, 843.8644061505624, 115.04639763308413, 0.0], [6.445412643487941, 437.4530416982093, 637.5181189448788, 738.5007141656707, 129.56116094941189, 19.0], [21.11275951116202, 232.5099608518545, 813.8945059868777, 1031.0586646522152, 96.86154046468984, 17.0], [6.610727283977785, 0.0, 404.33313573032353, 660.2525665479695, 121.7456365521749, 15.0], [3.060337638022874, 776.0176523205125, 606.6315115054821, 905.064394468864, 87.52582261281316, 4.0], [2.243011636975838, 539.3146091963246, 515.1714265034883, 371.3781826004895, 70.95826189212232, 18.0], [2.7859652583685333, 501.55283890544655, 662.5803030683824, 437.9727540204208, 66.58390966521486, 3.0], [16.977535711266324, 599.6122807144699, 579.3203811930521, 784.3288306125461, 51.7579343787373, 5.0], [5.962208179840848, 292.9006363015877, 436.7934414698879, 388.9448261578272, 81.75923625455445, 16.0], [25.0, 249.73444665389175, 482.2675543260317, 620.3948589485766, 104.34465131171191, 6.0], [19.394555783310693, 429.07882658046185, 500.3639361618951, 825.0813625470738, 88.75603711747235, 23.0], [23.894973470255664, 842.0246204409532, 761.3181849584649, 725.1756896733002, 100.93898299920664, 22.0], [15.08121346541022, 207.8147428767184, 755.1911132316181, 651.6642830479369, 132.04637052159012, 6.0], [12.019221492590049, 0.0, 562.6382535464763, 582.6338796756028, 79.47540355022913, 8.0], [4.932096739692365, 369.6497270453119, 747.3364120141616, 680.3338811887947, 165.2088949413592, 12.0], [8.515711985831647, 785.6749790587958, 579.2179392724591, 736.5872558563294, 117.65728996971716, 19.0], [12.714156822447576, 66.86399391141157, 560.8004689859775, 550.8658494520726, 89.09222832274295, 13.0], [14.5620783411346, 204.92909904428203, 606.8541408661409, 705.7604042018071, 88.41193717323279, 15.0], [3.849693260176113, 565.8866851445422, 634.1790334348364, 811.7646116298066, 64.75610129669451, 22.0], [7.560827822187983, 1000.0, 533.3466248561998, 616.6282800050212, 94.22849105338692, 8.0], [8.108529764411568, 160.4470828684511, 602.6238890618667, 660.2184275210977, 30.152116002843897, 14.0], [7.912811864033556, 150.7664771330102, 694.4787006775828, 722.5668251004167, 132.67298525805677, 4.0], [4.222282172689623, 584.6745922097963, 615.5447567047256, 732.7001324134616, 20.0, 17.0], [22.35146523991552, 801.1686099984261, 605.1339482549184, 781.7680359497659, 96.25252920103546, 20.0], [7.403694529446844, 343.97532191857243, 659.3016257136954, 497.8437566027075, 60.68032996029885, 19.0], [4.437501099013061, 94.65146096945772, 680.0047273030332, 570.6616246422035, 73.96464044229072, 7.0], [22.655621075591664, 608.4373052215865, 592.4861017664493, 576.2181961925519, 77.83544201232367, 14.0], [19.404797404797336, 653.7103493257433, 543.0982731506222, 514.4131752255985, 152.7687838238583, 20.0], [9.29092057787281, 944.6637816345622, 622.7043964169083, 380.2379117963278, 61.90247883588981, 8.0], [2.226689544256838, 664.2367780810697, 408.7243524212041, 300.0, 122.42778634382368, 18.0], [6.322346361405685, 707.0165190930949, 574.7613605395375, 829.317906762947, 129.81205487802026, 21.0], [2.618369665245414, 434.4637140330236, 303.4128110022887, 416.9160493328686, 24.84715399270708, 5.0], [11.508394351407183, 1000.0, 398.90498598762, 884.9763690058862, 76.53941699715698, 3.0], [6.041380533133817, 400.4644973528825, 380.85769806999474, 760.9114232924694, 139.533772784771, 10.0], [8.390358077076629, 347.01946964705746, 725.6822090943662, 589.146743029062, 55.5567964784164, 9.0], [21.548636781211226, 952.3787296296516, 601.677817916319, 742.46331572221, 130.94828109933908, 2.0], [5.757413061237839, 131.44068027614753, 448.5541172140398, 713.7699870070583, 94.46400255796743, 0.0], [11.224825494802571, 38.14790386802128, 658.9279252134428, 484.733845559034, 93.16708011296954, 11.0], [13.195180504354918, 529.8892311496212, 718.6385375239822, 647.7533339555554, 97.084748381589, 22.0], [16.68454087589113, 499.56586918107615, 661.0182667376848, 503.0241841187762, 105.27715494594368, 14.0], [8.020597031029919, 443.0280046123344, 577.3547845808896, 748.1192003256639, 66.33085445917271, 14.0], [7.512655563929333, 748.4824392804129, 627.8983942103241, 542.7168572300918, 87.18741400542383, 20.0], [7.403719281123057, 272.9484544278574, 708.1795839771279, 635.5055695704991, 106.11795241940634, 13.0], [10.600583360433411, 516.0417814061685, 543.6904994804813, 629.0301621483876, 100.30657518290025, 6.0], [18.95153904751424, 409.9735118404533, 603.5840307909391, 668.2368500809073, 105.476766876334, 16.0], [7.193517865999725, 277.19537497331225, 569.1368346472283, 899.3832621671033, 73.4393886755335, 23.0], [1.0999936401993275, 484.5399790098292, 406.3900976817017, 613.8559125716123, 113.96897629349904, 18.0], [14.036765051213628, 713.1489508237757, 709.9881816773162, 832.6890712961773, 98.79427026465125, 6.0], [8.88820731885618, 1000.0, 846.8726237992812, 721.9995491502557, 47.57899844978232, 2.0], [11.52332127199879, 644.8547457247423, 495.1446454510572, 862.7603819054874, 79.05885449099065, 2.0], [2.695561952419063, 0.0, 715.5216281863559, 599.2346322777771, 68.13486439074184, 17.0], [6.056536476285315, 490.4649786395878, 498.96074642860793, 754.4204362954353, 67.7486572108794, 19.0], [13.393105109251332, 512.4506977257081, 615.9345378092478, 641.1097966099488, 80.70559608645706, 21.0], [3.513853540899763, 734.0522523949875, 780.9614728547732, 300.0, 113.40415810075336, 13.0], [11.21012517139076, 546.2025756224348, 537.5473217529467, 486.5234622469287, 76.19623251319827, 2.0], [8.348437700281314, 813.2722293255191, 598.2376302729751, 737.3565418929128, 101.75173655773816, 6.0], [5.2414974463546375, 597.917653903818, 383.8196404190308, 727.4397374419998, 96.83487703552746, 6.0], [4.960965489375216, 889.6995173744722, 520.2954787264522, 455.7533917047615, 71.62842787079929, 4.0], [4.249686774239668, 390.3135199879266, 371.2473213599153, 683.0551323257006, 84.65554607998115, 2.0], [8.859721268490457, 536.0858668241734, 651.5013743849998, 725.4890339788258, 120.94547669896744, 17.0], [9.4889481892771, 706.6603838474741, 699.7965070741893, 644.800164539359, 99.49418920823256, 14.0], [6.755502368742633, 356.9514824266392, 573.4038136115034, 759.3227956489454, 111.46483584203166, 19.0], [4.067456561511532, 302.4967994797309, 634.306436070815, 738.8832961893153, 133.1748963172334, 13.0], [11.02393934238603, 342.66197507832874, 560.9123453699464, 644.7284630892939, 108.0481116253423, 3.0], [14.182092847316852, 312.6960957277694, 688.8654439049324, 573.3762771703822, 37.379333993440326, 23.0], [12.19490764556199, 630.6404802188318, 615.5354322150239, 806.3812388138764, 103.11793148346962, 7.0], [24.432591044814256, 41.29799259136109, 361.7203973035729, 719.4743146818232, 76.45665704979277, 20.0], [7.612830839475996, 111.00749342052325, 712.7976981711384, 442.7851021204553, 149.30907830012487, 12.0], [17.922783857158247, 965.6866230893156, 553.2548646500215, 683.1346484122081, 106.97808222842528, 21.0], [5.583322129902679, 318.7258197564556, 447.5544634549915, 477.34629358134, 77.07685107582361, 7.0], [5.001334271644947, 550.003712792937, 547.3279233187196, 505.3148101356237, 111.99535788323568, 14.0], [9.190541833101772, 127.71339552521796, 371.7516421744273, 552.2134946921544, 97.7237270890132, 16.0], [20.150015874975743, 523.531066947319, 450.01328265131485, 656.7309853489553, 43.461280588405785, 4.0], [7.097478023423228, 320.248333079649, 617.1539532138466, 706.1728366386928, 120.89182343758142, 19.0], [2.693999051688454, 109.22762545989644, 627.205402360868, 588.6587166138092, 82.83380257798765, 3.0], [14.0299532480291, 834.2301093911783, 636.7726601106202, 595.2226075559398, 137.61854878322023, 10.0], [8.133017291143705, 795.6414317652178, 600.3851841316197, 574.5134517047478, 91.2670764756857, 17.0], [10.191628769569144, 473.1613023214959, 516.8561851833585, 807.1270455631443, 138.17880451506406, 9.0], [5.176821070579779, 749.8547581694778, 593.4649032647321, 300.0, 107.74694449717448, 16.0], [9.139027719101575, 975.82317193179, 547.5351063662538, 603.6983885487825, 124.22370107887792, 14.0], [3.008113885170042, 688.5189832895223, 404.90620823097794, 881.7477193073549, 80.2454152855151, 5.0], [22.54270555102461, 619.9199348774907, 507.3102010787295, 727.584507541608, 105.59018541639486, 22.0], [15.51291853993579, 549.7374241208274, 706.7449477402829, 840.0270860850687, 43.212956479638, 20.0], [8.892851608174245, 662.3743412148099, 790.5952982465933, 344.8626710733615, 91.18889291374872, 15.0], [13.810282657960542, 409.0201516275909, 678.1530052942426, 463.6626152796316, 50.46405374206829, 14.0], [5.592836130291088, 430.8410327096384, 785.2437313190896, 727.1072211029701, 60.71068577440702, 7.0], [15.126312930272816, 612.7228438031287, 610.246633216195, 882.5945889886502, 68.27587202766125, 8.0], [6.900761750042477, 381.5337824601178, 809.0851301643047, 478.8145674732908, 114.81262807823934, 10.0], [6.593057753777392, 408.5500818016244, 846.5386128394297, 591.8357325940216, 90.57919595731136, 1.0], [4.452103946390941, 500.04666615874334, 391.9836602356575, 710.2196212800492, 95.94281028061538, 12.0], [10.94379846159488, 787.6518876482673, 626.5791968559399, 432.1252404368454, 89.76236827316725, 3.0], [8.660520714790803, 216.8340616100881, 610.3716226905051, 728.2536889371474, 127.4216149666096, 8.0], [18.24474028945157, 689.7946256763153, 549.1075421299081, 300.0, 95.12125873469482, 4.0], [4.0986104840831485, 697.9491032008676, 659.0298645781664, 723.8359196697282, 102.46511236254136, 18.0], [10.97286482682138, 696.4530023242023, 604.9350082370253, 500.7372548168236, 77.89240421515731, 21.0], [16.36349030440122, 298.31340494762776, 511.7461295529548, 582.8604214225379, 129.4822788099874, 2.0], [25.0, 76.82959612226574, 596.8073873955622, 747.7825921905452, 131.7316786040871, 20.0], [9.589941031377334, 639.9251181028555, 744.5513495242068, 717.0878233238376, 93.89333140793288, 2.0], [6.8176798298057255, 546.8846961034905, 621.9045716259211, 648.8028193472443, 77.47114910958557, 5.0], [9.444257341495817, 751.094158690783, 675.2345697907359, 795.3560827453025, 46.52070998848833, 9.0], [17.436359086143558, 324.86579299761536, 566.753627966509, 394.9735981071765, 68.94210708092876, 4.0], [17.432299696032853, 690.0295940640902, 654.2476812181937, 680.6621557252121, 29.987298154227275, 2.0], [7.931107089488799, 763.6030280548529, 454.8942563420184, 531.4744803712276, 67.68072439736017, 21.0], [17.263570187096573, 476.1725697376624, 540.0335400696249, 787.7047203019979, 105.70910292185802, 23.0], [8.414391430935702, 604.2323758162041, 587.6797669720886, 753.060060445418, 93.48800148202932, 6.0], [13.722922732198388, 595.4916143810237, 446.8396895248154, 758.5022328296008, 164.69800953364182, 21.0], [11.721038949850612, 294.6326177425875, 500.3739937329273, 896.8072749060702, 83.86829855598843, 17.0], [6.645512888483214, 632.774150571152, 637.573412228867, 818.9521298652467, 81.52604777136672, 22.0], [4.563809566730561, 905.6245852252534, 615.5798299545639, 626.3867999339883, 126.88308798586398, 7.0], [16.604801078942128, 505.5509836356591, 696.7638509589251, 649.8098219768755, 37.19191135263336, 0.0], [19.207326001834797, 134.76876533708565, 527.9386778177803, 740.140757106759, 50.546983322615375, 3.0], [10.821697096906462, 328.19336727149755, 606.9122671206289, 670.4262012121744, 114.915240837664, 23.0], [8.407520729843363, 55.14263778709312, 519.3759106939406, 755.7759982864577, 89.13727391091888, 6.0], [3.736023043146514, 822.5652822615161, 684.7354258931191, 315.4519362840457, 89.82511896807631, 13.0], [16.365856370051556, 691.0263240820236, 742.8895339230297, 932.1205278189974, 66.88546526497032, 12.0], [10.100928395692415, 374.6576820830454, 657.9989792703363, 659.1542875334482, 102.19310520747928, 9.0], [7.250646167133261, 145.48501950776586, 727.3433131531899, 406.630200366116, 70.75930236060705, 22.0], [13.944141604421482, 291.6762604393774, 603.8999835465435, 512.9309972624718, 97.18229986091698, 17.0], [3.2552728727875904, 340.4307585757132, 696.6712601994211, 330.6009423246166, 60.245493144402936, 0.0], [4.300747824499971, 527.3518997550146, 609.0501291134952, 486.4647379721374, 89.5280903785233, 3.0], [5.459090443408889, 518.9398357995949, 477.0086320097079, 628.6627357040321, 155.7460682148754, 15.0], [11.510357461679854, 835.3579938434257, 629.6065666438328, 877.2071918741605, 82.25192630787429, 15.0], [9.506571882593784, 326.81128557548936, 480.5643967451389, 711.6011878129701, 91.04725435424662, 17.0], [12.298408247566682, 122.9530364115178, 880.6954741882693, 684.9290917946863, 88.92431007592752, 7.0], [15.744864568884218, 210.75538121442705, 559.9094209469599, 479.73432181764326, 100.23594225033452, 9.0], [12.55251329824268, 494.2437863883748, 677.4045426538796, 526.3272041519733, 109.0445593905276, 15.0], [22.809219021440693, 317.8304832635523, 598.5104136569287, 537.9608459941708, 102.71713252029566, 22.0], [15.72428406681668, 426.99786362174353, 451.9663891156913, 544.0807081720842, 103.37947525317514, 23.0], [7.919603277353307, 386.491604920139, 372.1772363922514, 597.2694685627093, 89.57499846803444, 10.0], [6.030481148981137, 747.5596134262956, 689.8816746088958, 608.5983407425279, 98.00869167521115, 5.0], [8.068933432855008, 353.25857822691944, 785.9209112125271, 599.1382439559728, 88.5813371243735, 0.0], [12.803422052338556, 405.6644807630475, 511.823555134757, 549.3013654597277, 40.62923880718641, 11.0], [6.453962767221478, 679.3208952896939, 401.1968976528492, 670.257974871834, 173.91845029555617, 5.0], [13.005544072778504, 358.754522241981, 560.980472109405, 602.8377824949838, 116.8451824864165, 12.0], [12.16981888468433, 605.8030936142544, 487.545954571352, 410.6649130666659, 65.62042384146346, 16.0], [4.519098858348643, 582.8713155054397, 521.9621184165488, 655.1572733386903, 101.901812131502, 0.0], [15.55915306526745, 535.4414647402699, 673.2448439670342, 632.4854003327636, 54.61577832259383, 7.0], [10.60011118276205, 216.4009043233776, 593.9444343116594, 413.0558821167237, 80.17395352511885, 21.0], [22.193870316599583, 807.4435561304842, 500.45909293415787, 561.3047829955193, 50.54034376622612, 7.0], [8.4322563967222, 1000.0, 670.6200943513434, 739.5608841517038, 93.46445463078716, 23.0], [7.045066975114115, 502.2070345668487, 680.0246814090949, 479.5325092008943, 47.55934321609829, 10.0], [4.12659205707852, 238.41128165740625, 559.553101674645, 463.673139540726, 106.65268682938594, 13.0], [4.081298411646624, 507.2756770812557, 616.5499825101672, 601.2674567012392, 115.4666324273849, 23.0], [14.524816619372974, 747.4650351464998, 662.460225969237, 743.4398404717946, 70.19058201249125, 9.0], [20.956848465841908, 346.2030810710982, 651.1323757180077, 921.5491386084044, 99.82682612597186, 14.0], [7.999080028774313, 467.7752039704526, 604.873690435864, 748.8403032280526, 68.6010694393277, 6.0], [7.681193846503675, 585.1635341030353, 771.9103341870356, 812.7241610234162, 110.35192892809724, 6.0], [8.904655793039641, 800.7619630514812, 669.9216975618223, 702.0046442752496, 74.34145854312621, 9.0], [16.158614856582712, 693.8559332723436, 545.9185180202721, 547.2326030535904, 118.66910375140762, 3.0], [4.546812011694175, 704.2169923830655, 546.6397118761314, 747.7566614895369, 40.65224538995032, 17.0], [9.840539824708545, 393.4328608308937, 533.0296379835992, 637.1587926607818, 64.85072254700339, 9.0], [14.711025865935472, 600.4797387934807, 675.8545136673373, 706.5550756742019, 61.73551036140364, 18.0], [11.42880699481864, 319.5091682187177, 525.2702927997713, 508.0955260473559, 101.10061275214991, 13.0], [0.7683318852033196, 711.3815249713028, 549.0121092956632, 719.9668701259709, 111.6526909832476, 12.0], [12.588380484746006, 787.8049101964034, 726.9626865738975, 631.9237704851155, 103.4191210034806, 14.0], [11.493108163256723, 638.3408921931557, 622.9958354480867, 684.6886620776125, 79.7307041482789, 15.0], [17.912348430058593, 408.1473533012961, 743.8239152545425, 637.2390237762302, 20.0, 1.0], [20.035330404127176, 449.0682179896005, 695.767184802281, 608.4195326987476, 96.16106884272688, 3.0], [17.413792168998278, 951.6290526655024, 458.7993805115924, 722.7036169427613, 118.64147252951209, 20.0], [8.933618441732474, 412.322388844126, 587.7440330111158, 589.0863306662654, 175.41526614899738, 17.0], [2.6011421875282696, 762.498541862642, 718.793884796225, 411.6967804433616, 82.26814337868016, 16.0], [7.646831581086606, 584.7228995546304, 453.3388721818785, 368.9044886476332, 75.04053467197738, 9.0], [8.763215257572424, 416.4658878850393, 650.3652984430412, 677.6624854140466, 104.60764343864462, 20.0], [20.8091675934492, 163.8377431326437, 595.2916231561508, 582.9173277906716, 87.3152467033566, 8.0], [5.211803008738736, 552.9000951471266, 389.4253509453136, 390.6351233851924, 133.26120221789623, 14.0], [3.3271415678678498, 522.7752166515306, 614.359116542272, 665.9802450580671, 70.75434025255586, 11.0], [6.480825744997678, 951.6336678648332, 499.503838064816, 635.9188167539961, 87.53456843026912, 20.0], [5.402349147224797, 284.95182614437545, 459.8817861460288, 575.0579610165902, 121.2863922281704, 8.0], [3.0922281792474937, 1000.0, 601.0968810987159, 687.0213977271279, 104.25092937895656, 13.0], [8.106383986062774, 594.7026323741484, 368.2976472896392, 847.2641015161706, 129.83623015274085, 12.0], [8.971418766872373, 594.4573053055394, 519.0851648898141, 842.6886078027709, 81.90301845711002, 5.0], [2.95813443076497, 351.7319663919425, 906.1409103003176, 671.5646964662877, 132.89853507106113, 18.0], [7.017375768280975, 578.6765768886172, 678.0266872088332, 637.293435070572, 45.08478880169133, 3.0], [5.1754530015113325, 618.2627320583032, 494.2290112599176, 673.6602811417522, 46.28584065582793, 19.0], [2.006615021802933, 734.3318229552192, 768.4999128120476, 696.3510399161522, 133.37119749377763, 16.0], [9.1486346173723, 255.41944810201423, 480.954772706744, 609.7552734412917, 95.2204872639425, 18.0], [9.69659526138706, 851.6952139470096, 533.5565816237433, 676.1814589929653, 122.62447597639986, 2.0], [5.176153467087435, 590.3538809429949, 462.13783080848543, 413.9555643437304, 97.98412874327452, 13.0], [15.48148656435962, 652.057223826828, 630.7295854399225, 662.2023325020147, 94.788772744079, 16.0], [8.64518418626177, 444.76443462380985, 764.2355546516554, 721.5220392536908, 55.15937422627326, 12.0], [5.340992793968628, 267.96434043023737, 480.1192520928173, 576.5237086990392, 94.19481295048544, 8.0], [2.815657963927788, 518.1811568508218, 352.74683551090936, 635.6791033570852, 135.84821708950255, 0.0], [10.00672169137078, 546.8238874558045, 730.9160009423304, 483.2741928948314, 96.70010882215016, 16.0], [11.477325692081989, 153.64293297137732, 786.4560601305217, 462.452962556103, 115.4623458722068, 15.0], [6.249783554969763, 698.3213512912813, 360.590344030232, 681.6222616938786, 114.63121878180854, 17.0], [9.32362599040133, 749.1311584741452, 655.2065209735733, 540.5655476393575, 64.61875950015106, 1.0], [20.034798423714648, 317.5838440447183, 627.1806422538347, 383.4761507635963, 120.71390995258156, 13.0], [14.13013288521458, 244.35366243197717, 505.50296341346944, 483.2332551298337, 85.58813830820458, 21.0], [1.5047198425697907, 424.2596557814247, 432.677319999692, 505.0431359311897, 87.21042775115917, 22.0], [5.300007805536164, 505.9737742465749, 719.0100414535867, 766.1870746597613, 157.9308556701598, 16.0], [7.865652626297018, 433.9572475211664, 621.4884637748634, 676.8671017052701, 112.91910100494556, 2.0], [8.941261753790123, 484.7705418073042, 630.5311748912065, 519.259670276405, 75.84493927168376, 1.0], [9.445340362764837, 555.3694244189791, 525.3876776600159, 561.8351142543261, 96.93614189029168, 7.0], [11.057325633136704, 157.93276531859271, 503.7708954660051, 846.8126624160839, 96.0985977907467, 1.0], [8.028451394484854, 186.6760931741838, 305.4031750471264, 790.4153652795426, 145.5827946301698, 1.0], [8.63221299211182, 0.0, 572.6366364258145, 482.2909358027258, 96.18381741392314, 8.0], [14.545901851861936, 735.6363071133679, 772.6591655345067, 824.5288789439489, 161.23995959992902, 19.0], [10.123377345702943, 503.8067551543623, 493.1694695941616, 637.9056787325285, 56.09801907739882, 20.0], [3.843435080394996, 761.9627828662892, 417.3102366879912, 769.9417439084366, 94.02941738987916, 5.0], [9.113284302235218, 215.56550104715197, 652.2797479748402, 625.503806749774, 72.11433673410335, 11.0], [7.879119745989872, 506.1018348832905, 674.4095998135604, 827.8743798113262, 80.15514183741125, 8.0], [1.9933525108895325, 553.7066265291503, 529.6912847182589, 476.0569082690302, 80.80635486171933, 6.0], [9.396229545421315, 359.5543762924963, 558.148264236043, 611.6824352927727, 73.6607996000337, 11.0], [4.210888994287481, 111.64305038330178, 631.668374066251, 832.4689666848676, 146.90515414956866, 4.0], [14.667939777286865, 936.4678340337892, 581.3340637644808, 300.0, 137.6768098652754, 15.0], [10.96599040168232, 849.8208353175128, 720.0155202904778, 643.0907893813302, 107.07933534392367, 4.0], [3.69221494174981, 404.5169320686588, 892.4691091109064, 590.9680520190219, 80.76944315331886, 11.0], [14.293972099449173, 627.5569112924019, 394.9003733771453, 752.6991845815618, 74.50300871666936, 23.0], [18.073947509041723, 378.80233091077673, 747.5706156669577, 810.4870153116458, 117.25426224433306, 19.0], [7.292252558808781, 251.1490992438131, 612.6311641228001, 759.5425728960447, 104.00452853708524, 8.0], [7.850558101304784, 956.8153762392536, 766.5841126624994, 686.1500380095869, 90.65522139201244, 22.0], [13.51019892207163, 86.15777964854959, 686.9225413518816, 656.084682214304, 111.23579021901293, 6.0], [17.325043982982844, 428.6240757572213, 607.3387751922854, 598.5934067780153, 121.73211622731476, 18.0], [2.460501584759748, 521.0809568105585, 691.202753092924, 681.4958932286946, 90.37879909919084, 19.0], [4.00771780459384, 912.8091446718216, 552.7013074578243, 868.9482244378264, 71.38022261935762, 22.0], [6.561578809214598, 724.1471635590046, 507.0838293474175, 525.4231552583688, 55.36652999518964, 10.0], [7.022765628256349, 81.93755635817035, 697.3035738718019, 589.6877787313889, 102.5189502302439, 20.0], [13.290302434991167, 499.31243974145445, 498.6805861452595, 638.6735041149789, 160.01737919398985, 11.0], [17.37509981178759, 505.2650495910899, 677.0847109969882, 800.1399018793503, 61.680101901462685, 2.0], [18.101699937731883, 442.91541617513946, 603.6897099704998, 683.4953509328308, 127.98407243102172, 16.0], [2.4130377784289485, 654.9522768825752, 320.4670018652909, 797.5222843317786, 74.73331521959632, 6.0], [4.976934346528697, 147.72326806100574, 531.0741059444473, 805.1164985755402, 80.7790539226074, 21.0], [3.571843844345064, 397.8410218658324, 640.1805834029547, 738.805523872997, 122.44770440353494, 9.0], [5.796686447043925, 619.7046870532688, 551.279920330981, 490.40095751516697, 87.34225530013411, 22.0], [4.963327911382531, 669.8939212901603, 294.39214658853774, 899.1640867845072, 102.44779389471438, 14.0], [10.946648344846928, 537.3548683772992, 758.4900153391393, 861.8571006639052, 101.87284268637092, 1.0], [11.718137916525258, 427.9019499249253, 620.3250061525933, 659.5310075753505, 74.78157014709777, 12.0], [8.566937420402736, 470.01468159338185, 653.2487149523766, 442.7344636274668, 121.85292667760596, 3.0], [8.079864259967241, 685.1017805943093, 543.2068617386108, 415.9252763188465, 93.15253438301627, 17.0], [6.742170569668639, 508.475196031365, 663.778762577599, 688.1473439830565, 53.34469253248275, 23.0], [7.251156736904963, 239.53811363986784, 820.1784173504043, 544.7374336803821, 68.22547942602188, 22.0], [19.909763876567304, 191.6256040137741, 511.7912419647171, 392.4216299959432, 20.0, 0.0], [12.718022044195289, 127.5906446409727, 800.0415246428486, 413.09514509554975, 96.55976096523932, 1.0], [6.90509816109599, 863.8952046223385, 647.8276879070409, 579.5026140004987, 134.1219939004651, 1.0], [7.984809265220178, 701.8654589031112, 495.6680702355764, 679.2394360716468, 63.58550516660128, 3.0], [11.093240382129329, 473.3780292048632, 865.4868765109211, 860.856957652374, 112.56614775015484, 2.0], [3.69135354775285, 386.5518787089791, 646.9968287866719, 722.6331161520263, 107.91702152185724, 16.0], [9.302300529963276, 356.5174980081606, 681.1774402527649, 592.7893204292878, 72.44367423237478, 13.0], [12.320621498103614, 773.3092768955524, 567.677747432789, 706.9924336509199, 141.0960246895451, 1.0], [11.427221692020687, 976.1083770436852, 504.4408025604579, 693.5851378382987, 86.8201867932525, 15.0], [5.043201376445592, 348.65176901811526, 620.1375893610947, 760.9317810793909, 155.30805139509658, 10.0], [5.682825950185045, 784.6038801639273, 789.7756468179994, 644.0086829624204, 45.92963092999334, 12.0], [1.8063731441593625, 507.0032288397224, 562.8408225377174, 805.4296626518166, 86.92593345226271, 0.0], [13.068621263780086, 446.89276752238334, 596.7898505311034, 773.6677700341228, 90.4924392795003, 22.0], [24.607781228637265, 876.0535707074807, 774.7594166375502, 662.4050005457856, 93.78160936037956, 1.0], [4.574234687650523, 497.45856981430575, 563.9607785034912, 632.1163465601144, 116.48992952864845, 1.0], [15.6762571632886, 574.0911098136589, 661.2331573402945, 787.962271848564, 196.943819529887, 20.0], [23.402724282800904, 562.0684832278614, 705.2820608087637, 542.9788522839442, 108.65198460604272, 4.0], [6.325453367562881, 93.60015833513704, 576.386286134762, 424.0080756284909, 95.72720727876433, 3.0], [7.991579007502579, 572.1027871790154, 637.1805111839907, 423.15530124740064, 62.11401429297434, 11.0], [9.929131504717889, 359.32699388994183, 679.3323333217407, 530.5438287365737, 38.29680582038739, 2.0], [11.218044703745145, 144.71171496359472, 551.8901767747047, 748.9478448165875, 130.10782494104262, 19.0], [10.246418589822571, 370.7157298878094, 697.8918197987991, 586.5801718085205, 78.5964753225287, 9.0], [6.645997606493902, 368.65278259076354, 534.3616699853474, 680.6774004216838, 173.12475059736846, 19.0], [12.210943327686191, 757.4706347825827, 518.9894905247411, 575.7055893678396, 121.04634855156668, 17.0], [6.743960238941563, 234.05259597449208, 416.7550727015452, 576.8963951024051, 105.24678420343888, 20.0], [10.65947792063298, 302.5381402376863, 539.584518731873, 496.18044882332697, 106.8238668767633, 20.0], [8.381329795566856, 558.8470651370031, 502.1396757755735, 636.2156561876468, 107.11930295166395, 20.0], [2.216902975482657, 168.5515571614381, 551.8951252494031, 623.7254396331565, 74.1629916390869, 9.0], [16.972059105512717, 335.2483104419166, 673.4391158297386, 711.8997534772733, 42.45491888538948, 0.0], [5.783786141252481, 83.88081616669444, 392.91908959498767, 693.1546410517576, 78.4794462045248, 5.0], [17.91902756771584, 730.002865001349, 363.039492649547, 620.427892856238, 94.44931327418789, 10.0], [15.326926907157606, 424.65921988995353, 510.11049400184606, 424.0866732409737, 75.56548096315731, 1.0], [9.753851147447676, 496.0205387494394, 655.3476592303988, 810.5869201367417, 56.41749350120154, 6.0], [11.55644364854297, 729.0025799477726, 634.6637936423624, 637.2433491442965, 72.55369055112575, 19.0], [12.368509169329734, 319.4327790865283, 508.1925200506889, 661.9464034840845, 126.07197400918824, 20.0], [7.543106871533344, 422.00324783799897, 740.3289878587755, 890.0742598716587, 117.44587353182442, 20.0], [7.767000912684357, 235.91800017356465, 520.9908077857991, 786.2333189893925, 100.14229387710516, 20.0], [12.025984331886654, 235.37406975104693, 655.4944783352552, 716.9673798225095, 112.61293778900888, 17.0], [8.435942141842247, 269.8542764985303, 556.4656406958993, 649.6541318943762, 61.041995972170014, 20.0], [6.6598531559554575, 376.8774022915836, 495.7468378807613, 648.2531676748771, 71.13725624735582, 18.0], [5.09825636292276, 403.80111645673367, 527.3919854262313, 686.0171746582794, 104.33045223134576, 3.0], [10.84204952371692, 572.1580377245949, 688.080046177627, 711.0360088784948, 64.86473507201529, 12.0], [5.659467398882716, 277.1957629895251, 643.3230174171306, 906.7102633676202, 98.2519474426463, 17.0], [13.879691636950223, 579.9081689793691, 716.1723196531851, 1020.9899124144216, 57.12426217841679, 19.0], [14.545815439330022, 621.250322880807, 643.3183299044498, 693.5391208105324, 45.51161211699159, 23.0], [11.505111636631812, 383.0444542919335, 406.6596644190937, 327.00858919345507, 117.08236268134632, 12.0], [12.91086148876122, 298.8188568954047, 853.7297801170552, 507.238329006242, 125.8113755653328, 18.0], [11.524847608076088, 521.2697579949975, 645.0202816870162, 921.7245322807652, 76.07537018277675, 16.0], [13.436518365028167, 449.2706342473888, 661.7605865265875, 757.797894238362, 49.38612252587341, 0.0], [13.112974585050026, 555.0879573183863, 554.7794784604127, 713.9069730867494, 28.748834329612933, 5.0], [10.504754174493463, 892.4666090649918, 733.5813951960849, 624.9743132892859, 150.2600044860329, 18.0], [15.436004619579528, 517.1592621069288, 468.8270927279854, 471.6466911161024, 98.3417256051488, 19.0], [7.709263029617202, 340.192359679967, 673.1794893196192, 901.6684632118252, 76.76020108495258, 0.0], [5.202659760341091, 277.2775970144853, 350.44400613035526, 526.5184622788867, 84.08265308517564, 7.0], [5.026159051769053, 286.9495921696041, 681.2650225620221, 759.3967724807418, 37.6000324955321, 16.0], [7.023099511283398, 134.3084232222198, 470.4539506032685, 454.8947345368785, 113.59695421298696, 20.0], [1.970123459816224, 733.1859020863947, 633.2393424512551, 617.6976972757585, 155.00323075656, 21.0], [9.697147680711703, 0.0, 762.8808560095409, 798.9435695679447, 71.5172897336187, 14.0], [12.462645851899044, 622.0262044298264, 698.7260849400527, 612.5458741299809, 166.50371109940596, 17.0], [14.430135027708353, 282.9840233311924, 746.9258004141074, 594.7325186950959, 91.50785301616698, 6.0], [2.5879245433018285, 371.8986963439705, 734.501813237737, 651.5954402523022, 145.0025938563101, 15.0], [5.1359841073349575, 275.50308961390283, 537.0860708508145, 449.5995661347608, 104.21663132887068, 6.0], [13.643136132006209, 861.4661649105077, 671.7996208569425, 421.3779639430977, 104.18393567014986, 16.0], [7.560024354857319, 504.0789004214442, 393.00997248698286, 757.1354724040892, 150.3177722754586, 3.0], [12.303053065541787, 697.5946312607131, 517.5676298466899, 586.66851115733, 118.48111815805552, 11.0], [3.55750741978763, 825.1317146490015, 767.1185865940861, 755.9727140872185, 106.2454423499846, 12.0], [17.801209851994596, 540.8513685056009, 553.7955837088535, 817.7274525726061, 36.75989485784434, 3.0], [3.358481251046763, 201.0940572861969, 375.8397735995108, 829.7963913493195, 49.78374302431295, 20.0], [9.530558536986389, 444.839312471057, 841.6438700797629, 406.8487573135492, 108.75664922056764, 0.0], [9.67648140277812, 314.4379966084494, 667.4700339803663, 545.3256067619153, 139.03349916113726, 0.0], [6.213445432391929, 374.1239595281753, 691.7373864366948, 730.6377960565708, 98.8535320173888, 6.0], [11.272467731658391, 476.3440518376778, 482.3436558093723, 732.9502507855478, 77.26086083363163, 8.0], [11.456613768367095, 466.4610894769339, 634.0867884635844, 987.4156376294436, 39.82488832485179, 0.0], [10.752575664937735, 603.0272902550716, 630.9089965067034, 513.8579700150377, 71.49244827548652, 15.0], [6.140448816942536, 639.3260734451217, 501.42181720515896, 825.2252602181147, 98.92943387591107, 17.0], [5.393522748009159, 466.4637059333856, 579.621637737221, 705.841191505781, 69.30630043262717, 21.0], [6.568299674437306, 600.8672295269794, 419.9438744660378, 403.9646200004795, 66.06661188807603, 14.0], [4.916807683224824, 362.6600362661396, 558.3948560548798, 567.8277144910107, 93.46648209570084, 5.0], [15.834104949434863, 828.6949563991259, 716.8615508711783, 689.3792018484987, 138.4523501036717, 1.0], [6.932017869902271, 358.6221779455884, 510.37820892094976, 394.34499375556015, 115.54158163665252, 8.0], [9.370255401401057, 799.296621155607, 828.8757403711863, 338.9140992542531, 103.57615356110348, 3.0], [13.611889676020382, 130.98322705706954, 427.6690184957933, 385.8136132045035, 100.65459142281264, 12.0], [4.756798020408098, 404.89818564728847, 568.7789580245287, 688.8700482626508, 111.53530107071418, 22.0], [3.255571812132668, 417.2554565965986, 521.228404366305, 577.299911189984, 91.6960527265193, 21.0], [4.928258048912515, 569.7479486871185, 578.3783002352557, 557.0066454029572, 82.66755146964549, 10.0], [9.980216110786117, 715.4542008047059, 764.5769738569107, 562.1567247251409, 117.14581877774812, 12.0], [9.585454624666516, 187.65119923511804, 481.8653404243084, 436.3816382513093, 88.23814153513031, 10.0], [10.74068553058789, 223.00423948927312, 545.8475362884564, 581.5395225739838, 59.34207792597097, 18.0], [8.005225224843356, 149.52298307094054, 678.5406903267767, 376.9402927892247, 138.29900030389365, 7.0], [1.0806701097426286, 830.0460492996044, 611.1790535901183, 828.146170903836, 38.90218114855752, 6.0], [7.850704312035988, 574.4012356992806, 621.0209939170803, 903.8494581015826, 140.54968473701942, 15.0], [14.637519503290653, 228.90719788571928, 484.16683924805307, 301.09044299162423, 110.19365739523288, 2.0], [16.75980470325804, 389.2262815109824, 591.4099834077003, 579.6305511013868, 65.92547801782021, 16.0], [9.005055256914126, 795.4403489153736, 735.7894141847428, 589.3982854047804, 95.72763661660974, 17.0], [9.60650858441599, 430.4701924221905, 406.8574820302825, 623.8273240074462, 82.13409797908197, 14.0], [12.172491675576968, 415.7029655522068, 516.3229019161167, 800.22956365616, 119.23177185065144, 16.0], [16.496921485931253, 304.67582329423533, 382.1350044569596, 580.0683826709104, 118.6379375748816, 10.0], [8.53609252091548, 243.7519215453093, 664.4370836332489, 636.7622805903069, 108.8409056324028, 4.0], [12.076303870984844, 795.8309526107892, 747.0564654377363, 1096.8582445702082, 117.39149242950694, 15.0], [5.967646426179045, 918.475397185515, 577.9908015249549, 633.306063859204, 91.55502724741336, 14.0], [3.611865933854488, 253.70139930316307, 479.3375982581644, 914.9359790449108, 147.85025285581935, 9.0], [6.377220209761189, 851.23558230453, 482.8504258254558, 725.2981486083249, 131.7082065037288, 13.0], [18.596014984929997, 378.15264592747366, 564.3531350552709, 566.9436510386225, 80.6898284503593, 1.0], [12.215159181319605, 633.4119988422218, 563.895354553017, 541.8899019236854, 119.32585148099672, 15.0], [3.8584786360078502, 852.553451203602, 709.4198134216044, 648.909110750044, 116.7074685773017, 18.0], [12.538371678638429, 463.730851074092, 519.7348758668385, 481.0290312880164, 87.20715034136971, 7.0], [4.068355115832442, 418.0145199331602, 652.2094005823031, 520.420105040645, 84.85230067258769, 14.0], [10.655291104275884, 700.2702409470464, 563.3435964902558, 586.5409098461331, 131.4352403533341, 9.0], [4.634264320808507, 222.51179074093773, 510.62916537121953, 694.0015312890844, 94.5591874092332, 4.0], [3.7048592590892806, 786.7653444601076, 560.3548723990828, 923.8995817611568, 102.80515671009364, 19.0], [20.317218274885445, 535.7023481106459, 640.3846058245176, 577.2568359802606, 137.03312196392244, 15.0], [6.928725663868466, 355.5623729614558, 709.7920067858042, 719.2277069567169, 171.0001700209353, 20.0], [12.08935502954461, 348.95809917103634, 569.7632175409813, 405.81532301269226, 79.20125159058308, 21.0], [8.748950657649212, 154.13437204290852, 455.5274243042677, 809.0432961330976, 88.37299247236679, 11.0], [4.294608040785216, 365.9569583440026, 719.9657763918528, 391.7476137609566, 97.22012035828595, 22.0], [7.393641378830551, 433.9905595013997, 492.21967402174624, 769.7579708152152, 97.90209298214042, 12.0], [4.320574968646523, 234.67661707212957, 631.2987801213039, 637.8612570077295, 106.29064665835672, 8.0], [25.0, 488.63375596774887, 597.0200798755495, 631.9020585890315, 80.29464475588998, 16.0], [13.32504379470796, 618.2292711313739, 580.9715889184537, 604.9602923121754, 57.01426941161522, 19.0], [8.651269798256672, 446.63196600014527, 541.5628439774392, 454.8121063411853, 147.61475734549245, 11.0], [11.723528613370313, 665.6814566298167, 674.4178536498557, 462.7594235196796, 113.36247502411348, 4.0], [12.844074147796844, 298.2150042551659, 250.0, 775.4780234058474, 82.44392799981537, 23.0], [12.863330862817651, 750.8456910549851, 736.3764610040445, 651.7905170272082, 108.11339023762196, 11.0], [12.554456815659885, 530.4536764437145, 490.4990670727158, 603.8664897228107, 149.98851577861262, 0.0], [12.01454169755192, 886.3142924386858, 648.9548291700778, 716.5096925954085, 71.35935143673032, 19.0], [9.076734407627006, 1000.0, 642.6030741521914, 665.1965424831877, 101.3342063923088, 15.0], [19.457044003189253, 382.4391019241797, 752.9972858424982, 531.060267509018, 53.30405777295412, 11.0], [9.423584468578763, 392.7340307746354, 623.7298962645683, 1019.1494338763872, 91.44975128378536, 8.0], [8.802022754055306, 438.4394695128207, 500.5068983239146, 761.1134567754661, 113.49786158343672, 12.0], [9.471572246188783, 648.9255625438786, 407.4294014506837, 669.0602860039334, 61.55830232134975, 16.0], [9.87266424379004, 831.4924926187725, 558.143621415243, 572.958304667909, 89.3839589267457, 16.0], [2.603656161927268, 336.7978789575492, 365.0189485340385, 792.9139533791285, 118.99781898589752, 3.0], [4.89485139397863, 595.3110666195182, 524.5817152427094, 531.8648195410856, 112.67600396697767, 13.0], [8.005059936020183, 466.0181775360582, 564.8843255603754, 519.2337888546106, 60.27104957213417, 18.0], [2.504789722651498, 654.5272756351471, 517.210082636529, 903.9988453610572, 106.2132462470672, 6.0], [8.642943299252233, 877.3644233519226, 605.0128388837894, 654.6012764106064, 111.76156511562134, 2.0], [9.796541187619733, 439.9179406417463, 571.667518689695, 535.4976219927331, 98.23956550058092, 0.0], [3.4968876301887906, 82.53308333156735, 560.6776150405461, 650.8528547059516, 127.68017927950076, 11.0], [5.700722299994864, 685.9754192200733, 309.1019259555233, 777.5279830772564, 77.31658044080922, 14.0], [17.13557445201988, 291.8236953486369, 444.6353783177341, 743.1881680932788, 107.3745338837454, 10.0], [11.791934235688943, 464.2210657590556, 677.1188078225852, 796.1393394629678, 60.04399138840293, 9.0], [11.410578970539513, 696.7577092179034, 696.5107527536933, 462.566614589078, 46.98122470449863, 1.0], [15.98544258473441, 574.348713382777, 379.3467938425205, 792.3723742512267, 118.4323694364582, 3.0], [4.958763355576301, 474.5699936117388, 502.2338182519042, 623.141183592328, 95.61687411289482, 16.0], [3.5043509558443207, 356.0295718854197, 728.6358441118953, 663.3651150055815, 81.09747286651711, 16.0], [14.802399623257257, 0.0, 513.8353423913821, 831.4241740467169, 63.04531957713921, 7.0], [8.142566656462105, 674.5802039390505, 586.6211055449866, 755.4088509906934, 119.81307145290022, 22.0], [4.870915045325191, 436.762075193559, 773.0166394819089, 702.6578721432018, 49.75956882101515, 13.0], [2.587685928772336, 878.3340695889315, 556.6107226267927, 352.4004975279568, 124.78632404566062, 23.0], [9.535571616375467, 599.5070074904675, 539.4336707775461, 643.1747216929571, 108.52992111297844, 3.0], [7.578293885177988, 364.7186568662893, 581.1555971288844, 532.2986918644249, 51.26625835955452, 5.0], [6.798444799617582, 766.7175220940101, 822.4092957479794, 756.6567956596044, 98.77884244635918, 18.0], [1.4081919455455023, 787.7823514895476, 653.5442203547376, 798.6437280553492, 174.61174626099051, 23.0], [19.316286969852, 445.24204641188055, 602.2897953335721, 604.6953465825856, 130.6825485339043, 3.0], [3.04561251313445, 798.4112163933069, 537.07680531821, 549.9437786316969, 70.15918462437023, 2.0], [7.214388390282696, 361.1729861534882, 664.6580313817348, 883.4451193273899, 116.50117046716127, 19.0], [6.481077188597052, 539.9119423519758, 679.1639723916958, 559.5278649479459, 52.52260611389365, 5.0], [11.993148875335269, 642.8520969296951, 471.2185788629436, 470.8193739445622, 53.86057471681631, 22.0], [2.6102448162454217, 696.1953960035473, 682.3953830335391, 525.0993270471585, 92.98082990096852, 6.0], [11.18346148940366, 655.3804621539725, 581.8617256051883, 566.7500917828685, 111.54956805737224, 13.0], [9.563540893233275, 416.8357048592069, 602.9758361554328, 520.7955420575678, 119.33990956764254, 11.0], [5.520474551119247, 773.4244101728603, 493.8853196414018, 776.8087074917108, 119.75551173660104, 15.0], [10.957918525288012, 468.8931692233954, 552.0021566605308, 320.3129831638984, 109.84250337605602, 19.0], [8.633783129391166, 357.74315943120166, 476.38292817326055, 517.6159382831706, 46.58026618711699, 7.0], [21.86115887146423, 361.4155081716673, 470.6138195860439, 642.8753831817705, 93.17205094558147, 22.0], [7.574119584858818, 984.9970978690836, 645.2508264512136, 681.536933000344, 124.06045674893664, 1.0], [4.447656106562233, 314.5041429770138, 557.3860121468269, 903.7752256286876, 108.29872779822276, 12.0], [11.247626318855335, 822.9088066901118, 444.0434796165101, 881.7856569145529, 100.4631809617198, 7.0], [6.999736383266151, 216.1333260474576, 861.2463176986051, 409.8894266083187, 104.8767760073094, 20.0], [8.529631052573325, 621.9763612685466, 465.0413952820876, 363.7928089145631, 104.3755963933166, 19.0], [3.525092536985809, 574.2708628727088, 731.0523553223201, 653.7456893556498, 36.05308151069468, 15.0], [12.098110092206053, 905.3067199315568, 422.8243630053838, 549.0222145305368, 129.6467342790618, 3.0], [6.305223061772968, 336.3995262078713, 654.7802802201354, 471.19604870252215, 91.68401277539634, 15.0], [9.999468886591629, 267.3339455099033, 569.9327557657571, 652.4181346190982, 116.22759372456684, 19.0], [24.88022014935812, 470.87240541325446, 524.5131922369915, 684.6045977501652, 25.83063484387353, 5.0], [7.306477019324586, 916.200457310113, 623.6090788518848, 524.1547340533733, 144.82975712421867, 3.0], [13.04282253248378, 278.960380014058, 607.4711827595198, 556.3872882443759, 60.30765068681364, 15.0], [7.620931973685157, 0.0, 502.50417795815457, 535.6937708918525, 171.24023096091577, 22.0], [5.569907524048408, 514.9419593433206, 497.07133430556, 588.2763138168079, 158.5772116168685, 22.0], [17.467565816379288, 737.3807294902522, 411.1302945606719, 731.2166149084605, 55.2525194173261, 20.0], [2.02384989158628, 246.54851684003427, 636.6677261228461, 488.314120827262, 119.51110058777572, 12.0], [14.95515668608734, 711.9830433383161, 449.543420747999, 553.8972509753598, 64.48364426299523, 4.0], [1.823495333417714, 241.0222537468859, 600.4769947820619, 590.1076785371747, 75.81781892376436, 14.0], [14.099129924594266, 463.3603660770802, 671.0973580944859, 648.987139226729, 76.49502197068497, 7.0], [8.608135877159752, 688.1705788215143, 624.9666038188865, 691.3976739368709, 108.4379202013817, 6.0], [4.456112051893116, 203.1336495653904, 513.224244237929, 956.3386760766188, 104.73910371288208, 13.0], [6.578162659764624, 545.3783328768886, 709.0574893619963, 735.8188576412585, 58.9522753497965, 6.0], [12.64564201830977, 803.7433778230593, 551.0434226278496, 739.389702055875, 108.19854210209331, 21.0], [23.588458496767103, 618.4835248028377, 439.900708919401, 610.0080480080694, 88.87672620519675, 6.0], [2.6914539528120063, 319.8434856023081, 565.9526994656067, 503.0201306453852, 40.38341355016277, 22.0], [4.645639208709594, 879.6795881791984, 670.4156853581671, 570.7678977725327, 81.97903208692964, 17.0], [3.2985229221950263, 664.5981279806225, 519.0702822707848, 543.4501264677663, 144.81280560480042, 9.0], [7.582729987654952, 34.888000018923265, 705.3818897964366, 708.8346013331849, 82.99156186391195, 17.0], [14.312376032068308, 716.0609036835566, 507.56087217388193, 402.30778645644006, 39.93389566922282, 13.0], [2.974606196417012, 357.1564368956808, 523.8944167577367, 738.8800579380867, 119.4340019824552, 9.0], [8.470317349280709, 549.1408869106161, 474.47051809760967, 632.4140615845962, 67.17469445870421, 6.0], [14.05746978981787, 752.0010010733149, 682.0451444289636, 760.0022349033558, 101.4328679123734, 6.0], [6.518046465609716, 362.5988538337915, 593.6108915892958, 541.413771626859, 89.84040513775466, 6.0], [9.615742978942878, 88.60184648954652, 482.4285662707231, 514.7945427736631, 147.85010696410632, 4.0], [16.756948538118504, 876.3862979764571, 552.4547589262676, 588.9996921442334, 89.86150026607439, 0.0], [4.070918263894995, 791.1458678004869, 250.0, 585.4026120869261, 60.66232173980672, 13.0], [2.2051050015552125, 478.1859967042636, 693.3566836159972, 770.9522616612395, 80.93457187814528, 17.0], [10.315058493635526, 713.3553628730288, 579.5104032602031, 769.5207249585599, 58.136715528510976, 12.0], [10.482131687204676, 407.65331698407886, 311.2514218788123, 935.7534625303128, 88.72205865006772, 7.0], [6.2321055942859225, 524.2425441902125, 425.19822704111016, 575.6761783223155, 148.09372198511332, 7.0], [5.46699685603089, 463.928014340815, 549.4087124930876, 300.0, 132.45404118268277, 9.0], [8.713046496748929, 417.7639786606744, 442.2995089074195, 588.5866140453774, 156.23571509582257, 16.0], [4.019476103691967, 589.8256450214387, 695.201918842327, 684.5986887812536, 109.37750287690176, 5.0], [4.023972515103657, 197.95529408860293, 650.2287463189385, 757.1796036578253, 99.29300622929296, 14.0], [12.598527063619075, 437.1047923086052, 643.1443132221443, 695.5406406925579, 65.05120825036792, 14.0], [3.827895574769329, 578.6198681208513, 287.47836777423544, 575.5583700365452, 103.0900336696184, 15.0], [2.3189052244473336, 959.5251769904088, 536.9449902951727, 907.1246536123336, 91.08916157061954, 7.0], [5.459790060084945, 208.116446171826, 516.2860259269132, 494.9396299156661, 118.35882093926472, 18.0], [6.329332725146898, 630.1382644331068, 485.5134655074495, 608.8032727157243, 61.33235063621294, 19.0], [11.182762023480535, 657.5371941069653, 550.1789662603546, 488.0135238708788, 96.58047362930246, 14.0], [5.776334413316727, 30.820317329315856, 464.8514151496725, 652.9365143135258, 47.79508627956708, 2.0], [18.332902317758663, 602.633003602313, 625.8264879540261, 888.3508638159304, 105.12837777665555, 5.0], [8.986979490657061, 720.0712697459452, 691.9861625205326, 628.8611114151781, 115.27255644527612, 7.0], [4.968200613740611, 647.1631698456088, 681.6460486108406, 629.7800221859782, 46.354697252335015, 15.0], [10.911749296733332, 479.0181862818312, 583.4506497946223, 395.6056987434112, 54.807432389419816, 14.0], [18.152775703521748, 583.3100091306072, 839.9087635600202, 606.9511721313502, 56.120599380566944, 12.0], [12.403575137098397, 542.4611743122575, 736.7970137400612, 822.5730575050028, 89.67125124070037, 19.0], [2.84227577663038, 77.63566542120134, 587.8130155592938, 612.4645751514388, 53.0754616150867, 14.0], [6.680015683988073, 548.3300811886165, 542.127334775818, 602.3314679131533, 136.3547066645819, 17.0], [9.908684291988038, 408.2323985045819, 760.5632595338304, 815.8571863764939, 108.08458944524108, 12.0], [12.112675442633822, 323.79445725050425, 720.909681931528, 602.9653830544153, 112.97118739203016, 23.0], [7.611377267133466, 1000.0, 430.74873554109246, 843.8080448340684, 47.32856343159991, 7.0], [6.964949378923439, 813.1539299048893, 730.2985583074494, 457.3040606918383, 133.79534256584668, 13.0], [21.77188922156756, 212.4569849214031, 581.4751758331286, 751.6780962261545, 114.48828320295686, 10.0], [12.507623989752146, 586.6590491414597, 496.690616328797, 896.9493811543184, 74.52963080362203, 5.0], [10.931389677758183, 864.5063188910813, 362.49529689112046, 801.178534618237, 48.74696225885607, 23.0], [5.935734650532247, 125.95021148099954, 567.7300566362638, 369.9089672547088, 92.71618330793274, 0.0], [3.97877881553125, 519.2457853517885, 598.5326897736089, 554.2545137853674, 81.00705762916182, 1.0], [11.042181539433148, 531.0181814298734, 790.4793184932261, 613.7072713013325, 82.25219493329121, 12.0], [13.28521952730717, 181.37677937524435, 714.8680569298423, 801.4933245060621, 73.83704011765299, 0.0], [8.504124290195952, 594.8875761530574, 530.0535934348758, 450.2572825682846, 116.64307722032254, 23.0], [12.526630716333628, 779.0615976295313, 646.7210873063684, 497.1594994411028, 20.0, 23.0], [12.708365540004976, 877.0118978547883, 699.1499307207388, 409.2272854730012, 137.89783370996267, 3.0], [13.783852639052627, 437.6589745935136, 610.3148865620659, 526.1299624868279, 124.9434946614836, 23.0], [2.8052922158691924, 1000.0, 529.5415009162555, 462.25900145415306, 101.32055564886436, 22.0], [6.800314160246876, 0.0, 536.1393052507865, 889.6858790294613, 90.67743288642517, 1.0], [11.42608688337218, 589.0541774536156, 646.2567684548343, 552.8934094028051, 127.49151966660747, 20.0], [8.796081154731732, 146.92642001753617, 649.5576678630057, 753.6387938195013, 93.73222531718908, 19.0], [12.730803693819553, 522.5456593742249, 564.8454461083746, 652.6954260828305, 86.64711546429578, 15.0], [10.927607683043671, 369.5233124737426, 285.7017353935993, 650.5941921386882, 129.16154203568294, 2.0], [1.502211208661205, 626.7193640976403, 445.13727272504514, 561.2834270690842, 157.8018717514978, 16.0], [15.64643663556796, 468.27982147445147, 674.3626705934535, 765.5033730696061, 65.86979529768082, 4.0], [6.267733488263959, 293.5039554933709, 344.2994403172949, 673.1516348963512, 107.74397940430356, 7.0], [4.772226890134261, 309.927263109005, 742.1017219640657, 664.511586940857, 62.4082436636736, 11.0], [15.356901892591024, 983.1148743623884, 724.3388713366205, 560.9869084733115, 110.04919046782577, 22.0], [12.180670081316649, 610.8809346156038, 604.9315658319847, 585.5336110770469, 71.654555648735, 11.0], [16.562366201444934, 74.46782336540298, 671.7517241074329, 738.3366676363137, 131.34644770735284, 0.0], [9.972614471950008, 637.6969337791909, 663.4529614030663, 684.5083417318251, 91.99641794116258, 0.0], [6.745448678845086, 451.3308344105007, 585.9752245960544, 668.520847176281, 96.39436007614364, 14.0], [9.304765872588744, 501.1502717742848, 435.1333058893716, 536.8936262498679, 143.9334529236675, 16.0], [4.205355909870858, 704.9720384954493, 716.0727570074024, 941.6161555288244, 73.00991242662903, 1.0], [3.236917600602212, 795.8067934282114, 768.8676013660362, 578.0439890663265, 115.54526582316772, 16.0], [5.649354694755255, 285.2968180198277, 454.9691370909189, 645.4621007411869, 75.99639674348902, 19.0], [19.31893578032034, 492.564586461154, 430.64939937963607, 621.9449556687919, 130.88892631529416, 8.0], [6.028519182579321, 363.4722493153576, 583.5843714291191, 943.2066795891932, 148.60266346490965, 11.0], [2.359587057700116, 649.2155452685656, 524.8530485236261, 606.3778080938619, 87.11278406375102, 3.0], [7.744324755650578, 441.9238765659576, 617.3939088246551, 592.648708853652, 29.177288007162574, 12.0], [5.976912835384372, 507.3128699254689, 744.9592847832464, 667.7466722175004, 74.35588813545232, 3.0], [15.805464252036453, 432.43080974745885, 330.51954740730054, 545.9688665806722, 167.97908972065264, 6.0], [18.56118108576224, 473.4701211076671, 789.7526341942986, 666.7399575352271, 77.23657420430905, 13.0], [6.549959155531578, 926.162751482034, 551.1997869747696, 1005.7405370251654, 148.13063451194182, 8.0], [9.966991546138075, 1000.0, 661.4134774380593, 694.9543333326637, 76.81142029612262, 22.0], [10.45884220400042, 594.273023435619, 674.2471005725359, 689.4868882589872, 76.07001932318096, 0.0], [8.281613166103297, 475.0075550457742, 688.2409385037355, 542.3142660636182, 118.37562731002411, 0.0], [7.385092184254865, 101.09080977757924, 607.4289724835996, 906.3810731529444, 114.78022985207336, 19.0], [4.557535946991943, 309.8954500471566, 635.4672722144095, 940.8549051196968, 120.56522543965715, 3.0], [2.064872935981556, 468.3159910535234, 634.6826460726484, 777.9020563740517, 95.1473167569066, 0.0], [4.5394053894051245, 321.23419484064345, 524.3660639064664, 748.0749834593569, 126.03080294043676, 21.0], [8.104705566049411, 347.5248092820194, 389.0333770275363, 614.7851972841775, 123.84500282911904, 9.0], [2.9785692128998984, 893.8473685140414, 453.6977006681766, 708.7006371266933, 129.47084188994882, 1.0], [9.662894924139575, 507.1977034346204, 587.4914424602358, 524.7263592706239, 76.2932283410258, 20.0], [8.900850997522795, 529.5529978504842, 505.5832011724338, 481.082903633189, 126.00887608336734, 23.0], [5.98353917032416, 331.4523599154433, 505.8604713132558, 426.89601249113855, 79.35991572598542, 14.0], [7.191039623903015, 794.3861811923965, 815.9101254823435, 711.6010959912198, 90.28696310031476, 2.0], [3.3844768749159098, 446.855343002858, 703.0673755991847, 443.2438291518968, 89.67118887130104, 22.0], [13.64497657701678, 369.7840837122859, 569.7508840908032, 575.0546895331275, 65.05688267156685, 8.0], [13.51437523319376, 104.8122956726396, 680.644193478653, 640.8155937197034, 42.93447168920821, 16.0], [3.8295210378737305, 484.7245215621585, 549.5677673290919, 799.2935599283047, 131.71172127514166, 20.0], [3.908838970091192, 361.9854995556496, 613.2868121023796, 563.266975006862, 73.9325137291057, 12.0], [4.580726602986006, 921.1449365042248, 652.9954378102699, 816.9908469109089, 86.90780826274722, 2.0], [11.751689415994427, 960.9746730004877, 549.8872838673745, 790.8738369716273, 73.07849659704053, 3.0], [10.43971074629229, 191.17987630843132, 517.7699342955485, 489.3037508493538, 87.30036389893105, 3.0], [15.25954841786934, 219.46169052975068, 746.8718182162205, 818.4238240827913, 79.82291178822906, 10.0], [16.22037286923599, 430.92009121838805, 673.6935560637728, 607.4628316787696, 133.37251608327267, 6.0], [2.7724809658584686, 295.99052615143205, 329.4466946048739, 575.1573314411123, 77.28487554209707, 10.0], [11.512570664281355, 554.3541439216783, 519.8236883962068, 608.0970086880012, 146.43662443283023, 23.0], [15.86033791507282, 418.34412772860816, 506.2473000366728, 649.4741409861989, 129.17814197591304, 10.0], [6.903928676582108, 570.1330851388146, 525.4634293425122, 587.7433099343548, 80.76659025516562, 9.0], [8.410850464973334, 754.0233804117418, 703.4882746510674, 524.0462509458129, 104.59434904388534, 4.0], [7.654272892942112, 233.94462241612317, 602.304716160443, 537.311660945799, 110.580389887914, 12.0], [7.864585460922626, 748.8028056193497, 433.2444394973345, 670.8414214556024, 120.24821611275172, 11.0], [13.61866031552654, 317.8368964199558, 554.0142195649661, 625.9304672274433, 123.21401023164287, 9.0], [21.329726635643, 1000.0, 665.0085177139428, 729.0964540099345, 100.59897969538216, 6.0], [17.979214996426236, 905.2439915623472, 573.6425981359495, 657.8544345804894, 66.3673070107427, 18.0], [6.165613497403449, 471.7299160284412, 522.638885860381, 661.5520993975638, 114.2266497354086, 18.0], [9.985587880772274, 259.1041388674896, 639.7463134486234, 738.9218352251431, 87.9048087362743, 18.0], [5.058002785562724, 211.28350508135415, 794.1830987326892, 862.8377107869026, 20.0, 11.0], [5.40959537793875, 718.4071255186325, 655.3709799550222, 868.8533522437833, 106.70835431178638, 19.0], [8.187079711671446, 492.39930836737534, 756.1129815638038, 851.0609630171327, 72.97669846283776, 21.0], [5.312926486849574, 731.7167214379896, 530.6862938562116, 780.5673754611961, 81.2483576943894, 14.0], [5.846947718891614, 1000.0, 598.9965466651128, 962.8115933138224, 95.18063407450076, 4.0], [9.846491596326487, 307.4042653209691, 596.5737618557181, 715.6735649365359, 48.39058869890177, 23.0], [11.419862856281402, 192.46543906047805, 614.241490088123, 698.8318722983108, 99.1712443134016, 13.0], [11.245088550529069, 548.1400586331565, 652.8623527657099, 986.5881913254576, 85.4987766717719, 11.0], [11.309571067738702, 786.1899028630744, 489.7237054734266, 300.0, 68.68340042339321, 20.0], [3.562163643837012, 264.8849275535607, 677.6799222616174, 617.9155267033523, 96.38725475054598, 10.0], [5.133893398418216, 508.65618468981154, 404.653687325374, 837.8743593725, 44.29411347101596, 4.0], [8.73311965036413, 842.5109152927475, 439.70920049919823, 797.0394429082526, 97.26880335171212, 15.0], [15.457952925491965, 91.78054037740692, 856.2751321407877, 1034.8477362253443, 109.1179674051008, 17.0], [25.0, 558.8294903597749, 538.7855112625349, 666.1228577497128, 104.15745978852712, 11.0], [10.221455970512096, 364.97422660030384, 687.5423221183443, 585.3773596417707, 118.66652622645928, 4.0], [12.886543326717764, 531.0391477428258, 803.7948470912347, 600.1872879988707, 129.96057147656347, 15.0], [3.2755829038612463, 677.446072645922, 322.2652705465829, 697.7102244291872, 92.55956818892754, 11.0], [12.69052521312151, 555.2021269720228, 735.1495416186851, 893.2471393926982, 127.86348453526848, 22.0], [23.754295444668152, 247.44439947203625, 677.8732327511125, 895.1211513328237, 92.44744503046935, 1.0], [5.121351652433884, 374.6884654416304, 796.8476737119647, 641.5977992249493, 74.01617379352167, 5.0], [5.971680906174837, 856.6482450811166, 458.2172206002448, 687.9512730016688, 83.21748144808761, 22.0], [13.859424748716272, 582.2332234810245, 774.1505588345535, 558.2597631195704, 103.00932769614975, 20.0], [8.666691239232083, 710.0797818962121, 511.27026820767713, 573.9176678591682, 127.5553354451695, 3.0], [1.919319721272832, 562.5738275457607, 726.5733377131988, 806.9385820155758, 95.0438856444137, 12.0], [6.207159998932622, 702.2790170604801, 595.6774757094672, 890.3317083097513, 74.79029385190425, 9.0], [18.518515854773117, 400.9634775156696, 692.4922567722166, 545.4074479295278, 138.32780140461136, 16.0], [7.157023503290308, 308.1220883081411, 605.8643019966183, 686.4434528197027, 51.95550465960034, 13.0], [7.15331905700476, 690.9244289620117, 379.2718098975925, 613.5382915881188, 74.98383846571025, 23.0], [13.365651086074411, 781.843502611389, 250.0, 779.580748932801, 121.78353134680968, 14.0], [10.436495688647591, 271.4296723008396, 625.1898640780189, 591.7225966465317, 147.99371739612684, 7.0], [6.916499060639092, 356.013797144068, 568.3517954842382, 361.0317590810678, 109.42043792661184, 22.0], [5.71720960167873, 459.19339294227336, 647.7865141704352, 800.5536066668003, 102.4966267449149, 23.0], [25.0, 904.956151245662, 569.741584098771, 604.4774555214891, 86.27404489427975, 17.0], [16.829963318688964, 640.7838637920648, 842.1692859261384, 620.0834426167917, 111.44213474147784, 2.0], [11.307157107159815, 168.9782925335283, 627.6644553393966, 715.3381875037759, 98.88103013032438, 16.0], [11.724469445671222, 476.7543021026247, 546.3662726900801, 702.925046472893, 72.36405937956371, 23.0], [7.894407016310999, 438.5303110623914, 640.483567961624, 718.4182488128583, 85.73018694888539, 23.0], [10.363923641617369, 206.68261099901747, 548.5590243742433, 405.5400252130706, 68.97662009958212, 6.0], [7.387945062912474, 473.8020207763181, 611.1516280936785, 733.4463518054739, 95.4084684314248, 12.0], [4.812668355762188, 427.4906400984069, 453.9023775316999, 363.6415474473298, 99.85518419127804, 15.0], [10.601284820383404, 409.5918690957103, 598.9697757768473, 765.4347715671511, 144.03149595943293, 4.0], [6.997576865903321, 398.1546581545472, 542.657712475779, 817.7571923165871, 77.07808622928135, 3.0], [10.922195332962184, 867.8483870872009, 465.2716049557585, 444.7219785912175, 89.57669735589538, 9.0], [12.11351719607966, 409.0742209748712, 837.0883041687171, 699.2649485872323, 132.77543964539615, 12.0], [11.255328507206052, 512.4177640750661, 469.9517556071801, 808.5768694421649, 149.1526805137367, 9.0], [4.550151149163836, 485.3088219308177, 600.8357068130325, 582.64945694427, 109.24132932195997, 13.0], [6.532881888705724, 373.745787408166, 503.66303580614914, 751.0454548562934, 87.39310485655393, 21.0], [7.082336255630406, 519.8761830919244, 635.644736505585, 300.0, 80.80946998187139, 11.0], [14.715677616606833, 516.3376490244408, 447.0737838637299, 814.7294177365117, 39.70799555419144, 0.0], [14.288815438870934, 519.939131724804, 700.8428188722661, 654.4991358039764, 67.1669510271289, 20.0], [5.336508657533452, 285.1917144758947, 844.338742363061, 611.3255935045952, 98.04687270510824, 19.0], [3.501675155659659, 448.4688731868397, 499.2552915479196, 797.606217935672, 89.57042401957034, 19.0], [25.0, 690.0804688374644, 746.0362635217584, 909.3934513585966, 113.70223921985082, 6.0], [11.001640278517112, 455.04584215498994, 554.1744786725172, 722.8573762030995, 89.14391639512496, 8.0], [3.1690848143990653, 540.4454781503146, 780.6949345291303, 891.3070824573342, 73.21828110088371, 7.0], [5.9092224506856965, 541.7592501815888, 862.619878331519, 417.007512120953, 105.2403277803597, 12.0], [15.060757201190576, 604.3487826834793, 571.9553985949849, 749.8923256751099, 170.15795736420392, 13.0], [24.760123981591228, 639.6627287534527, 417.5680702926418, 630.1994220718592, 85.47952497810418, 11.0], [22.51942016738972, 456.4623475626949, 713.7701264923307, 531.7671342925524, 99.1666256403613, 7.0], [10.735504265877672, 272.1661742269759, 626.9300649532939, 360.69778477057366, 120.4077674363804, 20.0], [8.550984463183562, 195.32116783128103, 660.214940133317, 741.7680573503994, 77.85144825292201, 21.0], [4.256012855182565, 754.4216031840897, 559.2185949786762, 781.2935080124794, 133.18272609906148, 17.0], [8.988975725944305, 692.5626119927666, 493.86086176071615, 851.3034126489871, 47.62879666646214, 15.0], [4.657626745455153, 662.3411710843816, 739.2417817160369, 826.4735656452548, 82.76818948194145, 10.0], [6.57348697684152, 702.8008819656684, 638.3747993467239, 534.2435469412445, 85.6633730747434, 15.0], [13.851218607246144, 459.6110040935279, 426.77315589795296, 843.6058272968196, 99.53473235760823, 13.0], [7.365084469873892, 343.78089555861243, 682.4250267843472, 859.4521255867767, 35.85556100030254, 2.0], [7.061189503061262, 254.8981475504905, 658.3402317559801, 611.0024112120942, 135.58219683983287, 20.0], [8.27366453822429, 227.66250085362515, 560.9641290811724, 654.2642395567578, 107.42652754852504, 1.0], [25.0, 803.0396721375828, 794.3775881090786, 509.8768689425687, 128.68494336497383, 2.0], [3.9417769906268982, 376.7137860614628, 525.8192205183283, 872.1329435979999, 143.39107212391613, 3.0], [10.085432141009358, 763.7147673027217, 579.721620221419, 423.2319281046596, 72.78308664988828, 21.0], [21.416817575500104, 453.2785424048373, 692.2377705561837, 730.2456008545654, 20.169379978316076, 4.0], [8.258746256265784, 330.939697939394, 609.2115540396866, 560.4756217826431, 129.39098225890493, 9.0], [16.657066600725734, 226.40880061246415, 433.8692214769876, 483.8220770665233, 86.40851989479839, 19.0], [3.329304332361508, 352.9614107530624, 622.0559247592115, 368.7907579157888, 61.004945123838176, 1.0], [7.515983070117301, 854.5596957265725, 550.2732249641721, 617.6890808043511, 112.2889895912153, 11.0], [4.998236761759813, 228.94748005504687, 702.8395493899516, 460.3122328103142, 168.9067079639678, 4.0], [8.205666497732487, 1000.0, 618.1462012794168, 658.4629179910706, 91.73280514585004, 4.0], [8.311506807530458, 289.84814313258903, 611.3409183656869, 915.4005985100936, 123.53348921906957, 22.0], [8.840140924357355, 687.4874926628114, 474.2098603537138, 679.9260989658333, 65.37491619591594, 0.0], [13.779024401284842, 463.68717643625286, 572.3969630503193, 949.5920753426972, 44.397267036513405, 3.0], [1.3538379893360446, 419.8601043684536, 716.4121312815184, 502.659011978277, 122.41821407968976, 20.0], [5.739389998229105, 182.144805718716, 506.8156386550355, 457.8588800000979, 111.38219515424802, 23.0], [9.68571416574942, 475.40175431219046, 596.0939732294743, 495.9476617657583, 81.02891496022777, 11.0], [2.0038684783953693, 210.4285009084989, 487.6264738021189, 640.2349191867044, 146.18657924076183, 11.0], [12.198602103999365, 79.60572505813991, 457.9388830737997, 1058.4423031586457, 42.06533091029287, 15.0], [5.668898544459281, 529.8996695543623, 589.8169146012777, 400.6827929672839, 58.099097722339415, 23.0], [7.604945238190339, 341.1442232910612, 620.8561397852806, 742.9494677243637, 78.71397215742388, 16.0], [14.131792349351338, 921.0871290312452, 839.2635527079113, 545.00519959392, 84.30808978038185, 20.0], [17.079337644943294, 750.0054287732056, 545.3447539384549, 616.4295423423555, 138.96064122585852, 9.0], [12.410763954646589, 924.3580453341924, 593.979687883934, 461.2970133100494, 96.47593813432287, 16.0], [3.862012077432844, 329.7792738730253, 449.991177843489, 781.870748605931, 89.21138031200698, 16.0], [12.060420985545065, 534.0344350328219, 673.4688017888296, 643.4112300567317, 56.89979696627339, 17.0], [12.523923195379757, 0.0, 647.7713735223937, 813.1180326041249, 127.59900625524592, 1.0], [11.152873378420797, 429.549404828319, 778.6073613229714, 676.2867791048841, 60.90077349598285, 3.0], [18.62549738751871, 524.6245870061703, 733.3946581810451, 360.2564034234687, 56.76166125919479, 18.0], [8.978914040881618, 538.512019425443, 642.168800261074, 814.4814193962821, 82.99225577158482, 10.0], [18.229468627915384, 489.40789736458174, 543.591420049356, 348.0499051535479, 99.89582073965502, 16.0], [18.16809070481123, 516.648142937917, 624.5602125468073, 369.5145975625381, 73.39572199519078, 11.0], [12.7244926911525, 0.0, 586.6783845441887, 512.4195979802546, 116.21052508872816, 1.0], [10.178826119450722, 564.0554590204945, 714.4077039885576, 514.4216687071496, 86.51071169057751, 5.0], [11.189796645269094, 620.8599729825097, 699.1477787954962, 859.4032048072223, 126.58306818834755, 8.0], [20.85565177958861, 834.5063506572128, 564.4301467614293, 624.3265419874455, 110.15331836249224, 4.0], [15.446813252190545, 592.1845148192326, 424.56074642077465, 837.1077937961995, 148.74438021092408, 19.0], [4.195086538215468, 467.31573332169046, 760.006476026451, 564.8123782121071, 81.73313222367533, 20.0], [10.786207667135518, 776.2354805577429, 669.8248748187846, 697.6469553515952, 120.31913073216782, 8.0], [4.826137720383586, 0.0, 561.6611915312798, 721.1063467268309, 41.39566334558952, 22.0], [5.559235451458714, 371.26286734321405, 665.1448017724405, 754.4098125615574, 181.7497607922768, 7.0], [6.739950889243727, 308.5475049853221, 608.9833567500035, 580.2610738376243, 118.81839600413764, 19.0], [3.0942218255520544, 491.36955004217225, 758.2341511463369, 593.3217666822659, 86.1107833333184, 7.0], [4.614449830589929, 0.0, 368.4263721833968, 605.2804131101234, 87.22624368101448, 21.0], [19.189093477839084, 829.9089155609297, 552.337084018388, 402.1381414554794, 87.58930779088193, 20.0], [5.536702510656302, 437.5475250789385, 791.8865418574426, 637.4147205097555, 150.20109869305023, 22.0], [13.822903224488588, 579.6991727897334, 667.5838084160774, 592.447361738558, 121.9564850102365, 5.0], [9.463158842565337, 555.7728062649735, 812.712800197705, 737.0109341116331, 81.70045092843954, 2.0], [11.763173992870604, 815.6224737192376, 453.3831488203842, 663.6951756589502, 57.83554945627152, 4.0], [12.050116830427347, 34.51246628148817, 728.6636059126184, 850.4572705179781, 130.88313906523751, 8.0], [4.47508623433817, 528.0802913674531, 721.4669308144599, 660.513271281285, 120.15646778940872, 6.0], [13.777178440580435, 786.5445651762151, 398.1925757605399, 565.6691455669604, 116.19112091751798, 3.0], [3.5771845084681746, 475.5033883198272, 564.5397252560299, 658.6545833248676, 92.48766693854296, 6.0], [5.385603548862205, 262.1826517154245, 587.2165073767101, 758.6250861315746, 86.01147653267722, 4.0], [11.950406904411423, 349.02747108605354, 405.6532765479113, 384.018888831643, 145.55670534439216, 10.0], [4.273590680698479, 401.7644995483989, 567.0477208175473, 542.7127221773613, 148.3480771791343, 14.0], [1.8374984963566177, 319.73749261925275, 704.7819768934327, 545.8734896171973, 114.20374626924864, 1.0], [10.493964903225612, 461.79722073316015, 553.0892328570312, 711.6485304316251, 124.09522594519726, 12.0], [2.6900954521344023, 877.0618507962504, 629.0559786808832, 729.0929809537924, 102.08294492512132, 21.0], [12.5969653471075, 350.65880861562937, 629.9244339273182, 665.1841591752342, 53.90042682599291, 15.0]])
feature_names = ['Wind Speed', 'Solar Irradiance', 'Previous Generation', 'Previous Load', 'Day Ahead Price', 'Hour']
look_back = 1
is_timeseries = False
output_labels = {'0': {'0': 'OFF', '1': 'ON'}, '1': {'0': 'OFF', '1': 'ON'}, '2': {'0': 'OFF', '1': 'ON'}, '3': {'0': 'OFF', '1': 'ON'}, '4': {'0': 'OFF', '1': 'ON'}, '5': {'0': 'OFF', '1': 'ON'}, '0_name': 'Generator 1', '1_name': 'Generator 2', '2_name': 'Generator 3', '3_name': 'Generator 4', '4_name': 'Generator 5', '5_name': 'Generator 6'}

def get_flattened_data(shap_data):
    if is_timeseries:
        shap_flat = shap_data.reshape(shap_data.shape[0], -1)
        data_flat = data_raw.reshape(data_raw.shape[0], -1)
        flat_names = [f"{feat}_t-{look_back - 1 - i}" for i in range(look_back) for feat in feature_names]
    else:
        shap_flat = shap_data
        data_flat = data_raw
        flat_names = feature_names
    return shap_flat, data_flat, flat_names

print(f"Setup complete. Targets to explain: {list(all_shap_dict.keys())}")


MemoryError: 

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[0])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[1])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[2])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[3])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[4])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()

--- 
# Analysis for: **{'0': 'OFF', '1': 'ON'}**

## 🎯 Analysis for Target: `{'0': 'OFF', '1': 'ON'}`
---

### 1. Feature Impact Distribution (Beeswarm)
**What is this?** A distribution of SHAP values for every sample in the dataset.  
**What to focus on:** * **Horizontal Position:** Points to the right increase the model output; points to the left decrease it.
* **Color:** Represents the feature value (**Red** is high, **Blue** is low). 
* **Insight:** If Red points are on the right, the feature has a positive correlation with the target.

In [ ]:

current_shap_raw = np.array(all_shap_dict[5])
s_flat, d_flat, names = get_flattened_data(current_shap_raw)
plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, show=False)
plt.title(f"Impact Distribution: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.show()

### 2. Global Feature Importance
**What is this?** The mean absolute SHAP value for each feature.  
**What to focus on:** The length of the bar. It represents the "global" influence of a feature—how much it moves the prediction on average, regardless of direction.

In [ ]:

plt.figure(figsize=(10, 6))
shap.summary_plot(s_flat, d_flat, feature_names=names, plot_type='bar', show=False, color='#34495e')
plt.title(f"Mean Influence: {'0': 'OFF', '1': 'ON'}", fontsize=14, pad=20)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 4. Focused View: Top 5 Drivers
**What is this?** A high-precision look at the five most critical variables for `{'0': 'OFF', '1': 'ON'}`.  
**What to focus on:** The gap between the 1st and 5th feature. If the 1st is much larger, the model is heavily reliant on a single "smoking gun" variable.

In [ ]:

mean_shap = np.abs(s_flat).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[-5:]
plt.figure(figsize=(10, 5))
plt.barh([names[i] for i in sorted_idx], mean_shap[sorted_idx], color='#2c3e50')
plt.title(f"Top 5 Drivers: {'0': 'OFF', '1': 'ON'}", fontsize=14)
for i, v in enumerate(mean_shap[sorted_idx]):
    plt.text(v, i, f"  {v:.4f}", va='center', fontweight='bold')
plt.tight_layout()
plt.show()